# Chuẩn bị toàn bộ variant ảnh Hán cổ — Kaggle offline

Điền `PACKAGES_DIR`, `MODELS_DIR`, `DATA_DIR` rồi Run All trên Kaggle Internet Off.
Mặc định `VARIANT_MODE='all_eligible'`: các phương pháp CPU và neural đã tích hợp
chạy độc lập từ original. Phạm vi dữ liệu vẫn là một sách/một trang để kiểm
chi phí trước khi mở rộng; không có OCR hoặc dịch trong notebook này.

Cấu hình mới: **4 L4 trong cùng phiên**, `GPU_IDS=[0,1,2,3]` và
`EXECUTION_MODE='parallel_models'`. Model nhỏ chạy theo đợt, một worker/card.
TADiSR và UniHIR chia model qua GPU ở pha riêng, với ngân sách weights 18 GiB/card
và phần dự trữ cho activation/KV/VAE; không coi tổng VRAM là bộ nhớ của GPU đầu.
TADiSR giữ float32, UniHIR giữ BF16; device map/peak VRAM từng card được ghi log.
Nếu thiếu prompt TADiSR, tạo embedding trên CPU một lần; cần đủ RAM hệ thống.
UniHIR smoke mặc định 10 steps/1 refinement. Chỉ tăng sau khi smoke runtime đích qua.
Các adapter chia model chưa được xác minh inference thật trên bốn L4.

DocEnTR và ba task DE-GAN đã có adapter/source/checkpoint. DE-GAN xóa watermark
cần duyệt vùng vì có thể xóa mực đỏ thật. DiffTSR giữ điều kiện dòng ngang và
chỉ chọn tường minh. LWTformer/GLADUnet chưa có tài nguyên pretrained được xác
minh; thông báo `not_integrated_methods` chỉ rõ chúng, không có adapter giả.

Ảnh gốc không đổi. Các ảnh phục hồi là giả thuyết, chưa chứng minh accuracy
hoặc đúng từng nét. Kiểm runtime/load/inference thật được thực hiện trên Kaggle.


## 1. Cấu hình duy nhất

`PACKAGES_DIR` trỏ đến gốc packages; `MODELS_DIR` trỏ đến gốc models, không trỏ thẳng
vào thư mục một model. `DATA_DIR` có PDF. Không có đường dẫn Kaggle giả định sẵn.
`BOOKS=[]` chọn tất cả, tên/stem/glob như notebook 3; `PAGES` theo từng PDF,
`LIMIT` toàn tập. Không chạy full corpus trước khi xem smoke và dung lượng.


In [ ]:
from pathlib import Path
if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle với Internet Off; hãy gắn Dataset packages/models/PDF.')
PACKAGES_DIR = ''  # thư mục packages, bên trong có image_restoration/
MODELS_DIR = ''    # thư mục models, bên trong có image_restoration/
DATA_DIR = ''      # thư mục chứa PDF
OUTPUT_DIR = '/kaggle/working/image_dataset'
DATASET_DIR = str(Path(OUTPUT_DIR) / '_dataset')  # cache và metadata để resume / notebook 3
RUNTIME_DIR = '/tmp/vtm_image_runtime'

# 4 L4 trong cùng một phiên; chỉ số logic theo CUDA_VISIBLE_DEVICES của nền tảng.
GPU_IDS = [0,1,2,3]
EXECUTION_MODE = 'parallel_models'  # sequential: vẫn giữ placement riêng của model
RUN_PREPARATION = True
RUN_SELF_TESTS = True
VALIDATE_OUTPUT = True
EXPORT_PREVIEWS = True
EXPORT_IMAGE_FOLDERS = True  # ảnh ngoài cùng, mỗi phương pháp một thư mục
EXPORT_DATASET_ZIP = False  # bật để xuất ZIP chỉ chứa PNG theo phương pháp
DATASET_ZIP = OUTPUT_DIR + '.zip'
BOOKS = ['1']
PAGES = '1'         # mỗi PDF, đếm từ 1; smoke nhanh nhất một trang
LIMIT = 1           # tổng số trang sau chọn sách; có thể đổi PAGES='1-2', LIMIT=2
MAX_BOOKS = 1
# Full corpus, chỉ đổi SAU khi smoke thành công:
# BOOKS=[]; PAGES=''; LIMIT=None; MAX_BOOKS=None

# ALL METHODS DEFAULT: có thể chuyển lại selected/classical sau này.
VARIANT_MODE = 'all_eligible'  # mọi phương pháp đã tích hợp; DiffTSR vẫn explicit theo vùng
VARIANTS = ['original','clahe','bicubic_x2','bicubic_x4']  # chỉ dùng khi VARIANT_MODE='selected'
# Denoise không mặc định vì có thể mất nét yếu. Mọi model đều chọn độc lập:
# VARIANTS += ['denoised','realesrgan_x4','tadisr_x4']
# UniHIR is included in all_eligible, with full-page input and GPU/runtime preflight.
# DiffTSR remains explicit-only; UniHIR downloads use its dedicated local script.
# TADiSR luôn lưu thêm tadisr_text_mask. Có thể chọn riêng original + tadisr_x4.
CROP_TYPES = ['page','group','column']
REGION_IDS = []  # []: tất cả; ví dụ ['column_000']; vẫn giữ canonical đầy đủ
RESUME = True
ALLOW_PARTIAL = False  # thiếu model/variant => dừng trước PDF, hoặc không công bố snapshot
MAX_SOURCE_PIXELS = 100_000_000
MAX_OUTPUT_PIXELS = 100_000_000
MAX_STORAGE_BYTES = 15 * 1024**3
REGION_TIMEOUT_SECONDS = 600
MODEL_LOAD_TIMEOUT_SECONDS = 900
BATCH_SIZE = 1
DESKEW = False
DEWARP = False

LAYOUT = {'dpi':200,'analysis_max_side':1200,'reading_direction':'vertical_rtl',
          'columns_per_group':3,'column_cuts':None,'page_column_cuts':{}}
# Cấu hình nguồn của các sách hiện có được nhúng; không cần configs/ trên Kaggle.
# Không lấy max_side hay token budget OCR làm giới hạn ảnh canonical.
BOOK_LAYOUTS = {'1':{'dpi':225},'2':{'dpi':226},'3':{'dpi':226},
    '4':{'dpi':181},'5':{'dpi':181},'6':{'dpi':226},
    'bien':{'dpi':200,'reading_direction':'horizontal_ltr'},'kinh':{'dpi':45}}
VARIANT_OPTIONS = {'clahe':{'clip_limit':1.5,'tile_grid':8},
                   'denoised':{'h':2.0,'template_window':7,'search_window':21}}
# Document restoration: each task runs independently.
MODEL_OPTIONS = {
    'realesrgan_x4':{'tile':256,'overlap':24,'half':False,'device':'cuda','seed':42},
    'tadisr_x4':{'tile':768,'overlap':256,'device':'cuda','seed':42},
    'difftsr':{'steps':20,'device':'cuda','seed':42},  # đối chứng 20 với default upstream 200
    'unihir_restoration':{'steps':10,'ref_count':1,'max_memory':'18GiB','device':'cuda','seed':42},
}
# New models accept upright pages/groups/columns; no glyph segmentation required.
# VARIANTS += ['background_normalized','sauvola','niblack','bilateral','unsharp']
# VARIANTS += ['docres_appearance','docres_deblurring','docres_deshadowing','docres_binarization',
#              'docdiff_deblurring','restormer_motion_deblurring',
#              'restormer_defocus_deblurring','restormer_real_denoising','restormer_gaussian_denoising']
for _name in ('docres_appearance','docres_deblurring','docres_deshadowing','docres_binarization',
              'docdiff_deblurring','restormer_motion_deblurring','restormer_defocus_deblurring',
              'restormer_real_denoising','restormer_gaussian_denoising'):
    MODEL_OPTIONS.setdefault(_name, {'tile':512,'overlap':32,'device':'cuda','seed':42})
# Model lớn chiếm riêng cả bốn GPU; không chạy cùng các worker khác.
# max_memory là ngân sách WEIGHTS trên MỖI GPU; chừa activation/KV/VAE.
for _name in ('tadisr_x4','unihir_restoration'):
    MODEL_OPTIONS[_name].update(placement='sharded',gpu_ids=GPU_IDS.copy(),
        max_memory='18GiB',activation_reserve_gib=4,entry_reserve_gib=2)
MODEL_OPTIONS['tadisr_x4'].update(tile=512,overlap=128,prompt_device='cpu')
# Prompt CPU chỉ tạo một lần nếu chưa có embedding; cần đủ RAM hệ thống.
# Sau smoke có thể tự tăng steps/ref_count UniHIR; không tự mở rộng dữ liệu.
# Để trống = Real-ESRGAN dùng kernel, TADiSR tạo venv overlay riêng từ wheel Linux.
# DocRes/DocDiff/Restormer dùng overlay document_runtime.json.
# DiffTSR/UniHIR giữ overlay riêng và điều kiện đầu vào tương ứng.
# BEGIN LEARNED DOCUMENT OPTIONS
for _name in ('docentr_binarization','degan_binarization','degan_deblurring','degan_watermark_removal'):
    MODEL_OPTIONS.setdefault(_name, {'patch_size':256,'tile':256,'overlap':0,'device':'cuda','seed':42})
# DE-GAN watermark removal cần REVIEWED_REGIONS['book:page:region']['watermark_removal_approved']=True.
# DocEnTR/DE-GAN dùng learned_documents_runtime.json, Keras backend Torch trong worker riêng.
# END LEARNED DOCUMENT OPTIONS
RUNTIME_PYTHONS = {}
# Source Python của Torch kernel có thể khác RECORD: ghi hash/cảnh báo; native vẫn strict.
KERNEL_TORCH_SOURCE_POLICY = 'warn'
# Điều kiện do người khảo sát khai báo, không tự phát hiện degradation.
APPLICABILITY = {}
# Ví dụ: {'color_gmm':{'books':['1','2','3','4','5','6']},
#         'restormer_motion_deblurring':{'require_review':True}}
# REVIEWED_REGIONS có thể thêm approved_variants=[...] / excluded_variants=[...].
# VARIANT_MODE='classical' chọn mọi phương pháp CPU độc lập. Có thể chọn riêng:
# VARIANTS += ['retinex_msr','retinex_msrcp','wolf','nick','color_pca','color_gmm',
#              'mean_shift','anisotropic_diffusion','background_morphological',
#              'background_guided','background_otsu']
REVIEWED_REGIONS = {}  # chỉ DiffTSR cần duyệt dòng ngang; UniHIR chạy mọi trang/vùng đã chọn
# Ví dụ khai báo cho DiffTSR sau khi xem original:
# REVIEWED_REGIONS['book:3:column_000'] = {
#   'upright_horizontal_single_line':True,'character_count':12,
#   'allow_simplified_text_prior':True} # DiffTSR có zhconv zh-cn nội bộ: nguy cơ phồn thể
GENERATE_TADISR_PROMPT_OFFLINE = True  # nếu thiếu embedding, suy ra offline trong worker riêng từ base đã xác minh

# all_eligible chọn classical + neural tổng quát; ALLOW_PARTIAL=True để ghi phần thiếu/lỗi.
# UniHIR có trong all_eligible, giữ preflight GPU/runtime; chỉ DiffTSR bị loại khỏi tự chọn.
CONFIG = dict(packages_dir=PACKAGES_DIR, models_dir=MODELS_DIR, data_dir=DATA_DIR,
    output_dir=DATASET_DIR, gpu_ids=GPU_IDS, execution_mode=EXECUTION_MODE, books=BOOKS, pages=PAGES, limit=LIMIT, max_books=MAX_BOOKS,
    variants=VARIANTS, crop_types=CROP_TYPES, region_ids=REGION_IDS, resume=RESUME, allow_partial=ALLOW_PARTIAL,
    max_source_pixels=MAX_SOURCE_PIXELS,max_output_pixels=MAX_OUTPUT_PIXELS,
    max_storage_bytes=MAX_STORAGE_BYTES,timeout_seconds=REGION_TIMEOUT_SECONDS,
    load_timeout_seconds=MODEL_LOAD_TIMEOUT_SECONDS,batch_size=BATCH_SIZE,
    deskew=DESKEW,dewarp=DEWARP,layout=LAYOUT,book_layouts=BOOK_LAYOUTS,
    variant_options=VARIANT_OPTIONS,model_options=MODEL_OPTIONS,
    runtime_pythons=RUNTIME_PYTHONS,reviewed_regions=REVIEWED_REGIONS,
    kernel_torch_source_policy=KERNEL_TORCH_SOURCE_POLICY,
    generate_tadisr_prompt_offline=GENERATE_TADISR_PROMPT_OFFLINE,applicability=APPLICABILITY)


## 2. Mã PDF, layout và dataset canonical được nhúng

PDFium kiểm tra placement, crop, rotation, clip, mask và annotation; native chỉ
được chọn khi bằng pixel với trang hiển thị. Nếu không đạt, render giữ mật độ ảnh
nguồn. Native scan 600×450 không có thêm thông tin nét khi tăng DPI.
Ảnh phân tích tối đa 1200 chỉ phục vụ projection; crop lấy từ nguồn không cap.
Giữ đủ page/group/column, RTL, manual cuts và QC phân hoạch. Column là ứng viên
hình học, chưa được xác nhận là một cột văn bản. Deskew/dewarp mặc định tắt vì
chưa có bằng chứng cần thiết trên các PDF này; không bóp méo nét để đẹp hơn.


In [ ]:
from __future__ import annotations
import math
PDF_IMAGE_VERSION = 'pdfium-visible-v1'

def pdf_pixel_to_page(size, bbox, rotation):
    """Six affine coefficients: x'=a*x+c*y+e, y'=b*x+d*y+f.

    Input pixels have a top-left origin; output is unrotated PDF canvas units.
    PDFium does not expose /UserUnit: do not label these physical DPI values.
    """
    w, h = size
    l, b, r, t = bbox
    if rotation == 90:
        return [0, (t - b) / w, (r - l) / h, 0, l, b]
    if rotation == 180:
        return [-(r - l) / w, 0, 0, (t - b) / h, r, b]
    if rotation == 270:
        return [0, -(t - b) / w, -(r - l) / h, 0, r, t]
    return [(r - l) / w, 0, 0, -(t - b) / h, l, t]

def pdf_visible_render(page, scale, max_pixels):
    w, h = page.get_size()
    size = [math.ceil(w * scale), math.ceil(h * scale)]
    if min(size) < 1 or size[0] * size[1] > max_pixels:
        raise ValueError(f'Page render {size} exceeds max_source_pixels={max_pixels}; review acquisition settings (source is never silently reduced)')
    bitmap = page.render(scale=scale, draw_annots=True)
    try:
        return bitmap.to_pil().convert('RGB')
    finally:
        bitmap.close()

def acquire_pdf_image(page, *, dpi=200, max_source_pixels=100000000, strategy='auto'):
    """Return owned RGB pixels and evidence for this page's acquisition decision.

    `dpi` is the rendering fallback in pixels per 72 PDF canvas units. For a
    raster-only full-page scan use its native density, including when raw decode
    fails validation. Mixed/vector pages use at least the requested resolution.
    """
    import pypdfium2 as pdfium
    from PIL import ImageChops
    if strategy not in {'auto', 'render'} or dpi <= 0 or max_source_pixels <= 0:
        raise ValueError('Invalid PDF acquisition options')
    width, height = page.get_size()
    bbox, rotation = (list(page.get_bbox()), page.get_rotation())
    objects = list(page.get_objects())
    annotations = pdfium.raw.FPDFPage_GetAnnotCount(page)
    images, density = ([], 0.0)
    for obj in objects:
        if obj.type != pdfium.raw.FPDF_PAGEOBJ_IMAGE:
            continue
        pixels = obj.get_px_size()
        matrix = obj.get_matrix()
        container = obj.container
        while container is not None:
            matrix = matrix.multiply(container.get_matrix())
            container = container.container
        axes = [math.hypot(matrix.a, matrix.b), math.hypot(matrix.c, matrix.d)]
        corners = [matrix.on_point(x, y) for x, y in ((0, 0), (1, 0), (1, 1), (0, 1))]
        bounds = [min((c[0] for c in corners)), min((c[1] for c in corners)), max((c[0] for c in corners)), max((c[1] for c in corners))]
        full = max(0, min(bbox[2], bounds[2]) - max(bbox[0], bounds[0])) * max(0, min(bbox[3], bounds[3]) - max(bbox[1], bounds[1])) >= 0.8 * (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
        native = max(pixels[0] / axes[0], pixels[1] / axes[1]) if min(axes) > 0 else 0
        density = max(density, native)
        images.append({'native_size': list(pixels), 'bounds_pdf': bounds, 'matrix_pdf': [matrix.a, matrix.b, matrix.c, matrix.d, matrix.e, matrix.f], 'level': obj.level, 'full_page_scan': full, 'pixels_per_72_canvas_units': 72 * native})
    reasons = []
    simple = len(objects) == 1 and objects[0].type == pdfium.raw.FPDF_PAGEOBJ_IMAGE
    obj = objects[0] if simple else None
    if not simple:
        reasons.append('not_one_top_level_image; preserve_composition')
    if rotation:
        reasons.append('page_rotation')
    if annotations != 0:
        reasons.append('page_annotations_or_unknown')
    if simple:
        m = obj.get_matrix()
        l, b, r, t = bbox
        if obj.level or abs(m.b) > 1e-06 or abs(m.c) > 1e-06 or (m.a <= 0) or (m.d <= 0) or any((abs(a - v) > 0.001 for a, v in zip(obj.get_bounds(), bbox))) or any((abs(a - v) > 0.001 for a, v in zip([m.e, m.f, m.a, m.d], [l, b, r - l, t - b]))):
            reasons.append('placement_crop_rotation_or_reflection')
        clip = pdfium.raw.FPDFPageObj_GetClipPath(obj)
        if clip and pdfium.raw.FPDFClipPath_CountPaths(clip) > 0:
            reasons.append('image_clip_path')
    image, method, validation = (None, 'pdf_render', None)
    if strategy == 'auto' and (not reasons):
        nw, nh = obj.get_px_size()
        if nw * nh > max_source_pixels:
            raise ValueError('Native image exceeds max_source_pixels')
        scale = math.nextafter(min(nw / width, nh / height), 0)
        if abs(nw / width - nh / height) > 0.0001 * max(nw / width, nh / height):
            reasons.append('anisotropic_native_geometry')
        else:
            bitmap = obj.get_bitmap(render=False)
            try:
                candidate = bitmap.to_pil().convert('RGB')
            finally:
                if hasattr(bitmap, 'warn_on_close'):
                    bitmap.warn_on_close = False
                bitmap.close()
            visible = pdf_visible_render(page, scale, max_source_pixels)
            equal = candidate.size == visible.size and ImageChops.difference(candidate, visible).getbbox() is None
            validation = {'method': 'full_pixel_equality_to_visible_render', 'equal': equal, 'render_size': list(visible.size), 'scale': scale}
            if equal:
                image, method = (candidate, 'native_validated')
            else:
                reasons.append('native_pixels_differ_from_visible_page')
    if image is None:
        raster_only = bool(images) and all((o.type in {pdfium.raw.FPDF_PAGEOBJ_IMAGE, pdfium.raw.FPDF_PAGEOBJ_FORM} for o in objects)) and (annotations == 0)
        scale = density if raster_only and density else max(dpi / 72, density)
        image = pdf_visible_render(page, scale, max_source_pixels)
    else:
        scale = validation['scale']
    meta = {'version': PDF_IMAGE_VERSION, 'method': method, 'size': list(image.size), 'page_size_pdf': [width, height], 'page_bbox_pdf': bbox, 'media_box_pdf': list(page.get_mediabox()), 'crop_box_pdf': list(page.get_cropbox()), 'rotation': rotation, 'object_types': [o.type for o in objects], 'annotations': annotations, 'embedded_images': images, 'requested_dpi': dpi, 'render_scale': scale, 'native_validation': validation, 'fallback_reasons': reasons, 'source_downsampled': False, 'source_to_pdf': pdf_pixel_to_page(image.size, bbox, rotation), 'resolution_unit': 'pixels_per_72_pdf_canvas_units; physical_UserUnit_unavailable', 'warnings': ['low_native_resolution; interpolation_cannot_recover_detail'] if images and max((max(i['native_size']) for i in images)) <= 750 else []}
    return (image, meta)

from math import floor, isfinite, sqrt
from pathlib import Path

def projection_mask(image, options):
    """Analysis-only ink mask; never use these pixels for canonical/OCR crops."""
    from PIL import Image, ImageChops, ImageFilter, ImageOps
    gray = image.convert('L')
    gray.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    radius = max(2, min(gray.size) / 80)
    mask = ImageChops.subtract(gray.filter(ImageFilter.GaussianBlur(radius)), gray)
    return ImageOps.autocontrast(mask).point(lambda v: 255 if v > options.get('ink_threshold', 40) else 0)

def projection_gaps(image, axis, options):
    """Find internal low-ink gutters; suppress long rules in analysis only."""
    mask = projection_mask(image, options)
    w, h = mask.size
    data = mask.tobytes()
    left, right = (max(1, int(w * 0.03)), min(w - 1, int(w * 0.97)))
    top, bottom = (max(1, int(h * 0.03)), min(h - 1, int(h * 0.97)))
    if left >= right or top >= bottom:
        return []
    rows = [sum((v > 0 for v in data[y * w + left:y * w + right])) for y in range(top, bottom)]
    valid_rows = [y for y, count in zip(range(top, bottom), rows) if count < 0.7 * (right - left)]
    if not valid_rows:
        return []
    cols = [sum((data[y * w + x] > 0 for y in valid_rows)) for x in range(left, right)]
    valid_cols = [x for x, count in zip(range(left, right), cols) if count < 0.7 * len(valid_rows)]
    if not valid_cols:
        return []
    if axis == 'x':
        values = [sum((data[y * w + x] > 0 for y in valid_rows)) / len(valid_rows) for x in range(w)]
        extent, length = (image.width, w)
    else:
        values = [sum((data[y * w + x] > 0 for x in valid_cols)) / len(valid_cols) for y in range(h)]
        extent, length = (image.height, h)
    values = [0.0 if value >= 0.7 else value for value in values]
    threshold = options.get('gap_ink_fraction', 0.006)
    minimum = max(2, round(length * options.get('min_gap_fraction', 0.006)))
    edge = max(minimum, round(length * 0.06))
    gaps, start = ([], None)
    for i in range(edge, length - edge + 1):
        blank = i < length - edge and values[i] <= threshold
        if blank and start is None:
            start = i
        if not blank and start is not None:
            if i - start >= minimum:
                cut = round((start + i) / 2 * extent / length)
                span = max(minimum * 3, round(length * 0.035))
                before = values[max(edge, start - span):start]
                after = values[i:min(length - edge, i + span)]
                support = max(3, round(length * 0.006))
                if sum((v > threshold * 2 for v in before)) >= support and sum((v > threshold * 2 for v in after)) >= support:
                    gaps.append(cut)
            start = None
    return gaps

def _partition_box(box, cuts, axis, direction):
    x1, y1, x2, y2 = box
    ends = [0, *cuts, x2 - x1 if axis == 'x' else y2 - y1]
    result = [(x1 + a, y1, x1 + b, y2) for a, b in zip(ends, ends[1:])] if axis == 'x' else [(x1, y1 + a, x2, y1 + b) for a, b in zip(ends, ends[1:])]
    return result[::-1] if axis == 'x' and direction == 'vertical_rtl' else result

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from fnmatch import fnmatch
from datetime import datetime, timezone
from pathlib import Path
from tqdm.auto import tqdm

def _sanitize(name: str) -> str:
    return 'run'

def _run_dir(workdir: str | Path, kind: str, name: str, cfg) -> Path:
    digest = cfg.cache_key()[:8]
    return Path(workdir) / 'runs' / f'{kind}_{_sanitize(name)}_{digest}'

def parse_pages(spec: str | int, n: int) -> list[int]:
    """'1-10,15' -> [0..9, 14] (đếm từ 1); '' -> mọi trang."""
    spec = str(spec)
    if not spec.strip():
        return list(range(n))
    out: list[int] = []
    for part in spec.split(','):
        a, _, b = part.strip().partition('-')
        out += range(int(a) - 1, min(int(b or a), n))
    return sorted(set((p for p in out if 0 <= p < n)))

def _pdf_pages(path: Path) -> int:
    import pypdfium2 as pdfium
    doc = pdfium.PdfDocument(str(path))
    try:
        return len(doc)
    finally:
        doc.close()

def _entry_matches(entry: dict, pattern: str) -> bool:
    return fnmatch(entry['name'], pattern) or fnmatch(entry['stem'], pattern) or fnmatch(Path(entry['path']).name, pattern)

def select_entries(entries: list[dict], patterns: list[str]) -> list[dict]:
    """Chọn sách theo tên/glob. [] = tất cả. Pattern không khớp -> báo lỗi."""
    if not patterns:
        return entries
    chosen: list[dict] = []
    seen: set[str] = set()
    missing: list[str] = []
    for pat in patterns:
        hits = [e for e in entries if _entry_matches(e, pat)]
        if not hits:
            missing.append(pat)
        for e in hits:
            if e['path'] not in seen:
                chosen.append(e)
                seen.add(e['path'])
    if missing:
        names = ', '.join((e['name'] for e in entries))
        raise ValueError(f'books không khớp: {missing}. Có sẵn: {names}')
    return chosen

def list_pdfs(input_dir: str | Path) -> list[dict]:
    """Liệt kê PDF trong input_dir: name (đã làm sạch), stem, path, pages."""
    root = Path(input_dir)
    if not root.exists():
        raise FileNotFoundError(f'không thấy input_dir: {root}')
    files = [p for p in sorted(root.rglob('*.pdf')) if '__MACOSX' not in str(p)]
    if not files:
        raise FileNotFoundError(f'không thấy PDF nào trong {root}')
    entries: list[dict] = []
    used: set[str] = set()
    for p in files:
        name = _sanitize(p.stem)
        while name in used:
            name += '_'
        used.add(name)
        entries.append({'name': name, 'stem': p.stem, 'path': str(p)})
    for entry, path in zip(entries, files):
        entry['n_pages'] = _pdf_pages(path)
    return entries

import os
from pathlib import Path, PureWindowsPath
import re


def runtime_path(value: str | Path) -> Path:
    return Path(value)

import json
import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Event

def gpu_worker_environment(indices, environment=None):
    """Select logical devices, respecting a mask inherited from the platform."""
    env = dict(os.environ if environment is None else environment)
    indices = list(indices)
    if not indices or any((type(i) is not int or i < 0 for i in indices)) or len(set(indices)) != len(indices):
        raise ValueError('GPU IDs must be distinct nonnegative logical indices')
    inherited = env.get('CUDA_VISIBLE_DEVICES')
    visible = inherited.split(',') if inherited is not None else None
    if visible is not None:
        visible = [v.strip() for v in visible if v.strip() and v.strip() != '-1']
        if any((i >= len(visible) for i in indices)):
            raise ValueError(f'GPU IDs {indices} exceed inherited CUDA_VISIBLE_DEVICES={inherited!r}')
    env['CUDA_VISIBLE_DEVICES'] = ','.join((visible[i] if visible is not None else str(i) for i in indices))
    return env

def validate_ocr_gpu_map(engines, mapping):
    if set(mapping) != set(engines):
        raise ValueError('GPU_MAP must assign exactly one GPU to every OCR engine')
    ids = list(mapping.values())
    if any((type(i) is not int or i < 0 for i in ids)) or len(set(ids)) != len(ids):
        raise ValueError('Parallel OCR requires one distinct nonnegative GPU ID per engine')
    return dict(mapping)

def probe_gpu_inventory():
    """Probe in a child so importing Torch cannot freeze the parent's CUDA mask."""
    source = "import json, torch\ndevices=[]\nfor i in range(torch.cuda.device_count()):\n    x=torch.ones(2,device=f'cuda:{i}'); value=float((x*x).sum().item())\n    devices.append(dict(index=i,name=torch.cuda.get_device_name(i),\n        total_bytes=torch.cuda.get_device_properties(i).total_memory,\n        free_bytes=torch.cuda.mem_get_info(i)[0],capability=list(torch.cuda.get_device_capability(i)),\n        tensor_sum=value))\nprint(json.dumps(devices))\n"
    result = subprocess.run([sys.executable, '-u', '-c', source], capture_output=True, text=True, encoding='utf8', timeout=90, check=True)
    return json.loads(result.stdout.splitlines()[-1])

def run_gpu_task_waves(tasks, function, gpu_ids):
    """Run one task per device per wave; multi-GPU tasks reserve the entire set.

    tasks is a sequence of (name, reserved_gpu_ids). The caller owns publication.
    The event lets process supervisors stop their children on notebook interrupt.
    """
    allowed = set(gpu_ids)
    wave, used, results = ([], set(), {})
    cancel = Event()

    def drain():
        if not wave:
            return
        with ThreadPoolExecutor(max_workers=len(wave)) as pool:
            futures = {pool.submit(function, name, cancel): name for name, _ in wave}
            try:
                for future in as_completed(futures):
                    results[futures[future]] = future.result()
            except BaseException:
                cancel.set()
                for future in futures:
                    future.cancel()
                raise
        wave.clear()
        used.clear()
    for name, ids in tasks:
        reserved = set(ids)
        if not reserved <= allowed or len(reserved) != len(ids):
            raise ValueError(f'{name}: invalid GPU reservation {ids}')
        if used & reserved or (wave and (not reserved)):
            drain()
        if not reserved:
            results[name] = function(name, cancel)
            continue
        wave.append((name, ids))
        used.update(reserved)
    drain()
    return results

VARIANT_SPECS = {'docentr_binarization': (1, 'neural_restoration', 'neural_restoration', 'DocEnTR released base/8 H-DIBCO2018 ViT reconstruction', 'learned_prior_sensitivity'), 'degan_binarization': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 1024; Keras Torch backend; threshold .95', 'learned_prior_sensitivity'), 'degan_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 1024 deblur; Keras Torch backend', 'learned_prior_sensitivity'), 'degan_watermark_removal': (1, 'neural_restoration', 'neural_restoration', 'DE-GAN released U-Net 512 watermark removal; explicit region review', 'learned_prior_sensitivity'), 'original': (1, 'source_observation', 'source', 'canonical RGB pixels', 'reference'), 'clahe': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CLAHE Lab L', 'conservative_probe'), 'bicubic_x2': (2, 'deterministic_transformation', 'geometric_scale_interpolation', 'Pillow bicubic x2', 'conservative_probe'), 'bicubic_x4': (4, 'deterministic_transformation', 'geometric_scale_interpolation', 'Pillow bicubic x4', 'conservative_probe'), 'denoised': (1, 'deterministic_transformation', 'denoising_sharpening', 'NLM Lab L', 'glyph_sensitivity'), 'grayscale_luma': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB luminance projection', 'glyph_sensitivity'), 'grayscale_red': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB R projection', 'selective_information'), 'grayscale_green': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB G projection', 'selective_information'), 'grayscale_blue': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB B projection', 'selective_information'), 'lab_l': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab L projection', 'glyph_sensitivity'), 'lab_a': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab a chroma diagnostic', 'selective_information'), 'lab_b': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'CIELab b chroma diagnostic', 'selective_information'), 'red_ink': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB red dominance mask', 'selective_information'), 'background_normalized': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'Gaussian background division Lab L', 'glyph_sensitivity'), 'sauvola': (1, 'deterministic_transformation', 'binarization', 'Sauvola reflected local moments', 'glyph_sensitivity'), 'niblack': (1, 'deterministic_transformation', 'binarization', 'Niblack mean-minus-k-std; equality is background', 'glyph_sensitivity'), 'bilateral': (1, 'deterministic_transformation', 'denoising_sharpening', 'bilateral Lab L', 'glyph_sensitivity'), 'unsharp': (1, 'deterministic_transformation', 'denoising_sharpening', 'thresholded unsharp Lab L', 'glyph_sensitivity'), 'retinex_msr': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'three-scale log Retinex intensity; percentile mapping', 'glyph_sensitivity'), 'retinex_msrcp': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'three-scale MSRCP; common RGB gain with saturation bound', 'glyph_sensitivity'), 'wolf': (1, 'deterministic_transformation', 'binarization', 'Wolf-Jolion global-normalized local threshold', 'glyph_sensitivity'), 'nick': (1, 'deterministic_transformation', 'binarization', 'NICK mean+k*sqrt(E[I^2]+variance)', 'glyph_sensitivity'), 'color_pca': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'RGB principal-component projection; luminance sign', 'selective_information'), 'color_gmm': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'diagonal Lab GMM EM; brightest component is paper', 'selective_information'), 'mean_shift': (1, 'deterministic_transformation', 'denoising_sharpening', 'OpenCV spatial-color mean shift, no pyramid by default', 'glyph_sensitivity'), 'anisotropic_diffusion': (1, 'deterministic_transformation', 'denoising_sharpening', 'Perona-Malik exp conductivity; 4 neighbors; no-flux Lab L', 'glyph_sensitivity'), 'background_morphological': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'grayscale closing surface; divide original Lab L', 'glyph_sensitivity'), 'background_guided': (1, 'deterministic_transformation', 'deterministic_color_contrast', 'closing surface self-guided filter; divide original Lab L', 'glyph_sensitivity'), 'background_otsu': (1, 'deterministic_transformation', 'binarization', 'hypothesis: closing surface -> division -> Otsu', 'glyph_sensitivity'), 'realesrgan_x4': (4, 'neural_restoration', 'neural_restoration', 'Real-ESRGAN RRDBNet x4plus', 'learned_prior_sensitivity'), 'tadisr_x4': (4, 'neural_restoration', 'neural_restoration', 'TADiSR CogView4 RealCE joint segmentation decoder', 'learned_prior_sensitivity'), 'difftsr': (None, 'neural_restoration', 'neural_restoration', 'DiffTSR IDM/TDM/MoM and TransOCR text prior', 'learned_prior_sensitivity'), 'unihir_restoration': (1, 'generative_reconstruction', 'generative_reconstruction', 'UniHIR BAGEL MLLM reconstruction', 'generative_sensitivity'), 'docres_appearance': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt appearance', 'learned_prior_sensitivity'), 'docres_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt deblurring', 'learned_prior_sensitivity'), 'docres_deshadowing': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt deshadowing', 'learned_prior_sensitivity'), 'docres_binarization': (1, 'neural_restoration', 'neural_restoration', 'DocRes Restormer DTSPrompt binarization', 'learned_prior_sensitivity'), 'docdiff_deblurring': (1, 'neural_restoration', 'neural_restoration', 'DocDiff coarse predictor and linear-100 residual diffusion', 'learned_prior_sensitivity'), 'restormer_motion_deblurring': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN motion checkpoint', 'learned_prior_sensitivity'), 'restormer_defocus_deblurring': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN single-image defocus checkpoint', 'learned_prior_sensitivity'), 'restormer_real_denoising': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN SIDD checkpoint', 'learned_prior_sensitivity'), 'restormer_gaussian_denoising': (1, 'neural_restoration', 'neural_restoration', 'Restormer MDTA/GDFN blind Gaussian checkpoint', 'learned_prior_sensitivity')}
SURVEY_SOURCE_GROUPS = {'39d1b637eed99a6e1981c326e876537ccf96baa897e16f1dca4f6aec01e32800': 'G1', '3e9a2be8a104ad044cd054dee7c0a98422aa57d2c9a4ceaea985b8d9a56a0d0f': 'G1', 'ba8129a51d43a81a2ab566153596ebe482c72a570a0c9f319a69d79c95dde7b5': 'G1', '68b6accd0c037854e41e65ed45fe1ebcd27889999a75ae3df2bfc597275107e7': 'G1', '66fd3918224508aa4dffa5cecc5abe741094c563569a61ea9e976f893811b38e': 'G2', '4b26a2be1db3c95fbd4d4387fd72c417525a813f5ed2c6a6787e02a08163121a': 'G3'}
UNAVAILABLE_RESTORATION_METHODS = {'lwtformer': {'state': 'not_integrated', 'reason': 'Official repository has architecture/test code but no verified released pretrained checkpoint', 'upstream': 'https://github.com/INWLY/LWTformer'}, 'gladunet': {'state': 'not_integrated', 'reason': 'No accessible official inference implementation and pretrained checkpoint verified', 'upstream': 'https://www.nature.com/articles/s40494-025-01965-8'}, 'gatos_energy': {'state': 'not_integrated', 'reason': 'No pinned reliable complete upstream implementation; background_otsu is a separate implemented hypothesis'}, 'single_image_bleed_reduction': {'state': 'not_integrated', 'reason': 'No verified dedicated implementation and no established bleed-through labels in the surveyed corpus'}}

def variant_spec(name):
    if name == 'tadisr_text_mask':
        return dict(method_family='neural_restoration', algorithm='TADiSR predicted segmentation mask', perturbation_class='diagnostic_only', provenance='neural_restoration', scale=4)
    scale, provenance, group, algorithm, interpretation = VARIANT_SPECS[name]
    return dict(method_family=group, algorithm=algorithm, perturbation_class=interpretation, provenance=provenance, scale=scale)

def preparation_applicability(name, book, page, region, config):
    """Explicit analyst conditions only; never invent a degradation detector."""
    rule = config.get('applicability', {}).get(name, {})
    stem = str(book).removesuffix('.pdf')
    if rule.get('books') is not None and (not {str(book), stem}.intersection(rule['books'])):
        return ('not_applicable', 'book outside explicit method applicability')
    if rule.get('crop_types') is not None and region.get('crop_type') not in rule['crop_types']:
        return ('not_applicable', 'crop type outside explicit method applicability')
    review = preparation_region_review(book, page, region['id'], config)
    if rule.get('require_review') and name not in review.get('approved_variants', []):
        return ('not_applicable', 'requires explicit region approval; degradation unverified')
    return ('ready', None)

def preparation_region_review(book, page, region_id, config):
    reviews = config.get('reviewed_regions', {})
    return reviews.get(f'{book}:{page + 1}:{region_id}', reviews.get(f"{str(book).removesuffix('.pdf')}:{page + 1}:{region_id}", {}))

def validate_applicability(config):
    for name, rule in config.get('applicability', {}).items():
        if name not in VARIANT_SPECS or not isinstance(rule, dict) or set(rule) - {'books', 'crop_types', 'require_review'}:
            raise ValueError('Unknown method/applicability option')
        for key in ('books', 'crop_types'):
            if key in rule and (not isinstance(rule[key], list) or any((not isinstance(v, str) for v in rule[key]))):
                raise ValueError('Applicability selections must be lists of strings')
        if set(rule.get('crop_types', [])) - {'page', 'group', 'column'}:
            raise ValueError('Invalid applicability crop_types')
        if 'require_review' in rule and type(rule['require_review']) is not bool:
            raise ValueError('require_review must be boolean')

import math
EXTRA_CLASSICAL = ('grayscale_luma', 'grayscale_red', 'grayscale_green', 'grayscale_blue', 'lab_l', 'lab_a', 'lab_b', 'red_ink', 'background_normalized', 'sauvola', 'niblack', 'bilateral', 'unsharp', 'retinex_msr', 'retinex_msrcp', 'wolf', 'nick', 'color_pca', 'color_gmm', 'mean_shift', 'anisotropic_diffusion', 'background_morphological', 'background_guided', 'background_otsu')

def classical_parameters(name, options):
    defaults = {'background_normalized': dict(sigma=30.0, strength=0.8, floor=32.0), 'red_ink': dict(min_red=80, dominance=1.25, difference=25), 'sauvola': dict(window_size=31, k=0.2, r=128.0), 'niblack': dict(window_size=31, k=0.2), 'bilateral': dict(diameter=5, sigma_color=20.0, sigma_space=3.0), 'unsharp': dict(sigma=1.0, amount=0.35, threshold=3.0), 'retinex_msr': dict(sigma_small=15.0, sigma_medium=40.0, sigma_large=100.0, clip_percent=1.0, strength=0.6), 'retinex_msrcp': dict(sigma_small=15.0, sigma_medium=40.0, sigma_large=100.0, clip_percent=1.0, strength=0.6), 'wolf': dict(window_size=31, k=0.5), 'nick': dict(window_size=31, k=-0.1), 'color_pca': dict(sample_pixels=8192, clip_percent=1.0), 'color_gmm': dict(sample_pixels=8192, components=3, iterations=15, variance_floor=9.0), 'mean_shift': dict(spatial_radius=3.0, color_radius=12.0), 'anisotropic_diffusion': dict(iterations=5, kappa=12.0, timestep=0.15), 'background_morphological': dict(kernel_size=31, strength=0.8, floor=32.0), 'background_guided': dict(kernel_size=31, strength=0.8, floor=32.0, radius=15, epsilon=0.01), 'background_otsu': dict(kernel_size=31, strength=1.0, floor=32.0)}
    params = {**defaults.get(name, {}), **options}
    if set(options) - set(defaults.get(name, {})):
        raise ValueError(f'{name}: unknown options {set(options) - set(defaults.get(name, {}))}')
    if any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) for v in params.values())):
        raise ValueError(f'{name}: parameters must be finite numbers')
    if name in ('sauvola', 'niblack', 'wolf', 'nick'):
        w = params['window_size']
        if not isinstance(w, int) or not 3 <= w <= 501 or w % 2 != 1 or (not -1 <= params['k'] <= 1):
            raise ValueError('Binarization requires odd window_size 3..501 and k in [-1,1]')
        if name == 'sauvola' and (not 1 <= params['r'] <= 255):
            raise ValueError('Sauvola r must be 1..255 on the uint8 luminance scale')
        if name == 'wolf' and (not 0 <= params['k'] <= 1):
            raise ValueError('Wolf k must be in [0,1]')
        if name == 'nick' and (not -0.5 <= params['k'] < 0):
            raise ValueError('NICK k must be negative in [-.5,0)')
    if name == 'background_normalized' and (not (1 <= params['sigma'] <= 500 and 0 <= params['strength'] <= 1 and (1 <= params['floor'] <= 255))):
        raise ValueError('Invalid background normalization parameters')
    if name == 'red_ink' and (not (0 <= params['min_red'] <= 255 and 1 <= params['dominance'] <= 4 and (0 <= params['difference'] <= 255))):
        raise ValueError('Invalid red ink thresholds')
    if name == 'bilateral' and (not (isinstance(params['diameter'], int) and params['diameter'] in (3, 5, 7, 9) and (0 < params['sigma_color'] <= 50) and (0 < params['sigma_space'] <= 10))):
        raise ValueError('Bilateral parameters exceed conservative limits')
    if name == 'unsharp' and (not (0 < params['sigma'] <= 3 and 0 <= params['amount'] <= 0.75 and (0 <= params['threshold'] <= 30))):
        raise ValueError('Unsharp parameters exceed conservative limits')
    if name.startswith('retinex_') and (not (1 <= params['sigma_small'] < params['sigma_medium'] < params['sigma_large'] <= 500 and 0 <= params['clip_percent'] <= 5 and (0 < params['strength'] <= 1))):
        raise ValueError('Invalid Retinex scales/clipping/strength')
    if name in ('color_pca', 'color_gmm'):
        if type(params['sample_pixels']) is not int or not 64 <= params['sample_pixels'] <= 65536:
            raise ValueError('Color fit sample_pixels must be integer 64..65536')
        if name == 'color_pca' and (not 0 <= params['clip_percent'] <= 5):
            raise ValueError('Invalid PCA clipping')
        if name == 'color_gmm' and (not (type(params['components']) is int and 2 <= params['components'] <= 5 and (type(params['iterations']) is int) and (1 <= params['iterations'] <= 50) and (1 <= params['variance_floor'] <= 100))):
            raise ValueError('Invalid GMM fit budget/variance floor')
    if name == 'mean_shift' and (not (1 <= params['spatial_radius'] <= 8 and 1 <= params['color_radius'] <= 30)):
        raise ValueError('Mean Shift radii exceed conservative limits')
    if name == 'anisotropic_diffusion' and (not (type(params['iterations']) is int and 1 <= params['iterations'] <= 30 and (1 <= params['kappa'] <= 40) and (0 < params['timestep'] <= 0.25))):
        raise ValueError('Diffusion requires bounded iterations and stable explicit timestep <= .25')
    if name in ('background_morphological', 'background_guided', 'background_otsu'):
        if not (type(params['kernel_size']) is int and 3 <= params['kernel_size'] <= 301 and params['kernel_size'] % 2 and (0 < params['strength'] <= 1) and (1 <= params['floor'] <= 255)):
            raise ValueError('Invalid morphological background scale/strength/floor')
        if name == 'background_guided' and (not (type(params['radius']) is int and 1 <= params['radius'] <= 150 and (1e-05 <= params['epsilon'] <= 0.5))):
            raise ValueError('Invalid guided-background radius/epsilon')
    return params

def classical_resource_metadata(name, pixels):
    """Conservative array-workspace estimate, not measured peak RSS or wall time."""
    limits = {'mean_shift': 4000000, 'color_gmm': 8000000, 'anisotropic_diffusion': 8000000, 'retinex_msr': 12000000, 'retinex_msrcp': 12000000, 'background_guided': 12000000}
    return dict(device='cpu', input_pixels=pixels, workspace_estimate_bytes=pixels * 160 + 16 * 1024 ** 2, workspace_estimate_status='conservative_arrays_only; library_allocator_RSS_unmeasured', method_max_pixels=limits.get(name, 25000000))

def classical_local_moments(gray, window_size):
    import cv2
    import numpy as np
    gray = gray.astype(np.float64)
    window = (window_size,) * 2
    mean = cv2.boxFilter(gray, -1, window, borderType=cv2.BORDER_REFLECT)
    second = cv2.boxFilter(gray * gray, -1, window, borderType=cv2.BORDER_REFLECT)
    variance = np.maximum(0, second - mean * mean)
    return (mean, second, variance)

def classical_percentile_map(values, percent, *, flat_value):
    import numpy as np
    lo, hi = np.percentile(values, [percent, 100 - percent])
    if hi - lo <= 1e-08:
        return (np.full_like(values, flat_value), dict(flat_input=True, low=float(lo), high=float(hi)))
    return (np.clip((values - lo) * 255 / (hi - lo), 0, 255), dict(flat_input=False, low=float(lo), high=float(hi)))

def classical_gmm(lab, params):
    import numpy as np
    pixels = lab.reshape(-1, 3).astype(np.float64)
    sample = pixels[np.linspace(0, len(pixels) - 1, min(len(pixels), params['sample_pixels']), dtype=int)]
    centers = [sample[int(sample[:, 0].argmax())]]
    distance = np.full(len(sample), np.inf)
    while len(centers) < params['components']:
        distance = np.minimum(distance, ((sample - centers[-1]) ** 2).sum(axis=1))
        index = int(distance.argmax())
        if distance[index] < 1e-08:
            break
        centers.append(sample[index])
    means = np.asarray(centers).copy()
    count = len(means)
    variances = np.tile(np.maximum(sample.var(0), params['variance_floor']), (count, 1))
    weights = np.full(count, 1 / count)

    def posterior(x):
        logp = -0.5 * np.sum((x[:, None, :] - means) ** 2 / variances + np.log(variances), axis=2) + np.log(weights)
        logp -= logp.max(axis=1, keepdims=True)
        prob = np.exp(logp)
        return prob / prob.sum(axis=1, keepdims=True)
    for _ in range(params['iterations']):
        prob = posterior(sample)
        mass = np.maximum(prob.sum(0), 1e-08)
        means = prob.T @ sample / mass[:, None]
        variances = np.maximum(prob.T @ (sample * sample) / mass[:, None] - means * means, params['variance_floor'])
        weights = np.maximum(mass / len(sample), 1e-08)
        weights /= weights.sum()
    paper = int(means[:, 0].argmax())
    result = np.empty(len(pixels), dtype='uint8')
    for start in range(0, len(pixels), 65536):
        result[start:start + 65536] = (255 * posterior(pixels[start:start + 65536])[:, paper]).round().astype('uint8')
    return (result.reshape(lab.shape[:2]), dict(component_means_lab=means.tolist(), paper_component=paper, fit_pixels=len(sample), initialization='deterministic_farthest_color_points', requested_components=params['components'], effective_components=count, warning='brightest_component_is_an_assumption; posterior_is_not_ink_ground_truth'))

def new_classical_pixels(rgb, lab, name, params):
    import cv2
    import numpy as np
    luma = lab[:, :, 0].astype(np.float32)
    details, warnings = ({}, [])
    if name in ('retinex_msr', 'retinex_msrcp'):
        intensity = rgb.astype(np.float32).mean(axis=2)
        log_intensity = np.log1p(intensity)
        reflectance = np.zeros_like(intensity)
        for key in ('sigma_small', 'sigma_medium', 'sigma_large'):
            reflectance += (log_intensity - np.log1p(cv2.GaussianBlur(intensity, (0, 0), params[key], borderType=cv2.BORDER_REFLECT))) / 3
        target, details = classical_percentile_map(reflectance, params['clip_percent'], flat_value=float(intensity.mean()))
        target = (1 - params['strength']) * intensity + params['strength'] * target
        if name == 'retinex_msrcp':
            gain = np.minimum(255 / np.maximum(rgb.max(axis=2), 1), target / np.maximum(intensity, 1e-06))
            result = np.clip(rgb.astype(np.float32) * gain[:, :, None], 0, 255).round().astype('uint8')
            warnings.append('common_RGB_gain_preserves_ratios_before_quantization; not_colorimetric_fidelity')
        else:
            result = target.round().astype('uint8')
            warnings.append('grayscale_loses_red_ink_color')
        warnings.append('log_contrast_mapping_can_amplify_noise_and_create_halos')
    elif name in ('color_pca', 'color_gmm'):
        if name == 'color_gmm':
            result, details = classical_gmm(lab, params)
        else:
            pixels = rgb.reshape(-1, 3).astype(np.float32)
            sample = pixels[np.linspace(0, len(pixels) - 1, min(len(pixels), params['sample_pixels']), dtype=int)]
            center = sample.mean(0)
            eigenvalues, axes = np.linalg.eigh((sample - center).T @ (sample - center) / max(1, len(sample)))
            axis = axes[:, -1]
            if float(axis @ np.array([0.299, 0.587, 0.114])) < 0:
                axis = -axis
            projected = ((pixels - center) @ axis).reshape(rgb.shape[:2])
            result, details = classical_percentile_map(projected, params['clip_percent'], flat_value=255.0)
            result = result.round().astype('uint8')
            details.update(axis_rgb=axis.tolist(), eigenvalues=eigenvalues.tolist(), fit_pixels=len(sample))
        warnings.append('unsupervised_color_projection; may_hide_faint_or_red_ink; color_lost')
    elif name == 'mean_shift':
        result = cv2.pyrMeanShiftFiltering(rgb, params['spatial_radius'], params['color_radius'], maxLevel=0)
        details.update(pyramid_levels=0, geometry='identity')
        warnings.append('mode_seeking_can_merge_nearby_strokes_and_colors')
    elif name == 'anisotropic_diffusion':
        current = luma.copy()
        for _ in range(params['iterations']):
            padded = np.pad(current, 1, mode='edge')
            delta = np.zeros_like(current)
            for neighbor in (padded[:-2, 1:-1], padded[2:, 1:-1], padded[1:-1, :-2], padded[1:-1, 2:]):
                gradient = neighbor - current
                delta += np.exp(-(gradient / params['kappa']) ** 2) * gradient
            current += params['timestep'] * delta
        lab[:, :, 0] = np.clip(current, 0, 255).round().astype('uint8')
        result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        details.update(boundary='no_flux', stencil='four_neighbors')
        warnings.append('weak_gradient_strokes_can_diffuse; no_missing_stroke_reconstruction')
    else:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (params['kernel_size'],) * 2)
        surface = cv2.morphologyEx(lab[:, :, 0], cv2.MORPH_CLOSE, kernel, borderType=cv2.BORDER_REFLECT).astype(np.float32)
        if name == 'background_guided':
            guide = surface / 255
            window = (2 * params['radius'] + 1,) * 2

            def average(a):
                return cv2.boxFilter(a, -1, window, borderType=cv2.BORDER_REFLECT)
            mean = average(guide)
            variance = np.maximum(0, average(guide * guide) - mean * mean)
            a = variance / (variance + params['epsilon'])
            b = mean - a * mean
            surface = np.clip((average(a) * guide + average(b)) * 255, 0, 255)
            details['guidance'] = 'estimated_background_only; never_original_strokes'
        corrected = luma * float(np.median(surface)) / np.maximum(surface, params['floor'])
        normalized = np.clip((1 - params['strength']) * luma + params['strength'] * corrected, 0, 255).round().astype('uint8')
        if name == 'background_otsu':
            threshold, result = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            details.update(observed_otsu_threshold=float(threshold), hypothesis_components=['closing_background', 'division_L', 'Otsu'])
            warnings.append('binarization_loses_red_color_and_can_erase_weak_strokes')
        else:
            lab[:, :, 0] = normalized
            result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        warnings.append('morphology_only_estimates_background; large_letters_rules_may_contaminate_surface')
    return (result, details, warnings)

def extra_classical_variant(image, name, options):
    import cv2
    import numpy as np
    from PIL import Image
    if name not in EXTRA_CLASSICAL:
        raise ValueError(f'Unknown classical variant: {name}')
    params = classical_parameters(name, options)
    resources = classical_resource_metadata(name, image.width * image.height)
    if image.width * image.height > resources['method_max_pixels']:
        raise MemoryError(f"{name} exceeds max_method_pixels={resources['method_max_pixels']}; choose smaller regions explicitly")
    rgb = np.asarray(image.convert('RGB'))
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    warnings = ['experimental; inspect_against_original; ocr_accuracy_not_measured']
    details = {}
    if name == 'grayscale_luma':
        result = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    elif name.startswith('grayscale_'):
        result = rgb[:, :, {'red': 0, 'green': 1, 'blue': 2}[name.split('_')[1]]]
        warnings.append('channel_selection_may_hide_colored_ink; original_RGB_retained')
    elif name.startswith('lab_'):
        result = lab[:, :, {'l': 0, 'a': 1, 'b': 2}[name[-1]]]
        warnings.append('Lab_ab_are_chroma_diagnostics_not_luminance' if name != 'lab_l' else 'grayscale_loses_color_information')
    elif name == 'red_ink':
        a = rgb.astype(np.float32)
        mask = (a[:, :, 0] >= params['min_red']) & (a[:, :, 0] >= params['dominance'] * np.maximum(a[:, :, 1], a[:, :, 2])) & (a[:, :, 0] - np.maximum(a[:, :, 1], a[:, :, 2]) >= params['difference'])
        result = np.full_like(rgb, 255)
        result[mask] = rgb[mask]
        details['red_candidate_fraction'] = float(mask.mean())
        warnings.append('red_only_analysis; black_ink_omitted_in_this_variant_only; mask_is_not_ground_truth')
    elif name == 'background_normalized':
        luminance = lab[:, :, 0].astype(np.float32)
        background = cv2.GaussianBlur(luminance, (0, 0), params['sigma'], borderType=cv2.BORDER_REFLECT)
        corrected = luminance * float(np.median(background)) / np.maximum(background, params['floor'])
        lab[:, :, 0] = np.clip((1 - params['strength']) * luminance + params['strength'] * corrected, 0, 255).round().astype('uint8')
        result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        warnings.append('chroma_retained; may_create_halo_or_amplify_noise')
    elif name in ('sauvola', 'niblack', 'wolf', 'nick'):
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float64)
        mean, second, variance = classical_local_moments(gray, params['window_size'])
        std = np.sqrt(variance)
        if name == 'sauvola':
            threshold = mean * (1 + params['k'] * (std / params['r'] - 1))
        elif name == 'niblack':
            threshold = mean - params['k'] * std
        elif name == 'wolf':
            maximum = float(std.max())
            threshold = mean - params['k'] * (1 - std / max(maximum, 1e-08)) * (mean - float(gray.min()))
            details['global_max_std'] = maximum
        else:
            threshold = mean + params['k'] * np.sqrt(second + variance)
        result = np.where(gray >= threshold - 1e-07, 255, 0).astype('uint8')
        warnings.append('may_erase_weak_strokes_or_merge_strokes; color_lost')
        details.update(boundary='reflect', intensity_scale=[0, 255], geometry='identity', threshold_tie_epsilon=1e-07)
    elif name == 'bilateral':
        lab[:, :, 0] = cv2.bilateralFilter(lab[:, :, 0], params['diameter'], params['sigma_color'], params['sigma_space'])
        result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        warnings.append('chroma_retained; may_remove_weak_strokes')
    elif name == 'unsharp':
        l = lab[:, :, 0].astype(np.float32)
        delta = l - cv2.GaussianBlur(l, (0, 0), params['sigma'])
        delta[np.abs(delta) < params['threshold']] = 0
        lab[:, :, 0] = np.clip(l + params['amount'] * delta, 0, 255).round().astype('uint8')
        result = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        warnings.append('chroma_retained; may_create_halo_or_false_edges')
    else:
        result, details, extra_warnings = new_classical_pixels(rgb, lab, name, params)
        warnings.extend(extra_warnings)
    output = Image.fromarray(result).convert('RGB')
    if output.size != image.size:
        raise ValueError('Classical transformation unexpectedly changed geometry')
    return (output, dict(method=name, parameters=params, opencv=cv2.__version__, **variant_spec(name), resources=resources, orientation='independent; no_rotation', warnings=warnings, **details))

from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
import hashlib
import io
import json
import math
from pathlib import Path, PurePosixPath
import sys
import time
IMAGE_DATASET_FORMAT = 'vtm-image-dataset'
IMAGE_DATASET_VERSION = 1
IMAGE_DATASET_CODE_SIGNATURE = 'bea606181a992d89d6d6063f0b81168a5a09f689e659099cbf677a2188a2ec5d'

@dataclass
class ImageDatasetConfig:
    dpi: int = 200
    acquisition: str = 'auto'
    max_source_pixels: int = 100000000
    analysis_max_side: int = 1200
    reading_direction: str = 'vertical_rtl'
    columns_per_group: int = 3
    max_columns: int = 64
    column_cuts: list[float] | None = None
    page_column_cuts: dict = field(default_factory=dict)
    ink_threshold: int = 40
    gap_ink_fraction: float = 0.006
    min_gap_fraction: float = 0.006
    variants: list[str] = field(default_factory=lambda: ['original'])
    variant_options: dict = field(default_factory=dict)

    def __post_init__(self):
        if self.dpi <= 0 or self.max_source_pixels < 1 or (not 32 <= self.analysis_max_side <= 1600) or (self.columns_per_group < 1) or (self.max_columns < 1):
            raise ValueError('Invalid preparation dimensions/group limits')
        if self.acquisition not in {'auto', 'render'}:
            raise ValueError('acquisition must be auto or render')
        if self.reading_direction not in {'vertical_rtl', 'horizontal_ltr'}:
            raise ValueError('Invalid reading_direction')
        if not 0 < self.ink_threshold < 255 or not 0 <= self.gap_ink_fraction < 1 or (not 0 < self.min_gap_fraction < 0.5):
            raise ValueError('Invalid projection thresholds')
        if not isinstance(self.variants, list) or 'original' not in self.variants or len(set(self.variants)) != len(self.variants) or set(self.variants) - {'original', 'bicubic_x2', 'bicubic_x4', 'denoised', 'clahe', *EXTRA_CLASSICAL}:
            raise ValueError('variants require original and supported classical names')
        if set(self.variant_options) - {'clahe', 'denoised', *EXTRA_CLASSICAL}:
            raise ValueError('Unknown variant_options')
        for name, options in self.variant_options.items():
            if name in EXTRA_CLASSICAL:
                classical_parameters(name, options)
                continue
            if name == 'denoised':
                continue
            if set(options) - {'clip_limit', 'tile_grid'}:
                raise ValueError(f'Unknown {name} parameters')
            if not 0 < options.get('clip_limit', 1.5) <= 4 or not 2 <= options.get('tile_grid', 8) <= 32:
                raise ValueError('CLAHE clip_limit must be (0,4], tile_grid [2,32]')
        dataset_validate_cuts(self.column_cuts)
        for page, cuts in self.page_column_cuts.items():
            if not str(page).isdigit() or int(page) < 1:
                raise ValueError('page_column_cuts uses 1-based page keys')
            dataset_validate_cuts(cuts)

def dataset_json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False) + '\n').encode('utf-8')

def dataset_hash(value):
    return hashlib.sha256(dataset_json_bytes(value)).hexdigest()

def dataset_file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def dataset_code_signature():
    if IMAGE_DATASET_CODE_SIGNATURE:
        return IMAGE_DATASET_CODE_SIGNATURE
    root = Path(__file__).parent
    return dataset_hash({name: dataset_file_hash(root / name) for name in ('image_dataset.py', 'image_preparation.py', 'pdf_images.py', 'restoration_classical.py', 'restoration_registry.py')})

def dataset_environment():
    import PIL
    import pypdfium2 as pdfium
    result = {'pillow': PIL.__version__, 'pypdfium2': str(pdfium.PYPDFIUM_INFO), 'pdfium': str(pdfium.PDFIUM_INFO), 'python': sys.version.split()[0]}
    return result

def dataset_path(root, relative):
    relative = str(relative)
    if '\\' in relative or ':' in relative:
        raise ValueError(f'Dataset requires portable relative paths: {relative}')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or '..' in parts.parts or (not parts.parts):
        raise ValueError(f'Invalid dataset path: {relative}')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError(f'Dataset path escapes root: {relative}')
    return path

def dataset_write(root, relative, data):
    path = dataset_path(root, relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_bytes(data)
    temporary.replace(path)
    return {'path': str(relative), 'sha256': hashlib.sha256(data).hexdigest()}

def dataset_read(root, reference):
    data = dataset_path(root, reference['path']).read_bytes()
    if hashlib.sha256(data).hexdigest() != reference['sha256']:
        raise ValueError(f"Dataset checksum mismatch: {reference['path']}")
    return data

def dataset_store_image(root, image):
    image = image.convert('RGB')
    output = io.BytesIO()
    image.save(output, format='PNG', compress_level=6)
    data = output.getvalue()
    sha = hashlib.sha256(data).hexdigest()
    relative = f'objects/{sha[:2]}/{sha}.png'
    path = dataset_path(root, relative)
    if not path.exists() or dataset_file_hash(path) != sha:
        dataset_write(root, relative, data)
    return {'path': relative, 'sha256': sha, 'size': list(image.size), 'pixel_sha256': hashlib.sha256(image.tobytes()).hexdigest()}

def dataset_load_image(root, reference):
    from PIL import Image
    with Image.open(io.BytesIO(dataset_read(root, reference))) as loaded:
        image = loaded.convert('RGB')
    if list(image.size) != reference['size'] or hashlib.sha256(image.tobytes()).hexdigest() != reference['pixel_sha256']:
        raise ValueError(f"Dataset image dimensions/pixels changed: {reference['path']}")
    return image

@contextmanager
@contextmanager
def dataset_lock(root):
    """OS lock releases after interruption; no stale PID/lockfile cleanup needed."""
    import os
    Path(root).mkdir(parents=True, exist_ok=True)
    with (Path(root) / 'prepare.lock').open('a+b') as stream:
        if stream.tell() == 0:
            stream.write(b'0')
            stream.flush()
        stream.seek(0)
        import fcntl
        fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)
        try:
            yield
        finally:
            stream.seek(0)
            fcntl.flock(stream, fcntl.LOCK_UN)

def dataset_validate_cuts(cuts):
    if cuts is not None and (not isinstance(cuts, list) or any((isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (not 0 < v < 1) for v in cuts)) or cuts != sorted(set(cuts))):
        raise ValueError('column_cuts must be unique increasing fractions in (0,1)')

def dataset_partition_qc(regions, size, direction):
    """Exact geometric partition check, independent of OCR/text completeness."""
    w, h = size
    expected = 0 if direction == 'horizontal_ltr' else w
    for index, region in enumerate(regions):
        x1, y1, x2, y2 = region['source_box']
        if not all((isinstance(v, int) for v in (x1, y1, x2, y2))) or not 0 <= x1 < x2 <= w or (y1, y2) != (0, h) or (region['reading_order'] != index):
            raise ValueError('Invalid canonical crop bounds/order')
        start, end = (x1, x2) if direction == 'horizontal_ltr' else (x2, x1)
        if start != expected:
            raise ValueError('Canonical partition has a gap, overlap or inconsistent reading order')
        expected = end
    if not regions or expected != (w if direction == 'horizontal_ltr' else 0):
        raise ValueError('Canonical partition omits page content')

def dataset_plan_layout(image, config, *, page=0):
    """Projection baseline, no model/pixel budget; all coordinates in source pixels."""
    from PIL import Image, ImageDraw
    analysis = image.copy()
    analysis.thumbnail((config.analysis_max_side, config.analysis_max_side), Image.Resampling.LANCZOS)
    options = {key: getattr(config, key) for key in ('ink_threshold', 'gap_ink_fraction', 'min_gap_fraction')}
    mask = projection_mask(analysis, options)
    w, h = image.size
    aw, ah = analysis.size
    manual = config.page_column_cuts.get(str(page + 1), config.column_cuts)
    if manual is not None:
        cuts = [round(v * w) for v in manual]
        method = 'manual_columns'
    elif config.reading_direction == 'vertical_rtl':
        cuts = sorted(set((round(v * w / aw) for v in projection_gaps(analysis, 'x', options))))
        method = 'whitespace_projection'
    else:
        cuts, method = ([], 'whole_page_horizontal')
    if len(set([0, *cuts, w])) != len(cuts) + 2:
        raise ValueError('Column cuts collapse in source space')
    warnings = ['heuristic_layout; geometric_coverage_is_not_text_coverage']
    if len(cuts) + 1 > config.max_columns:
        if manual is not None:
            raise ValueError('Manual layout exceeds max_columns')
        warnings.append('too_many_gutters; retained_whole_page')
        cuts = []
    if not cuts:
        warnings.append('no_partition; whole_page_retained')
    thresholds = sorted(set([max(1, config.ink_threshold - 8), config.ink_threshold, min(254, config.ink_threshold + 8)]))
    found = [projection_gaps(analysis, 'x', {**options, 'ink_threshold': t}) for t in thresholds]
    ink = mask.tobytes()
    top, bottom = (int(0.03 * ah), max(1, int(0.97 * ah)))
    projection = [sum((ink[y * aw + x] > 0 for y in range(top, bottom))) / (bottom - top) for x in range(aw)]
    evidence = []
    for cut in cuts:
        x = min(aw - 1, max(0, round(cut * aw / w)))
        lo, hi = (x, x)
        while lo > 0 and projection[lo - 1] <= config.gap_ink_fraction:
            lo -= 1
        while hi < aw - 1 and projection[hi + 1] <= config.gap_ink_fraction:
            hi += 1
        tolerance = max(2, round(aw * 0.006))
        stability = sum((any((abs(c - x) <= tolerance for c in cs)) for cs in found)) / len(found)
        radius = max(2, round(aw * 0.035))
        left, right = (projection[max(0, x - radius):x], projection[x + 1:min(aw, x + radius + 1)])
        risks = []
        if projection[x] > 0.02:
            risks.append('cut_crosses_analysis_ink; rules_or_strokes_need_review')
        if stability < 2 / 3:
            risks.append('unstable_across_ink_thresholds')
        if not left or not right or min(max(left), max(right)) <= config.gap_ink_fraction * 2:
            risks.append('weak_text_support_on_one_side')
        evidence.append({'source_x': cut, 'analysis_x': x, 'gutter_width_analysis': hi - lo + 1, 'ink_fraction': projection[x], 'threshold_stability': stability, 'warnings': risks})
    full = (0, 0, w, h)
    columns = _partition_box(full, cuts, 'x', config.reading_direction)
    group = config.columns_per_group
    groups = []
    for start in range(0, len(columns), group):
        chunk = columns[start:start + group]
        groups.append((min((b[0] for b in chunk)), 0, max((b[2] for b in chunk)), h))
    regions = []
    for kind, boxes in (('page', [full]), ('group', groups), ('column', columns)):
        for index, box in enumerate(boxes):
            regions.append({'id': f'{kind}_{index:03d}', 'crop_type': kind, 'reading_order': index, 'source_box': list(box), 'overlap': False})
        dataset_partition_qc([r for r in regions if r['crop_type'] == kind], image.size, config.reading_direction)
    preview = analysis.copy()
    draw = ImageDraw.Draw(preview)
    for region in [r for r in regions if r['crop_type'] == 'column']:
        box = region['source_box']
        x1, x2 = (round(box[0] * aw / w), round(box[2] * aw / w))
        draw.rectangle((x1, 0, max(x1, x2 - 1), ah - 1), outline='red', width=1)
        draw.text((x1 + 2, 3), str(region['reading_order'] + 1), fill='red')
    for item in evidence:
        if item['warnings']:
            draw.line((item['analysis_x'], 0, item['analysis_x'], ah - 1), fill='orange', width=2)
    draw.text((3, ah - 14), 'RTL <-' if config.reading_direction == 'vertical_rtl' else 'LTR ->', fill='red')
    layout = {'method': method, 'source_size': [w, h], 'analysis_size': [aw, ah], 'analysis_to_source': [w / aw, 0, 0, h / ah, 0, 0], 'reading_direction': config.reading_direction, 'regions': regions, 'cuts': evidence, 'excluded_regions': [], 'uncertain_regions': [r['id'] for r in regions if r['crop_type'] == 'column'], 'geometric_coverage': 1.0, 'text_coverage': 'unverified', 'warnings': warnings, 'thresholds_tested': thresholds, 'review_status': 'needs_visual_review'}
    return (layout, mask.convert('RGB'), preview)

def dataset_variant(image, name, options):
    from PIL import Image
    if name in EXTRA_CLASSICAL:
        return extra_classical_variant(image, name, options)
    if name == 'original':
        return (image, {'method': 'identity', 'parameters': {}, 'warnings': []})
    if name in {'bicubic_x2', 'bicubic_x4'}:
        scale = 2 if name == 'bicubic_x2' else 4
        return (image.resize((image.width * scale, image.height * scale), Image.Resampling.BICUBIC), {'method': 'bicubic', 'parameters': {'scale': scale}, 'warnings': ['experimental_interpolation; no_recovered_source_detail']})
    if name == 'clahe':
        try:
            import cv2
            import numpy as np
        except ImportError as exc:
            raise RuntimeError('clahe requires an already available OpenCV/NumPy installation; use original offline') from exc
        params = {'clip_limit': 1.5, 'tile_grid': 8, **options}
        lab = cv2.cvtColor(np.asarray(image.convert('RGB')), cv2.COLOR_RGB2LAB)
        grid = int(params['tile_grid'])
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=float(params['clip_limit']), tileGridSize=(grid, grid)).apply(lab[:, :, 0])
        return (Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)), {'method': 'clahe_lab_l', 'parameters': params, 'opencv': cv2.__version__, 'warnings': ['experimental_contrast; inspect_weak_strokes']})
    if name == 'denoised':
        import cv2
        import numpy as np
        defaults = {'h': 2.0, 'template_window': 7, 'search_window': 21}
        params = {**defaults, **options}
        if set(options) - set(defaults) or not isinstance(params['h'], (int, float)) or (not math.isfinite(params['h'])) or (not 0 < params['h'] <= 5) or any((not isinstance(params[k], int) or params[k] % 2 != 1 for k in ('template_window', 'search_window'))) or (not 3 <= params['template_window'] <= params['search_window'] <= 63):
            raise ValueError('Conservative NLM requires 0<h<=5 and odd windows 3..63')
        lab = cv2.cvtColor(np.asarray(image.convert('RGB')), cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = cv2.fastNlMeansDenoising(lab[:, :, 0], None, float(params['h']), params['template_window'], params['search_window'])
        return (Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)), dict(method='nlm_lab_l', parameters=params, warnings=['may_remove_weak_strokes; opt_in_visual_review_required']))
    raise ValueError(f'Unsupported image variant: {name}')

def dataset_validate_page(root, record, *, check_pixels=True):
    source = dataset_load_image(root, record['source']) if check_pixels else None
    acquisition = json.loads(dataset_read(root, record['acquisition']))
    layout = json.loads(dataset_read(root, record['layout']))
    if record['source']['size'] != layout['source_size'] or acquisition['size'] != layout['source_size']:
        raise ValueError('Source, acquisition and layout size mismatch')
    dataset_read(root, layout['analysis_mask'])
    dataset_read(root, layout['preview'])
    for kind in ('page', 'group', 'column'):
        regions = [r for r in layout['regions'] if r['crop_type'] == kind]
        dataset_partition_qc(regions, layout['source_size'], layout['reading_direction'])
    if [r['id'] for r in record['regions']] != [r['id'] for r in layout['regions']]:
        raise ValueError('Image dataset omits canonical regions')
    for region, geometry in zip(record['regions'], layout['regions']):
        if any((region[k] != geometry[k] for k in ('source_box', 'crop_type', 'reading_order'))):
            raise ValueError('Region geometry differs from canonical layout')
        if 'original' not in region['images']:
            raise ValueError('Region original variant is missing')
        x1, y1, x2, y2 = region['source_box']
        for name, entry in region['images'].items():
            expected = [(x2 - x1) * (2 if name == 'bicubic_x2' else 1), (y2 - y1) * (2 if name == 'bicubic_x2' else 1)]
            if entry['size'] != expected or entry['image_to_source'] != [(x2 - x1) / expected[0], 0, 0, (y2 - y1) / expected[1], x1, y1]:
                raise ValueError('Variant geometry/transform mismatch')
            if check_pixels:
                image = dataset_load_image(root, entry)
                if name == 'original' and image.tobytes() != source.crop(region['source_box']).tobytes():
                    raise ValueError('Original crop is not pixel-identical to source')
    return layout

def dataset_open(directory):
    path = runtime_path(directory)
    descriptor = path / 'dataset.json' if path.is_dir() else path
    root = descriptor.parent.parent if descriptor.parent.name == 'snapshots' else descriptor.parent
    data = json.loads(descriptor.read_text(encoding='utf-8'))
    if data.get('format') != IMAGE_DATASET_FORMAT or data.get('version') != IMAGE_DATASET_VERSION:
        raise ValueError('Unsupported image dataset format/version')
    body = {k: v for k, v in data.items() if k != 'snapshot_id'}
    if dataset_hash(body) != data['snapshot_id']:
        raise ValueError('Dataset snapshot identity mismatch')
    return (root, data)

def dataset_page(root, descriptor, source_sha, page):
    entries = [p for p in descriptor['pages'] if p['source_sha256'] == source_sha and p['page'] == page]
    if not entries:
        return None
    records = [json.loads(dataset_read(root, e['record'])) for e in entries]
    if len({r['input_key'] for r in records}) != 1:
        raise ValueError('Ambiguous prepared layouts for the same source/page')
    record = records[0]
    if record['source_sha256'] != source_sha or record['page'] != page:
        raise ValueError('Dataset page identity mismatch')
    return record

def prepare_image_dataset(input_dir, output_dir, *, config=None, books=None, pages='1-2', limit=2, max_books=1, profile_dir=None, publish=True):
    """Prepare a bounded selection; repeat the same command to resume.

    A completed source/layout/variant is independently reusable. Only after QC
    succeeds is a new dataset.json published. Existing snapshots remain usable.
    """
    config = config or ImageDatasetConfig()
    config.__post_init__()
    if limit is not None and limit < 1 or (max_books is not None and max_books < 1):
        raise ValueError('limit/max_books must be positive or None')
    entries = select_entries(list_pdfs(input_dir), books or [])
    if max_books is not None:
        entries = entries[:max_books]
    selected, remaining = ([], limit)
    for entry in entries:
        wanted = parse_pages(pages, entry['n_pages'])
        if remaining is not None:
            wanted = wanted[:remaining]
            remaining -= len(wanted)
        if wanted:
            selected.append((entry, wanted))
    if not selected:
        raise ValueError('No pages selected')
    root = Path(output_dir).resolve()
    stats = {'pages': 0, 'source_reused': 0, 'layout_reused': 0, 'variants_reused': 0}
    started, environment, code = (time.monotonic(), dataset_environment(), dataset_code_signature())
    page_entries, manifest_rows, configurations = ([], [], {})
    with dataset_lock(root):
        marker = root / 'dataset.json'
        if marker.exists():
            dataset_open(root)
        for entry, wanted in selected:
            import pypdfium2 as pdfium
            cfg = ImageDatasetConfig(**asdict(config))
            if profile_dir:
                profile = json.loads((Path(profile_dir) / (entry['stem'] + '.json')).read_text(encoding='utf-8-sig'))
                for key in ('dpi', 'reading_direction'):
                    if key in profile:
                        setattr(cfg, key, profile[key])
            cfg.__post_init__()
            configurations[entry['name']] = asdict(cfg)
            source_sha = dataset_file_hash(entry['path'])
            doc = pdfium.PdfDocument(entry['path'])
            try:
                doc.init_forms()
                for page in wanted:
                    source_key = dataset_hash({'source_sha256': source_sha, 'page': page, 'acquisition': cfg.acquisition, 'dpi': cfg.dpi, 'max_source_pixels': cfg.max_source_pixels, 'code': code, 'environment': environment})
                    source_receipt = f'sources/{source_key}.json'
                    try:
                        cached = json.loads(dataset_path(root, source_receipt).read_bytes())
                        if cached['key'] != source_key:
                            raise ValueError('Source cache key mismatch')
                        image = dataset_load_image(root, cached['source'])
                        acquisition = json.loads(dataset_read(root, cached['acquisition']))
                        stats['source_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        pdf_page = doc[page]
                        try:
                            image, acquisition = acquire_pdf_image(pdf_page, dpi=cfg.dpi, max_source_pixels=cfg.max_source_pixels, strategy=cfg.acquisition)
                        finally:
                            pdf_page.close()
                        source_asset = dataset_store_image(root, image)
                        acquisition_ref = dataset_write(root, f'sources/{source_key}.acquisition.json', dataset_json_bytes(acquisition))
                        cached = {'key': source_key, 'source': source_asset, 'acquisition': acquisition_ref}
                        dataset_write(root, source_receipt, dataset_json_bytes(cached))
                    layout_options = {k: v for k, v in asdict(cfg).items() if k not in {'dpi', 'acquisition', 'max_source_pixels', 'variants', 'variant_options'}}
                    layout_options['page_column_cuts'] = {str(page + 1): cfg.page_column_cuts.get(str(page + 1), cfg.column_cuts)}
                    layout_key = dataset_hash({'source': cached['source']['sha256'], 'options': layout_options, 'code': code})
                    layout_relative = f'layouts/{layout_key}.json'
                    try:
                        layout_data = dataset_path(root, layout_relative).read_bytes()
                        layout = json.loads(layout_data)
                        if layout['key'] != layout_key:
                            raise ValueError('Layout key mismatch')
                        dataset_read(root, layout['analysis_mask'])
                        dataset_read(root, layout['preview'])
                        stats['layout_reused'] += 1
                    except (OSError, ValueError, KeyError):
                        layout, mask, preview = dataset_plan_layout(image, cfg, page=page)
                        layout.update(key=layout_key, analysis_mask=dataset_store_image(root, mask), preview=dataset_store_image(root, preview))
                        layout_data = dataset_json_bytes(layout)
                        dataset_write(root, layout_relative, layout_data)
                    layout_ref = {'path': layout_relative, 'sha256': hashlib.sha256(layout_data).hexdigest()}
                    regions = []
                    for geometry in layout['regions']:
                        crop = image.crop(geometry['source_box'])
                        variants = {}
                        for name in cfg.variants:
                            options = cfg.variant_options.get(name, {})
                            variant_environment = dict(environment)
                            if name in {'clahe', 'denoised', *EXTRA_CLASSICAL}:
                                import cv2
                                variant_environment['opencv'] = cv2.__version__
                            key = dataset_hash({'source': cached['source']['sha256'], 'box': geometry['source_box'], 'variant': name, 'options': options, 'code': code, 'environment': variant_environment})
                            receipt = f'variants/{key}.json'
                            try:
                                variant = json.loads(dataset_path(root, receipt).read_bytes())
                                if variant['key'] != key:
                                    raise ValueError('Variant cache key mismatch')
                                dataset_load_image(root, variant)
                                stats['variants_reused'] += 1
                            except (OSError, ValueError, KeyError):
                                prepared, operation = dataset_variant(crop, name, options)
                                x1, y1, x2, y2 = geometry['source_box']
                                variant = {**dataset_store_image(root, prepared), 'key': key, 'operation': operation, 'image_to_source': [(x2 - x1) / prepared.width, 0, 0, (y2 - y1) / prepared.height, x1, y1]}
                                dataset_write(root, receipt, dataset_json_bytes(variant))
                            variants[name] = variant
                            manifest_rows.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page + 1, 'page_index': page, 'acquisition_method': acquisition['method'], 'source_size': cached['source']['size'], 'source_image': cached['source'], 'analysis_size': layout['analysis_size'], 'analysis_to_source': layout['analysis_to_source'], 'source_to_pdf': acquisition['source_to_pdf'], 'region_id': geometry['id'], 'crop_type': geometry['crop_type'], 'source_box': geometry['source_box'], 'reading_order': geometry['reading_order'], 'reading_direction': cfg.reading_direction, 'variant': name, **variant, 'qc_warnings': acquisition['warnings'] + layout['warnings'] + [w for c in layout['cuts'] for w in c['warnings']], 'layout': layout_ref, 'acquisition': cached['acquisition'], 'code_sha256': code, 'processing_version': IMAGE_DATASET_VERSION})
                        regions.append({**geometry, 'images': variants})
                    record = {'input_key': dataset_hash({'source': source_key, 'layout': layout_key, 'variants': cfg.variants, 'options': cfg.variant_options}), 'source_sha256': source_sha, 'page': page, 'source': cached['source'], 'acquisition': cached['acquisition'], 'layout': layout_ref, 'regions': regions}
                    dataset_validate_page(root, record)
                    record_bytes = dataset_json_bytes(record)
                    record_key = hashlib.sha256(record_bytes).hexdigest()
                    record_ref = dataset_write(root, f'pages/{record_key}.json', record_bytes)
                    page_entries.append({'book': entry['name'], 'source_pdf': Path(entry['path']).relative_to(Path(input_dir)).as_posix(), 'source_sha256': source_sha, 'page': page, 'record': record_ref})
                    stats['pages'] += 1
            finally:
                doc.close()
        manifest_data = b''.join((dataset_json_bytes(row) for row in manifest_rows))
        manifest_sha = hashlib.sha256(manifest_data).hexdigest()
        manifest_ref = dataset_write(root, f'manifests/{manifest_sha}.jsonl', manifest_data)
        descriptor = {'format': IMAGE_DATASET_FORMAT, 'version': IMAGE_DATASET_VERSION, 'pages': page_entries, 'expected_pages': [[e['book'], e['source_sha256'], e['page']] for e in page_entries], 'manifest': manifest_ref, 'configurations': configurations, 'environment': environment, 'code_sha256': code, 'geometry': 'top_left_source_pixels; half_open_boxes', 'ocr_text_coverage': 'unverified'}
        descriptor['snapshot_id'] = dataset_hash(descriptor)
        data = dataset_json_bytes(descriptor)
        snapshot = f"snapshots/{descriptor['snapshot_id']}.json"
        dataset_write(root, snapshot, data)
        if publish:
            dataset_write(root, 'dataset.json', data)
    stats.update(snapshot_id=descriptor['snapshot_id'], dataset=str(root / snapshot), seconds=round(time.monotonic() - started, 3), manifest_rows=len(manifest_rows))
    return stats

def validate_image_dataset(directory):
    root, descriptor = dataset_open(directory)
    snapshot = root / f"snapshots/{descriptor['snapshot_id']}.json"
    if not snapshot.is_file() or dataset_open(snapshot)[1] != descriptor:
        raise ValueError('Immutable dataset snapshot missing or mismatched')
    dataset_read(root, descriptor['manifest'])
    actual = [[e['book'], e['source_sha256'], e['page']] for e in descriptor['pages']]
    if actual != descriptor['expected_pages'] or len(set(map(tuple, actual))) != len(actual):
        raise ValueError('Dataset expected-page coverage mismatch')
    warnings = []
    for entry in descriptor['pages']:
        record = dataset_page(root, descriptor, entry['source_sha256'], entry['page'])
        layout = dataset_validate_page(root, record)
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}: {w}" for w in layout['warnings']))
        warnings.extend((f"{entry['book']}/p{entry['page'] + 1}/x{c['source_x']}: {w}" for c in layout['cuts'] for w in c['warnings']))
    if descriptor.get('restoration'):
        validate_restoration_dataset(root, descriptor)
    return {'pages': len(actual), 'snapshot_id': descriptor['snapshot_id'], 'warnings': warnings, 'partial': bool(descriptor.get('restoration', {}).get('partial')), 'geometric_coverage': 1.0, 'ocr_text_coverage': 'unverified'}

def validate_restoration_dataset(root, descriptor):
    """Check every extension asset, including rows not selected for OCR."""
    index, report = restoration_index(root, descriptor)
    for page in descriptor['pages']:
        record = dataset_page(root, descriptor, page['source_sha256'], page['page'])
        for region in record['regions']:
            for variant in report['selected_variants']:
                row = index[page['source_sha256'], page['page'], region['id'], variant]
                if row['state'] != 'ok':
                    continue
                if variant == 'tadisr_text_mask':
                    dataset_load_image(root, row['image']).close()
                else:
                    variant_region_entry(root, descriptor, record, region, variant, index=index, report=report, allow_generated=True)
                for extra in row.get('extras', {}).values():
                    dataset_read(root, extra)
    return (index, report)

def package_image_dataset(directory, archive):
    """ZIP_STORED PNGs; include only this validated snapshot's referenced files."""
    import zipfile
    root, descriptor = dataset_open(directory)
    validate_image_dataset(directory)
    archive = Path(archive).resolve()
    if archive.exists():
        raise FileExistsError(f'Archive already exists: {archive}')
    references = {descriptor['manifest']['path']}
    for entry in descriptor['pages']:
        references.add(entry['record']['path'])
        record = json.loads(dataset_read(root, entry['record']))
        references.update((record[k]['path'] for k in ('source', 'acquisition', 'layout')))
        layout = json.loads(dataset_read(root, record['layout']))
        references.update((layout[k]['path'] for k in ('analysis_mask', 'preview')))
        references.update((v['path'] for r in record['regions'] for v in r['images'].values()))
    if descriptor.get('restoration'):
        index, report = validate_restoration_dataset(root, descriptor)
        references.update((descriptor['restoration']['report']['path'], report['manifest']['path']))
        for row in index.values():
            if row.get('image'):
                references.add(row['image']['path'])
            references.update((ref['path'] for ref in row.get('extras', {}).values()))
    archive.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as output:
        for name in ['dataset.json', f"snapshots/{descriptor['snapshot_id']}.json", *sorted(references)]:
            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            info.external_attr = 33188 << 16
            data = dataset_json_bytes(descriptor) if name == 'dataset.json' or name.startswith('snapshots/') else dataset_path(root, name).read_bytes()
            output.writestr(info, data)
    return {'archive': str(archive), 'files': len(references) + 2, 'bytes': archive.stat().st_size, 'snapshot_id': descriptor['snapshot_id']}

import json
from math import isclose, isfinite
OCR_IMAGE_VARIANTS = {name: (spec[0], spec[1]) for name, spec in VARIANT_SPECS.items()}
RESTORATION_FAILURE_STATES = {'failed', 'missing_assets', 'runtime_incompatible', 'inference_failed', 'timeout', 'resource_exceeded'}

class VariantUnavailable(ValueError):

    def __init__(self, states):
        self.states = states
        super().__init__('Prepared variant unavailable: ' + json.dumps(states, ensure_ascii=False))

def restoration_index(root, descriptor):
    """Validate extension identity and index rows; pixels are verified when consumed."""
    ext = descriptor.get('restoration')
    if not ext:
        return ({}, None)
    if ext.get('format') != 'vtm-image-restoration' or ext.get('version') != 1:
        raise ValueError('Unsupported restoration extension format/version')
    report = json.loads(dataset_read(root, ext['report']))
    if report.get('format') != ext['format'] or report.get('version') != 1:
        raise ValueError('Restoration report identity mismatch')
    canonical = {k: v for k, v in descriptor.items() if k not in {'restoration', 'snapshot_id'}}
    if dataset_hash(canonical) != report.get('canonical_snapshot'):
        raise ValueError('Restoration canonical snapshot mismatch')
    rows, counts, aliases = ({}, {}, {})
    allowed_aliases = {(p['book'], p['source_sha256'], p['page']) for p in descriptor['pages']}
    for line in dataset_read(root, report['manifest']).splitlines():
        row = json.loads(line)
        key = (row['source_sha256'], row['page_index'], row['region_id'], row['variant'])
        if row.get('page') != row['page_index'] + 1 or (row['book'], row['source_sha256'], row['page_index']) not in allowed_aliases:
            raise ValueError('Duplicate/invalid restoration region identity')
        if row['book'] in aliases.get(key, set()):
            raise ValueError('Duplicate restoration row for the same book alias')
        if key in rows and any((row.get(k) != rows[key].get(k) for k in ('state', 'image', 'canonical_image', 'source_image', 'source_box', 'crop_type', 'reading_order', 'operation', 'parameters', 'provenance'))):
            raise ValueError('Conflicting restoration results for identical source aliases; select one alias per source PDF')
        aliases.setdefault(key, set()).add(row['book'])
        rows.setdefault(key, row)
        state = row.get('state')
        if state not in {'ok', 'not_applicable', 'skipped_by_configuration'} | RESTORATION_FAILURE_STATES:
            raise ValueError('Invalid restoration variant state')
        counts[state] = counts.get(state, 0) + 1
    expected = set()
    for page in descriptor['pages']:
        rec = dataset_page(root, descriptor, page['source_sha256'], page['page'])
        for region in rec['regions']:
            for variant in report['selected_variants']:
                key = (page['source_sha256'], page['page'], region['id'], variant)
                expected.add(key)
                if key not in rows:
                    raise ValueError('Restoration manifest silently omits selected variant/region')
                row = rows[key]
                if page['book'] not in aliases[key] or row.get('source_image_hash') != rec['source']['sha256'] or row.get('source_image') != rec['source'] or (row.get('canonical_image') != region['images']['original']) or any((row.get(k) != region[k] for k in ('source_box', 'crop_type', 'reading_order'))):
                    raise ValueError('Restoration source/crop identity mismatch')
                expected_provenance = 'neural_restoration' if variant == 'tadisr_text_mask' else OCR_IMAGE_VARIANTS.get(variant, (None, None))[1]
                if row.get('provenance') != expected_provenance or expected_provenance is None:
                    raise ValueError('Restoration provenance mismatch')
                if variant == 'original' and row['state'] == 'ok' and any((row['image'].get(k) != region['images']['original'][k] for k in ('sha256', 'pixel_sha256', 'size'))):
                    raise ValueError('Restoration original row differs from canonical pixels')
    if set(rows) != expected or counts != report['states']:
        raise ValueError('Restoration manifest inventory mismatch')
    if any((counts.get(s, 0) for s in RESTORATION_FAILURE_STATES)) and (not report.get('partial')):
        raise ValueError('Failed restoration falsely labelled complete')
    if bool(ext.get('partial')) != bool(report.get('partial')):
        raise ValueError('Restoration extension partial status mismatch')
    return (rows, report)

def variant_region_entry(root, descriptor, record, region, variant, *, index=None, report=None, allow_generated=False):
    if variant not in OCR_IMAGE_VARIANTS:
        raise ValueError(f'Unsupported OCR variant: {variant}; diagnostic masks are not OCR images')
    if variant in {'difftsr', 'unihir_restoration'} and (not allow_generated):
        raise ValueError('Generated-content comparison requires explicit opt-in')
    if variant in region['images']:
        entry = region['images'][variant]
        image = dataset_load_image(root, entry)
        return (entry, {'state': 'ok', 'provenance': OCR_IMAGE_VARIANTS[variant][1]})
    if index is None:
        index, report = restoration_index(root, descriptor)
    row = index.get((record['source_sha256'], record['page'], region['id'], variant))
    if row is None:
        raise VariantUnavailable([{'region': region['id'], 'variant': variant, 'state': 'missing_variant'}])
    if row['state'] != 'ok':
        raise VariantUnavailable([{k: row.get(k) for k in ('region_id', 'variant', 'state', 'reason')}])
    if variant == 'degan_watermark_removal' and (not preparation_region_review(row['book'], row['page_index'], row['region_id'], report['configuration']).get('watermark_removal_approved')):
        raise ValueError('DE-GAN watermark-removal region approval is unverified')
    if variant == 'difftsr':
        review = report['configuration'].get('reviewed_regions', {}).get(f"{row['book']}:{row['page']}:{row['region_id']}", {})
        box = region['source_box']
        ratio = (box[2] - box[0]) / (box[3] - box[1])
        if not (review.get('upright_horizontal_single_line') and review.get('allow_simplified_text_prior') and (1 <= review.get('character_count', 0) <= 24) and (abs(ratio / 4 - 1) <= 0.05)):
            raise ValueError('DiffTSR applicability metadata/input contract unverified')
    image = dataset_load_image(root, row['image'])
    box = region['source_box']
    scale = OCR_IMAGE_VARIANTS[variant][0]
    if scale is not None and list(image.size) != [(box[2] - box[0]) * scale, (box[3] - box[1]) * scale]:
        raise ValueError('Restoration scale/dimensions mismatch')
    if variant == 'difftsr' and image.size != (512, 128):
        raise ValueError('DiffTSR processor canvas must be 512x128')
    mapping = [(box[2] - box[0]) / image.width, 0, 0, (box[3] - box[1]) / image.height, box[0], box[1]]
    affine = row['image'].get('image_to_source', [])
    if len(affine) != 6 or any((not isinstance(a, (int, float)) or not isfinite(a) or (not isclose(a, b, abs_tol=1e-09)) for a, b in zip(affine, mapping))):
        raise ValueError('Restoration coordinate mapping mismatch')
    for ref in row.get('extras', {}).values():
        dataset_read(root, ref)
    return ({**row['image'], 'operation': row.get('operation', {}), 'key': row['key'], 'provenance': row['provenance'], 'model_identity': row.get('model_identity'), 'coordinate_mapping_evidence': row.get('coordinate_mapping_evidence'), 'warnings': row.get('warnings', [])}, row)

def resolve_variant_page(root, descriptor, record, cfg, *, index=None, report=None):
    from copy import deepcopy
    record = deepcopy(record)
    kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
    states = []
    for region in record['regions']:
        if region['crop_type'] != kind:
            continue
        try:
            entry, _ = variant_region_entry(root, descriptor, record, region, cfg.image_variant, index=index, report=report, allow_generated=cfg.allow_generated_variants)
            region['images'][cfg.image_variant] = entry
        except VariantUnavailable as exc:
            states.extend(exc.states)
    if states:
        raise VariantUnavailable(states)
    return record

def ocr_input_asset_identity(entry):
    """Keep the existing pixel/config identity independent of new report annotations."""
    result = {k: v for k, v in entry.items() if k in {'sha256', 'pixel_sha256', 'size', 'mode', 'image_to_source', 'provenance', 'model_identity', 'operation', 'coordinate_mapping_evidence'}}
    if 'operation' in result:
        result['operation'] = {k: v for k, v in result['operation'].items() if k not in {'method_family', 'algorithm', 'perturbation_class', 'provenance', 'scale', 'resources', 'threshold_tie_epsilon'}}
    return result

def bind_restoration_variant(cfg, root, descriptor, source_sha, pages, *, allow_unavailable=False):
    """Do not put other selected variants into this variant's cache identity."""
    index, report = restoration_index(root, descriptor)
    states, identity = ({}, [])
    seen_pages = set()
    for page in descriptor['pages']:
        if page['source_sha256'] != source_sha:
            continue
        if page['page'] in seen_pages:
            continue
        seen_pages.add(page['page'])
        record = dataset_page(root, descriptor, source_sha, page['page'])
        layout = dataset_validate_page(root, record)
        if layout['reading_direction'] != cfg.reading_direction:
            raise ValueError('Dataset and OCR reading directions differ')
        try:
            resolved = resolve_variant_page(root, descriptor, record, cfg, index=index, report=report)
            kind = 'page' if cfg.image_crop_type == 'native' else cfg.image_crop_type
            assets = [{'region': r['id'], 'box': r['source_box'], 'order': r['reading_order'], 'image': r['images'][cfg.image_variant]} for r in resolved['regions'] if r['crop_type'] == kind]
            if not assets:
                raise ValueError('Prepared crop type contains no regions')
            for asset in assets:
                asset['image'] = ocr_input_asset_identity(asset['image'])
            identity.append({'page': page['page'], 'source': record['source'], 'assets': assets})
            states[page['page']] = {'state': 'ok'}
        except VariantUnavailable as exc:
            states[page['page']] = {'state': 'unavailable', 'regions': exc.states}
    selected = {p: states.get(p, {'state': 'missing_page'}) for p in pages}
    if not allow_unavailable and any((s['state'] != 'ok' for s in selected.values())):
        raise VariantUnavailable(selected)
    cfg.image_content_key = dataset_hash({'reader': 'restoration-ocr-v1', 'source_pdf': source_sha, 'variant': cfg.image_variant, 'inputs': identity})
    return selected

import json

def plan_restoration_variants(root, base, config, *, applicability=None):
    applicability = applicability or preparation_applicability
    result = dict(schema='vtm-restoration-plan-v1', selected_regions=0, manifest_rows=0, planned_output_images=0, planned_output_pixels=0, by_variant={}, resource_warnings=[], storage_estimate_basis='PNG upper estimate from raw RGB plus row/deflate overhead; before deduplication', runtime_estimate='unmeasured; CPU crop timings are not full-page/GPU estimates')
    storage = 0
    for page in base['pages']:
        record = json.loads(dataset_read(root, page['record']))
        for region in record['regions']:
            selected = region['crop_type'] in config['crop_types'] and (not config.get('region_ids') or region['id'] in config['region_ids'])
            result['selected_regions'] += int(selected)
            w, h = region['images']['original']['size']
            for name in config['variants']:
                spec = variant_spec(name)
                item = result['by_variant'].setdefault(name, dict(planned_images=0, pixels=0, not_applicable=0, deselected=0, method=spec, parameters=classical_parameters(name, config.get('variant_options', {}).get(name, {})) if name in EXTRA_CLASSICAL else config.get('model_options', {}).get(name, config.get('variant_options', {}).get(name, {}))))
                result['manifest_rows'] += 1
                if not selected:
                    item['deselected'] += 1
                    continue
                state, reason = applicability('tadisr_x4' if name == 'tadisr_text_mask' else name, page['book'], page['page'], region, config)
                if state != 'ready':
                    item['not_applicable'] += 1
                    continue
                scale = spec['scale']
                ow, oh = (w * scale, h * scale) if scale else (512, 128)
                pixels = ow * oh
                item['planned_images'] += 1
                item['pixels'] += pixels
                result['planned_output_images'] += 1
                result['planned_output_pixels'] += pixels
                if name != 'original':
                    storage += (3 * ow + 1) * oh * 1.01 + 1024
                limits = classical_resource_metadata(name, w * h) if name in EXTRA_CLASSICAL else {}
                if pixels > config['max_output_pixels'] or w * h > limits.get('method_max_pixels', float('inf')):
                    result['resource_warnings'].append(dict(book=page['book'], page=page['page'] + 1, region=region['id'], variant=name, status='resource_exceeded', pixels=pixels, method_max_pixels=limits.get('method_max_pixels'), reason='explicit pixel limit; no automatic downscale'))
    result['additional_storage_estimate_bytes'] = int(storage)
    result['deduplication'] = 'content-addressed PNG storage; identical outputs can share objects, method rows stay distinct'
    return result


## 3. Mã upstream, giấy phép và worker độc lập

Ô dữ liệu sau chứa **mã nguyên bản** TADiSR (MIT), DiffTSR (Apache-2.0) và UniHIR
(CC BY-NC-ND 4.0, nghiên cứu phi thương mại), kèm revision và thông báo license.
UniHIR giữ mã nguyên trạng; wrapper là file riêng. Không coi bundle này là tài
sản không hạn chế phân phối. Trước khi chia sẻ asset, kiểm tra điều khoản của
adapter/base/font; không tự upload. Kiến trúc RRDB x4plus trong worker lấy từ
BasicSR (Apache-2.0), strict checkpoint; không phụ thuộc BasicSR training/GFPGAN.

Dữ liệu mã upstream được nén bằng zlib/Base85 để notebook dưới 1 MB.
Ô sau tự giải nén bằng thư viện chuẩn và kiểm SHA-256 trước khi dùng; không tải mạng.
Worker/source upstream được ghi nguyên trạng, giữ đầy đủ file và license.
Bộ cài runtime riêng trong packages được kiểm hash trước khi dùng.
Mỗi thời điểm một model; process thoát để giải phóng GPU. TADiSR dùng API
`load_cogview4` và decoder upstream; tiling wrapper tích lũy CPU, kiểm tra trọng
số phủ toàn ảnh, sửa trường hợp một chiều ngắn hơn tile. Mỗi region có seed riêng.


In [ ]:
UPSTREAM_REVISIONS = {'TADiSR': '9d910f8f67224fae2f57e3d63d68396eebc0b671', 'DiffTSR': 'f9839633f5dbcd7cd80ed0a7609edd6ff5d3b5f5', 'UniHIR': 'f67fa3f36063bf79b957e617f915b584da543953', 'Real-ESRGAN': 'a4abfb2979a7bbff3f69f58f58ae324608821e27'}
# Lossless upstream files, including licenses; standard library only.
import base64 as _vendor_base64
import hashlib as _vendor_hashlib
import json as _vendor_json
import zlib as _vendor_zlib
_VENDOR_SOURCES_SHA256 = 'ce022c0d13eaeb51fc06acb32fa9f889e3f8a5ed38c465389b7995e3f1b36842'
_VENDOR_SOURCES_B85 = (
    'c-riJ3wzr(mgrx>IN#n#twdXjlcsLnZym=;eJ2mE?M`>D?T@A<N@gRGDoOd#H2?jc2LJ*fD9Mi7>FlPv6N@Bp02~|~oEHv0t{v~ZOpXrMwmz=C+27rJceJ;*'
    'wf1)ZSiDIFaazQu>1n!~&2IB#d|8Utpe_0v{iou;l4+8T#gB6lr9&~u(lSpj=4F-_Sm7Yfr%6#HSt^o3T*i5PaVy4ol$P<ZBSv`~i)<tYmr*{BJEF`0$gP;g'
    'c>xu&i!w@51Req0+3jfxMaxToQ)Hv^I?7{eGAfE}kVF7b470&}8mDDcVuPb(5*MOXUdG~d?MPNRU29W2!#J9p0&sz!)kkril$Y7O6nR|0Fa`*wBa(D5nGdla'
    '>hlD~EE~b<gb^4nfSngG4D4)2OtWD!!oM+%Z8pD{B*kS%43mOU1TPDGN!ab+Sk|*#6!8Q)1|SmXJPo+&9~Hvh&k(JpL{x!qt}nBxJ(dJ|G@9orG#XQ-VFrk!'
    'Hh+l+CBDJ(qiizCu5olQcf$k+RcztZ9mB`yBD;!dJUr`ZRzi2V7dR2KYI^0T;xd{{#6>Kz1}y`QBXdYO_Mj+%1d?bXW?4>6x<d`93qS6OqxY|lf8IIV6Z=Qv'
    ';PCxF_FwM3g!$Y#f)}T29r5%2@sIC+Iu=moaOd6eKgIi3V&~mI#ozbez3hm+|2{a}J312Y4^Pwmw+C<b_u%FJyWKZGz1)BITD*XY@7@E6?*pL&+~fBG+mx{O'
    '_l^+S+r7ixAK`K5#r~W9<9~Ke(^vb)?-1&%_lIIf9PAt(@9+NfX6H~G{B(Hm{%8+6@Dkv@+kf}!5SrP0yZ7!mfM(&H*!u@Oh@&5O-n_xKPSc&AU;u~MMX~$-'
    ';Gc*4uYWuiKfZtSat~g<*n__9ym+(6ZNa#9-|Xza?TD8<Z+BkrQLXm?28M^_xP#*7ANTMbwzvcTzk9s@{vD2F_x-!$LwM}KKo5^~-Jkc5_BvwcaQ_H#^6K#Y'
    'TNoqa6so)@C{Xd;9)m%A3u|Ja2tNOGw5K78mwP*J0N4>$rctWWFt2N!wZpxgmv8ri=@3}VBf*SkOB}~HWp{^&_Yjt6_vb9148{I58UtO-f!4Z*agj~twD7Ky'
    'NW4r&qd8IE+iVz5Xnl(Rn<Z%}j^c5}30}qn_|6>Z`x%g9GETQo*I?n5aelh?!)YqS_mQ~F;}N{REX!H3wZ0zZH_26y<>U3}qFC=g-FOykY(CrUL52VGsMmkq'
    '`}Wy);vj;4eZL<4Ab$5l8AG4u7r`K#t|OxyT&?qPHI(Pyd?$9Hdp;nKu99L7m;in>p8$)Ibv8FQzxzLro;}@o^6huxA5;V|Mnx8N<#IkALl;KTAcp?H<FCZK'
    'pZ1=}K6MF=s+n(}Y&`jnF<XfL6HRB|iI-6sVGDr9_1O=L(5bhD4(uQ4B!l0tB4DQfOm3r8yqN<d`se&Ig4h3qBLAI4*)K^fUgP`!f{G~=g7P$ui<}@HCsFzf'
    'Fq5Cs<O=38DQ}4nT_sl;bn*h2J)m(qou|owXMHyd#G7)syszIU)3GS>0lbIJBrc2fQ8J$AaTw);%cKNm2@k<9v+?N~kl3)igo58b`ww`ECMEo4<O#F10KS=D'
    'CGoXtLJs14;C(;rSOtOaA>Btr_k%nuif&Yv&_5glj_?;E=)xrEp(N<w`*j3Z4N-OlSaSp$ja(hwi>LrbJj$bKe4XVVfam98^9FV~Agya!WC<;@f)-h>7a446'
    'mr*I=n;Gm<NjZV7InS`;umuAKW9m6Dwi5Q*H13x36c{{maow~iVXrTsJ=g^w!}@c_M=SPCcDPfG6k6foF({)?SF!3S?fbaBO(tD_gFO+`sQ3_wW89gdt1KB7'
    ';{EPn_aeJNyv}AMN<dBS8tjC;15S~`Pg5YqAqosgs~rK;2m4tx=?I`Ks5r-;SUJh_cs!p(IX4LenE@hZKsbTgHS>591IE=B$m5tHqj+41zh@JWjkr-B2N3G`'
    'A3!Rb-DfYvRg?prmpF<T<t_dqizf^!2lPyV7;twbv0?xcR6O`FBSOK+-MxhQC?r-c;z2Y=`7MEYLsF2?E+d%6GKb;DPK`0lZO)Vy;qc2_8r!2s;%y8P&``Wr'
    ')LoGrLMty(pc`cK*(63;a5@9rq$SG}UGddd$BmR8eDxL6!F)CZS#Aj#2~m;IB)Wx_ASW{c>0%a<WC0`t&Eqs6wxPuW<SdYHK%ha<2geCBiLyt)o%_EF)*19}'
    'M+}h_1IFTLs=lDmRDjG`z>73XaGCvMXAe6#iX&X#1<f%mo8f#QM}_ONqGDeCNV!0Huvq{R&jPVa*x|MiT7v8<B7qb|EQCuVfD&`&BeaqjF49NJUuX(p1;c#Z'
    '!kUdHu^jBHW9UY43ky6h5*pttT*4rl!=GVA^wq-Rk)UJ@$Q<%}Tvj|CIHNe<fMNIutzLq_*!>kCAfvvB1|P<GHcyAecQ}_U)<fH=m?k%b7nr4|8y^rB_X<`Y'
    '%tguL0#P2uk1E3~O0ad34|exN0oxKRTD|mWm2nHm0)fd*E6E@NI5(&!RAiOF)RplJxp04DL#i;-3tS7Q=?qH4s(*8^e{j?xR>A}$;vq`(CZ&Kt2GTERG}%Bg'
    '3dC7=JWpVio~F)}Okj3puVGQyJNSKeswW{LA6N&d0}D2)S5#O$*%CZ45v_)*D$bi?1WK2piT3Sc04xws0`XsW0~JVdPf-%EqzvW;7lxwf?1Nl3%>Q#D93+^E'
    '+G#<LBA8X&18!^k;9U{8c_##JE<#|QMMF_Q+eIVRHWy(H>vyrfjjkD>3YacnMxhn8W?%`({191(*%XAPNnFw!z!)q9V~4+(%<~$WfcQv&HEHkIQVONyNeeuW'
    'QCmYv0=Ag?o+iZ%*<ZQAm8U5#!f2c(BPo~#;vCn~&bivp;E8w^lwpQ77|da0x4^cd$vk3)f(m5LJe_F4Vc@6f@8U(A4lbt=2sposgQItc@aO1g|1JEXB@BOG'
    '?Z1S7sdW^Re;3_uSJ1y(i~nF*WSQOLJ$ToD8f<<Ge>Z|}o;;(!y{CQn`=lTAp3w8NXML_D8C4fXB3eX}@vw~`HiL~m6@A`kP(9#30OEPD(c|~e2?TGugv|e!'
    'Wjs)RgaKEZ`S0Rbu~pbVBOH42TMU18wPauqSkwp4J(j9~lv92eCx5?=68X5!f6iJBDn-30j+OSB-vwwo;_VBB2puXLVKk~G>lc&kVtopuTrZ<xQsm(WY9bt!'
    '@cU{1*~Z33F#9lR3u_=+y27d0#pyfw4SI=sogsKNcE?jzkLhc@5B)B_0g8Co(ZPa18EZ4dRXm8p-ZxLx0Do5t6<G%?`IwAdA_W1z0EzV$UKB9>MMuiJlL<?R'
    'Agb_EPN4qDc_lR#&@!x0V9CKRz=zJyT8-b@f!G;Fvl4}$s?2GMU|}H11&WR_Nbyoo$mV4Mo1l1|l;4r~*=35AhOdHbhDkanu`HU+CR*6shaEGTOd?iAPSf-A'
    'bEtoboFE-WAbfy`6^kw~N<x^J-GXqMZa;ygB)Ye%7)ni7SddE*Z1H}EK%>bPTI`4k9TzYwm-9#*+#c+MR06wBPK*v@H?o%v;~9$F=^&w<dzQ?w=dgW{m@<ah'
    'izYz=C2qR&9HeVG|Fhd!5`bnD1*L2uh==oFkggXA&K>vX^}$b~wfob{9nk|G(yqFdjeu>UID2q=oPpjbt}kQM1h5_1i-gex=)>>y?^QzLO}AU`&oH|Nbvzmd'
    'fGsRa29E2?$@*a4>-TJkUED!_orA=M+jkL^APL|sV4q(BYCv^4_^F$vFxhc>m4GUN<vO@gOn}~L0IJ`pvLG>{B(`PE=0lEQHrE~e2{3rx?V^Z>HrwaVHa9lT'
    'Sr9xonmorncMbyDB#AC2xT4ZDCMm6uN*Hc?Al7F2Eo^n?E<(;LH4g`n&r@s>Byrp;3)6aBl12v#_3JlpVTd2%6qaaS(bcUuFHlxV2k|V42l2U<YPA}eq$3c6'
    'p@1OTsT392E0lR*Cq-F{ZDS}3T+EXR;~j9LKpj^KRsSV%pz}cdj51h(`)x_QWf-$TAC{1a#wb%8Z~@p4+ZzGwc^l{7RTGF6L1e=&z#GJ<R)<HxjYKv<!NedJ'
    'P~d<uNmIjMq7-RZ|1ae<OB<BJhM~eDK`HBt<Vg`}dLU2Xl2?SnAT`B3sxVps+d}4(XE&fj5&L@56R%(35+kkx%2S@qfEOx|A=V73CXgYJDfdV*ZFm7xH=CE;'
    'A?T#nF}VozCc=wek8c^n7o$9zipH%2Dfma;X2dJpXAt-?Dp#QPC8JxBpb=n%A~W(CQxmC=4+ND2HSn-v?l`{FQxsvlUBl2|?hs2ypz2HehKVP3wRELMji_<E'
    'M574x4)&1!;aTJ2T~)lx#d(%?d7B}71njgqOhHnhb}E(IE)dOh20HfiA|4J|W~RjmGWV^t*U$nOGM&{?4x&BKYaBA5Py@0CYV4dYHR&=;akcO3MRXMp6?o!2'
    'MVMxT(Kwh@3>}wNmM1_(u-i!K&sb_`Q?u3wh^??b#6Btqa_vDG<@V9OHLk6A0B)#cCq;Q!YTIb9VJp*jTmu+Q$z}+)ZXqni_tZ8*(4oDGIm0C8W6<abhDMES'
    'xDqKOAbdWYv+<6sCNPo|kq0m@lEEDQ*PYMUyg(DVdDCHo1>j&17dV}?moT#9VdpeOo9`rIJ7Y=CN0eyL5D6S9n(}^9ZG?G(cCPax4`%839I*j&foe@ic1(O}'
    'vn_{g2q`9sBs73^z~|+9nXTVu^E{m7ArBA0IdnptN$Pj&@eK|2!PeC(<jMrut?2!3l&j&8T}vP<L9B~!o;}Ag@??On_RJCqKxRQP3m9Qw8D7&k9u}r+5=fk^'
    '4AXc@)O8s#K|cqwC)pD1etDNTD(+fRD^5`N2B{4hJK3wvefm3+>mtzp00_E5CrGov^AHLPn<%crcyJW68z!R;mCUYTg`M*lL5{h^q2g{0h-1Q%s~?Gn`d5+>'
    'tVqUFq(H#o1sikG3k%C<oJY5Wxem&dFwhI~@j@f0byJxkYV5F7YPvS8B&BZFScf+6NjjCL1G%Dcknzn7I6Ce9uzblhipXh}3LQDLOHHse3VO$20BfAK`$;rU'
    'L3~sTgo^RMnqtlM!e2%tkrdUsA=R#;(YR!rDDU+l>R4fxhgf<UX@{Rp+B~Nru2lXhcYgUNGzE&gAwn+QyW#^WP8O+3NX=T6kxYTXO<=QhORh^p2z?CoY`Or@'
    '{If!REM`^zH<`1L2cx+YEULq5H+vh+*w}M`orXX29KOZEwc_C8IkV+;OJqAo;}583MKp>{{}3#ySv=6*MQa1D#C;2HN!|bn^F1?GcH?yP{zt58XlN)F1pQh9'
    'IbgSy?ceGm&s0u2c4h3ZzZW~8k6XZ`*+&YD65`$sc3a9VM6zUnSAH7HIcq@!MgzKE8W$3f4k}px$If1p1J+T$Y-~vexPI<h<<Q{Vk;Z=v3_`(I+GyC(pZpea'
    'Gz<fDclvWO>!ssKEWMt)Z}w{#w-^1nvQ=Rtj?qeUY8VGl9B=VzESE_Rlpf{f?fPt%4RV5nmRe`%^McH?uv?<Et03B4@#_>c8h{dmATlsrS+aPA*aeCJ0r_zU'
    'c6k~hn!dVd`eM~bg#sw$)sV)8GHS^-nr&eMt+-Nr1saz|Hp9Lq+pjEa%4{nm=cv%m;6s{SPtbKln^<MR8^%6qmY$^Ez{Y&+{D~2ech-_|dTrQ~4^>$yi}SXN'
    'yDj<+kosBlAQv7=*lrw{HpgWn)Wy-0T1o!}O2-1J0YqbvSvu_ZoG?b^qLUD)QF2h{Xt?T<>lyW?Lb~Q10wVJS>98l-wosW*jqn-6lDa-rCEFl(pB(JHI$m03'
    '7LUqyN3)!c>^4u~!9f>jJCNQF^74?7bS}L_=gI-Xj@%(@<T!ZPb&-Rnw@CKn5K$7+u8a{>Voh=Za@1-1pM=Fp{PA+$-{?O>*{XvZN12q9c>AN_B3~MxK|xTJ'
    'H`8!+E{GZVM6E@0&?WNij~JjMV|9o_V2q9m*ufVZsY5|K_=v$eGHwSSDPBhf@96q6fQK9U1+)+l3%5V+46mYez-WCJ&vOtJK^x2RDR=LH)hg`B(QQ%2Q`ylg'
    '80!@9^5i)`&j1m{_Qz+vr~Q7n3x77B@P}KV^6ig^(od)9C%LPxb=Dde_S)9kN33|d#=QtBtO(|7ZA<hz{ADTwqPD~dz7XOg|G{s`5Z|A!H5d=pI#p3=_2BZV'
    'pjD!>CUGH;+rcWYIFZ?fX0^<23V0+9>6_<IH=jP){PwBwvAB#-V<M>E4C2A_;l{VUZ=P&C-FWiN;366gE_(2PqvySiXZ>h6>^<u}xp?}lH+nk!ChiY=@zZa='
    '9mN|^@M*kZwORlj49p#X+gL?HTDPW<Bsybc+{CBOviV`Dp^MI<=DFcQ3umV8<61s3en*vmR(jt)eg5p(=F`oFS=qdZ;?eVq-Y6PA>3uthM&AsdZ9E%1?>~L='
    '?Z%U*gJ+|`x0}zO44(H!-)=tJ=so%7o58cq=e;NWr@i=_(KoBi3h<y9SFIdi2$E3vCd;D`C5Z6HI4Y;nEX0VB@TeEQ!k~u`6xI-UsAsaw5zE=aKWDT|VTHOT'
    'bqWfV&L8bRfCUN?9B>CTYp@QlZ0(~YFUl>$<5D>V#&OvNIOw%rpI|p9x9g^+^uu;9=m#68soqeOP5U%8HjfZC6#UTt20x_p>FkzjKEp@cSny+S6Q2$ebe+T3'
    'jbH<xQT#4pU(6WvX7Ci=nPz^1izSU1Cn&lZY(A&AzYeGP^o-g9Y^`E7L=66IeGbCt1;~JLdL?~huobQ0{~hFU7q_jsG;DHQOrt@QWIPb>q^Vsk&wzp9{zltj'
    '7>?#Bz=ff7f54WIX1x87yrbUcW9T_Wr16X5wvZrDLz^TQsy^yIWU1Rbcz?8i{QmHt;o<xD$6}klw4gy)ZDH6BWPD(&9UxACj!t?&Udafh%9c~Rjj`&u3qi91'
    'Hn_z^sU8Cm72~{YZFIb~+o!an591M8DqC&Q{Xx9T(pZ9_q2<s;HOycqA5;9x0exu=W1d35%61qIvjGgitQr7c4I^2nb-HF6(=g&)nU}YZ^EoJBm+@q_eY*Ar'
    'satrq4G<7=kn=#Mg!x+15Od>NBTH%x<Ew-=JdiO)(VQY>2J>OGxH-v#kP9i@+h9#X!!k;8>0MwuC^QT@;z!UufgEw^6lI5E&M17ej8GZvh%y2d9mtmtk?ck('
    '2pW6ukHn!E0E^C}j!+z_0)KfsBd;985VOk+B;+m$V{G@IMmNcHK9xl};z|Dlij}}9#p|3D*p7gX=aKmFrLO^CuE^|A|3hgyn->xk)hs>$WzI5krt6p7Ck*_X'
    'fudUYphAHWiu3A>$}rZlRng%XuBu8~;(6uyN#ilaDlM8Q*uy6CJf{u=&6RZar6}MJ$grSiN%n#Y@OAY-QOPi(>j-s(DF4u*;3*28?1*1xG0YqY%)o_Fcji&y'
    ')B=#4Ors#Rb&F}10jF5AhXrFPU>(ZQ(H|UN(V``cC0L55Jvz{K0L6T?s}zv8my+;&Aj_!wF~5dJ78Gt_KnR+Hwz2V`k|%IDxN`VC|884=1I6UYEkF#pMTL19'
    'IyA&UO-^XN<+Uui6>!3k&0wL}-i9d+Lu6iI2>U&^$h?ep7-`1{L(t{RFy#G&dA;YTMs*0oe^SHUB?8kHB{8kv98!o`M;WP1!364uA#C6{0+7bn+)BuQK<KMi'
    'Tsy;>Qn;tur)#iV!*;r64tH=FmE*}Y)Sh`Bx9lP+40^xH3jH&W<*>?I^83iwZxZB3D$1mThZbazXerBEGJvXfBwrrjOQ{iMxoW|X5A+6^$H#2yJXYQIln!7b'
    'Cc4ffBe|vndnj%1M<q%^<6B=vi40;RAweO(w*tkKJ|xdsN4!80jb2H=z-h{!k)s3MJ4tQ0dn)>B74c-$VeOSnv|G6Ewznz{^9W5L<S4HOslK6tPE#WH?gB-E'
    'Rt2nWLawS|n?W(-g$N|!$Gn7tAh!*WYWZ{DUciR3PTLA~Pwc-eoOaR>IN-3@!Z2=V%W6Bgf{ptkL~~NB?bN%-iZB`atMRv5m3Tpw*k5H94I8V>(+^G+iSOny'
    'Ab+vHf)E}Ond?zj;PMOvI9##`s1R&X=jHwV?**9we*b|N_V>S|cwRk0KN51)Rj;xOjF2T-#g}KsI6V!Vyxlqa`x)0g8JiD<@la$l{t@GkGx4>r$2GEYTKo9P'
    'DM2Ak;lgs{O(c3kN!2R3W$*o-Hc#l!nVL>Xf1YZ-j+~fv5C*;WRF*}=*W04!46u%*FuuVMX4)u_1OI-y7NC`?b&|&cNW{@BZc)LuJ#-Jc&ol)g8&Nz#vXO$g'
    'M@R0UPR;FC(WHn?)&d)&x1@t(U}I%S$b!yq*xzu~oSfP3a0sNa(h%Zcgw9kDa9Yc;14C##t;>!Yo%o)i2c+A!eC4TU0V?w8uw}1D=Gr?2@w%c_oK%3fEU;(o'
    'b{%jTM6($IJ+b@1Pxz&M=Ba0)&?sw{bcZ>K)0P(>_ZVAkIC%_GW-7|+Z4BhG<t?<h8H*yBpd5Mxe!T_%WVqQ76#|L6uSk_d(cuJm@!3|5>4J;_VB6>$0GQQh'
    '53845Xj_zyeelq(vDYAjEA7?cty4m|6Jx_M$iG5rg%P$rON^9T)fj2SFczZ;GKh<d`DjFfHOh`1YkgVlq1>v`PsKHxIGlfZag$9;2}2;hcF`=y#WmahntCKI'
    'kEnt5GGpfCc1;hJRxPRJ$;{NAf8Pl2HZnE4@r`GKWuMhNGp>2EVUC+7yGFcxTw5+JhO)c|gDk}}o06TD8Hz1NKTH<F3JKZ`um;A=GRd@1?4*1JKt4lhHbC<j'
    'i8p~zicBM|Eu=D=k1U*HFUui(9B2F0%+?2=3~}W2T@IdTsy_H?>?s!!Ej(g4jwJ#tjJSyK;DKS>Va=IC^wK@OJo7hVa-NPUDtAnY_6)<O)IPG|Z{Hq>7w`L^'
    'S`qGBj7w9ok<r{*yJ>m!F@=1Vktrz&Y2qaCovu}{rN=<4pVHv>Nli2>AQa@M0i2-5i!YL4!E&lT7)w!v{EnQ;_=-xMnUv!wVO4MNs3oa~A)emYK3$u{Bhs-4'
    'lQ;rha28s+VKIwfwQj4Oty%c$c(~+OUwybnn}tcIEMdYhX%8@w%(eupT-9wTZD7<cRmI$In<{Cms?c64$-8E^?}6}sz?rjKQ1n<U9{Y>O4aH#&e8ri-tJ1E1'
    ';l)36-}~OEi>8}aJJ3X7k%0NIgbWsa?D)4yQJKA5y79hU1V+=7N$IGC%Glecjo}Yt=t_RGfLGJ$j7*G9*FMtkPbli4w<1z1poD=k&_}&QJ~c71rY3X;x!Qlu'
    '@(gCaysfZ>ihs3_cy?~hK);8gKU8bd0}HA3!C|TWy}jihx;fo?A+;Ak#wEC7FV~C3oz$T2*hyv%i#ll-S{A{r=3T4)_Cg>yH4suTIwVL1HwLszxkLtKP5$BU'
    '+maANpAeF?w?snJnvf<OrDN{7VKQoeazF;01A@x|(}{;F)>4|q>(U_Av-UN|EWlcYZ=><!O;_iNe#X?yYc0+b{tIMN|EPQ&sY{dLzgC=F+;xk(v_R(70&o$A'
    'c)BPZ-$5nuH*YIAo=L{gU|v=jAa|>@e|)PQupJ#h7|!x|l-z8i-?)y>>iH<TiYrtRWQw!0c7uXX|Ked2)7fhXNjpIEuGP@$nP{B@7S4T!mSal9qRPQ+USB43'
    'Mw37|oeLZ@m_1zC3arCSdPctTwYZZC^<Bt{IJV(X#VR7mp~@IXG1;8#&l<DjVh;TCV@ys%gm}J1Q3Qp~ghxkq2wc!SBOoJGz6QWc%SNO-9!oM+70K>dAGb>^'
    'j%X{@h7((7a7+X3g|;lKH+FL->L`Q|ww|=*bfF_2=NFILxF?K~)B%pUp=iAUZ1@`FaWu4z{S{mj<muY=1?+kVl+HxC1`2)|(4UrW#P%Zanl-`JWIv`P(?ns;'
    'mGxB{#q9lAdkNYd-}X(ef_7bvw?Lb{qR7{<pn&cWUFFeOZC7X^rU>x}<Hp&M4qw}Kn;@1XoftYm4ii@S&{-lCLggsoN5e@1#KSE)?wAEg7@?o#BbNGvE|xah'
    '3td%)O^+zUs{VgE#lRzjH05xrmi#c(s119Os$h2XYX@H4R@8ucbosJEEz$n#JjR80nPBdMTjI`)p>2gxM_rI5QA{^NfYg8Lt8M$!3cK^3q$HR?oK0ENfrqCv'
    'V)D{qxf2W}r580WfW{IoLDRc-#!i1lAhd;`sUgGIR%{PTdpL2`p!)SSGDMtB_=P_4U^t)7ik56mLOf$ZAuYEzQFJOOmtHYQ66t?&I;-$uS@gY&!yZ(Jx{dS+'
    't54AyQ#q|s*=N)0e}}es39|W_=wcxpAHQ&%l$KY-lrk>R86suda5kA2trHC&S11(?We_1yBE{bT$$`u{NsxenBa)P_g0);aCul{8?AMV~TXnadl#;OVaKkXw'
    'o9}yDI=10N7eH4bE&>^Y)8xx=H`EuDo<JTlrjBx04ye2_;2jUPP^t;c{DphIemkOUx$=SRan*s3k`k6((19jA!<a^;3~@IYC<iy%qEsMu#EZSd;|ek-wli}g'
    'abVS!xdC+rQ;)8;CC*QF-yEHZ{}v}ldk1IdW?6SAg)Y6rrNs+fYoQQl%oNC08rxnu4!>ff`!8j}V3OicfU;%D>x2{dlHwIKwb!%STPD#dVu2B3QWh~W`eIEZ'
    'VrC^Vu4;QrkBnCh0zGR5!TJ~vT7gXZh+cQ0*KL+S=-39HWuN7kH`=9uW>R&91ij{M*}~ng<OFRy^}ja1){RgS&C&qm*<Ebd=F`^}gs;8Aj50Qy`FBgzD?=-s'
    'fjH-37mjO+BZ4H$+2X*b6O4O0AvdTQF?8mH44pZ!rqN9Z`okC_f;79qIhTyaDX#64GM;J64RE5cVe3us@_qR3{c(7(^Rf;Mwa4jvQYN4<gxP4NK1g0|8{v(b'
    '=<Ri?5~D=>$^sq<-By@YJhFvp1mjV;&Y8J!N1x9qH+UYybkSjJMxLNpgd6oF^D@w;(_X?~oY|}|?3bJ*9o5pT3dN{8Y4RlO2erDWXtkT1w}WPKP%*e_gd=&h'
    '53wIh{(>k`TnSUxN`5b^z|Idyh=HZM?cjeuBNHt1Ic*+vvN8HV;eD9o9glp?)6&B{gS%2Br;O*L|NO~fe$|%Y67HM#?_Td69TR1!6MUG-wJ0lQYR>d3FyHR{'
    'clc)S-Rt8Yw@kPO?X+a9Z+jJR)!Nz;FEip|Fh;+Dp>oZrX+kdpA7WJ0tnB8dA7aagE$+@^Tpf?M5Fni{Box6bZ8uPxZzBL&Sq=nVZHqzpct=Z7Vq5I9WegcT'
    'CpZOCn?*wlA{%oQ1l1mFSyz}(lruC0K2+lQ%0k4FEP?ESyhd}RPgu;9jtK!F%>m3~5-G(lMw`R<-u+|$<^B!`Jo!-Hh9J~2kwG+*xq$1<p8v2dz6qY5i`GA~'
    'Ng25d%Fh(*U<@1`#dI*0)P*XG3IifRPN)B+%(;p#w+N$9p2bPR0?34B_<+k=ABi^@SxO=&*OIUbf6SvveXLcnW7IZB@ZkyOo7(TD_$Y73HFxgG#+ezi82>7{'
    '9n{AtC>yB7xsy?i)nL~ec*a%}TV1IkqqrfOaF*j5-Pek=jOzd4tMIpNvC&@C!<GqRecc124H8eUExz)6GRHy*U-0`yjG_%LtL-`(|F%eGpdooshPA3*+R?)M'
    'nB){s`o}I219FK+;(_h73@ohI{rc(<)50y<p}RU1ltoT@XMn7V-kb?{!w2ryuoHrSqy~F*OF4v8+ijQJ>a*muga`Lz{sR-~z`vK!ax-R^4Q_SJZXjlj9)7(-'
    'Jx>pv3Ak7P>JzX^?`n}F3)*PrYTA@&dT(ou+jHeH@7nFzXE%(z68f;pL6&m{!j-e1PzHxPA8VviXnP_fM0J2DDp0KNRG$_da`YS`*-39Zm<ML2k5l}o2GxS{'
    'zs0Gt_^5Cs`$&wW3m(r7Eb*K;58<SkgC@vgd!!C_8fdI=z!Hr-l_5i!_{g;JhE6Zj5dwb3WbZS0G1Q75%Ji!C1dkh_DkKj(VXjpuT%v1+O9w%4W(RqP2&!_;'
    'SAa80V{umZ;4%8S3ZpMP-21Pe_7C@7hOZCb|8#Iv1@xGw@<CWTT`Mp#3AGs-2V&%zfa=>-6!W`ZbUqq1Ju0B;IJpaQp2npg7S{b#oiGjY*h`Dik90y~mn6Z%'
    '2O?2pFD-+yBqBNsE%AIah-anPrw@CS;5I54`zH3H1Z1v5K3h$tY9}z&nQY2YB6A&r&?VPaoth|p2p{LJQc;3(C2s*p3Y8QuunsA*P)7YzwD3rr%cM*p=A1dl'
    '-SP_0`+}x$k-$u09-CzFf$yIH35`MA#Dh5}O53Llf5m*psmvABgtKP=)yT1ND#8(T8SgF-AL4jcINXB0Srg8)PT5zK{%21FO)*($@Km~<&<@`85aOY9ItdIE'
    'IGZC)nBy}~-^$rLzMb)`+_wb@PD&d|K}ptt1Gfl@W;pE{N&Ewq4kn1v>Dt~6&NU{S!B&WERTJK!EVKzaW<PS!I>N@lGCqU&sXB_R4B;hZnp1R-nKo%j5ToJh'
    'TXn*B-1i^iTQ<wd8r4womAm>+%te3J4w3@1|3N_n)Aj}aRK=G8RttY6u&S_ykkhq!iW!4a=E97Y`sv1&X3c_Ag>@|MOu>`Uyv)$S3b<OhLy#gAnY$%k;&AX2'
    'C#aG+oLCX%^j&hH7gZ7`l|Rf1>Zky61~z$mpN`T6@y!gtqjRlg9OJiW)Ts{`B^Ph2PNJ{i^}DQmg%?I>O0C9B&_?)iOMG;oelh~D)Oup>>lAFOU8!XH-}Vx#'
    'ECrj7T?*|OBQ-^=Kl1dDyIy@T$V=9jLbpq5OBU%*spfPQS$tGLKLtP!877s(E_8HGUdlJgL>U-ZQQP!H1<x4%OJhubGsc93qsI*&R3D5W1p=zhEI^kJ(E>s_'
    's6WcmpYSw@Nd<W2PsXWQKPu>-%%1wC595Fx+S_0``Y?K;uwzOTWN$&vf>TQlN2qO|t#Ftr)$lS?@2KA<ocf3&)@iAktBS@069<hNoKyYBqdkF-XZ6ErD!G33'
    ')-H}p67{t{8hX8~<3~;}$@(;%mA6)R>-0>NZ^&GR%cK&wEQ~Lb$v%FvyKhW{QL>s2B20>_O>Y4(P<q6A6$P9lb<mH*x_D~wBw`SpOlD|~fRUoJv8EZ{awRu;'
    'dooq7LVMJvWVvvpvQ)z;aX@@juRy`2MX8s<3y=;z;ND=4Spr%%-sbzV#4wB5@FNE<@ai#{pY}42hSj*^NsTJB&|Ec-@^UjVC~0~DZm`IGrXTAq!-GtJ_fz*>'
    'T;6BCL*zu;jt;?!zx_r^NxV~>u1%Bb<6<H*8d}J->J)^i$VvrtHAI-&b|I#cvmpgEXQgZ8YDU)`?&b+^^VdNZr$}0{A}M6K;*g=~;8wql`G4H)zhiC-v_Wkv'
    '`ZYe0LcHxtE;uD1<`g9bK6hNEyv={C7GNvb#wlQAcif$Go0R4jKNx9Ul*Gu%^+}Kk3H^#@?kG4K{%0NX%NaZE6!$#EI_oN>n<NbpLxW?=(Q0gLd<<Ak3VAQs'
    '2E#sjPG#9uW_&w|t4?$yZ&BbJKhIuJ+DhZZ6zhDcw6c*;1XRz$lx<0LqBnT`^t9TGVfjlnM<>bw#!ym@N7tEfZz=cRN7@v8KeZO;R3l5I(A}Ekrp30u<)5m}'
    'f)C8+S^!l2)&eV(Hsy(B{If~e;K5rNjy{|9w(TydIbFnq8~JZ|H%(m0j`q^y9#C8XqFXn94~cAHmL!q_%-fdD%a-e#Qe*L!76>f1ZnkabeO4WK8x<e+tLrf('
    'Jl>0*ZEn2v_hVVDmOY_lR0%7Y!<{N-v&vjk8K%v=*OgtGv}cw5smr%W&2uu#R@s~}?X|jSPm7lN?nHU3#RFNYSF%v+oJW-Z5KV{aOqM^`f1}D$5~xMVp8Tnu'
    'mfN-^3U&!wl0vFnT3fP{HYYaU*778pA9*UOosZMkRof^{zgiqEp0O>bPwj^Nv1UDSXH%^yWf`4jS1d81Q57$$Ft-`Lgb_o4>$V{7S76%Vl!#(3ZJYlvz<{mt'
    'i_O4m`%*VGF-)Prr)%E{mC!{3lp;y{4w=^6^lBP0+qEJF_txpvNLphQRD@y5Ymx*i(T)q(D5Gs<x1$tovM4_}+4E$Vm`6~fEdzRmtF~LAi23${mTe228E7!b'
    'FCSmjgf>NUwy|Hm!*p!hydncScEQ4kJR}A4dHDK;CWk1`Biu*RSr8TU&>{%Hr_hAE4e#+%lk!QwE!IW<>C<4t>NOjmah^kB5G{hy4rXyaotJT|-w_)f(Q5}F'
    'e8CPM{}#74OhFY-qu>x_qqUJbQmE=K-)>Vu3@~lt;1a|v>Mi^o%;xy-B%01z+ym96UPoZB+5vZ1B7*US17R%iK*U!(W)R=dfCvzyf=t_hmpM%C^ESx!Fou@S'
    '5N3=EUAry(C!{-`i{-<kh=6Tj)CuHGE~r1q(XKSu#bcv(mJW?`s9}~-AQx+x)mB%#l|k=!Y$%|X)CR0F%38XE)F15&-Q*>B9M@FJD=(vgp<|dz0l`(fCS7+>'
    'Q%V6KJArJS01qtiMSMEEi|);o^))LhGGXpqjsqE#oC|gLfV?5f)lo|83cEn1<h^qiUAfkov9dc(ZqGz}l@E$HeU;xhkO}?KvJv)&Yyr}o_@|kagkIq5{>#&}'
    '?lfEqc6H!YCXF|Ei~GAQpT3%>B$2h4pP;%j&ZA+V-YbsxpD1}HgDG7|Upb0l?ysX9)Tr{NBW{#8ts^dy2v%J@vpg_RC_um7Hh!pHhaJ3IIZa_a9elu)kvD;x'
    'p>G;TDSf~n$V$*hI83Id&#p2OP<Tq~Ms%qju%PU^)o(AWo8n1>35Cg;wSoV(G}y0cRKOE3370)K;yr+o5Pg`0mI`1;cT6FvemaHU2PigRCo81%IvinT^zUmz'
    'aZ_1+q}@Mjx_>lnINza6U%KT(M!oNNLP-vIs7}fl{!zT<Y8WD1C}@C(Avj~~_H>Uc+e}nN^0iD@T@wLc+22g$ES+aJ^wa?4uZkqV3xwv~AO=a(ZvQo!TK4eG'
    'P8RcN3tPaA1NOxyZ5Ig6u#LclbtDy4`0JQ`=~OT~^4ySz9rS@_)8K73oKN)WM^D}wMQ#*=LhF1>(3S|@?Zw?^R%RJD%ZZ!yMdJ!uK2R1P$&a9)MN75Tt|jbb'
    '1bOQH7U$W5iUt>|^D-Xd&6b>Tj@l)G!F*BoEZ2ByK-Vj`ZaU^TJ5|>^x-)i`qa9+{cb|wmVSNxW=8h-?8e=4}LAq)>xr%qL#s^t80dnYpGy)$m<v@Sfs`!k-'
    'aoX*NMB@!bM^X=4cKWB6^JsEZ=5aKAjrl6P_*vL`N8I|lLvc`cIrGW%Ox^%zxareAG`O6nA5hH3^=LA%TPSmjWp10xsIm57u($C27JuCI{iB5`YFRUN{{e&e'
    'OToc>B<9Y;M-AWxHCcTGmrZoH_OaS2O^Z4~;CKySIUpD=gNx(VgO`rDZNkND_0{neHv+4pR(hp1LV>ciWul<^O6QA8|7lcerAopvz(VR6oowA&>+0?N^*$l>'
    'eV+WmT!$po9w5o4Zks|pv+X%wX>3`V&wH{|E#h6Mi6Q@5sTvoypPuNnZQH$$@+z4{T+r^b!_t~RYfPx!q1xM)*ICNH4G`{(MQR23w^)zx*{*}gKNkeiP5<t6'
    '<EN69h~o37>T}S3@%d=KxEI=+m=qqnarqRT&iYJVg>Xa8KqUdnCw1VLSw;)X$pUSscJU#Zk>h!U2@?!}FqMV}n}N&1e6Z|myTz|Gz23Ed{g&=Y$7ra&vsE3g'
    '_0+7`(6fs$w;1;_uS$)Xv+;v3@1zSmld8Hl@!%GIe(qi-b$;$dsn<4!Rd)it(m*|W+JnVb{nLN?tX)gqu6w8q9NX1?jt5vBHgIC|2y1tfO}9A$M?RsTqWW~!'
    'yd>wd72B`xCax)_mCmQ(1bI`j-N$gbP_@5JXuwz@`wd<eqb^A2n}`-9WUOeeK+6y)7~80ob<_*MW;ntK<szBP`T8MxFE_9!{W}0he?>?NbXZ&_BXoTbm?|#+'
    '3nCsG5wJ@BHzI<Dy#x`BNpJt!m{`DeblhM~wXi|Jx7P?9r5!?|jU{|jjX|h!yB)z6k~crKJny9}L(qD*?CVG}m6yW2?YX+$CsU5^K{lE2@i1+*exCy^#6#oi'
    '9{ZG5?UEn9`-k$(@r^%pR>82`AMC&J`_gU8zWEfSDHYN_d}VjnXT|1aCbMJy{S9>GU0n8G0)mEgeD5t^H}&%6!P}$3WsLg-NI<V9+4WmWz=6*%lVSieL2MM@'
    '8_h*^xm6X|+J8wf-R=bBj>C(n_yQf(y?;b8xt`oRjcWb_+*hw6YXZi+2;^%Op@6cLx@;-SBX+?rxG4z{q*Fe3#Ouk_9Dwvp*Y1F-Zm6;oeLuX1<zMer`NRDx'
    '?>b_)BYy0NpZyM%`HJoo?F*T&>PNXXsz~R_-TWrE<Z|$MN)WQ+LfEbRBe$+wWVhyBkYf1lLwHCS=Mwv7+>?cQ%Y5(5fj!Q}UX}4OUrK$jI87nd;~66>WNp^m'
    '@-%y6L}z<ZHQg_~V5S$;JS1|)(NQx7Qb3Qsg5Q<DJkATiw&OCntvn@9s-OMp=SI8Z^^w#|yGPl%?#j-PrS>pmTdk==22?i1EwGa1@fbF>I1hCIZ0j*k_T$b9'
    'UXoS7;WF_9b6qF*@tw5)THcZRY)B|eXTswpsmVoNpGlMS<hKM_($ihAx{|X5#JT0zs^iFluoP1=Tt=cx#m+1k4I2I*(6i$At(aYF-S*qk(_+28>9;@M?*f2p'
    '*`<+@-fC}&PRv3clbb)yV{!*~N$pY)hYnU-Gk&khYTgnz9|;~Np<ZFUbdO>?jR0=^a<XyOF_i}Q_>eq%*fCHns|A?OwD8{MR*i#h>6Mr_z_b_C$(Psh9Hyak'
    'w8I;-pD}1FbZvUeotVY_#ArwW)xL9QJr!47zISvv9g>2qRsMy<xikr)VI$yx1pBjTF<2?9AO>uL!Vc2@U|u@6$T=cJp3O2Wqk<ncf{l*vkd(Uavt~Zv&mjR>'
    'p=-5B3wNW*(h&9SC{OTAsEV&Hu9M=@x<#5$`5~FHjUpN>9w%r=FdGI=7G+rUqLBhmbQ>OX6_xIj5VzWFEweVe*RJMqh|xE)^_r16G_2CbQ&rCFm|cdF)SF)|'
    '-?;h3W7;u5E+~l;VBwQ<L|4x)cQvOLJr(A&!(%HFY{;u^6xfyzHS{8CdpajZ%@$=JxD%Fn%D^NG&@mQ?Mwo`MuA{KD14Hle<{{-O3y6tG)LU}+K}P_$pYYj{'
    'VT=k*WxlF4K&XnR7ie1fMJHPJjJdz!m=Sx<xb0r9HgL&s%tl}GtCsv^>eikmrRc_e4ZF_*^1~{93*UC<eIW0Lcwk_fe<8J5#;BIX-s+VdHYK*}i~Da4<A-(N'
    '*0xr$^U+I@X2Vrx?&{!It%JUsg<U`u%00Je8bt5SCSRYFdNFgBdY<ok+UlRwJZ^ZtA-~_QZvma!{TWRX1}8ORGl=73Ht=?t42N;b=fapRSGCMZa?4drt7Uy^'
    'FRfD1XRYRvy2Q2aT8Ws<-7DR{my(a%4K2hV7PO=mfKA=a$+pdDG~3X1TV@HXDYgPtbuCa|+ug6puU2Kt?C^5`4t9q>GA61iAj%f02hfX-Ee>j*){=N9sHtLL'
    'Rt{eBc#Pa0>%gV<vXV_ZDf+Ff9dsp7?9N)DpfP?=bk04(qEu8(M;*#9D8$?~pspMvwFKS*MjB4|kHtErS-}jlv5MNF?}WuFxuRh7UsuMbNLPZlx85+CYJF))'
    ')uAo;m?!a798U1=h|;_!Lps@{ZPyJz8pkOr$hbjhO@T|iR$sK%AlO4pCk748h+bk)i$2>o!aFFU5_Dm^am*jc9Sny_QPLescsE!DR$lueL2fl<)Ku^S16aV7'
    'T6#cczLwAHPLz+Q(ajyZgVkGi=|f@P)w8ID!h7<Z55Vp~8_C7I)JI@<v-w0Oj~SpVQ%D$iXiVqlwwsX$?wPcv(T6zf6=ggN3-q*Vb5EK<d;AW)uUrP1u<5kr'
    '^OZBCPr7B86XEdbvnW|FhFuuuMTTJl#r$GK*$I)&s`~o0r?)jakcyfHJ`}jb<>^;-AbMftXflII2?z5j;DPL}D-;V5uyg*^ThGAYEvCv;cR_}u>!H4E4=GLI'
    'lb@`;I!Rly3>TSH@lz_1*&{luS7iyqCmHRa^3PYV%*4A2mP&_XHDvXts&5y1B&Hw=sN1;g?!y;f70(z^JVUx$S}+<cFH?s@pXPZ^3$sE5Mmx%%c3MYfJ%-%U'
    'oB-lJE&;z3^GNoSx8+dfr!;k!r%yLzgkSF-@qH)Fp|-!av}D%X=x;8JhW-=E!Vpb|+1ctbgw7aiVNo8D%?y-Us2pzN5IjY!SgM9HtAv@ku~dyE+~BIlf{UW9'
    'mIhZXmRi4H_J0m$k)!;NMC=3O(vK&bNc+#1p|UBq@~@3cn;SoPa3$jH;esm(nj2i{=%Guuc8qxcvb}%V-oI=@UoiV+d;hY%f7#yu9k+MR%cK*qeg1x4PvJ%u'
    '&|qC=OJ4uGbuj`*f1`oII_yw0iQ&!T3gD-8zE#D7Ag>qC18J7VP_~*)33`gY-mqm<=vcMGknj?QZTp5Vz$4$~hsP!d8B80WW<-lhVRu9&%c5uqvbb0w3rd*L'
    'rx8upXp3y`+Ka6o$<4{vY+JUq<|g}|Ch1JgAJ^m)Pm@H7e_WHBO-<5JAF#<}oMw5fj`dOnlRk7Ga2=qPjqV3$GV_pw4nv0Z{j_~wF#U%C^W<T`Y&L?4a$IvV'
    'K}E=mov*pS>hXO^<kzi!0yi-pbTEW|Wqi!&{j#%iuvb*bmsd1ZXmnpZabIk%)I^zuAGjv``c?nB(!Zdu!Nv-p&MXoPF_C@9X(s`3)v>5n&L<a>l8Z?Gid$GT'
    'FN!2eS7^&SN89x&bCw0}=rWaG%jsiB<@lppfuJ||9dT(fY}z-O6p)wK-UnsYGW-*+0>n25_|C&E**N%C0k<R0cqRA~_lB&MXShbE4Rl^7Z7)Heu3>zbUKR9^'
    'sam5^0NoAx*3$;2KmEnrW6$h+7Zy>~=2EJ1>7@yYZO^<}wBvTgLm^T;8-dtIffjx7)mNg05!;$pRC-tx!K^r>w&;3H1<ql{?fN;!mfP;Ow>K_idB<!houjz#'
    'D8~zpT=vuWY*Mz7$Cf0f6CmQQyr_35zseh_&&-+SgIuz<zQx|WHLL<$zdX$J02)!ORqk90zNt5HLUdS_I$b-f(v><JDf{X1nopeOVTfnm#zl2cpRvN?n+o7o'
    'L%TId(DsQ_*X>MgMUmv6+@~2`v3bCsypf*df7MgN!9T5&{stZ`a9y$i-uqoY&7i!eltk8bIlE2j<F4Daldem+q^?>gjVgowvb^u^fiAwTFH1unt40Jrwwh=W'
    'FNx$5ZTId-eqB>a^m>Cm#M^Y+zu&A>wi^GUDpfyqHt6PVn#rQ=FIE6f*kU==(|Ioh{ICiDV?+v$-F>1FfB)~(xOb+giMD-ODeZ`GrXBQD4n>M#b-=KGtcGrR'
    'Fs}aboMs7UcF+8w`ifzrFzSxPA2dx=w0s;`><=A=r&~T1#6PALs`l?+^~wFKZmu>7=JHf?VNf%r?dDwW6kGzrf=$#z<32flabjN#a;A24qb1`8iMCU<5^!ws'
    '$N=P#n;al{lzx!+VPUOJ=X34SEV>l$!<M+>V`&}uVXY#wzv_6JOL8ADR^L#^=-sF0mdPTspkbWVJG9FRgWIS)`i+^&i=mu7bAGE_{d#XxQAAZwXGVs7eNzv8'
    'vc)F1iWPQd$#w2EBGpgt?C6U{Ty{Lm@xBKp8XAj0ylioLn6hD{|4sum;37S&3YKZB5$Zi}Tw{zZymrm#ud&=0blbe7%)LLxo#hH5Ubj)7VzN}CvZN$V-w{Pv'
    'Ey*}ZBersN#30XR1#rwP9~L0Wf~-#o?Ie%kGIa9;yd!y8+D1`bWQ%5qN1dy4B_<)6ox#{J^f+4WW2S0f2jIQY1B+|V@XDGYjgw?|Y#uX1G#n;1^(ZuLQ;*d#'
    'Dp~t0KF@1(wv8QV9<DbllDZ?T)-_v`)%S}8903nIYm$beIKpiIg>yKe0p{Pfy=ZVY{n{;2brj*fQOFZyLj_*XMrw>eObFdqaTSv`ozc<fF({lL{aur{v_?jS'
    'kG`&c64y}ya}UFghk-c0lougjKreP80lepV%y$Xn+=&Ca9)fRO5a;H!R<ns+4Ut&SDdzaxn!<iiUG!qi<$2)o-87Q0t%{i+&(sYm^l$wnyjj*<wMgXSc{0S4'
    '<I-55$KqYU*KV|VHFTr_d7_t0=+qTeoF4QL<Rvb0%`QlPjgyy`v_h_f;xdA<(V_RM#5E?rfko8AjEo!BL7r$bl8eCVq*z}UePiWJFN}?5+_I32PysCEx^HQK'
    'wwcBR0CcI+8-W-!j&&V-exScys8$1VjanKPIoaYQEN4&_KAv#6{??hl-2o`}GRVIGDFR`u?4pCLpy#uC>Ihup7H0lK^a;IN=tIIo7U+?44qF?orfD#eiA$nM'
    '07bE(j^ImQ+O=7Pk35IihMG@GRbcg^PQCEiBw7%bKpF8JjHu{T-r#eAsm0_*f(}c1U4%1cs?%Ai_ZNAqQgwQwVOOc&m?xWR+3Vt9|4qO*-R$#s+<)5bdvVEg'
    'YT}&OQ<>rIE_gc(E~s;}(esWtcbYv%`4n3w#)y|y)1^8Dul;Eu?~&LI<sBpHu_%YWlbq@(gk$A2&O#6%eIUazhFXm^pMZ1vWG&T2Alo`0)MhOo)M}lFVK3`D'
    'VrF^Q8GkGF9E$onZu)uOk47YU<6fgqeBg-A;}oy*liI~#HV-o{c}`m&e;qa=fo$_6C&_BzWtM%g5K#ZTVDPLt+WR59OlI@it-F$K&4*Dyw|Iwx2=-kk3yQo~'
    'jBnmTrpXpbwTkV{(6|y;^8=`CJbdNn4vs)1@A)FO)jLBjwplx(iJk~~8b@}|1mekY@`!AsqQI-8&}ILRojtx8O2}Q3WRni2UQ9}nUBx-Pzpe`Op4E&oE<u5M'
    '3)}mAHi(MSW1F#&DXqi}60I~oGJdVi+`_X$Sj8nZUzMDYC<*uV>U0-MmerhiI=TY4lpOP)*&GP^DwdKeh=jxhvr)z03sau7J0j)C>K0hMH+VEcvF$46F1oDK'
    'Xj>3Pq3l@==2IZvTC+@_xFlUU8}r3tvOn_jud1(sue}<^=}_7Nocre9rs|C|@x8e&aFUG2|8NV#64==CV?t4tA)wGJ6lhuoOpwNGpzV&#LRW{&1N@Bs{;j#c'
    'D-xHS<XyTDO$_I=Nix7><Z6%3M&{gpwA=rL+qXkW+*6ZQL=$n<CfvhvP8vq%o+hmf;>3|zBsfVxf(Pxhhm%lALi*Q9a@KHT6lH2cp2T8rH5yO(v!#QmXvX~D'
    'rwb2D^*~&0{Qgn<INKdrY>ju;VLCg9@n}|Zy3r&zF?U@_Fj*k$;p5WHwFwprI>ytTAvq|qXaRIEOTbg-tBol^brw|$>5E@C`*I%=S_1zotGpvQ7%Tk;du}yr'
    's<SR{F%QRmB=~A~yTiC5D|mJdWs3p{PR18#k;j&ckU9Ihv5<C6j&FQH#=oH9`syZb|8SV|@cz}*ua$YduUTUG7I!CEoyS}R3qp52RJkIeD^%9hIfu1)Qbamm'
    'JgF0K@Y|l;W|S7oGA31N*`5Zzd^ldAyPn|8mu9CN)KW_-iNhB+gQ?EsgJktJXPjdQ!$eEg3tRK*2jXRnNvQ;Un~2Ig+MsOo-0PfGM_qrs*~rsC9I&xNP77Z;'
    'c`-4wv^X4Uv2}!v74|#BNaDn3do3m&)y|5oVOyV@6wti1x+criB6#XR_2%Jdz|V0W@U%V}iYHk%a}gtTV0C$>m2RQ2lpVhkuw~RJ(yG_$IH5C6KVe$PgM)HC'
    'B^SV;{$x0hCT2NKJAh?P(|yZlO0e4YsoSa}Zni|Jfm=C$wk>44SqSs2D8i_eR~5_nA|D5+1tD$wgQ=Wq%z4NsRXcFerroBI+KoAav9>Dwxzo`RT8gYS*S2=F'
    '%Q!0WKp!&LK_Elg!to@#fL%*{Y@M!M0ATSeOkh&pb}p0gB}`}sczYgC<H@bSYq66FbOiamW#j-%rm_JAPB(L%<wM;_$oE;d9wUgy?Ev4yYoC=aGl?5yoyEdR'
    'Q+-`)mK9|PV*@fnpL)wE)?Qj!c^*~1i+feBvQ604y=ZiOp3bq%mMUt-JE&L<QkoXUo^%$5mo&yKFA6MK2y*kx4#-J5qNUDIhUiJq4|rLX_Y@SyGgFbaoig{g'
    'WY(&+tj}kz8h$OBVptvyQvHBz+Y@w@fJo?5=Lwfds#1K?EQ9Lm>U_B`i^$2=R=0QNS9HCLjh?<%ZMtfmQK(~5tT9I|Z<%%@9H?!OL&NHF7KlYM=5jFZIe;(b'
    'Vl-w$pBVEkq*a)JP6%^<K7y%Y{I!cbv?RY;)#irXsx_ez*r1XcRoRZthljfB>DtU@?keewF0JrtCC*cq*#KYPV_Lk!nYNBTtF~>PR8v4PI>Qany-Vu7WudRN'
    'NK@$&%ZdO4bAqIRp0!jL{T{DZurt2P%2(Mu9kS+ydX^jj2&9Gb$O8oO<=;{hBP_oqs@Pv&0B#jWY~8nW{~%)1aRE@(wLQxK8nrM)dZqYVilV{We^?M(p5|Mr'
    'Omy$lNm1r}_Y?j)Ye+#ojc&q%4t;`V!v<&NxA&Ys=1$g9$?@xk7NqCjt%7eC<f~R80yUg-&APsT8#n(tbDRyzq5xQEbIerTpF=H>^sI$RO#>|-l3mQpv#J;l'
    'n1YH#2V6kVM>?G&{YE#^oL@&jzu)%Egwsfw&r^B&!E>k^qdJONJV-z=<%0meQc~QNcjris<-cXM!ikDFsgM-t_r{r7xqnKx#kK=LyUNS}s$o^S)q5toqUAvO'
    '57B$pZqpx&#~4-X+dh6YgJeeY$;4yLX?Z({AqW+8UnbB|GMR)|XeZjXDFve*8213rQ&Bp~>{fjWa<$fKrCrt8nKb{zRi+_{nvPS*6A$85!_AfGqOVfQ^{!~E'
    'l@v@ESnp1{{j){7e_idbjKq3ltL;H@+qDn*7>tz%AWayb$yQw}Nj#3wol`eod22WE%#4q6C!+)pf9G+m?NN(v2xzdNNp|iXmBdbsKg(-~{oQj0c`q1$>{fEO'
    '?X9}H>hty!lRq(#W>1S{4@8eM*H~=wwJO<bqcX;O`k@w-|Mmj!N_T}1HEQ2BB`g<?V@&-OOVzuJ_q;7)H+Jznp2fhvdX0fwtAnB#n&r_R9o6-Y==R)9-3!(y'
    'IUxKCqyha-VNAr~yrE+Su5+MX%^K8+^WCLl=d(%9&L@w<vrgSKYzilnbgVwoI|pLBF`VNVN-v4#ICp@Zi{g?};9hVrMG;M7j)yBVaW2bn496Jy^U>R(PvURe'
    'i=sF_nnU~46o65SPB3QVZ@kJ^#>6UtEj3htHsrkeL~(+qW6IYH8RMd`wY3lxO@OTc7a1dMMl(gF3c#sPp$vOzY8RSy;$*Us?(J&fFbzwxf!ozq#~s<8&HDUp'
    '*LmQ=el`HP6KA|n1E0+=ebQK`p*3G*Z*@3Y8az~kbryza<lG}f%N%T`Uxv*ud?7+@S6^uWx}60grdNY>2eYz3=*>#youb+eC3XOYUkR*^t$p-lb}$doZ;3^I'
    'qtms98HajJw4l$Vn^S9xYIq(AxY7|GgIC*<+HF(uQCIt<bDrBqw`NNl?YL{4V+vr6;vCn$v_xIZlL_q0DaI0JgJ`1Rb)UTuFZEs6@eSUH9H=hi*%CT6GQ-2?'
    'c%Gz;U_%BtlUwm7JKRzEM+)9fq4+OUvM|o6t21NYF-$Bpn_o<lf$*8kU>7dtv)LqxF;o{F(<S<&%F8^Pk1xgfxw28610OZUz3492W%foRX#@KO|8xtaw_?C1'
    'B|K)w-C$8IE6pI|qzI#{D4C#Jz2!Go9?nyAB&z)TJ7V{zmph8hU_iAF73E!(K1*px!wxLVVkM|k3XO$gx=gZR+ZR1)Ai^{ojK%@lcm8_T*;+qF(2AMuW7WQR'
    'V{{G-K7eMq!s?OH3J5ANn|h}iWMkazHZ49x0D{@A7$td8E{f6lUzrPz)M!?w@0hggK55`MgYF;DSYraXKccbC2h2A2CFk3hoNr%pzWo()zAbj>RSv{FN(*4_'
    'Q_gOroWIrE{>zm-idBWBN#X1!Yq{ajkKYR_D$vg#oaZa&*-s%{pzi?eI{^C*!2TkDPHH$Ocif5@=e!B!4AeAJt|N-9kY&zs$ShAeXHKV24|7S!BI$-fEULab'
    'y{?`$>)y$xyh*}dO|t8^Xm;4c7cY}y081fmzIe(Jh-(*2xr;*u17co2_p~?y8IANV9bL#HB2|E4l@zV(h3Ymsm}CWf2$SJW1vVc~leEQ+d6P1IegZ+dzBn$e'
    '!<-~_@_}+`@qPYB<{5mbbbrq2(1*zO_W)_o%thr*CuSD--;fl|;xfx~ikZd#?A&5sQl9-!OnGKzOuA#rv)V?wYUVTV_FW}FvxK!JdC+QQ8?us_3umS5WdA-H'
    '&>k!=*wRhJ$dYB^bJgr#_FOe)`Ff-iXIT|g-mE`1#aPw2-1i$BT`f<Tv%o$#RTyX18c(A_&bo0(gA;mfs-vNHh5?frsJ`bE2)#FsntC_;rjw@%PG}~;Z4v|T'
    '+lZB1cU53!Cozd`lETc;RpTcuU#|gl8*e<vBtCNS!rH}EUg9FvCC?m`;&=&#CuTD!KGFFflYL%aCKJk7(4rbDJ=6C*hi(QC6L>IBi(lt){9D}W_B^rESf=^>'
    ';Y5d4FAjb($;A%6kpMiWSudr5=sFpa$=xa1?QdDpHg-C>ZPBvo*DXxTiuu#gZBs@i^JHu_`$n{*;j!(sJ2f3d(ty3PFm$znG@Ho&MA1akVl<X_ZDDJCrbbRI'
    '{j%;<U)__t!VyC0H#c)aIk|u)%gzH-he%aXtGlEM;mtgi{<hJ0bj?UUpn>_Wzi#^=w5oIPZllZ3!rT4<b;7hdo2}^peFXfnXeN3)E1mUKr={~95N(X*v=@sF'
    '-uQMu;7v=+ZpK^cZVfgg%zCa<WplViuNg_z(|rLpA;$auu8pjAC|4`zreJo*?X$i3IIRKMvGLh~%S+GL-u)P@>g;yY$y|#rw(V4@Uh5*2Gn1v(&h5_R#BKXC'
    't={z*Zk-QRY)EbbUl~aM6Ar5W`DaayBL6O@Nxz&YZ8%T*<wWV16QzII6QwJiDP8NVl@Umb`Ffck-wbBAYg-)7wRRLo({6N)vK@cKb7@0yacf49Rbg&0n*3H!'
    'oLW#tgrUzxgyCq88KT2bBvXuU6O<Asl9Hh;HOM9tjx8yI=whIX?nV=ef5LL|6$xARlbuW!vmL(T;5`aQgY7jhi<WVm1FZ&_V1|BTR>Vm(qbEuyjOp$udyHJm'
    'u(Ywb_<mcLHPBm&zFwok(>dl%#ndaX!{!*yoT}Q|qLk<yg5Pg*k80DVcTSGz5L^y+Oa)7R#1X1I<(x$gv_1*E%XPj+g@wVis}6A06JRnj<h-h3+s~j>5Y1+B'
    'I&4{`Bq&8@RM0>^yE|J3zST({Gb+!Q@VP0q^EK#@*|nCs(R~&2J8GpPKz>Xz5JaRLA{OKg105Iiq!8U|y}0|t9!ra%n#q*q8<kJFEq~GHYS7q?VJMTfNJ)_v'
    'pOgQZzsPc1n$}P3^H4ynb5P|@>=eOP5p}9eV;QjX%`<gBtx4eM6nu&|&N6nF0k&dW4`db^jBh12MM0<WP1!}3A|}~50V14JG9B6G41*e(1Sm86BrdGzR*nSB'
    'W+V|56d*gv<7jx>RhQ@v#o_B0GQt|$qjd!+E8_er7T91hi|_(kHFZTcAvHe`?=ei)JOhx%D!vss2QssG5aDTC<Y+?*!o|=w8B!^yEtQFDU~X;&4s8;5+2CM@'
    'zgqb)Q<>_16YbmeERsF<H7~v=i`mMaOj|D^*$=`DE1f`?&}q((-Qw$Tf9!~#+f~?T*unpIO;koFxY(`amK|{uYObO<sH)A&OippTqrVeW_p>B&%=60!Wv}T*'
    'ubY-+WD4j;yJNnVur?}AMIaD!1jLiN>lI}YR~fZ8WNQF}e_dh&KyR=~f&$9f*ev6|ckFb&*ElD~dvY<KY6gT>(xN4?ZX5j0oEMZDQ?eszwX&F!ZOPJNkiWIw'
    '!?z}5Q~sRBfYYU@TAN#h0;=lzOnYR9&dBAeW^Bt=QporKpKt1)nHG^HNa9ECzSSXsTlDYOl70!umQZ(kz&j^#AxKmNNlH`-s$%=}Tu-{GYRP2wbRS0b2%WEh'
    'RYl|saK~uVZ9E6&M4R&+87v}7{xHm94sfPJRJ>Ukk{q5gKy4?e<<{1ue;1sIuebG>VFRT}chB51iQg5ffpsrjNz8WzD<Mh2Dpzzui*Q{5H5RAb_O`2~>Xgy~'
    'xa`T6;&@xc<IXxJoQ5)%td5><*Z|hm5eQRkKb9Z6bVdXdgTbz+GJ5ji6fK{7Kn|laD&n$WkV;(7$FnHMh+z5Rmm;gg@vP84LGT=AQz_76&NC&&<8S#ImNLo*'
    '^20F-Ackx|xa#ZQK%SMDpU$VVTNLcm*&;>YD#0zhGSn4ywSanYu>VGNVV~2ibLWWbf%q^f)Mvy9*2KSNDQ-!Xsv2x)pu>M&C6f|e{tSnZA1LGl)5F*Y){9i-'
    'L6Q&VphEz8T;plTkfXgj64xNP-himdFAcFfnU4F?91TBVN4ps3{d|1EDLzzkc=o}XnS#o!s--LG^3GKjP)2h9KBbm@!pT&*k|*I76d$THHIE78v(3lel=5R*'
    'b>ko^=p5~1+5Y1iL&AbUGr{1h+{JWv+;4od`~J=Q!|=uH!#+q!ruU2LNY59B1QKOoJ*FDavEx>sga+#x5*UE&#g?xVSihG1Uaoxjrlp6GefbAYjhkIvX@1Kl'
    'gY9w9dO}SopPG{fKti*C6&`36?L>15*OUb>N0VeG$tIUtdqqXpGLxe+|Dq*(u8tbWUJwLKS4e-6M3{C1{?kIQ??&*H3t!2?s;On=q1?9{E0#Ct+!&)n)YMD5'
    '!sar|li%>j-Ndns19f2`o)!{}#x|i`X&4{_mmp{^gs|a(K*MSzw2DxBRK35D=F9<yIA<;V=l;XS+B9wgV;90#ati}h=c{ee?cEPE+{nhVMtZ(Re8|H(cnS{w'
    'EJUAERyK2j1<%fESjHm8(yra@<6GYN7WE2a$*NZtSvD+hwiyv@zRju*8vmFY%SgyeKfK9B>iwqD{#zdy5OkG(n7Jm^2M?vuNC>HX?TA%Tj)z{?24q5}zm_AZ'
    'BK1NtBQFO;iZHp;md|k|p|ofnbeK*%Oi7DVtQgfR7E0n$h{<X}Kutg)2-POv;`~QC2d;g3n13H&?E;0jDR8hZ9ydlAY(6DHj?A!#QTBPwa}=i{G+SUp0&0?A'
    '(wZeVefVGZc%ENm?LgUFfC!XGgu%T4Z$?+=N%-7#RItP)G!CjbCXLSKg-|09*^I5UioOmB0}8E5SPb4p)3}&LXz4>s54{{V42pYa;y`v|yMnhP3lny7waq>f'
    'biX?3gm?u?R#f9NA@TVK2?F$@Zg(7^x3U<J;hfVR(v|jfqz69^CdQ$jMNqI&50okV0j1)K&gPA$SuzZZrSK-v^kNu=V>%V0fx-6Gp&TXKv|X<TV=_x-g$d<8'
    'F#0BdUa)a@5Ep~^vti&=6v@;YObw1A2&uy;I$hiVQFS=VCU__X;S)&AZL_2v21ao;r`9U%DaTGcvq=WCdjE#muF?7awAa6DhwceiVeTrLB&>f34C<$)wYXI*'
    'hUq1KHQVc}nL&rjQgjSHgajlRnlPm4p1rQy7{7|=Q%ekiU$%rWl7wx?1)amtgO%O~NrxL7Xbzps0QkH4^dhDkKV>~+e`pv+wt@-z5zL$rbEnoZcG=&0|Nf0H'
    'ZvHCyP0z>TLBGm5J<TmP4caW`BL5BfJMvLB7!|?N;nD<Fx6IO^NRzruDrDN1`%+lqzO?WNr|c;19tc)lY;_I$$Ayp$BC;kKbUF(R>^zbQ)C&<^pcl8)u7(Ko'
    'zNCbT^E_TWvg)Xq4CKK+gc+H#E|=n^mJz^@l!l8i91OfdCBI77A=AcpXHMDzOf(rUpE6(u@DNssMLwr%Dz~1rEvdt?X4(>%A#PHjzP(ESqRAj;i77-Ow7Gw)'
    'K_VvgnSj{bvj9_K2?)A-V<E1WLC|SK7c8<B#zN;*=#&XxVO~{!zB1dcV8}B93dE)bLM=uuI?7eK7`TjQ7$)+_1Dy*Rn&$<YYSLYgs^Xn+Tg36u&oa5Xr1l+w'
    '1&K3LFz;|aNG?D|)+W4F7N7J_`kSMS4((NB2;+JtKd)cDcwbb$(bfzWtqg+H(sVU%AX4EG59rNG8=$Hf#<Og2Su_lFV+kb8p~hpSfXAjSE!@CeZ5e53euUjx'
    '0PpDvz|c&appF|226GVeX>(rK$>noVi2?K3keuNhrwZItw-Sqs80IF%2nRGaljbKbU<bw%4(nWzlI8{2k5xxu;RY*urLLv$CUKP0r03*VHKlU{y>a*1GB|l?'
    'oW}Nx5-MBAa`~aPIQs7~N*C2Ja+L=X<gY;NUS?T=I4`f`IK`YEeDA%=&r!gW`8z}>d<ZL?V_-O)KsLEevuQ$N=;I}Q5!S$rCx#4@Je`ANf3^TldjUR{<DFYr'
    'wmj}Y%S^+HpD=hjLWH_{ULk%tTY=soXBpfM=wRh%ud++bm4sm=WuXL1=um>0jp}o0b@a0^Jj<>Ps&*C!>Bh3B2Ut+ZlOVE#6*0M(TikBNz6podS^mM)b}b|>'
    '=YAmKO+#1|QLkwn8^O07Q+lVu)h%b59(Di{sp(V7hQ5>rbj$q(Eq_~YyZ=yavtJhjQKQ?&NsPOF-GiU0f}jeDudLP-tea$FND1{;HlFKd_PDU1B=ZeJT-q#d'
    'w{-X#!($hiY;>z{qZ^?%5rLIf7}{i#U5Dl9NxylY`8<XWIKPZx^f|^-aI!u@t%~DUPx=^Pwljt8GZwE8eky>+M=3^Wl2H{(7BiIZfJbSGPQ1dH4BT{+6!%=('
    '>oI6PZc09DB%!@5%UQ9tzCOEUhhvtH*N54lSO>NOa^pJmFD}+8k>F)HojhU}9bviqq~Aq6b+fb^QIESUo)wGQ{S5XG0&SLK5{yRmojP}s0pom_Z5LzffPTo{'
    'L`ovN(l6<DJ7|-$T7q_uQcxg{bllzyWFd043k**_ce-}IbenC-LA5(rnd}Krjaoa$O?QeX^<l$455!LeH2@TZC5&DYRhdI5f<I<kW)J@?j|WRt(7|Pf1hNg1'
    'N}9>EN3-5D{&g|JUuR3%dz>O^hBti932(>+>c0)+(I^?P(5vq9yY~|Itx>disWzAM5ll;htOt)H0l#87tXXWWobTHE(%=oFPur^9%8z6CZQ5QhQ)5NlJU#`M'
    'y_ww`&dZ>fO_H+pxbxVKMrLEI$HRn{-!Gn}KftqeH%G!3hrmNP1pavA{TF-oeeBs^ETH$YfHr}6j9b`*!ucPYqezK;KmPjh7whR4>**Kk>Hi|@>D_HKe+i*>'
    'nQ%H!-2kYElS>usN;2I=>)`91cmIZ>>59VXFfImpGLsZ`l%T{T;v15Dko+UDi|048$-B6$de3!iiDE=fWsZ?0#(ZBMzuCVdhyOQ>-1;-)wvwSGll1NpE``hB'
    'Az;*F$a)+1iQj@Dc|bKraB0F&8W>t;Dh9clLMH1PI+-O5t*%E&1#PU=sJjSNs1Dt??wz)T2a$R<|3y+yC95nFc|hxX2$@G}2wpBkn$c(JQ6?WjD{q3KI#IG-'
    '47*Z8yo07?i_t3*tf6P!tA4Y3M0Za7F(D#)77UYvwEm7!uKJ9ixBfD!u~nu@8N16`y_@y?vxDmH$buwBKUZ=$AXVwQ{NTEG651J(X}GDH3PiW3zL3eC$}Z<A'
    'chOmz*nxDOF*&ZZ>#Q6E@x{&h#m(xtSvMZe&w9rgnwrn<UlkT}*ASR5;VyrCxXZsTs^wv=NCuN<pC%@ZH+1_ikqiH(kqdV<#og08ty!?u4B<YyiWN;o?45~k'
    'W`Geg$rKYJdSz%-7RG-UH+<#1Zn6Y6+5z#_(_xkLzEyc6eMZ1t<n;M`fN>vHLIsw+X7buqx<7J=Yk@8$oW%pY;dUrf0P($yTm!_A%WSCabqD%34%u>7RJ{=`'
    ')QYym4y8vUPYYViF4SolcqBVxtrfpDu-lZ2Vi(hK6frhLnK=raVxg{yCVprQn<Qdb<B`0O6`qvz+n9Tb2Sm4))}6eDY4;ejAk@0naxVKaN*oe2zEP3VMFQXE'
    'Yyd`~ee(n6o(bOUyxaYM?~mN&Z8jZarCsPG0yhhuX{&VO=)(T<XCWIgX_)<|&u-v<A)VuWkO8-#g^qxK+U~e1qsIyZJf!Z?09%-R4KuK{mzMugpTmEJ2jW-3'
    'OG46^Ld;Vc_)|UK(7Uq-L5dp3Vx-)$&oUChe7>P~57K8MT_Um1>@(3{nuXlUMO5~N?4K<*gNmCiabxq#WAl#b1M+6PWV>a{$VC&NEiI5Atb8r@P0O~)6SjJc'
    'B<dJF$23UNg>IhgH~9$uw~zVs(85RJ8T(jtFw6yoDNvfNEE=*aW>}3GUl>(XP==8y08`&ooLj+F(NAXTQgYl3)V2UED&GzFtGpxzS>*<lX|7asRl{EKn9%ar'
    'SWh^Rf=Ue{mIXx%^UTrtx}vgea!bsDJJh1Q3Aj+EPm6g+W=1NFc!#IU?$HdJW3O#kytCL-tTryx*77V<LsvJ2>-~!>bx10q6oQ1d!DK#+mnTOq<`-RsQZmXX'
    'Bgz$4GJOHA;$R$f#ACL+?mX5%Up!VWIPN@^H~Xw6pFc?Pqs6r!qg3)3qnIAUg4JR!MNQDC0E56%^8^ho=?Z<v`I7TXCnXC?nwX>}iX;N9Uf1OW3SQN+04L-n'
    'T-!nm%I`4CWRZxY!KE<sMk+x;QUwF8h(!B5<tsInUIGo8m-3?P(lHwyA)fddA>HC-)%AYlCYH3dt#9GRyXIRi&rxAfJGJz21WMXk?*{8d4VFg3iV!5G0?J&r'
    'W@Yqnun^-h(FZ^RM&CUU4iMC5p#T9iVt_J-W}vOK6hzv9p$w)y8~IRSG>zc+Dy>nwb}8hTEY~_Ou~j$t5Z{9K;8w!@bAj66-W{md%XYPPth`6zBu?f2xM;_-'
    '1$deGRjo-7!7%@vvQ^_%X2ND03l^Z<==B|(YW!`xo8!1dz<MiBhBr(4n<w#AjH;4iU07d?ldCwTw73bVU^ju-BgTWDa3h6jAp1|0nwuF+cs1@4x7-}|v>9IC'
    'g60w4Rb@$w34zp-4Uhp3nT(uG(u7Mwg2Cd5yG_{!gu6V0tZ)CU$Elbd-e}VpIc|gr$1VN~Joh5Ax?xBKYTGua7<WL9kDz-as5lx}QTmbQ5?BM*@r05G`dP6|'
    'cpM=&zDkC)O-V{zWkbgcUR&r3liLc;nzD@1w9%OZo^1<Won$$S^bY;SL;3lwD+mBjm}J-Z1go#hW^5GJ+?+^VAUMgTX30%F38`oSj1g6H&N!^BcX|e58`Vq^'
    'RGkgxg>YC2Ui(U$`bih_o1O*rPUtIy(4}&xt{7x^m!51ib%ZAyn<XlaZ%Tt^-O{{QBtvgEs}<lQ`gd7rMqnXLM1yh;EXbyHHZ`{zDOcqWBgYyZlL0=1PHT_>'
    'fd)C9wY@sgF2z{j6-hctd6{SPF)yj}iV>e1uAU(WWTAYK=;j0Jc}@J4)*K!N&^3s_g%W|!^}0D1(WJ<#1r?PJ^bX!12Yk)g4lc9#1T%V*Wd#bMWkvHOa?}@`'
    '%$rxJHl4~cNwESiS36pDKZD%iPGA+AIn8+bDPJG!mbAZG+jLE)=xFffrYKk83a~7x!VgSwtZjQ6KT&1R?7O{8b;npE8=anw5`9a?aZ8W9T`%$qZ)~rUP==*-'
    '>7>P_jn;rQ^lIoZTC5P%_7*c+Rj->$ss$A5k{#H8qICOYBX~+TR}T57b-{*S?E0E0MaM0##71e(L%u^{+pZ9BL`n6%<7Td=L@yYkri+lbBcG=OGc~77c*o-|'
    '<h3Big-k)kq3TzKe6$;F+zK?3-0};#HcfrKO2uCln%p85Q`HYfu}^hl%F*G2nNzj|`A;u%3>1+M6Z4^FZLzK#eF#15az&kiyY<<s$E6GAk5QH$y0^>J5P*Vz'
    'p|FQ>-d!PCu53Jv3h?i``8?QO@z>r3{yg5<!0>*O(eQr(`|*y}Nx5sLu~-M~o7Q1A3YXWJz`IL9FDu0`n*tw12{)VP19{gB-BxINLNJ^BfMnS@3;h6}SnXAU'
    '&ng-VY8D>~SpFZR;jUX|e2WX)?AQ<BD4((Hy}U1`&_j?i%3?!yhi!jk4u|1N;XC1E^VPkx&E`XCVa5q$os$FhQS)GJfb2O!TWwh2Cp#nKu2VNY4?j~`^;@4`'
    'z&n_@JVqP+tx|I|2~;X*F~3m7Q0{L@OJPSyGavjQvzh#BrfqkO$NPAVFrD)$8_dXo(D&s?oe!0unrV57rT2fCp_kX%!&rLd)uNwc>0MpkxNFB3SZD9r@49t%'
    'k!iASbzK5~B-1}oI>{&~Z}5J<r50MQC21%u@&P8$4)A6zAQ235n6=uCVjD=+vdThrrn#}t&~P47%UPi5+z<U1_d|a<F7Jwdu5usrL(RV``YUB0Hp8J2%&n~H'
    'wXI&ea%sM>!v5RWWWh=c`(AMxCLOg=w0m-9UkZg5I}SGZD)?nK<~RfV8SCG(RR6q~&eYFwGO_~@JnaS4r(p{(y77p|7JEWBY}u|U8WI@K{GZo(Xm3g9Er!_8'
    '`3KOF{&g;yY8u<(>)LBT&@x30ZT5?X@+vL|m*lHK*_Tc!r=VeswM3`{VDB1Gl5jN1KsHb|O@Lz<_AG}@&A@C&RlJhX46-FDEYdBflzQpcIbJY6l&*-xa+A~5'
    'qU1ojiW0i4Qu$R3VY7<17^sUP(PK|mTnTb;nH6ywh<)kkp!f03AdY1qMm4$IP3N{h6pr=TRUQYog!enj#0r5PsfEanY4xsZMU(63wvaIKsz(V05JR}dyJ@d<'
    '&JOA@YC{4+qI^rQE6`faNh3_n;Rwane7r!Hm0_s2+!)aS=2p(CH|Eh1MbD4VJC-vb9tO5j!)k;UFOj>D(@c_?dH6e4K_bIC@!5X7VkK7>&2p2o%bLcAa(Y#W'
    'Bz2Qg!l>Ib^+`o}e7aNl3oJjG&pBo)=EyiuIi~~Px;=9^OX>rMsn?e@+fd&Swp3MXULB<J+%~(l%r1E0SSx6e!^`ZE)%fGWfvgE=AvmLBRv%UZSM%SDB*}g6'
    'W&ZX1mgr2UDGOJnPYBZ4!n<J{2=ZDg&eB$E-8O<zHI@aMtMS%kRZAwaF~eMEcB4&qkjFU&nIFEk^%ezyoAm*}2A2RRW&5VZO&R>E4z_ExxX}f*u<gdSDw~~t'
    '-)yw=H|&C7L3#;@Caa&1KU6(iSXxqi)rkgIi2~c8f;H8W;2SiDMxTq-n)P-;9kVU#ZIjdHFpegBM~AO>-YF~Qik8c<l241e)1?NqLp^FRm5|U{X&68nA5YgR'
    'ncEP#D~X$>Z2F}|@1RUZ>g9EO0dGG!GBXFTvaD}Q`_lFRLPa$$_f{-Sv5{pZ-RjKIj8Ghe_v>i#p(TA8p1TzZJl?RJ$RNlqf9xc%N~H;q0R}MA!t(8t-dR8g'
    'T3T(snasx5R&zXQHE4B@yrpaKXsC_46Gz5_!}fSVcQOZfLu2p_{?j}2mt|3v3iqCNL=P6OQ*a#3iz12AFwK%e1J_pz>T>*9!^4Z7H(XdVC{y6lL&vq*_6D^5'
    'q}FmC-MX!AI%Sa1DnL&*I^x@>XEnqWF**#wi^)8%Ks|59UjI($>)#cB{bdO3-vxvH)lt}Ag2VnoBw~|Lz^x9ej}l#M<Ea5VHUbfj;plh-tUB>IO7a4aA?tfw'
    '9e^}Vd|9i4-0wMM<Y#nIVxJ!;MTQa8Q7V6o<224$vJXA4bU93t8%OiddF1bB{zj|ywfPi@qfh_#h*~zDn7_|_Bg14lktVQunYfI`=Gp-0o4!IbYq9m+A|SsH'
    'Dq`=gGBLbN?l3d`B~#P?m&{H7b0?=-{+{)0%uj!%3F<GMp}ud5BDDYEX<sOX48JrD<5Dnz@1Z@<&uhz8B1Y3nUwkF5#Mh$786<FfqHirgPOsNK!?w8&=dYoU'
    '()98Ts+{J8KN839U%uZ`6}S;fuXas`sk?)0h}MBB@J!kORFblo_A-~r%|T0$0<OM>__sLEidG*H*=ws?gs}%GX=f8Uqs$?USX)ICUgSjKX+|UuYHvIm#e;G|'
    'gC{+VZrnJ7hI@CkXgw0$CI5*xSu_NBMOiYImVabIKf-IIw?yj_l<<xiw7(OB%P2*}XF0{*ym)>1owN<mRBXvmO?`o}q4ojWMzKQaY$S5j)HRf^N%F@lXOk(}'
    '=4HjUx_3IA1^Tk(b<uzN)TU$auwG{B0%bPH5C@W$hApA573Gi15TiS3TnrncUeb6gbKCABo{2<f=PaUNy3LJ^>MP=!Z;J(x@*?kzEpsi?iAt!d_9}BzG+Tip'
    '_GQALVB2+}3)k+;ty*kbh;ufLZdx$4cKxonY}bolHy6LIUlm?W_aK|yzKuS_FNnA_QUMHgLjStvtHfWtKYY1&7#{Audb78C9QHPPZ5PMxz32u;&%_UPWLQOd'
    'wIU8`md_OTL>n&$oT8)5_`dB)JVHM_84}TH?wpj&W|frz|0F<Yn6d%TX!XiLDM9sByk1gK&jtTDfevh8C%(38b*P0cYNqxn2!gZatC7~zYubU!ShDg$v{bB!'
    'S7d98RallS#Cc3K=H_I(;(@Ek^7AwyPYK>oZtt60?$lC*vaw+*5eK&<WHD-N*d7Io;HGx+4Q?q8nLrQB&VMg@PdrLnHjT$oh{lzAcj3(}0fq?MD*be(*JB7<'
    '72;DrM7amU;!X0wd=lj$XrLIbE+w{>gxxVm#!I-&<Yn{=4RyK!?pMfnE=Zg<`CScZu(P-b&Zwt|+1UmFmE^WU331Y$qB=cG&;v|A&Kj##vfiR1v&JHZPG{!a'
    ')*FHxoe}F)?IR%T0E;k&58Es3ukPw%Qh|Cce(s1?tKtBfxxj1B+jjdw*h5`$_)GAI{?`AHMcx0yS9SmX%eue9y6%7O!tSq9<ep2rzrxz?ueP`!etGxZ_1*tN'
    '7P!4cdCS+@r(`s%)qS+;GIi8fFInE5HGJj@rtW;+PTKD+gvr|ME7X{dr3<I}zIdtHfNGknJ=35IqB3*%`b9_l`13CQ-u?0C4#o`3C<aqi@(G-M>0LVEm!Lpi'
    'jPrgzzL46KRQOn#Ix(%33&vR+VnyLih5Mf9@o<RlhR5Ub@s?0Z$zv|wd#p?m`c+@Qf`#+gz#zOgp3nxxdM&({zdqGGSgHf}InO`}cH8vhyg7Qge|Se1j*h)c'
    'CNXtV9;+e2pOhjD0AgG0swZozFuxjfKui9(If75QJjp>bnyH{Y_McGJ8jk0~q*O(#eddXWx?7GKTFacNW=N@m^td~G8g9EK?RFcBQl-@$Id@OhWGBG6J3VOJ'
    '-Ti=zX}8f}Ry*LLiu)YYf4~`(QJNXS!P-ZrUSRGSJONX6%Vs7`j^h#M`rShKe$VQ%b09~D$8k~Kckxw6-Bw*=zVlvA?)5t#$uhxtV-5cuGMC+N;SgT_q9?=t'
    'oRzh7Y19^-?jbC(`z@N?<HMud^z#=DhhC={=BkJP9&!y?Ww{kLf5MMD5Svl`PDCaN?D5A>kAwfu-n+K9aU6-le`O+{91dh)NPv=TaSdmwo6SVGP_&hV;z9#p'
    '2u=iGU<@D<_Ud`$t87QMd=lI7HMSGqHXF;{bvBVDUp~D5WRE~fpK^Y}TbJ&h?wJ7yQj{%w>CIA@neOW9>guZM>gs9^H^d_{Hm0w*+sbF0M?tgJ?R)guT5n5v'
    'z8;I6+7Y%(Wxvve|HqgKYV2+mW06Vo?XM5B))@}VZ8n3^p<(G5yfgLq97vyGf)K+(r)nwH)ON(sX{R-5%&V07S4%5ci<iQVD#Mjh`zS(<G*9I;#eav)Mu}gd'
    '2nP+-(m(PjOc9a;=JKJr6I(W!eGG2E`<BGC-hDggG&P2Entt4l8IQ;4qNcWP{_FUEU!_6l)i!`22Z1G_eFw5Y?*P}+wlrn0aAE7#!Xkc}IWl{2`ao&==r0c('
    'JU;he=5`8`2q~5)xGzzng}N<1sU2^p+YBG+dLEc+6W_g{p%I>v_zw{^`}h6OC5`b_q*E_pygigb?_fOmgHli<yCHL+oA&!BtUB#p%qws^#Mf4fF#NCKhG#~H'
    'h^5zu34PxfSW#g;5qWE3@)ks;B2=QX>Ze})@IZ>B;XMSz;I6MHZAX{5vA*eL<?ErH(JN?Q64<vr4vuanZx0wEm`4=O>k4Qtz)Uu!`kPuvG2@ipQ?ul`#MmrS'
    'M*NsSn;Nj45)+E;c!)*1Y(y7xvfN!skn>|m$uSJvI<@2A?Ci1G_&HnU#nl?d<8-Z#b&&p{`?NwU5p>hWn2kk%+HPw}IylhkF>LkRl=YZ*E-?Xgajl~o*N*v#'
    '!me%oxUC<5aQgB4s>SXc74*GRC)-NmHYtgF_T0OzBi7XsW_lU93z5DVVvfUnax(--z7?AxdfRh>*x=T8Lz73jRE>g(`Yj^1?GW2)ZMpTD@>_R_-GX-1aW(mN'
    'xHl|THlZ6OO3@*ZSdElL8odhIexh(;ozl&g<E^yZW)(zDBg6<Hx7~Jowk(p#TJDM;7IXQ5_1j*{1J;%=vlw0!)zRlxivGUMKDxNR7Bo#Qhsdk4_*?hrrD;fA'
    'LG8{LwiVR3tDv@Mw|?st)MgReR#4jtYFj~VE2u40P^sHcg>4nJ?PuHGhidaRl1=#KRkwGews)hpcccDiI~v)1KOA7rht3JV>Xp=bfbTRNqAEoSRFP6QegHvB'
    'lFB7;-Jn0q;r!<iPwH3rO0%djEEpwyEYbOH8)VoLop{Jv@N-%G#4L=Z5Rqrd^x8m1q_T+@w07dj0c)Jj%Oh+CtI@IPEyGLri$~3|1XLktTv9$673fqt3>p2r'
    '8#DjJ07l=J65qw3%|{I{n93Cs`$w9UFO;JsX~lfu-Ynk7-s6)Id=^lnf}p;UwjhnjJ2_E1pQxY8VLFNAXl9H<Zpe`;PeIDi=k&%g9@&~m0kAwOfV=<@%TOkQ'
    'yf{~rEofuME1H-ju05t$u%mMaXG@1?jvkzz&7?Py$d*8jaf$?{hVf)3P1t|z=!v=MqjPfPZ2Ct$iGMP&^RFQ3)Yv96Dr$J0?d%Wgfx-yi?LT(-*lcOv1G5EC'
    'vMK%HAT&3OqR**n71MnbMTDA>@xw{NV4eXpxY6x+c7ZydPx7_M+-X|rn2T=z!L9Hm;c5cUOwfs@+h(EPnj<JWeq~@gEPc8xs1l}&16C;~NMwdAEMc6_!WW#I'
    'jteaesO$g-bP}L-$JG#>vFQUI*m7nwtBAz02LmLX^)SyE-P+<IKiO!QCi72^KX6OShTVlxAa2sg&n59)-^AAfWu=Kfz{pL>5_Fqw@DY4E8}jRXzisPnv-vyR'
    'pEtBtVOwo&tF7Cqwl=DyO5!So62=-1*Do8@RvQ4lVYP*&bJ$3(+zE5oK<;c|BO4S$mDgyB%hU%Zz({EssXs{8X!UsUwBelzFqzZVaHH;5tXtjeSqz5uk*xO!'
    '3}=0q$}fD^I|u9OyK^ZAWMaDl2`Mn5e?zz$HnGLIL8nMeylYU|Z4f;V^$XODv(T*a{m^fQ9k*Gb1Nd2CkI~<FR_wOxhn_VD3qVqz_TtzK-FqA*_H?b|GlvBT'
    '>j8lN7-fgVmd}l4%xk5pc+_gi^J_~Y--u%-O9cF=kad4i6q`!9i(Ij+LBMgYIy{leu<G2Pj&RdkE-^mrsG;0K70<8R{8l)9D1px;=2x98m>!-|)K{arc%5E9'
    '!MTSbPX-K@9l^~;7$)2oNw&$1;Alg5)D+3B80+i25g$t`zlnj|{21P;!xtMog+x|^XwF#lyA8H8z;b<z1{0<>O0H4~exXz?mCo6w>-DiDU!RNf#bZgvIF~_N'
    'J1!b$gKo1rSSW6R0Q`XqwUHTdvkdrc%8h{wp56KFCC{C*<hi9FvxGr(rv#a89b((7yW6w6`<9n>+p%Zcv1i|-*fT?2J~VyLo_n_y<_}Y0-XX_^whpl$p2FN('
    '9hna8j>&kn70Tzk64M(OttW6XUfq_hccyHOFZM@~Kck6fefMS~LUGsLJsWVEZLb-&*9^CD%>ZU)P5KGxxAOcX7}qfzwUJbJMD^%}a1&)MV0Cjw3Lol(>1`4b'
    'fu=yZ%UXTy)BZ~zUcd7E%?lS+-~GGcdDHH<2{_05gmcdTk8La?JFMaN#J1cuB=90^Tc?Edw$hxnBYLS1nPCyz(i0Z#u*tSYk^uuky9LzRt%5YdP6W_s4{lRn'
    '(=J>dXc`nY8bDK6H<o>b)#hE+jXyo!f9}r~Opc1?^j|48+!gY98k}wAv{@3B?j=eKhRNv@DQ3vfg@X~Ntv7=VmS^<tw<hTJsGfX^5ql-|@+YbrBU&%DWG|(p'
    'K7bL9m<rvtSD*^O2U)0#lAe4P9T_ybwOa<I4ZYg@-XJ)GGrsyu@}hz}$gd*z`t1RNJCl{+;nD2X`7UeO#0zqub*4+k417PlINw1Y$}Mx_B-a&pUY_`^K7_8>'
    '`&x>5v(?PE;6kCDM*G|Jo7{e5elebMUK#J}UXrjVleKTg#OKOeVFPba+OFdr7=B@cgtv+1zcx*8&9k~aQ}1wR=k}!iAx_%u8N01px2NeHKTU5f%x8P1-r<oq'
    '+mrSOGimQ&y{g}Arp>jHGUl`Mq*TQlJN$Y9M&{eRL~N7`#5S@Ilhz^L%+Tn&v>|{g05*GR3r{_J%o)R3xAogA(6L0drS_e6g^R*cFUT*q_zIWxwkq(sI8RCe'
    '{w!oIx7qRCy6=X3BqtST0fygf1V+>&1w_2s2$b994qR&EchRd&UrVD2%EAvyQtC;qzSj~mnVoBsl5r@Y&FA5|LuKR^g{^LlY}`ne`why#&B(k#m?bt`2iLp3'
    'n;ZkAT6~jZG`Y5GTdw_}<yw?x@V3jdt-9&zhaxk#EUU?PA#yjL?d1C*qT~G%RgF{acesGx+<Er_dSLgq9{8W92Y#TE`VJJhce)t;4#eS5SvP{LycKD;XIt8B'
    'OS{cwzTTF3+cNKaDf4if_Io7u5^%o1g6+2KEv`35Wf-x{xGOf2+_2?UI;A%53hPrVWiR<q$`<mznM9gn%vr3p1CJ%Z46l9Q+%mR-iD(ILX(&!OTehUqkw{Dj'
    '9P-reW9cceBg6aNd}W88!z-DWG8zqyI#X5(xoDV%5YfurZ?rBr@YD|aKl<JO?3o*1|E~Y^2WxLU)qnHr{*#ybU%q?u!ll)V&)t0Gn;Tz$vG$h_`!7AacJ1~4'
    '<u~9>|LTYRFaCD@o3~dlKHk6l)y;o@ZS~WaZ+!FSjrT6Defe*zFMs`i&p#oj!gRY9PQ`OG+d4v-DT`azoBqd_7yDoRaqaoPUB7bGviE1L2ea16jJ5jgtCqbl'
    'YsnYXP_#^!RHY+309RoD##3u=US56o0;6u!G0y_p2|5Ch)l0wapZ{|8)i3&AB3d`Ey>{dB`~6E7*RFoEdj2E+s&#mlcueoJL4AuU)PL#47_kAgpy-S5^gsHI'
    'qN_qzAF;cv)jyqIedhx)Y}ft@{o*hFqR4w)_3;o?v1m<B>1PELEu)u}CAZo1>Y+>>l(kfbpeXA)7x(H(_`6K_+UP{@%Xq9NyRnvxLqnV=>0!)5&WVyv?MP)='
    'CPn|7r*FP@ZS}QJtw~~;T}Z^p6uHY<d+#!^a{sH}^`CoznWX>Qi)-(E0zVWP#o3+vi;U#dfOJR^*A33H)TYokTVqg}BrId?WbpD3ORV2Iq(LninVHg>5~-bb'
    'TJZxecUjlJ`eyCrkELCr&tWbRm+*&TsqEQyNzLc95SR7or&*&AlBl6bq$VZ(NCH*KenCpy3b!z66);*WXm_l|dVuvXh1GXEqtxs$W5RwY6Ya+3_tq}|{^ncf'
    'Z+!STLU`}r`%ga&)#J15K79xcy<b8*#o&G`r!ws6<vKwNPnF}q+N$2TRWQwnQaR{!0*r>E^kpdpV+4sn8?`57A60(AAzfGs+Ww>QKEw_WV!67v*fILR6_cH4'
    'iy5!_5HFb02?1QMt~w#^guvtWvS?QJJR4QXYBX5D;l26Z)&A8>q@(+P|1fJ^zw&(6>R<XCfBhz-k%;)x;NSkIUtGWX;_7o>05>pWp>jR{()BBU?q7Wwgc9>g'
    'x($eS{Jb8n7ql@Q0A~e2`4CIIEvvFo3y++LPIOy{3;YvZkI?|_UmOEF88i8H#xesSA~KHK0^_=BN6id2U0H4c%*q+t0Fpj%K&Ka8ff>gX8VFVPiDdG|dSysq'
    ')5MiR7*BbZQLi9`a<D?1AEpQ<Gjp!ns(bS?DQ)y;A;sogefNvi7cQ^9_-||PKF&g-|Ng71FQS-1lkJs@tM5HC*o;I35?18(s~6EwBamrUWMnZ8veX85z*0!{'
    '-+l>9Nz3LrLi0n+4W1X7RD%uH$|z+J??|8*&;caym4XyHfufbr*HDh^zURK~Kl|CuxBqbS&A+WZfwnCp#NV4iv-GIf4#d${2cGfK#*0+2Z$|oM%2vtcoo2V;'
    ')fuY^ll?w?V)YMi2_&EX7zhas16^-?^K$>qKdip;SDddg<`m2{K{=ZV55Bkuue#A5WfX0%)bN{XYQS$>qBTDSAQmzXE~Dl8Ummg+rQMaGrB{Z+&L>L`fgLX`'
    'x=BrBxfkMCwKz%}%^*B}%(JS5Ct#5XgAAGY%mTiGDK|BkH?^t9c$u~Jr@<3Zu{WQC7n=UJ7kS`s*Lp6td`p?r3~7`(tt=p*mSqp7)`q9@aA7fw*dnQbvDPz4'
    '8Xt>6hyY7sx401UQcIKl^e^%!)SKe+S%Nf0Fc!>OL}iu^^Fu2L?lxM>8ek&!2K`s>m1=G`41Kp*D%ZR1boM^uwZSZ<W9P-)xO%8H+$oEKQUR&FGcl68;?pk#'
    'G09HAW%8?^u73R9+UL)IH;xCbSIYCiNWFWd|Nck)&n`i6CE-^;`BVRc3#+euX22j)ST>PDuMeC9JlenTO#knfH{fMRk|ybK7~olewej_D-q!krNmmGAr8e{q'
    '|JW+TwMNPsLa&id34PU&#Gbz(W3>&lCxz!sF4KMsek;;nQ-&LWwbiG%gf1qzV_X$ua#NxW{Fsu;U$bYyXyaCk6Y<vw)p(1@m$ZHoFrd!vc$wsMJJr;*$7b|~'
    'ZonzIisnI@8bKG=wWoo()?Rsi_0<cjZ~diz@oN8_cW}-nMjT$QJ@q%>THPrDpm<sEebPu~dUzoqZhZ9Ajf+qAFTNV{PX6P|XO%;WyUG(6-oX2ReEFOKI^q+8'
    '3!{c33E)Lf2>?#}Coir2^<8#UvIBD>35UCDfDzps6-=BHW73euRhNkjV|Nlpx-=B^BWcL>fsb!lQOnUt)zt}BF-A0gHGu{7EIYqWsR6<_?^v<PBqSeNHO2Im'
    'm=I;^cBy@$fVB_p#>cSgIYYb_94mExrq~zRxZyC+l4aYKtyQh9NheJgJ^NYGG)fooX{JRCLtWg*IZ(>{h^sW*b~&(|`nUdnyu#gQm)7-bAKV0Oef4GK>r6GE'
    '^he_~4e2l9mMP)_;h>Xk?snu3KpO2w*-$eml;E-rZ?DNFMZeNo)cd-L2C=b&C+0LF>;a}=*_K9xjc)eE5K>+b{+=;_ie3(encBw)q?m65ohDhmA$_BK-`t5K'
    '@al(nB$AD$J1;7z41FyPN@aN~2YFSOvsxPzr1R_6ErzLwUIWjuRZ4ZY?A23R*vc@VZ#AB0)JUE5@_Hc*iMrpKP=^^cuPEE=+GJEO<iuakqE{wlaZrQ|+248q'
    'kNG)q1$pfI0MmZ3d7oxj74!MD60GZX4M@*@i_s4;+71aDGO0dV<j?`1HO4I^O|A+cptq`iH<SyZ$vZ!s(o7{`b_0<x-;5$?*{&4Uu+>@ez~=3^b*tHJl%ZpR'
    '*j@~A&ji1Uemc56x`hIp!5;EDl_ihSA(F`J;%l^TZFSmR)VG+;lDh0w*<l<<YWs+1nkKl*IvluFCN7Qb_w)4UUDh-j{fg;<aj8$HFiFeu2?7+1Pn#M%M8BOQ'
    '?h601)23ED*XMplN=^-Im~_+tN~R_?86M4=US-(fsq`ADv8{KB6wQ`fuWNk(CA0$nziuD6vTe&y)XeC8NvS$wpHgb3_U-#{-gO4KC-lB2Mm}Z?@~ZYN=&d*='
    'M5bE-&-Bh@FheboQs#2FqZSr)<)JVrWBvXC>mYV@Z08!-&NXny$=)BTXGN{4zV8{PbUCs~sTKpV8DVpHGot49$&2qw3Vh468D{H_?2ZoD+hKbg3_qAq&Dwa?'
    'tAPnlwm1xZZ^zUQ5t&;#3x=xrjS;)*fihX)okGo)ddHU)s7Kn>In|_S->~9Caq$7Y*}hxXKVb}BCzJ&m-FnAA>$ZJZbC#N5n5x1A^Q<2NqK;;^E_Rz0MBc4S'
    '5batSeah;NdKOZ5S-CCv&$??JNaX(!J0YQUE08vX)4Dyhx*fl6Nn)*w8OkSeVu8fywEdNW4Ns*N+QljLvbYTFxEW(NdcRw*bkS8nYs7ZYA+G^%ATkik?koqe'
    'nry)Q;}PTc8E3#2GPS98AI<l@l=p;Hv|Bj~tnN0ghUbRewg;_toWYxu%_?1k0JTyucamF#RY4iP5rMBzn4f2ickaZH!ij~fH9sLpM(-Bn?C+*lhM$!nw3)V9'
    'D+kXZNMU|PV8)BAWt3O2<t5`Sry6u@*-}=%sHzw8lQWmeM4=X-VwPa;_*GkTbDAUlfY<cea-gxih2=4bAGe%v!_)0p6wNqbT$4M~gU4kB23m{_3@P1ohRsss'
    'qGcO9jK+f=slBqY85kOXUhftF7~!XC$lYemvvWEn#>cG!81i|?01~~Hv5^?n*kdqAd>ShZbyRo5tcb5m9NlcMte{RpTx}g!Yo|%EJTCWoJymva(ZR>$Ht`=c'
    'n@MYPz<A1FPrN^dD4|6ORToK4DXEnu#vCN3jvqLL-d7yRvQ#<N17d?4#1vk2SMH*?93r1oTf-QnV6r13r5CS>`Bb&aJ!=YO^weoJ4Nu3Y9f$q1to1Vl5C2#`'
    'J9L?DWOxh_c%l^J!H9I5Iaay211k!oh(4daD<U|ASgr1ZIIn@Nc-9yI(P`0?ynDvZO*p%OrBIx88@3QkEJF;~cvOmC7c%DXGej+}7l*z&(y$|i_D4gH@%KQ1'
    '>Igg_Dfdta)bX2Ec*a|XUmIeXYK&b%q&osKHO-yFCO}yo3DIYGo5C3Fv*-j)T5vY1K~jhF`A9G3mCB-?7A=DjswN8|8bxTpfge=D<Og^gF@qY(k{lX49*Gh+'
    'Wte4(m@FK4Fbq2FpygNC>6NbgT#aE=1uJM<ZZmpRtfA%-!<yl(o~R%m2gzDNv);2#6MCm{l)AuZ4!`J~rp=hsge`saYd~E1z5yM3JMF4eu7xhO*f_t<{A(TH'
    'tnX8?eD4v3?Q#&*hZ!1%JS1KpNfs`0j{zradCDY2)+)jOa8jgNRnfUdiRK{XGLn(GVj7vT6HGC9>s$9R**!?R#)^p_<5Ifw^Kj`MO<3j;qWMN9Z~2y|u-5Rv'
    'S$q2Qap1-Flyw};w}uC{u5~c%_zm71I|Ot8h$j222hu<u&~!w2`t<(*flxMz@^o%V0`kyd478lpJn06h-aA9OY>SW$iI6R{7hA!y*S0rgI5LZ6wpLEH(3Et$'
    'w^1~S7IJm}jAz&sF`<-)F4+0ISN_G8-IUz9=(W9O#hbF0I-OQHH9k&`Ha_y|t>DS@n&(u4#`q&$FC=+1UYMBNQ`kH1hdIE$?d9B@M1v+uHK$L@ZP~RgyS8Q5'
    'Fxl1iDn7>G!3yY>WEZ-BwnBcfm%g;l`e;4x4tIK`L@ulmOZP@BO>Scj3mT<(Gw^|IRF<XddKHEV-&vv#2rW4sH?hJCM_%GZB#cSNh+0BQnl#x0aTzsj+46wV'
    '{DiY7Ywcc$7woNHP~vCBOXhIf$Z7dmOZ?D7H=>c_3Wh)=A403{R{-!?CC{i%Y={A!dv<57yi?e{uz6WGFi$|r0n2>W8v9u!U;)>#1Je}7$_lDg5!UI98DI>e'
    'W4y+Fu~&E^f9zyCXx7XR5{`H&#0uXG-fjAH(^MRIHUT?D(jpg84g!rV2C}^*vb{mb_Dp0oSSIf>u-H2dn)yWsC)yd(|6Zd1y?Fl(W=(^Xz&4vic9?eQDaEpq'
    '+M-JHqNMl;t}2A4x)dR#8<OgmcC^O@UW`p6nm<~~2{i+e8Ja;SuENC_Y6W=!^%vUc(Q7L)bo@Gv+$P{`0<Ly$SY3b~o5D&D?$mp?wMuebpTtJCLpG*7H?K%<'
    'L8%(28NmG5M70g;<*m@sLtHr9YI#Jp9F4aD8Z_*^+sb=ed2cIk)~z8Z6Hl8gOq)FXzxFP_5b4?kwizf3^J*&QYe?&u(M(pY1Y?~TSUsOLk;)7)LPm1G4>U;F'
    ')wOMwd^=`InhSCrHM=3>V?*}F#!QUSt%{8r5}UOh1TnYGg~8^+ctIHq<AwE11`s;Cwhe}DgJIiXAgwod?7*=pkR&h+4Y!B<gL<t6V<XQfe2<KTL3>}DvJ1AY'
    'g7vL}QS5?-zXCR!P+xLr@PNxOjVX!&AC^ZbMm;R#hm|x$QPE~4lUct;Zv!>VIa;(6%h{o|kr!)GYq+pow|r{@<>Lm+dL{^sTf*!}w*+uFEgS!W(sV6)q%%T+'
    '%miq4dMz)E;TLvWbS`3k3?|+f%mCWIz#kaQhyQA3KR${HUZPSaEcK0HEF^G5UQ#$#f_B>rTLEXm5-4H`x9r#bPLI+>dv0aPY((qJw)5m$w}k|uk@^^M$x^Vg'
    '6!;a-6fF9XM;UygtdtC)Xc5~@rAWiU+mUJfU@981e2A@5;_DD&I%>ZYQ5agX|2X!{>31&vdM4@A4(N;ek-L)W2^ie1n8X>f&K$O^9J-3sBzNq~AR@(*35`7B'
    'dfzuN`afint;=M(&KQYW(u&-&siJA>Y|?MAf#Ek*+#TK-ePB{7L^LHaR;g`Q{j)w^IlxYbF$y{XX056>+8MBl-^Rld?NU8hZh>Jf>`J@R?F5U9v4c6B!ml`C'
    '>XO0%=wh>CaVD4{L21>nH8dn`_q~%7`SC)onv+=7d%3XJXn38rU&#T6IYcpMHWV|U81X2QOFS>OJ!q&|>EUUPHU)x(mfi9#TBogKhGv$gn?8ajrgB0D51vv6'
    't<7$Ntnw2C)v6P}!$d-WCp}aPGsINA&~MY>!x#{ZI}u0fZj<vu4_v|nxALs0#?*DUQLefc9=lRB+$qoWI={}_>K*dkBY14vm=N}gl{1dZ!DBu{x1~_*mUXvf'
    '_l6hA;BUJiqCx%J+f>$?A9NsX!HCD2PXRNI@oDMr7cg(BAQj>QJ=!IEw4pdJUtY{)-LE4TyKsgYFIu)JnlsKQ5<)354v}$XydHGKmXNXvCeSDC?U18dw|L-r'
    '{nOXHe!soPuKB$6Kp@fH^U-0*$Wz;VuM4jRqx8De4_o99z+8g)2@E23d;CR+{q)#j)jkz<2Bzn(S@uyr$}a*wXY@pHBD!=bYb|H33MK-gg$*Cv%5v`V$$~R5'
    'aXxpR4&Yn1h_0=&oJBZ=-Mcf|swF=$<Oo^}O)dbyuz$Vb_zHAm=^YK+6XC3q$Wy6qAyL_mJVf|U)82`rB`;K&DlEV>K-ow{8OQEAso=bo7AC)}`i+oXOn8ph'
    '4~qpn6Wj9Kjy<W~q+d+8A=Cz8B@d0WTS{FgQLV1Sl9jsOB7+J|m~OKgGH0&GtJBr$Hk)pqP34%cFE{jo5d0H#>54+2>)kOYD-I^vGCY)8*SHSm8-OCVjgLEt'
    'dnSNQ>7v67-7$pZGWd!NY^*d87(7wcyl#>YKp?f@G(LuM>q^&Bt20EvPOQhUmVLZ8hbc+W^22l%nV~GKt}Qsh3{ZyA&sxX^F&;=`S>{f@F5+J;1xn*xokyLa'
    'OsZ-R<<EiWjmn~Biv33b%xn(j#L+Ljf+-9u1U!CS@od4Y+(*h~o=G_P<aUd5*7ufe1#<#0%=7>Boi7DC8q~q?frO=~9L}uh;eOo6NfEpQ$}@K5I8Ldph&$-O'
    's?%mEHyBMyqf%V((<cQ<i2I1#^M$Ehz*1FQ+ZORP{el<XS-&FQ(9d;rm24(@(7Q=&qJ?<Wm)%~7h>H{Vm7or*7fsff{ZC-MIfO@~nCu1AxAR~N>y*D+@9j?Q'
    ')5etKI+A6e*4!33w<L1b(Te7%Y^h>oKCZy>c#D*y4)FFW$id4P-mBWcYDU|QT;DKJTWZiTl;AWPxRVfRUhT+LY79U;0w5XxkiMg?ni?d`ab~VB^0iBX1yMI_'
    'j>YzHhnm|u!JHVy)|Ds{=M4^z*2w=<<8e5}OqsF)o{lu1afz8`(u}48ZIksupP6$5b(;A`W?V^_A%x6EQAmNN*)sZMi`^12e98=!)$7ih!J~4UCQe#|TRC@r'
    'kox5~kRS}2mVII==wvqGENd|ULw2EIQiNe1*;yul#GDW*`xG_5v(zoq=0sTW&-k4jnJ?x;xgM0q8?N6R56|!$OuIPFz+1h$4o}{bJ6y=`{nnfvni;o_^_HUJ'
    'M9LiN<dTQ!0HUB3vX8k_5g4NK9qq;A1g({`1*dYhvmX~F%r*FC|FOfzW=s1Xm@PaoJN-))n8vqZ;MAl!%2#1w)Jg?DbTF#rEOeU#T`HY=Par@IDUFAZqs>?X'
    'R;0wiP#cVvN4mt>(xooG7jWAE!Q-v-cp^Qh6>R*LF+@$OOHeSpK#D2>3M`v-5K{6XsGOzJK9Arb6@E*4gV*6sQSQm6w~TH~VAHrmQ(so$sxHdeDwkKoXZkJp'
    'Ak%06_N@F?oR)(|b=zk-NXbQZMyj5h7@X(I?NpX9OweHXq(5OKcS0KTCl~M5EjkyDitCZu;ydR(k<DcslkG;BHi_mYa(Q(L(`c<skmwiEB~K41ceLsDeQ>>g'
    '?|q}|`S~zczHnH^bAj{pfl)8Pbi@4V7;}Q5w?!S9&FcMF<A&L;-j61P7>#a7t>)@M(8BPLn%AN8NNgD(4qY!)QM6VFE!DM{S;Uc}hKtrb8WoIDWLFSI^FHZl'
    '{NF+*L(qJ+0|t{|VF17(RtNzh3IQBdDF%qSlfB7c9mur&s(5AyyXeap&7zSMmSzq9JHVB#>>7HpzQ3C{Y7v{sO?bR-&612W(uh2COZ>Ag?~KScTf#CCyKIP4'
    '25?C#gABHJZ`*!C$~G9=KLfYHNO2a0icIhG2CeBdlVTKudSV@88o5ogJH#Dmo7}+Iwm9x=i(|NHZ?f3-#><EsFPrs`#a)$ymRZC6qRL$@1Ww8pOay|rV6^8u'
    'VGCx&#bt^wc{B9dv>$nRW83>L!wF_emW?p<*6qI-e2@Dt#>Alg7vs$xy8kk|#9nX8j6Nt^Gi%b?+unTH!aJ`<i@g5n@q-V*>ZQ?Ydm);3-_G4vapi~H`0G}!'
    'q#?r0@krOj6Q{+!DTeD$kDHjg?T)|bRyyIg<`EXyMLyv;N}_-x48nS_?7@mkyv6NidLKXi6&2A8#=V<~VddqjyNEgO0k^H%s4@0QXq@!9(5wVinBJIg85fVR'
    '8nSV#X6DGr*_pY6CFHo$&mWpTJTv#pj2dAfU`tu7AMN<yrDF)r9&>;{@c5HjMrVzdGBMQT4hjRQdIW?))0IhS6OdMqDROccXG%f2m!bzIG6N2dyLS@58i$?d'
    'sn`OncW_BAo(8jW>=+}R{2I>PCPNQcjVb!wF-$J!cA6D!(2?wD^G!Z>soQ!?lu5nfu^xn>CoXrSs}w8YS$SFF!FC(vhIwzJ{^T^9`W=gIXdOxvpvxeM;ZyF8'
    'OuH_U?~~*^AbF!ay3)}BJs<GtK8a+h+wgjB!$l}HuTu(p4NSVNU0UZiOJ!7u(A^UH&uG(hO3@B;dgB#iLKyHvP_Hua@`ZayfrIpsw+#i&PS9<KuqyV!<bYoG'
    'MAb#VS=}$b+6;#`kY^9h&6W=SV(#GV(doltPx-;=nWLrY<Hu)@{i1YY=E(8G2cv}%y3}Md89b~vp`Y4nqp%y>=T<dTx*>WA7u_?SRW1iBj^m6aFB8@Bq{};('
    'U^eK#;E%k334TciYq%>V(G=!ucUSDF;WJ*ZNW13zX1eXUiY+T0K^PbU{4k5e*MNzFfUhVAzhGtC?1VN}Ha-LH@S$bHliGmC%ge>Bb|t>ZJi5^w$}w{CT;i0q'
    'f;KmouX~G#o2)iJC6iXQ{o0c8r7*Q%R1x6uDG6UqNxM_eTIG6%_UjirCTszM9*4rfTLjt&gNiEd=+tpFaX-L^e|6CnHENgPXGc2;*lCM8?RXv4h+T%Cop$1;'
    'NWn4)oC{+k3;K!RcF-!(M76_qg)g?ni-6@E8yg#wd=#FU;yLfAs}AUjcD!%RTj_wH%32Se+%MK+3~Wk>682Yt;04a9cv3y>fAD0i73P=ZtS6LY=R!~jP|B_T'
    'x*K)`^29QtQFAVGd2vb#=VQUR&{+TzyHa$o;yk}y05d{{G~{AlST3QPr*T{~Io0ZfdDG^Yha#jE@ZY@nw;=wVRC9OPuhQMkIBTQ(3Gn&`8)0G|q#Qv<#>-l1'
    'MGC6O<bsR<S@HxkuMc-}5Zt^fQc!T=u@LL0l2z0d^Eaa*M3#71Q1XFxG^FYfcj)oABjk`&%7XC8QsZFo@wH7}iM`%hMu7dmoLTFn+&M*V;16@=B@}pQ_avFC'
    'xxBy)Up=Tj8&}-%tt0XKBj&7s20Z^c{x%b<#9W8!vP3hBuY3iGMtd({O@xnhJ&(Kw4tfg2UX36xybJ6Up8?&ucgS%UL8p0qJBPTlhO3`IKLlp-6Ewjw1}mf1'
    '1lbPuQajY?re???2O*VsD~X!b(aVafE!Kyluaf25)WC=(w`2p9PcZ=eg94CHnKI{O#1rwB+@eaEtbC=tjD0`R2!hTMNEsW9p(8=HTldm3=YZAS#S3JltvTIl'
    'VX7w?Kp#pd7TWAq9&h-dbN!Q5i+8jf>kwrzz%qVlwY)ZRHKqgbTIBhtM=6smv2%^vWt5_l?gi)Dg~?EGt*Y1REG?r81#eLg>hTL{<lZ;sBLK0;$q+DWk=zPd'
    'jLL}XGN~DXZs=IYXcrwMP`z&2!%Oa2j}zWCld!;yTlVU8XK;Q60~=&S*?ui{H=A*u)2t;g_C&VlWKIl2b0)`uRsDu&UQ5X|hd2AEPN?gv6-laY$4y_3@CGXm'
    'Wx=w;79artV@m|Z!732J$TLt(CJ`x$plHCPC{FI9ZYh<(+V7M~u|x==S6_^+Rb*3mhdj=dcuF@tlYl5PnU<1fidQ|D>d@I8D*(vJx&`sc*vaeyL)#fgA+2`P'
    'lR+v51!9@M;=EHdGB%)U21Et{7IaM95Tys|&(jYDNDt1KOR}T9fT`66ApxdwoZ;?@D#$ezmZABq7|Th?#*mNR$vdAzB_!1?(DVE(E|+G4u@l);2qZ*^UVbwz'
    'sRb=)MG;`S4oIp7U-Q(o9`#!(nIE$eT5(D?JQuTJ+D3+r)P!$4i+r0IGzj#j{A!fhqadGJJ{@1@&d|khwrw+JIci5G2LTWnnZ?5#wIjHG!c*Sxr-8&5K@iZj'
    '_+Watst&EZGdPpuxE7}r%h$-w&GN<8oZBTu9K(`&*^5m7ls0&upEvn;$^?@X(KR^3q>RCpGaYK8DYjGQBXNM)><o>DfiDURs9W<x&_bnhcX1JB>8Tw8!yQsG'
    'WnwjI?V#I2ew6LSCNqjvOh+@76iv9OJWFN+{>WFO666XOH4E$3EDWeQxo*u#qo%Rh$et-2v004KX?Wx<0fHRE{Lgtb`EcW4QdbHtaTaA<#9Yv@9`^jRpu!(J'
    '=bYN%pgxCjf$|aIQFz1qWzbbNNo!WJ`aIIDj5JFl#oFL}NXT~tEvBGYjobRFOF0EGaw8R%A)1nIHIf=s61p1)eMo-bxI1TX--I=$2FfN}vtpVQ={^4@yOugf'
    'LXM0}WW=Jzlo{QgJK!(Uxen07<+^vUTJsKg9auNQm?jK6L#6Vtu=btA5ScJ2g8)238hUbc#YTvQjm}bO68JA^UTXS1<9HyM>`ZE92NPHMV6b9nQMF{tEMU6i'
    'Ie{pyUH*Ollb7N!2zm0jG@bo4d4zn?4Ub8b{B6t`mZ#)CxXQ+#W#0{rv|};2y)B2!q$KFJMtEZa8D(g+xXhGJl3%R8`bGbx|3j2nefNsijG_p$tq3sV>&kga'
    'xBd+_^93$k7^$#=ypXP=4YpB)O`=ywaYm;$$Scx`k;>-s*pK1pC21#q=?V)5doq*O5%DXfY&x@wDbKv6?q3${urh!XhgRxkhPE_liVbOJ_^eEd`4ycz8AO;E'
    ';|Zs*cW)Lg?0fIQfAV?C=)jQxD{@V*=s1}Ol?7e6BJ_QVKE!cbF_!4j_Z3aM+AC55hlvuHCk#G|VWj4>bWuWatnDq<LBQ&c>XlNtPEKY`p4lrIoS!%U{k7Fs'
    'KVAL!y|vGuzxnzL{n!7jvjY)zMOEa{vKd@PIR`TH`6*;y{JRjV+%qfpMy-68wfgsK{pa89e|Bl@ji<a;=+}d0|NNKNuYGXy@o)N1zp(nsXZU#Kg|&}A?_YjH'
    'Gbp$$TJT=(UhO^To>IfDfzaw!wQzSdcSS!%mO6pBcoOhpqMor^{o|+os~_I@)8qZ;{*0Kt_MfYt{E=U-J^Res6Ia%4p1m#RN6poe(9tn4fXV!W*^>+UF_un='
    '3gH%133n|RdR3_1to4hm^}t-#+6SLXxRB0TRFovnsU7vgsU32Xxb=Dj3s+`hfXd;c@bn9rdP}^e{X&X9jY5*cNA(F;E+Z7x@H~}XFy;;Vg`rqtdnSp_-GGlV'
    'ZznKD1MHJu$Qim2TZ0D}^xZYqhWpzdr36QBxpMPJ2RHe=DSW(*vD~o8L~pu<#yXgtTDQVfDk++jks#2iSa-1XB;1&+DAA|5OT7%ED5#e)E7UpzYdkQQUS)-_'
    'hZ+W-*xYmyd}?9QY$o2r)jyqIedmK4Uw^Uo<!ku+zkaiJ?RTqxcx&~wPx~)@xcb&}tFJs8Gm{s^%<#je+q4<EjN>+YAWeg|@ljwSzD^~zEUzTw65^<P)MAV|'
    '-x&3TUr^q0c1G^VBoI>bgOb(O^J6JZQ5iuSfi;GvD-6q_9Ro~jrWokLdZ1G_h@D2ryeH|A6+@bZE}>5ZO2Nd&#d@N1DXXuDZbyU847!wY+2d6u6}!zd&0x7n'
    'SW&`wYwR)Ai*sXX^wY;ob%sV$Jvn~ILw$KaNbNmSk%YZ|1=P|@%tlE(dDAumxls<FKH$!NmvaFpkT<3_N*vXeHpvIcCEl~rL1Y{p1v};i*Gn6)WN~d%(5+WW'
    'L8Z;(jZ+SH;Q~+xk9r#x?GjGh07?kbK~i7r+=CQrc~uesy8M&xYaQ+7`bq-Ljb9{G0|8MIY|B*2ZPN+s2>t-CAMcn@*$Dt(`F<?+TTno^=T*IGj4{jl<`{~1'
    'UNZ28erz$5?3cRHz7^DawYb)41tCMn^v9tMazzX}?z5?f!UCI}s^KoQd@os*OWEaYCEh+yR)C$jtc6B`B0cK$<A4wr%3AZ~GPGb*l!*M3N$uCjT2$Lp2kRMH'
    '8K^zAlC@4+QG@Fp7&xz4P1l*xJdnkb+Hysn*D@#5-CAYS{(Qj0%Z$9LPvSkaD6j{K5Qzglw#BOgLFu1T)2p1p8E?uC2S4H#J>p@ni0dZ8rC=G9e7RdICjA}F'
    'Qp@$*GI;_4kEb?<!BGeFj_sc%X=Kbz3FH36%)CogP>1|zTw|Q7x>Q=308FCi`6&T=VPFlCDOwFtVQOLX1^|duO8aayk~7iV0%L%i8EOj{C}d3xbi-h*88p4I'
    'Z{I916cFf4*)L7y=}~xE>C}Kbj$RWHcgc`9{3%1*yp@&~H<uPGH)>2CPdeTJ>UIKLH_ApH#%l`7`Z+-9+O?PcW)-HDwv{8*9~DspJ15)`yz7F=)wP;5(K0fZ'
    'IiE8qs0587Z96KVl}OQQ)ibduM{aqX|D~-JBN7P}M~SPO=!@M4*@062fGCEMAgLYBJG=Gp3F$T|6||ZW7^l&&wWp4qQHLtb7>uMlMM0B8VMlGqqs0P!@}L-4'
    '>ZGC|Jipcrx}hCQw9dsMN|m6)4UE9ZQ<zaf0*XOb&EjLp1yca@jB$Ntqnp<~K@3sCk{;YRJ1A$o92E(x5Q9~?1z7n~_y`6lb@NUOxS<d=B0m&jAMz!}0Q<BP'
    'ziWBf1T>?gP-v6=#VtufP-~%(Y@xs{6gJyJ!E7O4Nbf;m^F2^R+MlPb{)s$IY4NVW?<Tc($akfN+d}2To411OcLNAIHKl|@o?ePu9c(e6cgq1(<)h)bTO%ab'
    ')H1*796XiWQ4(k=p`M~#Qj6v@@qeR9cH+d$k(4>cTOIlQkXc1zV|v~gPQ!2F*{FuQqLNY&pCrl_Qp?&UZBE1QQig-s@<tvIC6Gptr%~n*gyM&#O1J9ju4@TP'
    '$TR#*eR0tL$ICV)tm1`HBbVHe%A4}+R^*p9LdQ=Woh1VVR|_iEvQ2@+_>Z6`9Xx?GCM<lHK%Q4xUHHq<b~0iPEi|y3M=kF+s*8)F5%Ru>-Zkh<TNK{V6l})j'
    'Wz2#gIhQuqih@4*i3x*Vg$WraQ4ebQ2|Mphz^p<5f*L;VXyS!|r4oWXwWCzRTy>?=sU4Cm?f{GCtRLbDgGb5kHSC%ojBFFX;c3Q9@^nB5qf@2L5g-6xRL8IA'
    '7Q>>tyRPhl;n;0sJj0kX&aSX<>{Jt0YYq1dtTWo7Ez4xVgo5QkDL5m;2<5<I6e?ZECGtg-Kz(<_ezZY2@fOBNgyF%gvRNd_7w!s$OBmmWw1hG$z<A2E=1dzo'
    'Rx|>Nlo5-*jtGlJGKm|Y=xYlGTRa01I;F+BTa&pOa4^CSaH@QPNUGoMU^$F7uJr+fk;Wm$yoD5~cb0yMLzE-YU!}HaH#DXdG%gMs79%p18>R+Da4HkuoO0Zk'
    'B6Y}61(H7cF-i@6_>Es3ctEP!s(dEBLzIPtG13NxgIS}xsXAp)wZ*Z=BoWVzKZaPGb6U;XSVTfKs2VhUM`oqLCyOA&yKR$G)@O>1j=u=|L=PJO6pPc2->G}_'
    'dMs%BHH>7&dqiQH<Y4B|O{2`^^1{#!F_J=BPrZTy0~Tsyr7<)KI&B+EIrs;y^0wEiyA?d_hmDUZ#-JoMK(&>u<VW<pgVd(zT0uP{f{h$`<e^AmJ}1@Y*Ee4='
    '(^SiAm)fNA?$!4r)0td&N|~vn%9%~EQ|U1tnR)8-!iLyu$GPkdjF*`^F}q{xu^oqJ_8&ZY;vmTV^@0bi=@yX4vkrsSZ-(B`XoH724UAR5n3|uu&92+-fgYRK'
    'HMkb$W1SiwUtV5zTx!S(+O_dIHx`bIk)As^d*p;Qee{5}|Jcz3GjlV?j-Id%9h<ctI&m;-%^o~Hd+fkN`|)*_N*$OvF*iH2?;(66K;-GnEQXb!Ps@p%J*Rfa'
    'uI<1Rg7rFwII`0Z)Gd2t@zPu24FfjohO_}9;zIFOYzYkugRbqg{W1jv;+(5uyI?ewd)5h0_|)=vge~aSmaM%TR_a5!ph96_DaZxwBzBdc)#C$7!E%#x&|oZ7'
    'r)Rm{4qazQZAwsSRXR)P1rBjnnv+&_qLDVS@M><|I!NG?m~@*_IJV_h2$Dnvvt&il<N@Wlq;I|#a{IUhZU-PRFxE~y)Cu)0_6gs1f#ET=R-+NQUqwmjZU#Wy'
    'q+=bztvvKUciXK1xF{m9Vt{PaH30{Dh5f<k`Hb*hFXOfc2sJzv$C8GA<ADmR;$ku`EgG1=5gpoKwl!fU<5*(bVYjj*kON~^<oN_<pq5<%ZqR3$68Hfa+Xq}|'
    'q{1b?g`gJwMIb6jkqW}LcTfD7wz?WY+lxgf2i*?FMbi!d3>R2k5+uMZdrcq(Uh`*y*Dy7P|Cd1*=wm~D`VGvO(Rl#=hr~JSSGx$YZ5iy+l%fc*_USAG;NS2='
    '4zMKF<H<o|l*WB0==>O9K&OIY%m?=PqSr>FoW3p+`Ojc8jiBl;($&k*kQ#BnS*dq<W2D=mj9qoV;Ufw#tYNU&S*DmzKC*_#!?S8)3OBI~e!>!2DIOO6TDQ&T'
    '5*B?B(nbf4m46M)k<tOT+2eO?71N1O@+o?_4#Ep4)%2C8qFl*j^ro(kLbw)_iQr|;Zb~4r-s0}0k5erm{6ueMUXH5SZ$rdYXp@OY5CHMCZh!^g@}1>I?@tEp'
    'GfAPb3~vc<5?088(c}O%E;<uU4W^;!UBj({q&e&Qb@aXo`Jf4}EXrNv$%-qO%Z=oQlz~7ED6NDbQ;aunQSNlmHef<Yd?gGU`rxh5nJ7Bh(+==9)~Y<bGfi7D'
    '{tC>qIw*UC7zezzf0l0pN4kV2VUQoNp>$FR8c1MF3W+{i3spRegwwQ&t>BaZW@2%MjnUBI1X*74D@(exfRO`p5#>SKW0NsN&Tfi(Y~jHJN(XKE1Te&y>vK#('
    'L(S-g9v-}bVQ`_jpiVQzlJ;cE=qKg25@m~qXvkV|8VU+h_B|d%f-m$-+rwqCnzkP1g+Qi3+E7oxT*RYTb-#Irh*t*2L6$+ID<g;B2PxHF1S=HPNmf^?in0l`'
    'kV0OtXpBFeHkZah`q;-Og__BFql%DX8cOY}kU?n1;1Ew&g%zVf1xzEV1_cInA*;=j4k`#->_&M~!)_UbijXSOViD#!AWsO2xj`+FuuqsHYP?B14pl0;iGyOB'
    '+DGOudqB`dpvDlhXEX}|#!7W73TS5=L5XcV?Sm1K?Er*z76+q@2cmI`wXtTCjFxUw41$G|!l1nukqfES38TqF!w_bNDy&GT>8-&3BCJ4G`gN?0+pb|&*KEp$'
    '3VUJ4YlMb~2L;{rP{~wCnTjIJ2dE+0#!+UPPW!AbS`2p3h|Z_)!U}HW;Q7WT?QfwZ6?W)Jl7pJ|@>V1v%<f9ogx-WLzk-`7Fr6E2`;1prl4{wH7B6@$bFJx$'
    'Kph+@qHZcvq69m&<0w$wwe-2`M5q0XX&txJRrlqjT;EdCNr|$v5o_970-Va8hhb&nI11pdH>Ax=K0?{4W5bmID9@@Lt!8aTN)c6XtOw8_!<P3eLP(p;I?-h%'
    'ELb;neWXu4U1(FNi$*33B@5t91|V4xU|B*>!8mMrlvhB)0ijs0F8fvTdNp&z1|f7CA9G+QwfV~Ro?Gwqa*J&b&lQ3}cs8i;E@@ovi`6AIDc5iw?P9l(uM;9r'
    '6Zfrd87c!UfpJ@P%!Q!dh||4Uq<LK#A^K|2SS~0zO(u7u%@nOq0!NP1q&7~+U6jZ_j6t%Yp4aN&q=VH<N19ZCV#uqjjMd`qYJ-UGmMkApM!Qg<M62`w&j^u@'
    '@anj$K!4do#^a-;gD|m$AxY9lbq>kELnYhAGgY{R4e;z3ks4*1K(11EeIPoQ)%ypeBq%zGl|*gAYz{-W?bBpk#B<|vVdVKrJL;0a4l}U24w|0OnIIRzc+pQs'
    '`|WrI)h8LIMXU%tV587PPqeX&;~|xyW6dlg8$=7L5TqCKw;Gy`Ut=WPnv0)F@ClEA9qF^kCT|B}m?NrLGKV&-7U|i?)Ae<CIqdo!>}}nvv4(Iv3VqFTh>6N!'
    'LXQ*%qaKR&y9THd4N*__OpZ!}Y+HcCR-4I^%D9qCTQO_oA|;w{LM2N3i?s&MR~$pdGp>F#U2&lzF@VDbXH|-ZU3f}ScR@MSdc*BS(k?ETKsNf)3Nl615Sx%j'
    '2CYnJt=q^lLx~K?L8P_KRXFSU0n#lSt)z%58k03Vk?VXhsKc6yjQ}amrXo#jyBY3o7r3s5I3tR@R)GQbp`R$obZgm6w3q0uOPV9T2>B;gv#Qy$)-3ymA_E-P'
    'VdxTIzfl|636>sy6M2T$prMWd3O>c*2ozlbHKK{ffZ~l;w8q<B2c}$Bn%kOBfYud&RAL3HwXNEWm;<wW)FbVg72J-gg+SP%m6g>^4&vBOGyz2iqaeZ<MhRJK'
    'T6!#2B@jZd8jukTDhPW@p{+Crarbi3>`YRd$4OVspfppwAeN6f>R|L8o158x5X?D>+eFIaToetVt*$qw&p1sqq)eU!g=ye)SaK2T!f3ixTC7Gq=%tburIm|b'
    'W*v@@cO)^mGt`UhDD>6gq?0<8zQlL{r0%&PF35}x15qcM!nCauPDw;viDg9ah)O1hg+sCUCtcJV3}?)B)4zp#C`$gIR;@)kGTA`T&Zdysm6oCAMHB0a6iO||'
    'Cl1*eu!!U#1BmTv4*S_tBgEUSm^rf5@?2QT&MomW4#iput+in%8-v&6%8v+Zk7x~ymP`yGG1o|r^^DyCrBpD5CHjPMxumnE4n|@xs5Hu?$np?`Fd9a;XhG&u'
    '?nJLy?KY&DYjBQ~HN1qBgPIU@L}lp{!N|+BiA7#8SZ=Yk(C#J}k%=>S_av22(Q1V3T-qvNJ0a#%(?<lMh|cI}qrXocg^%pw6tiD*9HwkxF&u5{Dp*XxC#%h&'
    'MOtn42CQLp*PohK1Q6TSeT5MUiS08(Q7cOw9oJ-iBSV><9<{`Ytsi3JvYXaPVi$r}yOGrxI@Uu?(6b?powovQReW4R6HINBCi0i`;%2AjCDYtv16^X;0s-5M'
    '`-a#wEbE@`4Xx{hO-6`noWn5K*sZFaQ3h}{=wL0iiA0*G9Po;c6JMdULj6L>QL2Q{t9ra4g>yn58POD*Anb$%QmWNd4OWCS33_6-&~naO@hZCLCwZ)h()Ma@'
    'n>V=Pix#y3u?Hliv{OQqmYVfl4M?DN*ihGYrI7q$dylOgwSVF^&?BcTA9SaAZ48LB#3P_3vjtIM$-olntme!T3te%{oS6d@eh5Q@DxZcHEQ6qd+vtc7&>C!$'
    '3UpEoNVHr*Z*al~Ea#RSKA|U4bb}<<J&tw24{71W`LRyoA)qGis)`w*ur2p^4NU8IT%c)!frhin*-E5;EgKD-kOpCdt&QlS^C7Xy)r+Hh)EMOq<z8!otu}qa'
    'nmGZc+P>)%Gba?0PtMFec<iA$>*Vz8?DWyOnS&>+W3ze;!LdWu^wD2hKbtvv0IVXPcc)h<8L!qIpQKGybHzq8ihQxIlwLhpDigtIIn_>)y+G5snYqIUvq1i%'
    'xtXJfW@nB*aPY{%qjOp7$idnD4+5Uk`(`i;HgVITnYp6}PjIxuv;=Z|dKQN6p~KU&*71jCj~_d6ko7U|T-I?n1G><HW_{Y}pdBGzSjIR<I~CTXPA`glbHt00'
    'DAs(k?8yv6FrBc+N|5^@3H>mr_-eJy@?LEHlFL`$`%SKhB_H1Fz#B<etajLU%RZ+*pF!npftl3o5Go7>Ue(E$3TVQLU3cBceY6fnMQQGbrdO-`HOeoSRh#iy'
    '!($xzlGf#Zo2?1lL#X>@aaALrS3}=Mv{fP-?O>opNW1sxvxTLqp;;(twggnCg9p;NKtt@(wM_bs0alk$3sL++*z$1iU*8gk`2}_`Z(X3F!(L<D5EM|Q=tfr_'
    'V2+OQHt*)6o{zLSrE82WmWiZYC3m{~#&3!t)x@pgd$5OY)l0k<#Zg~`UJHUMCNI>zl4n3`2dx&Rp5^1Xfcv8B*D+}r1q;+={3genGzUbj;HCreu}%|i+6ytv'
    '5Ezm|6=VKA2}C*V-RfDNwrUqeoC(YZK?oUwA>hl|u-CDsE2u7z6jCB%<I|BU*CxoxCA9X<*_+s~9J<*jP1ed%5U`Jx+^A*<0Qq#m#9Q=8!hrZA95ELkccjHm'
    'Rv{64#DQL;iD59BpN)xIm*`tTxh`B=WZjITAmUsq$d{PPtl5zZGe0zUnP6G+V6coUC|-vuq7hX!$Wa$5{-#;iH(!*IE_PwaODtZabd6-I&l&BJGu5VzCg>yA'
    'lg1Ci>x_#nf03m&&VQc%M4!coK2>iKRykY+Gn}V5=-qaMM3Xee6(zOV-EFs{ojT#121)Hv0&d}>%_cq5<({xdv_AEa3K3zI`M<2OrDkC&Y$PuqJbD1NZdyd2'
    'C^3EfI24-s#T1TZl)j7uEA&`I{6^?3YdcsNEjVJ8tcdY2of66t4`3FxTXepZ7lYZzAF$|q^=gPWO~CwTbx_8gQm=!%h4Ty1hOvlU73#1jS&QVYST<_wamTU`'
    '1kEuO$DmJJ*}#u77OnbesT+b30&D>*1BHuNRcM-B-zCP`6ZSyXuc!@Kat<&GAUvREJq+lqFP0XrSS8`9Fmn+w9R=>vqK)j*mQ>erkGJeaVKuZ<t8fXi=BXWY'
    'a2r0Nd<OH)4P$$-B1VVL%7=awBp?VV_kfkZCvur!o`2jymz)vpXU_NFzXfZau*Xcc=9aI_J|K9esx7)qZp!KruNJ0T)}u)?|3pyaf(C^IYdPVumrDS@Db}+j'
    'ITedV7Ln#d2+HJ$a}9r><ZV|!$GuKqtq1_l0pg<4DE9pzQ$<A05gg59myFB8-3q<H5cB;OZFgz3GyP6@o`HsBeX@jriJ*VYS{LsNslf`j-Hb;mN+(QY?T5+E'
    'Ids-$_Tcn^BL|&Eb;s0>`>Os~PL5r~fK5+K8_bsx)Phqx?w7-LAKvK_wkz;u)@ubE2|g|Ywo85mPw9YY+5xsqua5aPDC70io?R2I6$$3PaTk8vHx6yyPuE4A'
    'Y9at~HRL!tY+oHio90k(%+T#9Cvef8oApAGztpn#KYV<)Fj3f*0Z^{B)b<vOW5`P(FjLU-nkf3i(1C7O&SFN##<*Bjj?0?kEf+lj<G;dfi&6}Q5}{B+6iS3b'
    'DUL!3;gk?ii2ymR>f+e_ia-h(PUuc@*ztq$vReg;Vci_o&6Vq2&k4`gpeoSC`rplR*!m~_yZN4>O~Z%N-TM0di#np$+faW_L>1e;+z}VEiM#dO-pXQXzfLvh'
    'f7yK$IFfK%uqEsszI+tl<??&??saM~Te@XzCUYO)c7Ogp?Ara-zgqJ@_1$I9j{^w760Eyz2Mp|5-DBd&iSQGUYVG3019PpJ>6vNx@23p3pnyK;VpLe{<@~Qe'
    'b*=a=7_=K}>8B^FVX6Fct))o?^l%rEe=z9A8w0Icdjx0ouY&GI8v9v3cl^-OFH^fUf7oC00OUT9+#7+pXV1)`BL@y@Fa`Z_0QKj6p8spac}DH9qxZn8EHwj='
    'A7IVt2%SI?H0OgxgSq6P@)O?$b<kvZsOeL@ATKo@#4V1W1x<0g)m?X4|L>jO{n0w=ErVH|gNHZH&xsR*pggOd9(RQMJLnPdn!gUvw!`8V-+}cArne6oo(k<E'
    '=~>3rA1OXskj?-6e@`#icjX>&Gh{sIQkC|V)n!&}Teux)tYbxvVi-?&S31tS*$9g2<gjE8s^mnKTvR1j>-xAdZ+aO*cjr!=nNGg2b0_um1R`DT*4U*QcI0w|'
    '!r(%7M^fJx{S}OXI8GE$ZbCE@L)s|u>T;6>9r6`VZ*5|?Xl+J*<v6^dgq3x7iMq!YMd&`5mO<!Kd;(BSC<~*?BvrZi4vxJq^J=icMD5~Q8ErJ#ESi(;PQ8}^'
    '^51X2|Bow|coE9Bfs~1k#yr3Miv=TlswrqQ>$#wEyhS{mFxN&?mkQs9r>^{Zct0_F1HGH~L*)w}!~cJ7UBB|l>Z=zJ&Og3<qyOm_*RQ_VfB&QY<-hfx{=EOq'
    'lWT80b^Xfu>sQ}gyY|eD=byOw`^z`~_}V|dym;fA*Kb~czi-}n@8b2V&-B0g?b_APuYdifsC(&&8|R<2?6oJp=wG<j|BqMtU!KocI4`()L~Fu<9t$EuE70`X'
    'TkrHQ|Ka+TSJz&-*njiuwdXFt`_<3i>|cC!_54TBsK_)&Z~D)E2K}w)PGTICbpq7A+xCO-A78$+bEmMdcJ7>>0$PeT`fq)F<MJnK*Pgub&4sm(e*^sn6mPu#'
    'e*h44KyMqddFiROk6yTW{yek<4X<7MX!W^opxC|>vky!k?f>>7wSADD0FtEJ3#)HG!If73>v3oV@&Ch9{cBhM@A)ULUwQez&%d><MeyI}--cRLEVuUNe*zu)'
    'PhMJm{Eq;<|Ji>41y(O!xcS>>W8_+W>(6WNe85d3`dTe$npq}~1fMx@1cw|X6HM^6mp?`fU;7kqdjP95r<ALO@85p%=7ksfFa17+rq{2$0M!6A!h^C54_;lr'
    '24wgv6B^#*p#A52t8YEmfBxMYUww7`+O_Lny>j!9PjLZm0m=v{luz)f>tDPB1m>d1dsqH!MR=`#{9gYLzfmm$9#G`Qg})+!zj_9{^6>OQL_{=r^%<*PZ4io!'
    'yXYJ6fprYbj~vq<lhR3iRHt#spZZ(>r5B;_>RW%g`Q8_6Z(c?i`{!nJz)Lxj)7I)MpRK-d`R1F?0FY=Ze0}-GXHNnI{(%_a#bFZ@6P6F=1bm>pW+KJG9zcBc'
    'llS}YBb)UvKE3*zzgTlX4WNK^BshZD_Mdqg5u=Xx-@X7tef_I%VCW(;(kHC{_J!+LuIfW@<Hf)Czy3Y6apThuaR&3t%U`d3b@|4pe_wm+H`e^k*Pp`dr*FTs'
    '`s(YER{fR@w2VBdXXTw;f@2GsfQZKZ#@D~=Km9@fv#b42uK<y5eE9h4e>?$wzxmdA_}%~P87_c6Bh1W^&b|>B9iFwg0@q)=_73oqs{HI{)@ic*i@WoM)7H3k'
    'n(x6W<_q_pHb(BvcY*SXRjyxqS_qdHUIDHE0SlkF9h9Bi1(I+A?Mr`#vG2e1@5s!uyqI0c3CIxu^XB;{Za)2j4ilSx|BYA^*HMsv36o;=m1i+O=X0N|f*|?r'
    '#hWj`e&f?m<ve>!ka-fx9EsZ(KVEz3nVaX|>0kZu=G%YhU%9gS^dE5+zxCYpYv-?j@$8L{{?vcw^ZxUHSpEE!bmFeP{2w3;6)pPjT!lw{3O|2-^{r14=BJnX'
    '?>`02L9hE?zRN=E=96DBO?X0Cr{_*vtABa##y6i_zxJnpeEICnYp>n7{64%c7A7W6|KrQ&khy>J!RnKruU>ko|H1{m14x}qAd)`d!2ktEc+wx#xBUy3`k!6G'
    'CU)+`DNl&syl`Rl-M{x={}=|;I&fg-2tomwK`jC<zyI6yYcBvfZhZY@|AVh?y!Y|ii=PoGCD2d5uuQJyp4^1SF1|_)0a8Lk#swP}!XbL|3MgO;7pFRKp81AI'
    '+6S`?%`CQs2>y-NzXI_UeL;qJ>tm$y<u^bp0c9u^=L^@rdJz=sjVHf?Iv_m(EvDhx)eo(L)j$72j3E&9Fa8FK45bn%`uE?!%*}6IU48x2>sNk*viQq4R$qH@'
    '^|h~A6UE4)Q{%>OuiX6b5@Eag?5oTpH!fcVHf9;cf`9eRzjD%O3ul6obIk1AFdz*bfVRa<0uDO=2GB@+C7xP)`JL5gU%L6kpI~4?x+`faX233*0n$9+rhfl7'
    'Z0Ch{`tPI8?SBZ`<rS8jq}AThQ|^)6qAd|s6cjVC1s9Fa53?B_QP2|Cue^^vrY8Frz5>c7`1Npsnn9|)L6u$v9kBXezv;jJ=l-7zCY1U1w4$IEMCl7x(M0&-'
    'Kl>NXTc=n41(+JxfiwC7W&5A}UP7Wj{ipwa{j1-tzWu?CPyZ@0%2@&xQu$xL3u2Q(VsuspD}x(eed|07BCvyX+9suqQtHB00KylPH{4d{G;!cJtkj{!{`(iW'
    'Q`f)z?8fEyV1)b6zZrE<r4xj)26N6j{b;Fk8VLK|+U4KF!@t;_T%KU|pLqfFAIb4Y{{_W`(CZ+@*4}-bUs|#Rk^<TQ9U{mg1obo5P+*c_fUG!&Ve5#2;z!o9'
    '5iOCyh>(`oD0`GH3QG}sJO&8oG-kHRAtU8fvC|~(PuDt$_t(Gp&CTEc73*>EwudK)T0J5I0FNm0$G1TxiG;ggdz=;*e@r+Npuu2fufF|`F}`dWfRWL^`~^y-'
    'j~=(?XK)b~83V$7fffL{JAS1+8`5Pq&dav({A5N<s@*uLBC|?kYqAnil?=AkrQqc1*Dm#6e1>Mob1*}0zV^4(%m2mdLp^G<2rSl%dUK@jz@UFNF`c4`NYjK&'
    'W6gFVW>ivufuT+~eYgZgPqV!SIvZZ_WDtWVhvKw8AqB`e(binNtvPwzm@LGUCg((xtT*P5(COu%f$LJGM4}OM+bjNA@<4-eHD1V16kshpdC%U+&f1lNoud9L'
    'e}ySeh%)nAnjwNS70es10VZlXx@fx=Euxflw?*h%JH-_Ur(^O3G|DH&;J|-Bny0!^k8r`i;@~P8WNt{j7@n;;<hA(8;nHJJ_+0UDsc`p?ckk!RVt-tifcG#O'
    '=d$oqy{4I<B|pWM<~GvO9Jchc!#^(E3otQZRK>0T_?~_He|*mYe%JC#kF|oZb1o`UUW5{rQU}TiWVE*vRC6!=4}R~*|EV&!3LO4W_AdOYwX-)y0T{2*DFB6$'
    ')<p9-y^%U5y=A^PrAyZS@?rnd-@z8S{>5{^RA649=6~^HG+ICXkOa`3wkWasm!7w#>$RW_qObwaa;iv&osfDY$zX-nzYNR2r`N81;y4b=xckN<H%z)u=KsBO'
    ';V<z2A6W;Rn7MZeeKJ#Ybe0`_Ttn?|ExR1Tim6(&6dH(|J+39dkyDJeKPg`W`M&H*-LKHuBhHzbN%6=$KylPESDEJM!fI(Amq8-D<DT(C>nxV(;x-r5q%&*K'
    '5lmqw+>d?}H=HYU0?Dd&>F!RMHVtD6pkIC`il;-L+KLW=tzMPKf&DY&Qq(*&4#=st7f_f2ZyM70pW1=DYT{v-PX_ma#KslHml8!o88ixu#f~_sPu^4xRiS-F'
    'nMhafa#kGt!n+PJqJu&LIy!+uNm@s_Z4Lw?dXEewz#8Q$`a%<q8*$6bPvXOJG~n2E9RQ-xrc(?|y|B`zbnfH?mcg3!z~zIqAVzYsxM;=EB4X-2CN9QoS2|rz'
    'kRn=%!lkJlI;f#ArV$(q2{SF81Yx1S2HTg{LM9Mkc&E|D!|t8alDv3?POreb;kxHXy18B2b0<%86UukB(x$sdR*)git#*J%shF;^GO{AJjgg4mii|<lJwSr<'
    'i9uYEKtB|BwC4Bdyh@Y_weT|3qKGbP*JYgX6grSPhB97K+-4gaKZeW)^I5RqF0<TxfKK9^!!+aUStf=@exEls$Sk=*c4zpF9OX4SDUWzYckU8}IY^f_-0A*5'
    'wJ$A&7SZJ=mYQx{w<-4E$#ai@s)^}^EO2;MloxRVyg7lDLs1*p18qDG7nR8rbPp3EsQZ}W6AT8k;u+k>kxdk;RYjqM!wmv~*8LQ*BJTf-dczw%d~%LYF)<80'
    ')=JgHRr8LuF8~&u5j;n{*Y?C=e%!mzMIzsKYBtSbR1LIL<M{ASh=^XqspX>#<s+<gV6hUkDG!v$Ug;=L6>Y7k-5kInitOW#0cl|J^sKNREMsh*#t9tySEra='
    '4roq{<B24J^;|EJ)aq#4f>07Up0AB&DhOocm1g9|(k8!M6;~V!KUT)UC@Rj!kmz4l>3liNQ|3Zu))PVJ<jKm)s2;vVyp0_|KbANPiYH)@_~HOP-zAM2h#g#x'
    'g8K6=##KKmm9H^Huocb&MV|xO(K?G3iP|LN!`g8<Z#}|L-RLrHZ0n&=HY2DG`XKJu5|SMcDxqkrEn#1fDq$Vx3r#9IY%6+(;XT<Xn4RLYR_NY(C|dace$T|s'
    'bz2Cxj(!NAW&;f)hgKr5XwZ;T8(@1I8d~(=I2RLnu%M&J(m-dGA%vEhB9D}4#0v3j8H^ClF{(w;);LWtU4?1|x(o)Md$uP)*Znh|IPg{%j3$o15uHWwQ#m@}'
    '&e4G6=fW62am_OM3)Dr*sM(fL4J!NqtDjg}d2uI4%;)7t_Xi|q)5$nIkVDIX*)ZSv3bssjYJj(E%6cdidQ@JMfzjmXJaSSzURrT+Ar!KfaZXKufl`$^CJiCa'
    'jT1G0LKAIMNrV(Zz^GG!m)NJShzG2iOdnFFX}=LoCTS2=aFe^Uj9bxUnDO<lG*Ojv4ELsi6R|3lLGvA0tHFVi1Bk%+St$CNwvQM6Hl7bHO9od?GGRfX15F&G'
    'rCe5`%*5%e$eN)gHo;|KRTZFUWe$^LGLkN&GSD>p#4rMnD1*j6w)iNtqx@*h)DalPYs&p)nZT1CFfk%ct<Qq2=5WWr+NnOcj5G-;qdWd=cQHsXUN`Bb12H}='
    'OjB`u5oHSMC({k*ha#Va%62A4PGiDDGSQ$|`Qh0o@78>Cj`GYIdNDR1iHl=vu~dV{PzljI7jfg_EEdLEhaEmBLDAI0BGncJu{0VcIx(p*r%4l>>nJ*@WnMs-'
    'Mld%;@PyY+t-j>JbLq&&AvXuXXl;&Ln~PdQ!f9T+VGYkw!-l@x0ta3SmWUIE>9*C_ND}E59M3;3E*_T?+FFoSCKC6djy#Q9Pof7xWy0Nd<NZM0ctS3zHDlOo'
    'l!LmK$1Rk?6C_s$H7M&7>H8oNcIPf43Hf1uK1WHxxj-CJr%UIm%F!2h^@&)mi|>g0;VC2bfG|51Gk2;<rc+m##eT>#R$s;u9B~{WC0mQW%$*W`B$V0BE@&${'
    'YN}3_g}fL<n-N3gVN@y1tl&h)9;iSWQyR26Hwq;K#MNV+qSs()b|Sv;MA71q)rnwSTrPt7h>XDCLiiaj_N@Pvw@#g^dyl|hWcYUDF)DP<J~{qiMtr*We>tSM'
    'W}51-OIE5VNk=l}$Zc5mLUr7gj!+xa$l|&IH8A>4l!Z)YRWt0iy(rZYEa-8d9-Ng<J<1w|`5F9>qRd4WphI2qcZua2dloR261g^)d=C?4p(0<bdn>s9%vy_X'
    '*zv^CB>E(+d0rxuN$)k6aM7VMR0~FW`2%b2*nwjhF*v`nU`<CyYSg5Rg&wj2r%$<tm~xJSQmtyMPKs6JGiP4qOe^rEtE5bxQVmBD|0qFdCV+szr&M_TPU!!i'
    '$NzEVlJx+sdwLW>ogRxn)2N-O`=z||>lVi0vybG|j>7Ie7=b^2e<UrSI;;RmL>&3`FDH)7o!OnShC?lE9O|#9`PKmH#GCJ8C+4sNcfpGT9_F6~(&3sHdyN6a'
    'Zauo@E@vI@(e(h<{Y5M9+=GWea@PKb4ouUFJU*X3eHx%yV9x<3qc4W0eH3$gsu_J6-S|7ztW+)a8ynyoYk={W0aX{>lrFH^Stuxf$|AW%eDd2tlX$S_HtMnz'
    'CLKSEWfW*eABhnxH_2yA9(%<=DoVAQ{#~4kk}I@_W)E{VUuhsa<4~|OjuH+Oio;Z$pMQX&z7Jsb-DvEp_^1bj(~EJg3Wo5{_MF-MtIF9!lgqjMkpo9dwO{;v'
    't{NV$9r{(_#F2X+>HVY|wu)4zgh^^M3zlqdpX*n<2mJ6%)L6@HI%Ry#3U)ldwCMi)(V151;XM!DJH0gh$nNT5``&2)GJo6+!{wk|owClJ+1q5OOf(grEC|`^'
    'b(Sb$g?L?((Z;PFjUkdvY>8C)bd9!B5HJ|BR9uR|fFB&bnZ|^q&e5pzY0-2R`ivP!j3dZFAu7>3C0(Qy5Rr>r-XuBe1CNPyvT!UfiOe`+eP!nHMh0^%Y=VNr'
    'g;&og6Y9QLmF)}nk2o=fJZ?zJkxxeA`6PrYXd<QPi`KETm`VarjP|%rHGecOelFOW*r0<kIq@aOo*IxK87A>`Ii#cPr<${1X#rFw8E*Lia=^p6c68t;5}veZ'
    't>CG1uUR9#<%xsh!k@Z-33O@b8CA$Om-}GQY;Z=1oDg?SJY#DNFAGfHRXT`@j_<0K3)@AqJj4wGbn1yJQ<4J2v)!c_c*JW7Z2#ONc%c!VC6AN=WhBx1>JUN5'
    'XOK;}L^#4<s7E}t?{R{l3N_fYns_6Na&O@Y$|cb1p>W|xJ8spkCq#tb<~sy{{doM1#5QW5kPjM2NlgVc=UK&VsRE~(GtKA}1*{y&J4xp(-KN+ojNF??Xc26K'
    'F`Z$DDB_qhct}-r4OW=SI2?yWMj_K>1YPPWi-Xo_n3B<z2_{sBB7jUQquY3ppA4PYb;Ou3jE~(Bf=2$rQ%&Io$1hMNAQv*{@D!WQwXRFa7CkXV$yqNGT11<o'
    '(&G~C2^v$CZmgj(X`@-TEcUY$k$F!xLE<?iG0CU6Lz36_6<1i&>>LPGZm_hO#M=xCK^%^gK%s7%&NoJ=9PwA=rlv(}cN7CW1>i>)2E~T;R7U1UMIzwTT@I3*'
    'UkPA=L--I5<P-^H(hN#TDLjN+3oOMJP1+T_MaQQ!Jy73gI};1g<Uj`C$r0_evt^7aG8*iSk|G;r<R%U)3JI7Y5)b5XQbjP(-H7H9eQ|SwcN{`47_l#>SVNl^'
    'x3O^F$ZdL*T-uS^>|we{&G5;iFvQdt=sS`UeYhM4b&?n2dtcn3jmt2(0j2zjk=Lr>fsU>Y9|4Uvj&%%kdr_s~^h5c&zED<owCJiJo1zDJl_3K$#BlWFl^I4X'
    'ie+ihD8KtVZAD9(&oUv*kP*v*K^aDCzWFYPXyzo*H&m&xnM#wJskCb|m3D8Y(jHN1fUDre-|d&j^Mu**&#>gjJw^{DXZRSMEJas_5)yQTn{KNrKDOa`eU55#'
    'hX3MYw1&u9e6AHaN4yXRRq@SXKTBs!@%KskD9@JCayjT`#lcXjj*sd%D89k-o0y#n4|8(c<sdjCus<I4Q|gkLg9i_e?|=9Z=E{&uRG4)8WpbH^=>iP`ecbK9'
    'o2E4zDFto;^Bsi6#|pdeE!>mK!9SCG_(Kmw4!j3qo;%e%r<u+;5dR)s!!;M%5>`j}qGRdvB3l6_ClCk!7i@9xKO<Mb(-jN}f>k<aHAcX0^Nb88LRGWuQM?!0'
    '926A^>yE>m@HoX!tgQf%9zY6m`!zc`#Tyv{u@A_$gh|urbg8f`i77N%rNTYbt-8&JXU5AKgj|Io@CJycf-{4n+fE=yqnM9LkRQ$C{U;y@!>DobCP!zoow^&8'
    'eaEkK`Hc2Ya41WDwK!nK8=F@<vOD67*d0?lFcZa8Wc7<nP<gK*XSI6b)^L0fJM|%67Z@$`qmr;rD^5E1I=jUizf~+uIFtN@ce!?gpnk^h6nUAi!7mC=0KU>L'
    '4=hdqv^>A28&RQLey>wd@P(H>2ev{v%BLC@q2exX7(b#DwQ@EmFf8tJ@=n2-oRFVe-D<fAm`cd_j^vdez-wo(GZA~6TU>0kyc)Nns)(-SXbT?g$zn0De+N#k'
    'p_5?|f#$he)L&43xlXt2-4F2YVdCL?&|@A9k>8>BlTO}}Up&7Vw5XGl5pLYLRFm<K(e6x$1enyCcX6jar<YJYaw})t8ug@*kH|SjO^K`N?}H&48v-lW4jTF0'
    '6T1}hE(phZiCj6fU0hOohOz|uP2F}0<rgKgENSp$nMarK9w_;g^DGCrn~ciw$UhL>Ei0H<JP|9=WwI@K+x;WehPu2Gev8+%^9DRCO@7M!>*LK>v72eNdsq?f'
    'A+vzE2~%~0EWhXpXDMI3MXPDB)cMj(`YT(;JV_WXf>$hK;SxIovT{-;K(HwuI!gG-7dlkBE)s~}C{ctA3_=OhV>U2jA8ysF&wO<zSl_qM4%m%uDa4Ddx^+tB'
    '>4)f5Izw5Z$!$==ZE<m%Ym6#$+!TiDsg_C&ZRb$5Bp_w2dQdC5)oKY=2$+2>1g0`E*c5n#nlAyDmt0ONwo(F&@omubmIBc(|Hdy=gbf-_TO<>!fbc>-xB31v'
    'c~K}{!&)lg{!yu9hhBX#n{t0;!(Az%qXg{ZqE*<HRs=8nr1t1C)jju2?nx`9&8rd&GL*b$&%J2{!-`u6wgbA-#k&(u{@y}<cUno{h!So-3-%OOF{hOTikA*h'
    'XheJc!myq!F<-4;_TlM+B^t^RaKPQL<06m-2HdEiU*1Y<Av-p9bfw2)#!&;N-K~mW9KNs|@192GH%3>`^2O`$h}WO1Yvk+<-SrCvTivk;N(`Pc@X<r8lsWd='
    'Q><Ka^kZ*g<zq~sKWfRJxU6IYyf8jpaM>|V$vB<B)~J?oyt8gSK~jxthQLnp)`;LEJ{tl(!HgpUSByCXcAR-fC5y?(!$_0L@`Ia68Q5yI)lgEDc;8m)p_reT'
    'n1G2PzGAv5(&ndjWUZkW_^SVe_6O|S6eBbNf=0dso0EbaCUz^49v1KQa+8_SV5fG*k9US%;k@A<6|h6}4in4ZR{o>dFA=~D9UHA}f&lslI#A&2+Jev%w`V8F'
    'JU0rNRW;NaxR1n~!+OJ_Ys7UCh6%(=IP5snNep2f8_S`lZ-Ex$jCMtx5e7_bs#}U++R@MXF<EGAA&XWlyo+Ce9-qrY!D;2zff&uqp#Y;|>%dd9aVR)dYA`g)'
    'HE1uMk{4kM|1Cn5rxhN<fjD<+hcT)HrW=p2;)ZCxWsT{h&zoq%=uCmu7H!5BrD<z3Ysl;0t9Yksk>4!>x}C{F8cU#I6_Qel+j(|`nuf12$p@iaOvApYx{-p0'
    'TW>899(e~BPzH7bi<c^0j8?!))SE3wJlL^9SzNb-#fi+iU`nmFca~wfbQ!Cicd~;D&rjv@3+Yl;l2i_nZGIWr{bU{flo^ExgDT4+qmy7gNL0?~y~ZNN7%7}^'
    '$!&Rq)DennRhiVECM{E@l%GDOOwm7F?CTOe66iGXrv@n+9mSNzl^wNLV)uA4hC!ALI<r<WhV^LrD7j!DLPk)B5I8G)EB-ackhA*^;td|g`H2OqSR6KnS!+zN'
    'GwqJ@Hn+Uao^!HdtdA%@mMYRXT+dpWvQ`|57O^u9I>Ue?$>1@?THFLm%*wV`!yM4<Qn`y8ANH8y&9UsjX_^@Yh)JSUXt6oC39vNVMu5$;(&+wO78v`LZe0W('
    'EE=1%kJzvVw*J*xDLvN7<<DiJs|~|ZkPIJbcZLuE#3+^+gscM#COeZEjULn#umP^$yWt3pnrh@0S{6=$Q5uG#Th}i2m@c0it+(S_+-f7gCG7$=@|ieGqSUjb'
    '+scaER#BNq)cdst>TvZ!D)rb4sknx^#cV1I_-P_C4e3qE;f?(l(*px^&9{IEYC)+EQ<TH{ETaw6ST;UzqeI|v7)uQ7Mj=R9nf7`E{YjHI0M!sLaZEEN|1t$i'
    'AaYCc;`N9rr70mvZ|xK-I|6SGyPY2_Q@CF+3x^fT0CamWD-EeRczP>cvmp*Ksi6!XF<$Fa%&-UNMp|O&Cf%HFYNvUY4a+mNmcGccqc4sd;(E&-Bb#K;uCYuE'
    '06Dd>C@(NjSInhZ_p+=cvsMQMSGQps&{Ap&C%34ly&17)yyY|qMwt=pEu&=^ZR45kID36&O6j2nOMAOvh(>=jn8y1};yDO*%2%z0h|S~RnZI%Km+qoy>kN*5'
    'YGgxtk9In<>m0G`lqmx6CP^qJeceBjh56Y&wF7<Kr*>qfl9B`K0cTKkWsS${8D#2U*(B<+pZ_ej<AM@K{75IMT=cbA3V5Raqoof1!65FeRe`B4_H*Eu*Px)Z'
    'Zrdy79Ub#>*4kOZEjQuU-w8l?L6g-z{!s<5)X^8R=oGV8ggd1IV>GW}v4D}Gjbc8diJfAHuZQwhI>m~)%95hQRQminUQF6GsfFpIM&&f6m1rlmlm>)kaIzM?'
    'PTSvbOOaa=gQjg8Xr1T(WwN3h^E5yUasY}67o{6{9)<#<V(UU9Y=`5;9a<k!j9SBCrjb@Q5pO|vF?!X~$^<~RfvHd#CmP*}{D^AaMYcr~!GKWp+ocsmjSUk)'
    '%$%gd&YiGE=2{FXs>uhmi;7}9war~yn<I>N)?%+WS}W01i=eH@QDh9FHf0y@mZ;v%?dY9odjPB{U}R;LxC7`BQx>d9=VM1~OBdrCfU<{Tx80`atne^QiNS2c'
    '%D2g#77%suB`~v7_PV3hZOUqxuoCc?kVp&h!n!*tSJ!V~&{8zLDeCJ8LUt}GWKlz{5L6+>bei%$7Z!oG%Tib{6dX63Y8NF8Llz2X5im^fqd~w*j7bUPtcRx<'
    'D{kF3n@kRlx;#GKE5c`lKG8?YhT1eb#2ao*S$nh|VJXHJYwOS+!SES8wgv;s4GKqk)VM&i)X~V7?5^2FYR?VCH3F16gGjJKA~PinLg1?7A*z!dO}`urTDIkb'
    '^c7~UTt1t+Ydx3)#Ey|2$J8XSAnhTC7$#WkG~5+Cz$`(mi&II)#4-t`kBLquxvQaQrjzO}be?r`dGwy)%$opfi(b(S>@&XFsg;^Q@hwZ}yAgNs4c^Jx2EdGC'
    'c-w^#$fG(!ABzrd)@$H#(WY19d-~40^)5zLQ@SFNRE5$bxXs?O%g1yUaj_*5UbF|aL(+qAwWIg<Srb|D=~1s8ge*kWcQlFAZ*u?-K3sZarGqvw={|jqC=sHh'
    'y9PrO?XZ4yS14G*&Sz<rTlboJO$QTHfjv!bD7{<dcHmYkc-=g7j9C}SzEgtCs0BGp1<@%D?(nXdl+v0iBC!(OrPq#m%8mvdZ^|kiJvLXu*$HLxTI#)}U|AfP'
    'Za2Dhm(m~fS|XVhj_{9u^rILwPAlcPA)OV*gY3&oJw6{q$rHMsW9`HIS`;5VrS-Gh#B1b1Cj-hTf<qxHjK1F;A4XDG(kQmIxuo|~fMF;#U|E3Gkv~{j7OhjK'
    'OnsTeCtf0_Hp^>x@9av;PSg>?B5R%TycYfxyT7TzV>pT_VkM?|6f?(N5{Z>2$zfp(*oI~_ijj0QxSbULs0;hl22zR+Y$h&{Bc>4Hdoibz1_L-{K+j`7&tYB2'
    'Nl!{VdP*a~pur?;)AVK<-^R6P+5oOIaB)hmH(2X{(+Z^m)u+7#wbME{ztUL%(v!1Lyv|s8P)tAD+*s=4ILi1*RJLH`MzLuJ*pg~E{f^-t6g$L9g5#<{iGC~_'
    '6|G?gih4;dE^<oJ*xFAC?`|va<t9O+%OBeMlCVPsW<b1KV&}N+8X7inn<nZRZJ8&cS#AE9wT2K_qbZlxr^8^v%2-Ncxj^YW_^eP?=7C7;sMjCi5+cST#6%oD'
    'TdSH$EvtTxTZP@bWoUdFKuTm%>jqr`T7FN58^v)ua{;@^>`YWgI^%}o<aC?iBVEq}D`R5lK6EAWC&dE!9AT-aVav7P?=aAhmJZKK*iqb**F;+`4X7xO|2Se0'
    'wXbFMNtp1-rF3G=jOCRNg(@c;DM`l(n1*}z?p4*a-kHHm^`KcZus0lY${!$?Qj80&(AXhhE`p6%m4us13zpXlmndNjbJ=3uZ>4S%bb=1<R|vSOL<$`_hbG11'
    ';*kYAY9dQ_;ugnbu-OT*`$3zp0KTX-VswdakIT|ATRed4!HlQVxwdR8?#3mVL1PkOowIy%EkWH1(?$*}Y<TEsXAB*ts6_Yk4><%;WS?DO&ngZhIav+CD_9_F'
    'mGr?=&WlH*y8C5Ilr0ClTcp34QTq4)v-fVxjburJ*snxoS{ovmK_mfY)z)Bz-mR{z>MrZ7te&duX<IR2(E%Vxpom){0wl;>%1knuNe?p9t5&uq)6<w<wEL`w'
    ')&9)R?&>e(?&p3B1VCn8vRN~oL`0n9=jO+cyC1)Bh4KLdoSu`6#c7~Nc}_e}Rt4xb4ZenGyHGUU3!eo3jg4HWDfq;DK#gj}{!wN!>ll)T%=DgnjLVO3ZS1lZ'
    'VPZmN4BryFB&IW!W<GQkZ<pW-_!$foA*2ynaCO?r5u6M@Xyh;i!1(nX?BC6FQGK46`;m&nH#!l}SDwKJp)%@16N!%%ffBak5;w^f{(1C`5P-!<?{SY6;pSvH'
    'C9pQgT7kz4)I0@xQTGLVr=dlU{6pAlOS{rThY-<g%V1yPq}i=nmmrE}Hf06>{JN;f3Rb`|`phO2-5XK%?nP&(Fom7&=lXmDBQKm4qt6UIkoMu!);DUOx*8Lp'
    'yv>xyyWW}O=yUzyh5pbrl?ebK8_t%aOK3y;#2Y5BA_H(I0P9%qywN1KGAZ&gH1N5OQYAH;ZB`grs7Hdh#+Q~bnX4R)e07gHyO@Cj#R`$8<0$MHa&$gBF~O@C'
    'MmE$L`aa(DnC4$#l<TGuKq}2UOQ6!rn?*jKj><Q-Z=eT}0>~alc;8n(-<{9ur>g7!Ebu{_O5B!Z@ea49+L!DeMmV6+f!f26`6i|U!<64(qZpH81JiV(^Rs%p'
    'XwyIQ;EG#@?XwsDBI|;Z_YP-;TtbC)qXKi?XB;(p5i;yiBo^86meI}6ja7z{;(v_DYnNO0k}|kRj;yosp{P-?-3FAYo<8lMQXyJpm)J$H1lc&TU)7MPT0-R^'
    'S019}5U3aSw;g0^FJje%WcdhJ_j6x^Ve;iAVkRzXgs3q}GU;lqsM_zo`u@#V<PWXu&ucJo!Q7`US}eX1mGCUV73}yS$6u2k?jTKSuBzo_u>b&Ti>BKSZ1rpc'
    'QuKUNE%`*Wzfiyb_$o-xX~=9PwT&|#JM;x!E;m55vRijSZ$sNpZKq!JX~#fqV@n&A*mkyFjwTMOjVA84vbL19osP}rtKGeHH0b0YIy1<UdtUH^*y|Ly`!J>*'
    'wNg3=3y#Ctis*N)Z$5gURdpWvJ6-X4_=QY{U%=|CCa(GfpT$%Vz}E#2?~1XiS4A$|>-lBQ3S{l}Y%LA`)}r1kvIlte{m<>vZ@Jn(oO?aVa!LuBVvem0L*=fj'
    'jPmz<R8IR^TVkp)7<eJ^gou9j-Iw2I_}=H|FHl}O{I6#1R{}-9C7>UT^S=nBmr=Ac6z@-p<CBjdP5>4KGRLFBXBd99rt~+=ZlYo7H+z*PzDQD?a5*4Oj7hyL'
    'ZZH6WP~f(Fg(p_wNnK&0`Li5v8F+E^##>=ITAStB1H5jzAr}166n;H%5R8)celVV`vi8Cn&%v3nmRWzaN?L0?372#h*nDA?wpZ79c1*fIU<ic2@sJbdWi3UO'
    'R#b(ef8xkh8Jy)%NB|GHYaSsW>@#mRdDc?D!;bS7a3T$CVxUv8vGA#n@TO6V>x&hGK+mmR@hHw(lqe{pge}~^AD~TchU@7VR{^Gi?6&O^tLc_(49l{$%kxS?'
    '!z?}-aqkYqfu(D2Fe4f+roQt9Iyo+wh3@&nGc&s;#@k0<zYuM8VrunMVYPql_VZgic)`2~V$A!mCJ4%bVMP4+_4FxsxVF*|d1ix$J-jUl?=uz~4v31$AII43'
    'G(v@0_tVds&V{*Sm_P{wdrktC3dPv2wpHsen1pmJs6O*seojaWM_GGSO<GC#7;95iv<sh>7{_Ux@o@6`jVkFc@IKY5tGSfg_p0=%Em9_u@dR3_ntCTQAj__v'
    'I3ZUWRR}Kb0V5B?I&>zK+~gsHsYn#VBHX~wf|M2K(=D0zgD7~hSgwf$DjkD=#)0qzX1fCBynQlSZ76h$c7OZ!i8|m|eq{{q9P5DF*mRuOW47AXAJ0sv10Rio'
    'wzZL=1twl4;{dI8=yN~ZEweN(=p_yVzj|(@$#{SYXs!4B`w_Z$*(T~%Vw~rAdYSaXI~VyyV-hY0#%T4W{mF)FxBCf5HZ6*MV2n&J6B^^9xXv!2XP0Macx=`f'
    '_^TMxLG=d~rnFbfReWjCuKAQQv!>BzfpOC*4Ir=NFl#y}$()I3RZdu=??sGzEtC<9;Qbzl#2ZfTL>FkD=#%OSeAW7evu^Jw9aQw&=)@)!qBr02M*fH^7rp0I'
    '?tI|1=IbE-S#^v9RTb@+O$|67i!}YzbV!GJAJStupWz<pLu0#H+V<Ng+A%Sm0o{k=L$>+;&OAge)9A|Xb$ePBbN8^Y$x3+Ji7of`iFR)T9gCjvU0AdKtS+nV'
    'cV$?(972UH(7!Z7tiqTfjlwO4LjU{AJD&1UqDR|6LLWW1C-HJ{D5bI>eH_H<tk*pm+(qqn`vXxetASrl$hn_C+MF9`&J$y9S9rPF$Z}1C$~B26XGBzc&|qp>'
    'tgnwH%-4aqc~G3N2MZDQfue)`S^|UlhW`PhfgRl3AtAu-H@3avemy`_&4Yb;sG}L->oG9TV_=@Yk-$8U;VAAEjzUKF(E}AQahnRtDb=P^pGh$dhxFS=V0oX7'
    'si~slgTXsQ)op;sKyf@CA0u@<M(TKs)bTKpI%+~?JVv+pIip)V29($@pu{Co5HIQ*f$sP$8YrTxC}apcy_6mO4$(p`H9laVGc$r2`98+Zw7=WFagG`<&iCUw'
    ')dgZSASpI&VdEs9or@MUUwg-J%#Y!sd!{ffrdIE?W{nC{Z&Xgrj<^KOHm!|C1BpN4gnOQQV}+>Q(S)dkq6O?azl~O!h6{L%dh!_c<T2{WW7LyhdDIgzg%<Hw'
    'I{}xWqkeYviM&MH*jScoBl<A5O}xSsnX+cmMx{}J1-_g$3&wI$pd7l|2)Tido76BTP7Rac)G`WAO}k&?xLM0^Cj|CPhpOGKPV90Me>v5#ENS*dD=RxCiyGZD'
    't{k;~O7EXoyiB9kFq%Zy(I`6pEpSI8d}z^K!n~L-*Ebm@VXs7JUl~D$zxRZJnfTh9JbZh`2w6hpaG001h*~2ey2y)^BggjL7|IU7ll=g@`%KP@njtxyRCPed'
    'OARNPV~)Ufn4;@^E!m6aU<`rKpmcacc9Pa#bj1?9l9m%+a1g~5du=lDk_GL#;cT&0KiG6IypC+%x@p`}m7CgdIQq<}m^iYzx~{(QcU2rQf~WCQVtpBEsIdZO'
    '48>rJb@hwYxUF*$pV4L1Cx&=cIu1;s*0DM#8gnM<+vT@Nh*AA9rrgNdGVw6wU(#QHu$#U<SGU=0-PbVt7QSn1?d=2N6jvgd2%_{(%u}EyoVn44Y=2i|b2coy'
    'KW}{f5tHfdlcSLIO}!S&#@np8&@5;|m|LHbP=;I1lT9?7)N0QiyR!v6$gl)rH!t!99nxnk7|iLw+Jr)Tw>_~)X{~PCUQ&Hyn*GpP`}U>QS}?FqFf@<1x7KJ('
    'D_?ymUi;tJ?2}G$O}HGIvkcFCtsX@kJt~G(AAhoC2_wK82LYG{rAEMaNJr{N*~ACamiO5gL+ipEj21I2e{4D9#tsSE)wBuEX<z%C=y6L*w4948l^H_UY9=0D'
    'M-Gwd9vSUE<KgD<6Y{yNY`s=RDzU5)YfWAa#cK1$pGUE+%28~Ro}tD)yHWTMP1>Z0nmB29py#|vV#q0(4!X3HFw<@0$lUWEURn-~hH8f)q#Ya1LSiXJmZhmx'
    'fbGSJ<9k!AY$-Uco(9yCK3k#%C?tN`Mc$|)suSYtorjUrT2o!rnKS7xttk(<I6{B2>DVyn%u@XSJ(xIwM@r^l;2$(emM-J;9Z$`NZENDRHThs|nUYrD<1hNK'
    '?HS@)-*q@mOoUGjw5Q^N+24oFx1DT(z_8Go0yyTCfS1F3O`3V)^)3qWzymj_r((4nO}q()kvP14axq&DdH6(h5o747-W}AMZT%`07G>E?CvcU|rem_}dfbYS'
    'TXC<gnEed<knlaQHTXAsCn{WLKGmCbkx^(D3~N*p!kdaLF_wTIKZ{c&O!@7E<hqZWeiyz{ngkYC=8JJY|4Yn(;dTN!&a?n`;{{j-;M^*=#p3p~qywHT&z;E4'
    'T6*+>kRQ4iOyV;?fBWQzq9PZ8%}R#FQEk|U?RE#wT-<%{N9UHT226-(C|;tMGvOM(=&N?cuTXl<7IeN?1MvfBEwVWd6+B0$#9}eJX?enp0rz8hj&K5e20a@n'
    '>-n{7*s_v_EpHcyoTh`N=09Xa#(w(*|L?o2Vtu`yg6yKZr;=|2WoHmRyl7~R*7LyguM5U=J;&$-RE_V8Wx-rlFyhrkfoVSG%XQIa4Y4xfq7u6FRz-UPX7M=-'
    '%kx~1?d8OLdVtVP%|j%ys1qFER}1bO|5BtX34%dqk|B$xWj+a5FQ@qh_UQNdrYxuV;-yyT@E}lnC?O*#+I`~(i708_pfNyK+1Gw3u8a)5)<5-qMh~|P9cqFA'
    'bYHi#fwzP_LEk>1-@p_3np)04>;++eL5D>dY;nGX8CkA_-<t-ZVPOrl7%HIvDtxV=1jn=Z=txYv4R0(1iwG+yZ&~#Fk&A<_rXD{senysM4gFR?632-{G<){;'
    'iHMqKI2aTBcN+kU&{UQ^*q2|zb9Wqfk2Dx6z8*kHqwbMJoz$>1BCx1Z&1zw5wZn}#aAhsL3l_hPX|>U)#!-*Y6B&Zr)slW64i3+P*w({upNOi+>T62+_%wAo'
    '!*c?fh`{l=rlU-!z>D=o1#{S9qwWG@Io8X1rq`<WJK5=$?^Y*<$#8=&$Bms`!!Gf<XJt8dohj=aDVsV`Hg}*De39up>2sZIvrD}AG6^_Q;!#$P4L5kc`%Tx#'
    'a(tLKq~#LGFY&k~wLZRyV2L@+i}kqp^W_|vd#V0a+dH1<9TB%FNCRum(wKvDWm*YA?=x3p^IWfbpKnEnETR+Nnwn8jjFV{NGjAC_g)bIKqD{G#g1R(v@|oxp'
    'pv@)AXku~x7Ay<XaAfkECn*O|l%lo@GAeQ1ECePY^{+v|tiaeITX{8a1$96x)vYjc@AsoqtLIp8dj@huO0;WWg50*@AfIol0<UFHiH>&fbzLv=itekcOd}~$'
    '+WHcjnlhvWvG@(6E6`!s7#kLjCv`B0p^oXP2{USRgbZ!1DSgUD#P|F98OQu*AXH!FRy^Xo5DGUYlxMlL3Zzg!tP5#PXRVp>=)LySUjVOMv53cgIic!ZjOj*J'
    'vuOqA28&eVh@VO!#|DQ7v>4|7^?g1yhMc;Fii{mPJ`GR#2zDSqCx8H$4iJQ$(He>`a)Wmu+=oxr>f@`QdHoN?fgl!$VIm%=A%9Rzz-AsC7vwk=_tEbD1k~yw'
    'Y0JsI@ToZ-!9-eE^iaE_^Al1;cNE$_fRE3iqnX$X+OnM4Uk5cxTUl#&8JsYWLA+l0;q|vrmtmT%jg6H;S1Pk~ZSJ!s%c5ZLWA$Dz6x(_)Fcu4J!fJ$Vq@M<o'
    'fsqXQ@k#qrOs4dAmWEqw0B2(5j%XXljnR*WNWZ%yng*7RT+y*XN@4O%zE#^NHX`ZRDAp2ZFI#S^HVZ<t+r%2tn8;;m(%$7_cEc!ZDailP(Ito#7p3LIL9zEn'
    '29vtopFA^W*@=mvaSf;)(0XOF;cJ8a7c_nA<}>5W{8?uZ{np{;c`ymC|6m$xREvN}3a8<Z#mF>>vs5zJ2?M%YQcg{^%+?n}r-NcFb}@LBIM(~9av<pnDV`-E'
    'Id+$>M3#6VT4w=+jO`n%4h7jd@b5m8bVEjA<M7DteC}+uCdR}niwl&si9gVTb#ZK%<Yit}!i7xNXeGV-s=ySMaOD@`<~Fe~_G{X3-Ej<TBinK8gcE?R;@$R9'
    'yglBwfxN)%KL@!I{Q?sC1|7xPepyc_;lCMUww>zcYm71b0|qmTrHXQw8a_J0?VCnd?=rewI?qQ%-R7pD=!G%1nEnV`<;{05zw1RmOqQG37-Qz3!C5rC5fK5R'
    ';RbJRiqR!r^-#L@b&+32%Nc5$%L-PaUh5sw{>Z`jI_?va4MoiU)#-3T@53el#O0_ySv@aLyk%Mgp#1EQ`gc&@A5+KEbUgOm7+x+tCu1{lPsOgs#N`?+()6le'
    'D!EBjt;*i>=NCW|H$y=2`SUNA^VL60%jfL+{8$2cyAS}(yEEq|jZ&*D{OMM9UCQm52xX7#D%>!4HXVwR{LMUHz=m7zOa;Tb_HH~EU+}ly<EtsnHC@9Q2~#fC'
    '({ZDuEo9ue1qPyo<NJc?w#hSVh!Mz(El|2rW^hTPX*thVX@rKR**Pp1>i9PcRH3u${owUq;6k#P=W>dt)mQWUK3A9MbxdZf>3+AEZsH~Z(WR1UWaV`FfH`fr'
    '`7<56%^Sj$z49{m<u~6_w%lUPi2twa3SRV=JI`ex-7}6ZUc*_s?gAf(hpTWp5givXP}b;Lw(cswems@qBcqOR+>V|>Nh9IS%RhZvb2$yn9TuC8s34mUfov*f'
    'SV|Wz<Q7$~a^Xwf=wf@L5<7U)mYBiWJtYkM(qpy<Zpt%?1qbE2feY#fxC^HX9b&$5c${abx#MQyI2Y6d>0^L|Fy)cwyB<M*=MB8dg4jQ!c<%6zcNF4|)^9U_'
    '6cy7;-eeW7jTgXdx)K)u<6!5FFpr($_(e@24EWeBj=O%#0xhV%uK<g$z$DGk+5#JT0mAGoD`vwb4^7%XY5R&^X_rHoh#2+?4~C1EkfX_RNdZEYjCGZ-Vb=g3'
    '2q$tmy_n||+~fEPH!fi@kj^nW!d~$gzIlL!qst`TwNBeV72FpIwIzrUq2Wq_eR?ZyXEFczlax78qrFbKnd30dr`uwzh9EXv>CHT48kz-LXBQMiAp&#U)X0Mi'
    'MfPxzES<?Ay?wGhA$x<?XtQ3!i8s3;2^k^dF@nS7g(>DCE8nth9_%*yPP3lt+V@m9V@_~c2EP_iCCl!5(f6AL@oCW4_udxnj#aKccFa7nRdIH#W}afoE~W2B'
    '3<;z^dz5>8dvm9F{$!Q2NDq{?-gqr>korhd9D1Njj2pz<tAAm4o6{leu`nC4GE4C>zZ3`3U)C6Jbsg)ci?b_?FiDELixo(a{fhN)Sr%R^a88wZ)~8~g`^`P!'
    'gWu2toqJe)#uI7{CxOtwe=Ld>;>uUweo;ota~9rM=>;XsHj~*=9hls;eL7-0ah1Jkbpw<A%vim~rk)m@E@=c6CFe}LoHok&Xf&p<^1QZ@+N7R{*ybDS=Is}5'
    'v%}SnwCa76=ctMPD5GlX=b`pHwOA^Pf!AC?rIYkdRqw-ua&|}*=jMn&@!u{PgCj=ju+KNM8AcwTWx#N4ioSLN0h5g^kqXbDhG#~8bSIJt77lw6W=$KgwKeRK'
    '!wgyZgiOr}lWq0IwzYjCF1J>s*7&3*j&}%G4hZJm_|B{@=(id0+ZtM3pzmOi?<N7h_X_SE4(z=vp0|nU+LV&<oO<+^EXX5WJvhwNuE^l`3kd$fqjx)zyBkLB'
    '4#z^?H(K|ONZmhMl<tNRy7vw%eOP4fy_-2CBKKiq*((zFJ(_49fx9;Pb^vD-Q-wG~*g|+i@n}zcCY_lS^ZG1qmd&0Ci50!|eG|-(8GWh!RNS{FBHE+Q1s>TX'
    '8-&u~@~Y@X#k);DLo!3t63c%@j4#oVYTiM}A_&)WE@<yu9Ct4CHF3Y{%l#A@nJuq-bla;f2gzHj0`GlBIdITqujNq>0GoIc<rRG%LdTeR2;4ri?-CRMWTEMz'
    '`8aUfh$b@L6VG;mKXW+r=9{@^lqm-D>YgMGuv=9oCHiPtoOkr*d@RLL<flCfoetZ~p1yR$eW^icyMwiTa-}9$b<h_I2aCm;UCoAEB$2jTe2$KyZg^wie_aut'
    'z_Xu^?Bj<*l*ug~KH-X4{UA_NQa^^1_NUtXvD5qwi=S;bi{tCIF%pQ3x*8~1%crh$4Da%KS$%^N9^Q-><FD51C0=FEHzbO2%}+5lR4VM6HkvH2k;0HBuVA+T'
    'Fl@~aMIRw?M&_H&VCPY$6$gaWAJb?BM>!T>xLyF9L&FiU>6w;`e8J6fG$0AA=uBM;|4aK`PTm#Ik!xSBgeW6I?bz|__fc!K$v;8UON3l$JyoGAH<>VQ@hL>e'
    '1BgYo4hz+$0*;rz>qS3YPb(BraB;u-_6x4HH8Vj~>l+K&&9s<}Ep^B_>KkS^ZaTA?S@H~bUF<VgdyPLd2hx2+7{eR0aG)oUp^soVdzf!flZ2egGss3minKL;'
    '^e>_tCT~WzW?n~z6&X{pB9+y!*MVWL#A6D|eNs@qWBiFYt(~+YItuBR$@t&zi%Qu4OA7!^NIXe2L#ZB1@XG=BsNUJ@Xd8VA|7*9~gNTXV$aB)=z^o`sZb|Nf'
    ';Q$sG%$2Gr7L{o(Vv^L*CM^0xu?8mcgbXW64Dd}wbeQZ?yBB>CRT~WT8e`EkQXHtx$0jLVQbC)1EUl7VmAAtah44_W2`2K`H+hO4y-LYVKJcCPiJ#N*z&GRG'
    '?f@)rZ1zaW;@CUYO|*`sKViz2516jyK~lA_O>5_+kQ>&GoNY=Xh=zM~*(Xx@CSw1TK0X1-JH^Hv)anfyw53qZXvqiIC%iUm#8)%epzWEOu|47Ia(xTNe(_Uv'
    'mNP}af$c@Z<#OgG@X_+F&<nJlt&;nQmsSFu;EvuJ;VETr?k5+j#w9@WzRZ{tZ{7B2md{sNe2m)mPTS0uO+fcuIO*-I?Pd`oR%ZOcu&|*_KZiBD+u^{<_vAqv'
    '?o!4~GTFkp%=a*jj<Qtc|2X0X+i1CM=l{@$lBHS}DSNVP%-;58(j}vV##7>^LsRB|vFsM>lsPOzL|-g<{p(4wh*$<19rOaDTV^8aC2A5_=*tu7C7ht(g{@(J'
    'lcQMAvm-s2g@!e_+R-z!EdanY9qzOxGAKMhPnxbhKv@c3KR!LIAIw~O4!ajP@X}p%x!g*D<g7MDN#=Ju+CH3Ln8B!HuQ{RaXQ=fABS8U#fhki>QEWof-gu3+'
    'B!CMs3!i?HdTx{w3bI{}`Yp2TM6SFdrCpWJ`kvyp5KaT&Xv((`yi4_qU|4r(WNHn{yl}#p-c5X&OpJHUZCnM$H2RIl)CLeVqPg;3vmrkEq)mPD#Z4rP);uJ9'
    'IO8|7r-?GkIXeJ~>_%XNbTRi-x@7Ns))a#cb0N!sNHgD27D9%_kl*>s+*YybS1JLeTEmncO?&JNn)$YQ2k3igv6H{RoRS|jpX7t&k|f_hbkXG+sx@(W6iAPm'
    '@jLN51mB`rk@Tb1tkbg9*gRl30|pxqVe<gt&Ys4WJo}TwFoIXR9F*0QZmYK^d{4l&_M8XXj>Tk9_e~HU-T5I<aZit#8(hByhhot>TtB~ATrQT^Ebu9XZi-s('
    '<^0^WEQ>ZHN~(T-;v4Cz`IGD=E0=+jl0na1Ai7~QHzBrPhKx^wA*1i^BNlKJ@7wb30ZhIGwY6<f_Ckd7w@?0rO=>L$Y-CAOc~jU|I_5O#ezt_Za;7a}!OkIn'
    '{C_9E<@pqy6<ef9Z^>zwignVNwlFlMch%aP)fi5J`29IzxBz%Z-T>S<vDMQ_$DI{DnHMtYi?{=<v2UchS2N`Q+?%7b3ZtB3k;I``&AJZ2c)#)Ioo_Al-ao6c'
    '?L3lT_MSoBg}V^B2!u1=$6fF#3Rj~vnxxUSl3UQrjPVpcVJi*uGEc?r-UK(707Gi;YyC|cowp`ze(2kDTuo3D%G_|~o3cU^toNe$U8_kF96R^TcJG`wCZoma'
    'FOypBIsAphoKk6=wt@Uj9=H34j7KHMj)u`A5Oyg!MRtx9&O0d?UR~)k!-~S<;iiJg;3bUIy$B!QqHN|9%-9r2NQq|-`Q#Irv*}>quqV3~qJA+BS&#==6l;cr'
    '0JU`|kS`Fw9ud0;bLeRGgQ<;uKxz>Ilktt#NTBknTCf-^jSylx@^J*@mxP`oc*ThKn^dpoyILQMr~<9oI~MSw<yp&-L<&)5;4h?>bkt}`MZM8|mz7u-V>~Km'
    '`9UKJ@X9;u@fA^@#prZuDsEh-C37$y(R|V3GB#r5Qj(E8yfnFcl4C+oJ4BRCOlx9}SWY}U6E+x|W%v2IZ@Q2`mI!^EY=pj8NISAXgk<?WLWCWZ9>V92cX+~x'
    '{d{y(P$ez$Qo^|E>|(v#tk9t8tTkJ=27y3p@GS!&7|oW}>BIEgBgQ>&q6>)xAWCuHs~{qYYGjPS!x9TKDf@IZKsSVI@UXnnd!$m&q82_1?pb_3BdS5hDU5s@'
    'U+;F}Js<M@k}vFP<K7LZ1%@v~CRkQ0bWt$FveZQqX=0#eT9;`q>qrsT%j=Tegh}r@5fj#EaDm|@%4}7vu?lCuL^)j%&+a%dUWIWS@}h7kF2q^Lj>Uz0Fb09>'
    'ALw{pC`d6pB|BlsU>X*z1q`Zhj?ouMR#iDXg4XN)=g=Ou(C><=mRDP(G$l(j=2`V%v9-eI+-sbc=|UHQqSyN66<=_>%)*)^Z=|lX-s{1R_)xk;h^iE2RRi$#'
    'u!a1!z(2CY)AdopC1`urqe4ddTqGxzV*eZ5tH9x1cX3heb#YM<9^(z4;qw!pRRfSWTOHa9aKsGz_Ou$ZtUbMsRB9{uf|@*!yz&4FJ?~;kF@rT=YnCvczg}Dm'
    'S>~fqMp=g+pe{R6V+pO;<~5NjbnLx-@~7F7(!;nh!-&8NY9>em!%21&vT4(9fnEnLfgGkP6uyl=c*<$*@0vGN(w;Sq?6{j?v1(JV%VJb@#>@vB+7{NN2GL3k'
    '=eZNl#OXoc8b}+wIC2Z*W#X^~TH|-SLd|`!EK0SO%=;!#($nl{;oV+-n@Okt7(>*UQGMnZ#D|LNW`K<X)95EdI6jRQc%f0D2KiBBS>k>$fKfPrj8GVfm_|Dw'
    'a<yhKAV=f4A$oGzey8pAU|vnP1<M)9Lhi7WyT0I#fFU)UpC=%OJ`$9+tylVPS3dgvh{0~JmMaW-C_bPFN7qFfiBJ~cb~Wu2Ry;G}>)LTloLRSD_*AF%XayRu'
    'fr_Vam=K9vd}i#|;hJhTeL+6zTqq=4FkW6SO3p01`;3tcjPS*u;lMzZC|ToZ7fF=!c2^T8)<BH3$o(jkz@7(W(4xGlZhE?$SjlN@<m*Q*Tcw8@y_P==glbjI'
    'a_mr=`pZ?}R`7@Xs#vtNYcb{|Onr<999^%L-XI|^=`dHGzuXaKhNzcSW;k48()h~)6F;wcxEhu`&2pI{?h0gR>4f4!eDEL}i8maouO~=!gG9)pfau_EzVTev'
    'T2*RtMyaATnWG%q+9Z@kopsAEG0KIYsI?(Ztd0hQQ|Jw=HWO398lnu+eGX^ZHsq~YCV!SOoz4gdjH$wigYnMdJcwc63WR7%f6R$P<eGG0==_^aWAIZO^C?bY'
    'caWs7Hm85GG=dhIt33i*O>Tzk=~&#Bc<T7a6o8^GT0D?;uAHMn(j<$JCP`R0f`ki8F)C0+Cz7o52)fKM&STlB3t0Ar-tAbLTAD54KlUnWpoB=O1;#ee`*v(g'
    '+V@OWD@I+3;THYciAa(isy9=d^UoF&I$%VjOQSql6_dp-OW#bQbQ*n`Mt?}7zwDuhqiP?SLQmXJ>YIte!vc@Pk(x(~)xt!Wnpy2u%PCqP$~N(&zj8o$GiU8='
    'R(CIN)Xd?QOb#fD>A-B;C_AA$fp;xw*ECfS4R=J{;KE>TkD}Wqi<jjW7jSUW_6jYeh}KS?q%8a}-;mNWc7@c9Uw>~iW&)sRZ{yo}9JE1LjEk*mnoJT3VB`x_'
    'GsLy16-KY=erRQ{`P73_b!auMM<)k-GhNSvpwzSpD!B_K4g@`a+|nYUW(^rK3#sW-pjLn)i%}X~mZTlOEGb36EB$^=pccTDzhxe{7wJjbNxSK(A)R0!mVn{1'
    'ysj<(o_QBwXc+F`fu8>85MV{wVw0oX($R~fjAM?z)ZbHy&1c=g^XD(r<0(ESiF1XoO92{UUt1XMY#SR}c<0V~X~h2jdI~-2!M~lsAbJMlKVzVK?EeP7afx5g'
    '&0X7wdhoiAl{M=B3SzpXkSlmO-&LGPh6EAKBh)|zJwt`V0>VMay3}I9uAsLw<ZAt;dCH%}@0;({X#i1bp$`uXj#<2<+**V^4FV4QBOgg~Vg}$+se2Z>C3O3c'
    'gn|UHcB*2&k}eMZ@x~)WnA0O_ugCvBJ7qXDNW#!fipJX7pL%-x%YHanD*f{)C=RO>C_XYwuSNl@D^JFmQ5$a_;so=}pncj=Xdz(ffkx(Na)S*1dQLv>66g8m'
    'fSpCepyhFQ@KbZq`}#Eu48mHY!<UDdR{0i0nR*|r#FAUAiqDkwSZY{a4U+b>EH^_e<eUERR^#x*6xK_CYE!6=3~Y+sY_es`_n39P_v)c`V;DU*sx>xC^P-x8'
    'V^o=)o&WTY|HFU&KmP51{m1|QKmCXQ7&&6_fBxV8{XhM$|Nfu;?|=MH|Ih#VU;g+1F^&G|fBrZB^l$&K|N0;Q&Hwj5{V)IZ-~GS;<v;v8Z|?0KbUkoRSW|*y'
    'fdl9eK(Qw)-N-)J7<zh6w~QfK@Q`~rJm)5~u;@ArPP3AJzw2A#OqI2FMOdSu?F^%--Fw5T!Rfnr;h`qABRbtNqPnw{Y!|R3Y(G3&tww@0<UaEy+G|KeoT2E)'
    'oA{74n*yEZcD}RI?_}@wqRRU&+KXQi8tS`U?e{sz1s7^hqw9vg7jNWr=h8!YlC-a;#dSOp3C_6PN{8G&I+n$cUHh8X7NtdP0ZkWf^0jGNes%*=$AGQyvB#_&'
    '%A1+HW>uaIm+Nt{4vf!^iNhoLOnTA83Y*TqT=$rRi6>Bx5G;Tf!q`I;AArOyJm*+1bS2?+g?1(L=|Yw6vv8NJEV1_Ud`q8s*ucEv;6^C3uEXP)Ew53hg+l;T'
    '146DQQWzFZ@DedaRa(>;s70Yf<Mt9TD+YloG>ITdxs4b}Wbq6+ud>ZbHjNfy08lI-2*E|Mg5TWI#qhf)01P(XU>YzmhCx#}#rOh4a%qGEA!s<I_cXPeRmdiP'
    '7|=Q5$wus>REp&q{(;aBc>HQ>XECx>0DRwTI7}1oFU$uto33I_L&c<7%#q98DcFVb+1n>{H3>5z+~J-Kb}O)VLCd!qv}mVdLtxwl)fW~a>EI;6`&V1iE65-V'
    'u2k*qPc<#F;X1zrjC^7rf3LFoNoI?a)K$_ZTl>)T2B5*BVE^}6n8wRvz;!Ik%t2~TI&ttWi(xM3TWqY8d302K|7c$V`$sBgi-tSNod~olH&LS(aocMT2KORL'
    '!Ry&6_~oQyRMO|wt!RHe>D+y|9sf|9T}P~6(3b!8?e-6_(Lb&|e!iTIv&{nkC^utR%tbNd{=KMG7{lMqc^`T=hJTYD<w;tOmy}{IS|a;|T79y@CW{7)5QKI<'
    '!15aQUjdeH+atyJ@|?1tMDehczRZb_*`w0H(635WXOTv{n$~@$?bn_=y7tlT4c1@qcFA6zYwC@FPcpzHYhR=sRApWv-yS1}XLba*w&#2{(FXUKZbkMAx%K$k'
    's?#y*ba)+OPj1dpBi12)b<BP{v0`C`B@rga1g(v}vDV`$ixH12y~@XUG1os4D9~qrENo=BqX+2tGsC8%Bduj;E7+C3#wy%1mHH)|8M?gzS{&BFa)^w{{MDm_'
    'I>xrrZXc6Iw<-LRHCW^N0+WnkuVqUHAJ6!7iV(&nlaiFTOV}f`eAQq}GiV&#?m;lBVN4Fm(10Po{gomJ|JL<&%UV_#{pQ@BCVJV#IK)4!-SzmmW3{VUH`I=$'
    '9wd9>zVzhj3V`CpsTh;FCA#<96Q0TIh?gH|cW~&}Y&aikpVSX5v}49<$80|09RnxJ*xfSp_NU!5yd!GUus+;p*Km;9w&CM?*nQ)f!j63Vp>_@jf6a&V!S>FM'
    '!FhL2b5kUb$2E>R<=F2S_&6MF*tatj#YK8U=>|v)gG%8D%<TR~%r2?M<wZtw;O561vJ4!Ap9RdJuu*obEu7F_<R{AQh#gnaOVhr8Sk_pXv-l>xVL90018UmF'
    '*&Rgc=Yj4T{P5kvJ{Z=TwtRQ1?<}t!pKE3XovrfGB(8Th*u!8f++V{GZR=UHeTQQJd?1@P<c`@jh&D5>dX8OgI9KRltkI*C_|0Q<)c4%#8csMi#83KOZ|7bp'
    'v51TibJiN=D;xpdYmS!7HOwpeXlCfY5xipCHi`@5hGt>Ok%O^WOwbJg9%Rl#EjW}Q9GZ!f2+?j5-sYB#MZ;@j3HYG?h9d~s{oc!j#oy?ja}j_~u(z(mEv9P<'
    'I<WyEG!GDb^B{ICni0r$loE57P=sFdz0ZVC=`N;obiRsKoAL0U)4%N!Ee_WCC~Wy*1S|9z-EBw7VRlC~5O>_X)|i=WN6wPfo|CpEUb8hK+2FzU4t;0s;03a7'
    'Yg1Q2^oG^Z+iqyYqnfoCg&jBCM(cs9I}l)GRr3@dInFk6Te8<iSTr=wxP?7G@cGVr_&u9lS{EawqINw_+%Rozm=rXjI71E}Y7e|ot_^;$=K>MI7e?MNZM#`r'
    'PhT0&dz;D(DWmDwVs8yk7dP1Txlz?+5pLV#={%;;dM^UT<-+jbVxDi~TNZI%m_~ZTV%!GZq}rmVL@;o;!{zc4=xepUiEnu*`o859EkG(~mfJwYH!X&n$wb|<'
    'e9h16eHuI_!+8DV-V(T>DrzrLR}`Q^rcnYCdG*e{V8Q<`Wjt9gpi2r<0mQ$!Xh(6U`#Feu-OoKFn~C4Guvift-e)h;=(86dXs9@;fpqom+(?EN(7@l?8L73o'
    'BUAq9x3G#BZ~_hb;o&+!b7+z<160gC049Kr{1Mfs*CEp>;lNBWrCClc6o)N0WqnDdT7!2@sWrmglXFj9lps_$C6Il(DwfOJ)n>xBA>3$JMYrffVcZRE&b`gx'
    'E)sP}Rl!{`U=LA-^P{87GBG_d6u$!7@B3QU3B8qDuFJU9q9BoZisAAhzG5<Ty+4j?NaP*P)2Q2yfN0NdxHByV?<ajr1X2TrN1D5ptgxp6GRH`He5+h7;Qyy}'
    '#`1ySsrnEHzCqZoGe*!*@|!vCG<Te7;0$wsQ;Y$Xuk6xV*=1lYp%aWa9M$#rDPS*V;N<ss)e>-gqFYfZpGM)wLTJ29>WYY8maB_U(jh0xhOJ){1w-p_X!}It'
    '*@0a<^>zC}*@rXnxcJ@rbYf>`Cu#I)rwi+L*8TJ|`uz-k3<icPU*qGM>0z;WN(QBM8l9Dl3q|RA@U(>1lNyVWkfDr%MEI&b>UWN}x^DQ^(GBm`+3sqFYiYwG'
    'u&ov|e9&%%qYA?A5cOow-NDfd?eP>Iy-sL#%ISx3P&4{(xv3q!V2)}>FUVC*&fpn$R!gbTfqB7C{0d@_@<kMs`!?VPhy7<Ace-l<-{bVy@6-^{uxpMB>zMFw'
    '1@pvs*puR6+$9yQ4WsJm0jT;g=*W9lkBCbL0UIAA9xr+6AIZKw#jp-Vl$TLH1k{Q^dY~wJC?sQH7{^jex{VS0igkf8C5Avk9|g)sf$~wHd=w}j1<FT(@=>6C'
    '6exc=0;R{aKpHGt+{WYKX`|C&hi9lpOI~ZJ)KPRbB<!t@yS?3K@%Hg7{$-xUTk|X)pZCY-N$p9z-TNf|g&o7qPE_tDy#I;%K@QYLCuZH5`8Rc7?miM9;85fU'
    'vQ6%BBx*4BKN3AU)@_}R_4)G`T+K>RrP!Ni?9tS>?)Htjm!4H?{=;cocUa>7+}hTyrET3R6&8#6rF2-KpkF_)zEukgwvVoLtLF(3BzRk+LTjpT-HWPkKa2{M'
    '$1f9|(7NP0vM=-ib|JgaMmg5}(dl8mM|&lrQATuqU)&gPT3civMoBQ8mbdFNLyI|vsy55gha1ho_Ol|smtMp#W_Zm9nN^w3677>ctOa))*o|t#ZSB#m{-NF4'
    '8RclyAerkZn9sE)h3=>pr#@s(+DEMVm1C8_sSFrr(%$C_SPmk}GK<)7xy3No$kAq+r8;w&e)HYS?|RW3Z)Jez9_LkF7L|VagLZ=qGy;n8A14Fu37SvN^7$&J'
    '@5p5zJMA%dWA}B}KHY9?a2+qWjvw5q4NjyYBT(pv<?8}4H_*n`4ih!e>z)q$?XkKyc3qK%6n5*)2?eK^cpQw*%-K!oJRjnD&em<9uQDlRd6+s*Jy?n?!VknQ'
    'l{iY=hv8y{f=#m;%y1iVpIMSWT+}wEA!sp7k48D(&QW>|pc=(?RTU#ly8(DNguO@ou+!bHFhSAcJ~0^gvx{ypD9bb&p2F7Au2nP@4;x0)>`+{i5pM%Sgq&$@'
    '4$$93CqUw>cr&=GmGnG`bO{SWX8OEG)QHtt=5NyyJ#FnePShH;Skhi&ApVJ<L|*uC_W-w%{M4clZU8c2aSBA>h*7?XmW$a9Y_)vGvb<sH6@3^{Ud_EAaZ1(B'
    'i>zc$521m*pZ3)vYg}Mv#KTaKx(=@QP}&LNq*QlKS9FaXPxsecZ`VX;_kr|vMiV}398$yWrDY35Uq)v$l^6{${nhuG_EZ)Z=)%h6uqVyq__Qd|5KLNYp!gpD'
    '##OOZMw4Q6$+M-S^K4Ztin8>@JY=+C0fa7|R>hk3n@9neh!l&CYo1`Ata?}ZY*QFD>5W^L??#e>M{Ij1vh&P40#SvQi#GUtxd!B!(9SP_C>8TSx^?7fc-sYY'
    '8ulqNM+%!q!Nlzh7e}?YIJ%$im{3R{<@*s^#Iuowor60YKSy==>1Cc3UAJ5yd%@N5Ue(AvJ1%8|1Mhds4f?JAZn^!Q*d-)7z(dU8+{T|ifPOtQbYU~cDm**@'
    '2F>BI4FFy~yUuS)47!LsN>$2$0@}xpCwFQ_TDL+^xq^%j3!WnwUn9(p2RY6+GaLX22C4deT|@BwS6=>61F?>BK=tJcb!s&5MFe95JT8%v!;V8vY*Iwr{0H;$'
    'FkKyQga?`={U~f9_~g}|!%R$}Fas@10}+RY;b`UDE{hpXDt5%cb#VcNgm(--AqGX7DC3!PEt6fX5^YUlGPJv#Q5@>h%$29uamI%8j`yQ32wG7rn+aG-AU&v-'
    's|e#oM(9CJEdWw+jKHbKIO%Y5;@BuKFO3feD6_n!_;OvGBQd)6&MRA<r7?fijI|PJdyg~IC_WU`bx|y$_$2jVTQy@ytIIM-EMjXSjv_sf3|DF|*3oZVtVbkQ'
    'h`Ucjxlo%Ov(4wvcQiUlnm5Wad~wLlufd-3O0(%42?Xw`YQ`opWszWsc(7=(#>xOD0E<)2Kpdoh16C0Rcw7*zPJ_`oQ^_v9_FWktn9jhQYnnpUJ4&qO!Ky)c'
    '64|*<qg<>+OkjdYxx?YdGh`h*?IcN6fg4faMlY~OcYGZZdy%M;v2JqDE;kE0GDt3`bTSjVuv~!{JH0BzmakvZth!v>l0e0Ddass?3zifo_N0GoXQXf7l#D4~'
    'k~37c4R(9ehw+}M56q9=F=B6sWmr;>>kGhSE(Ng&!7Q$wwCm}5Z%;=XpocU<nn$3BEGq;ZCpupl<PQAm2@0}+@sBsqK_GDWlGD2L0&}{-Z=(91gcBp?^gov3'
    'w=Q)KV0WCg+ihTpj=P>o_4IR`uPUz0v+l&tv@pV=)ihcSgjFyjI`+&nl<?j0SWPKIU;P72&5W|BVG1<(l|+lyZdmntQ@!v#`o+5wjj?a6ljBwz&H6f--Oaz+'
    '`i_D#u5`}nO%58~m>b|}5Os1IsYy|Qz|?WD>5=iole#!RpN^&;N#LnrJvqG53@C}`aeS_w8$qVxlO7aI{d*+4jW>cjM9YAVLfc(hyC_jKp&!)-WWGdKm{gjw'
    'gd%`I0X}Z-)y`!`nzjX6c|;TPFfZ*{JHWtXhdJgxW+flSq@<BK$riZNk&3akOk6vWT8*Bnui=ia7-pE0g!RB2LMj$m=z^6-%Wp2i>zVP@gT^Jiw0Rk(skD8D'
    'B940dP~SYh2Yzzpv5`x?q2_omt?>hEjGq!}pqQx%e7y+f^XQ*__IdQrnP|MnBIuySV0+Q;j=ntl!_i;h?LUux$1cc|ekV$!FXPb!1V6yS{%IQhA!dwp(j%aM'
    'iHXBbdK|hATg^@{dVTb}qgO{S0rqPqJ(Nkmn`#ZgPK_CllP%Bbjt&aENTZiA9MtMn=d%-i{TiuFe+4E-y#Ko4k$CmfnIAPZKoXN}njx8Q5$&v3r-P5j7|HAS'
    '$;W$d_fOUY5(J9?N%kvDS^SN|VnkF&&=;+m3el5_8K(SuLKWF`(yc)><+mP)*LpBN)kcn}>U`IChKcVV&U;jx@io~tL!P6m<KkBhV^hZ@WWraoSd}be_?9x#'
    'OD>_JRl`#Ji#`etIR=ux&MC@Uwah9_<}Tl@D{OYU%JSK2l9$<NGsk-;f(p<XsZU^e;2}CT8iFysed{pK4ceM_>RYoWV7gpq!<Doe%A8@^k>h`ls!5=4R5kkJ'
    'D>xz7bE(a^%GXm`160;x-kxL9Z=}(@$ji-|WNOXQ7R1!!6%>Nwv8`UeL8hL-f+)};!F;)@K&ZVf08(B?Kw0>WD#ny5QR$403(c$>be|2YN2ShQN89L2_#cSp'
    '1FnQX9RzVFW!n>kKWA2YSm^Ba0LRQVOff1%VGK!1_dGBXUXtJCv}*$J>t{6YpOg=LR$3yTl53(xnUw&E6i8{MB>+3hoOe|QD*~)}mf~;B!A?uZ(nC=j@2G@)'
    '9dlTKP8yuc8N(sGT%#?RUCv^x*Xhe905|}l7kWE{6N_&Dq<tdF(1UBZMEy9Q<y9=2BUxtEhFWo#YABqR22KV>3qHTX>*i@$IN@mNE!otc=OA##^bGyV=fiQ{'
    '(=6o{K+Z1ZZ0JrboOG)B<H@Xf(Y~6%zU&75tHEFr3nS_RkZzu1OD+b-g9E3ClgPMh%s0@L^C|L<EwV{d1{L&(Y#1|}0IY8Lf~(RdDlm$xV5a(j&)X9?fK3Vo'
    '!^4{kiO@b18$U2W<1jRuqyy8yyAxDWHf}>#1+Qa$=2PA_Ls4$<)W+ZO8R{t0=yc#QcU(=knBWf^p_JgY(P#IZ439)EPA{5jc7D-JX-J|FO&?y+da?0Bo4wIt'
    '+lmV2bAq;fq^sSA3eZ>E{x4I$E}?jZycf0xXSB{VLv3wkVg)r7vNGjmgeomtS*qg;vLqa;S;({h+MCrFxuEywt@mv6-96V}fvBuKF}zCd3{0h7GomQQz@EbT'
    'dG<Z4%<^Z|psor{kv`1loYZ012>~&Qo=NETc9N}e09=5C;|Ra<u{nG2YBoG{1ER`1=9@4{yl~lQXYjy#Q^G9p9$}G0cB}cyK{c^#@bk`|#Al*ad3j#UhD&oY'
    'sOdf0<n5V=q4^l33WUkmiX$-NiZ-GKkLTgzdH8r9KAwk<=i%dd_|cz-+7*wv88v}iJ1$K@S9hSoV+AuFjte3i<6bmeE@#pmLWb1R;)IEqTe40~Y88D9+4e+$'
    'C`Ze6fw_)hF9Z7VoGS7;W1c$BnT>iO)E9V;Pw&~8QVklX=HB5Mnv-(F?B8yOu3g04tYq}sYzoz?)AO5*Uh|cb#c9zh=J<AG&z3y~&oFy|ZjgF{*-goJ&G`b~'
    '>DBn;@}F5rb4J2*Sa}YrhhQc!7l^S{ysfs$11pP<J8d$sGU_{9#}9+FR%aRgnJ!@FigK_!2A=0=WaDRADw(H7Q&6dQeg&EQuJN?IhuaJuX!Vjk(M~A64(gjg'
    'RSQ8Rg?ZSvq*i_RE&X9l><$dE5>iDS)fRM>WdUY;D@>ISpzRI`_kgX6Iax_>)<qwYp8#az^>PIpuz%7%nCuUM++{-L3&?rI9d=a|2>Yf=@qxCE7jbsfA91o{'
    'PNMPD+z!__ot}5i0lJIeyxF^VuyFEM^UC&uS<jxe^PEci5eK5hFAF#^MnvWRlCR0M?(LHwHY>_`gBZ0c#%&6i<eq0z?av;4#6t#eXi7#JKqhBi4^)R<?1ZJm'
    'arG5!S22}@*amD9nA;z3-LxHPEQ7H#qu)j+ZRawcH6}AOo^KW-cI(yV)r7)i)i<CeMuO!fhjcwD77=BjJJCn9z$@7dQ)-bNM|2#;tWeTEpDptW?`OFQT8u;h'
    'jVEwibdkIjw~NP7CrS3_ejK_DLXXiUXy>H;Dg1GlC}B}+;hRt!5nhR)w#;hP*Pq=7)O~ufJ=yQ1I=rn0=hk|80c%%IuLRY?UCa(Krkdl(dsVE5OE|9HtT%WV'
    '<g?i@A6@dZVSibJ&-nZzvq0ep`A;B*?8nX{YgapaoM@?be<}Krz2UzYW6Ym+HHzy(`Pnl!>m+wm){69O6^HzyDGtOdP1cgPuYFI{3^RMBy6O!JN<)!mj~5J8'
    'W2#B_qL^lzj8Xg1nW~CjVQ794i_8BS{I|N$_p}1tf<*}=Ohgk{J*LPaHt28PuXFMFk`o|kNNae3_5dIy35O!HQ^NZ{tX}ln|A!sZm52BlW>Y-t>2vCLcBg{e'
    '<DG`(0;i>z=c}0M(W7mTQkN>wscMqDo^W4X?yKI=3Y`ql%|4TNt$;M}PKoL{>xEw5mUr2PteJCtgP>Q97Ey**)=z+{pkVEYKkvr#c+Cu~$1yXEI!P6{gksQ?'
    'qkij8iiAeL=Z=1uI5SKis%4eW`U(_|gwt0xY}u~Ob-BGTrUTX6u|_S)N0`oNdsb8#a@J9gsF2g>hh=3&CU<<2Y>=hgb?8ME11kViH1%yec!xdhv(gx~Ckcdx'
    'od73UEHY+wJem~a&8!H!-quBDz{oo+QisQfrWzTURe=jRt4dPPl7n{82TAufaqgA)%~rn}3B`$XTSZqzj*AG;9X$Nd5th77rI5rMpzj=dxsM|v_Sg`m*pLsC'
    'ry9#ZnruVN!+tI3XTSEFOc_l+z<AoTl?RoeI0h51R-Xrx<1Gj!!9Y`jO+p2>i)u&<%EVg(y|hKWBi5N`z`eJD5yuDQ5#hThO2ahzjmdrhr%#Db@t0xk<;(1;'
    'ACNNHqQqr)mIV9PlfUArTS{jXd&U#n4O7=zi@n#o(_D*_p^2@J3l{om9qbNXS8<QEEVP{116gVx-|S(p8*#8c&h-QEu2NIPTtka0FC9hE@ZtHtJz2jR<_}7l'
    'ol?8Q-?d%b`Y(>R`2n{Bg)L~M56re0W8Fb38_-Evgj=t6d#3{=SO{5vrA2Wd@nVuwRDn55E&g!qP7SByL$&*z@DFgRH#o@;I^urqXBV{dArCE|_Q~jnUpeGE'
    '+}t9&4?N@2f+3uy{^)#Gd*Z31&6!K{s5|&<^Brm`SybwaCpr~g{C)~7bQ7^|rOq}CI_3cx<Buxn40&bQ?>d&D(L+l9!%{d!lANx`mV&+v+J3Ez=m^yQL|35n'
    '>e3sWzkR|r_r2)%(DnCF{mzJFP*OTG0(UHwK<^dwJ4d}APzUfilAXU-Cv`yg?5=4lWQHi!nL%2!CPD>cHky?5ZY|$Sw$tE$jlhb%$N%cvPh$`s>h_AZDE>oR'
    'bbFFCB|50LW`u{PR@;x`@pKN`1ASFlXZ3w<>^dxCrvs#F)B>_PFJFH1?GNmU8cPD?;Jze)fKfYTYBP#@m)e}-YNOLezMs(QVjaz-Sbs)Sop`!|JsRpfSH!Sh'
    'BmBN|Ox$k(-d#ZFhPUfthkCoK?2p6PhwK5ZVrhmxf8GB)jTRVmssAG7@cH?r7OUSqIpNPBuFz!!?>`dg8mY?NZVdEvvC3fx;zV7|mX#=iT`7$Z`ehA$4C_-E'
    'py~PfrUb&Mm5%(IX@ja07#5mZ%h8(LwqeqU<>36%jrDHH<^esh*9HFAGgSOusKX~+hrusn5705OjG7}Jhv)zRyf@OukhaPGU_`8ZLl=4A1PuWhjeYZ)%ZOSO'
    '_2{*pCxb`!v&+zX)0`FKb&``J#$xU3$t2iEGR7&Z^=8DPc2XgD`2d00t;o3g6cH>88OqFv-SD_IK}bP3IBVe{R*bDM7(9AK6kt#^*JC*f5!O_BAvXxSY-o%|'
    '9t<$Dw^69e(<5rR$NxS%jkvdYO|UFnAfuzSHY2w}fQxI`y+98bdKhuCp7=U&NFeOcVn1ADY&i-JjnfMXycLd-?bNbt=h>YEk2sKpX4G!8@dRh0CYx!y7ry9J'
    'OzSRYiH$BC(%Y@US|I!Ni>!jTc}XGl$exsZ!iO;7lE`3$dGWhDNKy?_^HE|QHbT1}9ytszKDQ&^@-oN<9XBzRx#uji^~DfDSnR0h(CyiN73ex6i$qYwVqf*z'
    'IEN?!IsutUyN8jV&w5mFV8^vyN0o}_;A$EJq_K*A<EU$;3Wa|8y}kn*rJ~?zXzLW4zT*`xbE`v1oMBH6`}QGYL#b?E32y>wzgd)a-Nq*_Q0;bypV(~!Ke6Ca'
    '3UyM*UxqjCUIy9-p$FYCzES9|1z7{(!tj$jjlN8yKcvxLLQpR-cYz+{&HxI_v=E85S<dZ+$hv(1K>S2GycY^$3{Oczm>8jsemgK1$hMmkE?5}HcL=w*QIu_M'
    '6QpBTe~)wI0TQq5pLE6SK4G$-7BOMNr9>3;j@m5dIXXJ5iumo5KLLrO_$^-cN=jw7x)5UuktFvsK{C}%tw`+e3pwN-qOdNmX<|0YYeI1Ir$%hhdC_shNGuJ@'
    'u!-$Bz#FOYoZ}_DXv@EGu1NrMjyOk(?D^6+vd?^LQExKDF0CFrvgrsi<|GN7oLU*askBx+>Uu{ima+LXZK>a%B8|S-R7)Oe<&UotU(G;v*Z{n5pcB=!y7?ME'
    '{eWM5DE>48fqV(nqFl}L8)h=ghk^HG+Y@*-ofQn5ypdq*H&*obd``yEOFk+7z4pu5Z_W1HeOvHrP`vR~_t<7OV-ZAcUOwBNWTGcyXCJ!ECR6Wt@rdHB<`H-B'
    '9HBqcIw}G9+G7GTlzBk)EFZU$Z&trgJq$Xx-DbQ4NY!Qhnn&o~9cTGzL=m|FZD%tINDP0i_(^&*D49+$midd`X>`PYV$bC(_7i9Bq#f+40LAWnFhtVc8|fQ1'
    '_iXp<*gx4pdnURke)~5K>d}F*e?3T*ay|KJk({Hg643oejjD)!WJT$hHwHN>=x9OqUC8f5#octUW*w~~+g!<ybs@!^`&QBwU2XkxZy$~@((GW%fc6b<t+hs!'
    '|CiSbVRYXkgI|mE54_#EfAc!Z*{R=P$ogSj!+Ys7#tm~|+PzlC&h;ik2Ww#&Du^EVjS%oY!;@QW)g~slEJjsl98>q832{Ab+=e9Tt`~I(;ndy$V4qCj%p@KM'
    'M*2(5I9+S}&f}VYp$DhIv?bGv&AP}$Ousd|5YiEMrlLF)>;^hRg{~gr7IW1IJ&mT*mR`^}Ew19*m!Gp2&X;wt3RGU8zh_96jb?dSW*PiiK>xDrt`~iQ{4AdY'
    '+{d!PV$H|*iTBoAj-CH_rcJ^Qt)?+M-ikP=3`e!2awEj~xn`kh*P&U(onaT+UySqlUt*CUYPRmr)*dXv<4ah@ul*)-Ef%mtOF$*cXKV*px4_~n<#<9dszPCw'
    '5hGS1_JVQva5m>}pZrj;(_yoUUXioOtM64Sz}GpGs(0bco$vR4qTG_zT0PlN43mkomoYb&zG_$e3eor$Q8^5})C16PnupvuTIcKH-3G+a8z|{XymUA4+Hajf'
    '&+yO@9df7@eBXVQ%@xty<=n{BjYqxk&U8$7?c|c9<!reI3h18N9F5s$%c}NN>Fql#Zgq!6+fK}Wn0~2%E?3<#MwDPdSo%g2R@H7BogKHByS9xNlL-dgjNx~_'
    'jT8OtanxzM!FHKB0B!c@rBLtvWsR!Xb-bFjUttH3u70~DWaw!jX@LixR$0d92$h38j6cKoG_vWkK6T{=TS(dxN@9f&Qp!da)j7P|oR)1+?)l(!tzn)3OvT9L'
    'Es@6*f63z|4IDdz+U(!xzR%BrUc@6)bN5RH!}wxb^ZEPgweHg*%vdu_Axx4zbM4wED&2f)NpQC4)+k0NvfwoD(P+Oe1$Dz_ue51RqkXy~g4dyXxjz~u_e)HC'
    '|EYVZagFK*c4(9tL8e&*o+s%OYV7$pFTZ_0Ct^}Qhn}KBak*%(Zl3g>P$V_j<(N+kBq55CFf9dTe7oSU=6N-d4>Wr7$#nC_DAZrSU9fb9_*p<+z$6vW+Gftb'
    '!UN`L;h*L0jEF{nqo8t(QGs!^`8d>siWi$|Iun2>{mp7QUGUhCNCS#3Z00hC(^G6^K3i9qXVEk%Vl_64>AM(uWR`!A$u3aI$I>BD&NC=<$E6KbT?B^&_<%|R'
    '$kf*pvo}WX;Amy{!(w{lG510ptES-O$ri;0%k36d)&P&WL%?)#zKkoLZvsy%1%M8OnB9cR1BdLc*gKVshkYw6RL^(d+(XQu0P7gIfFm)l1M`PD7_;~oilg&~'
    'Fc9GVV#Qb|o<`?IG3InyoQ}W$>i4hy90O`KI0@$ztcx;<Y^uFZJ&-tmWDLQp=>jk?wt&H5-%V;x6NAG^BTNdSX%Kz{&I5$}&EhidGKylPAP`IeH1c)(qL^)B'
    'te@yS!q+e{U-Kp#&V}r|2wwnri1vdQvyCv_Q*jEKPS$h*Qb)$H0=a1Xbwr@GLSoKTY*hY+x}>LJfvG7*JSZCs>jz?3(F4Z1ATib@RlTs8WGMIhuK-=NHWI;F'
    '!noO<VHjlrazZICSRm1wPO09zy?DuEoQR+%X=psM2MTYszBj};-8LNhO?<A{)hsv%;Y&O>&_)lq|KCC>I;tpffs_R?pno2*YIKbyJUB$)ejSF)vB%0ABBj>Z'
    'ReM}i`Dg+dMtzG-g1{=P{=XJUV_>Q>p(wM6n>>Dgzg})uScCG%Uf@&NKZQMqkD_v40|ZAAl;{-4pQTZL4%D$wcQ<11v6cEloSDt{-=xULC2l!fS$u%5>_wFb'
    '0AL~g)zTqN$KztbccFldHyUO!VG;E1@`_>dmrTH6eN!-wqEnKQhSMAyW1%x$Fl<LFFbV=6Zl@`_4#^x>Bz0<AN#R&`way!2+q*QnOrtBqj04J$?FRl}lMsy('
    'Va0*(hT?4$!`r0Amqr@8;X{fg`=@&52Kav2E|=$(2+4DdNzILHJ;=C8UXXD$acejkFVf+pSRDTrexgkVb~!O(jQJ`RDb~bF1<EuWJao*ds>V)KwPg*E*~7&8'
    'RERP6^$}0{xCs60#7|1tqQ{ITsr~Q+b5i^PitM*3JR7*tHqh$Pu_K>cq77T0WHlenu*$yE!&7mb)+s{+75kpWFN-x4<qm93DzI$K&C2q;>irXYoR@zi<GlJC'
    '87HzGnNC3t9`0eIK6LO0%z<Yr8XYi`%ryR=$jetG5b-3_4-{!tX}-S{?=Q{w@^HFh@AYtXqvd?H0nT5_*ZP?^N+&5Ec<-D8&pgbCoNH!{jG~WPR^L>OOS^_R'
    '=Jb4mnS0<&e@urzUczxRy@b;N4_0G1R0aMy!X!>rJW7s^;^UL{r->h|sY?~7=5@y8ovt-gG(%%_Z)b_3%=uMgyJ|W%RRx=bI@9P!R^!KLs+XSZoC#?2I<spi'
    '_>+wHNjqvs;)v}}5+B-$J!_+kko=xaGW~@864gojovk3FtZx`I#`y+(;G!Qp`9Ief(a&DoSNaR{om%^v2&lc)dh5zv<tV3I3kv>GglJdK8hf~2L!8&9N0Yda'
    '<L!s|-Ae<2(fy_%uJ#8}!*@53jy0%iig!z=%_L5xskGuD8l=}?s`c@_)46(sr&oNmg)B(cJro%iM-=f<W}JELyH9w~BFt4iZF5)L2psnIIN?U%lZ2;F$K9Pq'
    'F^6Ir#FvEZlA-|lK~Gd5K4V59fE)=T3p++(x@F?P)q-S?V~2fZ=4Yy}%(WTXYPZ|`sDNLIZL4nR1p$De350GNx<GG(<R>9p`%z6xoY1IDs>f%e&7IcJC@+iu'
    'J67H3i|<H4dI#ipl2_4nfyoU?b5m4Nu~=>{Fu|a9W5jYaV_TMxW~2k9?V~TukJiHA(lXus)lNZ6qJwZbAVhKIpn(+iyS$u^-T>?p^`pi5ceCZ_vbk<yk#QMr'
    'Oh0sL?mGccp$KbnE+RZh-8fTs>?4igRCQo#tV2rDo4%&|P`wlfkKJ8Cf+zuYiAJZ;#<7QiVz$xesG!hvp-lT=!eS~;F=Ai$yF0tD&!A4Xn=!2i4qgm>!9!9I'
    'QKVZ5J*1H2V>|qG$#04^>AXzksw~%SSnjkzIfXcQnQPpzG&(grT1y3m*>7YV08m<0IZ3S)4rQFAc9+b|t&RJ!m8m}vvtu>Ca7dvE&^A!CtBQ7^sCJRZ`y8ly'
    'ey%E>x^4JO1h~pYBNr~E&{eTo!Dxttg}DQx4dF5Rn?>Wi%DD=coC;E*%_apfOG6*$mw8mohXqMr921xlm<6W*?tMr%m}3s=4Z0k#l+b80Fox-mDy=Iyi3hS>'
    '5EN!NsSZx_i`0ixuS;@?p=Xzli&Ztj{PT5m@}aa+K7N_^5e|RW$^A4}J{^F#H91t*Ay~y<lBUCEe--j%gvTWtGQ<2=XwGIfaOOW&HNH4_aLUsoK>IxwcDSj='
    '%A1vNEE#yXX`k{&znIZiagSwr05#72lsN~hbnKN+4mv?FK|odX*bSe4VqQ$I3QXICXS`x2gm^(=^+J3`{pG}#?K&S>C)JW|VW#g^=c8gSw+)m&h-{3O3xi2_'
    'ci5&a7_Xc*z}7vT>ifbw;HWq3`+}{b;@J9bwx7|=d5uwQBaY{S;xgluW_8fFft6?H#t&b7`=?jmy#9Uk-RoEXsx8ivfc45^Ghf}Hy~<)G3|8Ka=faKze;;at'
    'LT<q-$P$_+DGnypK5OBxRDKS_NKlIwwH)F}T_YiIw?SdU*?;?FM*4xbPs|`Yjn6t^3P^u#gTe7?o|%sKykIEezhS0J8O+#Eod&Fniz((i%7&Zs^FkPeqD?dj'
    'Yir9CkHU-7dkUvW67Cx^Epctvu^4w#!BB6kFnf;QJ{fJsx!*?w>p^d+dbN!G^^BRTG+S1!>-sne1;hjh0S_?(=Z<3pJ&&j52(}Ya=(kU}RSmAEjQE5MjUL?4'
    'SYWnLQ)`i2^ovzn8DQyTIqqAnv%7S&R^RIJA>Jtr>Pm+iK(EvaBd1L{#qcPz5p3x7QaTSaWHi4yhNtFd!$Bm}kEq2eMU(uBal-|!DQvYdvkS{d6B*)8i|t-5'
    'GiDLjkF+cY?a>U5H`s;JJVGKFSxcXGl}n_Ez``9Pt%bJtPKyE<Q<z-VunvU2CCM%j=Je>W%rE;w%2w;*D$q4g$;7^K$!DX@j2w(g#*WDVChy{HRz?3B4xabb'
    'ap#V?VJ|_vS(m%Ih^A~=c*KhpGj_93X0+PGNwRwa2s$ivHlHpwWrKDoy6l0CVbG0+ZJ|fveVU?it&7og)oe`9@88h8bDyR>iJzVlJ(Vu<nkkfK-bUy7^kSY1'
    '+RT6(4}`4-jrDK<41=IgW#8Ao`+NBXJwhEKYw#c+q+P?2HB7?&ORmj%E7ez^Sxq*(En|iDsVS(G{i<QVeH$L28T(uBnGTjliOMV$#7|<9ILO7cVuBPFga^Tz'
    't4P_=nC{9$R?&GSP=NBKg)t&f3jhWi+h?`SkxLc>W^eb*B>P2~cE<RF`>HOWXdt|<?wAk=!)azha6T_cDO6@9^XDo8#oU|<P7xWYBAkYkvywFdubodK0JihN'
    '%8uq4*sg48-t=gmDfXpHs}KF{jv^Q`Tu(aDIq_t_+$`iG0}@_rreh59lF<d9ci?7)1{lKKzgOstM6TC?a!;dbGA%{wfYEY}%3RXv0;ZF<!aT1~)WuhNyD3yH'
    '+81s7CZY#c%S#ZUD8_};kPsuC4`xUU#?2Wrjwvhjf6zo2JQ<?!(eTt0x`xLP6S+8a!g=dNQ|E;Z3v}|oZN1EadNMUMa*-%)D<%K~byJ9L3ab&lhkx!`Cfi*b'
    'qS!e8V{_K;5nNgBb@AgsI0oI&9}a|96Tf?()Uhu0(PDWY2!>VyeUsIRK}gsvCuBfyeXvwdKC-F(J`mcj1X2eg->$4v2g2PaoV@#=PN}uu2STeQ>vbO=q&?$Q'
    '2!A?w(_04$B{s3%9SA$wBPM!1duCEjH|y2z%vOi(_@5^OmceHPfjwxJ^k6HySel(IOn_z4wUTC*ElnNu%g(iRKr$_lj-2O+@Zcs-VC%}|mfN$@3z_{^1RD$h'
    'x1%UfY|yNrWRb`U8OgeY5NCEni}Kcy#%zHiq+{7RIeOKzp6b(}F2~vX)|Q2}Z1q^avovafcsT|pL%g{&vk_->!9|x{R)R4t{nyTg0jd3bHeIP%N_5R`B@63W'
    'op!-=C(dC}-#+<%v%u}em{Xblj6z?%*BZXllB7N{!>w9**3_Jcq-qK@Osh<*reG&d9x`<4EmmewdC`u2i9*5PX;Z3MI*z6u6!B&-W3hwxPGa^U(_J!}WhV|p'
    'N#)IVFTd+SXAx(o3!r|}k+|26R?ynNc);vpK=(^KQ5)$Tt!!;<0s1^Efag#Q;Amqupi$1%`*OE+{Pu$}yAQ|Set0JG!?KO;G48xcJV}B%B?h!a?0on%bsWB('
    's!miSN)t%9$Ec7^hIXdiJCo<lAe@2EQ%jslvKD{ga|F3pW?0(s%4UWv^Y~t8HryGD49}hagG}+>?Cr<Hkq*=7C`%P%dc+NYFrJj)Nexag+@@d+5#{Q-j+oeK'
    'dy?|G0$M+GdKD_2Ex$QM8%#5LPP2G)b&#_B;$mG~&}FND>Y3DC>AbiJl^Jn|h1W2U$|+X)ijACNH8d|8*)jKFycw&I?wCc2213Ces7)Eq!v%=$1`;fs-?giM'
    '6vbHOSgYcM$Tv67?qsySV2{&VS~o<Zw@-*Q^!3|n3V}?|Z!&tV-OSTm$rxH_Q{>4xZd+y4S!2*v+9?RJI*zpmziF*E9N4D?s-P)%eb;wpgSA)s&6@Utu<rrQ'
    'R(D^MHEfzd7<%?%hPL2ZAP`#Ma)pSDIhU900$~F$7uD<rdCYP;F4lN!z@-muowCZ|=P4*33XmyZ#xpvq24VR;7Yn@u5Ea`AjmBKxFE%Z^Az2Go57W2Gq6xbN'
    '!_b9IHStwbGNUK?Ynz?ulA_?bC@`Vhu_N5Kc-IRfXLrPA1V-oJ@7WP=IMbEkIcXxJleVBOplJ+`W@a%(dV#|r*A0$TTO$ekbe)k`Ah$|k>whek?&zqS@IkPF'
    'X&K{qnjNvoN19P(ERS>{Lh8&Bg3{O-D?X*h)<nOJPV7_L%<f}4MDuDfl>zqH6kK$JT7l+lylRAEb)SiedBz%at!=5IWWk#m(DLH~Eoh%cs~Z}EN*INW!WnV}'
    '+neXLh*&7_^*+(z$i#NxH@gkm*DC96Ud;-}qap=S0vb+yGU>OXr=FKD$)4ZmH-nd_lyJRZSEN%BYLA_sz-zn|LBLhqPK+-;%!kt%6Z5~tF$hK5*Ep)LbJ!fk'
    'JARA$yUlbq2Hb>AxddKdrn*(O8+U`nP3y8hDWw+1)k0ET&M<iBjK|hw_RN!lG(79+N&xtQZ@<&_rx@kw?Grhp(KluE#cUS+tK}woU0^N=Pp~J>YRn2vUC1^T'
    'S>SdA^_t|fbM66S%sysX=&~a4o(!d`2{S=Ef&a)LMb<r*Q59KP?!LcWar?-ed*<`<rdCK8Pr?>DhUx|&_!n&|IpI|UF^D{GQ-0*JYefzS<Vv9kk5AegxE*EO'
    'D>KZQE@JFNDn2bsdLEd8Ceahvn62MLU7FUO$a;ak6B?VXwPw;CD25HBQGAki(v$QgF~e=aH?@Nn%LOwOpH?@qjHm(Z!1c787}d5%VmudfKw=}}nAeAinGMT`'
    'TzduZQa|q($m`E5uBY2GSW!w!Rum<!_OhCnwJ?x|`0YlJm7n|(ju%(Iphz#LKTV{U#-YTv0kkI_%4#`7b@UeX4<Ku0=Q$}Ky6qF@BGG^Q1d$!KwGrxj*Sc)A'
    'B%@becNfEPV|l&!RiPa^RRMXwnFHs)hU1K{2^~6QwT^W#oMZ_{&N7--(Q*T)+w98D!Sa#dZ}&q+D%!Oe01z2qM!W6M!rK~}4@kd4??#U2l=e8sL^UakxMReX'
    's>f=d4kyrZesz(pmdhCnqpnYUH<k0bW5ax3l#8Nj9uDEi+BCpUs}n*f8W&etIRWl*w5h_%OnHj{62MTw__Wh~Xvbi^k7gOf2;-|zBa+Onf3aG%K-QE*g@7?5'
    'EH1lx_J-B#G>xl=pgIq;<@G@{t#i<B-}k7}&new5K*t-D5Pc^)72@%R<s9+&JqCb%t#9<%3dMm?D_B=r$hcYu!y$v@qIW%ue#i>`+E(4!^q=(IEg49m<LpIW'
    '>QXVm`i6a&1NGkX{!nM_!}ycetW(XM2iqVpfgf~wA7FOx=cWH3bK8i)VF#!k6>RGX62`);7%}rOz=JM^+8@;!L!TV_TXQ{v)hE$2iwws`bJTN`?RzQl4kY30'
    '%@}>}9?*B82@DRb>kjNglNf{flLZYfux10P5D3~P;wk}~(|b0tLTi6fRSU-KS&BVVKUXc8LM)QyqfLpyVj72uspa8N`C@R9EQC`>3*KZ0WD#Rf19*&q{1^lI'
    '=N;!`4CKF!7|0)K6y$ep9t$j{f=Abh{T-;}9AL*MX*b1S{LtJmom{6QJ8ZuVhzi2bYj5!4;d~yyOD_{X0s=X?rb1`KbejG+=$EK?j^5GZXmELk(0&~JBEltK'
    ')l3`jR&e5Alrdf1M(}HSj*(5vf(%#S;hjHjGS1djdY!51ibe#8fq0idVLm^ICNOc+=!#xU2hbNjqZa5%W{x7gHs;`fSjsZI3I^*l?;<FDC*2XfeR7pArn8wA'
    '^F;&%5oIX*oK_C=5~_GQZ=X<j#wUX`I&Y~u@8!?CXtTInEUy<WdAFmAkvWcp>;^|v%!PZ3lM{diiuJ{X1L*e67<VIZr9MaoU&reh%vp&0q07#udb-g>sORu^'
    'Mj46X4q)Vq^z($>?Ent&SXMsbNTY=!%h`rPe(2%+PAHOKK3`>0naZP`?jY_89y5v*H+{VJh71RAxe1Ykkz_3j?PBS;S&jy_r>~@jU{3r^io0ot$zy6G$;3~N'
    'Uf4__n_4k%7<r~x#C#1Be_<`+p7Fx*-WOoWvqC%^%)jbO07dmjf1X+K)w#!zP~=@|Kpm$C6^xTWp*HLvGVd_s4|)>8+FCunC=E_~eKmdIKfI86`FnyN*l|$A'
    'y_GR-N=VeqF7L^FHJ_gQDVg@?y4>qXGCAF>*SH-5=uUG$`-)7L(&SmS!P~#UX2W@Qkje`CH_c|n70WWnEFFA%+V-|#u>xx#3xc|<3;|;64U|O<?~HeA;499U'
    'b0}Y5n2k<`HFQVEX^t-*8+{Q*_4=)mhJ}66`F-?V#to7j&OQ*#3#2`FseK<M!$c>6IbdreJL>UBI9EuEnJTSQwCC>=78BE}_$WD{+6gFs*<G}eJNA5U3q+;A'
    'p6?t-@sDdI&v%B<hgo~#*yq(jKqYxYimu(!tKr$qglG0ON+5qKyC!Fwi^;^_7PXyE$K#nD<QrawCQp24lb7Lr+r#Yu5B~2{{UMOwMt#r73F$f_#8v~`49;Hn'
    '<DWep*EXxbxxQKLa}b$6wB8fx10P8Z&zSOZx{}A&!yHG?I=%ym=n)@d-^WS^2(8h<o#*S`N8d9J|M3X?IgUVcQ^n5Ly5kSNHaz@JJOhB*e5>ObCygKPeZn1i'
    '4>)LTx<JKA41)M(g$lP7TGlhQ6{c_@ACVGug3)qKrsFzQf`Xol48F(JWrpaz&j2By5k7zg<VuQPuZ?6HRm%3sA3I&Ryz`I)T?%z8o=#QZTd#~`<tOo8w?0Kp'
    '+GbuvuyPj|6IW(+8et4F^wy6!fr#?$v<#V;q4~6#3Pd-RelTqSpN}dW0f4ItLVgA^A4%e^W+@WRP6q6<*<0AXlWqjF<G}P>+48y~&;2fue}bF;Oc(s)t)ArI'
    'U;*_$vY)z5hd#@LlF`yL4s;MtO3YTz%@Mq30&2lLaFE<49s{l&&0{sKbx)I%QXD6f<UxcP1<=zUCFJm_NOp)FWFbK%J+=v&xDijBNi~Ww-rR5Epz+-rm|YC&'
    '18|@BWA0@CH>)7N`H@9)0)g?uf~}UTSb2OjLQlQX#>i<`3Zv$3X2J_puosrWiglP97BiX8^re}>k5iT3Q%v-85Q+0FM^|)AV?~qa$vI`fY+o1XFs0vWe8rsH'
    'KAwAxTkY~|yxa_DMRt3@{f4iZqa!%o(QSOcyb>Sm3gqNhdG+zAaQ~?k?nZI-@Tiu5RLeiA<sa4Zk81fxwfxVbmj6X5z`94bhD(~@8JQc}?Z@UuT(2E^RG2&}'
    'On%V{lb>GPmfr2fM=U5~7Ez3)gQ19pSX||c3C@fjMbVX>Lp^uoX(>HF3eS5B&qg_Ytx!j=J=%liD~Jar|LM11S0Yua7<vq^Et$%^ie}4eI5)8WhX|ym3@2nV'
    'KP?;V5QvGzJvxg0we7XK%~-Y*>Y#@70<`>Lk(yh|bI7~hSWR&V!>~Ws@-wR`9>s-6apBR5=R;ZX-0$}0Q7QFkDD!A2^JpkzYXNS3CNa0%2&U6hRxq4j!gtn$'
    '(ru$+r*4H~<Gr;D0anWt(x!{}wt<ZLc!A&O0)O(+U&aelDP2IaTClIxKd4ec^hbzdo{$}kOkshY>WCjGQ0hKPiD2k35X~vihs<=jSGWd>W9#~g`7dtGPIMaQ'
    'eGc(4_Y78JurysvIdeaFCWCWgsBG>%*8BI*819REgZrSPeetlP?fx=Gt9C{A=ppun<ly|lG+AKSL2>|7wIl4^gNCFyFksX#`I?1i3?tJlGi8_}nu#YB3VGn8'
    'n)q}O2tUNEy|yeObMvT0yjoI!)NPv{b=xNT^Q|u=@dF)Ms8_d?=(={>v$RqR-%Y!TxqTa+cQvYrpIWc1Nw{83aaeB(r+nmPLbcfAy5?HzOGv}P88n`pX?OX9'
    'uLWgy@Y=D~>IcaE!zfvzXOWl3mL(olEjv^#79xrHGO<H`Pxik13}iI(A*E>BqxhhcavyESFI-Xakr^%c2(J}t5qI=usuLD<=zajvJ46`dp{j~X5@d+qp7Z-)'
    'WY@%G1L4msq5CJjF@Ke#Vh`vdhNN$H;Ip{(*1<oYj)QE#xL8$_ez*Av4s$sjZ}J(VRy+kE3!k4_AFW-=`UbPWegtm>59WrAaqbzA?({<iqzkhI9arcNoq_v~'
    'Z0DKy5CCm~?m#d)A4$X`Q)Gj%(~!St3b_Jbg`Cod6P)kv<9s)t@9+t$w&&i1+WTbPKWgGW*+4A2JM$!z2bi^KJ?RbTgGZulnsEe2N09*6%MCDB%FXsjqnk-W'
    'wALzfYL!WRA|_q1H5^}e&+(Zj+O7;I-xlkOV$4speLSa(^9)5--4IKdiG3iOusTs#8KkWXBXdlL!=raBL0QJm+D>E3EiJh(jBjDfWrm#(t%FFo)|XysAziOE'
    'SAAO(Fq-sr!#iQzv~vgE>dl$m*SyKNL7ZBMe!b?Qei8fORh4NvoRzg->~5A}S>M_NV)qb>1`}`bY#bX*<S@|Lou|fhU8ldE={61^u}2sV`Vez(%E__K*z{YC'
    'd1o`Q7fIMhw{`3Av?f)sQQE~4ziDYW+e@wC=)>>-3R+>?UJdmqy!m;SMRbPYv&b=4OpdlUIq39|y|fcOckeX=|5@ybdU~|8O`A{!XbYoj54wGC1A>;E<HPL8'
    '<O)6DhV|}A`IY0(&J8PcKaCG=<Zsv(Wxr~b(rg(U%cI;3LjaF{+ZQcrDP94Wq)ruCsKpAiM|NXT>-lqDCrO@lG%kr&J0oMGPWxFT%SXQvjU;+TT?Bpp_6g<e'
    '#4Kp<X<D<4rY+0h*CL-6S$0SFO9DhMde48|MaRd#RYQ3%e%+BJF_)g-%U<0Xndq7!vaD<$K_MhIZ`lq*<(iwY9W+aHN!!c{jc%K9Hr)dlinD=_L_xRqnhb%='
    'XTrkU61EgxY7*B^yO#k!P_Mct2hQhXR^%X^Np}aHJ`Qu+GJk@DGZAvY9y<v%{)In#B*hcE^7XR9t>)d5cjzHeie?R0E!uk5)7dOyz891p_k6mAV(+mhcdS28'
    'EIs5p$Mnw7<~Jf6Z!!LAy<V;z+gC=vjI3^<m_i?Qw#64eO}51yk+5tQ&d$H6f5z{AbExe>2@x%=0Bt9U_$iBw1t<<vwl2k<GdiR{c=vs?^xn%=vB;-dl5a{2'
    'oexJMt9Cww1%Orn4JXU7KdZNquv^dVjqj?R`My2#gC)MMN&an*qYbotCc4bS?TjYH=#rAibJQWl#bV5#7~$d%B0r`2jJcFK1IFJh_{~(V39fFqFZhr&&sqM%'
    '3ZUh)H*iLjfS+?_)BASuG#YQ_^BYV-Kg&l2%UOW^2l^q{!fKhFcRo|;m{&kvM0x{H(bKM5R7+LF`t>K4qtiJe^lVCBjPn&xIF>;C%U9q0sTWPEYE|~0KfeIN'
    'ycq(unRCQ?j^Cfdxi@@1&!>y$quF#NQ_7p5J!g@5CTIkpT;(H}Hq3zCi_`+s9>j(4eZT{lk~gWmSXK$(HderJh`QYNmKWITd|NHcDMMt17)+V^r+f`(5bXW9'
    'nzUin$CUP)^*GhXg@y-Rx;2ZQt3lyg%pGE<Yr4D=nIk!pgM=GtfKOR2a`EU~Q;h1XtC@*_tEmGdXIbQR7dhSC>37xqd~fulzyISuNR*6CxJ`K{$~_VtWhL8T'
    'kqlvIGEiGBxd>J`#vCY5qt{=?fB1{7+~u*lw(WV5FDNtEaR-yx7AyQsSK`nK9=}Tm^sHln+KOazK12}-<1uGECZXT9DQkTVI0Fa#JffUk|L2igRh%T4&K~~%'
    'WS|p_H=OPT@|zjtVsR5OG8A1;3Sg?lq9sw4UjirQ+Q64c{&SIUCSPA{kg00>=*7RZ&!^yNZ)G0TyGY$!t!6jG@bS)8%jmD$zmmh~BKj-T{;RL2Z5N_DOJ8B}'
    'ABuMyT!NgkqNu0ToJ{W&!c>UjvU`)fB03(JLFhgXW;6l$bPXfG@(dbKFFJdGWyEP-M8y_n5GWpO@pgzN`=&D-Llk8nY0QUJJhpq~lrm3`ikM0UrXgt0=UU@a'
    '+8Y<Fd&KU9Edq)jU@q81m$GZWXyzMy7D<HwE45s4_6W%|7hw64^;USYMCxo)BF2I+f!4Pq5r>iU(yS@cLH1zh4S@}SWk($|oNy?0oWonKQIuP9*fG;$Y(fFj'
    '2W^Y01;q%#)=S`Lq2Exg^D>RP@IOG2)0ge&n{)V+Vwr`9!PjPDIU0e0qecUq@o{y%#1@MMvVvuMCqu!oS7{H#n2BVg_Q01UTuBrAumGBK`u>TzadLJf>E;=5'
    '@TaY2fuX=x{Nn@$@CePid?-$TtecwSNZn2YTcKu$|806aTS109%Av{~w-Ec;!1~dtN5=Zvkf=PnG>x>Y)>MyY41AV)enHQPRB)h-08~joDfq?n4x#UB=i*nF'
    '-nwoBfw0A%6*F#PgH(qfK}hz1%P7WS>eG6~bs^oYzTuGQpb-FH*;a%7<k>x!W&hKkYvTkQpi%iD=cPN2O+0teT-RJ&LIJLj2&4Ko6j^k$;x^?nOp1ytQ-3I!'
    'daA$iQ^A;vpX*%sD2@e&%ka8O!Mw1@CqQ6XG%pkaVd&kY@iDSoX@`g>l3(*<g>N$U$2n#8A;$cqv1Z?sCOb49O*h03Mx7vt9vU6~5T~F{N0v4n`MxNbDAquh'
    'FwHiZATiR&2#7)0`UO*egWx>`TC+Ttm=_`~W@#cUfjYWrn_bV9WdcC>4K=HeX3JQczcfo4xzOowT@|D0IZ&z8YJ3^&vZ8}fA4PpZG7VL}1{f-!2+M>I&C^7u'
    'mLRDVIVDtul5?1O=tXDSF$#ekr&dFQW6^O;<%0DV#mFW9`;uRG<WiL?$InK1vChX+j7OkK5KYET0azxhTJTmD3@XWR5#>h3fmnk8qF-NAKNi}DD;GU?8#=;j'
    'uU#Vv_n1)M1PVE!!0-d(4$H~(yvk6~>=fb$kp)i9x(jsWf`IukUOw>2rOy&m|AlmYeB$spV!wW(GtitfQBJ<_l(yG}*Rfx}^@JxQCXIV`Y04k-m@zIDVtI3J'
    'U3MXa>8n4-_Ip!aYbvT44zr1eP3B>MfnVmk>aCgLWwY`f{*w%9>Tyz3<NN{(yXniKy4|-Xsd|VY)bvB+t@rN99(6q|htUlKuAXk$oq60b&9Ekb5A06M!+8Y='
    'nD`I<z1@w+nLTF<5Oe6Fn8lQtIOc5c19pegWYcW|?_V-H6R)Iqo>fM>@AJ%`M=@e<uU$K?c5+<F-z0#F;`$Jcvbd?E-^DUx*VM7^E~H(&eB%k&hhyK-mj*5G'
    'xcF~;6~H;<CGcl&K$t{pK)jNmE1P1BY)<ZBShYI=&bw@TD5syjj<(U4@V|DuJ>US5`8zLiVC{?2tp^}gO??mR)$4C1dQ5Jp?SLuy`=*2oAL^)#JxiAHiAGf$'
    'n`FK?D{<*r#^P1z;N?<{<kohrom7>rJjcP5481C5&cql4^Kd@vQB-5=rD1I=ocTB`L)~UL`VqZ%V)kr}6Tn-Wch0El*%cF~_Xae?4#rYhM+x39CvChzj3@ji'
    'QPsMhOlJkT*z@{K5>2B2hF(tiqtzAF7XNhy2*n&m27WE3eTG;7arfeDrf0!<j{5ddadt9Dop(mmEkZ<W2+Cng{LXf=SzL<YkwU}H5+`O?^a54_&cnEam$V2j'
    '!5;|FUxm=3!QxE-4DUu@W!1oZ-l9IT2q*whCth2QMt8H3x^~yp0U(MluS+4lM&g#5ZWXBlg`_L;(Ii>|om_Lu@!fiQF=ZMAykQhJB6#bYMngcDoSTmD%gNWr'
    '<7kwZLO}FbkxZ69E*ClUX)#$U<0WQL(kV{HqhfeEJbm$ZpALsb=kGe5&+<=;&hYfpzx!-7KKW-SpN&3!ajNQN)j2ahDW7i^Mb%#2Jbm@qXAdhXN<#(1g-yIw'
    '!r^ge4_OXAFN*6-u2n|i|2~i`2m99PF7hBw)O5$-L;h}}kn?d_qP5@ndvIcNQPKI^C-2hB^h#BIFMiz#+sg6uYFbW*K<9=x9=PwFS9kVpk26qNSu<?O>ZH=f'
    'GAEZY17e|T_G6mt=(hE|heybUbzBv$2cMkGnB{6Vts-n?Se&ChEPG)s@c|Y*HQ4b~Z^UdDI<iNruZoL&bQ7{3Ql?t#<&XR@(?x*VNyQ0ms^WDyCzn_9^GRe-'
    'L^JZ{yO-bf7zhmzEMtCw9u_52NQ!dj(=Acg_9UO31AU#%KKbMm&8eUFXjNt5PHoBBXmvxTnRN`A$K+~%iz`)tVYEQ*SZ)b8=X}8xlvNXy_7xDT>pM?lw8EMs'
    'R(C}AIU~x9DL0EMJD)9AB@16mXGj`lz>ePdb?j87Sd&^3*W$0$WV!k)h(n{x{DN$Ys04jmt}yBb9j_Eeo?E0!w3grbLG^8sW6dRO4&2FOkoqt6T{O=}B}!g^'
    'OC_;D%N5SU`P7kY**I*T8ASo~P{zM0moD=8%8|B5o3dKYGnkeBdwt{ejBVJ~X7A3`3Fy=2;Li>4d^!r@HdbG`a}Nv+u#qnshx#=7OA*1ep}kLpmx1%Vns0y_'
    '(v>NxzG0*3ivN|6eqa}o(f_)vM8N-&YOrLG2x&dtvWo+5peBk50AzQyoDYE&VhhN$?N|8>cIj1t1^}Wp)-Rkxx`R84hRAv9<3ux1WJgE50@==1%PMEJo)M4x'
    'l6-BMKo_4WpabKQ$U_}>Tslj;y(w^dd_D<|LeF;eZC(K(L5~UqX|w_xjPU6Ba)Y-)jHcs)$~(NBm$C~^xc6ai7K$Sg{78saBi-1k3NQ!kNa+AWiP!NT@W_S#'
    'eVM@1n?Y1bG5hafJN*%TQvqlGp;ksGAQpi990`iA8+NCM6x48BP0(v+aZw>Pgg)BDnDb#&5;(5-R44((lv^wp#cN}wf$edNYg9bN&_Y+dSxH|@t47K^+)qU}'
    'lQy06F?lXqG8zeD=$aQjC*Om1`EFgs>epzZ9}s-6(%vYhi*hr!DBs&B!%>wEN9FNvhgFIBR^DOfnDBThernVT_w~2Lic~qWl0-<(JD;Jvv>+Cft}1Df5WPx&'
    'XUgSyHP5#(d)Ef4m{)NkErywx&Oh3+dTx&A33Qx~U{v7<ST&e;_o&OYMZgnR&bTzAg<)0Gk!?h)(Qfadr!zRUhh&HhJpcis?L>&f&d&(NJ&-s>ko5M)N@#Ec'
    '{^60(C_=+DJ*PiC@i0Hfl9VLvt7&oVCwy`X+O;nat^KQ_QT3;S9NsUSl>Pa2QGJUa>**MAl$+U%$}<*@8}AuJEO?W^x4-6h88GDII(@SmwWrk_=4BzOb1_7S'
    'DU7?UF`ltoWj6=jW=+rRp2lpO21q5+b!EF@S&JwYg`R7K%noNWel@f_*Q%2#5M|(^LqkJNIoIUji@qljIoBZvdxOFhdYk}m&Zc-}fHSgD-NFC9M&G9`wUROu'
    '8W6qFS0D#;k6tiyMQr)S*04oMRomK*!%lDsuEwxQSW1ADGtAN3y1Mb&Uy;wbT%QP^rf7)CL{QSyvPg+s7(q<50HMZ%1@D*@mZ6PsmevfDO3PRWG1xRU37GNR'
    '#})~3igsRM8FEE2b7cXUFyZSU43KEA(OhdM48k~;S9v+#j(^qQ|25)2>aC$_jG#b5Ul-%e2)0!*U#@SMzQ!F8Q9auXPGKwKFfcwRE~0u*7X<-mL320m*RFJ#'
    'NOy!C8jNLk^aSn?FA$9#(<_}}pdN=hCL(V3qU+`Qk_+UjJ1}Rk2Ms2U&B^Y7P3L^3EE}@uWwBZpBOnT6cO{)`DQ>p+_4L)Z$Ea3O*S)x+^8$xGE$1dA*fcEg'
    'EWd;#AU2U>-FhD07$K!B?~<AjHPI{l2?rL!UM(wTLNU8(<|sd1jMhbt`lI6fd^!RI9^KsIDr7nb<YvJ-tb~Wo6^VHu1+3?+%M)*SAn)Pe+UFV36Loso=N8UD'
    'x{Xj!1}U_|!a0U`8qx5Y{MFy<oUb~cIUm#|((xX*NbW9kZ<%)58CuAE&|Q>ljOnnlO8MT2SO%mhV;aOr`z)LjwYXU7qX?;$nuy#C4)^IpcVzCFV*l3+&Vo7W'
    '24EO)grg;GPdEyuid~tOu=y~RD3Nwj2ak*4a(PMiAf$W*Huj>3uHLcjqB_9TY&ur%nH<)$D*hWfZOZgGC27?9dbyl^@=5fsmKz-0#gf#~Ku~6zF|!(%&KEeI'
    'o8@L5jn+#zauw~;32VFEZdq1Pfwuh}4U8((QD!<^$m?x=y>YD?_SXo5jrZKkIWEm|ao$E25`RoDM4$(ZK>H=VaXev3p3=rk;;<^;^ID_H$>|fnYSg!;(j^Pt'
    '+gx8U{YKXh(+i3(i+k7lNuMOP)>l)s8qlCD*PhaoyyVv$z!S6rJ*dB;wtDh#Jz$nN^322Q2>EberegD0<iU?-+5T=q|37<g+TB)iB?$f%ROZYOR6>+Q<z|)X'
    'ZkIE&+NbO)kL~K2X{8PglAr{OBAHy2l5?^T7J>u_68l0BAVGp8xC0=#0R%vt)APMg{{|&u`=q}x5%Cth4O}vvu5wvfWfAYaSa008d))Xzw=Wy=RN?+a{rg*n'
    'bt`#q+l$U0jWQ5R99Z&kOV0-L6Zx+=Iq~TSIpT-J8?&dL2^NqCA4~L;acm(5R{Hj^_lhhS20;<3GRjrFE|%cxJu1J{xwtv+>6ds+zg;w0a!#gBL8V~4l!W)U'
    'Q+R06(R9c3q4ri6xyChNy<(WMMh?{Uc+=osu`zY!s~R&5xK$!iXMQ8z>o{QJ={kg2el8hn%!)R2{Tmz*U>Q|9-2S5O^|77<bjmDYEc&Y>XVv-tG;450(6T0Q'
    'm;LvobBuSt7}Y9~$4<md0SEq-0pqc{@Dq(`xKmRcs~>2xP%xv1h1)~acU&lmkI|(Tn$&%5dTEqBLJnqsY`g&S@x0Ee*_Bt#Xpp;e!RynusuR^Su-xgakQoJi'
    '_i-%hTt=hws2{JXr_)Z1UUq2109&6Jg_tn}GMcvG)t*DrUB?DSNaJDLSTxbVmDdoGek-#j8Q1BMibN;M>Ds$uwgzLlVJ&_IDpN|X7}ER7F+KP`khns95xDUU'
    'M=R`A>29OWBhkdsUTsvl_M)ioyDQYz0QuJqYqWQ|1lJo3(@4AT@cxR{y7^8Ro4R_<^vQXy2Xq@A78zZn%*i@yfkmbGD}G-w@*C3^9V3^psuO#@BOqUQr?%8g'
    '^8>aA49Jp}V@kaa8eAAGUymeQqo?3`9qvXF_=Cx~>y8sP2HEYTlp8Y28*R56Gt8SYO&#OB(e>sx;g&y?HvCYosX80n=>m+aH-?C58pC1dvV7w!4p?5l;Z@q7'
    'MFCf%Mfr1V?CuVNL#G~Yz9mID^-iCu_ck=(dze0MB2yU15x15dZYD2uk`b<w3w|6~K)R}aNV@ioHh(ihbp>%Uqy^U!q0WToD$;RVa`AmhgjUhMRv<D2!fi);'
    'Ghyde0?rQ;YW|7^8OuBTy^(c7M`R@n>lG%N$4|Piv@~$5d<Szw_v1(G;HlA7nw_i;o$k-H4+=*}wftv%Y~0-aBN-}KRQJD)xk77!yylv!|0w1PqiMf0SG+S<'
    '{MDE%WY5GqyT!i;yM=5Q;L%@fyK90qja$0XG;kYSeW&IB8)^B|<9|sVUy;t~mOA{+QCZZd@7?RHP^;v)AH_na7aVVy_3Ng-PNNgb0uZ%+yem=>b&!iCPpFZ%'
    '*vJiPAL<v2@@u|-$aJrsis+|ihpsJJ<H2apug0GKPCT$69Q)t_SYpP+F(HhnpN@*|{@0w|vGbxVD0g&h^sY28c8GTDseNkFz)P&WPKDj0GXCIWex&EB(>%Wr'
    '%JV2AcG`1H3cD8Kk?FX^n1U+~*BIJbM_!u^Z-ME)!oG2vdb+)GMa2%_dD#2s0?)(VKM!~w_I^L`=<D4OJP&)X&Oap`^pNkGQ(U89F})W4-nsLf)_OX5<4Y?9'
    '$nhX}b|zzh9m~CKDk|aBx4xD<Vs`Z6!QIb<)|J_(3@VA!dZZ4b9v_*0`jmy8jGA8Bt-=Onr)+f9UyCDEj*rJ)O!Ctgx)}aHd(tZ@QORp%L<o=0Gk3@jVh};1'
    'pePpF1HZ3L2*`hJdd)Lvvplk+Bd?`{5W-5blO@^-D#!MWEAa}AaY~Fx(cegAmY2w46llgSTpXuNt)C4-2+vigm6CEPw52UG@9YP}(TI?JEu0kO5S4Nu-+yB1'
    ')@7R#GEGUB=Ewm!=GUq*Jmc9VcsOM{giB+k5f79~H=!#Dh(XFGk<~s^Uw5KS9VH7D&x}$~C~3_VEg1|>MQsd5CtXNH8+|5Pe4cd9ZguTkc#bLoe6Mt{yM=V1'
    'Ojo5860XaUrAlxurtD}RwG;0+zr~3=p8n)}p15O2y~Ok?lQz|9sdr~1ZgV!`haZ6WYd!zKGRF6P`5w=BW`Ai@of5vo_`330Wc4=-=l^koyWfQ@{{=#p|CU0L'
    '|JEXk-$nPni|&0F-TNOjy7zB0cvSc1FXxSU7mN8W7V}*!=DS$Tf0wbC??MUxLxvLGgx&qXn8J6FdAEwpQ<Y$r6}r*bAC)OYc;9DI8eI%8DZJJ_dlR4KRSKl|'
    '?dRe!e*W_5Z|K;1hs?~Trgp$y@7ZNUS9`ljhYr6O;b^BDkM<521*GJ}RPWvD4FA6wSY^r~Le!Xzm=!1U@7<$BzT2XT!0F&9H()+DhNN`U-w#2#(VkxkrU(5_'
    'u$X%9fB0z!X#LjK841M}S#5)>a&A4p{+GS~22)fa%@u&SYB&A>K<Hf+Ry&oO{&{TbmrM$JL8ogQ`g~$yY{ESJ<$M1+YI`+q8)Wxfe8b@PEr%k@c>LbqOi;n!'
    'Mr^=MT>O8H?t88G{GV>;H}BvxzO7@rTV+<IDhJh+wO;QZ+2yO(;bjS=XL8CqF*V7Yie?4%H?kj!Lzlh8Y9DLQ3!8P6$1a(BEvARv-8Xb18#!lhKCbUSCXM%5'
    'Y{+7=zOplaT=(@(-75&|l|JaJoWx&_zAxnG<ZfaiH|gMnbhVm@w309R_qJD4Ux=LNonQH#U-_@eul&w;{LXj$&UgIIcl<Z%JAUUae&;QI=PiEcE&i+X7QgfV'
    'zVrXS^Z&l{|GxA8zVrXS^Zy#3K)un<y`h)bNPEz|(`{zjEI;gTobE-fPAGD<Fov#cyt~`TPbT%ynOk4OcXj<dwm<2>C~m}gujaT0MEy#k?m9mEefaAS;j7&`'
    'x86Gk+&c%{I|tnNb-?wryjB`ROxrd0&9^_6Yi`6kIn~d-L$`L>J?mji!hH8ZH`l#*L-MPR?m9kEiAW2TZts{gXzG~?sA7~2;~$E${ERVsr{WmbF(73F9o&77'
    'C#NQq<tXcrA<_Lrnv>Ka{i-P}%};k_qS*eIqwiVX<4APxm!JLmp0F^J_t@Q4)9f}F>BaPveWYjn^?Q%rd*5Uo8XIS^66^{U=EZw9HAT0{y?amXQ_rRcdIrZ{'
    '+#92iv)b?3UyM(Ty|fM8mD{^V;XZKhh1EVPf9!96d}^JZoU~g<+plP!tP6fsoncQS(}VV*33i{~y&>D^n0=D&l6H4@$6r&5IQUHH_LEQUe>^ZWJoxeON4=j6'
    'fAs0l!-vD4emL;Sz(>8kwvQj!KDIr4@bQNO58ExZ2huOKbj7!82P=Z*>9l=F50v`@)=xkA)b`PXPX>BF`FQZdkA@yR_{m4Mf!?1C(*X~E();m8Ke68bXh5v4'
    '0NbbtCe|yvIrRDD-w7U8@hY3KN%<$cLREg?gH}KFjE+(ir=pbnn#$HJ4+niW{^cmYHOFQjWfz%?--Wa2tT+BqxR8?KQV&C!N&c@?lwDhAWy;l*t)CU{9~AG`'
    'VgJW?!VBwjx-)H4Rw^hwvvC?B{a{%|B;scgjrRTAbKdvBcK1X172UCYy|%l(Oz)+uM1_sMPi=P}O52M$-o+M$C@G<uDng14fYC++aZZ2{tur}p9i*FBF${L^'
    '`~9Aelmn4pDTmIfYM;;$=gS{|*v;>ZH}Ej<h%WVbg;}Y_EAxb_#%vX>;qpjWe!>o;xo1)plK^R`0$4}JpK)llM{&<&exRsEXGd|TXWULd^cq#<ozv20;Z=9f'
    '_}JIxUZXz5ANRx`kMBRxui69CUFq|KM^D;iS^h?SX*#gM5&O8AKU+-J{ulPqzWe-#^;O^fo}c{WCs(ZoyNtO13KY5mUzCV)J%(I4f*-eQ5sRe}s3Jgwu)6<h'
    '9htVt(RA|oy1tQqe%oc59(_JK_Vs8F52;h|AwJw5PGrp88cHz4lCH^kW#*6kBDQ|vu3#ZNR9#AXH?Dd|Wzm7LNg+x>K?-y&GMIKvkJ?#cw=Tsz2Ss<Mx?L%K'
    'pn3!fmzL_ifNj=E>%@fhwV7ud753zP<1r7)K|FlfjmP@cv~@^1P+ga1GwoEy0@l^IaD}$PbncjXKJ7LFYPUe4y6Nb-c*rhoy#gN$!z{ag|I9XF6H<)uVM=%E'
    'iBX~^aaL4A9HGL+m+yU}oAI|_zSkx1#`h&rOg&UNDT$OM&uDR&48wA`@Pt!OZG!4FY-iWXa%y3E$uN4-G{NuKHuKmdcJnRgaN=cu{DyK`eaJdxH4RNquuM79'
    'J)t6Sc#r*Jz=3*UK9F3UbY(^4RDf}uwl+mU_B-2%5SH4m$izqvJ;9zk@yPUz+|9RL>b)g@FxpqrSo^uY!*2H#3Gm=Fvkhx^72T%B+vt5Jg3?h+#F-6{$~sgp'
    ';8*HW(7m)yP?>JfCQ~k<qME9+ULUJFeWEPKvXZ=1Z=uKSiua#9p&;$l5U2f5dZxzsrt4Os(BR<M#E>*ljs<k*Pmfqx(y^CJ%wiI|B*|GPUfEyvj7>bfXC0Wl'
    '*L(lL{hs^1AKe$3+NAyI3qBtTDuLzkeP(VmBtD03fj>&`7<W`W%KS<w+kG1(Q7nKOL#T2<(M0(C?cdW!^AuC0q*uHv8F!RTe<9v7KKP7wJu2)9rqFU^;J0<N'
    '{rnAm{MKUX(uy9#UHnF8N!b)HARitZ85#SU?-mBI{;4)Y(~g{?HixMWulZYxIAMztVr@+4jHRD8+0Q@s4_GGznkP%`3hb7PSm+LU#YIq&`5S~3{$hd(yzGWw'
    'R;)dGDtS+}X&R*#Y_?DBFWC*_y!k2CfoXVpl;0{Un!W@m(gE|UwK=e-83M9nC}QTjDF$-$gS)J|ZWC3RCOAD1s1aA~?^_-`(cj-}FjMWN)q=1<shF5!Pji*K'
    '1Qlij8L_??7_thpWLL5C+q<n3Pp27`r^Dc*xzE~qpuSmL{h!#;jix$1&YELAc5ICa-lDZjo-S*P_Mo{3Tv}HF1oX-NV)})!gu)G@f6uQO6)8VOQ3htJOws8~'
    'jdkdsVH}%1QwQlpChSvo`TB7s#z-=mWP>qC9B!~!N|V6U17;03-RrclijuvI>)Uzy@tc$v1TfLoL8ad|G2&Zhs<&IBNy77k?)Dz(R_{wjn>l6XZ7QTW+pDPw'
    '>)=#ByEwOhirdA^l!8z}^i@CS*V0r6ET%jCOn4R8xbRY7pw^zJR(1oZncW6DrS6!P=%HkWRWY+B*s(26SecbY>VmoKySo*mYMZgmY*Si9TOf;?q6kC(4v$!$'
    '8WDic8IlMjK-$ebrkD+M&^q{x5jWbZZQ_M($j-~rGB2${;4OuvNdG7nNV8yDVo17m-YUo=>`cn&q(E;O{6wy5lL+a6b&v&;4$<Liah;7yz;k(g6vu7iu+?RS'
    'NeiXr?L5idpo3h%Va>C@e_(C>L0wsZ6eACE4LaPmL?KpUIg=@RO;Rb!G(dqgG-l(O<_xzLo%e`+&_2cNyp+x`(}OfF?KJ87JYvjIyrP;CU+Vy+#7`%zLv}We'
    '_6oNHx|It_tMWD!s^-K~t~^aM-?{*rZn~9(XJNM8I$PQ=<ZiwY!7W<B1+<cVP3%w`j|T1_JPWO{O6EIa1!Y1teK%%Nx1XLGV{`72-b+aPpQ*YxWgirzXT9lt'
    'aO#!CG%-EeFY_ce)~76&^&M>@OIo%^S(E}v`KT@^n1zLnJ}F&z%vyZ>Na#R$6+^p%0AE#kq4*=UFOSFXi{`4ael~Nn)_f{v;%g5<JAU&W-9YsVt8PUaLI<cy'
    '`%wVv9kDJOoeq#%_&k2qGek+|;4^bKrDI=B+gZROtJ!bnqBdt3uh_=d)qY|K7TYM>2*#j!-}Ht36=UCYD?X+22h+*~{fFr_W8<7JDOdW%7-b-Kfn{r(5}h2A'
    'v|D!hIjuTALKRJgbjT7)a+1d^#!SEyC)zb++?LW!!o})mymtTK*tp#$0_&xnIqWCRuWksL8)AgIE1jR4FQsf*kvAO&@>P9lb0_N%-~-(ruo$+-2cY>RjCgvK'
    '$}U6N5G2J6n6FINai`D17}Q_2TS2o^%t9tyA>Zl;N`STb?bYrq9(BzGi*eh8eQc=j!F~FlQL59<5;+yarYFbjL+pll-FY4j@6h@8+U~l4ro$hy53)OfnTddo'
    'M|A$#1d~QjCu#Ke&uo)Sw0zC(A0K8Ck21}z*fE{wBT-(Zjd6KZ6kr7q>!nr!AUrAW)Wq0`AUab?jSH~D69VIXEw!Sbu+}$8J63up$HykcbhY}r8fMA%vSL9L'
    'goPim>SIkaqnN_X%f{E<G?ce%w#r@Rk?0v2dunb^m5FXDtF>U~r`W)(`>ZPiOXWi@sIrfW4M25%^N{T&UCh3=g-CeOt!Mgedx3z-PlT4uY&?7(kNI1b+g5;A'
    '+l<Z9yDBW&jEh>tZxlU+7h=v=)zLd<;iJI&-%Y)bbjfs$`jB5st+RXtll>$1=QgwajZNgygD1KUHG+q-3t@Sc8?$S`boXbb0n<<yZKY4R8e6gR<;v}iu1t;a'
    'qT|WD|3*u3*`8soLS&>UX_frU>}a0YkA2DjCmHqU3`;2Aqp)Qu+1_}Yz4Eb{)l(2Ex5qC;si-Dk5ndY2<9C83@Gm(Dsx|kUg#leT5q2oFpGvrNJ^6MwJRr$r'
    'Y)(;VVZpjUc?wUqFHk+1P8ksev16h6_90Y-b^C!J0E;CARZOLZX%kF{p$rB?{WVtyOw5Tm#1tP@O(Aw)O+BT?9w|&7Lad1N#ehc2VPt$^RDE;?z&}omO^^RG'
    'E&H(7{E#^xrD;e<U4QfX_iUS*9`t--VpckK7Rkbup|5T1;2(F8whMjKBT4=I(30+mt3d=!w_cTaB@<F+GNYW4N=!<PLOE*Mh5UHCUZi7-Qpr~m+3ePy3l{Zm'
    ')e$D)c%(_2Jm1jZ_TV#%sh9qbuFJ#Ecst^G4r%KcaxK;m^9DT751me$MNd5#GHZvS^t_1mdpp+ey;1##9qT{rRNu(IHa#k$3l*r_d-IX$8`jUaV#*2u(&E{+'
    '+kjZ8(ixuZf&a?5@pkXlmrwiI0rx-0T3abyVhV<snBxbmx0u`Khb(zJJ<#`Fb+L&bW8sCZ{a;=WOxTBhA>BnlH>iq+7#vy`HuxY5%AgX47@NUgNEbec@2)$t'
    'tUV|zwk8dPhxHl;f|xa(>)KKd<Yvujhu%2?QR-VYW0m4A7LbCj^fQyk->s@8H6~Xynuc<)dPBrFT~gCsk4%rH-&CyYQdAQ^NsmvyRq|MhPJ(dAIWW7$j0wS#'
    'ex+8eL=m4?V%8XL`F|vG63W$wMtNSWJA$kA!J{YLQhR-(Y~3S@k@5`@8k?-fA}gc}>fD=Z%zY~PT@Gh!582!;oTy=M#GCts*BwILCZ{Is1H!SudlDWCv2Hi0'
    'pn4K&8AeyB!l>HY^ydSU+%*1~q8SDA6%YAnBY-=**;NS6#(vwn(!f5m(_tw`Wi}eBQc%zk67EP*kqU<Uf(x-v_^KRhr<p3Uhv@1U&(Ekvdsfz`Ow;{GG0;#o'
    '!Lflq+6Jetm{MCXXSm4h`}FhVlzmXWecE~N+WJ7BqOvx6nhMQgG%r<yUs@;Hir%y7(dTAmuf6))LJTX?O*C#+e6g7BKD^&^pC#<-xv#CPnhI0mmCu^8lT$+~'
    'W&VE8M|4jsdg|6K_=Kv0aZxEBj;d>;!x%zyx}hkPUm2!K<zKTLZ4<z?2>42l8)oX!2kox!s40|GMAUF@(zjMvSB~cSh=_Az%;&}Lmr{=ECh(F}Gd@be++%)x'
    'zvs!-`Fv>0yh~=fHMj>z(Xg=eqIxm?Li9zj*gXYW_jX@>KvD7E2kjN6e4*z){a1;>H*SY*kjFyGnukxcHxn7(YujN)9nR?J=ShdkfhJGdbWeOiM{eATVz!F1'
    'Ml+{Hh3{7aLObg=DvgdvxTHD)N(=W|Xwg4ER?%@zeLXgA7ov5TX|I>gNelwj<Nei8mmB*)0~5tE2nk8<npAj4!74#m12P6G27>Ne@e<aq)@adBw<&wiY#e-C'
    '^ap$Hm7aTzRGi7N>4`zxqgw}v_kJ?id*Ax$uyx=kg99Ia(mOQ#(a^^OLqi`AemeBv!QiLEA3nT4@RQyT^#S4|W9)h_DxJLls|TWO_7|cvnOT1R=~omcRKa@6'
    '9k>IwsjqD|y2M|PnWTVx$xiY+M%<V`mTG9zQ)4ffAIv&3^4c^oF=ibav`$X3Yab`a1cNy(IcQ}ep4@d!D-Fq^6wG*KWgN}=VvMSD0?A?V7qkt-5DLM5L{X}J'
    'MH@5EP@c^!XE$605y@HjKGPTmZH(tzRLHFLmdX9nlIkx5R8xH}hNuaVh+<jAEov5?vDGwUvohD(sKUd}uaz5MimuRaJChhsi#FH}o8!)RT2gY4nf^h7HB*G2'
    '*+#}Er6Xe!SY<HFPBp>QWN*9PO<#{qJm(;6^X6_5?lKfb@TpUyywZU1l?H%fiX#3>jP-cM`JrfoR3C}qzB{wFN3qZo2_`C1p`Z|VDYa}reIQj=6p$vPEY+T|'
    '_)#vVWLF=U%mP}`uN<AkAXu4Lu?Fs{>veZHPh)#wHPQ`)2rh2&lhlM!rMUVl`_$(ztj5+qjcm&MWMb){^|cVX_7jV6=6U)>|MWOjjVD!Mo7XAt%!0w%GX0c7'
    '{l1AiZrS6{7=toQuBtFrbbZFlN;oc2l>6Ki{-8^rLFq;RC>4)-C;1=Eu{2&_!uFJMBils(!1VAi9ds8TO_xQDXZ)qHZ>ama1_d|`F+gdXy>C&r$bz4xlw57#'
    'UL&69_r%=?x>j}P#TYa!vTwjP(!*X1^>=U(NLjVjAp{2E&ifO#7h^AZWKCC3*WJNqR;Jh4CbaH)JB!v~&@q^uyj1KDQ}>zM-%Ojcbaiz~xuRO-9s$_qZ`i{l'
    ';oY~LYHC+YC~jegHz-x5C@i~1SU9#GzXr!fgdo{=jLG9C9c@7z)=J0<sNW|^NA#i7O`0MGnSR!1y8o!doug8j4^;WCl>M+Uymns}ew;>IU5F#TZ}deqe+nOV'
    'sAYPxUus>DFm=<oK~&8P0esxiJ2++Tgp=L62#%q#uiKF4b%jCmsv?e=)#H}-z;oMcj{idaM}~=>#E*|@*(cXxKOd5ayn3|cZFH!loX2C{1V4dk@R~b5ScHml'
    'WZgG!mI}fH{budbQPyWThPx^N7-c-_p!X<b7q;(3Q0sxBKAk@Px}S1$1=~@g$?Z|(Ht2C<#JDX|s0uS0(!=Meg|{)fJdvo*4M2WdaI?rNo6i1p)e-sMSzp@3'
    '=<10HCTc3oPz-Jq3baioJC0ClF`<ZN`T4|C^(pRl%}1tR(my=9a%|Z22kES6&u^p>fB1W?3!F=>T49<=IaIEt8Uvr$&~+&Zj0iXXB~gCCdoD(@D!NxucVWu6'
    '1LJRnI@(zY!xo7p{qzU^;t%5KCB=09b<#FG%@Y*7w2s&rVOFFq;MG6>_0N?pOA*9YitdBb;TZ)TpRhmWCzQs<r|d5zr?(X9Zu*_kU<Yi&;@fsRe-JmYJY>_)'
    'hZ*k|N68vB3kUt_I47+y*>NT=Nt*a+4rMI-gaSnD+>C8P(fMBzuLQ=YSQT-V3`e6~XntTxKrc%hlm28EZIxOK?1Vq^yp-faY*(TW#=3!!&-zBb2f3{H;tGd@'
    '_Bgk2c8$e4j<Faru@pN%!P1TEk8evoyyBS!CdKs?tBE<dEvDbMzx`hm>=3~OS33XYvwwNSb^otfoZAG2AKmd2Hu?_>)R2$tn|SJPrW8!lhUr&!ERUlZ3;L!N'
    '7|mwt@_#bjz56ptJ|?G(oG0RFZlEXKEZZ_avcTIEDMkms^_A|FFIAe{FWA34!`RIRYB4Et(T++NwDX`y3u7F8UHnzot$eYJ;q<ZEn)k~1FFLf(<?u|oW5sZ;'
    'aG`Rek1#v~gEeP&hDK#!T+$HR-`*E+!7lJEMuJ$JujJoUpU&ldb8ak0@2Lx@XKEhQ8zKjl3#LG9dD%4hrnc?<Zj4Hl%B>>qRWYZilIj?&_NWjP%YN|k7SqeI'
    'LF<5c%?b;RoV0z_KVlp0`{WbZ5&GqOgVRG+^;H4aB7snsyLs3TZ4PF|1o{kiH$SU1HTIn9uzj6IsQ7(zt9?6Jy$FHHK51Jsd#wzgA=~)Wvp%s*_Z7$}3{rK5'
    'iofALBle?56k;WZ48yk<Hbu6YKglV1+F&pXbuA*!v_qLi>xvGk{?w7Eyy{?S*2HN2!{@*K{qI?)6wcS6Ad@*5-5jT_G5Sh5Age9RmsK|nyf!_XeoEztr^8fT'
    '?=k&XwuC0OgpW)Q?|;<WYcl_*?rTt0gM1WIzv}0womi7X2s%dx;phg%*PEiJU=3@-T4qxJKqY1w1Ya;ys?1|J!x&?n;wy<)X#K10+4t#N>f!MsZGf4-d~bl3'
    'xjSH;wB4o5lKvTzQx@EC4=LfnT-}OCt=UTk6I1%-hbxfU_DYrSnp3@gj#dKsR?pbX%F|jCY$i;!(=U%^95P~ypRY-3YV1hsXHlfX!YMb2Scpfoq=e*OzNeV('
    'dvdSh@dEmJd`bm&dGD%c{P@fFgrf%bAxdGNXt%U(e+kXV2YEEsjt(1D3v_ube7Ewi>RTGdru2x=$&DtANAtRUn8IRutb7p4t8|+o+Z98V!*gFvX=%#MxEiM@'
    'Orlm25=lE?f*ad_lRp~k>*GX8#{dlwQcIPXbynU7BHGKux?jHYr04#vm?v-b`RJeeheySG9OORq_CQ5pBU50z9W{wpkyCNoC^oDD6#t=F`>orjt;p-DTj}nb'
    ')70yiPelNvbrcA}`<+RUfkCJ(k?;-r7ZN<Q6MkPjdBikuksE#TiLq!R61z4f`JHfG)o*S5G{TKE8-sNnWqFC7+SGUav~6?TPSMvjU>Aauy9^6Wk4)LeN9=U7'
    'vy@u~B@}mM^gT2xpvs)e>8(Mb0lRh5Al&OZ!XeF&X;|V<T~0xDLFPfw6%ljfP&Mdr7Xt-Vh`TIZ5?B-v>YAFcD_##y)Bn+5w1&0oiB<}cK_}j>AUx=jgHoC_'
    'J~m>V;_h(KaA1}#HqtRl>xiPLO|TI63FFrL;9gkA`$gnaUlF8qqU*P&vZB@mBV|Pq5Yr@KCM9Fz<NVY;lQO?iAj7vzfc!?;`M2`%4U?7I#weWae_}_-wXQAo'
    'twDnQqfc=vKzTolDi`kq3r8gOPiNAk(h!FEF<bGFt#~X>i7;AQlpMIGTsY>e7fv@(?n0mYGzcK>?OVWG?nVs-uEbmw;P!n~A@K&dUt7m-Zt)SbI!J~TPBw3~'
    'b2&G9z#^_+_G;Ev+pdLW)T94@-(q^oj?z#%(|c8O{R3j!>VIHDEjxHao0Z0RRlNz@$h1(CUnEOMG9u|-S1>x3C6um?VRWOid9zgTnsRjbs%17V#Kx*BrqEt2'
    'ra_B|>KtG5%(IG=A5wF|QE#(Y&~(T2b$1V4uNTu(JeOH7yIPqA;!oM1PLEAbnhm>Yqo4Yl4h;<ebPL`n)iyNTs;sDj)m9LTCz;LvE<Xl2c1?cMS9g`l@x7@D'
    'n@z~1s%l&ox33R1v9MI0lhZGBHe&@bXO`G6uC*TX&n(FtO9Uom`<!eihv45uOFF-=O&SC9@bTmt9de7Zd@`d?<#P(cSvtDl{J;lS<UafSi=Y4NuYTYE*)M<H'
    '|6e};`DYYsKK_t>_@yEP%cK~nAoOMKQhn>PNa0Og<19h_*zgpSbNP`9i}Yb^a!UQ>%TXcB@>dMmzfWJ!>{R_4os_%a^08f2Sn5Mz@MZ_zY`>9@Y%6TFHz0-b'
    '8GA{AxD6o`a#XAA!%B6))IJ%kkQr!8YP-7dN^6U!t+Ee2VpyeF>agc)8%x1Qnby=0Q`W9bA+Bn$ou@(Il_Gh0{NRaf!FtuvOdxe{9_<#@Z+4LSiHaB5U)hF!'
    'z&hMG4r}yfKRbKQ<2Hnl5BeXAL^G@AcVArH-fsEO`aRyywSiAk4_y|yv#S%evlZv35G<4`4o}*s8m%&qbOOCt&tL)Ma0wA>#8I%wXWoHg3Sx=}3aRn-xdKkb'
    '=uAu&(;wMkb-U@0T|B*%x;<^a_4aXkv$9=&P`}%v>WiP|@9s7@xZgeouSb>ld3dHxNF5_2!lJOBG{3Ztw9i`c>hz6T<dtgB|5>Sc#;VZH6}e0GO{tqxH9sVr'
    'iE5K%^N<*)IixtRIr6pjwW>v7!?M$9S9!pUyX5th0n=0cwy&nC;$+-`@$u!q>$|Sq+qc<wZJUg&joO~}^W#IBqw`{oyJB>qeRNm~Y?W_fw!V^9|GDk;*JBey'
    '?WkW3I*i)~pZ8lwhx*6I#?7?wq+*H1M08>@Bq~^##SDD3+#}Oak65Lf``acpnb>#?4Rc`&jS6ik58ZS5w4W90A9&qQOR*aph7{e4W(J0reWu5}J&yhCuOf?`'
    '7{fBe9Czwy#1}kyK%Z%V3Jkn)K3WxJ8{%g-r_2oNd2!vgV*Nj+1)p%w9<yvI;<KdGGk4l$Es80!lU<`v`v)k!8=dUu*K1SuK5S<cY)ZSH^o{Z!ZlRzKUIM84'
    'g|BjJ8kiZtR#KMrL$a8}u2?gg?gpfuIEv7!FZ5js+mScbFMSl=QDFUmfoR%hx~uSA4(w_p<f}&}^Q*`ApRi<I6679HD*uWyTbBHUi$VH<_SmOvFBnnP;I3h}'
    '(yXMNCHkdhGZfQlZ^U}}R-mH(?PMR#u|@U$Kup#c9=(!f8qrPmni+89OB_bcuXN|Ym3aP2I%y})f!9{ahrhM4gK#&3!z}?IPg3<(3>c7ub<B#>c;*M)-G4!t'
    'JT-y`M+>G`-&{JjQLX)f?e0gnf{u_3T1bCHC9}J7y7ad2%#(4e5^r7wL6&X$8UU)KT{=Zso?#CRYJ#M~*n@4H{T4FG-BFC#fH~gdZHWg+SuT8O^m5IoDuny>'
    '@4gp!KmB(AUQsyycMf3Nq<x4=I5)}wW8sEMR@_kV`9S`x$^rA+fBi@6)Q_OK2$Pu*4OO(8U)V-Q%p41@4kPL?kEbleuWc=snm215tD?z(tx(qhRRo8aMqsAD'
    'x=lk}TDMZWxYVU#tq!+;OAKEzGAR$Oke*OtSHm{jkT^*N<Ujw%PN{wLP^T}1zs#9+n-v(J82h8J1Y*J#O=G4zrW<S36r^mf67*Tq>};2CPQACYa>S_g7d=mz'
    'C6LwE7<Fw!Ux;1an2d;H7rZpwXVLc|9oxvX<S=bJcyF#~QyoCOs-6j@oY?TDm1z`3nDxnenfOL){$QZ7@r=jU+-0v?7fX8-2G`gB<LB)9dIc&<!3K2Rf-)yy'
    'l%^rwVVF_d)HQ=@k6u5D&jiStCC93z(|qI+wDHzp8{II)lsIn1mXh!5lrv??ca&a(M|u~I{(!Ll4ZmQtPit+|loKv5{ZyOr(>gtA9T8DebE3vN`P|6YIAtAq'
    '&Lni9L8F?JRF>!Q2jhkfFz+>m231BMd}LUkdHY}3Ui3eI*>_(%TY2+n^N@~m+o#Z;c;fNq=LwZVGCV<Ol$7Ml_juZ4o+wbLtjMO#GyhNsG3&f&oK<}NJAL4m'
    'LXzO1?I2EYsI#l<Y&?Of+S|t$ljV=)KXegYlh1i9*W`1RBTYeE+I>sy>jKvLztU1{#fG*hebD-<&%b}wm*2Chh%Z8IEqzc<?|lic4I!(-c_wWspH-No`eGDY'
    'qqO6<8K71S6jFBx*9;3I>NfK4s%-zaZB03ODYLnWZCY^yx9OM`RonZU@!R&3rcJ&l6mNKC?n~f0o3!Dy8&F^vU-XFc(sbUguPCAHmW6$Z>FSvY=$>d|(sCM^'
    '#)$VP2@mRMxptYK8K0TtUvB{Oc1sp{lNl~3EA;XQG^hfEw0!JjL#MUm;1r1`3v>{E={KOfi4>f<6GP>4n}y2-Of2BSfYgPM^r}NeL)LpOPkg22X=9@-7|4kA'
    'bmD1ZmHtlOCu}4AbgvnGYG-z4-f^Q6>?#g{IT?+}_M)#HNumm%r4Y23`neZcMO7px1znFjq65RCU=9`ol0`+A9&O~^^OyZX!qK+GD1w0rI!aXfN-n6_jL+ql'
    'QbX-;FAct_c9!TNpp)0ehP0bixIkz*i}XcubZFD}yzE%<rM{xkVAKn#PfgFxLp~dum=b*179M&^w*n<IUyn_X44H(h7Ohh(%3XB6i9RCnjtRzG#zw_6W0S&t'
    '9P-r~8fiwT*C$O{bW`2!WhZLC(!NvIZLgN}>jMpvn#B*YlT7&2-DxGBcWOsk5*4zvL%sXIl6q1!{ao?=xuOMm9TneSnqM0AKykss%PB0qQ;!r1^OUG5o1dA6'
    'x`h<c&zR!eMK@F5Gl~diu?;oJ6vAdiIY}4ZWgUIZP0Iqh&^A~>i;8q~>+Z_^N@`BpVnQW#CP8lVVUZ-G`fP>VN71cY8nIQec~RLC5m_9O=tWbGuR|-+*n}4y'
    'NB6vqaW@>NJp1<2d?ofl*T*ib4D<idf4Vx&v0e9hyY3(5y7d45sI9A7NO2H%#1=cHN9f4t`3H+h;>=o{6LON%K|Pa9Gi_uW=o?_j))o~k3NEi(R2}JJ<)3$?'
    'clzQl8Af*~-llVH;22$4tpeQ_J>%9PMFnXtO6lW%x9GW0$9QorKE^NdmVDq0E_Tz9g#+l1ZCD8|ake8nnJ8J`{gR_Y?E+<ebRg(5-__1j92fudL7P1*ThT__'
    'h{ZJ>T_x-rPX3Q9RP4Dh@#o^iSwV}jZlDbyGn%C+nsk_>`1L9<b;bu)iK~saq5fg}$Vk8WzbkQ}y(-<>Tp=S0JTlsFf`Y0|jKD%91USo-=8=ZEO=u%H?8doQ'
    'm6)TVA(U}<al#{z4uSHReyS<)$94%L`l8w%;>Ms=*bLg+L934%L-YoqGytL`vS_riZsEYPR!94(3Z@U9=tBL+shlZ6p4HGbY+yg-G)SOX<BxlKdMqZ|a?kYW'
    'B$bJ5!Y%MD4HfebTV?z+03t`!pVFp&1V|}U8;g3QS=es6>FA>9*u8EVqM&R}v>}+qq{ufW3t}7BlLc1L(m~7bOcw+-P1m}P;tt6@yzJK2Zf^x{J?;|unNV32'
    's=73SlRCyw6<4S;;HoKWpZ0}Td(tjUpYF}oE)azVYvxvA^>9@|jFOVfBPS<+1Y<2bNAyDtv;3HOKUe673=6OeaD-&rhM$?Og)>;@?(0}$;>G0E-?>BI@UzHt'
    'cnY$C>A~l=DHd+tp%7a{mZ_R0?CX2b`>B5Uxhh40KFltXQdoJlV40qr8harf<1*6JDJ}ZiKKQJEaLhJrsHj^f`X`uu*4P9fIN`1s0AH8u%e}A;S|`Sgzp>50'
    '!ESK6mHDUS<_&90(FnJ*e-97$3)QsoIR;TaU9@nQRIXc18s%2{uWEpoR5qb@VR7?wpR&E4r?x3`m-LO+)H*u!ZY0^7tviJ{&d)RRzecARCIuUpu<3urqT3!n'
    '(KpX;^rfgPglZaMsMgbIw<>CM4A5mJFrFvRs7XPcQ>&#y*zDMpb>g+6eKA^+C%RDaFX5z?!S*!$w9jxuQtIEno`x0>FT|D18&Yj4o8pL#-N(spre#JBeR<{t'
    'k(%z(1^@EyWzAdXD;72e1GxB8S4Oq<I3rU$8(rAYudQM5R@n>dRmeL+%gPZvlKonyscj=C!>NDfLp5wdj8Tyq5J|3dF@RK1Q<>YorD(Z!DYXGrU78N%mqU!R'
    'O}&=;;^bS=l`?{AfU<u|CwW~VaLcHgcAaPuC2QicC^Jj6*^ySi5bO^910!RDOcbQVp^M!c!TuTUQk_pVYWo)L8a1gq^4eiI!(CeWP9)&hRv+%Nut0Iu2V+5A'
    '#k;g>7=PQ(B6#`TkjkPO5J`%!#K#PhK=rF=Al0{7WTa|@8o62|8V{(9F1O3bh1?be5&dlvK8y+Hk+}HSbv1m1;7cmoGZ%*<2AcugqAsm9m9W0W9owGLy?gBe'
    '8c1FJgIu_6m%n4`Wpy4h+b;b@f3zJY9J&<LQ)9zaW1W8bl-)Qt#B!AU=I6hqZH(I{^dtJ(KExqwZuj?rqHl40Twzc?7=l~BDiX|jFBInDSCarvp8jl%j(g&%'
    'P0BkbfTBx3P-z9!EAoMav3E@7_H^zV48>qIM7G^-{Z5DZsgYT(s_k4@BB67sP<OX6x~t?mEy_y=n3ZLikx$T+Rd0VPZVC!q>4S-V7`6}cE1TJQxRD_}pW~If'
    'u9_yNB;2^A>QF-(*xaRgtLLNykLt^n^h`YlZC9cUWF!U2breqq?o&n91Xh-=6h99j?r3Q@TiI?_G}NunCEWfhC!JJO(5KH-l<(^^(H;!!n8FV3Q$*fyC!?Tj'
    'i9Rt=#}fXqSYCzHsY{uu?|1O~_IulohOg>K8KlO$x^qVs=({^t6aI>h1CgN#Y+VRgXPF9wLN~mpf``d`{%Vk-6`S=iJG{m6Hi|v*$$ZnE*tcD}l)CEN?P9_e'
    '8IFQ&OWg9ZfYN;hWaL6-X&$=%R4SF>?Hi&b$VH)cJgPdP?#k;slnuxsv%0Y;R1kJSI~8%v8a+hcwkWoljq;HQ(7SRx>W@#l|J^#@U)Wwu8f)bIY>jwWwNvDm'
    '_*qP=-N|C5{3bgDbY4LIMV3QH4RjSYR_3Fie|Vlv;c#2q-90IK<%H{cC3lC0nri8zogIHNxAzWxU~6v?c)vxts4Uom?qk}ZZGxpRmve<G2&zSiZ90#oeUgrX'
    'A9LWRp!&pH(G)fK-Dn1z=_X`ZTn!1wUv;8T%)KPRksT+yuj^2nv=Eh`8;nY6G#CgwEhI$Cf8VKa<0!`?Mr+tgu()oH#Db|>6B(j41X$##5~coov)n@MkP`V9'
    '_fYo9!27=Z%Xn7Ou9gEt6w_2~U*&^EJIwaZ`Myj~Fi?xb(k<L8rOr<YBcF>LuYdJ2v#UNN_0!ldsvsB=`tp}I!k@NdMrB+u=txO@c>6tC1Nlu>w%wnBNuyJ*'
    'D$3TWZ{1wwoN}^>MpFDmQgZX%$L~f|5_K(Y(ok)Ef&+6$lG7*tHP~SK^agzz7lE?00VNVn5*RoZjMB_V&ITz_jK=A3#rXoGV`-v((<KP8T>{*tlZp(Rt)s#M'
    'O^M<zvjH(tPw2ht3KOx)1);q~*B}|KoUHAKc}?yZM1zYN#<cQ8r;Kk3UTV=@S<e#$7>Xyacu)gNCZ`K8S7vb2eH6nX#wK1%N$QNk=Iyq2X0(o86_Gb^^Z)Iv'
    '_#z@GBxofRw5~0xlXYp#9c?o%yA5h=SVB;C!Omwk30qXwddW~P3a%=Hy9jATc#-%<Z>W?LeTu6o26>222Ia(M63dBC1{$iI#IF5d5YkRx_ZwsoQQ)048QlRU'
    '+2npz`V)EH%Je3T)^)hWPk{$ADOG}5;eHUMNzvec-yknazhMNt;zPHAsa-f?Bf~_D)m5z@7i+~O{yX9Pqh$4#OwGFUUTTAZO5jy5onYrz8qm##IHO78qKsD_'
    '8GCA<n$({fXs@b#MoG0Zg;}%%yrxY5S%W8H;+LK&vnJkJ3_2H)&djd-G(8?1nS9bMWOn<t@csA7G{PEFqK=X%wvX-`3pO#1YFE|Qy)%k&SJdbn>aK3KQ-r#~'
    'I92Ca+W;G(Olf$UrDl`UvOXUj``XlYIjKNu;d1@BZO}e!w+-p!5V|cWN=>NAUYBk(hSHsKd)*L*-z#sB7KyIO9&~H%Tc3gRHYdBp+Fai@_UrTSogLd96!p$-'
    'X(7g4y6{_svMR1!id(mIKyZT`Z!(ZIDd0ZnQ&DZ>1aS(=s1Z(IDbJRQ^ieOrN^>xbBjx&uaQnD!OEFw+2iZu`qk08L8{Ph@yM6Jpc7w5tFz-I2yK=<F7oE{J'
    'A#J3fz_4f47V8u<+9_JpC=gDBz?G4qT3J~Yq$TV_jWW$~i~VEK+I@rs$_9xT<OMU(Gz8Ew+X1_i;gL9Io*CqkjEIk(=tHa2{DrnEaS`3t6FA8$WBrEBL<7&Z'
    'p-Qc5itjZdr|s*0(bl9*!ZU8QPdrkf344}<f-U^^7y*uA!`<B-Y*e@Hm~l%n5>P9t@x)W|O?Vc$6|E~t)RVa4E|oC`pn)FCk%x|_)7W}+fnye}TCY<CuNt7-'
    'pMj<~CLDQDC0v}IvX4yOvyO~Evkus%#0zqyM>=L9J0i@6T2(g=Ht?y3lo!F;0@}-hMK?&U!v!Z^-vY}X5LiUPJ>vG_lOtU65LXQxB+=<PSl@--c`}zlfqCHF'
    'f$;;d-~j8FcwrTkE<qs!b4j$k2}?1wTYXz!C)I7@biz~-%oXw6CElI~wKQ3Y;(0$f^TT53%P+qiC6ztwuR%`<?uAjR0unA54@1W(Xp~SU2M$AE_82zT;6)Jb'
    '>|oasKEG%+F0iKw(ig-PMa!!=dk7BWBp<+M^TbtZ)z`r4F7U3Q{auu(!j(h3>;xwT7&!#9HIgop*ePDz2BldP58_e~`wC>`j5w>vQ%0+o@OTXu4`KNb)z(qL'
    'fn#A9x!?mmbb;+<&|Cq&0InXqt*_zmF)lhtB!jk`Fp&g}>T5J#L8%22IRT3vRNDZaGnhNXO;_t;8T!3=HcyuJfV+UwAzUevy(MtD3rb5Ql>?;%95`w<D=-uT'
    'D<QOch<xc*H3()KVCM)fFQHlqr?cRA9n5=?dmqH-fiDcU&Y=GgF3*rmnN%})r@}^xPfoxg1zM4$)<NEl*B4v0=4W`(2Ym(T*v6hZz6^tlZRDAU*;VLsqPi0w'
    '?cmEQSXn{Q8YrCMlRSwf$^Ic(S%sNJaOr`kbKu~FWSpQ-0ijjmna3{ocg_>w+e3kK;A^4{KiW#*#RxcGNA(Jh?Sn|Xb#eZkV-6l&g46<9o`>ZoSlGaUdE!|>'
    'v-51QD7;Lz&PgPW=FdqrfubAO5dgbQyplrBeLOe+w(k1>&T%sOw&6#`0xmglX$x;;!ORBPI)dIya5+a-!ziADGy5p*!RbvHUqQ(^Tx@{253WVu)*E1Z75j>~'
    'xCBpXAeO`PNxXc3{DoGf1iXtd>cG(%NYS$qhnWMoI|rB6S{Ji;*?~d{d|JZm9yU~XnZ}o)xAhqu-$BJo<Ub^3Kl07tSOpyKf%z=(UJ#E9Z=QgS0NG2B)B>pI'
    'aHE2oP8d9cdk$3FgUct_A0*iWJbMiGj!85_%5$Kc9K<v8q<V-BOQ^Yv*UwRG3%S$4u}2OMp@WWV2h7cocml00{Es8BxQ%khV1EOZkHA&}E@n}p0+PodSOvvd'
    'HdYWSVCOuFIN??vW{Uq0^agOEK%yHUR)iZ4usaW<0Tged(+s|xA#`Z7Rk*Q%>IJgo!{IbcxpC2nH#WhJ7cN(S0<+#$J>04<!i)nZ3uw&=z3U_zg1dPX$wF_I'
    '<Y!tJ3()69ne+cqLg_NxSVP%k95}~`(svFQnms_d64=}W=iAVq!qGaO3G$IPk3gn~f<fqyp!_k36hR@4mUiKMmUy;EDvlTS;mX2yjv%h=!R#8W?ZEQ_E}X;c'
    '7W8ejsxfewBjq#XS%4ISFRHED2AoObVjS)iQDXg#BZ>n)w6+Y35nMR})do0dqP0A7S4pghw<CmZ($zg2TELkMnycc=h7C~yl3jZfaS&G(&N#tN2CeM#!LGW%'
    '#XM@pVc7+j=HPM}cy`c5gG7?8%QW=o5e0g^2vTQn!rLewz$Gu-oq+`xsK()ji@2j`#s`m*|FaBFBX7c+Xl@I|j)`jzB>b)Be5;xVj!Uwi$I(!$86e4hoSsM7'
    'E#yA};WSxTN88)LoyY!#R%3%WqiD4W7cb!HGNy!hX#pm-Kw|~27CE*aonco2hZ?vTCayGeE)&N(NMzquH^FuW#kXMglsFb)c8TORQSB54ym&bSHrK&&9nQ^z'
    '$}-7h;K4bZ-NBhll3M_d1H8F~PtHNWk0O51Tq3?j6gYzUL$r7ZinTYMP3UZr$_^-><H9M4L}6;1tVckdvXL<K?SX&~=A6LmM&45t&f=UO1W!qn6D^dAfjp(B'
    'H4v(y@C@9DqNP)uibC%>@HpN?TzKvXIuGG?6Bc}ED}(0GK(&sd88~-(AI&e~s1L5@NF)J5Wfbs}+7gN7QEK&V{fIb1xVVh=PFfVzW`oe<g{3lH%Mr(_1uV{^'
    'g$yokpkNp_cj0jZmO}7&9?n-=6+bRH(Q*pLDmYyQJ7JjFC+>9)+(Z-?&rrRL5?NSqgNrCwY+}z5%58w-8N9d&lV@+dXE;Q6<vCbLwHj+Mat<mXSlYo0&i_?`'
    'JEveaPEIf3+5+~v;Nd3NNPxs8+TOtNc~Uuq?tSQ40O=&AEMoTr`Wqy6@g{x*yajl%h%bWRatQ`!T9svx-+ALbg-6>UQ$or8QP7C=qcYt*1yG5z!M5sI5;+CU'
    '5Lt}?Z-h7-t&126ufp&Qu6l9V4|9|kh4FlaL}x)|j>IA)dI<KG;mjIJW$?lgi8*0<4@OGxxDFh<Aee=bRS=Hj@GQ89wHk5cFX6>=5M2hI99TMpwPkn~C!SMq'
    'z5`}GXuAZWVHmhXvoV}m!mBR06UG}3yig^_85pMhQntNHVw}w4U>W+(@M##hJa1;)@OTMr#DT{PN=G1g4mYYGxCT6ayuAUBTqNp5dv5HH!Lw7autIis@%#$8'
    'aDz$$2g>N8f_9eiX8KKX4sERhp9^fBApaVm=tP0*U4fB0m<_|V9H_=nd=G9rVWo&SBgBzJjVj)uj3JC_6qW(#P2y4-&Bai<ihN$MdW6eaP7+R1IO0N$5YD)X'
    'r-_!EFq%Wo3Ty;HJ=?m-!kG<lkcQ<rD!Som9nH=l=NTw?h?|O5OE7r|nyFUp99J5s?j?~mytoERIk-3Xw(dr$ecU)~RpKDJ0`@Mk=MWudfx85QyEuA*7L%|%'
    'Lt<WVyomyFlwJfUF}QI`j#r5zj4M8PMt5YYb-9Of3%`Yd11`txt)bl{Ow6GZ%H-BTW){@yq@2J<6_C!Mx&xJ~X!DZ9f|%}sdL2^fE}lilYa|*5<s4op!ubZ+'
    'ti#ee%3ro>TVQ(&M5(w!iOW3luj07_c)HeV7I3T%8wHs3fK@+sACj3O*o?!4EOCcnx&-Nt3TMD}7`y6tbq9x*TGb2;&O%onti@ZD$gk%(Hk_uhr-m+<QGE?<'
    '#Blr)t!=<c8B{NTF9tn3Amc?bN^BOu+&)TD;bIjXQ=Uo}zV^1U3Mj{lCR!J9(Ab6L6ih}?wgNY5Aia)@1+sNP9C@@A#iyCKbq@$P$jl6MpTXoV*(yO#7*v~N'
    'Zytn>NSeMZ!t^=bnR^q9;h+m$>;uX|UB@Ug2fbb#3b5h6t%h2aO%&XKml+VBL5oW`Rlt=XNJc<;0p(8MmJ6M1<6HvNz4&6k)ucTB5UzVbx(UNe=-30#gJ`>g'
    'Ph-%10%JvRk%608d{M+_RkRmD+e>hzgj{uUxY(+cQF)J~eXv?a(eqa21g28JA0m|o2-6MZ!l!#^^9<x`xD+6j9LWUGVFV@0;K<2|SbY~ZDT{N%^JAFWfzCa2'
    'wD|wR@&@$f$l1(q;8C0;OJHXGJI5kAh_x<fkhcaiA$+=m1IM^I2R9w?tblWL8I$C40lRi#gA$N)7~F$}SrC~;1rMU);@kp?t>cXywC{)Olqe@~Z~;3btwtKn'
    'lyESNTuWqaAE$EgBLB}YxeXGvxAiLV%(iOIR;|I=L!%5%!YH{%W|u+14;GKm(J_c@0#6)o&7jH}m`{=P3~n@`V+YleAb3FX34EDA&SI+)#^<LfcL@WOkO#nF'
    'm_(!ad=L7HFn&&!ny45Bkr|TR1HmFVj=&R2WfR!71s0Q}x(pUBN%<6(v#7R%=Yy@vB`MU=ObQ-Rz&CMj5k$+Nu>{YlIL?^^4)3;VaS~qu<rMaw<J{r@0rRUU'
    'T?hUZRE`t(C9H%=EsaW4GOUtn9$hp+G=-LqLBjzq{opVOsBp66!W%J?a^b2U?X4jHIz01z=kUSp1sIqk#~~cZpmYTngJhZN3juh14(H>@dro49ARNLDN2|I7'
    'j=X4R4c3oQIn!zc@Oc(29K+*dROA={G8s|}gTOK@dq~cYvzK^%A2&;6i%K9XWZwazPLkOm(MzzGgNG?_TEvxAwC#ufDD=DF`T|G<@%bhwG)QO(og5JN3|`)W'
    'D=zG<!o&jdEkd849MoD3s=zpL)%Ug@BI|XWI)W=95Z}W^%AL=Un<_cwUU)PQmZLbD1+E=fS;pI5<UB=Nbw1MIB1utoHHBPpxE=z<b$qc3^L4bk1yV^cyV+`N'
    ';*&awl(8d&OPjC~!p>cAetrj4Yh-T=)>d(40WTIwd=96=aAA#{@3rbw7+Ar}8+h&z#MW>jkE^@javmm%ARB|J7~Iab8VAr_#2L!Xs$ek=@(EB#BG)-u^^=WY'
    'tGR)8j$z4zLUdQTIDK_^aG?rnXVAMu@<F&A!iyBX2e7;jt3}{Vf%AQsxghR+6xsrpkya%KLx+S46tiA%>4PT?d~%507bxRHp;_#Yg57PndX9ZdXpIU-Mcl~4'
    'xCd+npyw2OLL?9ab3x<^fTaMa=CD74n^D*}A+ELW90^jL{mzkuE3^2df)}Wekj36Cm_HU&gec4$VoKi1O>~%sM-DJsLfcg9--NkSuo^)-eq1~T6iDF+cKFGv'
    '2kj+5&i!{Fbc&KL5~0*60*gtKqs?EU5CzmcNT0w+0q)Gh#u7fE1hzm{{lFCg$0?B81@r6R#D$AnWc`AacaZCV#Hj$d4MItfkKj}lR5x*P5uctx=L$aJy9q4L'
    'VOIrjIMHF5EN{b^CY~vy4JTgD!BZ#9<;l$P|2%13_`yXMolwPi9iF-|l|v(QU~3t~LL{01!3uQLup<FHzE*VsuJ}+s2omQwv;yeTpIv~}nO4<>qD6F&#|zu7'
    '>MmZO^t%FfX7K3@**(K_M>v<UYlGDNU}F;&*5TgToA4GqKEkKleAlOfFq<WgBBatlphQx;z)OMWBIy8Jtby<n-pT;FJC{-@kb@_FP@v3n3D2KH{}%Qhv>HL='
    'oh6=ikn(|G8SGPL7{PHjPM7fJ9z5H_UMk<OliV8GZ@jH9efb_q?YylwF=es!JjxxBT!>^=@k$b&grUPpoLeyEgoSflng#I#ke%aX^x^=ngh8b8opX;Y1@OrV'
    'oS#8QHBzH2<^Y81D71;fPL!y_^&+@*z{(<A^nhv&B_kx_1BVXeZQ#S17Uis$N$Af(=O%J5!KEF%G6%y>Jiqm3$%8XnaD5HMGx($lXExE^B90`1cNd;)lfy8i'
    'eExhBmh!NYBuneS6aEOUZsXDp*m8j+7XeW^3>S06wSwnnaiNLnUO8DJt}x2%qSOXn-)+?o;rw#zqCnz@sI~wW*RXq*B<uJxLzc4Wq6YH?yl@D-ha?w6%Y|0`'
    '0#cG1r4mo@JBJqsk3ixWc&KK#j-3mzyg*W?C^my?^C0L0kpMnTknLGWNp@(aMOS=_F4r<Du9HjxRAOW$fg%nz&^Iv`Os(R=G1x3Z|1olBNh*yOJg~4#s!RCX'
    'kAh{q?fk#-UKErsP<;ihZKEvJ?-$@w7I;fAumG30@n#HA{I1Ty*gk4(fgP%n<>CAR^oPLG2?(x(#UOTX;k5s4eHk3)L3kc)&A?m~E$pLM1?6XOZWA}Xz~zNi'
    'IzSI8uVGIV)#_k}Q%|xH!KZ%U*n+MaxTI3u2|O#3^e#A^$KfPeJw}BS<jI5T*>{cwGJ6awCwMW0x95o44OjO;F^kvN@d;Hyvglw3=3Ow|M6L#zrEHOM<T+IL'
    'f!Q?_al>>4`j^OV6{Tm1dk%Szz~&4*ZIbvtK2O795$qp<&^`!L@y-c1A}GFqPu5W;4;}l0>1q#DD=0`M*9zIEg!ByOT(G&0)-F*ngceTFYKfGoP}@N6b1>^9'
    'a{-(_1cfG9*#*fmO2%L@1w#=S*uWbn$XkJ_bu_bt(q6FUBk?8-xk$3uYGy%}Zh<^p^~1vr*jy#)9kj9s{qrQYgFTlpeDNmcAju;*GfNzGklBI02u{qA)C|bF'
    'x$Lw)+p4bOKme{#PE<xSM=0)vlnuuGWUJV!H&AUAtggRpxRJk%i$%1$hiI>}KD3&}=QChux7Dbjy$hH<f@=;G%b;y4cqd7ms#vSIQO3<ctG)>uIh>gz%PEpu'
    'BGq}gxJy!H*l>bi3CB0#DOD^IZz2oia)zvK;7d0Q)X~8#NX(*gqgCN%OB~w5?rl6z8Lk6XXGl4Rya#aR92Is^E`UPmH;H5D*m?-hk5Ds!ih1IW!)yj6=RqTi'
    'OF3LULCG}S@Wab>7`lM*6x=M4-3DAaMVSgXTEpHmc#;Ntd%!z~&sX4X6`$<jN)C9dXeoya+i=AV=ffbh2nr>z?<6ZZ5_OZf1D5uQGYuBYe8<;zk>7<<m*8X$'
    'o>0wZ4b{(pCjux}DHqW45m~RnKoBexV0e{WRA43vq7FQFf?aE1I|kO=AiIqAizIpiD}nDEVVFNB$8J<xA<j7*I6-tkB?=Qa*(xJf1<mA9l4{FU5cT4v3ZBit'
    'vt5{^SQ&tO2e`ENW;P0D{G?t(baC@e=ySux4Gzy$H!NjwWdm)5{?A|FlVh+m2NG_QapA-!iDb#rJaN$B`C1oIOy!!@G~RHvnoDSjep-NgyQmoh#W^s0j?X-B'
    '=MZL#Fhv1fA$xNqQirKa;yA@c3hPz47s3lWq`C_7Ww4Sb*$fDtf!aA(brWX<C7WR75Cx*Gi&I$N$1{s?#)lV{!6H{gz~&a7rIJh*F2-S~f&K9}iFL5Eg@Xx@'
    'UPJy}LKXTz1W;+q*T9)*t1=JP_JBVDmi^Efe-l{-316$`8vg?bEWH4pS+X=sb`QyJ5qR7!+FxiMuRGAvCR(5pQ65rZpg7mMSVZ-UR&5T3GSIgOHYo>K1?9wl'
    'Lx%~nx`-Djq$y9?1epyOj`3k89H<Z?yXUQH1o)euf}=2=FW}2nxG;w@bx=RWn<cR5B~<^dxZc+HU@ZvGR?%#VT)5y&;B9jY$G!N70`w9jwn>Bv#AS3)hZMM_'
    '6m~R8g$fwEpmc&uezYCLm-`@?go|e|n`>1ZIJt|Gi|}}lT&&>r6xnJ%fc_(Jo=22Rx;D`MDi*B&AfLjUDY9~m_ERLf2iN9EdW|e6@j4YWR>A%r+zgPt5XvQC'
    'tO^cO`1}An&akHdW*vZP_UnFd=0fu)q<#t4y(C#6nJ5VqNHm95YcLT;dw!DIBbQ;~rnIJxmt9~sjhFmrw@H>`D7gw<MZm-eKR(UE*<F<1#@^7|dKn(a!I}dE'
    'cCldiMKLGZNutIwh{y5HGTchSB;9a+v~U253FwT#NCh5Ik!l5&_MpoLsjQseM=Q&q9>?xYkT^nH73gyi&m3IeK$Rj|p9i)1R?R`m8Iqa<^(~O0#CR92MX)o0'
    '_GUmJ{}g2pKxGNeM@ZR;jvBC3fk{7D+X00oR9Gj@WUEfK=sYNgNj$~r=ZphWKo^e5#`fEK6a*8v>1j2>$hm+HeTdF|brEfy1Lp#8<Z&R1T$>~x$46CM*asya'
    'YQ#|O420sSo+X|GFjs;rQ8crRd{ll<p+pd^9)Q>!Sy@BQLlR#^{s;;caApy2(4}#c*cqw$$kq-_))19Fy!*IO!~PAj><4oVH0J>R3wV+eIMP0dY=EUR6r(c9'
    '5j>^(jthk|t=bvd*}w~Nlt|)@V;nD&ats&fGPpr`9~bsuDF{~9TJ;*(OT!f}Xl|n>)!dHXG8?)dl;W@!LuYwhDiYrbbkoLtaB&-7c;GsfQP*K4jvBfD4WDJf'
    'c>($h=wb`3MoBJ>y?ZDcK~#yo@WW#&gB1mdDTru$tKt8SBNQVWaKQ)t6j8Pj6)ECnw9q8EBycwId<tDq!8-vr58*`yM7Gg(9?Vs7&I=X~;L+-v*a8UGNS#va'
    '6rvJ;jq0`sc%^{0XUSF-E*^qL4N%0*xo|v%XO^Mw7@wVx12?Fq;mICaZNLImr4L~!gtiY`)ni;LgWXxQ$EgyDY~jm!P@2b%2BbKipZf|#Lhz&v^BL%9!tDlF'
    'O2bSXZ!dsi9aJb)^rP4jjOVduzExj_=XvO=L)QjQAA{L4+S|h09_-o!$ql&Z21ix2*92Qha=8q4B5&)tH%UL-X@G2mY-LIEoH)<$`2tBE!tye5dESKcc)o}h'
    '53p+qr{{i(ibv2dP|vjz2p#^c)ud|iJfK=*D%+}Tqs2>D@WX5rXCly_fjjG<o(9E*R@H-pyR8c^*v+9@4M!HS_Y~}}gJcvq<1l^FYVII^jX35}J#i1MQqj%x'
    'wtj%t3)thur*t&?aP5**XW;n(yzr4^4Q!sjtyf5L4(Huqa}z}KX!jiD9Uzy)TeC2bfGdKL-*tdbw(xQh`l7_)Lp3U&RIzIT>@T-!2@*|$xlQ6cgt;A9N&t5a'
    '&&1H)5@rJ5Hc3@LG6o!3m|npF58hZJ(KEOk0+9qxUBHDsv>b-Xb6BN3lPU`<aCI5axbSHZT&{!V1F#!M<uIZGnBPxgTW?~NXL!NJ@c;;Jfx~R8nu6|GP@Lh*'
    'z8r+7Vd!tdYy@0pS{FMY<U`p@u;9hX3~<EY@hK_0K&%Olci>S7oaf#)5^w7Pkf*{>mDJbZ&JheA!psTIu0e`y;Ulnm3D44CegmbCU@A{4M<kjivnRwEBk?m-'
    'qkJ(>whqDe>_36VA*=_%lD9>LGAaqpJ5YKHxE%k2J+qK1sZqK=b~!jRS>OynKUKTEpm7eJ2VkQB5_533PV#<4>1^ByVwdRf2v=)(`3OY|z;l2uHt^9go^#{o'
    'F^R3CVhS(D!0rVu)Zs!Agwm+If{yBl3eTrI=p@%_G+{6Z3#V`<OWc%2Zlhz$RS!}0gaiv<<s8tB7CFU^JY1_|ALaeWWY&+<yYO@u%*_ex$O&@3R(<>B+d3V{'
    '3Mj0izy-WGfek9up5pC1bW{DCuEHfuEyHG*9MoI&B~socnMIf?fMgaoD0|u>u?@Id!6z3WU4^+E@Fm`aPw>Jq$t_|^qRL08)&$`^sJme*j*I85N*dHCx6H$%'
    'GKu@(QVJ(_aL|D$XU>KpWkZ)fP(Q?%HNNiUFs_zhXcPK<Z^CurK0%Fes}h635O8e~cL<)RTD4P@aKp<Ob{w~=(Klhbc5`Ui3p_F0C?V%1@g(88gET^5v4R)e'
    'pq|Ix2ud{Yc>|@XyuS~;RBS(js~KDhVfPMktiAEZ(e@(Pp8?GX%%tI*7i=D(0M+qV(43Dfds>wMJUt-GmpCH0%yawXaE3%3aLtbzE?9Gd^D+#$NHqx}$GCWj'
    'FFi0_ep?Nr<{HRv!a7CS8C;@tFadMhWc>`KU3ex$av2<Sl5iLYGsL-w8^<uPgw`@RR>SKhT-rtZYsgnXE*~zegCJEq&tWkN&l4~oBkuWDa~Z^fD4W5#bCNnk'
    't{SNvgNr06FB4aR545;Xa&!{&WX}tNdtf$)<|D9ih<qz#V+Z&*!R8|J?6exYps<B%RE0eTD?z*+0D&ENN`<B@Naa90@y1iamnB?{5Lc08sODHkO%HY-LAs8|'
    '3*^v=-5C@*BkmQrafB-|bm@f2#2c!^7t>_D0p`<i=?JgPz*-b-I$QM;imc*F7ObBBuT_qqks8U&f%G!TZ^5;5;+O^dltNP7It_fgq?W;j3o^SvTnDXY5M&Np'
    '&Eg-yxeu&*fG30YGSGPeo3!X+i>`YxjQwd`OaZF?FH!JQ{EL9*EUE<n)z9MbH;x+IPJ<N}JYB@6RMV`YdI~MO@HwT$epHX)Q!l3DjvkX}3S7D%W!6rL<ufqL'
    'T_R{JKn|(QngdR%ER|Y~?N)6cP%Zw@NA^=)(3b|Y2jC<ColZh&Tx=Utd49J99jR8u3(H$Lu!gqgLF5RR=kZ1k`eMjegyDHuXrkH@+DNx9(lF!zt{e^@0tZ#u'
    'oUpP>+<UEBiEJ){?J7=}$(j!ZYJd`eVgx4hxN(9r$yOr;vVKCj^nzR1_XT)SM9n3zT?54xyct62G}>vjYJReQj^^j#`3$KAF;$sdDR6#C;xRZ+Sy3L$df{3C'
    'SMy+#3Q4D^S%+&L*th_x6!iJv;SyY0!Kr0b-fCU$;L9Z3TE?y-a&Lo~IWQl^jcv4&2FWcDNg<yL#g@^<8aYm)?Oo){gJVCcEt6vpIBLELuaUeU;_XmX#trEX'
    'yj+CU0NLM1xhyEVNb2C{sMa76%4|14eGR!?FhAG2Y=Y1_JX%3iNGYCzqm8%CHC*)LW){tywW=k2c0}?~>|BEJ6YQy?umf%s(0&$|V}Q!MlwJj+(Cx>C3aWYG'
    'sRuM0@O1St@XWxxmmEaVN)9gsz+MHfH;^lfBV|0l14`S1How!l*hjGqau|FQaT8DOjd%HNy+Gz%(2)Tvbv)w(4XRD0K{n8;dZDj`QZ=yXM8{6(-UYE)6wjlj'
    '6YM{Sr#?~(U@Calk8tRy)eNJc2ZX3dy^AiJAClxHK6Qh<3)3B%-)~j6@L~`pwjhP#MwslKfw>BaJ5gy4Z5N3%hw>LFG7IX_e|=le2<FgDnAjzAK6E)xmVz)y'
    '+2ks`xPTOpwF^|pgS9!7*aWk6uuuibBG{*5zz#mmlX4pEpJ5jz&%0p5L*@&x76!!%P8aaeY^xrCXOtMY@J8jy+xj+|-9do_E>r!WL^4zin}KI^%u67&3&K=;'
    'r$|=^;W`SHV4?)GyMMy&WsnMx>@F(MJ?P`KYUK=P9dIQ<s!bF-fIHbY392)lwd!fIw@h}ANb(H2mtmqz(jGDw1Qg~CC-BXKxiu6BleFV){hU+|U@Z%_VrYqK'
    'DNW!y0PA6}&_q=EOc&v{8_(~P9m*ghXs3ymXW?lG9Z(vX`yEK_foK%eBcNEq-ZXF@5@(Y*9Ast<FYc4$1i82bES`ru$>B;JHP_+l269x<+6qiX$nFWC@<aF-'
    'y0UM=r?BRI<5(dcx;#{MTK@mpd)KbUk}Fa0ugK-)g_K272~{pv=a{~?{V2Dm%Wk(_ZqJ&m99;n-Q%({v5}9Px)GR#^An_0gfdoQ;0Pz-xx87Ro_J>)sX05yK'
    '{DrjQ^(p_t?1=NwIY&o8mVNu)neOs(a&%6_j))yQ_A7S8X6NVZVvDYM>B<}_U!jvUz20U(X&ZhvIs2{*7Skb)9+1F6J%7Z4A(|?)%rz?e=xL1|Ezr;;+MT77'
    'el(vVsWkDgkqh{lM8OP-T+rwmDQ9R1Xk{HO?vc#{wB<#o^LR6lB(FGLOt8!`o~+SYhO91<(h^%Nv*0{kI6^CH_3Ad-nM7+Bc&C6blGMLL0=w@(EEgxyW%6w('
    'gRU#65G3(3N}Zxuh=o8zT%f56O3l^tfWT?8yuuDaJO$Cs8I3QY;u5a;(NYGT?b7WWDx}##j#L6P=_S+iG`)>e8)!Go7QrT8!ahIot)tlx4NswDR&dzOOPnpU'
    'i4&GCbl`B99Zloh0h--HAcvNdWWJ0RBP_9iVlg^(h8JhaW{!sENO6m-?IZ6ji_Ej7Lt0%y-Za@hCL2d&;hKg6Z{u^Qx=$AJIB`vP7EvI~Qc(;dFtkB{O$0y_'
    'OQX{wT|GdVNmAWI%LTMkVCVBFIgj&Ul1-51Q+%CbfjzRdN~XYm*ca#xY@)3VTsuY=C6@HDg9|hR#{EJa#N!I^<1C8opvls^@-7~qBB$dBnh9lSGWL#3)Qi__'
    'ZGzTTm^VVheli_L)hn_QCu=J#HjW}bnvUZ{3{5Q(SZLJ^5-Om|Au5bh|1_BgQ5eT(Ieff{Lg)B;3RO>VIzYfy1*<O^#l>rUlOQE8PG2MM3Z9xFWyy7Vc1Sjh'
    'D4W3tU{?oFX&Ei$==2<|B`Fvh%ZIqML`qc(YSsKPo1Db43<>R##39-9k+m#2xF&%qGIu~Cv-o10_>bu91S)1pa1AeQ)7lO8p0bN^l-aBo6F9wqfx|6VQLsWz'
    '_Hp6}Ws_`ti{6})jpT<kd_gv9czK-c-{9&Tz4DUc9*gec(hdSsdr=5oRJeQ#ihpI66nALAM-CTIb{hpEI8*))1h5{c(ZDhW^LHA|npx^Sz+f!Y#&OxtN@*NF'
    'qZ0{QsNva4J@2EHBfL|iHxp=T;%&@_cJgdDN{VSxPNS)7bbf^|a&#|*y}Rh-2yaEu)g(T-pfhn2DWZ54O-`e&bvjoerK5T|&bB65aEDDC(uHF-u`B3zJBTJ`'
    '==3a}I$^#DtrhXw8QZuZ+xu+Ai{|I+l{kv*;;Bov5<<CEy0c2BGh_w0LxrtH>b3PcOylYtxmaQO12jF2E0buxN^6Jg@P^FKyXjhtZl+Q2=s(}Exd<!l;91ab'
    'g7xw#@&!mKiLdvmf0-R7(cCyoPSDsTtyJ+w4P_76T9!_v@LY!FC7TfR>-{*r+QNwpDbKOXDYm;$eTUcw+;$fwtK=g8BXr?o)d<@Pq1jbbTw%$JdUX?Tuh#Q<'
    '<SWtBC_N3LXar5f+1fbv1(?r|&yuvZk50DO(mL9mqOl@fS!da4lq=v_;0JjMt9U*@XO7Wsm9C%S+#cDVBXdV&W{a(_qNQE7S3u`^mN>^HSol{coe_jP5g|uK'
    'TAgIOlQa(6c^U7XvFOR?=*W+9d9)TLdlfPsrBmxP7e-3~yct5iDKvA4uea-k5K4LRLV?B4Nh*V93M8=e79`4HoW(Eb!WHwD*i;x-cF^q60Nag`nK)ZKA{!^<'
    'e200rQ1Oawf!a|)Hx;xO#nBzQUSx?ScIYRW3>jZT)hSxt$Kf~$NnyZ+AUj;E*LG~|u>7t(K{huKu<fmVTB*>}1G+sy!W(4zm=(Yt$m26uE(i50D2*qq5J9tG'
    'on6#RrMF<)6=qp9jPgEQ+9hQ$-!f<lnAsBB*rVB9I=M|wfWr1kc^-o*o({5=d9(=PxmvG*+Ml7(3O-$82St)utyd#7x6OPf^(ttUTki^oB(RT4`;x~cf)jgh'
    '7Y^{wB|cB0vuWnvW!I-@YY|V))@!?{I>VOyD3U<g6ke;~!*P_XkV7AZ_QF{<IZxI$Q96in5d_*``k0jVSaFdhF6i_ny}2OaaU7|j$UfWJ8)5Mxo0=pWWp)@s'
    '`#xImzV%-Gghax0{1lJRVxaJHn9e3pZ5hvO(}l~o!A%?qkw}izq-~YN622+m>s_2$!xJ$E`fVVKc8>6Bl|(m*cL81Ga5lzbTWorQ0C$=W(6bOtAJeG`651iD'
    'Nu1dvYm>}h!GTGXTmG}xOR8XUhVa=M+Q_~ugN-?jiV-@OKr>4y<iphz4rS@-e!YB67HhbCN^SypCQYZ--W6grK937~Xst*Nx7kskUOcC{EDiwc5Z-HY8AhOZ'
    '#JA`PD4W3FGI(+Uc_VKZw#d{Z4xC^R@Hdxud5w8b@NyZ4mkH=l)p0z3j1LmH7(nxzcs+;139?&au^Bv-d{@3^(F0aEBI#wEI!DnX7A>KjJ)AhDr(t|OgNtC4'
    '@8X0X&m7S(7)!@AcR`mz=yC%2BBXNgHVSrc9A9pbg&d7+kiF2miiovG;VnA5ji$lADB^h^z1l|U1Rjr|ofUL6L%q?_dM!l~^JMWF7mjcykH=RD_q1%G>zjW$'
    'Kx@a$A4jQKc6|P>93@vL)HjXuMYI*Hm(mREs)QHqU$Vnlnp>kcc^V0^r8K(Sz|{+Sd5)4v8cx(JDFnuFZi7w~@nspGFOp=H0=ZmFlBq+qynsMYT@;~TD1U+q'
    '7ie*xrB+bcOMGBt9?+{f7A?}NMY4Q>!bfE4nob|RD_`M_aWWqy)8i<x%obK~Iz|f%cqa3%eEKQ#?c-(OQwumTgVsUeoG0U_=<Jj&&6A@H8js>~m989-(?e48'
    ';lm^9-A4;v6s^$OJWCdEIfTKsDXx>_Ll)h_7pKTuW-C+p?0_vT(CVV#V$*4QvO=cIWZ#RUpvkB4K^iab&@(VkS1>5dn>Xn40B<MgtQT$Up#2TJvx7<r>;>KD'
    'fYeUjm3PR|GTmE7(?B{olD#C6b&{S(`x7L%hjO58#89b5;vrHzp_OA4+o9D}l%J^=FVI4qj!)33AXyzpxfqVjqGXCC&q*YMO9kxRWJ_zba7|_oSyoUkzOJF!'
    'wRhzyoGsv`2=am@2(z+CLd#@s64f?w$xqTC1#)D69i{e(-;3sgxOju6lGq3K#3Vku#9-po)@UGrHX<mvhXX}ayQ01rii3=bqR9~6Kg8ula<WthWhh=@>tL2f'
    '*?Ja%{Wf<&z4NGegKtXs6lC-T^KDcAIXyX{mpQf@r`J`M%945E@1nkIvb>7JX9P@x`4HN^rc>#9^_W2KXHs~2m6Q+g^fHDio;#$cArd~u%SWV=uIJ%v9QhBZ'
    'cL`T7X(Gr#N?ctMu#n4>bSF(t;`lU%XFyPeNHR>r6`HJK;5)|`csGeplHcOXIGM>1kfDJTJv~PAQ)GPxoy?&80-B$|OY85-+x5yKiLT+a(8S0|3@1ProkW2P'
    'T)G@Wn>%!RjRaz7sY+vuq<DyPC#)Jq=f~volr9v}oF89i$o2|~#@Ny#1^vOlfv5M#(hklg>D3HA+oz|?$P30mku4mNQl4zzpp7CaW$4)|I=Er0l`qKnGFsin'
    '>)Co?qh7s2)eE*2q*qt?=HOj9j<0U;E-=?JTU=%98MY*J1r$0Z+hH;jW`Qi81mk(1&K1~U4#yMq(geLqlV}2^(hTgX(`mGDMJ|@e)*g-?<Etq=Qz1JglA5Pm'
    '<7hHP7I)DcFsBW?pP<uwED%C_i)1!Rb7{N*{N0N$_R(>O`X@<X4TJ0e^)SANecQNrNGkFFbB>f2sP_it_tEYFEAFtxD|9|jm)Gz*Xk;SVg0Ako^{$cZG~V%{'
    '_%aKgk@16n#*uBD2+&Fqt#30x$oviitL*CHZG3{n4p{J#USFWe2^0aP_OxDEqoIp;<sv#;N4fEOF^iH{Gy)o378TCXdY-IK(p-RMkJ0QEEvN8Kv|df2d<ubq'
    'Q}v;#DFUkc&H>qvk<bcG&Y+zd$(Pvi2Hlvz$rThX|4(9lc_&bOmnFcSzh;FTss-wmYnC};u>$dDF-WUag)A+wK%7>}=q!S#4_IjhO<d!6gk(3__6GL)S#lCh'
    'fMK}I#{FnFj`Jy2Il&icvR7ikLl!(`#R{GB(%=HiZ+{=BYAhB)kt8|XC0mPRvy3x0WIB)Ml5}C4O@vVRiq4j3IQ*`hM3FNR5?q$}PY4*or6McO(TN4R<EMLm'
    'G`EDd!2&y=XEXl-THPKxEzsB%nYkegadH^1=M$uOfiv6qB9CHc%<pA^Q*^mZZ}#!}20hKAlPQ{<z~>hvc0<!ub^;Rp49^r<eiL1svh7u}y39bqI;!C+@Sr^s'
    '@#1g>XJKh<;g}cu*YV~Q0b4SeM6)MkepQNP*vHdJ8V#|<B05cwT88EiXdVc50>|?-ea-^AZ{e$d8)ah@m@NpJt4&tCtk;g&S{1J@kaP-Xj_^$ikMF%JgSzNP'
    'wL?-~BAKareir3?^q|1}TPSf>FD%ntAIa?@KybW7Gc}x;dj|wD3zq9C+CKiXcaAQc;p_~H`EemhbCakH>g^^??@N8%dmB8jmq5h|puj3#xx~k5w3?$g+XR%h'
    'B@k6UwmXlfYGfXiYA~iw5RbPl(5qE6dx6fgICz6HTP!hEuLQ_cj*Nr3c!T3UdU8pFA(Y9Z`6#ZG$>{{XnWl+7wzo!tJ7oS4pO;DTh8$1fsZ|74*UAjJ2-gd9'
    'bbE=+U!fvs;8%Eki6vIaaZ(WL(iF~bqrDm(U%=%{avVURH4;xy|0VX$;Zr}Y?BjeI#dpx+#M@{X`&aS&COQKSc2F-yS#lOlUXXH@WUtWrE-9_xwJCI(p;NQJ'
    'B*8g41=F#L=6&d75ra-1NwDn|Qi_sl1(l05Jd2O_5OBKq3_4w>$upEaLz7;XTw=ik7ER-+c{;vKvM209oEO5UL3$n}rzy0u^IyQy1<U#xC#v`)j8`^rc>*QQ'
    '5tuAs>#vot_Yh@sXflBPo20f%=EJnQO=@AZw!sbqELfzq1w1>CQc09cvBeNtJfS<+XlH|NgIrl9)j6~b>fr)D&*R<8dVZN~`S9ft_8rlg2rGg4TVjb4y4c0D'
    'Gi2tBMnFy8L6I1XCeh`d<gQyFkxjI}hz>IJa*`ZQp<09_ymWI3pZe;RDSGC^S9z38AkZ8aCQvfW!bwuxtrz$3c?n&vl3gIIJ$hN7xf=@h!^R~F@1vU<t?km2'
    'EZZ6<$rHK(Txu3APLc}DSP+Hw>eT?fo@PFf6_<1`L-*!MCeAKq(Z&XeU$V>+*||h(huD9PHv-rjMeEY`La2iByLj!87A6t5l*S2YC8Zk_onhlEs5nW13M)$l'
    '#O3}hInJPy9b62uXbx}Z*wO?FE~D8fzF0)_b2y)6fjl|}rFpJiinCITl-B4vOw=Wdl~5@}R~PZ6@6X;vdN4(=$MNnsUZ2DVMFi_1QYI099V{g1%;h>;Pq4`u'
    '1idU~NM)SvfpIPN6-hOQ%4JmWljt7FR_RudErEU!!z<S)9mAW;Xn&J!Ezqfbv=>Hu$27KqulMS;B8sfw)FGV>(A6}}=iZes=|+L&H|ph;cljKexWX4zmRNpQ'
    'US<bXJaxhR(>Q&=GI^3NlkzHETxJ(39Nt9n6j_}nr4#BMCx@FXb%{!gBn<p(p9D|X_}YJ2Ayemc>rzl|Zn9q7!NDyI6PP}tQ{xQCDLe6JpAY3~^g53->-B0J'
    'r$Mt%vc(`voYBDHyYdFwOw;NtJ)dFQ3B2fM5io0fEPsMmCp~mNfUgd5=$K`LGy;P>!61kyi#WH)g6p(cq_Z1%cY?;^-Cuo0=XdE^35~DdaGB0rlZ7n3Ss=+v'
    'l$xQZev}7e;sUQrw!+*Ns_fFeHRM~x-UzzL;t7~LAKqMG6Ujd2OJLst*$UCXDNSYm%vHHn5?jHmC-^E%qVwc*fh<qI4Q?^tJju<V`7m0&sTXT3dW7Sfcq>4B'
    'Wd=%lZGpu>v#QpsF~%o&vtA0~y*YYvj<z<@R31+q)=S59u|_gpa+1Z<<7|DCZ2HOM9$O0In@LiRvy||OvLq}JAju_q8h4<bB8t{nbpy|ZSn2fbLK;tkoxMm;'
    'Q#3b@PY!WDk7_%3We4XoxI9${?PFq}`a#?*lDz<(-X+yjJhOm<ODLJctIOngoG#9yjE@GV@p*+L=U4%3!bR-Mzboeu=-|~X*#o^kfGcY_xr_ZEw)ft86DYYu'
    'PcKO#!~B<IW1C(cldumJW&{Dx&7s%@%3iVES(G|L`8~2<!0|Ztk7F==55V3?5uo0~D>PlD{;hf~jW6>klc8&?_3}Jj1>@ozhmTMg7+sOXr)VxlV!%|Z_1Xo3'
    'm6j<Z(6hkAKi{L#Q=G`slN}n}M=NLSV419YnSY5KA7dXFXxAtO{5_6NLUeu>U&qM$CX3CZPyk&`qsbbINXl(#>jlki;>|e8WN~f+hYIB67-zv`+5dmW@kI&8'
    '5_IZ<Mq@a;OXrGY{sP?u>%|P-si4bCvR15@0;B}8VjidWQ1KcCmhkp5nFmQ1f4fjaM<E&mt;Sc+uVZhDL{7-^CC-)*=-e|Ax^~HeQF^obXWt5*o}t%8w6TPu'
    'tL$P0%`DT(Ddo!k5{ay!%mUe5rg@1A&rRUUIy;PzA{aw4wC}Cg#z|(L#BvM>;3|ffSD3eo{WHjSiB^u$p6_jRf}ZZO*g8A7BJ;Z>0T5v-j}YiE*%00?q98E9'
    '6PypCg((uO;8cdJP18^gZx+bxJWI^uxp`8}ldS~X+(vUX%$<kJC^E%XbIcpVi$SumM8PyXE4-^DNfuPlJPt3S(;%7?ele5@v7Hli?ql0?v^r5Qh0*RJoe7bn'
    '!r#-I8@jPZPHzZEvS^je&(Twu^<z3&X0chinnRZXvbTW~moyheOA}-Q)>4TT7SKtAt%Vq9Hl+xKHL;$d!6G@@#@CBzw#Y&!=mt0%Foa2TS*2%NxRl0MNj$lR'
    'K<8hHSn<*fnOw!k=kzL!rhIe-nmj}Y*EA~VnXc7v{1`1QGJl$`hVfjkUiOh%4i`brL=c!EfhDr!Mbil!4&llc+DNg38anWzn^SzS!iuGOIfOPU*jKLS3-ol8'
    '`B%`w4$EAz!+n$i$x&gkg<sIQD2uM543PLXTG?lb7~7hnAPgd5>|I9dSrVIlSICfKFi2-mc88?rXe3AGPEqiZBsWkD7>bWxUe)trPmLX%u%i+?Org_#yaK8v'
    'ki;q5NaF$+KVWmsvg#DwSfJAvcr{Kc!FT0-R;iKN1*xv1vpEVnYWCp2Zem}705_hGkk~$sU8BVryaw7(h{Vpx@g|v`BB^~ew}H1qC;@iUF-jCj1}u_QGPg^X'
    'x6xUMMQb>9fr4k~ELg7{AnzL8TR_EKHV*dnMZE&@DNj<uBh8kMadDp1N@O}k18L;DMiD>rSJ6&{q_cGPkZi_KaRM#eAmDY`E#~!;%`7SJ<Eu4Py}>oG6e?`)'
    '0>!RLW{;*q6i^vHuU9V6^*Px}vdJ($O|yiLq`ahjfkSinW`fkJsGLC2Lv}MsV`=7{rppnu1+p}VLwjf?hBGldw}+>1=;AJ2y2MivG9RJ-DjPo&<WbuqV6JcF'
    'NOgx+&q!jIPA|~VG@007)oE0%;7khb!F&c#^^l%y(YTKVBJ^s99D+8O!z(#fn8k%E%1_j*yEJgk;+JT48SQP;aFpzB;e$i$E1~5W&4B$hOHZc|s7t4dbka+f'
    'kEp+b6CpIeK-R!Gf%RErz+eu*il4!yX><@G)pL9fcEJ(4@zd%CJ;_Nt`UGu)`8dtaYHY1Qj@MCQgT{{Wag_#Pr7V!S^L~<EVjH{cVumcN(C{ItF5~k+y#%@w'
    '(A*6^g+a{INSalBES1Em2%epNS4rXYC5{JZ?iw%U>a{9~@6g3ey%a;yDBC!q)hhD>A6)sfH_SG`8ji8WIp(`!v2!%Bg3qfscS$z$B;Z5YHSEvQxqY-chvrgJ'
    'DAN@K0=!Ps!Z{13Xmts1lvpf>mdZFZNtPE#3aE9K1*a)kob#9O$_u2lOh7X`1&*7db7gcg`K~;JLp3@-PNwHbZI@ikpy)K(TzFSr!Tu<VAJS_dIgg^#TD>-p'
    'V^y?1kFph(oMDS+B(;s=Yk0XpCrZqJMj~L2ETLK!UrnG|4X^LBWQ5EGQ0frR-Jrk~OGq|;Zibze@%lAdPvQI~_4)DTI7+S~?>?QGpl4a^O_RwbJRQXTBNhN5'
    '96<Xab^!|g9$p_uy9ESFYw?V3`B*81L5o=i<uQw6HJaOH-c_=EMH0z>0ShcfXD<H*ByEg=T{M%VMIW17AS<8%PvLbRj$G2KS$q*9Yd*Gq^fnqqMUcRYI1t1)'
    'n=~*(3*&SuhJs}los%|@Z}81JS&zOA?cnka3*L}Shyp9Q2#^ez#9MC{_L(<{{VB9@jr_A@w?Nm<XmAN9Ch*)Q_FvLDSgtGd`idq%7fIGj8}%YEopn6t{Su`P'
    '=;=m14^w!FBWb*KLDsI(d;#Aay$x=onG#-@q^nb;cty^usJM(WQ-2`I1PKQ5)E3<vM;XwH_RwksSMqo*g`<;F6z(qFm`3p#k_C$x`1TpPzDD5`DaP1p0$-i5'
    '{0-Tt5HKh~L*B1qUk<0{(5as+71%_PBz*s}NpfHrm(gLAEuN6g3|Xi$7~fop=2qzT8J^xpyCJ+8p(j<g>&1aRwt7Kw5gG==?id#~(B3?XAF}O5I#s2hkNHk<'
    'd>mg!QFxO^Hc2dtQsK8#CwOb(&mc7Yd6L-_i}K)-lmiqj(!+DKbp5UzL2H3Odza9}EIRYzYyged>V<g<1d^}f<QhBP#?vu&a7=1I8c_nm_+%2F7t!n`n!m&|'
    'J9N5$qsu5)M4MMAm?vjGQp~-LM&C|E|LX?YKBAE<nOVYXF_Z}5<N;n^XMRw`c35<uPJ@NEfW1kybb^*EWOa^hETW?rx;a4CD=41C(`B~5i6;b|Jjcl@31`{L'
    'Gz-qs@iQFQVCgFqOQQX2vR%Y8%cML`XR|06d<!enAH%Er_#{d<3wUY)Ut~xTl$=wVj@3&k5-#ACIL@5lWv~#p>-h>=&DKlf_0l5UPO%KoasXGy>&0twl)~Gf'
    'DV_eD9893K5S~AzD^;{q!o|HY<{f9%B$^J;*bUxT#rYWo>OgD(?M;)VGWCjBzIvrh*N-qL9;b6SG(#pXNFqUC-1%j8xP?F?Upu9hD^fimnMqQb$Dss0E22uA'
    '`X}(r{@bZ3dUZoVkb3i^aze^M5;&&Ir*w9m6nu2;hQ?M2=<FapbBF(T810;}#3cfIqY$IHGvvELixqMNYGaHY`p{w-?}k`x4Hd^JXxQ;1TwOs&QMNll5>+}0'
    'GAxCqxDJ}}k-!Sg9kcQxKA)$4;AJZ;4ysm~1lQ5|8Oztu$`MMfqTLMLIYN~)oLZ;3S?r(18z9iY2#CBZhgf2ko@D6hI=-62=lispB)b(7o@9sXBnJjT2!+>a'
    'VTG<uk?ak<^3khH42t2*2|n9lhdX#~9|!k-fae$SW`!)nfYz{YmID6Q!Zff>r}8LrfRA_ZbQR4AYlf89>ZLgpU#ORN$;lB79?<FKdUXM>C(ywaJ}%<aKHe?i'
    '&4YJkKzR~RfEbO`t1xAGwse7}64bwnz!0e9NT!IE=J9d`FKsbE)Wj*?y2hdPcV!<2^q(j2$ujfau*GBMFQPKo3k4Rv!09xtRqE9YDQ*6Fe3_I2Y<vA}VuA%%'
    'Ny<lO{HU^nFRxH0f_YrnK1%Gcv=mM0_0d9@fjkJ!&`=R4B4}olEN-HmEDju_e3q3qX}pGJZ|KSCTW=0cZ{X=;a<oAGMZ9_V4f8Iuqf=IzMGIRL6zpo5o{qC9'
    'Fo$Vcovv3v{+IC$*vD&V76jTJDP53AmW{`$H$eSqvap4(rt#U~TkjGsO`=qZ?FQLWoE3a1c!I7HG%$-maSEN|SQK3y;Drf%aUu!LZL|bNaQ0nc87~~L!#uhH'
    '+kc8|Y~giKvn~)&G%)${7F{ZkKmZp`h`;bFJORqUHrh+!OaT|eX#I$;`N{GGT?CSwB*{2konc$6G%`uIgDif4YbV&ZRj-_)i*3BOi4ImtB3`ea)=MjNdIO!E'
    'k$4c@Wa`yD65B=L103F=xj3o?QSuBY!enlfMGL61M-q9uw=QY0C33uj5;eLKBWp=IcZ60BSinc-vJ}S4gFXG|JV7R|@YNy$<`>>V<x`x^khvt;$UemB3`<v0'
    'u1qEaXljS~uj-YFw+q`ORwbbs6yL`eThw1;2W32a!@MAvj>!TvbAa+mQl6lzd-!w~FYMv7IRXq3RIl14o{b<dfzCtJ8>Gn<wz$M#iEqxj=xmu(rL(<@ng5xc'
    '=TOLt!$~^3Mtu{^w?IHjo-E?x7K=@=G#LJ=w+mHTO`wHE8s25A1(d1Kz#<M+>xDCxxqKU&$Jgg9y7HK=XYsj@PL^0Ah^AwtoWQ$#Wby=0PvTS+2j}Y55UytM'
    '>>OTMX2ES57$@go`kvxhe?5PQ(la!-N6ydLIf&Xx1V&-Zi|0Y|&;M^|c@6CcX;_3aq2Ox0SVS|k=w_A%^7VWcPetpsEX$`!s!CRUXm=dVOrSK_oI$j+j8bv5'
    'zfF!;P;HhiZIg>Q1LJ)yO}wjU!;jB*NhnGKUIZ5RQ36G>XflmzX%@f2vnT(N)|O}u^q*DgouMfZR~uwEh=VJnSgcoO5iG6>=xVTHgKTS&B~DPhLiQF|>;{F8'
    '|1aPkTll;{#!J$*HCZ&fj>^;c7#M$sPVUjY3I;>3;IEfLC?BEuE4&2u_$&?`(R>(lOKhGUrAaJ-6IZCXfWkGF&#=WEnwn#=6PiyUe~FBT(P9XDy?8xDK_?BL'
    'l0XVw9aHZFt<EtJ$!i64HA9O2dSMfvrO8Yc18Sn%$Tx`!>o{D)lTo}Aq_r5@F0jH3+KJ(tG_Lw-LBjWC7F}e}_7Yo}Ac0eS=0nRqR6Am&ag<&~Q7^3>QLvYi'
    'F}4B%dXH_C>cs*&NwL+nx4|??F4s%vblk_1RTkeM#|tRoMIblMGI(j3Ozz^+0Vzl6)*0HJK)|2VM|8Ye&%^4Cp{Y1qon?zP>W$N#Fuw7=D^J&JTlnG#&%;pn'
    'N&1+~t>E+`g}PIK_Z@mEkWG#wII@N}E?D6LAA=>Gpi?;(TfsLo__9z3@}Ahj(G-r)kyMn#m)Q7ry?l-r*YR1F6+-mfj}kz?`!u?OujlC55%Gs<Erk{jS)f91'
    'mQiT_ZD<okw&>*+0SRBu5nqH&hv*)tO4Dd1OLofmd`i0a&-Ajuff2jA_qmf_ramIw7wCF9JT&6rH&hP4u{hk8!C~!f>|co64*%d7dHMU{pANZQgU;V6)#b|)'
    'mByq5G9Kq+e%-X?wafF|;;{5M+)n#|xOum;<56=3lRN2OLtFOIVY}Po9Co)`pu;a+gVJBenE31VI7U3u<=1`AUdJ1y=qu+)55Fz@jdbJUE5|_hBi&`^BIU@4'
    'UDtx8Ti;@HanT&V0aAW!E&&~szQcpbSI3zA)iGxJRXYm#26D}trhHm%565V?<xz*>Q}g(tRP#&iy}$}B$haRgPYIx8KQU>b`Bqwfd#U9It$PF=e{2~Zffv^P'
    'uRiD<9T^d+&|s*KE#C}&%Wq@t<!=T?9=;qM@VJHtTtI7<Z?vQT_NxzU`b%;xKQ^CUV+J#hIIDJM9_r`WX8EWEojia>KGYXB&A6mY&%lW3MW5Q&CMljcp;-qt'
    'z0!4p_s-gia$41CZUD&C3JVBUq|F39TB`mKsO*P4jhs2kpomDYXMXK`CsZFC?&uwITRj|o!yUse<MpTaN1B#l^g0FxREE)lXtl|@RWcuJH)Nf-lgo9PusQeW'
    'Fz1<W`R-+BpT*^Iirk3&*5%%z!G71X(Gf?_fYb8I<q%iLy6?pxJH88t?3Wm1zr-K=753Qsrgy-}ugZPqd43P(s7VuPAh~WS6<@zJ+l^n2G0iV|@yNe*ZQK4b'
    '4&1LY;C@s7+b^@<eiQE7FEihM6W*)qBTs{>{cS8e`qD1Gi>up*`EU2=OY4(IPl3e6&A`ndZ8kH`-Tzl&-2JY>KKP)1Q`UlxZ8v~D(&zu`9Z>TkHiq`3^YK$N'
    'UjAdijd?0NvA`O=Yf7xc+y1cI()s=pNAyVHMNMPXW_uFZh!vKk8I1Vz^nX|x)(?f0CH@`$)3hp~>KpOr{&G>qk73;Xs&o`9MlPgaS6+Ye+m@12sihH^`kJjI'
    '0B^nDl>hht1u|39U6WO67-Fl+?;%6`RkGy_NrA5yq~_Rd6#uIU)mu^eSCXifTSTgqC`A>f*4qRvZ|dLq#J?h+__xO?TCL6ZSe3W2kipxT%y(U-CT2YUS7*LU'
    'e+FDVyn_5rWNc7>=?r|4KB#wSV8Gccv{t#k#nIEN0U*)j$WX7-?G~9n+?JP0WzX{wr$glXkPCk)UdUhnH0m66DqsGI|Byep26?_6rN0)JyMJ_0bV2_9n_qsR'
    'eEa1~$1`WU^yg<Ij@R<z=R<=c6M`r%@X`Tz=@5A7P;()u1Cn1ztcf$0fuU!1&zQ&VaNAukpV>JWyHd*f`gyx$te0on{My<5LsdamzQfY)bB+niOUf3;m0jhd'
    'JiXtyyyl9%NJh`qdaImY{kh1`qZS(LwMdo$FVpK7mZrfoWEmb9l2zOuxq-1>xfXO=O+}%uU$t)nSR?VQzRx@{NpFW|NU5u4f^7#A#MArS?$W1u)z{Xs+~<xF'
    'V_TZMvQD1d6N>1;$!*x0`RBvVA4}zb+ycucny1<YC2gis^9ld*^nQtyn}D?aA8p#~cUrV6MbtWSL72pDeMx0jI5v6&PDdPr&j111p~E~vJweSKgU;91uRi$g'
    'FMjgX2VmJf=jP+U&`9@JAH04JMx{=l>w`t0)Z+`Hga2a%U^eY6463mCO6l;7j5>K5PitGpM@BA!mkT(%T`$$p$@Ift{Nx`xzWnm@HktVOKU(qGE2WXKe)5QO'
    '!Pn2Z?emeP>xZ9em5%`s=x!<T2q5Bvjz12$IQc&5aXr<^8_&oa!;J8;b?`*5R_h}v(T{)y_N;4Q0I<<B;OKq9QN#z>&HvFf>l8`CEx#Xh@tj{$mcgHoj0}w!'
    '8j&RNS0DV;0Z5V(gn=*~;Y>_qjURgU44ykIzkA^t9_CL5F)7U(s9cVb5yu;WdBCp&SlW==Y3+i3gF?t<KxfyFK-+uR@Odh$S0SXvq{(X<fP&!beZjxD^`E4~'
    'J09ntduT-aSr!reO@q6++&o8(x?ts0LYf?ouskJzUkZRG`kf~2wfsgTY?c#f2^;?tDR;W^?n&oUP>w#fX}jU#6!_nxBk=ZPo?%k|s#DWBIWdviHzk)U^iFhA'
    'Nx?0ZR<h-RLf3BT(~L(31Mf}xt{TMh?>;GScSGJ;`{blZeT}AGIvs=d{sD)_>e1mu>koSF@eI2kfArBam*@Ft57+@OKN<p(a=1QvHtOng_C4%#_4kjup@oln'
    '28Mb*>iSV{*CWSg{f?f$>FxRCM_qmWKkWOor?2nR-p~3vJ9|It|K#IGJ%7{niCo>@s6IFd8pij(`0P`u;V&I8oFMg`qL-GTS3Kv}zyKg@@S))DJRubHnu{?L'
    'H<m)IVq6^@1kaw>EyR(;Av@cJu#-zjQ*5(ormc5qbkJiT<|zW*R>8^?ua;9v9;cwoGvsj$$OXB=H~P{#%=5fC2VfHTLLcUQSyCN_m5hUKtHQ`&nq;t#wTaqg'
    'CH@<!9#~A=%(2_yX%HaT?cefzftGJ{jXwYm;C%S$<BqQWZ!N#FNN+n-rmi4GQ$@3c#6t1#q*0QRBKZ}q(jUXR`sxGFnxBE1^8p{4R7CH<m)?nF&UU-YZSQxD'
    '@Z41%SfzHmF*CN@F@3Ijkr%oF=Bp0`93bbR!9F=fsonn72loLr;&1`w|AX_5;L^a6e?G>wLT8`lh4YQ&KS^WxPh1<)V4(?bP{uyg^G69gz{h3OJZT`~sm=P}'
    'f%LWmUO+=KmOtS6%X&c*(h^t`JV`c)wCY2SK0T4W6bXqzTn`8u6D2!F++e}+W;%EaoehGLAZSn{RR;}pn9Ef~dGC;(?9x}O^hfK_Q42}{tqm|us1d0vZ-?9t'
    '6xsU%C_v;`w28rJ@L}&85>yFR!hpTc;c?i9N1XjmSnF2K#o8_Z1XzUE?G{(xnE1!;8M5~e>JjyQOYaDGl3V&lU%q@JD)8LViX2en>-Q}qqk|Tu0YIjTI~A^u'
    '%W|yqURb3bydD~P;T*BM`nsXGj0=O}XQ(3=`lpWJQIGYB7y*=cs(by_2S4L<`+){2EfAv7z@UN07Vb~s8vqK12JPIA0+A;uQ>x6>GEi*>`-D2dDW-RLRCo`7'
    'jlOcqc?fy(-p1%;nb<%sAjk+)N!k5G2m7lJgns(LQ?+ViCH^x{swl5T%}0re|4c&ga~JS<7f&$jcEdvDmWnAV1RI2}Jx+@gWR#S4Rdh?Tc1A36r#ckb@O>o<'
    'x-#JnB|%0Wuy3i3B<`CAEZC7WyvIf^qaB+l1A<#-Ha(*vU96DDmS=#pM$c3dGV2dm$sNDsJ3c*)l;o}Y&e@F3Tn}wBN&4IV@OQuc^)G+@i^pGmz-t3HhmnHd'
    '0(JFHw8PXmL~D;NzZcf}(BQxu3r|@rQf9gZ=K5Uj5$7|<h%nrN*Sb5}nsU+W)N*J-u{im2hXl)N)4Q3?CqyN_$EHBJ9_^MiY0Y|UVX+xI(qf5=8UGS`pe1IF'
    'Gwe&yU!-~QJa>33pk;sx1C%N|IRL>5cALo<w4Mebayo_=3Kg^qTH;gC>luamZ$J^?MwjU7w;fs!o@!BWX{ufCnsNLScQ6lk2*-)F^U))%kC}H>`urQI)25i<'
    '5N@lkcx$Bf=N@*v9#oOwR{MAVc~$6?v)Ctkv_DaW#jx0zbQyS>!OJImC3soA4?(fJ?2?ex@>MN4J~hP$_cwBvWz#N|Fe$_|f!!uSvwWzEZVin$%Kbno_dVUl'
    'n$l_us5Tu5{43(Ena@1oJ$vGMY;ke7D`4iSRJ-A>c+}n)Z@VaBd)ksw$X`_Q(Kv<-U}a2b^=c^0-&rM6>u#-cv~yg!yH$ieaXmGVwbb~B(E9MxhFT%*9vf&|'
    'Rp);ve^u3vj*bTRSNy18*eLLny=eZfga4yA)1Emvldwxl#*-%!M(s^Qm+_)^fT%W!+DoIr=n(Dd^TWT;@9Tnq9&p0TAItRi*qFN+(91O(uw{)JI`8s72?e>C'
    '?8R5U>LpFc#F6$w-s8Q8&#i%^C;b19o<0=`tleOwD<hMAIX@SFxLgJ{+-!((i$N~%*uu3Nr|rGa$rJLr2KnmHp;VF|vZ2xs1%Q*k-#iJAm(9d({7r+zz=(P|'
    'JOD;#hvmsP8bacG4t$!wwYZ(V9ZxmRrxvWZP3FXHdL6{zXwe}3(Nmo=Wp1j8-NnIPK#oAr6OEi6k?d#T+--mbppG$@yZez2m){2w*~hiNQ4grn?j{e3p_v9a'
    '4K!5+GaWrw2DC$$@@X4=(uU3&#td~joMRsDlN+&_CtWr*B_u0Lg{s-4`PI;``2TggGy0tv$?Y{LNPBRw<z5E=SKTKucLr!TT%k?70AU>*?D&;rIX-R`Lu9g{'
    '$m{=p5Lhs`1zQ#Pl6yM=s~}ohmUDyc1HA-IXlSJSQAg)zU7bHPE5~!9i>^D}-JkyG<4?`Xz%aQ}*e>$XcYUUd<{TY%j#zCS>cEWgHX8U`+id(Y1_=pIh)A>U'
    '<S}S~zMm^O*R7gr?b_ZL8{?Q09&&T1je=ZbH4Q%RsI4h!1foSa_eE%a>OzKwMqY#FFK?*#+CioG+CCu9BRtA8Z;}3LQT;Wbcvvj1MiO`{uPdtAq+IY?l}V}a'
    'pu4$;nw=!OjP}L`zGfz@@L2=$mmC>tNsdda+)BF;#$rRc+oMiWw+dYY<fl3y8kEY7HLY-|)pogJ<FjsC_RF9BO4?d+f24X(?PBkAaOkDWG0@Bkc<{34wyjww'
    'R2^0~F+o)y5n4nO3+YhISH}Ra|G8RV_>CxHAK;sggYxF>@W4HHVI}ZGAYWg0Xd}jz3UYAAZ~t=MB|<%5jY(qPE<DM6Yk;pipei?zl}w0jZL+EQyU~|!;d*X3'
    'oE5EsyQc*abtrTBcc3$BD0%F$McQs?w+IijMc$d@yIt)T(EYSqUUC;6bA?{?s7G>UsP1mJ#VYR(ba$%M^IJhVx<>6V!rL0Q8AhmVNOXW?;f1Xal+BFCN~b{P'
    'KJ2u0aNqVr?cN68-R|Tu3wDRqVc6i1W)(_UJ3-ZOfZQ7fr2~u|=fh7P8DKd2`t1BWm{{u8<x7yuFGpXhYyf^s-*v4hr9RZOCp`&x(6U^@rWWBJdNd!t^Ii?C'
    '`w?I=yjL(|fOoNrAXn{aw>+@(&P%wIUP?NHqeBwqfJ>;U9}m|MlzHGFtk%wU5eXvpqU0v{UZ2jF{I9rc#HWj^QO|}(T)=0o{P&m=1<Zedw%TLBpSF-p2M-!$'
    '1z9OPy|Ev`H|0G;c3upy{(Q)zEL2%Qs?icsDDjve5L<`K<9rG5e{9jo(*8DSht~3qzTCH#zwNesqbd5WyeP)xMq#=>?|?_UNe5)(4wU;*U^bHe2}b`Tu~;~{'
    '@t_R-eiN)LSb|n*tI5!g^4K9mwCW(s<pn0yHNmM}t=2y<1d`I$0o{8!Y=uSH4NL9gM>aiy2>?x4{&x(k7*^K2Tvki>(i@I1fjEh*4YV%ab$|iEk<h8zP~-uK'
    '_8un~6>7Mls!IT)7O~oo9(`uE4=Ytx!xWWy;ie&v2I%8fhq718H)uzkuUyX89iX>-1S-eq%m3i&>wfs_pZ_mE`-c~wJ^Nox&KSs{@oIlKqn7a}IZp?;6nFFr'
    'XN}Th&4CXQmg2Vef^IP~)F;QS54o*Y>lly6lVbZnv`A-4I;AH$o)rE*>i8l2d|>Hn0nG5?Bf|@@G_@5ui5LoWK7QI<>+!?Rr~1GqCE$L*H7r8_^N`E8Zay=e'
    ';)i3byR*aA@c8^5_@nzzBO<cO^W4SpG9*+30f0};&?p%4z_&-Z7*W62MI|}9&8a~9g=5t1b~%6<E4m$r@b^O&Cyy@cXznYABKK8Kz|uR7jBvTR@3_wnLd!KM'
    '>*ZXN5<5S7qWi~J&XLz6uz+P%OH`2#k?<ilwlO?IAVyyR0!Bxi7E$UWCyzzt)nr>>$SKLr-sgOoS{#FKUJi}$?bW93dfp|SjoDCYf2Lr=Iy}(vg`#DC2fI_~'
    '@OHlM%T0Pmk8?m;%Wb#WgK}{ZWMW!SioAp}U7y@xdqSC3)@Nhwbz8U_-Ly=@h~SDDl>E`BpPBY($nBPW7nZhHJs^9BU;^A8SFh;+<iLax*Vp1)5wA4%$h2gi'
    'b8yJz2HkqluI$cRTG{5@ijg<{?<}io{r4%Vvp{8|U#s94?RGOtY8UioZqdsU&xjRR16F|-OIx371WcMXofRjA*eawTJs7@f$xnSY9{<r%o#xm37~;B;%~WG}'
    '(=-b1`fu1_-OaP3Q;3bx?7G^IP2!MECfoSR5HPNxXNjP}Bx^I>=xqDx&`UrpH$pkD<rOWXZsljs&?uPXFqa&Fy|LzuJCo9XZI&-NyK}k4F0R-&0|+p^E*@Mo'
    '0w&oIkD~&H_C{=Y_KKzjv2>WwMq};3fK@{L!w3)48?iR6EL$^SD9cH(<y(9>O)JmV!TGZFE|4;(?>0yagHxI>i^h>IBM5DflW~UlC$Mk+Gz!bdZGSf6;L=@='
    'GWd=TH7%!8pG{zC*{k|&DA&ve4R5r)RHg9knw`zGjxNkndfNf)S%l6CMgG%I+H|q5+)5S-g`|yv;;$3>N*P(V(Uz5u52_G!9nx6vat*CbudUNu!`POwrtZ5Q'
    '4f63%jp?s8)o$#Qt;5j+q$JEljrGA}-F|o{NL?S=TXnuw6J{+~%}@#!dGm;G?YDO8;-^D{ubjfqW)XA$ahKpGBhF`Fy#xR48SU?f<zzg_seHFf->q$Gf7;p`'
    '1f$r6w)@o9puMeWBHjy<4iz#i-MjD5FEf}XU+xY03y&SxJ1oUFZAy6X*Uk}s)=gc!_kkfoMQ?-9&`&jfTKfy-7_H(7>1;f9X*uT$&grdcDb<70cqNmcR8#Lz'
    'G5CC5=+5>2iSKv!al2M!kD5M>ai_g5oJb`qJ2$5l|Ch{$)%EL9tIL74E!mAZ7)=3FVA`dlfO%l~`I9k^I9(;&jnZqIr4!7)f6%wjvE-K7mBK0v7ZXSg^3&Fh'
    '0k=1o(U0Vz!kCTX%VrXeWAGS=J~hvbZZc#aThoN-Xx|*81}^B)fxKmsj2ynjRNW8!cbf$%1mZ)pk+z%>eYgLr^TKDv)$5VE)wEXB{UDppA@7mLY3e`H?K{cU'
    'l|7>--AXz}VpUo(t0+HpxE*}k-)ehs+n!icE$%k5svyKJz$4-PZ#Ok1ba<1z+XhY7YQ&6*9>AU6!fW?P9J3H>UHaL&Xs0M@oYL0iZBM!$Kh;6Eaj%e(D8=^R'
    '=YqD~L;W7^#2ez9el6IkQpvD%hq#Dvfo5m2e$6@S_bq?2j(Hw-TK;#-7~f3bm=LeT3yyyN>Vc-zBkoF5N@y{~j|4hLoXxd$0Bs$*vqpT~jJoZeTs7s}3tDi$'
    '1p@~RpF!s{(1GDWipp%aympBb86F<BILe)nj=?vt9dCq<Ai`My9h_W8TC^j?rRb}*@&3pBSd&JQ+6P=OG~aFQ4HyQN_49wW|7py_*Z<ffg$`vls%M~@`uhDh'
    '$C$7+v|BDZMkYCy#qNuCNIfG%j=o-pIHM@9Add0}3P=u2-s2o}$^+8Cy?q#ZENPnR=7?&dENyMs`P`VuUyJ)-?7#lUFYSC8K&ehmsP*%q*V2|ItnSePhj3QD'
    '8RnZj18>Cqe(!tVGeF}<SKe5h4)+_bySklDfc8dm5%St2qs|V?Pk5-Rh*o{98R_WYAZH?de1AtP3Us6YP&<iwEk(16jFJUFND!U@u>u^A@}NZ1@w?Nsf}^=F'
    '#u5)3`z@ebaZ%M&M`jW2mKRRvFn<(Y)fQsq_Act=O_LtbO3X+Y8G#iys8GLNSShu2VGB)ygqISsMq!4^6sSB+Lv!7lMpe$%-Ds%Q_!|u<1^%NPV=yF~0*aqP'
    '!hkHv?tuddt|GWqG-#MW_x-S-5oJOYh#u(nJ@rks2<B>xq6HLnOfHY^risx(7hi_z`Uk1or!o$gN2|I#<B<Xbb|G(}{KLl1pypA5C$`@|eq>W0H0F)y9cO<!'
    'h#xAwQ}(oslCUu{;g4$Gwv|4%TW(RmG^8Pw*2~jo0%)lWOZ?+53DBzhw@tVX`hEL8I55!MZEc@>WWJ}9F<S6_&R4Epr=gVeQacE@yZPQFk1oQx*%4;+&Ux(8'
    'VFMt6hPdKU2vT$1(jOtsY=EV~XZmnf)l`*LSMCaR+P^S6deY27q?u_|($g)^rA!H}>honS4Z>yutMSp;ZGpcJyEWFa_N{391?N`=_GixC+&Y6nt@p1$21C+$'
    '_}eDWWZpQ$KaG8<W0|I_rrT=bOy$ZDPy}zyu&SOr2Kp7C8dlWg+%;E@lpfjiQRp%@m3|uhDcXwd`c5>j*rq@*1-Q;*qoLzlm3dW~aEyB#z0d9Z*s=a_qJkN&'
    '7#Y#6ir1D#aJN~@-Hje}H>*O5oV`sJ%5}}8p>798GpfhUeblw2f6*b;G#n=wxMd1e{InZD*^B_1r%>05Y=78+4Ra#AGj_?eB|VU)CN-aRZNGH5UkFypRrO|r'
    '#%5F8luc}0)O?_8m%Ft(M9iA|1z&D@Mx|ptjr~tfzGC~vHTcY~@$HFzt2eRqjp@nL&9A}b`zTy>R3=H<2D0<_QYc~5b%TICamHD#+RlSho&l0Z<fEnh!V9PJ'
    'V`xzHsPW4=Jk<NVS!q^)Tfh3?*}zZ_&<L-_&tQlzns?u+J!yFwDE-bo%*Mr~B4`zOHQ(!!c8@@R9U2|xhaaA_4UDvzUMdB{fp2z4kDEKONB&AZzl929bw(Wg'
    'kZC{>4DFVePS5k9zV0?zM{F~kF_k`xZC5G8mLKzIGr5`!%t48{D=u-%-&#5!f!+u@kgfHNj*@#y)4|)iW)$4elnA3yR+xx}Q92>}_9b1nQz*SWjOS6ODGYwa'
    'Ot<=@^C>rR8*wo5Vidc+QXtG1ogIeFO%LBwZosg7q&!JmK*qi|daYrBRtbs}1}9;~-QB|viw_StdY$|%TJs{2TIai!qOuZR$~ARR{De+!5DwZnT_8V?_M12Y'
    'qnULD`Hx$5pBpE9=iOAvY`4IK+U}u-sDm(S10I!ADmT078n|sx+Ap^r_%$BhHkxi+Bxt#(&cEBP@uP~bPxP@KPZee*E}P;f{bdep4ENA5Rdo^Yuth~3Y==jq'
    '&ezPj6qauo;+T4~?U-RA+a<i;wh*;T-$Z+EIO@$!Kvl?^*noej0nu)`zinV^>4}yYL6?Qw03E|audQ9SrjJ~8dZl!)!CzV~H(dyr^R->Wl6VknM<)n!-X>>D'
    'e<MAMCTtpwsY!Tf+I532!?z+0EdxSEdb{-vwtytqv<ioB!<fP{u1M&p&1B~Es`;G;lN>%d{F+}?C+R+dyNWnUr%-XUMPK#UW&H?f<Reb9AF6G(U~LcOI(qKb'
    'wD|8qzxSZrzjzt__ay!Q)|2<%GiNlTwF+npKJ4jxsZhwnhEggxw^UXV!#n_)pNA4^SQl5OO?kGu`2__ufql(pq|$7=<(oEm7DH`&tiJwMJBo$`&}rf?bmByn'
    'A}{n8#TsuT``F0fIctB?(b3T^!t$Q#L8_;^I5D^NSe{#6TY4=I|JKq2f8dAqHbMcZ!!u-+Lfh<I{5w1{8{j15dNw*V>XzS|7&1x>{l8ly|89-wt&!G#{#T9x'
    'SDy&7yDxRvHKbt&h|ZUezkoty2%6!GLA4ma{H>W({%ve$=*FNiRZQ5jscx)MdY|gOpc-H%-cNm~QIj$(n<{o`%fh1byI8p(HcIh!9d9>UDSBtx*LR>$-?-t1'
    '2<ci9C)ks2n7iAY@+P;|x_;34=#hNCce6quU;5DYJ82g~fn@o{_4t#%Z(&V42f>8?>Z_49TPq8$e-vzV$xd?_kN`i$&0&fIckr35S$q1rL7N&?<ZCTmn^2|H'
    'ar3Pdvrw6@ZOtTT?|t4l&GK)65Wj8$W@d8t=)pD2?f(JjSs~Y-@`?NZ5VcC!f8u(2OEA~h_n_{to4j*M3LmFkBf!Ps6s;=MNR|YeoKT|peM_45lp9DJ8Dbhy'
    'SY|hm>MF?(y8JS?1@>!&i`Hu+4)On8pX22ptP)7BQgjcD*xpa4k)yZTq?=gU)WZhNr$kK85Z!Vz!W4|n{O0E0IX&VW$>^|p2SK9^?bS_Y9eC2rslPGLtk9-b'
    '1G_<K$O@<w`R4}2i|$6d%9Gg^rvRs9gaBx5d~LP0D%j6YVmJqT-$2RE<}%+tHMPy2K+m{KOX*OvuAsQy8b|q>4!gY1F+On?Qr?@<-$#3|n^`osWZZFTrn#Ng'
    'H=5{_HoHs89-#c>+cZu0nrOo#^<A=>0PDUDwDyOou4Z;yh;$39`@3Eh1duu;NZKG!^^Ojadgbpt9_Jue;Wd|?sJGi#v}Zei>*V6gs@^Pe2Msh&xCG#~Lwo|$'
    'O60QI&~FqbX4OhjaS1(-`?K9P<*m}wNR2$OjA=*svp0i|m#$v7_THDKfwmy{78zJ{`a+8`h5+5}7Ux5aMV~tzeeUkgPw!K<&pGUQ-rd=CpRzpzL%lBmaI3OB'
    '?Sk9+C!Ttt+x+mXK#{s$q>Fc29gh{F!{Y(N&+&$Ll%I|t_3*>n_vtHi6|8p&6QA9`f}ovG?o(dUms~vq@)hE(>hL^0u2)XCUC~B_?b)KRmKX5r+@IYJ+o+C7'
    'tI32|Z(CYsCrkOp*WFCO#Gxtq+&Haj>ASg{bhX<(3i1+$WnfZDn&+l9U<Oand(#RqDLJUnd-D=7DXpb_&9fO?tw!B#SPdrSdmW>0$AIFb;KyV}Jwp#ioX<vh'
    'Zk<<7aWC?a$K+1n{>Gva73E)2Ot;h7Cv(K+nkLJx`ISjm8%>OGeZb=A=O^AcaXoY>gk%vwTC{22Umj1>0AMy#(&7~$nn^*!3QYUlxSmOrv^2UZ>IigM|MbH4'
    's_~`642}qu?sW|E5PBtx55FwS)&E90vFsdk^m>d9n!z&m!Qqjv9TvoDwvV8W|20@JL$4)9X3pPCk-*v1us9LW?{W_GDZYOLM$9(;RW@kO9Zg%~X-peMEQ3xD'
    'zjcP|C%<*d97tTUVb-3WoeFQb{*~DpGV)c;VSM#n(=D|oJaR;6Do|V2c?=Osf4O3*xa^G_`LCr`Hns$mdvjo|b+tELTdej$8nJcEW@<l-nO|lnT9zZ$`Et>!'
    'sb4>buDvp*8f#o0r)Z&rgZH?z?pCdKs8>~KHR7wOO7p#Q(oya%w+93#-{xtg0(-yIVZ4)2%&t!*CaIKPauoeWMbUk(-ZWDaT`!;gefP(v>KlCeLwDB?P1Qg('
    '^VF~d#G#tt=}R@iQxBMu&tE!5UUYcIJmwdwyclj&=RZDDklq12s_ibb&Ul#yj6>wJc=q?rU;ZF{`Gc7~pbSErPHWiFbXf=QAYb%Kr<*#U&RMr^%B{^U+01U_'
    'u*0}jn^bEi@SQ##44bW^ceKx;$AQ>-^(Y%-ipR^*XH`3IYgPE07tS{wa!8T&s}Cg0utWYMrEgGw*d9v<zJyGJdhPO0ej!i;s5sy5w%Wcm>|V8jr43d#FVbeH'
    '^0<{g*}#h!kG%oHs;T}&3;OBp_xZQLIjkR^ZsIrc4d7O@Vz$4mm(5gbtZaHOhn55I1Zz5mperfoty8WaGC8X%GXhD?7HnU)$E2-qfT&IGtX9?<(+%F*A)HRb'
    'Lj(MFWld>0znfKEu<eZ%s&yhtw_J8>8&XbbK6<Kks){p7TGL(HA?#Fq-0DJUur%($>~}gq_=xa${hc-P&RR1>4Q5PsxD8j$YswmTXT@8)N2K0qYQ@lN%ZK8U'
    '{LZE;(?<t9IsxCHA3d@gt+fkW2?75~8GW}N7oZ)0MWJk6>K&kpjFE0khvuKO@t4LO<Uvpyb#uZ`2+5(C^6}XQtoAM&=nXb;{kp!jp`Is?+l6QS{!ku0g*p$L'
    'VQD%eZVr~vpmbEKqaM=iwAq^Vm?nx*Fu-xy_EdLnR#95aa#YG6hk9Bju+(I1H11Xzu=~U$|H2>dLw_<K7Grlkee%brJdZ@PfZ^vj9-F%ilwK&=1RJ1i%@tZ)'
    '#V?llh|}@Hu&1L#pPH^vlc>+Cv@T9->sL#Yy3NzX@oSyrHLc5sXo5731r@v*-gNLDbW4OPMs7L`d7_nJXlv_FW;CvyyPNZfD}kgkU%=qy^o8$7M=}jbS~TC+'
    ';kV6L8yc%sKMyd(b`x#M*53B@^M}2I#*I16aGto@E&P;yBi?n`A>2N8sgQ8Ci@cix=gm964N%6gsaAj_;U)$EBYA(xroiRGMx4DE(<~-_uZqYLC`0GDLbbVR'
    'oxLyYvPId}W|-hs>s<k6+~$|bp-KFUMB`RhbFK`j#@}`~<q2=<o8|`y7!y|j@2^6(QVQOCCD_o_`}9X2|IpTq1KmYe;-#%As!heO<!?2!)^vARpIree{yur!'
    'DEstQ^-o*%^A7kWb);R~B&#I=o2=Y+8wMkjrwYf^er+i5I@?uu_xkPoEy7ZCjPa9D!#ugWl;zN=ZlK83{*IUUjaoqUt;zrfgcsJXI$-SYwOymNH-E?PNf>hV'
    'iPoT*dl{VE+bRW5NU;bYJjLB3lJD{po*(M{cJE$_>*!vZT<NA<=O$@q>YL^}n>0O8i|gLrch^U_LA1vZB$y9O8ZrqQX^!3(#}lO?s-ykrBjp}kLC!CG1Y42='
    'R{73{xcS4XR@GleA(e03bS^zPB<Y~kF*(xv5Gpo-83T1m3MPdT@al<_vq5`pincwWZ@~-D+y||&miQyT1lT4gZFp#Mg9?<<lH7f^5bi^T>ZXFJ2c)r;VEVb`'
    'd*Svs`=U7hO+RXV`<52h!)k(eOiD-&-EPZgt-32*)ykErW@V%u0p+qqbEFJ9?Jr&_kXo11IAL!q*57aMeLgz)LR@Fts>HK_(Yt^s9({h%t7)0$ZH7C_-dU;#'
    'SZJSnc*vt@1)VJ>-MEKf5tp)<UdL<ZZ0C@>K5KzwzB(Gb-+jlFG<mqCa8<LbeqX$luq7hN#0BcMmIXO!^2Cj|^i;^1KD!WG18;8aCZJXJ^}D6W8uzUvD|@5A'
    'TP?$OsFsN<7ddLyrR4`LrpP8=w9^KKfklgO8K-jJ^UG1UxcsQk^~xnJh@LlM3q%57k@9?ahIoF~-cjxv(d^^A;3uiwmfnF;e)7BRJw&3m-`!g7(sbSIu+V)C'
    '8$du)_Eu80@lAd*N?n=Hap#%**yP|6DN&4>jWqmNhTQBKd(RygC627Ah?(<`?+2>sf1qcQoUzJ%dC+xBw>WF&`j}l3bxlf<T(Mox$D1Z?U|4GTcY=sK{H&+J'
    '+Bfz}?Gp&!PmIxC9MVv~XBf!GYVu7Fi389=Q>XEPL>#Sd!jp_ZMIn_D&`SMmbX%$+PO5b4RvD-^=F6<T)$(iMKh<rmYdw;ZV(<kIVr*?BE5o-vzCENtE~fWO'
    '7#gwN>SB<vYhpUJXzB@fPRY%gdr?{|NG&R;3QcFbUK3FymbA;Q)gqL~_SO(peL{L@8IEL71k~sDhV+*n+HVEW5ZR3%x709I3GNk+&_Yqr{%qH4K(}sNx6~-~'
    'FK(%6s$?|3yahr_-J``)Yz^tVZKRtj9*rOEMnyyW^Y$6i>mB+Rw;{w#p=f@28};9&nza6C!p#PyTfR8Q=Ggc({Ir@R{FA8AL_#zy8JltmL$m5AayUWdU-ogP'
    ';dx{ItH!l9b|y4}(yHBjR7-#m)PLYL6sas<ov1nOu3S2*j#j6Gv}?2+9b(6H8UA!(2=~<)wBPm6?ue1l7{X(EyqVd+y52Ta6wzhv1+wC;MA^at@u7)ihDwsN'
    'cue}El}yhqCRfnt!0H;5lRFDE>$5wz5L-qG$J=`jz0@^fA}HG}I^UGGB>~t>az59mudwoV8gwL`e}&Hf@oCdx4teuaJ5tp(%-5mS<aDpH`oy=EZa)#gaPP3{'
    '`({<-P9Hvq;WxMBZj}8yqn;s=O+haA&yEon&);x&hQ_Z3zi)B$4Gnu>4RV0|w~|TfnalHhl)G?Ve)RchpJVW!f7yuf01yv(eqpEE`H&;`A-@Fj*`QDg0Vz)B'
    'M=yCcAQ?+SP5H;4{`Mmd{>$eM&;Y*IaQggY$m0Q(#M%2o@7uZCyVLASY{0rRHk@ySbV_WM-@xD1r?pZvkDbPfJ%e(!pYYul&a<_3o86B>&4J>-b$;=CtEt`d'
    's_P#8_*ig6liqaQ-kW>Iv%6h=qY{5<ajCPwmyVrkJ_1hoP08jTISS;jEg^Tct_~wKt^HxQ+U@%{t3S}HlAy!w>ir4lIrnuh4mbw+7P|5m)Tcq|G?@uas=&V+'
    'O&Xed;T#!s%5mo(^E7NDu0AKfdRE@b=)NDpO5pCGR?&K|(-<Vsf^PpBJ?*-0Pa7ELCX4#x|EKh>XfB#je@}9QHRV-T2WF^XRX+W<Tu8rlx_|BT+>?)UHQzNT'
    'YZ~oRn77;PtPh82ws!7ZJxU9oe)3maH=llDwq`o-xoC!mhUCz*UpdBp13v(wwTnlzv|FUV_Yn|$IKm<OS6X2_+quc0x>^s)c%K5Fx=#DOMrTq{y|;m96O;@?'
    't4tJPm^>s^CXXkNo|?npCg>|2mP9S_PYcm0SL_*V#t2kvQTlccX;!D}yW*wm7Q8eDC*!5l94}qp6)#=yg_o}HfR~TID_%an3ok$H>{9X4WsaARoADxQiGOax'
    '%g3g8fod&Ezdv3+`L1~R<QBX%1}Ed?V{^QGVsB<QqFU#pu1{{qj9k?eKfLy>mF|NkAwo1mlkbR;2H319HH=f8990T(0V+KiM5s=)xXooRJKI}oY6`O2O{<xx'
    'IPI3(1DSO_(+9Gut~G7ck@QmWC<}a8Z`EB}JIy<__|!)MHMh~a1<8G6rby@!{XkX*-Xk!hh0->rRoP5G2>EUPi7!N>J4Ua?wP>oVbyHof0BVG&d2Ok{ZQsOV'
    'y0t*h;H{s9s&`xey6);<*KObK>R;FG{p-5Be;?n~zmISGc31yCzP*1RoAvMehC-j*)x%G2`)0p|><nD^9)(Q0+c>7BrY;{4m&r><LfS3AHl#e)?}$C&4y%qn'
    'r*oJ;$dzn@o7fxN<kz;RHm%v9*-zvtuA0JpT8EoSVesu1b$3RN3he%oX{cFqLwDNbjcp))U_|<19qyK2>u@*hG(_~uol*O#y?aIOiCV*<UZW$|vUQ9Ox`87)'
    'MZT-%LkNAYS9WphV#|Q9M;ZimHSW=U(0Xe_{P)nR4>;=(A!o`Ev}17c2(d?xA9po^_j&M_I<^ODC!1<@>T2;gvAb&V%^{}-Cy(yT&N%FB10m#mzwo{fRrjgn'
    '@0PmzcuYw=_VkvWA&V}#t$Ht>nO9k+A9?lZlAiy))6wVbQ#X0<<&t{dycW72b#%6i3<mt%cOxRSIV47tCfm^VJ<?>$&02oI-RZh4B+@wTd<#V>^wCEa<2gIA'
    'TVPZT#XgtVPLr5VWACdmN|_$pElrvw-_&f<knXpb4VJeNcVdIj`0~zLc4<7iJ-p|V8r_O8fyPJPv};b>wOiEfw|0wT!1ZjfX)BIj+|75aG}*Zg%9^~P#sBD-'
    'izX48I6@%Ky@)O}a*;I3H1*9StJV`P;iN4+1*9gApmO^)5Qk0LP=duTluYBr%R+L%Y>#o_@wIE%s-+LGhMM|fv+0iLjI}q;qJ!-)tn3tvh1Ckl{Gawan54>u'
    '<;A8YFJ*$nzfH=@m~q6KOI3k88!_1$iAvxx$D*0Du$d4I<RBk75y2i7H;3D$#Sh1qbmOeydahOo6&3WS#JLTmAK>k<ZeKxGU2&VXEy>vGYSosO^~%)4qiujl'
    '4FuBAd)%WvQplgQgDY+cQkI(tn-HjxG6u@Qexis}!cbdE#_bt#jq(iAf~PfLKp$Ro1Dcc+;G?cV&u7X(8|9*5>GC^%@<nU|S`7`@Wag{%{%$IlScyWvW{gE<'
    'PJ)ALxWU5)<8t>7jcBqCf!^?PbU@RhhSVQ+aoV*t6FBG=Lf;VC8zfr3o13W(OAj5qsI$-B?;04eO86Fmo07`Y_RuxhuO?JC7}moM<&N56yDT_9R}UUa&^3Ga'
    'YTpz|V%|L!lp^M(7iLo`<4h``FtK(Sf-X2-4H3DYZrnKNDK1TNKK!&<=BqI8zDmn3JB5r_XbD<Y+G=u4^|Y!cDabABNoO_0L5h3nZakb;@K*jH5^x$aF{;o#'
    'eNs{dPv33DbQgTV7#>)xV>|;7Ko{pPIG7LBuG*Am7_OOJ^V;Qhe(rSkeJ&5+eT^G?f4@$xYi-yVTDo&uU)wu%YKgAR=E7Zu!u|bbwoPkQ%V;&d$2DuvxK*b)'
    'e=coIDZ>C(F5lm#VE-4+K^{#qH1fXqel^tV=xN5W3hz#=4v!4|(HyN3=zGQpya#r8iw0Xl4b64m@8s4Uqi=w_bNB<JO%dP%Xlq(gFJ+LQ?uh{<cuZ*(FaRsH'
    'Ax*TvO%ESQDM;n}RT^PMEkXgl`hfpWQ_dm^O0Vxe5+{e+_A7_5<h9U*X=6w3=FO6<F*P2acWauKQPHh!zm@UROv9p5bA$^Ip6WD-V${eTBk0%t{ZOG5cNBFc'
    'ESa^{^<Hi9(Y-ehk~Ab!53?Ap&b6jSk#WG*#%ffjhQ|VqHK6h>Ok1<p;rZ<l0cF+(3Iwye6^7Q2Ctpb^C9%2}Gi)YGx%NG_;YC!XNNObql((SEPa5<nZ7yjA'
    'oDt^efVAnO5L;vR6&0de-eYR6`ImI$76;00UF>?FE)L(i8l*0DJ^JLcTU-429xXOg`u|JYoAykzZuebZB?P1#;{!xV2yldf@8LuTgF|?c*bqCIDX6Nut9q!O'
    's^@v0=ecWcX>S|Lv1A{A1Kdxo>DBFzz_o{Syobk;C4}@yd*An6U3Cp>{nzi`b=6hhyt4AUyM%miZSOa$SHJTh{a3&F_T4Do-FxGg3uS-cn11m2&uwG$tKCh$'
    'I`%Wa{><$E<l8^~`sc#!+duyF@BHw;zcTi}_RSK4@4am6zyJCRv_JS}`S!1d;)mY%yMO!Pp<kQyZ_lJk&>#HTk9JG@8;t26{O-T{?QfQ*{`_n6_0!K=;vaYL'
    'yO;mrPk+5q`)@U*|KP8O^oPy){A7J5{C+SiDyF~v&1diS{jw9EvEDx_#(ztC`?}@7^UcPf|IV8Fw~(;!o=bK2yFdQ;k-2XO=jUG?q>lX#b$xx`{72{eQ8D~y'
    '-%0-O+yMEdBz=9B?<vyvZgKpuJ-_qKn+kv9S4*+JUIO*i>Y%Uh&Hl!BS0jFNll3=#^B?~7AN}+n|MVaJ(;si2^z9n^U!L=qCFF-k{PSOu@h?Z=uhIJ(`TE=_'
    '{7stl?NbNeJVo)faD4m2*Q;88u&2>CUt4^=lhL<No~n@gDwuxoCC0a|{d*~_tovuaBmF^^e|J=W^5Z4s>YU$c=-+V4Z-f3b8^C|FsnE~9pT6He?BD!o$DzO8'
    '65=;L7lc3g>%sc|{id(;>_;E|)dn-)IP&jb;oBEZ{c_8lAH~OCU+U*yy%zK5-?;<!)l(3^`Teu~^vA9Er{Dj|_s8e^8}#tonEc7-KJdMA_5-E;<!g1nFJ<44'
    '-_M-%$8-LBzyG(N<gfS6`q}3i@R@I!|DNilI{16N{^|es$ob52lHdQ`-~X+D@txZK&HcN7{V#v=orwOm|M<6m>-T@_$2)fY-p{}Lz2CQ?^PTg3F6F;|GUeyL'
    '^?N`0%s_m7nIG=9_w76RKC_S0*W1+ma>KLlKRWW=#(!?tnqPMJoB#Pt_rBM~-~74}Kl`RBzy5cA*}1Q`Ir>aDe)HRg{r#W){qG(9nMXjs?cDdz{5QN1@81zr'
    'ziR3K-=Xxi0sgu|fA2w<&usqrcYpSG(Yc5*z>>y|jP!I^=3;j&{RSS}eA#ivq4wa*71Zl+@~C>C?vgV*zU`#@giAvF0dy0R$Z=vqx{V!|9_=`4!eNe+63^0j'
    'nc`|7^#`4{w6|q_q@xbU7p9+b8PakMnGRM5aS-xs46hQO7MRn(-OBkz#fBIgS(KnOjhzNmx}UTr+=^rsF_)A26wHw{pIp1+b%gUHHO`WmqPL`K4)SBH7;<iM'
    'rHzRl={wGzWqrf$m2Z34uW4#Q+Lq=WavAbd*lnX`(x?Efk~C~duCP>sy2+;sWPE5=xVVSh7_>d#^U^(XX!31=rw+^-v6G^H8q5QkHL8{HF{VbJiUEYGog)hN'
    'Fs<^gOWP@w4>G^eF%6DQLG4{LFyD}<0Fg!B7@Q}+oVT`wO$YP`HJ4Hv!fD0T9k~Ok_V_qQr_bd)&IdTPcvPp7$LkT@M&ef}dxVN6=_!`Wu&+x?qv(i7J@zd+'
    'o;h4nu!d3=GAr&cuu>*9+kTEOIh=*koIu4FdkWJT-^X0PO6|(W3nyMu-_yvVM3JXAUiE2U)4~Te8llfa8!tPs@6cX}UXof1^&B-_d@Q6t!0Jvads$B;S;S2D'
    'ZMHZ{e}ris?9ubkT1&M+eNE<1DqLWjkXiNJPeI4w+>xy<AM4WfA!ADCMEV1>I}X<z1W>ra%!AAmx)D!wSXIeBad{@yDLz}Am|^<_m*)S5Q;$3sj1?^okj~58'
    'k*+1<7Ov0GUqdd5mpwgqyb3TsgyT@gPhPM2-hxdROCeQ~6wN+@%)3$IksCW6T$fT=v1wz*=2}1_P1+~60t&{I@yIu@5x~;theiD<HFBKE;v*|H7iL?^w%M{}'
    '?!mc4(He_w$X(bvi{?YJgu#^(6*3m$taz~H+w;GX`I+`*xO5<2##)KBG+kcMzENw8g)t>+P~TxN=XMwG8_+wfIh2@EI!Bi!9xI@2|L+xt7no<DyQIqudv$IN'
    '$nKEcrEy;RU39l_J>q=<jXq=x{In=@iG7dGI$GI3uDx@0m^ZNN#Nm<?4bJyt|HAHr^AFB7C6S<X53Lt(#?VNTRbkcX$c1rECTVEB$RAR912c=08ZSb~t<YM~'
    '&Bci+caPl)o-R}laQs65g270#6L_RCt3$!~2ztD{@~npWG5Hzx3NRnZx<l;=wwrw0$fgO!1lQF`*3xfFU5S=K%^VK*G%d4kk{<DSi?<M(Pgu=q<xA}(dK(T~'
    'N*(E0!NwRT3*0TCdy2!MObl5V(6A^yq-K^Lp>zwdEMPqYrNT$3VM4{o7Mu(?%h*4W>GD;h?T}X+$T*NKQa6H136chmHr&fXeE`k@&Leu9S!<)$KyM>0{jE*-'
    'UZ#!*lOBu=349o(p+68KkIw^bbJR_8t|Hn(+-)3o=zbvm5c_=|ukc!Zw-eY{!tzeDEe<x^-NSQ^?b}DN5j~9E5#J|qG!iXAbw~7pRF<?}OMOep5W*&_$$y<7'
    '?9=4|N|U_^F59=+ltmSLJ4*II&*F0nnOy92=@dbKjDswc64J5Ic*0_ZnlI@V=?;`0XkB@I;qZ*wBKBXjZBwlSyHgz4vN5Q?h}|j;im+5&*rWb{?3@h?f+NhH'
    '=tQyCqSZ+B8Wd(S=}~4Q=`8k7bR9wJ#m_N^$9HxX`&Cq7Jz79DCF3P(Bfcc)vHxUDV{Zqi8)haSVH;0ZD4k+&%cV1Q&JdM3Q|5grW}ftX?4DWm@!p2&f)jIG'
    'wQ--tc9lB`xX!+~Ks7^|C-2gzuW%K`j*YJ!+{<Xr@HPGjZAm2mCm5e$W&gtCdqTn~^jc7I>2l|2|8DQ#k@`g_+5|NforAy7BypC<krQh^4zax9^aig7$~ItD'
    'qo#pRpSvYIYS1ss+JRGB3Ks26?2c%-$8{=x*>VXX6LKMq=>?2yG>l}DfaVpf3NCZJKgqf&`z$P1klvB4F4X75MKUjOub|a9J!N}AdW*s)gbg-3FuK1RPrQt='
    'euw@LRnb{2p_t{eLmORsRWLV{9K!4R5v2dp-BYJQy`gmLoWJ0-B8eSdO3>f%JSFR#bQ_`#(F*yX;aJ1Nm9;0iA#IQFvOlE?GM};^emF0_Utw^UN(g0(FKd2o'
    'I9iF(#Foa*7cW!j^(pM&Fd4fJReE5g#K@7UKBcli1^g2VHZ)prXi-m<`%r=+I(a(WAefV#f^L_>F}qnAwP;tDoiAFMUn5!#xcU-1#V3msq0~~bzu{(`wj(a8'
    '>eiL5FVi93r=WSXI!otFt?d^FK9^-;%Wwhn7ETZN=*n&N5e_()K|>e62H7R5D7LTIShGLGK}ALhncpFCr$qlh!f_RcRSYg{>~WChYfrQX7AJfv)1gW8Bknv='
    'MM>|;eE09e=?Tj!-p;t6!>udVAqPoZCGc>QRyGc&arAgMRjefN+LU^aYC}03sI(Th2f>mDg*ZCK(F^k%c%5PId<09%-eLWq?Sj)9)O_fy_}Ieq0aSI<O5pg^'
    '(=h#lNT*_o5(k*9sAF<@g{M#wRjl=~reSiA3t#F9P9-qyU`Y|o0JD3_u4Q3j@&uj(3zbM?SnG7TV)BZX35yC1n{3!HzPwvU>fZ3Q!9foP6SB{I+CZU+^#x>K'
    'JRCtyWmt{J4KDV?3-M_|z9(N%<|Z~WxHwbkfVGFHkjggRN@ZRH&~LoRb5Uu1h1-L7qrevfoFr5#tYS_P)uYeHBKb`*H*uv-*$}1=D4N`@LAnGl6U*KwO~+0X'
    '&7CysSj<qV%(tnue0r{7rDiLlbcS|G$R_bL`I8!1E*^4ldQoNYZfe-p;oihr4IHH&WAGDnT;rpK<sAe%cscBkNguF6HI=6-%bp5xBOKe%AH5qFX?NlE!bw%O'
    'I;}#U*6=<8Bj9X{3r8OBaG&OU5nFSaG_YMk#fRPJmm~5X&@4hG3DFjE`(FfP&ZVQu>W(sdE?GFAesWA~=0$(Sp-;^%Sxsp8{3^5h#I?=!=pRVCkGmA~iyS4X'
    'a>B%o!|iXu?unU}m=o-k@s)sTg<3-@+|j+@X(`hqE_=}E@i`+~<2}$IIK!nY{f=k@%()+7hNoAkrl~rkNS9p)7CE|&MSF7k`d5q5i)!PKz{a~ljXY>$T$j-b'
    '{_FzBD%}j}PWb6dZjXCM_9-&z;Pj|5hA2ykI*x*OttU16Q{pLu7w;ERe_`^-`iavfr7Pqm@S+MwN#+kKm&NRocY(YqeOvTB<d2wn@xtI{8B3L4JU|QZZo}j%'
    'dz~CrNg{5aX;zM%IaU-<bY*#_{+3=YJ9o%v(%Vy`L0Tj;g+ex32b@>g^kkUFOB<Hf+c?ohV{^o>G=?4-S@4uOIK{86v~2LR_|kvjLv~Ev1TK1VIZI>Ack3g#'
    'LFDo*2YLxNKDLUS%JN~xwI@suoaxDAi7J&A9bQ;CIOC>FDV4vfoFAduq+A8|9Z5974ybm8LXtCi^sA6kaO2WS<Ls3-eaZEpv_Qv@sEg~(N0^7oRSb*UB{8zx'
    '?BjTj`Zf0F@7{@SsgIydn+0TAoDSah0{k0ktN7KVLydMeop+oYi(Z96S2km*<*{CZ-Hd`1L{HLGpkFX5!(uI?6D3+u$jQ}^p@o5o`4#mZSkF_X_%^O^D#7WD'
    '=iZ-d6s)+L`%AY=>3!@aV3goU>3<8WMK-T8F2Qxdm%Qwjcz;rJg5?Kwid3;a>3O-8;b2keLfVpF>f;Tmg|QMsBQKthuR6Ar-SaR?v7Ub`vl+JN+Q3!oBX~fj'
    '4a$r-b9yZ#>abOjSsrg~&{}`4=s@9*#m*G<{YTKG;g*}(kFZAhCl|^vwd3#s{jPM|lsIBB3xP(dvnaCY-Jmnz;+E1rFEh}*;8o|LvJeZb_vu=uMu#>kpK2T}'
    'FmdE!3s1^91$?_me-37bj3w(KmFDqlLyZ~a24GES7f^SGiea}MY(B{g$-m245s)7{p}1M-DIPy#--cOUE*aQu{t;aQu-o7|oZQB~Dxf`ix;(6WggH)av1MXy'
    'iweWW1{Cg4U$W!!=ptGmmjaFMwBEwuMC)7Zr)83-bO#P4KHf#;lyXaFUGzIe)Bl}QEgnqpT$6qRmoLmHJJ*nFDt0DPRdSic(iBP?+Pb`RVVi|Z{@pC{DF?G2'
    '6_0owW4i>oGxpjP9KLu$Ziv$addI)?3y{)iSs*pyriNhvO7+WoRyfq0k~@a9`BuJokQ7fvu?Nux&I7PDC=WiI`(G46TTuBS2@l;QlpEYlOI=~hlvIFht{kPn'
    '*oksMRRb1%yhW5bv2no4McjKF&GC_zwT}%o$QDm3v3K~C+(>f4ZkqHtlrFrk#DNFtER=&U*PsY#K7wkW&ZA$vK$UoHD6W!>!n;;}Hy*KHki}K>H7#e*8B$&8'
    '-VnknCSC5fWgYOTAdcb*3pGPRi^CNM{r^g`23kcJjb&b;;`%2LrO~wEH4<loL4zh!3~E>z$Z*ZM1uT--+w*in;S-CB-}mSqq11j??RRDf-5hP|TnS~OOC^gl'
    'L#joX7_>E5Us0)zn<m%S+|AN*N`oOL@?5RPP8ppX-Yy@r9T_=3T)=C9!iDh)&WirVcvVb2MO&xIm2WwiC-5|-q)y2ec9tA2WO~6?O|%Zx*1SGTRn@W??M$c^'
    'rRvBg{}J|B6|+$jn<ktswyIE=px=>s0LEU7gG@uVTzKASsYbz$l_@lOA7MjACR=k@g^-x=`U)!k5)CNkdF0bdh24%ydm1W<uCSj2WsNr%nrGhGIp%H3Z!ucp'
    'B_XvAM;9)Ja7(_83%Mnk)>#>jl*iWvKDJmbya(gh>w%S~@ea%@9!oUX(DU@kRQ0(-dy8*#*f(J4pugi%l_omQPTcFl^UOydM=p;eysTwhfJ#EVhz27*R5>zm'
    'cH?6cyc%`SP}$<sLw!p7JQdXJEZ&1OD3P5N;FN^cTB>6{JwL(%Ty%Qc&?~TYgQW}U9yFCD%5p~G=uC_<U6uc6#jhj14DPjf?aE|=$su31a&e%!fl8h-4z|)@'
    '<|N@z^9AcVW(wG-(_zoqBb79o)}f?O<BYco9P;?sV0#ezJ2dj#8%z7ngADi6=pJOF65+(l1r{~fH7MJHW{<aNtlzNKLbrv^9#(6aEU3N{qeiJV9!!XAI$3cz'
    'g5{3eA?H-(JMnEI>6}z2P&rcbjMfA@8S)C!D^YPsK}Bk}INZf<1nGs$PqZ2GbWB&DMiD8%O=m<;kgemW2(1bAdobPcJp=uK8kcuA1N$m8l$+V`EP<Ju=qp$*'
    ';IfY0E2WE|9GC)}>5&>++V0_;VC66difhx*-SOaxrX|~rG*|x|Z+Y3a;c)vupz7iEgm+W)1Njx~sSq1O#Sndp^(nM2R4)Ba{BopIkY>b22D*iR#gna!bS~#0'
    '>GLVc#tWSpo^>i^(4SCah)QMJJs9TWu*zBm&x5z}<JFpYL%dYrHl$LCuTzOqn7hBV1{|7bd$d_|#(=#7o-);3X**od@LH#CnRoqns=>zsUvso7vbUu{{#~8Y'
    'XfLS*cz3BUKEe_!oKFOhc}Vikopl@*Ie(^!DeaINO}Ja&8lsFvHOtcywKSO?-t9TgUA*Q&Ptss1%`sS>C})>iVBrAG1KksGXo|n$Qd<0^B&K{as9cD>4Aotn'
    '?qI#()0_)6Xt?NjTnVtC2CFCS3|M72=P;3#u9C_bA4YJwN^J)IgU@+3YEYX<KP8Q|%qo~`z+wRVP3-wlN#S|+Yp5`Bq{lCh-4}KnSk|~OgK`CCXE?n6a4XFd'
    'UHc!3L(;j}ZDYF(Wd%(mRA@8vVba7-3c?dERy<#@c7<S#!>*hznCOBxz)Io|71W2U)Uq8>cl=i1*PpRbhRZp23Se8bwCUiGHNsX2TX$}h*se%>EoX;ro!Fa5'
    'GlY5)D|;$uKY}MNZtsB(W96C@T`6augu;mqXB;#?<)&~}DAJc*lC8m~WC<TjspP<}e}r{@onTmmswUMJboTU^%B=N;&xIs4j#9aiS>m(@*Nt>dxRkLl<d*Xh'
    'Dy`acP{niO7ZEzOPg+eX9tJw)yOOl2cH-0U4-2sD;aQX2863rkX$lW<*pqCYZw+p&@KK}54fO(_vlJ=Yq>*hve??iPs4CFbQr_}dX|+j<1<qgO&7`-a!vbEF'
    '_rQd4fr1GI8*D8>4OeYP{sZQdcczWXaVv;hyRevxe#1<J)e)F=T3K)H6o)04A7W>~(-3oY%D8Z;NpeDFj%#zM>@eriq|3zsk4@T*-u)Tg*L+&hxIx_u6-QFd'
    'kW!n~kUMIOs-md(zJefw^)_8pi4IUxu4oOpwX|}4G^ufCC6$#KY-_MmhT{nvMZi6Hu4%4|Z^w=U<q`~skS+7<L5Z6ba@?GQQGrT1_E+zLBEx}Pl!w1_t_1c?'
    '<{K`k_<E7`fLa969(9X%+B>}j1%BQZDtkz~(wK5}$P)_^K1^M>Ds$V1VNyB@gr_uVigS#^q_obcrp~LdXJB5{4kfdBJk>vfLF}whC12EnbB2<8NtF2I$+bki'
    'mCO%xY*SJ>yk_j&J_3tJkB=bt5$xr@L))e1Nfu|B<e=GslhWiD7jj=Z(0@|hkgW3N3M)2{Xj8F@i90Fv+w7#WmQ@KoNBkqV(tI3?o06&G>^%8ftUHj&fYW-{'
    'lF)AQXu?4jN5+Q&ps~U2j^qy1Nl^Yp<%w7o^m;hXgEOT?0S_v$M_4)0WR2rS{90r7!JGOY6?ikCg&s#GJkPn;6W0ZAkJb<uO9`r=QgNA$8+}w9`cM+2!TyB3'
    '9NH~NT;tacl6Nt;Qm^2&1MLmyPrNnJnR2&?{uA{bw)^l{vtL9z2dk9mI%E%08FCO|!C?D=tqANDq{g&q(P2rc9KX7l>`?Vds}6Nbc+tT=)7+E%g!@G})H!h`'
    'ttI6wXf`ZNN!+DvNv+M1$qz>=eH>M}q`ciL_G;jmG%v?ti&c~qS~4j^cOcyX)+dnAAUu<=e8>U(seJX_osL?A(g7XEeBSU?mmne~)cGQm=4eHbchTrj?un-{'
    'd3$)ZX;7fmlRXbNHCirdqAt;aaL4NfST7vb$Q;68iB<w%5mkn$>Z>)tyQ<9@s5l=%UM5`%6MQr%%JWjE=ps`^Djn+2{}W1Pu+gLRQZx&dLTsAwGHB$`_y7-&'
    'UI$jht7iGS2dgf<#=GZ}HHP94y{u$sP+gIi=4$}D4~q;R6Fk&tRDpbi$}lur<acP4K;w?tEoUE}Qg*E3eZOOK1p|YX*LdDB^OVg94^_Dqz)awNiIK*Q6R6ZF'
    't1xd%?f&<rJr?84&wT96#Ggy5jvEKuiwu2n_wV)?FI&u%u#u+7{**dHukx-XV4mjJj3UK^5uP{bjAgi^xrV#3ENW1)ur<cvjXU|@$7v`#lWX={yKwSKQOM;M'
    '%$Imb@?kBT9JIPv^}v3jmqh=D{VXf2ys7zJyiJ>*CNCOTZ{f0x!x#0kygkR!LUIP4LzpDd$jDk9zy6Cn>y4jNRXGIrpZpUjFZt%7v4T@q(k0v|>`YO^fkA>c'
    '2h_%R_u`jwdjY3?v6VIHyweu=RjEI5R+Ns5qctZxylUa)%<BQ(EVR=&IKnG0t+KT9($&DJqnV(Q2emU?Y+MdSbK%gZ`9RzOR?mMm>bpOaWQi^cbRLk1a1_3K'
    '16inKut-%iH$&4WN2<&#`KiRN1zR11Iuwp{o<em_n$5KhR(AYa!}&$o1(em5FR`!xafP)cZ7uGeIJbwB3yI9ztBbPS9Sv1yj(tc6(8y6Vpsr2J2s4lWkY6Dc'
    'lF+HjTwSFShxL!wT%O7b9HE`ZuXttt7fE(<?3};2QMjh`0`mhZDxdhk$sXEMOk6OfzcrT+C#jiqX=6VEqw^7zxt9>*!JQ3et}x#6RF#Pl7cbbic$vUKo!fTo'
    'XJKT%8`s!>Leqv|Na>_>AODz!J@7j0oFIHczD_lp?@6-jeDA`nB%LNc2Jjk7Zuey<vko__Xys|7Y=w(2o39gUEyO!=E<vpshG}lul<MHi`>#duUit#3H0m>)'
    'rf8>%go~F36b3T(#Z$;$mg@4|I-#-T+LTU9X+5yG{-nV5X25O6XP0xT{#~G~4E+-`WzOH|alzA#R~l_bGS1Wf{@c*jp`7CU9TkjwXI8?IpHu22sU=K3JdVLm'
    'Qtyb(6<^foUdT_QVSWUP1uvA-pzp&x_2mM~y*MaII*+b`h8EqL(5cfn0YR1aUDooL^Ep+9>zeB~YLrmdMTvZ7i{~+RCbWKudBaEM|H58U%B|S*ps;|v6|0o3'
    '=Qy8`RuQKsS?{H6NHT;dL#4!9^WaqEM(>N$A4UAKAv%a(K&8#C1XucSK62$HS`vpnyeK#w#$J(YRp=a9dq_LQ={@#eys6^pDTd3N3%EBhOHi4S)uwUfBREKF'
    'h?O;2YIGdz9VzEhc7oRso(8rbd{mU8|5-poRpwUGssDSRu;EhUZG)WL7h{<p>9B=nAFE?X`(T`DaKiRb+7Ef$Kn3YAC7Uu7dQe<{1RgdYuvRU4QB))aceNK('
    '6*Ky=Q-|J3Oi!GU4hhUDCRK^s!N-x)33k`eXv=o-&#1Neq$lC1ih4GVRv4Z6Wd9E|9Ej%8wM7kOj%N6Hy_KcBPuTP&GGfP)oQp+=T_2WnY?e5);c`zlN9rD='
    'Q{(W&$1VCLSWie%>V5`x4wDPAC$LY-sQj)rKY|o%I&P+1TGC)n?E^bm8l51UgVGsF4!*Xu*<dk`dGFn~sPFP>%DE)>>R_nQEn=z!2c7Fao~mGFv0RgFf|d^V'
    '6(=?L5Xd-;!}dpDa%LO99#T7Sc7go{`fCYQ`sZn^;iyf{8On9(MtC~nx{YOBs#zK+7}euYDNF$efec@KDoMQtJvHhluzWa-abKoj1m!+=uTtGm+mZGRT$k3x'
    '_|+xX1g9)fUJ};3s$AiTo4&NK&>WL<poK@530D$usJ?487-@VBvHyhRoKETgoC60`&?}mboOgAC^C!6#+O6=USZmIMDZ4vVj(<Gk>x|PYDDN<cKIv<$>O3~3'
    'uZZ{}W(T|_?P{Rls(%M%y+&hJH8h=t*&51Cs4Ve(;g`V$N3<1%E>zZWR3Lv3#RCo3;^wHnkjjYDL#}6O7{Z{+t12&&V)V&Bpz>Px8}^o8`m}wK9YNE?(S-at'
    '90#HY&^mHP74s73Hkf)*aHGPFRf#eh&>czR40(gDCn#h&ITx)&eigp*!bN=UsB;5V%;z`Rj>(_CGgB$<xUk_>#Px<uGn7qZc|ge`G?WyY@60lG(tlipeh0^9'
    '9L{9Epuqd9X*!F?>PMi`JJ0<CJ#>yj^ix#3<H6!tnQb)^O}HJvxM8x((-}`pE*EgT#@7*_ZLp7IDerWKWuLvYB*)Ml<8=eo5Q|-$S!5JNSH!Ro#q>d1`eRt6'
    '$tqK&Ova41N_G0Ag!`)Ms~Q=DS~;2Q#q&Ob5)=~ef$^JCHQ=z|&J5-@`Z`y9C{!spf3YXCEfW(C6Y;8;ZIj=Ve2TigIP5}t1a=1d34F|GJ|{)9T^FibX_@Rt'
    '^wRnHglYOCNOIZc&4x}bPDk{-QGbFe#O$V6LG1WgoJz-m=L*dtmP2-Jws+9o&~*vb0D>oPRSnO?K}I%d>N=zfp_S%z608=N6C73FjZEw(V|R$BgJf!)S0lCt'
    'Uv;n|t)`5eFORWm-6qAa0h~h7Og`6Tl9KUJdPVY+I9A3ZL(LK1Hk{1=rBfhp36&QNO!VuJ=~MCY?k92HhptBr8;51E%Y4jnb-|^IbPQ;@v8!U?3Ww__&4S?*'
    '{jqFx8JiTna9Wo^L3R^v_OO*_RZ{PB_AP85A#LHYhSn2KHEtInYeA5Z!Jd5oivSJ-$Tui|z`ZZpfpQLn6KphjJ>mPElTDZxIg{YC2e&jW6?>$ldE}&y7Xxoa'
    'n)i8DW!K`K#zzNK^vrTHTghgQ3Su7zdQM@lvDt-r9hLj6&uKmae}p~nPt*LB7TvkN!pxW!Lq46UbD(A#tq#{(xXVFt1)+(~Iu0hB>+^Fa&HdZHaq>jDH0D%6'
    '+DdVPyNM`k5M;qn<Wr(U`9H%{7t0CCDkfj>&4+6R`sue>k!ZrD5R*GJAAHSVz9@ylANQ$bWBo$sBsM*E16eBuze>}i!<xJe-L_I2elh*>&et0yoH(*^J!H+I'
    'NKs4y^-~-rVcHVo$oV}wMZV9lcZNcm?<p|m@Rc`I!l98HVY^O_La;*oI#9--c7qE;ww}1ikbV@s8NaUNZDn)9r$*TdjBTo`=^b+MjM*zR5-^L%FF<1@%Jntp'
    'AI_DRS(sk2Foi)!+P&DzL-7Scmr?=b)ljeS{D88Jxe)4eY^V{OV77>Q0*gtAcAvDmEHW}O;ClU(wBYi>XCJDGkFdy^8G98dD^loCGD-ds4hN{3)EnR;g-+p5'
    '#^`jw(B7FfB^p%Q!YK=d5fpr?*ErXP^bFQ2O-5L6QfVMch@2x^Q@qV#{RHC$g(qkojCy=fgS?=51M?kDY{V@=xk&3Yj%+$sK{4yOE7zMiFG(x?G~=%lRwrG3'
    '%3?)8PgR;jNgins((ERi6nZ7F`Y=;m;L>Kr<0+j($=PwZ#!D4#lZPheCs5EnDP>-4q-#ifh}RsB#?p!4Y~ZZI?Xon}ly827Etu+LramQ=5+3N%lwp-dPt2w1'
    'Rf+u^4Sc9wF=@tbO_Yag&S>UIU4z7u?@Gp3bPV}2mez{~h9qvB%f-Pac63RkXzgMx`UtDPIEw-^)kbHDdl%+eeAMBz<F;aTm&apC8c?cpa!i*4WQw5eKIvP$'
    '5AmF%TtQlQNS&oRg4UGhLu%(?UXx>pr}mdyP#HDNLa@h6O*TtjCCKi5gef{^`Lw2@Mc$APA*`3uuxOO0z<^vf4$89Z%4Egi0VjI=nxS#vPL<jU#~V0zIO%ZI'
    '!;><kHyG6<wFecD<F?F{ORK$G0S}dvxzpX0y%M1t_*c~W;5{JiU}Xyv4a1tW#(d~-!33iQ;Tn2VS<I<6kdacU4f$(GUHNdwo+iq?+UnAET;@4xv#NIPf;tyI'
    'T7RfhQyH=xH^w+pRp9upY0%2zB1@YJ*N6D*kfNQ&5PDZJcf9f>(Z%5ewt49+IpzM_3Qku4HYsxhoIR=tUT8E((fP_M9r7Dox1cmbtH)E9dsSX{K}E)HN1Fm!'
    '`!8MW_OQ7?X8~RrMygn3A$>;WtJA8enPd;))S>bT>;ylP9PTMl-F5g9#LDWd>S7k;<YP{eRSokIZvvXwvQO|XC+!W)I$vBt8L@eS2S+~Iv~^^mLBRsEN!=lq'
    '#<Xd`(Wl%I)+Q#D^KGIr{V!l}z(xpF1&WQ2z(=DXmAN#AvbS+Cz+#JfqhDNkT}CGb=QFG>n5|K>LhAzMAGADjQIYo|j`|c%q!QrkfY%8(Hqu^Uy+MJpxB>2O'
    'nA*b9g!EYY4>}Ye*}+QsQ&KrUb;1IA2EVpAEpWw(Lr=`CG#ea6n0(-PB3p~DJ-X%O+Q4fHry7{SN08-S6QUM(E#B>LoFRpY_L8GDWbcq~qNYPpDVK!@l|u)-'
    'UEi4ub}wv1I8Q?N4Bi%RcR1Jiu;RlU(yv%aLcf4#Rb=<9g>Xo+B5@Tm595=T_*5KoX9wpQ>UX>xq<s;;#}fnH6ze(crr?mqOIfxRR=(4?bE^e?8y`C{ZL~6E'
    '4BojpJ8Nt=WTNJ93&R@uZK`k3D#KG1+B2OT_QsH1qpe~(1+@|Pls#EMYYCG!R_d~?QN0KIkZ*mwq&|W*1y#{|c<+3g>yR(WQs-h8`x+faqS=3syF+;0aXQ87'
    '0?sb>hA{8^xk<|t7M5~*eR0O_0+bs`RB7Ab_D*^|sy?9e6vgX_J`@@<XrWO=r50Y2E?K&ddD@re7AmVx&n}xKIi9FC7d1ueJB&i9cUkeG-NIsm`!~q;c{HV#'
    'PMax|%bZ@)aQ-Q&3ib)-wb&b?ca~NI^$8y$?gvs`(`$_34Si+Hp0qH)viYd7eTRf67n9~qTw0&BK3Yw%W|UGjqf19++B0I7p>fAboAnNta@1DIsGP0~#{J7Z'
    '<&{53lcQ6@z~USq325(78K0IS%z-H6pA2Exl+A;ehM1=hcg}~Kbc$j`bXA6O|1V;<BhLQaGND?a$vO6iv`NWuNh9@d4$BG*UD0xwdqK9wjUy|gn9e{$<Mx))'
    'w@=CSr&OOt0a!^GDRqkQ=};q-iHCWY%SoyCA;@v20Yi&+ZD=T2&hldlv%{zM87C9j7pZZewMYH3_+8E?In@xqB$opZQ*8CIsbf%uxeM(8YdzRy$@94DW95m1'
    'D=dz3*wE~Xn+e-lvJTik;^8jy7c@FhxU#Xv$?<L9IqUIZ5Bn}&kLXsR*x_-Lnl))p(S3p0VcR2H&CXfo7v8n`=*n9CQ$}t8W+C>*d^$lVfRjVh4zzq4Z(**^'
    'Q;^As%oQt$@9p7mk9`|8O`NgRE)Wgi)uL0#EsKu}+IjJ-#X(JK+Pl`0c7Yp8k*{!U!RSQ$EMKzFuxXLvQ4N|$s1&&~1vRh34t1MgD*o$AHSrf|>M2BQVeooq'
    '2ApzYZ^_C~G)<^0Pm!izNp8TKnV9NNii1`k!Gg-2_;qJ%fd!K)89Z%8)8Mv4zy0x=La~L(9tA2Ddos%7IKk#xv@ACFA5O^k;yab(*#^dU$}O?2knPHek#uc3'
    'SlCHO=JtnGO4#5|F|<E|@c%^b$yG}hs#+(heCONo-H#}zv+CvJ9(!w?=R_~l>KX?_neU+yegw)hwqxJtM;&z~`o}m7C|#%U%#ka4<x|o`<&~dTtlcFwfkZ*d'
    'V{syESa|cW;88szS`8XY)(032(W*&f0<8>&GioWxu9LaLhpIAZ9t>hn$HA7gsSFm<>Pb~$<6Tx;RK98P;7CD9Rf;}&E4n{$WO23sw&$QEbv)o@DfUAO%itL3'
    'C!sLJVT<!7_ODVOz%i2k<RfgxVVMTi_@%t{fO8>c2bjH+brRi{lp5P69=E(uxTu30Kv|E20F%l_6tOsji>d^NKh6CvPZB)b@g_&rBi%GCE0vtZVS!I=HeA?L'
    'a9Q{W?zkR{=E$@|`3>E#QagUJd#6qxq4_qRu(KfVqd)j^C)x${ysW#NxWUchTNek<PsxBMDO51rp78Syc8~r1r`rabN!(8{yW!d8t+g<(Ea-z%XDsAVtHWW0'
    'W2J$2@l()K++yQA#Ax=m*ECu3dK<e7$c<#|a^{Jp1BShK)0AY6)F668Q9vApklEns%&8nK%j7G;RAq1u<qXtwP_3esdS@OGR%B9*g9lD8;J*0O<a{2LWxM#4'
    'iQp1~zvHOP+6#A)ICoeyNa5tU@K(`aW+~am)D(jq_eW?=Dd|B{0pXcuZEP>GRh7COSJts-@puwDV;Jw)+i@ZbekAogL~FLXcz9B2^ydi}&DgJS)Rsh@GgmZD'
    'aG7w?{e=VVGixombaA`@tMEz7g5t$R34>A`bg*fCd8OkF$~D%XVodo|#6tj1=OY+?O4c~pz{eW>1Yb80-DTvnQJ3T%T*ZD}$|`zuC7Tk}NTY-I0~elgbP(nB'
    'RJGbD1)Jc#DEkf{Q(T!z>jw2P`2iQEQky_;1&OER7jm6TO0m&SN@K7cu5HEAIcdtqk<JYcAte`Bnd74Nw#RU+vvGydlWHj#M?9|b*`(8px&~b~6sDxr`8S{|'
    ';W@x_2}*ghhc7o+Zpv1pV;}oP?DsJ7!2U=Y712^q-pcjB$t0#SwC_VNBi4pf8R_I;JN@!P-7cJR(5^zE0QV!94{0a_Ug7C3$tKoZ8CB3)$>mAb0MZYN+V7gm'
    'osi0N94l?E;>DCvm9yzjT85l%94-G>KF;1+KYn%irA&I2GG|I(Kf(?)7qF<ldmXwhanzux{x)t@KS9Ca>*yowLVt_ei!wXuu25;2vF1$rf01c`-86M;|1)+^'
    'Vp{a-fZyT$kxxr#4N;Amy9a+F$t{kTv7)-C4dIfCTUd8uw}o5(BXD5;z>W=GidQ)(t!R?{a`|p4|JTI*26hvg1f1`IvfQmKSdlDt^mMV4hFgPN9ZxnG>)1VE'
    '=fI=kC(WklhG&zb8Wb`RwrOzX>mE`=SVUa8fOEpop_fX_JkKgrQ-IQ@`vr1ancLVelYOLD4qiQ&WO-_m`2x$LTa}AdtkzI@Ry%z6+V5(Of*~}{vD4y0_T8Lg'
    'rb0K9pDy&96dCLsp?u}j4(|@;=H#iuGWiSrZ7*Z5i@iG)=dwRaZ5q2}HWqMP$#p{a6qR>;EAiz*856S>H8-@>_>sfbh|_aihFlu|2b}cqF%|nlK?gE6mrAi}'
    '^Q=zJgjx|ed93Ehd19?1t5fWoSgmkRflK$l<yDc>$|GAa9O1?J*B?$K3YpvMPg?H(`D#UA3wl26uUs!fr^MxwRN81exWCe<&!ryBv`>18%n?sAU@ReZky7u^'
    'BDf_mxx!;8dRN*;><zFmq1TAI6}$wLT>hzrt-AE9px3#v_~azt=AG)Lcl|=wJ0CU}p0F{KM20#RmyC}v3)?Z~6CXjH+EdDev}{4y#Dgc33?DOeJF<1;QiFXB'
    '@@+WoK4}e3KA^75QwK^q`6uq3@zBEN5?;;NGqG5LiY?ud99CF1_@<&Vi<dBVlC0<?NQ&`7E#P|x>qChw>Gx=wgY@>T`BcjCOoQoGc0)8CyeYtbO4l}B4Js;Q'
    '*9ENt)q`xe=o~p&07V(aG}&#)RIpcrVihvV(X3Eer(R#GCiyE`DXj=`Jd(-_$`4G8=&DQF=Gy*k_;^dPo5FIH4=?F%@G?iO2)P$JWk}6q{~_B0SgPbI5*^}p'
    '4q+Lq9q1?caAb1<r4FBFP(R_~fL|>-_TOezhJD#QWwXM<1D_4PET!jAvy1i;caBWd@jWO<qIfRt(dZ_H4rg+dIm$7I+b&%rO1fh9pjqP7g|dCj*>C#_-4`f&'
    'uAJz0#fFW^u`JSbufy$zl_VBNUrsr<llqBNhb|H9&s<jacnymwI1W^Fx%WBBvVW4avWa8qMKC&&yOC)FP64NEn5u&>&>ON=$6k{ZI-b&8J%HbpLLIYF>^C5N'
    'gvB*}tsry4Y?g}+=p`|0%kYZj66)pnHN;jQ<|VpZAg^KnjQ1Y+PjqgOpK_!)!{OHi*IRyuJU*Zsv993qi0v6HeC%wYy~W7Hc7m*sf{au>(gUjY_|@iJ@ekMx'
    'u|I@#jTFdUSE$XU-h^_2j}>ejKimnoYq(C+BEm=|SR2xN+y{_OL7~R440@BdJ^I2U74nxB-X7AQ;#?(9mx2>psn}bgcf&#vs|l`o&|iQT@og#XbnM-^yx>Hi'
    '5+2_MRBKVek;?SXZZz!5bilVEmxg?EX};%7K`LW3x0J|AyUfbd=AK;hxqFsL0Xtc^?a9k?_VVt#ad@Zw2r>sUR&3jlJb*iff)1?%Epj+4fmYznNO~ck##q{e'
    't^B72{fD%a!*7w5!RrcYGuA8Ywz00LFC(oDSUQ+9G*>VWxznSz!j>XU9#mLCDakigt2_8S!y&^?`9rbgd4io4cXZj!rKh~YTHKkeH?iM_W{8HxojMsIEc5R{'
    '3X2{W#+)%(e?a+04U>x|)N0ba(j$w-6Pqox=9tX!agS!1Ybq`R$+ft2gzf}0N=b)OzC&XO^(}@43U>G!g5SVgN4ljyE|WH7_x7KHvxQe#qTJgoWAjYjgp4*B'
    '6-*j1+rxGtsR0dkoZDhY5lvne`rCA6+=bZ^?^)5t(#e2%$E&hpQ<z@jsKHB<YfTI%GHu~w2#+9kHeBgpZ;OQlezm%9F2P7<21XgYrMa)l$P)_#nBJtl;N6Iw'
    '0hISr=!#yDZWczHx8d;C!FC51b@XSFyOJ_wmuKu`A=J>YsWyeceP<WE*pRM<aSDfxjBKfvadJXEDN6&dS5V_~X+x@olO`!gyIA4f!lyy`I{8byWbkRTGVUd1'
    'K=(K<bACs`3|bFL&uN=Ntp|ppsvHz8PNeu=Vy!IGF4+T0gfiU!(H2H^(gxz5=(M5Il(H_&+LCX9k^LX}q;n#Yo(82ZxXKw_aa+Z!&$l~N6*(F_^zjkMOhK!%'
    '8yP%%bgQv#{>HD1v%o=)Uo~2k_^5-X#&{RI4LFv0{De*smQ5U-u$u?>!HSCJFB#tPR3epT_Zxa;m>j5Q$tL)hP<gQCbFzrnNV*l8tJ*WB`W;$LXch2Pp>kI='
    'L(-O{(@@QS`S@^c$R_Z*#%>>ud01{^-;n8wyAj;;*goKS6gwwY{2gp%(&WMphc>#(507wRve_fQMe8T-rrh&!9emQ#yh~8+DXK`{YS=o<rbDYHKU28cKw*iQ'
    '6PJUJpa%KAWV>{0%iN_=14E1JdueXs;D(Jh>)W?IgS}0a7HoYi-N^6D#gj`~s!xh?n9t(c;!GF1HxBBMPE*$WVu52rx<#?K_|mY`r%{jUZFGmY?#eL>XH_5O'
    'pex(r(7pk$r5IyTXj~a!(_p`V)3c=3T%Sp>i-Rp6UsUvHxA^a1HmB<lr>;0t7+5@Myc=6kZQF#gn*{GknF74FXeorc(ksu&GnH~QbpA_pA9NZ+)Wk#$ycs;7'
    ';*V(I$~u&Og-=zfEM(HBPMenw-Oo~0{Zw*3BgGk$f!uAlrzNq%iE=R$OpdUb<@=r|1$wGR1Ttx1p$>YHA1f~SZ><e$?N8cb%%#7^QUM1JUKUszz@ZJsm5tm-'
    'P+;ZqpB5%<R;gTAQF8`E?~C;>PPjLzw}8P=o=x=U*ht`WhRH3g9`UP859KOaR4x6#?7eAs97l2}`d4bGW3wA-Gzd`AXlOy#+@#^Xltv~sGaiF{%?7#vwAkow'
    '^+FKT$Orqr@B6+J`@Z8G&+&QZy$`<j`v(t1W>tOS{RuB3v-YkAKuTl#U5}1Pc2#9YMn*=)l9`dAEKW<)M`Qtfpd)2<f*du`HgF1oKZfb@tOCrx-Y3u3%gq%^'
    'SF<!x_4epgZUO$@q8mf<{2^&+mMSLbX%(&MA)q81C*`dzvNkO(?vaB5>8Osh^-3GZw6$01s*<WfGJv{UrH7}q`%vjxBBKjrb?0ojS1BJqtJyw3?4v7nbhw2!'
    '4V)jJp7oB?iXPJ0BdwOpm37L&kyO8}41#mnAgxTuCu^i`NC5+Sb4lK=kY3kIRc%V$h%&WAj={FBKHuw>_ovTKhNQkq+PCzl8PW#hHZILAD>b#WcTK8-*L%uP'
    'wKNI8m+0=1yuYE8uhaD{Wwuv=>2T7g4Azp9#y1rW^2RP%2CBSP1}14uzcMpQC$|;wQif~E)S6OV{q|^8>4D)GAiJy5^z7NlsIs>}N9x}m*PPYtpC2rfu0_&Y'
    'OLw}+?gU*MlNXoBe&zXbpR|5NS0>18i_$YEbu3HZJ=Sf?!=1FFgKjjF+Gb_3Qd(+~>JP~v_?9zr)wDF!DvutVAC1ZzUCP8LIoTvrJ@VR?ymxSZ&_n7Mq=O#$'
    'Xi?r+B%Mv9bxrB7q1~%w9!%C_x;y)(5)}D}v^Ao1ZphnHbgl2KYl8O9(Aq)L@>*(ICoL1CvPQ0)phr{k+$i1HCT)iV45rp`X{1)^IF6I!X}PIMS?(tTt<ui0'
    'QonM3*e#d$DFd~nW|yoU(1vDd3ktL;8y(WfsZxI+PjxCib;{5tty)nk*QD`5y4-bB8m#{32?~<2u0d*_r5*L8{(#Q)(~T<9T1_hI$P_r6HPY@O>8&P9HDt4z'
    '4sJ>PlkzIqf;D7+l1xvYH4Z3CQ*;Yl{2tP{c~-tj*T<xeDP?m+>Ti<QCzQFKv)&1*ZiO7RNhfn>!#)3Us0@_Tl}>V6C9e(1t5Z_zrqobRR~p_PgDu@e$42Sw'
    'G99Uvsut+_ywo~GhZ>aiLuCROuua|t!Q3D(tSEak^2#PzJ2@NPI6s`0nun#u4*7UXu3A?1MkN^K1Mr?&m5L2Iv_xuNQ_%T?Qv}9nxm)R4rrT}OX}7%HEl&?B'
    '3oFvh9_<?^!z-k@o;Dtl(+1MjK~@)(gGxHoFSj>It-Ew}C|fG8As{HX8|9wGKkd@01Lb5`p4g`=)6&WkIX$G{UG9OKvnXx!NOOmBO{=m1#{Z#wI-=BeQcyZ8'
    'Af-p?(1NryB)8AVJuu5F<<8afqmlEIZRuo9Sy&)#ePp;!Y1t-Cy>t}R>^>bjIzMWaPlo8sGFhG_?O;sr(am*bd*e+N2!<&-S}oNyk<M;1-%tA+m3i=;7o^H^'
    'd1IXHca!5yY3x+#?jy&`QsusMv?sN6(*Ah~I0XiO`b4UnCiCN@ripfy%iH60eqRRLq^4IHUOPWolG<0LgC$b6N!BX}7|m<d%0#!c(k7p*|H~<@JEH6DWN$=1'
    'K9)AXlWwO&^-_DIvRrdk-=XX`%M}~)$_nXjlZH>|&W2R6CY7(zo(>tfWa{LscTZlQBp{UM+oh=kW&e~emecjRv(ZX<be^`()7@Rt&`fq;%WIQT@5!5r39{ay'
    'bacuaBebW7&NnJkZDhVmnVFHAXXL4QI=&~(4AI$9QoE!~57CV=W#Z&bc?+4}l3ve9Gj(!L6{+7>%17kYV_H5W*N)KRS$Z@}+qdM-eWkx!nwccSwaU;isRrM?'
    'mewyyGh?K(Ls<kpcpx43%V4ohOiL3Zw0lo(>6A{!XhT13-I9T2+M4CzdfK`qkAtVbC5<deGxKD*|9r2UZXA;OacTL4RQ4+i8*eK2$=(j>UXsAPo2@)Qnw5`_'
    '&ib09!=AI617)#;937FG3c0U>pr_kH7PjR5E@i4uo>)8IYg9mg0*_R7(5)e*4F+mdns1S}>i?g0X<&x-O-fUT=O@(^%&^`%`Jj!|){wn6GWwb}tdf--WpRYm'
    '51sWco^>q#%epjwMEf@7##UvZQmzFnZ&aRKRQejF;R8B4LTeY4-Wh2F?7KE)Z<CJhNPY9lP=&PIMwdW{ZIS6(+Eb$}Y|5`s<n=*$s#4moBWsoN>N+i-mydVk'
    'nsvE%pDxUvpH`jkH%n7BZ>mP6>Itc%N$vv&Y??N8DvR}U$AYv_c{Va4?Q}?c$7E)cZk);!r(~^#EH%rswbJVydFzyP&Ysn;kbbbc7s%A8)YD9>T9vtWr3#$A'
    'Ia&d;Y@Hsjkct_)c|glMWtg&kV{&s1S*s=u$E0)d{B(j=EYhYS`FM$rjM8yXmAn6QOgTEH4X5&2v%J}*ER0EY;6>EP+dK56Ra&f9R(hn<Rtf%Y?xox9v}fo|'
    '<sRu>lz=va9i(UF9Hzkb8lB%$y4%hVM=AJ%^-Xg5wshPf_iQM$)wE$jUTsqbc1hK~+&urMb-8|r&TqddZ>Oy*(((3LFZgy7^3H%Vu|p=u&i6<Diu5)}H4CJ@'
    'k=AWX{R{Fzg}gkhY<DZ=+e*zEojz42PL--IX|Vo$Z;<rYD&yT`q+FWcKO3&4ZHr3t4w)X8#wJP2f-*WKpAIXPljmSYOoAiVMa$=C7YO?P^P|4EN9*T@)l%h>'
    'vauo8kI~^{(gy8Togc5A^)Au6O1Z6HIt5pKL27My)7qviEXk__a(T5}Swri~>A(SP-6S2;N_oY<?vtJ4v+iaEqz}xA<3qXPluXt9%eFLFrwo86F+pl;-yUr!'
    ';}gn3pEP)?jE>6_Bl7E6I^RW3CP@2)vU?y^ttzc;Wa#yq_I0_UlJrbV+r!e<kW@LKboG$cYTCT2Z0^V%o6<lf9dDx>n=-g5%j=}3UY=T~4g1P;)%g+V-w_2o'
    'o9?;ur2}OWSiOeK^qn8}{Oh(fGDI=xc~#zSAl<cc(=cgYkj7Tz#u<5IRM{LQE#M6{)0sX}IZqDy<l!v|#u2=-scosMi?of=_Fk#J{rq%XZX1_&z&#(7xAw?!'
    '^IH_vb7Z8E)Pne&qRSg(c3SCQkypn_f3wsuEuFTV@14jE`*gZRUM`nc8qY?dxdSq|MO$~}h5GZ;O1Wu5>6}#h>&aR(X>K8Vy`*c5bPh<%D@x6he6p(Sfmc5v'
    '9WTo5$MR_f-Pk96y>w}u0KrxdNaqRd9a3snX;TwjKar~kNniI_U!4S$+g+4F=IjlTwoTI0pv=t6GaIyRkM^v+saPRnJ+!l)_JNovKi?aW)~88Bh14)c=jY{<'
    'UNY4v0oN{1N)v-p<)$>$D;*6;8|(7vm<%Jfu%nc>D9x|wM3r0x4NXYH)ns(?e^x0g%W~_yRM{vuwbA1t>0n#gm?X8s%Ekg|Sf#CVq-jjro|PwF%QZ9S2kY|2'
    'vT}S%I%cIq@E;dReT&@KB-IR)mR8cTPU=s|a2MUKI0p}XqgSb3r%lW9#=J7vPwQ7`M>py2P?msOPsn<^G&ZGF?a{6+rJ|8mgFEz^R={ikQN1j8yq0@cl*P95'
    'qejwMONX|k$|DKv|N414GbjV&cY-ZFtn_WrwQ2caRXJ%Vy*spePHJgYUiV9#bIQmO9hjwa6|{MT^zM?q3T3fNsoGM8tLa{!bUH&;=g9u1yuPc{l`DOxq-#Uz'
    'T=~<HG7E}xhm<#v-PyNCgGy^JIb0;5#HTCdjZ>w!OX{mvD#rh`MS=H5M(7-vHk(rYkkY>)pH@nh%SvbG+r1syd`wUFrG;jBsGRJ!kjX>2ZG-ew)45iuvrgXG'
    'qRr!^x{377OPh_fp;N9opr_-~z=_l_C!O|?qXSykK?V+`!(I8bUumnSr_*G%QhB{BH7$|KR=PYbpY$j-gKv+k=+=?cxuDc^(%ns^XGvL|r;V?rwic;#ULILg'
    'Iu_|j)%i)Sbh=5ZI>=U^GC6Tpxg<BNNR3PK)EM1u{%g6soHTSO6*FXWjjs17gTrK{LFqb?Pg~{Ft+UZ8I?+v9R!D0N8Q+rI7U<@-+%hL`9uRPrK^~1(k@Y3H'
    'vs&rg{u{Y{Njj-l8akxr6LQp{Y&D$k4U<8zAREs2>y^%vv;G=sXHVYVmDjrI<|sX^p<}D0ZRPx6gPfF01Fz})guL4;wXTqp8fmdcYM+<OXO-UR^SuS7dxCBp'
    '(AApry;G^WmyRq;>tKcV$!qm*_ot<e4%&I7w4BK09a4FNJT*+4>*R_PI?$xdwv+ifQn5@rd&%@3soi*cTuEEo<;o#x_LzVq>>vHp1lb*-r$c8o>u+cFN#{D9'
    'oFhFaZz}7gp;4u0Q#w3VU~28o%X`Pl;EvRNLKeUYnv|Os>Dui1(VntcE1!VR(kAUM|MS3`_Dbn=OxoN#KbR$ZGfK^#G|@xaI>>y3v@?EIdq}ow>G0{>eZYQ7'
    'TAd}sz0XPIl-zWv3~nkF(90IOGk?A}M*Az3u^rL}o!(F?n$89mm7zMh`8DZZmU@q*kqX+nD6bqSb;HueKAo9Wn&;?t_n)SvLzo`jWNSgL>zA6U<+)C}w?w)s'
    'q@LI0c>n+GS9-xW-jcz2YwD7_2jzh=r4MxWoKoH=H>}Ccdt|O#p4)m;KB`oW$TN#_Wi6@dk%m^K{e7uv@BFltjIS#d)pAX(JUAj(bSoS4(pC!vucvxd8aSjw'
    'r^?BIGSx+nXO;aL0yDa&Lar~DMvkSP9i?JVSvVm*y|iIgsoJHTE6P#3G%~NWR!S!m(qNZVyM0#ME4Lhx+THV`b+SB0N5Eldp<{!}bUVdcmIlc}qdd~1bRQ_='
    '`*ePlln+Q#^W?Zks+vAOt(NQerNav40949^T+vT@wxyaH0z5O>N)J1flYUY=AnhE;oqPXuEVotA`c}EESvo8yjoZ@b=^L=CSBB)u6{%-S0;$v0Kze57xvBqa'
    'h_0L}eS37Si@b(OS4T!Wl}?c7tI}~dZJegtjej~Iwcv78pH)vvn=R5ouL8ojsZ#D=ChIfwc$qA1(fKiDb()N{$!p8XY^O9=McNuk>x^`;CRZNF#|?C%Rj#g;'
    'SN4?U7V>)GO-0|E%4K=9TIpIN%OkXTTG^j{Q}vp34=H<-a&I?X-zVjZ=f|tGzWJZQ>^P!5yY#S$Hcv@2M`XQK>IGZ8S)PC%9g)U5rM#1F?$PxH0*Y#JR_dN7'
    '<)F3KrI}XwbrapGA*+?Ne^{vkOSes)D3`ZZq=R9pdygLWpVzlaH5+v6P+kPLs7LN<Cp9Zl^A_pv|0fWvwQ|c0nO;-|k0cPjYZdb3usl^mb{5G|iv;%OP_1;l'
    'c78BUddAKV%azJm+OtF^8|CJ<KlPEZIr;chUag{)>&noST+>WD59#47>2H-6JC&{>X@5}a8&x`6m5Bp+vH5(zO&$U^pO*)E=sHO4Nu_O)9G*&x2eh$In(wCD'
    'by7zksjoOcU6vOc<kkauzfx)$S2p{|K(_=0JQ^n}HU9@a7?V#Im8ojEwMN;jQhM9u?M7*`MV{Z1*UHJ(F*&T4pli$h|5mFE4k?}MWNl8`>5|v$$SP>~E@iX%'
    'tp4?z9&l5JN%Mg;aU?a4(b_q(JRmpqkiBC%vrQVOXiEb*>5;mJ<@H9nep}iFCw5L=9hJ7${;iEp^^ulk+P5NA?<gai<fvadYLM%@$y&d3*ht44<QkYX>q^zA'
    'yt}G&b<0N!-_fm6siKW`PfPv%bgM$yX(Qz)%Ivb-xOdjEBR6%E&OzF6B%f|d;9&P{NvkW;N;zqJEj8}Y${GnMyg5Lp_Z7Tmt(LqVmyW8G$_cq+f^^PE>n(KY'
    'n9lCfh9S~EB<;MG4#w%;p3+-IYSyGV7~6HSHAnlmq~Rmdzx8&%O4<SAXj7hOCiBh8{;||BBu|V=Ez8p3WC5w_mFJgfYv=jivfKm&uRcGVl8=U@ifwu@Lyqgo'
    '>Li`mk=v`u_5zt+qV>I`W}gnv(uS&kt|luda@`WC1zU4NE^m?6x6V%%$V`(mG$<_sKCNVRg$@sq{R4SqN2*zoS0>2fDDCT2_QA^BBHgP>_qx*CK!y)UTR#Eo'
    'c6U&2E2sTCw114&57Mc*^ZijW)FD@Q$yLKj=b^j}KNe}%q|()^w9F_|t#Zv8X<U?APNboFy1Gga=V?njZK{*ErX_G{_j}LxK*N{Q?JnhHOloXAD<39J0|dm*'
    '>Yj42CpEN?cJR$+<>6gv<3t{8B-5*By{*#Zfn2#NgY(nYAdQ^J?X6@JW>-I1uT|>WmGU97cO)&<%k|Z?Z-EXkl4j`L4ym6e;PnpGD&TSMjnnaZdbCLD%H`uR'
    'WqMrcsy?f)p$9u;t@f;DM;baJ;7?BV6EOF?*2zJav^XO-_bDS=WU`C&m(%V=X%~!<HPQkG-XU#YmcVQrZX`7wv|&m*Jf@Q)%HZMKqy9IQU2pfNq~<00bXRU}'
    'kic2rel53l%Ex;iX>?gSJybS!<P*?qtIBK(scw*G#>sS@ygDcK?~$fGx#p0Z4A9LZvaw9+j>+(xG7siso6@&`Ht?FPPSd(NscVYto;<lmW@hBw`SXzmdeEY*'
    'A4yx2O84>`jOObn%iE;lSUTJzBb)Sil{5}1gNt;tMj75$_N(O9-8WT@w0=yg+9i!6^4N?tc|ySCSsNxDlVoUwOoLxFOzJC@r7qgrC9f`#ZtzKZXhWCMu|})F'
    'Dcm8$O;X2>RQXz-sgNsQleT&?zfK3M$Z3l_IZeTMuj`dN2G1JCmHAydw5yEQ%T+_?d-Y`KSn37$W<}}Tqz6aRLajW}@@BqL>K`C$?WA@_I;~QUC*<04T3ICz'
    'u9NOvxpk48bdvpMdQ?doTfB6&iR^CDsd0JrgbaaSxvO-y%dMmTvL{uX(ybMFdWLqMlFg~NM-%dP@A=UHY44JU8_$n-pOV!hsqI9WJ3T+%JU=+4ZG-1WGjB$K'
    '{^N3W8yOg&coE;!S?vJnu9N3x&Z-+|L;3m9xZKw*4ev=ueR6NLbTBJ7&yo2W>1bA|X_4x|NL*BE>u6Pv3@WF#QQB^Qdo)Srr)c#UnQE76z|@(RYG#xQm?vHG'
    '*8IQjO8Y&^VJq34lJ;KzTcunzCzp3iN99sm4QZ$&W0hoelU5v)o<*giM;YEwx;Dx3<p14FR%hhJx}em)DUZ~Xx*BP@lQe9Q&SNsr_ND@84>aB(Lw$06{oA8W'
    '(l-A8?#)Pb70N~hsn{l66Y{`;G}K6!cjdkVrE}<fzlDrWNy7tj)hKB^A=6s~-2RF~val_8Hq+jAvOXa<?8;*&^2)B#SWf4UDX`;6y}Udq*KCoKX7YNC);22L'
    'J9J}?)}F}qqhxdDO?fMwtWmlv==d(FJ(e3z=+uyM+)nn#<oZMT^_&b&W(Ca725F{2I^3j<OJ}v+w5gZ&f-l-7jZY~XHFUJ%tP9MD{Wk++WTIah>7a+ra@Ud4'
    'Rq>{xTACY^=Riaq(`8`!YN>aMt}VT(sG%U3XNRQjLwT;9)Q!lEEz;DIa@={=F+~sR<SF2VeF-KWxDZFZv>cShDBannogEY$`I9+m;7Fe8p_Q|weUB_G%VQgI'
    '%dynga(>iuzTZg>W*zPc$G^SNvTL`l1xuolSc2al|0JwF6+}WIu>h|sXL<jMLXmubNbrZEIhe+xpx7Vt7bGx)2fQlw=ZgvB-IH)iF+ByamJ_oGDrhxb?1wMv'
    'H?E0)x8~GL01mC@+H*jpR>wq@m?|L>6~kssE~<lf)?<qGG|;>;%nsxB#uKq%)aAYqlrQGTWH))&AAfuM(YMz<!u2OlUit&qP50EZTlN!T30|(B3cz-;zH<v+'
    'e-yeGa3PpW(a{QDzyI~U>zC2N8HA-04hod~;YVAS`2GA?&zd$M6{I`0lL#~q3*hzt3{*;+c0BQ&cB{X7o4*q`X)Kx$!|_Ng2L{o_f2z!9VN?wOw2+Ra)%2{u'
    '3oO%zk)Vzd0lUt@Thm7ek$GJqjVA%7J2Tsznjj?+3@6R0wRtli8UvFTd}`@3FJVsb18*Ke$<W<M_=On5yr7Z%?_lm<kfBfm?h%4T;YdvM75L+#LpF0vJmE7H'
    '!Lc%)Qu8F6%cfy$6VG`=ky2|498)371a&B)bNM&7EnLoX#icf8&%_ffNH{U|MGS2pJ{^$3$mU!;Jk<@SIHuCDB^VFJgW-6>A1>gDHas|v_7+$M4lZG=S-rH6'
    'D0-fzCiT$zpw)NC-hnB6uoR};4Tg=rc%O@9ahFrR53Nm&@<#XB7(8{SR&hv`axo-ENE;w?o)JbWw>2A!6D>{Z&-(zTgg-wd+WVx#%|pO(H=|r!BimC)Vqd@R'
    '{Vo#J(Xf~B2tI3~Bd3BXDZF^*7cNCq`~#TaC6|8)h7+HB<o4L|njnYqj47rBbaAZ|{NQYpGN*xB*j=?4%XOW7D?P!nP~Ht2F#v`$Q>h)&sFY`!j3CL(G83@1'
    'k8~P#NyF$C865M6i$quYWHqJ?Q*lJ(Gr`Wjx6-GnPvKR)OBy^U=C={U#t9C6<#~Xd=Cien`rx@8hahg9R-(>_h^U8gI~!HNTH_M<+wFGDmjmETdE67RzstGf'
    'N^%{fwH`~O8<Uz8Qp=l}FdPpCKurK<xtJ!36?I$1qfvzMXim<jpW3=>DI(m=u?Yp2`TOU&!Y3c+e3~uz!vRah&+x;|Uw-7i0C*;={L!bD^m~S?E+~$f+p?x6'
    'kC_!V60ty6(;?K_TD!cMkG;dGhyU^~+N->v!3*r&Q{9G9O|%rlNkcf_pD55IC~s8-ZOvHqEf+yd9tPd^*4&`4!$mG@>M~1COU(1GPC>0@BLSE##T7%zBbakq'
    'I)Jw^KdWRODg6GOZE_&8z}#*jQ%JG660x#00)`_A_42+GJ`}Q_8syB=Am)Ad)HXN8mjz-p@$L|>|06mG>cGQ*=@i7AVLAnMK>kq3Mm?jFBiU^EzJ3Z7Q|3h>'
    '6xD27y;Q1gYeqg#OU?D2g$U@MJiVF*a2&SFn@B%$7mnT)aQ{a$=4uhs{6d;&18nP)Di@scF=}mkEYiS~Cai=d?~G@KBJnuL`Sk8T=an#cNO2*|L{DKXQX(Y$'
    'C76m(1>;r0y!bvU#*h?A@Yz)W8G`VE@a=Ez{pOZh35*R?Dqz->31xsnIP%{RtXlk4F^V;{6)_l&MB^GHaHeC7uhg%sC*E)GedT4p;@$)%6pa+bv<jvKqn0WU'
    'pbA_2(ABTrhHmdeccGZiF#2sx^7{C<R@HD5Jr)<xV+d-qv-R+s2j4vYeK1jcFBmU~1%WawH|Leo!b1%oV->K{B1JF!G4?(8hWCa?cpmbViov4dL~b@dMuRWK'
    '5Dt&|tplSnm?-kq1f)0BW`bc<0^u;wSASOsDj^c%@Y#!tK^r__!7LdUeE<kE)oOV{88Dz$Kux@=zE6iPb}RSEwc;T1#5Irbq6ip08jQx_O98Z2@I0DeuD6<!'
    '6uxJF;`q&)l$58Zbx!UMLkB}?w2#6r&xFI1YcKraU?}8&a_y<jQ`AQWroaB%o_Q9B%P3|aJ)`soTBcB)whWtG&R4GVq)7WE4SfKl<9oMPeBlqdQd8$TBXoIj'
    'qto_x=3YmV2{ZwK?FHcbAbM&xkbFk){PN#``BEaHiNxg?3^k~{;43lL<#vN;GR>aUjKcx*#KeV5mdo2ZU3o^*a6ztpvUt)v0crmWYm?758Q%w%d3Kq8zXv2Y'
    'dI6y_7axnbMy&Lt@&$46PqQ2hVdYRq;}JG}d;xVv6`H13E;<A1>|zQ~Q0JFg+6gF^Pg>jj=|Y5c)-y>nC%`7rq!l*3GCR~TRSJ?Je*y>}@JAED7orMQ72)9!'
    'pC6zq)PF%KB=~si63Ogo>t8XSF)p<@<TOU0MKOQCNCN9mCL;L8i+{O{Mh$S7jXmK9#)T5m;@odzNzt9+rWaiZw&>k~I#_2{N1o4ey)gj%Gj`$~fE(Zg1_!D3'
    '5==Fh-Xph=CAdr<inP~Y%!`>Hxi0_&wn9nNrM>nDOvPu5AGyIDPxuRpUCx#DpBd@dJ;jg==pWLi0Y_<vD?%c$Js3JkzA(mfNQ<#x1Z0&E0d_9|5esr4@X|qa'
    '@hu$?31T#b>zS<-P=CgwAuwv5Tm!Q%7zW*%3Q848YI)ubUm#eJx&RFrYqCZi{ror=-f=HmJTFQ{l5rQhdbZA4KD$#_s#7)C=fVveZ%4JZom8X~8lJ0SVe6az'
    '^TTw=WvIxQ*;kudW&l|#oy3<>mr1R<HOTzYl*yZc7Xv;MFzu&>XzG|<AjX_PskJgvK{}P@sF#TTuq)*=PR5j%Jg2%vWH^H}L%E=`#vHnt_qm-Vp6k$!xdX*r'
    '?sUaWNIW-ic%FiS*;ookN`*wRD13SQYawdUz<{7-o}!+$qxgZK=U^9-(Eup36bBw>D~_bYD4L9A9A`%*>W~UNQWOLmqtO3c)YHma46b*W7ipFNRHS|`gj6~0'
    'Pq;3`jY~E+`Zy)I*>+Tmv{BA94YR3fw>=5D#$aZ9=X2B!@3iJ&(ijX{daLC^d&T(aPPaB9JW2vBO2qrISR|GPn-5a>g-L$4=6MPBmW>baF5OpCd>3%Z2Wi0r'
    'ksZ2uacOILtC8vtz)Un-hvq@`Sk+~sI3~u6BcVX<g=V+gnOV|HBhKk2No$s;AKKm53|%aJXtp%W-O5n;#BoC5cIlIAKm3@Lh$K`c_~e=w7iwS@F|F^u#n1us'
    '2Ly}DFn0LFaHE?I)m0soQq`oO^qT2?C<f|zIy5u^>x_Y}j#BFB%E4aoVmvhcn+j0-@{Ry?qb>!Y_T{Q1GCNZ#z$`p(DtQ^uqBOlNddI&0=(sO!ScGXr4qIA*'
    '&$bPGry_aR4Q{=TE4UXi=w+Zdb22ba%z9b%=O#4LFz96<H2!2T(lP60Ks26YK+-YjWk57OWI$4w^3vJ{uQ32=8SyeeMMh7GuF#J(YU2VS`oD>!{ENu@uOsp<'
    'VZEOy1ux*t_Z5t~R7(Z-lVw~g$e%3nQbGPC`Iidrr%J<Ai0Ncw>baha7*X50<eo&N$fGyycy>fDXKu&Ih0RWJVdEn6QKPu!hP%_&Mg8}4YgNZACbDH<2cOVk'
    'MMtNY`)77~(*aJapf17P<j-Ezjf?%+pVM=`pvnJOw<@4H?N)Ve-P~}Q2<{Osa_P<LUa#(?!!FeKjuTsxv>GB#HiZ81-9ch^(t^Z~R}zpvzN-+qM3Y(dOj?R^'
    'srznfEJcN-BP`dDX-RuU$WHV%LB1M=^+kS1q3bAiO=Zm|7uc~1-*4_d;tYP36AKOlZa$N6!+Rs!u~4w2=-0V7bJBhLeQwUjpE!Qh&-1fA@9XxMmwYi?P+a1V'
    'J@+PFCaya|Fm}wZvt_6(e32DN6pOKVmUhy_8%-2nw;XhLI67e4NyI+k32*o(^~>-1m*1y3hOX_%Tb^xY5l<+h<DarUuG3vL-)!4HPG{dOAq}Y**|&okd3-=0'
    'A0s>fFdfBV=e;a2+{T$R15h*;3@2RIA4dEEJTzowIss4P+!C&Pj4oR$^B(!sv-w=ngaDGuT->?v56{IiZ%mAa`~{-x$u)k$Nj>P~<tIG!2as|v{Gp`CEDNaT'
    'Q6Gb0Gf*!E+khhb(Pq6|2c?n_S&{3uDo*hMHlnir_zVy(Q}sME=%*K-6;3l?FstwR6Mj5E5ekVRms2tKPs1{GYS=3~9vQ*Upq|;DN!ZwN4SVm#fHUrg6R}9N'
    '>>)~~)CSB)x}d7{gqNkH<v422N9p%Ond#ySmiUWAuqqNh{Vc8lFzpGwqJ+nOT-P|%tHZMFVrp%fJ6@O!7x4WHJmN3Lq7hufcUdswhhB66<S`_U+tNTh8Pc}{'
    'Y{Ov%nA#~59QHCA69ddkLYR5i@zH$Nk%=BWt(WUCt}Y#dPAYfGhP7RuWoRJHqlI8WIf?}d^CmX$g;{c22HeI1=JV!yW!5$IM54>|-XmPk7m9^ap+Ly|T*!xi'
    ';K3YNWQB3rC|V$2Z4|Xb+NP>rtm(2IicrTV^(=L^<56I)a{-pGQ1k=+S&RV><}@Hi@-IJ~T?jHEVTNiuRctEj7*FC$lti6cv0*KI2tYp+vQt7MF5qwrv5nR|'
    '0_Q}WoLcX5t+YTkjVJi&5ZmTIR1LC`y<jr19p^}?&oU->3ZDOpM+y`8WWe&|fFqRS&dYvkZ!MCacgque`c6=8JcT-$sjy7QFh7lxK|_?+@xBO(rLN3uU>g9I'
    'ZQpycbBsm-q~&$XeziX3eHgN53t&+{M)I9LUUn?qMqQ`CnD4H_f2!hUeP?IEg<tTa+E#N0;|0>JzG+x!ku?UWcME#U2hsD~A;eE7pVojS9z$y#*yh$#>ugDJ'
    'M<bwyK=u37Ii2mjaUs!=3OaZvx>?|5<Z?!3J$LMeI?!4-%(X43h5{vAP5M~u@8!}5<Ka}QJOl!8%DCT+msCFDXVtx`%63`egw12YqNN<HX3zCk#hC@+>-I23'
    'YnSqZ9^sb0Ac~ea7Kr)-0o`u!p*ifsMj1n;YDybOF3-46r|HSd02{G^K3jb0)ikFD7Zs5j;bW?psmEk8=pMnm$6in@7CiUqfs>}e*8-ZHWPt`OB^-;hNs2w_'
    'Eev8{eCpmgukm^60htusOb;Jo2ycqdi}>mx_z3oji+1LEpZxOX$8NT^NoFhQ=rCFCla^-5L6`0op6@NwjR`v3Ag|8A*SAN_0;wO6y5|L~E1Vxr(V<hzBvR>U'
    'SuVC4pK83PK1RalrjfT#5HR{J0cd%*jItM>12tFu!V6Cv5s%=)Xa*A#xm9l~mk}g|<#jmMe)L@XZ8-Sr2am4Z`r+E6`?v3Xb>CYOxOVH>FY@?%Fqpz&JXsqG'
    'VT7SS!J<@R{z5!wUf>S}U-_A`6pVWW3L>!>4-58afq-DUMwyle4F`l|Ri3&EgGE`%aIiQS%jB(OzR2-L!$nMpeHIA5fKdw;g>#==D-hA@dvfh_{PKaoQDeV9'
    '_&~Ua*m#8BfpJ_2ZssG_gm_DM3|yS~NGxQ>WQ1P_@pTYT79N-odSoCZlm=n8`4M_y5QNxQ4<CMoZRBNS+z%Iluqw{Tc<R!L<QD!Yz$a;?{QS5J*{ZkB`0l&='
    'H5Osjf01|l?n5ExM$RX~SN<UMyEu`E#&2b1fo<xI7YD^qAnpxDvhw|bA~6ff`mk&!md*V1cFvvqIc}qlzaZofCyFDXNKu(L5-ZBWvR-(bi7LyRo%8WWS#H+o'
    'H~y#?yQDEzCKH>ytpUxvb2Ix+_MP`?@+&kHFKLpM$z*>&E{H_M%ujMYy8Xo$ruIrpOVy|){O&CXXYr@3SFc_czA7ur;=`}@vJ5c0jHbL$=1p(*jm(?B{N*RN'
    '@BBCDOe8nT-v2fJz6;%p<(3wMk7YxJ7h)afO+E|)JI2LJ+Ehzr-o5L)^M}mLukXU2d%lmnH!_Q2q8K(cQULY}o+m4al$78CKeiO&wc}a&WtriE%y1wJYO3vj'
    '8BF{-na|?k;%Sj60$xt$XR$(A|L_lg|3!ATO1J+OPn3nk+<2rQ=nt9D;59&Fmsra)CeFzCL$WL={6jG){?Fpc=;wd^!ACiN^H~=D6#i*f`0vSZNE99hlTIJM'
    '5rZ+{lnZJH{oxS&DMASN;vb5|aB<Qv`~ly=^F#PpDu$t8nAJZkALq3*G7!7RktEFAyT!pUo4sIh#Dj@4AyO!OD;5-o8RsIHd!iT>9)e@aBm&B&-v#41^Y2DV'
    'z`4Pf5B>Q}`q;6&|2@X-;oeIun4e_g;;+N`@#x?1zXdB=01#wgJc_3$u_Zt+!aVQKvOugcDfYRLk)ev%V4R&qEeyuMdQYm-rvyBKATU0(8uG`$9fnC*ltjOn'
    'FGHa&2bGtZh-AV)La`y<(#TtHMuzYp!DV+`2>6+gjWV2VD{)C33W-I*d>BdWD9Bt;BPdmaBL5ykVG6&403cEtHo0$ck6M7qUbR2TP$)Bst)O-W?GP?t%YWb&'
    '>%A<*7^Koj?75dqbK!rDb<YT2i>2%kb|%n>$-*1I%)Ie2;D1)$Eqn{0qbRqKZqZ~u3TA-hPe5Udpv}ne3VC)hcqL)>6zkpl;+<-Fqk%&A(=<3g2p5Vm5&O?V'
    '-H}v4JGOQXsH<$Co&NU6L?C2H^vA&$Qps{QJTLqP#sd9G#z#>n-6#m3ffx|tu>yE+Xjfk}7!^a%6e`yz*Pteb4ydhaBXKzc2^tsxQ5p;+iUCOWjT?W(n*J;+'
    '%Gg?ksxA@(BlzRbGvi{^kC(gz9Kppk#T$pY64LlPQ4AvrLx&6FakdM|xH1fu1;7EhRwTyRN@BJbD9&tv%rBF{0DOFCGECF~z{?>Lp)$>85w!_`7!~RKukv%;'
    'JP|arCJsZ9gH8Q92<{fxf{av#jUkr6IH;{b??vn`G^iRwaiJs_$Nu4>XdoF5_;GT<J2r~AaDdmMh$ZkE;X=h=7`J_4PynoXq9LQ_M}S@>{(@M9p<NQhUm@`Y'
    '2Kqh;-$X2L8&l3?;^_E+eI4Vol+WpaHsitGD|il#CnD@?AX1RTjtQU{ff|5E9GMVg8cys?)}{b%V}u0&l`(Xfe(`G7+5ZsBEMilg*td1YLo*AdSqYx5M!v8~'
    'wksHJE>u%q1e=x*7-km@opys*jOk)rzKln7HGvLMreW85`85A6H2VeH+Ahn;@E`*x#As$|5W{s##1eR&DHaw63xWtfXzbx<CXx&?v!_K+P1tlXO#VzDZzROE'
    '3qN4MT1OcJD14tp2@+QuhqmwFP93&ffJ*?(FgK*SVVM`3dmYm|%q6x-G@Eq=w(&T@w6=N5`U?t@Fdg_fzX--rC!%6RZHto_^so>Iz6@fop{}6<bY?CnGemvY'
    'EDmEN87tryp!`vP1l#&yF#H_;d7SVkl5ydW@HI0tQ~!g7nP^8V9L2tFz`mobN4wztggvx;`-MLk!fTHNj!=qB2jmP>_uWn0)Fy121mHgl;wvyypFMSXv$O|?'
    'sv5_@m;W`^(Ewz_*j3@z0FPq^h-K$wdq0Bzf91`|&bjrg8#ln{v_MKmkqw|?kQ)vNS{=>7a^dfiL69Dgc|<Fp{~(_4;@JUJ*hC{>n!(J#&(EGc%lF5NpM=5m'
    '6xaa+;WA^?3m=|@3j(?e2r#skKsw(=Q<^s=hF_r7iJWd}3(v{;g#unMGc0fwDR3)47hMD)vrH=nlRNkV3-VG=!l1L*OvcMXGGox6?~}n8n@(}uF1O$Va`+xt'
    'Nw~|$FfwU;HriOuQb02R*mVj&jc=GXLk-LTbB!7ZqWA*ZqY8QERm(nOswWzYya1WWEIZ96M5P-^78MK6GBX(jwleojm+T*TF>*x+Y8+`H^WERxyY2hkgU1i<'
    'JiPC_cmH<}?%sc#dqX9&pmC}2<Oy0WnVDd=@cn!7rN1PKw{Lv$*q5E-%l`ORz8vp&(IUMh2ryqkBncAr6a8J>|3c)|y;*z>J+6N<Fc-8g>!XCGBzj#;!~+b_'
    'h@}w$Z>MB;7d+FUJ-fxV7UyAR;|2tED<p|(oqYi<RTK;}0}A8t5l$G)+rPn@ety9gyGIk%ArIbR)!2?!2`G4EG^WqRSdraUG7JDO_a5?o@%3-Mz5f({{$6Ce'
    'pCu)zF8yKQ>)$=N_u#g08;r9|ek|q7Z+^>-tc;9LZwO!B5q|gR_E%sG@GB<O@$(8_F{NeUYA`d(@HPQ1-+7+}lbS+&u@m8BNj?bUNTEr?XzEl$8ky!UCwC?q'
    'eqiYqG3!DV{<qaf5IFb5LO<?B;V(nyo0PyUT?;%jxaS#^f0mAe@Z$qEFq-L?#^mLF)L`rU#(;l<e}Ap_HH~UDbzFy|^C*L3a29q^pO`ovR{r7^0OK9A5vV~m'
    '%}Wkeoz$p5(FhP2<EzG8s>{IzgCWA_7IZD~4qgUGMs{e(;$G<520Z{FAjSdCMUI1B9t_`5G9YR;e>|BV7ZdSYI%1jkzI@1!t1?@!h}(KrlSeQ6>&K5&>&i4x'
    '%$l<sLVA_kyhd8XNL_UFL%+m?aJv9(7d}Xjn8P1uZo_Zmq9$N;sStoFC~ib9;F;@MAgJaKM*^gJS&SV&g6Rh%rMgBI;i8Iaa7dUdj0PEcM9i2Aha(C0q*F7X'
    ';t9U8UsvMdOCWI^%-j%XxiHwIOn$_0O$hrdBC+_I04M~$7!^Q1N@~k1#+ray7}a7hku;_|Sdhv55>V~v*`bqTu^Tkyf_($5&dLYkLL|(@0w}HoIs{(fHm|_s'
    'P%;kAiMpzQ;gW&~(>zip0v<3E{RU=b0-r@zv_QPw4ns+_O!<V@uw@*9ui}On<()0`7X-n!MY$Xk(e+ksX1t~eOdMn$Q)MuY#b^Qv?0q)_3rY15cW#mG5_ndT'
    't&zYJsw1}6VP5G#mcTt6whjWd;=z|f8FV(zedc&^q*N#au0jCp--PHdVf4j?762{$=I$epS^DW-o*#%&77xZvi#h6y(u;8E2uz54&m3CB8W$`$8-iiQyocL`'
    'xEH|MHxE0)jv@=A&sz#M54hU8^}#$&m0<&#WdtnooI`|?$vD!2@j_uF6jEcjwK~bT>T2+qWtbti*~**d1G2{Q*<ePVql7KQ>Tz=ww)*lS$j+DW)tArk_-HT`'
    '|D3G~g7%7)eg0XT`9_~>Qh>X5$r#t)ai)L-ekS#^EVV4I4D#BVWN@6D{y1=Q6r6Be9gBpRpAHoMhX>yXOn`xv0(Ne3Bm!uQK|FG)HpH5Kjw_sKoN^;3iGDsK'
    '5#S^W&or@+^-PFGA_=eXpin0?^L5o0*bP+bq9nq}7m?K1fIckGRP23VKSaFWfwTNf(AIfE+{gPP3IOaNsCF)yn8m=`KyA!eNo&*5#Y0*DM+0w^{oYr>C{uP^'
    'Kbl@KC+G1aO9gngSAbWU30zVIo1J}Pa~f!d6A9-v6Da1=>S2E<?7eX#``KA}*E9I>iQ43CYm2w_29aU85=()~3z?zae8%-~M#fiQ{;}l&Jj|?<hp%+&{sQ8w'
    'LlseEXmr)rZrG6PM!#3U3)O^QX8uDk>cM#_fLvu4y0gO17%tpCMS6h2f$LsseYPG2U`-6fcn86RMV$89;vMoS0`Um3B!{dT*cub@EAom>nAz=N)-CQ^fI=uU'
    '17X^V^^G$Ywy$xdI$a+I!^xL8V@mu5zj-XU?tnYOCQ!VXpFA`?LBA@^QU>`4poQngjU4!M6aIVze?EpkpZq9%?p(tHOdVS@5K~3|Rd+#t9}GhW;(|*xjD>H2'
    'KGB4ZYC%Z=ol$U=vhszrEcs!&FF9$y+)VrBqqJW>PW$DPv~;neZt(qw_BS~xZ*Hcc*?pBaRc{P8N^o}xIImP^aoaSoOFk5{%&KAn)DW!N+*4g35X=GnR;xe|'
    '4B2=ToobB@Ev<X`^i{EgNnvn!$Pb*F;8Pen!LJ$DQF;<S>mEMqE{Fex6sc<dXcmK6Kd`Wwcv<+zEK^LENznfH$G`cS%Wp>*Fj85LH%(j}#O+5079Nue<%4Hb'
    'tikw?taIIJMLwKa{5*mI&(@nC)T@+K8U15QdBptELT20yn0)suwN|%L%dsC>Kcv=Fx$#bnZFImqMjJ=em7qTSL1e7^D58J+?H8G!8bn3KkA@z%7reqVep)Kd'
    'J;!I7Tm*XuOfBXMGX8#sBJ`QydRCkSiSbNlM$-y26XDN_qKRiFi5bO}`#4`%Ku1O7%S6xENfCg@R*HB_%;9u<HFfojB?^dZwja2ZaUmmvnS7{m)QP6|i`kkw'
    '^JiqRiGiA&`B|W#^CK^zbCFm8!^lK#bK&h@Y-@&D-Zwxif{S>AuZQ?!Xbr)XefHTOp)QX&`{U;?vpr?mo|ieEvK)`TPs2-q3IE5>pLw;7R#4H}eJOZ@iilf-'
    'Xo2Z3*<vbck+|k|tMY@nprHsVq&v9l!k2c~8{I}y<0v_GRLh(Z!$LkzVB?q!V<@+Z2e{Ay1{HyF(Zt_zp#HdN=}GmU7$frPeCOEGK5G#-NV#bpP?r|8#bZ>g'
    'UUfrSH*d97`9^eQadAR(cK<Gk`%(!$+MtSo!GlaRU3rNBqZOQXAHO$4#zINfyNebIRD?o52_KjU*=g|60h_69YUD!HfP8%iZ}-}SN5D%Z{wVi@^Ikp$lCm9L'
    'W4Vp-oc(K>B^rA%^CAsvv~xhXUGO{{DGiB%B2iycgIafPe|i6*<0+5)GEh>uz>ygT6E2GtWWHz~cukRC-+Sa(>+#*+ew%sk;qUG~^yg=(1$nJ|!NS6HwaUa~'
    'xTv65443+|^a{TY#_vT6vhQ5pj#iS_`${bG|Jon^)i3|Y<u#46ZiB&x1@&|e6EAnQMHgd9Sly%795j?q`b6SBRo>mZ!ks_hv7WE*2_JcHyw`D_ixywJ!WWoh'
    '#Q2cJZN(@bf2JA00s>>P&}%*&!<P0%F;x0pF#Naif@H!g29n-_Fc*b-PlbP4;O8Xru686wU%tQ%h(PE?K_)n>-msW(dxWbTzA=t5bmHW=8|3x{z6CGBI0gfK'
    'EKK6!3|~0VXkqj_sP69Q2R+#8h;G*h^Anik=x;$v;{t_<PHp^$S9tCy6U1C7lby_9l9z?<g6IJ?jQ`~dKm0k*5&p=U<L7|X=GbW{kHF4VA>AJalOFXP6+Wt;'
    'wX;)33H_`TzWM_zq#c*SH^2lQjU#xg`H{$T-ce}v8-p%5L`cg|J_2;X=)=w2uSAiJx%%mLoE;O>x8*rg_*H`#T&<!ckRw<C{>$$op(GgE*|!8dI?nfI<8F=t'
    '%mZkG_xrnKh`qxVD?uYY4vR{nKp=(-s_gY+@KEms#iEEuxC?R_FbD_z9waQ9*SIq*#(6bubqph>ez-E$ylyfU5_C-7W$&0*4@81S_3Rto>`y=W_-59NY=;^u'
    'uFSsBdj89t>`yW?;h&q?KRyZnsG616Jl9b8WNERo{1$rcE&k=>x2&`G9~d$WXT`_D9Zj?g4-)||@<kRq-2kRmAy{a*(}ZiZ9|(`Xx%c;(+EK*^0ni)Bv$0#k'
    'ZSZr7#mpQGZ7^LyU&4ZBwM*v#ZlpxL{{5Q=54GFg*}9IYl6q@C_x!bxE`I`#o8XdqVMu)G(;qb;k(jT<AAiml#`#ZEdJndJ>jP&j78lt^m$pEO2dU%g1*z(9'
    'fE{K*Oz=A(zJT7E!?6qe$+$nnf#iDx5H{bl+gXL@FYsS?YE}XE9S^<p5|;SKWGLiAgu{`bhS}ACx@=YBBG$-b)t=)09nRl5yr*vAuP=dsXY}1>RAXXE<c0Vi'
    'i2kF?<%9xn0r7BZ{ujA|Bi?QcPTUiE7i{!lx6sP`R&7aZ#zrI_FQ)}b*x#8ldw2ww7S_oFe68urUIn8rXP&b}{O}m)T#UjfRCnL-lKkt473MW$2wZ>4Z}ZMH'
    '#pk7>?FAPNV;=Gt%%^x`yE&(c-WsoAZ4*eOHkH~^ng%!+OEZoH<>PD5c9V*}+nb2E)Y)hzzH|LF)l|+gM%xmSm&bo%+b&ng1Gb8mZ^t8qtixxQ9_V1^X1kv{'
    'HK(OT)WniY6ICvgDDgOn0(N(?ZPt0rOz$>CToi+=zz}h%#|+pr5AIO;37OP*%ik5SNmI&hN?@U_c=xTE!H8cVS}gn()`Ypr%!N|@QB4u5hTih(c=Zbaa3NSB'
    'D@(v=PXw?2bX)7Fw;~6_m|zF|ui|Lw_nDs;mqx!Yz{>%6Sn);8WM%`<jEo#$vI1*@r=X@%jLfKoA*M6LfRCLi!sVJ^Ans!e*IImVNs+G@8!qydnkmSm-Y|&('
    'dnd+yQM|_n%zKTt=5GMO{K$z5fEg7`Fj+I!m+GM4QU|~>F4*Srb2qGjwVA6HFTJFAsahO*j~$27F0G-wTOJmDjI%XePlXTlhA`}y(b8olx(fM9WVeK6|K}ik'
    'egx0i;_{oFlG#F5T2wp&%fNC`5iM;NF61zq(s#iKXJf>xnE43@=``M{a7y0^j>ZW2$%5yYrPYoLe~&Tl4j6k(WCH2LuADIiYFlBJUI}LDB`{44YeV^pd}~AM'
    '%Iq7B;nJ*V*qTh{a>z^uyDqjGaZM?j#H&qsJ1i5Qnu{+C{vXT<0|1!Nd=U;@sjt=b(+CDkCbZSyAUhSzSB?4lo*MAUjx+GxKXtF)lw(I#7a)bf+Y$NJpSs~}'
    'tQX^^1ioQF)Mev9u#l}hC0NgxA%H%B33;Q(w-7qp8o?tUaenH8*F&|*+C<RaR6?lQ47#FbS~EzZ4v&@mwy0#6QQu>N<+f()1eZ3XkBUt_7sOebK2ugU^I#(W'
    '7uagIQnDFYYUW+%DjrQWo5^Mg1~n?N94)-klWX>Dy%s{`A>y|f_8%h3mPg~t0+y!7g*J-YP62Ns77V%E!Uw|nUeozrgFM&yE)BAu*iiO`L8w0!Ys8P#%*&^v'
    'v~>nS{8SwIS75fEn*&k@l8yCc<9LfxLAWKP(!q(z?H2?(P{0K10^nvMUqGIV=ASPVd@j1IC9G4A-B9B8;s+2Jg+VZKT}EU7Xs^mIG<k9@GZ-!e$;nHmn9`O?'
    'rc|hGh|shs0vRp7jQRp%fxpc47)*mARc7;7m^MutIv6(FS9blS<^KP#Sw2==zVPBdc%2s5o{!y*$prIxQf?QjU>jtXz6Rgp!iYB-DRtQjAW+j_FvpU?%ebJ5'
    'IV~tR03N%aIwD_C9D$i1cd<m?s&NBN;@vKI;|Lh(&%uq+ueJdvhtCaogS-=of!ueguv8srKmh<>v;3ZTUrgR(Jw>{DOH3YzmKXdm`zaLDo}M}-)8!v|woq%T'
    'bq>y>RyQf@V8{oiT~4eh-(|Uw2yY!=UV||$AANb}_LFOP%7cGlo9AGxx>$moFfJ2e;0Zb*EPqs*1S?jjM7z2|18=?+4a0=-jx1I%ES9<umt0m78(~)w8Fve@'
    'OxclGt}Ex`k3E9+m)j1|i>LWS&ELbBpuxat;Bx2PeCkd`mQ{6R+Y7qGdUKJ3<qx1OBjUz`FA;eTLej<l!3$-zfxtUQtYu6qDeTrakMM%+M`|^_*5WYRquK><'
    'Y16j9aLn1B+O~ED#0z)&wB-N`C(7l(P+q>!C<Nnc?Vrwv*FXDXuteL9zOH3?PX%Xc&j6+&Yz5*@0nfIwe;RNO!Up;U+voWiu(ttnZsw=<!U9Hb0v!a)@EELy'
    '9ISG#Q9|=-K+&-m+KQ1@Nu4Abe<mX6K%|Zk<E{6Bp`))SwDm;_P1l`W3L}-)43}`={ovTisS!%`&)BVmCOv5Ca<2Kod`YO8F1ccwLe{S5S|2UYOg+mrKWK&Q'
    'a!l{L$J!f@jaLj&2kI$Dy$boIdhIKQ=rX4^_j>&MBq)#TCgqco4@zzMrC!1cgDtkQMz5z<%D$))SIADa!0NcLITdT?ny!~ZMuu?1t6dBe^W*yP*O9~nYz|jJ'
    '#Q@7i<g!9`8{1L|?aI=BuMMbo+;J*+k)!{HagUa<6wT4}J5qc;=;?~}?xmLT8XvXd#_fdI%okUx^FUuOgYRFllOV<|!e9Ac;wI|-K#}-WF#Na}D&X%&B3cum'
    'HD9tD3-zru>lMS?8Ruoyb76R7i3^8#ga--mI6tt?So}bM9Yq7&3qp9T9DYP$<XIpK`2V;P!^{{>*u;}DepXitMMnty6jvaKJ;sXpWo!`xWIWph$E!R8JSrT4'
    '+>RB+ZP#Qmd|hCPyKf2KvcpZxj?xdXY3Sks7j1u<n>Hpm!LS7mP%l^l2CoHB9O-!(B#y5J=XY$|2D${+RsoM$DdyJ%vo&-)4`czPN2}=c@LD^|k{8EyH2p6b'
    'n(A8(QMTw}M=WS&9Kn^sM*!{~j6Dk=vlPY64%}p8hl}#dtR~Pc#(1}$IWw&{rIlx1Jwy*U=h?HA0^U0h9zOW`{_RK4o_VcJJt)*TGA{7l9S=X}?F0oO+<*M-'
    'vuB=*KKt(f{q4J0$lB7yXz^Gc=Q4MH3xA$HV=Ns{Mx)G6vh{<X6f$4)@=cIMq^SNX3wcjq?2<<iy+vN^0f;OwcNz>lSR?)r%uF6YWN8Rb$N52A0;}*fuSk?j'
    '8W`r^8%uO73$alck#Pl#@nMUvwzWEk`^c@={F>FUHa~A+=PV!8377JL&!5!tI`8vOY4|!f?Vy-}>HTG^%c?za>uR~m^}G0tJNHam2Sy1^pycXZwUtl>MXuUo'
    'hZfXXoonj6{c>~5M9sC*!hZ9(&Jno=0W8B=;(v}8N~?lLy_}Hk;d)dg%|zn-a36&_R~PCv-EpfdXM$6e;ilDIlWBps7V;8!k8r{j!#w~OhnA%^j5#3qFU!8a'
    'k1?<>Uf|aeW<u4YidjG$Ke=sQ0vEiw0&j{xWvE51C5yF7-FeYsJp-^qSq&;9x1E}L0grk3)3ufsgx)Pj7%D|gZ5eeg$Ut0D#CO9$(C22Sj<NA<)4poP0KtV8'
    ')dRlPq6TMyY+><o$oO^vs5Yf3F(WFPpL%97Wb80E=6y$<2<)%=bq1hEtgCixbVP7XhbxI&c*G7zOFI*-Zfa;e8VG{r8drl)({<ds9KZ->K4xHF7|=#Z;|PAo'
    '8fKHt%(->(RgD(*)v7Y-VNOD;%=VI+dfMC|)yye9r}D{ut|o}+$c%KfK)X-Ow}OX@e9UO!cZ6EnFv~VHaP)$~aN2@qROFQjX{AqIS>SlcCnuzSO+!W+tmeqj'
    ')$a3?CR0;5BIc`2nXt?<P<3vM9CS<F^Bg4E-#I@Wkyk79v2>V@9;bO_78;J?!ntueRZhodq_Jh_ChcD(?X&c7Lf+Zb1iU<3LDwrSOoK#p#KkRqW>pmS3<4JV'
    'Xv!q(0)V!gnXgO$$C6he1Sf&KQzM_&85&9%XeaeM79tq3F~f|^=P8;lOA`%gh~Y|G0lib_X54PoY5n)4&u}?uZzZ)Y(!>g|D4!VTdo2d7a5Uc@)svy|^SuL|'
    'e@WY_Ty;!VkL0>7X}*P=tUwJ?ckuS8!OB%IhtqJ?=hhjOlDdnm!Hnr4?H%WPO{A@t_H4my1$=4ijC|Sw)73zx{3zwtUW3G+n_6|aT=!CSXu&d_Iy2A)D$?c('
    'vU04v?gvqlT0~<Jqv0$rN}X}LSkXG_`iqcEbH(uGE&VykpthVP6gy;SzS1B$nMZacQ;d%<yj<&ZEom`Y^aeF37JJoWvdgU&t4fP2t#bUls~apTUvvBUo0zB_'
    't}riCupO&^$Au2f^Z(0A$6r=D{<6~Xmz9pctaSWkrQ<Iv9e-Kr_}^@$L$hbm?X@ozq^@MJBiL?-+Bom<x5v?+wgPegjH?J{r_dxnj5`li!54^#{9uvz5`!!q'
    '#o*8BNWYIC{b%x@9Y|y`W1UE2)iFQCAx^gpl^)=qoeY0gH_F&5X0k)iAbRVaZ5^J2jJvH?W5@Z6`c~C`4vtN(@CBH7qQ}V+%FX^H1OES2M$V0oKK0mvVs_r#'
    'yc^z+J%aYvQ>Uf__CgN!ki}jx<80?QM{dH}eAs-tYXESrwk@CPpli+KrYE@F%FIna0#>^-*D@f^ZL7pLe%i`iAqnccQse*MRN(m!BR-z($<k|+tN3k14NkPz'
    ')UUEhSMzJFfQ#|G!TxBr2K3GZSVB&`#aRoVF;8zy6QC!NPHz#Rxs&TWh%QSl^<TGRRR>~VZRf%c{kIj?51ken=dc1IJ0+{$)o4BQTNLw``JB(QUVnJq8ur7z'
    'PIiZ~%Qz_P5%LTm7+-YLZ97RE3=6L7kG{NvF&zMt2gVw8Oo&EG!(16?nMd(KO>(bCvYE_l!$;uFS<H8>p4+Hrm{PWiR%70wrq&|ry==3FaR6>AW6_D`1Hqc|'
    'N~1(EoRuF^%hh~+=DNDgk@}ISMkoBOBCW?+@N0}P4o&@qF=pn1TD&NWCFB8sdT^`B%EYTYRZ#4%JuHWdOBzuAuGD-Yjvaer>s|b`+mmZ|p<TX$#;p|X#w)|('
    'HXWRPnHoZHLyJbt)EicbT9&qOu~E3#@ldvxkr2-&mU*Dj4|UM+!gwmi<NkShSPGP8UixxRpBjTx5@A6X`JrUYhc`hMywHZ~xflzJAufcoKSo=m2nJaXvG_oc'
    '4o+$L939`|FL-%ev5smnE+!BbczP%o6lKvNH*?%(AbF;;7}lAirg((nEWpJA*8!4-A?<6DkC$`|`E5|R`(%MH<N*DdCE@s5#q29&$`?5pQ}9T=8}BPWD4Ul#'
    '_goyXN+Ki4YmDE-?9(ZO^}&aY-Y&$2Nfir^@RdhMPbyP)$Z_ZSUO8#*)q7-XAuHGR#*5K+Ov*XF{{@DCGa;9)Pa-D6ftr}QS&ck*o1!j?yui98c)~Qt93BJg'
    '!;^ku7Yl<SAP3NeDHn|BwziJJp|OA%`xXZdE8O|bqkH!s`5xW>;^F<f-#SO&K=)Vo&ktJ3@q*m2L55C9RomO6X3}1*ln*KuT~gDM^ty?(>`HU1PKaq-$-|qh'
    'x(L^x3yf$L!}HB~YWI>=fO%f87o^!!haPL6-mm8ldJK<}^3hNm_xj^3O)GRM31Z|^w@KK>i<5<gA(6+-YEv1{J@YtFk5J@~@;RMj-A^HFwVbG=zLqp>obMfw'
    'kv`nmtf`P!_Q=`+S=~F|YoR?`%E0v7qXwY8GT26k4`@Tz`Tja-Z>3E~v~>o@WTa2&+d4lTk$MK;hcY-|UYaP@Q7Kh_;ppfXUQyU%02S|~T0jj8FSdYD3#dso'
    ')VM3H8kztutvm}$a;T)IBUR-zP8>@CV}hDsYmw?KD81AHC8kL27?m=NO4DJ)>zENLwIIa58~CVU)24`=0%V_X!dGSujWh*nY2+}oy;kd`QOiUo{RRq^9(h4q'
    'wpmja<WdLdrvB-mcu!JtA4j#s?IxY5Bj+)C=P{xDp$5|QU4L0@kGoV!rl+4~*^RA5OrUjiQ)yLZd@a@|wHjl`XQ)eUOCPQTNQ#iL9%~;~K-$V0z$@BZHUlsg'
    'E>buZK@?vI0u4LO#x^3`%qFzUgr>*Rq&t0#n*;Z8ebz-9TI7jlX|P(}-X(KQ!XE@O(oV<g%zTeGEm>-i>1wIH7Ak?Dk+vquQP2DFvI$~+pMk^QTX49T)h*pX'
    'bOx)H*Zt&R_MPVdz-j9xb%r?Q{d1lHD(~d!1bTYK31#Ur$Cs=eTkutlyN!N%YfDuDyR+wejq+S4rwgbC!ExTXEvH*7{G<jpMtQ<<=h|*N4s>XX_O9vU0hq9$'
    '_VV++MUFZju=C><>39YDs&;Iok9P^!ook;Iw4CNJl(x<VD=m=;KeWW0>-hb_j-Q#MZOFkV&L7UmIf)MoVixm2r>>#?gW;$jOoCvz$VvC-hy3B^K6XW6?hWrv'
    '?S=ESb-3ZeY#;!Ka{VtY97qGlO4U5PXDEXOAa&1!sb(N^et1Y4C$zI43{+bs?OA(!G-zAgyCvw`#)5p(MQUa=Q?DKzb2>jkj@m3F(`Gr_Vij-}GV~bmdNNum'
    ';UMjk_6`;8kv{gx1fkRd0q2DY0$tl8`<s_QFwylE4|f+hRTw>h7x<Y;0C(dbvmI)(0k|b3l6YYUuSuE@$k-T)lgVMQcX{{GwwkSkukeNqm-@$;c>Hhqw3`f#'
    'T8L{<l}k_EUPB4MD4aUnf3ej5VyXR?wA8p+V~mQ<dKoqr!O6g;u)0!bt(ajJ4DQq#xcqj*!w&*$uRYerwQh4j?5ZG!k<)z*)`}jZDTB{=${xJ9D~012d*MBc'
    'eC#Y&<7qH$5&EDKp_m1=6mJ>7Y208~l=?LZ?33k&1d~Lx54jzCZSYO4T)dBw7J^n+UumfyB8LakVHX`8*7c9(1GkdaF?flJzHUpGy!;Xi2kTkytII4ZwSje*'
    '=oYK@KhfOREL2^Afa+jC)7YqXV!yk!n#L%tUxN|ghLyF1c?{me;=EX3Tx1Jg==PZ}InnI3Yo2Rb0?cc-u4(t(#*HXv(FjxymKYB_9f&oA*%lbBL-;-^#uEle'
    'f2@ezmFpZ8ohF=`hcSO2DAD7h;kWu4lR$)m`P!Fn*uB8KLRl0~<*6U;1q%`$mfcMK%&slgzVe&D^_;db56i*8E?f%X<`=uXK-*=(1d1^uF#%p$4_xz6&Xcev'
    '5aEpr|AL_nN4*Yj`4N~BKi+liGk?JC1AcF4GT`?H<32qhmwq*g>AM4f^d)@$P{`Uorq~9-6UU<Aus0xL{?A~T-AN5S;z#@(i?TF}m<~hRW;d<ngSG8~VUF|E'
    'L2TEPlhJe~!VABs5f^{z+jF)@UlCqR^&*%sm#|+BVT=@O)@!7qG;RRV2NufP$J(?-%&O+9h?%diRk6)A;ca}@`;}|zp|!S-ey**%refw^=34ul1{jhjH<e%x'
    'H<%!)axG7`TPt)5`HIV;j7+vWH`r0K;vlFv4kDb3X;E&N8Vf|!+oN*r2QLZ=i+q^Kg&$qPE3#}KQ5mKEhPxUs{2@GSaf6-6dTJ_xatX7Y`{KpH!bI*(OLeq8'
    'v;y(mkL{45zeqCR_}m-bG#}Zv&Xa3Jp-4VnoaeETO-qJp)3~^Y#hFpd2&?U0;4y4t1KA{3TvkwS8+`Rhi?xu&!-=wdT~Ow_yt+$P_nC39ERAo`7FA8jr$gwO'
    't_;E>I&u3u=-bvE@G|^8)Ou51d0lNinLu=r6BxGojN#Xym^$NhBEtM1j(#r3&~tIgB;KupXVX*9?>PMj{QfS*V`5=QWOrn!xtntZHcWbQ-An4T8KCmFkda}V'
    'H9A|ERfNk#MGtRHPr>W44zT*!#xv@@_juF-vpD;F>}D084+X8ycm0;(zkyzj@gA|?UM-LDH|&cmU}P}Py{_iEFw(d`X5X&6&A`~@MOt!XlQ6h$q&mKCo{TtI'
    ';)$49DDWfkx!2QWLpIkh7DLh8>))uEf`jRDBwJH}Ue~AOmGIU_It6j-;dl-4Pg3yv`r-`JdKkbh;S={eBBwLPRjI0{Z%!qnk<1xriGWQRf3!;PhMl2UuZ|vP'
    'v-E>}Q95WnbC(rQoFmLiXXx_G3;0eu{$v;+VBa!GU<TR}Sp3=V31TLuS2Hj3flt9?_`XyO=ise;MIp?6=}#siFdTqk%#5k&bt<y10yiT3k#Hvj78$1I!IP~I'
    '{rMcbl!8BZ&pyDC-@;_%iQ|g>m`+n<xA^j~v-~pF;=RjS#Pq+2U*?UEGjncuRmi{5l1nq_rGCtW%rfq1>5dFsJVr`t7PY0OHWGMF!w<}vY!iZ0)+r=lW+5XW'
    'W)X^K6^4>8;dLZG3t1@Z;e)&PzkYn*Ywon(O)YJn4Lv>2ql%4J=LNj3ik*}db*Dprl_o`r7z{_Edg+*m+wWn+ABwQ7()^<q)fLv7i~9>jeo!v%wUlJmGS99W'
    '=MVA11XnSCYeI-0>czBL_c<$q4)K?0D=R>=NQmXEX3N!@ZR5kRgRmB9q~$~?4Rf>5^s=DX^Z@$NXL$jBYYCH|ct>I$ynbq|T0zS`7@~sb<~CkAHEcebrmlyp'
    'AJ9^?f6z}Zlm_F)u9vq|in@i&&$Va>izEF~$WYxZn9U`TV8Hd#avb`0A`!lWc)C&|M=gh<)l-u;(!Enim8mkb()6b%VTbdexOg7+ei@4-qj>t0t@#z<Q#|)('
    '4qvX*>gSqZc?2|}`MvLcc5ujg6jX2Hd)~m^Nccrgz-578d9WPpvUlnE&jIAc&jAE%*6)14q3;RH7;*h-VRguhed&3OZ+n<evAT+VCd|IqK7DU}|6F_j-1`27'
    '_WlKbZ`%3sm@;*IZ_d)8wWEt2*Y{xLU&|K?gkmTSDCY~=f(w4Tg@Py7+;|Q)5iCkZl5v*}B?H9s_W|+30^-H{fMCRg@iQ*S^YXwZUm%k3!5HJsg2r|H*j2nU'
    ')+vs{z@*s6QPJDy=rGK1rL{O0>6(i>3AsSz{M>wZ`be^u7juIybw0GRB@bdU5P#{mNTEk!JS-;OUk-Ux;rwS1O<Fl~1J*LQx^MXk>J~3b=4S=XUu><~Kw3*W'
    'h{WulT_#$Zzysph>E(wRJ*xn@sRfVbVmca3039w$l~CA&oOkJgwZSWQA;;1ME0CYi2N>iOR{2G4>OeXLgM=|W1LHjkE)vk`k;`^M?#4y4)Xe?3z3|fdXt0#a'
    'ya~iUF~cI|t<E5>VvDq~K+AboXgM}$?@8pBIua&AtA>t)WGXg`o_o68_h+?$J`lnYuw=_hO2CH<7HFf$Z~I2G7s=t_mqP?1;Di7>fWsJiA;uke{5`b1!QeSp'
    '<RM+s-!gBI;T8f*uj3JJ7>BpDQLySRDD7f<qye4-4)ejNHy@zlVFBQ|H*6sEPO<?^y))Pty5C0s9R}{_p}9?{yc5k+>1xg$_0k$nlgm6#mDpjh0l;gtT*d!x'
    'BvzfOODJrMm^6d5_+OSg9c&uhw5na4`o`{P*%r@j-j?a>E5~NuZSYZKnfGnEsNv13-DCSZn6`mq>UJ)y;;5brGG5q98(^75sXe^A&GLca)dJ@^@HBKeJzJ}t'
    'e<zmKws)~}X^>}aFe<SWAs#$v%g%D8Bc7##@n9>3J>`AjjEXy`=AmSi=KWPWX5G>c`a<Fho?Vig%}%478s7jwFKSgy9}~W;j0w8}jC3=)nWrN5)Tsh@_*jjL'
    'i(?Oc&V5sP(Cu)K7yF`}`Y_w*CO*m4P`4T|xjKSwcN)OFyLvfmDf@8R35MUj8u{734&PPE<3A}^x@qIBnHTo?oGD~GP=tXsmWlwl$$rp!<<Mc%oX~$_sFs7V'
    '$T8<M$KSTQ0!o5`Kq&Ptnm{^YY2P`qd}!dm*GO8rr2}Q@o)D(7j&xCR{w?Px;b0~P(8$5dFxEk?=4cpgUfIb=S!yw0U^32EKfgiX=x(PI@cam-S<Hi`fBG{B'
    'JYJE{yLXi)y5)~J>vD?oM4k%tsjE1imzn+4VG0FV<_VVr1Y|<I@Z!cT=S|Tx4P!M^yADknr&8@--&=qfARRC=1dB$7OC~?|jF)t9uuPNQ*#k2saQg4+jI|W7'
    '%{z+=$QFCv(F;teqgtX?KTZ5emT0O`X<E+zc}&d<OxpBz@PBFhl$9&wjti8@!5eE`=_ZqHDilOYqRE74vXnfAS%KI0>3Y_Wn@Od({TMSl({__fGoaG7|MU_I'
    'm+AA%dSL1KD<50O=4P-(XJ`!=zihpK#jf8{dvFEA-VEGORqtrn|Hb&Q5F;5)<5?x67n%EZ-3vcrah_B#of7^tyB$BH7s2u<{^#-}tfOJ4Kr(uf7Xe?d=tDT!'
    '9#C855uD@tYSlZs2^o%F0>BQ2G2bq(#U-O|WJpZc4D?0OfXRSxsSU;)PyMhk`v3f<GavS!XfnUL%w$d@JFO-OSj?}kV0`^~46Vzg%+DY~u4>4qo6tJ$$!L00'
    '{t_$RT=mK}yuDKT<}RP0jzG03n=%sr5k_y&UxX{ZU;2~rc+eld(n_~>iUn^4R!_%+9Z{SNKlcTKB|L63*E*_~+L*c&<3Ca?P!%{+BkJ$>o}v6+uTZ4ug+Hbz'
    'TV`NguRM6W4FJ!zw(GD@r&7TtIuijMqWCiEy5ap8<STE=ZI6!OSJwhExS0(VwSzPmCK2Wrqu|$zgQ!ekEXQt)14~yL(<ndHlzPzVokmeTtp_B~+D?fGQUyjX'
    'Z_s4w+YTI?^|o5pym)L26)M)sfONNiVg`rtotyHuz<7&D(#l%j+N!)tLGP6lc6JH{M2$k4zxf|dA+8j)&#H?2Pff&_h_@o8$gb2gSmuo@eo_|OFu%1|Fdd0A'
    'RWQ9ZOC8ScS6fMYW!(8!<T6hdepDr;m<4t4`CehjvK_@#>Hf-$A7{3!2}Nx96<(H|()fp|?WQzq=53Q*^S;M>OyJD4GEVideB?1uG(L866B-EPB?~3NI9@=n'
    'F0O!C_Y`c`FbHC?ARz`!uUYG+WaFyUpw?%00`u|lr2)WhKQIhjg-@=18A%8~m;wAKsK3C}EQ{ZI0ysZOJh=w6d~!{D@O~Ew8a3Q@u)H9zk_rE22*3Q5UjLf='
    'A}mZ8|7hy}rMQ^t>9>A4Q0C7|0lmq0q5IF%V?6;e@|0h6fym+^+{L$8%5!8IlY-Q;u4AME$_ZR3u&dAX;yvf%Pc9JXH$AD_O=?itr>UYnJLgkpp?))sAkWKY'
    'hR#Ru|I>@bcusneoo(-c<($9`@6AuLv+X5JLIg@=XMg-FdnDNh!b2cG&+;OgLjyEwTgmLw$!|_`Zj2lrmcv`FO_QA=63_xB<An*+K)cOXE;GI~OAz$aMK7<0'
    '7s;_$CRaSbUC4rj77zs%Sg<H;_fXRdEOQ&yOF#VxP3s{GRMx{ISS-cAcQ}U@(7_tlj?&-32XM9vgU}Q9pcP|*9=@>uMezS)@9n?iIF3Wne`NxlcQ8W@7Qj-H'
    'Z9FYcD1KNxCMil}tmJ_>gTd|q7-4s3xwDH8D?G<}i9eD!H-7oFbA6M0a}&qM=Uv;$%gb|QEB<FhO8zPT!mFzOnEsrZ1we|jp*Y0OOm}s4b#--hRdw}D6n-0D'
    'iNf@7hn*v(-#3ftHCWHn;q5yD20JC91G!J{zD$?J2w3Vn0?F`A5=zU9{whY<Wd&Vv9*4b?PYRCuJA%6`4*4ycD9ge&e$G!E+{5vfYiQ|T?MQkb2G@UOwjho?'
    '``|e#Zl5dXPos1;85Xl)z5}h|FS>XdBO%l#J6f!ecEPrG&>Zc7aqs{mFXG@Wes@faCz}e1pg69ZP5E(UsGn#D((g?%nL3_{;$v!8>=S|K?AV!Ao1ds+qo9%t'
    '`ym5{JG;Z|xC(q0_50Hdr#Z8ojMDNzImFVzfAr?KwCA--b|ed{368;pue(EP1~V}fWllKfgEGmE0aF%p)uOZ#p%p86D)QYEtQ^(Mzq@IEm|ep;Eeg2_f9Li+'
    'Ance9p20Bk)5AALJK11>=eqB4bl4jbV%DpKPTHyaQf2Oi=HM*ma68OPJkXCPzsUk&eVLC3=9l4clrV1T;d&73ht54;K>WJDAn`$<vjh_?iKmfdpO;{E#;(yE'
    '7Qn@p)rXyRf|r@~KBpbP79)_rl8MK!v*A3MoDlyevvj;KPh#u$7-h&U*70-WjJSTp{L-a^BRnkF!a*pCS_e(!kbtzU9^9p1c&bu!%UE0u6)x2B!GblbN>vYL'
    'DLI5#N<Nq^n!||=o9`TVEGdJ`FIrOGIPi6kJICMs(YA6|T!AJr_1SyB6Y+$0!xOzRlk96eg2pWEg*BH$LqH40VI)pi$(}mzm84(%AT1sr#<0<4TJe^h7lg6x'
    '<1iiX%_%oO&zXTFV?SAg8){3nOw7th`W2&`5!6PA3)B}z>HQq!p}eV!K~=a{{cgJPhyqya;nJM~_9yw+QGb}802hqNfamb4?`J6`|FA3KSGl0Ea+(#B7pvx}'
    '>tumzQb@$SbHmR~yspVqzIL@pIDxR7z-pk}rA5%Pr!BASoxLVvd=uov>MM>&gz6j};U&697HtoqFI}QgFGs5$LMIttoY5$6tk<E*tC0f;^q~t3#GGfhbByi|'
    'ok+()8*l)sdr12=1DsMXc2gy)Va1|k9R;|S!V$WLw-#G#BEVAQng~+T2tloj0<D~Dwp5E#r<OHqR|o}G(?X#Ebjm0NBd3hAmm9oeM$rmG7kr!8LNx2kR^+2y'
    '^YHq+-*Tjl70@-@e$^ZBYNwYs-{EuGMwG3Btr;*aM`$?8N=obA5lc8ngftQeDggY80?cHD(-FsP*upHPdgqAWq_q|;RBb0uOIFFY8+$hZC8kuvlrzI@yoV+_'
    '+F5SWX-)NJJ=MT9!R%cFmV*f>xxowKwMjg4fT!$ukR50B<=H}n*g(14j_5uPP#wxzfMb;417!`zouq)R^$-T!*qcz!Kob|c7zGWWC=<odIBpkviQM^&DOiFB'
    '9L0eaGk^n8lx0OKvE}EezEKoIFS?vG!*mDqL86OMyCRUkHl+%qOCGK2t*d&`Z;OCFx{Cm*Z~)F8x_WVIJoDDph6v^@&%&dftQ!!Ck((AF5SbNn>>j8%Vp+ww'
    'LKQ(`9<ueY35e{rRX5OLEk7l(1jRAu+nL8rrbb(=_NY89%abQTCH!I+rdpDZ{l!d7;#0@WlxfJl^_A-sr!nLCQdj}X8N<@}1yHOGi0E<)K&&Y4Zamm&lJhEg'
    '>98+%5iqO^RqwL2zt2k}IF679s!x2X4Rir}mLXi=9~|vx(=1r)tWiuwS>>xBhoBwfvyzW@dC`1tTueQr*IAuY35h2@lz3qgzupS72~jdPJmt?)?#Xfi%)Mzg'
    'a6~Ge1SzKfp{W(f+isoMsG^eNQcZM@1pCg#%8aF=>Ug2){WH0OvRjD`h=FA6(GtEf%ROv|fZB4ML*Ba3Hn?w5+9*q!2WKkbopjc+icl!>N`PSui+3IKi1j|+'
    '^jf;cURwd!qu>`Z1%LVKjX-4Fa-`bzE(6t+3Nh73#LiomJfSr46mN#r(5dMooMVYtkgpgF4^!Dohdn?<A!W#0u$8^0tXo0`o{pqpt?UGA0VV84XB^&K9rTBp'
    'kOc;Gd(CO)2dE<$!neq%qCFG4t;#f^n!lL4phKm&3<-_pMN$5(CIr<ey2jBcCh1Ptg>_^$4~@mnvBitLjchuWtSw*5MxF@w@*#4s7%Q@|o>h(?x;xSI28GIk'
    '7b~iItmxyhhdhJqFz?&u78GFomLUbWGa6q<t9s|SAy)HXy@3JJZ}xP{V;sDl+`e(Q$1sS4FMU~ibR2Lc{CjnsgQjl3b?r;Z{agRX4GL`t5@h9k$YQ>|m&Ec7'
    'jWS#^XrH<6YWCw+J@@dt7`tf`%?wIrZga8d$Z%0RvxUfBJ;G77Rx1{!x)#NBxM$7Ip02b|J#=OThYNyE-t(y42rPI>q~<=h+m1T<EE|PU!)h(;IJyDN!Dm88'
    '!drk7%Gy!(EKSA5O}voXPSW$JC$Z~Xgy>qY`sum0n0@V4KjT`Ctfxw4Ommlud^!uald05|rnAt~@wS+GKD=x<?*wQ{UrT-4^0wicRWE5b58(wlbmM_5txuEn'
    'YT~uF*Qg-MtsLs%2KF%)8V@K0U&$w-renpmYLn~KgKUxg*xZPN4LmkAfxk!icMIczv3lt63s3?(D0&3{Mm4~Be0n}c!SZmazdt#8v~p$tX!59!X>(Y)NVbF*'
    'XHKwjw>Kj4PD`$S@l#nGY@M>l^d1?KpUy(<yAXNCKCi%QgL`)HyaRSrud7KYBA2CEpt-)w?RCj6xC&AmF938E=XU|NYg-N~z+18EAOoSgvZVmjX#KLk3e+Id'
    ')^?dqTdm{L4b?<Fj)+s9na-Zjzbj~<i|X0Zp+t7?MR#6yBG*pZnfnSw+A*N2<RVjkLK(i1i0?nPg&KDgZ#Y}WEb&@vhIY%}S_hYWqS+HJ+0<&Unz1gOJNCA>'
    '<(|_jSvvE{025IyfQ|1%?1h2p^PMoP;j~wK*P=B{%gQZ=p`z~1R9MSWJ>xm{n$3FZj-pPp?dgwkqphmvcxTkQbV(g9{oICTE#*egQ>@zz2i*Klri!@X{-e;M'
    '_NKW{HWPlZ%hyrQr`2P{DI8x4?!NWl#zt_M%zA_4WVSk(^^TMEmE&X$IJ#h$DF!ki5ck{;iMgMa3G2MI3S3d|hN_4ZTKyF_(r<ELD9`EKuF)$2ImwPEun?-+'
    'TqWQI3@Pc2xi`pX_LA(Nj|7D1*)3nPhRq65m)B;<5hYIQ^S4whr!7$#I?##*Z5tFVEqR!e3k@q<jTGd4>YJRX+en6Nd+cD$em!3jdN=WoJ}3iSm==?h&}cKZ'
    'Y>ROK@8?7)%*uzoTXH3*q;PehSC4200tPqR+Bo#ICkaD2^W+^Magb0S^R{E8kePx)DSA|}hN^!fRQ)<OV-v1_INGNK^}*|Ori9~Ss@}i>{L5y(<yiJ$=%$n$'
    '&i>?vPria$kPKD*-ZD7@DKyg2m;<wH5AUbo#xbacbypa|ZR*o8?a^LBcMVos8RN3S1$HokoOn4S-p`^VxZ%H5hVD_}Q_5tbO)=;ja`ay|AaD608~h~pBk$MW'
    'aI?(Gvb-S))KGzl0>v2_n@THFhc`i8mAkHe+Q+S|M}9buYra*4GRMGQXJ?wFMuG~5sm#T>mZULgt_F5s7KwC<Ba62n3AsV(C=L#pWsBveC?+U0SSjt&H0}N_'
    'J$tNXhsJ%hk(5L&VFcWmMX3yHhMUpqfK}?Zr_@1q!c1F^qI%{ZU4TJn8Bws;NE2Bvwwl7(%rBQx2+Dse-Idk~L0mWWBlp^8^#oUKoKY;v16F51V3JmVUDrDz'
    '7gZ%JZKfJVIr!ovD8oGRR@l^dWOI23c5vcxd20((f{MZ~TZJ>du%nTREu?h=q`&lkM77>`K`w{8q>e>xDyyFzgZy%!BNa8e^J@mO78K`EL5pfgt%~-A^vGyh'
    '#kwxISx76Ra#0lr!!{?l!!{f@OU8IUToN%^%%2t--(kN>7d=TMx+&-%E*ZmZK{USW5QoKhuO6hz6Sc6lE!%>P2)jWo?qYM_8nh)jeDJM;Z5ol4O7}==Lat^M'
    'A3HQ>lapdwE{j^f$^AJ+vNuf!RzPFY$Fo-nGinea{UMHcX>LV}azi=1)g06)qs8Lp<*zml(U<|}8Z_w}@7hYnK>IrE(+tkV!F>)UJhS3MIsl4n`)~;lL@jBI'
    '$Tnsklm&0eo9*0&rAPB@dSaQEbfOs3id7*SlPL*nns8Z=Y?)L8z_7w;^2V&D&5@q*9Jf<pdS1b!*Ed+NXFSJddN-QnCW^+XMB<bZ<N_Wdc73cfuzAS9u6u#v'
    '9pi&Tws}gZx;olznI_+$uCqdMbwIalR@ZGKw5>!t+}{odvB&ReY0PoGDnBryFHE$&)lJ;=6_~&DL5t6}b4d8s;p{sIevxgP4TKo(7jpk#s@+`4G?aZZl_bS3'
    '<L~9I7Fsd~L=<-yH%dAR3iJ*0PolgxBm^1Um@0!iB00fA^`2yPz8F)W;mC=@zoIrtS1gC?QS5lzjbf*1$~q3^DyE&}0Wm5MLN}cCsSUcQgaTQU*va@%#XNT5'
    '*AO5f$Fq79*~wGTlq1_#>XKP!b8V|t2(8*ujkD7di;~GTM51<Kwfa0v*hJ(UX>={B=X?Kzt_LHyUoYjEB4(k}%>E2-*?!^YzGFENB;GVdGy3g~;RmvJJp)$D'
    'HdL?uPQOj%rtSXv!u$7UyxTtKjj%OgEx@iozm^t<O4w9Tl^_~bQ{u6MHGte^1_PX7VrT@ai5nkHt%3`TSsUZXDeWS@0W)>KwxiWRg$wJ-*{v(UNzcf*a-PNy'
    '^)QZlD6G{n#dhFZ+j<;_Vj8VUt%2(^r3K_)X$#Wn4@QI-x2Y2|dOqhOo^9Ov!ZGWok5@lktoo_q)XyVEy-j?2Q*3(c-Y*c7e#R*D#Us$INK98OBNgt@8ibkI'
    '&10hUVdG4(@W_%*u+4LBp=kJDAuF>dh*7aG8;aC|=H*jbmVZ5r%4R0G7sRTZX6Iy8j%5HMtSlStvKPeaNzhtzadX;%rr?lYF^jSsIBraH_5Z@^{MocRpTXvA'
    'a)m%ded(x!Bb?8kkDc0Mrnb_}EpDl<O&u3`kDxn^z6Mc@+Hc0eyBHs;^^c6p>l=o386!g9#EawHz0?aILE(4X7Os=5q8}^}GBRg~WhBRwZTsyG>0g+rFHF=g'
    'l!=<nLY(mwc8L2-HW}HX29LOES)y@E-^5l>5D4yJv1X*fphXjxze`G+k$UScKYS{~=Gl%|O{Rd6auzNiD=+CtMCRJ8OH(~U+=jQ|%sax++5}j5s?{aEw%qT#'
    'lt@bbu(*#v!2q{D+6#w0)i2vv?HaApXZ<#0z1yU26lyZ5%TqU&HFi^c+KTEhYG;qPj=(MILwVudG8f*13-5vLJxH^f^z*4mLcO?0L6JQ*7XP}0OLVt_b6dKo'
    'F<!}2)Zlxt^Ch))g*+R#$*1dfXz4*{<Ds~4In=ovknfl&7nAZF&q6u#I&>>eHp`}E7)8wfYSg2ALX8!vlwXa~p3(Bla~M@E2$3amjXZfb`TYZYu=r6H75rH$'
    'DpgzLIWhAjxNI`#p5x@1<u`dgxlP*TH8CmMI-5x)*mQ@CFByg@?<22tKW&<hb4sRRb?l|!$)A4b={LU(U|4dWkN@TmKK|LiKKbDLF3%39_h}qD5_28Y8QeF|'
    'C%uQ|wXx6gDt;WiWMwIKm8sM9TV}YZRlXZ@jXc)xIDYcIAAItQ@BZ=|-+J<sfBxhb|Mlq){>PKQ{o|+q<p&>s`|BTl@bwEn)S3KHi}{hx>b3gS@<Xw?+r|&Y'
    '3@`0ZA)nIoW7`7!+i}T5`7EIfBn$Y0oNbtHr)T70Vl%<(W<nJtL~D1nfL}-q&#$F(s3Pb&8-A6Ma#?@R#f^g_{h4*p7=D+SyeT!Vm$<#Fg&D2M`YAjnlwQL#'
    '!cqnZY;8g*sq#V-dhERC=u+J+MNwN?*0^#i28%UjvwD9<w~il>?z5@N)lCGu?DO>Sd|I>pl)fH?gBSGn7&U*jj|kWk+371#zBfkRKMUW(b8#P>x7GYv82Hc2'
    'wEfvM@3%3?*X!I#e|N9F-tADd>tdqfW`)$&WiJ%eXSH&CntIU-tRKAa3-w~_EllX-846qkf4)LjL0B`+oP+pPK)(ng*yjM{<#{AW-Jz*x?$=Wkn>IelMcsg>'
    '@?$=`!%*#zXyaEtM~|fYIzo^Qinll?vwHUx@BQK%{L&kcOWH-_I3uRnxR{QTQ9iCpt@%okbw4O)Q@SacU5BNayRj@?ya}CAE@5NW(GmEVwL*bJn`?2fzNKBA'
    'hX9aGb&$lkWj`HebzNa;Z5+a5vCHI4P^Z5dPl05xjf7J1PhEYBp%%D~(Cm(b)fVX!8=`!)XAJsiuX%tdwdT2uj{@mz!@~fbdUUY*SqDey%NfUphFY7V=w2TV'
    'vucv6O6_r?xZ_ld^-IB%pZxRF@BH|qpMLeDpZ@U4cYpZgfBwXlEmd6OFdvryCZl77u`ZrsanBI6W?z}4!7{IEu<vMaU(eujmk!8T7btfrtgLI<vGoY;mtbQv'
    'DW<bvGT8aw@t;na%@WM7oP`bOGSKS6&*L;_xYw9H8jQj!LbE~@)2PnUVUIE<%C}MNZC0Y3B)kkUM3oT<EHazaQ=L0KXTcRsU_(91mOKky@97`bU3rsr!JGs='
    '<Bhw96cnO@e9<c1t=%YynHDWsr4AG6(@caH6EA~v*l2-<NJoc3Tpo22KWc79vb3Gv0$yK_meuoFjJyzpy4W)<Z4EVACxJWt<ps7XR_~FumBM;|@#22j#r?kL'
    ';mx%*v!4D~U!Jxn#<Qb_OVx5uvYs*L;+hJsk+1vM0+#Nhul-sF%^Z2(hkA^3UhsWr(JPxf8cTQwQB%g860oPs!`NTkH-MgR&5-Qj^L?y=<r+N4`&w-5p>w=%'
    '?vH!m>I(R{LCjOf1hu@FOjW~wS8(ORs3~P-hs7WCoy~j;7~(J+7X5s7QhzH-1${m?<FiPQz?(2b!?+J^cOz(;8D}HQ(qVvVGIaQsUBk`+=y;6}V%dFW)`nBX'
    'aX(Xw%%>TYy^vjFfs{UrSwGI-YH@yMhjXzNoSOloX=nQE4Hqwz&EY&O1?OeFIMdLagVEwsF*sbDqJaO>vdpG4xpZ~r;~XG{k=(oET|MicdMLUm6q)uSE$|#('
    'Efj991znQ{vq&Vl)Pl}CY8(My1dDrv#V<m@KdT|&t|;%b$ABxgGjZ>u3<7rywpA1U^-o9RHfdcC3$EZ2Z%jH_<V<{eT3%g*U+Z~xK|{&8*hkJ_(l~=f<y?&*'
    '3$T7O``1nZM@X$i*fA#zzHFQv4tVP0LQ_WS#@-o&C4L1aOnbxBc^ITlZ$>)33F|x!PoMf+L3}+^rDDj|Go^y2Je}v#!b>TJ1k}#?Rb@OPAM`1kE`SGvZA@Ye'
    '3e=mu6oiLl*Heo*u9GATSZ9SnZ92lLEIb&G&mLsi1b^efuNnvvF&z$j-5MIhz88cNvDNA~Z{p=(JzM#F6m|N;bTkPat_(aW(gV6jeiAZ;fkV6$czlbk<esjE'
    's2udNQJaklYkx&8=|I%us?_uyho{9;?fg0C?8HjiQ!6<mxe|4_O*1$%2H{oU#LRd(biWvZ#DHyaahOembT~x*1wo3pxyHe4e_m3=CTx|5CqXfWf6Dm;UHxUS'
    'Q=OfG9d~e)&-R0HdYI3=1heFTFeZBX<DIineR;>^bQ02sR-I%zA6e3XZEH9+%rH&yKKQ%E>=p(b<F&cj;Kp=XOvBxWOYa-~ek?8ur_r5Ep)6gzy_nQGv%uoa'
    'Yl|sIKY@{p;M&bETANJQ*;#<<irr`D6S;k?K61b5kgN)7fmnSDj;*J)yj~Z>X4-`Ae+tq14ib3|km<MEMlH=1z8rT89U55R{FT}f?};9bA6lPou{-B$z%Kcz'
    'o$>ih#DkyX1*OmAiU!X_Y2$meS)sNTogZem*kkas@?}sY?0m{buu~L6m;&a;WtNR5NYDAy>o&qepkfY-=xW9Bp-LJajkjjaLfsp&xupv}^+q&%Ldv2?`#CID'
    ')W+*-r*$s2Thzo!-MwR{^~CrAR~vh&6aeLbaq@$#86R!?UFGXx_g9!ZWqv9WFA%3<DHG}Ty1W+Cz0^j1=S_J$YSJj$5xJ!t6=?Qb<@E$u&X5k!Q>fEob@VP+'
    'EEdCTLWE%mG8hD<7*t#_8BwtRqCqRjRK*5^NMMD@+byPAN+kiJ%BEEB8+mMPwQC$JPUlMP{RwAwTr}FZYH#Z`tX$R4g)ChDrLv`c>y&n_#qC*Vuw$Lwe&xt?'
    'qnoch1s1VhwNHH0$Utgc$7ch=6eFsNy?j=B7<o8K_Oo;l`mAQwJgZho$0$vU9c)OyOsFh&XP66FMP^l9OT~uIu`SXUJbpLOd^pR;#VAjQA+^X(4KS?A-g?w2'
    '0sV~f$G3|McQlet_Y9mBEI%v4Se3Buee$MKJT?vnD4#a*0L069-b$=;%104NxW#*5w-lYriclj#pr-leW%H12-4@gT%>qLjmGfrbT}}BDHU(7v=wyV#Wjs(r'
    '*A-vVEKJP6)LXZSxiihu1KY!1Wy!R4I%v;IkI8owjNqU`_7B3%9AhIia8bp7HrOq5wSI|8XE+SBNpWpX#I7H=C;k1jW>-R_$-k0mrx&&BV$A&Q@@5P3i&_*!'
    '#%MBoK05++g6dgJNfx^km}-88yv)ZiI;de7SLF=y{>&5rNdAL3IK=VNiqp+$(8`9>Jjpk?Wg;#wjeNF0&0ukW66UZ<r5dRwD+gGw70CikR8rAb+fZZsKtAv0'
    '!(m;DxoR&PihI#^^F8z)CHyC@9z1Ysn4Z9B*!SPifzvIW5BY64^j^g&9Sk_vIj+XRk>NR`ojb9*^w(kum&rHsnYeh#v#2p!Mo&QUltS1&(THoK!W!vq(%&st'
    'iRw<7o_{Jjhq~vR(vF2gWp9&S=fvSA*g7VsZ8jSIxymq_bkg~(=mQI4&f57P#ay-7INcd$1Exl57M~pw80)XOaIyKAhTeNWs;P8?dTMqhve{I3ES-V{nJ_i^'
    'ubKCn`Hn;GZL%L*1^~G@^r~edsF5gMHZyW1FUVePS!)rtdKN6T2s@FjaB+SbWSGvn)}*Ylj^=)_7|rxK{925XeV&G=Uvyh>Prm`LRGwb-(@{<;Kt1`(b&|DG'
    'f_mz<&S8a7G~S+>K2P(0I5*sufby;u(s(1KLx=HaXwZ0bBVOBbRpgs&wr5*@{de<G0@(lVmE{PvY4tTRGJVjGa&rxv0eU!Iipk_O%*T7HJFr9!SH)v8pXI}{'
    'GdWq>Sb8bAR!mN&`QH942>VgcUF&v1>+ENsfCAUw8uvT2o4`$Ux(m8b4&5j_566y*uLf_NfWD5(V{m&mpo*(>6_Hi<{=MtJy>c5kd|YNLw?Gq`VJi6PMsV#~'
    '@W$_~+`SgOf`_1A!m<zcVQaIC*5&=Q40f_?96(Pfb$qaMLTz4eY7_Q4gyG8Ct1I2L4u|{}pr4P^A%lJd(5KljOG^Mehb`h%!zSiaa6b!n=fffFy$O51rosAl'
    'nC3&AfdMGx`?J}k+*n<Ojr{(62Xw&E>MkehG%K?dX65Sj{K4wZu-I80rTKXE_N{9-?%u!AnH|qMYR6?)3CG+*5%GbUk_mh|$Pzq3z$l&VE03UePEG>o<#;0B'
    'k$`wOrgL}nTO=14kHuS|S!H-NUw@V#(u^oY!K>FEbZ_3pW?l+bPWd0!?OqRH?~$$a3;Yv^U2&isOn51HC*$@{RA*lbfP@>TpfybdoB(ITnR&csNf)-L`55q4'
    '4mdCu1Z-F1U~ifaC~_kX`a>>rc#Il16SGCBx`UKz;3a(HiSQICK$v8QPbAgl&DK=n=oFP$l~Gpi;}ffN#L?c5aH-t~ng>V0UXkHMoR%}QGTYDNa~!0{d8zEt'
    'c_I8GuTR+Q#=*Kq82MAkM)e4ZhsFuAXF@m|)aEcS;zSUfgqTY0;7Xtd5vlD#hhV4HM|q8TFgD?t9mLsG0mOl9bt{T109y(P6tS!vE5ZM(;hma(afJD?9Qif)'
    '--0e0c-AVz$h<^xIou73T_wl}Pt~$(K=|;o;sA))4G~&zhY)7l>oTlheIhOi-@Np099&-&yoFGYP7ih6hAQDY28sF*vqr@btVb09eKhDYaCIEk+l~W#7Q^sx'
    'iT+O@#Y$|5C0$aT8L2?yl`>7T_(qVD4HrjAe566z$@n-8HUbLz3h%_c2LOHpA7z(RG-^15>W;F#R1*;<>2*2r+AEP-akZyFhT<I^>UsxWuCKv=moBk)NPZZ)'
    'SdJZZ63w4Qsa&PPhFR?&!;s&`vOJ0hD_2H?Wukq|S`&4MF=DI(OCjvs2J`+*;S(HFE}`-<S{fszT2-n~SP@5+cBv+=D>2fFgHbq7!0<5M*%r$O{PfxlegsZi'
    '8()B>u|EV(w6F}v<IWuzqhaQnLO0k!JPbxlr%L*wd<+xfFzbVQ0#uwJ;^P_9K{_@RI<3z#L?t}ROgSVO2fI^dO+Kl-q4cHKeO22Zm2=ovgi%Mrigd$qXEdBJ'
    '*VBChI79c4VGtO1Zi6b5PD5X#25vif%KU?7@DH|<-~O%8@t(J!!D$9+cE*Pw^w9T7#sZZ%82wQ;+b@(*ApLwL)(Tm~CPgJ;lKs3G^hjE$WLJk0|GPHLV4|UB'
    'I4_HQ0J^$NEF}R8Ho*Tj$i^MaP_nMCTn|`04$9ZNz(@nokx_)8ehC7Dd>%o@0yaBt10DPC!9H*uAbx_9d3Tx$U!&D9tpKQ#SK=Sq#ut2pXVs{vu@Qi5nN5pf'
    'Nfk!Gc%%8K8UO*w-Ek&hpx@&lyo+*2>AP54fPBu%Ce=xE=mla~P>dmfOXWAqjwc4oui%!WGc5MPs#}!yOpSV>4Qy`>Ef#|wHl<rDgoX->34T<VR=Gt@$rPp>'
    'SDx+>a(F=)`oXpstPn9E%s1(+;x<sii7HnhKTVE{0YZ#~qY~E;da>p3Am^lK;d*Nxvvs#o#bT%z?y}e;`HgNm14YQyV6~v==eCuLDY^X9+%~fWDWRUDwr6Rr'
    'NYMKXTk>=^n}$LpQa-okqYUi=%ci{v#vOGnaW9`zr^9Nvf|1aeMmSgJuT?k>vzqK8uLoSC)=+cgC2Uz2n1rl`O=T*`7H+n*-MM`aChM41C3{(OEJFF1X@}_X'
    'Ls_6aad8bog>n-tv<g)l@?;lP9kv_atUzv=lD@yrmMy+;FuWv6*22*5YQ=dmwY-n*#40pH3yhu?Ckj@sXfKp5Yt^LTxW681AEJf|8eJBv3SIVA;OwA%R@ven'
    '7qmCzbZWyJ+6^@F{uyl@UraS|!h`;~KNz)@Rxvkv^<<ROw#?(sJ>gsqP0vYnowx@vybVV0t?64qS(Jn3M=pJ4Jn<_tj4Vu|l%d9<`X?-{685Fw7JLRpj&@vV'
    'F#u_h0Xm9#VCMvt5wucOqSs?b@Y#k;D#vxB7Wt^&!8BR0NKH<XkT0CLUnL`7C`(0R39Ol@S_X~{i`n&NH`UTyxOlmr#VcOSffry7REO&>=E3>T1KZj1=qMX^'
    '6FQ>j^|ZW#`-<RgC<(ecYz2_g_~y;|-X1_gHw^Q@%;4*@^SNA|pgVS@`yzN;DqPO$Z`o%=M0>}o>cepOT+Zxo;jR+oM3RJ=^p40^ya>gq7P!R|Nt|IX1T#%I'
    'TTUjpQ{dm|zIK`9Dh?<$z2eYm;k;p?aN`bLrbqr1UgZAudvBm)ay)=(H<+Z!K{mx>n7(>2f3oAm0;9h*YgbXZqC6cn93+EcmT+^OU0ZcHk*iWvzJpWjE$-ix'
    'RNn>i?jsZhDe)#bOs7K-FNlb&AZBWZ=4eN{Md?w%;1V1sXZNbWP>5uD3tIr~`Q9{pFa@^C285^f?VUUK?_zM0_O5_k$;k~PCO@4_h9}9Cz6n_gpCTXjET(ME'
    '-PBqasFiE<9ld^u^32G`#wi(`jMGse0@YY-2FhdDjE-_N#=g4V^%PF~{rPA<q%ciK#aGt8@EXo6gLmXcxEM`fmWBFfw7$ZA7u(7EJemPJ-CKs0VFd#pX7h2j'
    '4CGh{s;Z_dln7O_2&q3+4I(5Ka%Pj!s2wn1vjTQ)#WZsiD0YK1*xY99g5$Qm)d_CR5ZW9?3RGp`mNOKZL8@>E8|MHBTujM&0A)$`OtL-`P_U()qv49u*6+fC'
    'j)l-}#3}+Q@)_(;=hF<@l&xT?R7aDwDy1y|OXow7=0`b*-IF{ULR+xdk8-rj>K!DcQuJRasIPqPjg_5j20h2eRqfqYC`7d)pJB*W?V?dfEY;U^qVBv0-9o$Y'
    'el`U*qgk=J4byMNrXMT5%@V}-(=poY1i86|O(7Js%fW*6PL>YvJy#EkK2evo>2Cv_J0l!@P<3I3Q*nv-)#;uZcX?R!(;cQ%hTE|9-;RSz1()uZ;)?Kq=lZMb'
    'U$|@q?0gvkz;v6QFs~3d2r3NI39chAWQ<)w3EIh~%m=tj-ho-b-JdjUoH){|&enHH1M%NBJ9C1X&NdzThH^*ypcQ~HM^R+3xV^t_(_@qsW`D1nMpOpS;|3sW'
    '5ncl@$_6-}+O^y5uB|sW48p#qRV$*mvDM36v&p-1tav9mB)pgb5O`3nirh6gH4u=Zj!xK|4TVaoGFZ}uS$!b%+GAIpzBny*svNEuTf*g-5?`15#T=xTGVgPd'
    'mm&D#w1B<iAb1-OUj%Y#2N}6*aJ>WVo`cZau_8@KW}@@`4xSnc-b9%q+X%utZ(fCcUEZdp^=&^`19yP?_SGmTp#1a*6H$Ee?W<yq=>+fYXU5SH5Q2x<lrIHz'
    'ylDm73}F54A5boCBw0#es&_HtfIXxilAXkBMS6wk0Kc3~LAO(rXwq4dlVFsk;|VCh)cBAI;zH~S*bw4MM0^D!Nv(B)n?lMCU|~RwkkCR`@g|hD&auI!v07--'
    '>2ZG8DW-d?lfmw4_ww4So%PGzS6=PFV}!#FTNjy;L#(bbBRS_tB**sU?smr#ZD}7=#ck;AHuNIc77A|-V{boPD)Zs|;nKFv;K10$cx6bY)}S(B1_Epl%3shL'
    'v<g&=#2jP+hw8i|AB@<hR*s(*bMJGnecpu~03?@`JRHa(IoVLn`1T3dbwpd#471#bIf#R*$!TF=m?q>E0;Fh%UW3+JXRYhP3RRl{Y~$(3i&VUjB*28^tREzp'
    '6D1ucO0(>Ygm@P0$V?&_IX6XLp@Q*rR1%Z~ebUXyu4gN2ueq>=>iG~*B&x907GVbr!~830C0*6#@u)Fr*w8i}4yxlY)R`^r0x^K{0T~B5y^<|_-U?3#ya%;*'
    'hBJJXM{i4kS7QE|nM554#sUUl9=$alo|p!!VRi^wC%SO9`Gz4c6V9ik*{d^ZmS@QkD7!kRHuaV4VBXHkUM}N8ieWh@w0MLLHm3MP6#)jQ4Ah}7CRqZYK&5Rl'
    '@aWR+tht7MCoReGoEHPZq3;y;Zn%B2)F^J3ku}5Z>v>;q)mB;9RvQ)uRC3d5Lm+_Ysynj33j<17wSi-HQq0+Q6l3EFpTKs{Aw98Rb_}9ZWh)>B5cmM?K$vO-'
    '24mYJkDyyesD#4C4OVCd4r;UG0ZgnABN`2hozd{baNXY^RDea14!A*b_z1Vzm_m29x%~E>&>;yk-&r|Kr#Uh+3>=XRF)U$gb&cuJ7*hbBHkUasEXTnz6Xf`d'
    '?`HAmNje?VM<5C!u^hYGT^<h8QF?iKiyOiuU}#N@tXwz=BdOzvO^DTL*lfX0b=%3szW)D~klr-iB^Q)1;{9wgtMKYDQgLQuO_3f!EX2337Ps9RyPF*q<xB#j'
    '24m1jWDkJU$83pa#~yUX+Xt@1Y$SY{DzP|Us*Zz8?gk|-#UcmF{tDmi@yafeifW0ExgD!*GpSXEk&HrfO|S){wu-bPWSW~92J<Ne2)Npu@GUJ`lpNiwbltP@'
    '4r6=X1JNNL$G~(#ra>b|!B!;2M(ApXCDu!Y9v{rdiXsQvohqN1#ww*vwFv9eCkE2-Y36QIvf?gTNXV44D-6W*y*-Sp;QFzPuxQ}`nwGQ0D2l6WrBHFQb^_^m'
    'B5PE}<nT^0N4@BT?cOkSc~T6S@DK6qx6~Y&obqM18%?DC*+|^WU2k8@VR7`F`btKCXWrL!S64xYn9h{Bf+l=+ne5i7`n<?oL2_X8Y~4EsEW_75yxS<XK%u)c'
    '2MLVeWr#A{b(fW?kU2t_U`<QKLgQ|h>@kD<Fjr>X*lTl83U(coM#_A8gvR2(E_6Pkpr}(V02Y<S;&orAuSVG;CFBt$JE&Hm#Hkvy1I>Ndw>?)pvB3op#r<13'
    '%qc43C?5}?muKGGkOmrXr~!vsdF&LAx!LgIrr7(QR)X;UyH_Le8aa;yXOl4no7gr?6%0mO3u@PCp{UrLgCWxdk_pfRTzN0vW#~!^TEu4kPBEJmBT-e<VyW%O'
    'o={BaEjgrJgs#-30SFVcPqi{@xOyT0&~{Dk)+)MVDx!2JAA(3BI|YNVy}dm-ne7Wj>B^NWf#p=y{Da(AY4`?3T_qBPSqqt~p`4uzGnW$y>*(9ep5|<eYJoHT'
    'o#89o8Y`cBqrN#-si*WN)JhsnUhTtvpwxY3Ejz*-EPYNSPHo{SWG|=)Pi`DMTq1X@@FN~%CwLew$@j*^G)vN1mCX&)dSHT2JvrCXmKM4uBAvVmY%2FskGYRx'
    'rzfU|b1$xOne@md64$sldY8L#t@FVyV4AnJ0+lhfC#<7!qZzYDX3Dt6rr1M^VeB^H_13c$wU5%iX69dWn5C>61}XzlkDGsDw;`uzTuNrI3KneRvoUy<-0XE#'
    'l34Pnr{m_$I>(|X^eqvLjf|||P`lbdJvt!rny?5l%>OnYW&ndG>UCxWyp5kU7%b8(0d3x*#HC9IM;L$a%57?f0~!}-S2LsfY1m!V8KMTI8PVw0WU=0uXLPK|'
    'W4&?IFz1uwouih^TV=d+_Znwj_0uThn}4477RAh2X}vLOXwy`EFz=C)Q&#>mv$i<quH&8Ael?q`pRO`uQmYRR9E}-lzcT6=<!j5@^WS=7)UZW@^~QS%m;7av'
    'a7qi;OX~tJZAQBSS-DJs6_WolzN^9oGa@RF_8B)WS=gyI{-DMMRk^$Tfyy|ic-pQFqXnF5%tzA;-apu2x%eo5OqEbkss7lg&SvRi0Qeca39{gHc2Z$o6|=&7'
    'xu*wHJL+ghHSkK<Sq(+zeifR?g)5<8wm~Hs;fCoHs)A|ND)LIn+It9i?KL)Nzo|r5vH^S}Ewj5AbIYOx--XTGJ<O7$L=Fnvof$~QF_oLBv-9(FETOij)cz*Z'
    '>VWwqt&$jqEFn<LUMf-TZ^psxIJk~sY*c~~L^M0#iG-@p%Ke~kR?zU^pMK(4J=dl*&3^NPW$RjW+&!X<+#7{@3}j6VKU0lkJ_+-|u|gkMCIw+?7j6qiWV0Ig'
    'C31maun3bHLTISbmptGcRzY@z36~cRN?m0l2fTyxf#<wgV>djG@LtclZj8qr7O_1$32X5)3i?9bFOk%h^4B$=g;$_v{)%*7R5UbUwwcHpm@2uo4a({WZ?xu)'
    '0qtHl@3rhK%M`yXfQ*R9sZJGol$Hm)jCS#H&3)#Tj<La_x3!va;DY10?}Q5}?U>pl44{6`yjnKu9Ol_kh_~5LDbh7b?+Y|;jhy!^=q((yZo$+so7!S!_0|s0'
    'ueFx+qZ@!6CEC$OtzCV0wcanPOKMm`3O}3Fq8>Z(aYc;R-fQcF6?+hl(G9{<Tqm?zXxf@DeXKrjVwyKK7vIS4Snuc%G&|4@Kejs4Y?8rhwoW;O985tWW6**1'
    '=oF19c}d#OAU|h2-kx)Jnj2iGOP&OQ#sqS_?U02L*e?s$5LuqGP7$w_s^M&+VO-7%z6G65+XRP2+Z~zDPrXJ(q><F;pE~hWJYr+XVMpKy3vsQKBMtl5ImaE;'
    'jomacTQyQ0I{YT&DN&9Ng*)s#QQ6z94i|N9+`M_~+N~RRA0$^FJb*v9-nzTx^TOy9tUD?c`jd6&Oj(3R3e1j_W#O9hS`{_r``MxB4Aub055~(8r-T`~68T$f'
    'Ebit}Usa&p@<<;;HNU>~-JVVkMNw>QkRj$Hbv8!9ic=8T8%e(=pzX|Dz<|=D`7A^63_<~OP04**o7KgJuCUdjuq_d0L)X}CQTY1M>BpxU@0(#wHcbng2rX-9'
    '9c!w?Lc)^0N2%{Jm7QVHKcKnPGxU_&gy5J@mM)K|RuFA&5a)Hbh%pDod2uhH?U$Qtq#|s{=1@^^D3(odtQF1D1l0594dAJKp(*J~fyj}G)-L7R$GNUFa_{Bb'
    'rW$KUS6yG^E#kg^L*1;ksc76s=P2CXSX6g~ovfzq7VV1u3-j7SXm@3i(`j_7U8`~Z&okS*Hn(&BJJ@q*^Ru##_0FTu@@{s4osXr0)f3*axoL*bX8+=VqV3Mj'
    'to-JcLB@B*KCif1^1*SPeP6;*P;Cj-0m$mL4odNu*uYFP=_7uz3aXLS8i98*I&N$<7XDE--Lr>L>#1sx=Z&?W>+a@!y?<s6Dn=LAO1I@=-j;x@RcXuaQ6mwW'
    'uu{vm2Ij}i53bF9U=@$mZgAYJ)rQE;BB2Rs6-D*d*M##^@~FBlUAtos)Z0+I@Km`ol3&sl2ek1}IZXk4&=#=gCRGJ`gWl>+24XBJk9w{}#UNkL8Y-j2Vb3u#'
    ')=e}OaeL}RCBE}9roZ&pJ5cK#G}Oaz09F>-!z%_A(MQ*pgf%q|sD>#T-PLMB1fs=vF3)MF0*j(s=dHibre<aA*%P`RRLvSz0n;aSi=9>+SXX^|9lUpn2Nyox'
    'bMx^U_~HG)$zC=-Z#>Tpu473$AK!B`LE1s~S1?0`Xv*uwdD>D>syVsC3@!x*;i|ORW{}URS~1gKv~Tb5Ik#6dv{yqj8y{+g;y5pY5zapt;hX~x6gczh;w;_B'
    'b-Nhv`K)5VE-foH8;cBzse2vsld$(U&FQG!w)~V$HcYsv?T-H0HLvFP{P^2n|Ky+k=aWDC*2jPGcTd0h?I+*-`jel3)yuCuX3}`O>Eo3xKOiPiJb*0zF)lh~'
    'Sbp-0Z+!IAKY8-)Z+-kXfAHwwu$GbhDAq^OV1=j^WE*E4vXG;pgA%U2i@kC7<9@IJ?lxRM<_D+3+_km#<9@gR?k>oF><>?ceG4<+xfrKy+uym{qFtuJGc;l-'
    '<W=DGMx&FmmJJpv-@&|3{r#|M=n~J^Rl?NkF|Hf9#5r;nQ01EdS&wZS`Q)?i`|F;5@Y*A{SOsPH+O$|FRMj0pxR7A37q>|pdF%F-Jd2TNi`XK?{iEQhUbj5B'
    '#-Pmt^Gt&ylh{3PPv=l_LOZKV2~?OL{p~Nm`?ZgL`bST{^W!Jq`@tu_`0htP{ozL+{>jH5{^-f?{mn-oeEp*j{^6q!{_I?1Dib%GJgcZmcXJnUm8XoWOrCLI'
    'WpWW&c@bH85m|W=S$Uq3l}U@x%H#~Ol}Tf8W%4T+U6~kj<0V|##F|5=2gQQZBJeV~h`jt%BQO1C%DTo9cWc&8)S^v2)aJd9*PYsG^HbBLf1gn}4LOdc8#WqV'
    'A8B$Cj`}Mbjw+Jz^d2tZ8ULl&VKvKiuIah)#|&?-B8Hm^8^fEnm!Y}kjW%Zwo<%(DrxO$VyyIe9)3!xq>}jK8E5%A3d=N$TgD(=N_uTItN+&GyPyg;8Kl$<B'
    'fAWj}`t-m3WAOAZ|MjDv{t&qL<G=X>ym)6kd-|OpfArI@!plePj=Z?qwY^P7fzx=|8jRrQfWiV^x5mBw^FhZv{r|}?zWd8>eCx^A{_V++{uT!Elkfi?b8`i%'
    'pDTF#OJ8>OgoIN&*4lK)vn>yKvISvUvUJPU(W*bb?zjxO8oq9^>7YW?8orHty$8>xjy0-Ylc=*V2dC?c4c*_9gERKT<)m@;<?wWUal!ZZ=J1TYc^1CD=i={s'
    'Za%;2ve0V&z8!oic=F+&!AkSwTYnXNIe^Lh%kO;^<ke|uESEZQ;rm%dLX)RVt9CvDqOCW|hS4lE>Q9Q)60No5sds3#AlPD3sGZ}efmj!(kuFXnU7SX0e;SFh'
    'RBkSMLTKxZN0QF9)zh3yf+g!wfNs_g|NO}x{s}0Z|M~N81qX+?S^D=sW^XQzE?pd5dO?mZy%bEQ`FIxYK3sZxBY5BIeR&)QUxM!z<>l8eS6?-8%)UQdiY|^z'
    '{rVo4YG<ch@HEw;mN16}->IE8x*S@lr;b{F{j{~ip+Q0Y^*mH&rS%R$Jl>&1ywCVJnR$Xmy>+<9TTowG2Z!(<c7c4)N`C4(t7cRm4JY#cf!p_Z4HGH4eUN43'
    'bsaOav21g8iy*_cM2X)xuO==I>%7>9b)sP9iuPecpTcE6n5RSgu!XmuLJAs;Y9-{<wU+x(rMgC?5xAUb?oG5P-ZP%f1nCv!dukAwFVT5(w1Iy_dj3zfOSaN;'
    'IT$b7gQ3#$=6pCj!OQN)nSOQh8Aj7592~>i012erLY3O}T3>6-5*Lp7U%fn0D7QRm+8aceKCTVJ-XP9uJOzzHO|NSyMJ;8W4yEUYZEU+F$M%#>YYp@odFCxp'
    '2Tz1SxL(G!8fnjXj%)Y#J#*{qwLNPL-1B`fFXeEvulw~vmtjQBTxrWildn&?CY)+|G_4O_?_DiaWvy5kz#{a~SC58yX>C^S6b}~IuNbS2;vW-R1oOrt?`}pd'
    'O4}=TUx`v!Psg!k?@IvpHrF66zBTe+nPQz>s@+E^{W`Zj6cjSkUVIuGE(5YJ6UPg;rEAs1`Sj*u_jr-&;Ti2yliFMf&Qu9)k{<-0eE;`8{)->T5G5Jv;76o6'
    'YPw-By-4h|)v7;m+9RLB)@c_`x5rP;xXDjm_+5V8tm*kaxa(U)@A*EswM!u_o6ZrpW7hVC{bz_<ZT*vHXYn|lTsZyQMjh9Yv#PDZrPbVj@do@Xa!FC&7NOj!'
    '<kuMubIFB`>f-RovphU<{#G!>`O495c@SYCGnv!b`)~>7V7vAb89YI9IT_{$Sz}{<XtfUHR()DLIMi(0?@(?~#M;KvM`t)tl3Z9eFD#qqVcC>25&wz3=SG}5'
    't@R=HA7(MCs_&kqowX+VN>dqG1l2iPXwN3PPi?|I7c*}Ah2RA~tJE6XRs?q)d#MeoZtBlM>apV-OhYX8c`Vj#`?tvYt+3ysC%oFP(dwq8?j+IqtiY!iR_EpN'
    'xN#Ap^@6(>F5<Z^Eb^Zzi~L38*spoynBOQk&W_I2EXzF6zIe1#`BRv*_vr!)85<qFU&sneKJ^yBoF-OBzaEZHY5jz@E)1X-2GGxz0rVma@WNPj5eE32!vN1_'
    'x;kG2(~D}nczzBChXV7QoD83GR3|rTFvd*=c^^+MzRz9OOV-ge;2`_FTC5c}EG4pr@!qWTz^pV$r?WgAhOY0i8YS7YBRuiU=wpWsXOMAt_omqc-27$(df8~i'
    'X>lu?WsOK_JwUTcWvRg&o1A?H=5<v?Kix45JJma*@Vv~BU!mmkonjetF!LI092sFOV-77+to@SRPjy2qTcOor9JuTxX4fJ!c9%5`&~47zbl|vUAxU33dd$Rz'
    'CIv$W*i>OS>{yiBT0SCurs;T3lw^5{ShrYn(UvED2ur_kZgwhR8RFyTx<2k1MaJ?}EWiR1D7`T`peMOmT}E#Kr^94K<~+*sz5Q89nMNqfOR$@RQXNdDML#P`'
    'faEgE;ZzAPtZC2HnwAe5w{Q-NnXyZ%tYH4VmcxYB7D~NMZNa-gn#~!DT*l?RI_IUJKg~eP4Fr{gO7n+)1Up$!?x)iX=0+csCwRA?0hf>{WCE#F&p%)qmt2}r'
    'syS*}QDWENYDuLd4J<7orIHz4;V;%+K6ZZfo8F*OC=k|20_xZ_O=zEph2DSMR@}kDoK0t5R#dYS8GDFbFXv$;M>?ZAX03R_LV*l!08M1&Su?dgweKCa|M90J'
    'KXvHqS;JqgRSj3ifx1593aPBV_0tOord9U$xTSHqy@7dABlCF$FXMlgE~%qkL*06=cSPU%c3bsc)CEFrPaQ|Qh!B3h!PBRUq&;(3tlz4ev{o-_odGYVQMiRF'
    'abXR9k*&ecS#Pmz4<4?1hB}PbLVNz2j7trWF@Lr=t;_XfD@GQN_!T3&vdTF1_1dX2=MlAd`sl@TRA!o;5G{3_Q}`j;`XZi%KjJJtiN*aAi~A<d!#{CWA4R*M'
    'URx)OPpuR-dIK6g0<8mATet<z{}93{eF0}XeQ*|sz&Rc}IEz=ndHdLDxxjK1coc_o^S8E_6_I7EzgbML(X6|Dr`BYj%yK}38zm7qitf#aOGB1<zQeveT-s6<'
    '?DMkm<1`w|kVkSaxC<_1+~=q>6ZmcDaXNKcejO(JojfgjH&fu6XK%nI%D@yMBC5p01$mr~Xi=S}Be@*oJJ@XFyKDvva~RoD@Zl1lbPt!X9#(M(r(&A#A<hYY'
    'M>@eDytj0i;$CSfkuYKasHJYLSQD`5z*>;PzIJlLIH5x+9m!V_Mw?nATU1u-|6Vp@a$i|&En2AiKH?V538R85ec<?2oEm$&!2u<xsEH-NwW}qXoc&DP+(6<0'
    'O=Q(e3rmHWwSp#iu5EFk{K?(tN;fk*ssgDUg$qZ-g(G4SM+9Sj8Ja{q^c>=89fa&T-km)P_tVbGz?q(t`N9@<VGDbPwy^e-`MJgzDwEs(auHSdt8@g{FZh<?'
    'CE`a=FIIQby==Hz=6l0@0!k<_b!T$2w6XM3a1B(}DQfXS*pGtlTKARU!G0FJIo|_~b?+u<!JzQ>JHadiCHLXjQQ_6#jgwh6NOA9Ydp79cZeSJk_8d0`ySw=?'
    'r^^qp?)`h$e|zP2-iHy-R&EV&E!oYp=|*sMf)Z+_+gXERSn|PsUczQ#m<9W38SG@)7|D*M2f@w>wRpX$#h@4yek*IQu5{Nr9P(RS7iZ9q0Qxi=W-!+W0Z5Z<'
    's^Ai{Tj0@ty*nQcgCWB_z|lZZ>0z1=@v>!DCxiXjY*KEluI@pV`3`IaMyvabh`T_<Rl1BAR-h5=R#tb0#m?#oRIAn7x31l|d;dm9?Y8VFVVGM8B0eSVRyaN#'
    '<br%ODW)^FK0F-nPKy!y9q<oh+7+|+3M!qnKg$nOlntdQdiC0a?#;WruwbDMV;=6hn@_1q!d{As_vz`&c?qAcQMmxfo7k^Cu*I_ztPIMF`3qwg&tX<~XNc<V'
    '+ccP!K-dXv6hjBvLW7^;P7wZ&9%#sN=H<|c6EM)TA}C=bGDm@8H%NocZM7ll%Jx<#xHUtda~ySmkdI+M3B5<ROUl}1wSq&c7QpgB{*|=fFQ&LF!dk>AldR9U'
    '36%W#2uO&Rpk%wW?Vu8DBa{NK<g=^~G9-i6WqVkPx2Pj8rv?B^=R=r!M>))qlRO(jTS1wRU~Wu}N`w?l7U(%vHIV0TS<hITn|npEH_W)`TG@eVurf(!{r&aV'
    'R=V9+wn8!Ol(_~xx>{c?(z`39=rw(GcHV>UFt+3UY?=cfA$x6Z1KZ6Q+p*%?TV(#|r(^Vdi2;Ld&!)vtAuyNAo!}kZ!@_&6m==8+71pM|4YY)ekc~ilg{e!$'
    'CE{0M|EBEy)RVp8Hf*)FVVhTQ#q3gCjTj*Mxz|2#+D@+LfKW+=^Vx|Q6){LWs1o~(uO~$@6mL6<;x?XTAYcZ_vF#AKy!N?nJ4_xfsn8!T1!3OFI&rX@WrG#2'
    '<VS(@-(ZyomB<qvQING_I2(~&w@|%1AMG#}W{?D(S|d>Ht=PS_XF);23K%S8o(xts<->yH>F=lGaW-s5>#`p!34^yvCMuB9pqrE`>)?xaJcCo&XbzPZR(G0}'
    '#c+=GB5a-dPp~`cJ^*AwySCO}TdXMp-LWPmthdlqIh$?=FeuyN?y8!Gx6!jQoF|hLTF?^^%=-_Qwq4kai}A{kth_<Bt_PxxAe6tLX+QzwWh4eeE+Ba8AI)i4'
    '1$KF<SReuI8cHjCnDP(!yJWu!j6fVuGQul(d^FGd2g#l&1JbOVPjHdW1|2i{TSOlNwdg=d-Oyw!YtGSu>iH15OHvusM(ikKe1wiR?6jH;Di2nZaWE|=Ai7&%'
    'vDR60;UECeszNgnv8X2EWLoT`JD{`9a+(GV!i{6H<jBqD_VzZ6|Gqc^cjd~Jz|!UXYFHdtI>ti<l48@R<=&;p(Q;qEssgi|3=JKpYD8BAdM=eKYK;_iAqWX6'
    'puhkq!F>@Crwh>dEZ7R`u+?a&RWLBV+UIT$W!-N{4rYSybM=@HxzID&3o4S#PA2H5fT_lIb4-Uym@RtA>6-GIxfDgA2Z}&kvwrt*eU59^%-*{B*;vu+0%Aev'
    'U5;JizXzifyJU55{k7O7SbI)M8T%wl59G#cam}Rf!IW0M!^_OHLka77n?&%sY*1;h`r-_0gf$M<8>5CjFxDGK4Rfp<?;N${0ICmW532F3ep;3fIkeT!*hw!O'
    '=IW<W#u#7y(QDckqiwx0YB-!O*2x%}oA^iFT^h$@y4OQ_cV%PdpFuu~>3+j~_U9-LP?Gi5*Wkal`F?h%S#=d>%FZvXaoAn>i!!ArZ)AM!m55z!9Y(<_7E;B+'
    'VZwo}ch=x*h{ynCs0qy+wKvTN$-bji>P+M)$bR5KP|-|>F!Pt)+=$7pyoDay3=m%DM6xkC@?dflISR5L&6reWH`$#M8xj*fWa}Cx{i4K|utHELh-VVvmH+_2'
    '_k%2CeTyZr^;kya00eN<0N_XiU@~g9mzN+%lAZbPE{3En5o+k=rXNvbdFlg}ryo%(_(uZ%Q9JlY0{)TaoU}X4rV4gj>GX!q|0N%*%}%GoxhA+IOY`v)qo}x5'
    '>69D4Qx?WGBv>&68L|6hSJrXDkoZ8hTDQGX00q|eGK32F6Z#bLI2G&=2EBqX-%p3T;jzH`cpch3X4q~7$1E+&=F0k3u(}#_IflpGs*qVkc7x-f^kL-~X8rMc'
    'g?@#L^S1Rzac~gJSY3hPOVz#U!C;;Vixc_i2V&<gOh+OHw+hk~WU3-??Qf5mn;*k$AHiq=0KA<$R0;Mdf)6bGmK+ULjRQ}O1HA?(4{s_K5Pasr8&%Xo`^;?{'
    'f!~_gDp?x?BO`vCr*VPiZ7!5i_Sdp>_0BqaEH}u?$z$=(k}K?!49#G1N6_8F?jST3n@W37j?#LR2w04(;(C`YN7EDvF<rvwf$2Agt?i(x5I>&`n&lzvM097P'
    '<8&|dfIwws0s~6INoGH($PTm9Xw?r#hT}OvU%J$FRph=tS5++n1p~OnabM6_Hr)B(jUZt2u(#c8Z1AKeyP>_qYzl)(Jdw<bgxM>MJcpxfmh#{;J!%F&7>F_e'
    'iIeqs$eSIs<zP7!3I@xAUeh(Z!z1Hp9{^oC%==ky8PQs{LUwCHb3Q$!y;#006Orgh@wmqhaNTG=BgF~eU&eJ#-g)g{9i{E|csnCH+_VyCVK5zF{%&J=q<|>x'
    '?58YHfx->Et8Am#bK~HO74zR|Cc<;_V-RQ}u!F7QH&HzosSK?eTo>`gO;vGAZNin59+V!!g$_g`OVX$Z{~Y2*J5!@I`m#?Hg|*$SpMT!wT&kNAXQ*ogH_^2y'
    '9h2SA@Kn~Q1vRmOmM+jDP_Nm<LpO|j<A?fl3_?*Om~M?{Rw>-muwol%n23#yjSk%o`JtUcQz6<Ow1L!E$TZE=viS`3!(s9WgHhrD&n9Se>!5(rPw(@{9r`Th'
    '#Ur}BKGdp4dbE{Z;h?^PdO5tPa_Fw&MNEC{9Ol_ksC88?-^|3rGXukKzH&QpuO7|-(deA|cnYZAL=O&8^9c6VkB$u5peX_WrvYYLt=Lu2F1N827^*0=b8w{5'
    'tF{%Zn5zxDD+g#waBth_Ypj_ai~#mWM;u2t0QwqhW{1cXq$Y>4+GK7rGH)q+Ofw+NRf`dAOWT=h_jEEpI#hDUb%cKO94W?PA5zU_HHK|0T?yFaQLlRNmpbs5'
    's>CqBsvi8M4*aEfs2i!;zO9W;c4A80kC7ZwzDZ?Jo47E9wmC$pWiP6j@4NlZ`W$!BmCH2QMUw=JgWA+L)KF;4yrH$f-u8fvWxS6pSX~z^+vxAu=hx&!s?h>m'
    '>!MS65MvMB^{!aX3L0G(s|w`sR^Ys(eYUyj#`sjJ_RnmvS;!=#X4ZFWmhQ7)m4icYAikjdUgjQ5$?c`PMIvXK{%R^VwcI^R%NL5QPm%R2t&X&#)*PsvR0`KD'
    'HXD6ubhW}g?a8~qwOQV3mMt8n9@A?K+XIeGkN$PkIqET0#;$YJV;%>W0m3APl1S4_dH?9N9GmW9V}rR)+g50Y{{qtW#j%99OZn(n8n~FF0G;HNSsq)-F?PP6'
    'TN{3DHstlYZ<9ea8tJEjd7N6C^c*Hst*Sm*Vuj#w;1xUd|EWSrlS7*5QKe0^_>5!oMhVz_A?99_w&PNcg!RqiT(k6!e7DA1K||(oY2nV<;kr_*1Lopvv5me`'
    '-w_t$Fn#kd5IgKcHiLWlSR!5hvRz6dF0*GyTu-K8JYTQgR-|cW=dYC2n5caR8rvGD&q)|AyuDFQrMpkiDt+xa-o=5ln!DS*+OMAdc#GUpHFow^xtU0aRXMpP'
    'c%}m0Y}TT?<;umVhPn_Sdb2nn5}p=DN2kN+$ik=zP)$0XAs#1GSxIsdpQ@(gTUpzB-c_jQUXWCkI55N0+({*)7O3iSBecrd+it6fZ>eHYv(6)1V(nqvny|XA'
    '$Y37-3N?9N3!bdw;gVwldV*ljv=BNnSu<<A5`1f5t}fBfD-pIoj|fUZsVI2TS*V>+#9(J}C+25;foP`BDc6Q`TN<9hzB<y63{#c_c1oC^D=12N%ygQ`pYN4L'
    'b^LU1SnP;XrjG_AaUubKi}RTUj7(7s%g$_eJKqr{9z3`SA9$I2w{DB~w@{0V*`Mpv^oajtg@w4eIj3p7pH8oi#~A1(S9ym7aA@=ekUF28^wztdcYeoY7fHVu'
    '7SmqXef8BiScCsXF-~Iq^vl2h-Y0+fFTp!se8X3l?Bc!U^3gn<X3NnA)?DuA)BZ494xWDVuLAK&i7Zx*m^zO5DQIsnB)<BmPrvhDo_z5AAY5BvNIv<|x1N0M'
    '2kgUt{`_0;lD+xpr+>m;Mka%ZZf+?3f|}5?Ctv&hkAL<D@cYU4|Ia`nLh#W~Klu2AzlU-TSk!F>>&b_I|M4IF8`NL0+sxa*Itl`o|1wRM5^+38As<|&2qjU9'
    '9#nBQNvCC&Otal2WsHRJOq_3c(jq+jtu7T+K@9eo!2*f~mgUzUKK$LC;ux34=?s2f2_HTjyd1^uKb=UF&&C5Ry{t+t5eGKSI<PDZ(%~?)Th$SCMW`!X!n|f%'
    '`o5Z)!TmIE&)~J=^*A_z|Hr`5lP+!1F~%8+VkYxgd0f{YyZYm0Go_t^RR@+n=zkdMjm5$8a^&oh@FrCt_)RcXxEW=t1$G$0(B>t|P81(vNJUz%e6$y<U?TnM'
    '0IY({rxd4DU+Vr?6k;a~d6-+c3{&UovQq3QAI$dSV4oIzSzO<cD(OW42xw(cj-(#ukmn#{Jp8190wQBz7XP10t^r6*0RSav2mc8<WG7jX9ddwY4nrRVES(q3'
    'U>lrGapz#rOdHt#Y!0-dFlHQo_;HDKKUfX$^g`q;e<I4CV0libx(dgF8{tQ*21NOjuA>28A69y5aGH*<_ZwjJvbUc2=99q*9&St0;bb2J*1}7Mq>DAt7+K-6'
    '7=R{aA`{rvKQ&d|3ENfK*^^6`Bb^>{TPGWW7F=fGrrUZZsTx;AjO&YbtD;@I<W`KSGKM`T@nS)Zo<^0Ju}ZJ&%4zEAGheEFeV%6hSvuYues+4T`6}kanUpdF'
    'fMg~roLQJN9js1da7oMnY0_^ws`Rw0Qd+!|VJ4I8eAP5mE&0L0wfLOXI7+~y))oJ-`cMDr8&Ce%UoWZ%#x3+f9fO@>IN-q=RkKgO^W!Jq{{E9c`tKkA%U?Sx'
    'Vja*!rrCH8Mt+?tvcgY({sAeB>noQdUyD$&ILH#lHfw6DfBL7N{KvmP`O${}qM<UUn_QeWOel<Sel76HH~-s{fBNGm-}<YkKmC`dfA{B4zWalAit#xp#geHu'
    '0zmhp5C8nhKm5~^pZw{w)0`_#+tAobg?2u=w#HQ(jm)I+(T2uC&=&B}fByWN>rcP;)lWYB89ZNxU%&jupZw>~|IpE(%2XST{A97x#zM@~Y6~HU83W72?^A6v'
    's`10XHu=zYGh)L+x(iVF+bE+A3jsT(W03hcyd3L;7DX=j>R^a=)OR&?v~H^J+-fC|b(ohk%Brj@9ZQ{6{Ji?5H2>{<e7_j)UG2<fyV`LZPr*0p3qE*r<w3cB'
    'GG4j2#tR0GCD+@Re64lSu3d7=dyrTfITD0o?$@XYvzYk56S+}kPx_e8%y1Mak<ZhU4O8Sz*<iuelD33$<Q&QmXiEi|2=_bLF%&67SvA^lt2cIwno%qtQm2GE'
    'g1zj_rt=IB;8wt6W8Ox7h%SnXy{2fbruu9q@L?oZI@r*rIz*yD4y~r@WQw~JV3nue{PvSS`a6)bpM3ZsTYENw_XV->+%;Ok6>J_m(^E62Pd}T?f*bS)^G&2>'
    'kZrU;L2dlWKm7A2|N4)?`?A5ufKFyNYV9E8=?`NJT&gt7r(!d}u)rv2_BC|$vTlr3puVcEp~CP5SH5o|&cmfo{@3@P{9k_rYti?9`1B9|<H<LF^3l(J_R)ub'
    '$%q9D)JH$VHTa_sfAR5O{NR^A`_3mn`Um#OBADZNip~dL{n?YR{WBYik3RT%@cu9xhprAsk3p&Up^AW$BUDdfGhbA^z_>uDVvHT!EO4q}JmnC!89pV`e92^D'
    'KBmHav!PVyW)A(-q4Hhc3Wz$aI;<c6%^y7d{jWXw`v2?9!L6UXvn61Eluo3Cp@ue>@!j&4Et|{v&Wy57;$6K^rbADqM>tOhp)}lxnjNg>ng33Nl@GN0I3)``'
    '{hwcb`gi{b^!oV!{(i9hKH~Klw9~)-w<q8D;Fo{#T{ko7?JB;fd^{jZVLkvFRpH~a2M}M$U?F2=tWBD(mE6RiTXiNgoqaXNGAw6b$tTV^9NP>XIW9L~M~V$*'
    'u4Uso<$!feov8%GuFQ#3R}7HL(O}{gQo%~FUMKGE7Nl@cM#X6+GoB)fX@3S!jBlKxZ8sebchdfWkUTB~HrKWmq>2XBOB7Ejg-I4g?lefbk*Q`)I(Tnh&Jt$K'
    'nM^R->P~T-2``7opbLYF<EefPUhi4s=rhRcbRffgI_%GfXq(TAc?cM+<6h<1%4vAKo{$wfAv-kvhJT^^SY>nLl`X4ymx_0(I2o>!E-T)(iyM`bVYWNtTPW5r'
    '!ZX<HB$*WxUg$(%XBIk6rrf}f`!I$$;$y+uDQ2@`#LJzC4tASVDgpm>fi%b;3G8)-<A#@0*D%d;4SYOoQf7UgnJg5It?<S$eMTr-TVbu$Vw_I1be%xq;H}W8'
    'GAQgB=@{8ny1Yu4Rq2W<T~@`U(tOMiX05%<L9A%~sl$>}N9+D7hYB=jKduC8q_72R&dnR;GE8(=8+)lv9y;sS8>XY3K^h!yKqoiXw}3;?m!@=z%~eWfg1fKK'
    'hVJ*@R3XeXaXBPwDZ#SuRb_J^8?az)6=n-MPI4qQFSMC&xeL9g7J6^1P8pT#SIjXC(-aU%9y5qhD<HcB2S&@I<^VtCG<9>DnK{k7oGqKwHEPmZ_dTln9@Tx1'
    '>b}S8zQ^mnXS#NDr{2U6HaQ8K_oDhbhE!p=H&xQ)73*|U6;muR^7v}i!xDxiy$>rI9qJ)SZ>fRairU3RP(!^~!|rvXt(v_t)TQx)nSvvLDbl)7AK$&l-n~cf'
    'HtIviM>*vv(wO08okkIb3D<*BCLjq*g{w!H4NGO}s3oZP2o$UC;qe+!p55%T?E8D7?{Zd(zPCZxK}OMcr)==qSnB(YY;(fc=0s(i6UH_tr(+wO+AkZ^;*4y='
    'I!qKe4ci=7^nb=|bLPI=Y{M|F`o19B99Q$MmTl|^77oXbMS{1J#(1PyNwTS~*3Mv)#20MxYZ=MJM(0fNOfhR{vwm~&*$jti)1PM-8q<!NsMxH<)vFpp>P1i1'
    'Ye<NS51&8sbqxivI3u1Cvr$h#N8p<?M#LEo!tFbeZ&I*>_rATp+BOShJ4Jk5C90T}h8Za+1clP&RK~bimqamhD4_>K545w*qOfXm1+54BE|}b$LGiKHWd7Ix'
    '{pnwR6HT)J<Lgg;^1+iI|8#>#zPwL=9t&K0@3X&dbCNKo2z2Xr%@ocroXE&(3J>upJ(yx^9T?K}HD8zkpN%TCCSyQV*FzQ+_<A>}1J~erLn3sT`>pc4ArAX#'
    '3DSF7m5PI9EVmr7-LgVL)44yIU_fO}jN_%Qpx>n~#yJ|nY+AIV1nCZY6s^^gbO$C0Ikh<$a(Uy+Fys+KjH$6tJ%>kvyy0TkgY{t1fV~R+GHs7kRxZ#~<gzTh'
    'zSS`DN=s;$pgUO<fbQLI30v!ojzF=R$dX@p6=Z$|%~uIK#9`~T*h^0}E*^Qiv4GWTA1^ecMCD<eTc2mEaP|5UgBwLoV}JvWXR>%({d^mI+s_}njYTI?qb18W'
    'SD(^v^T|(s|LI@;s~Z2Lc>m&{g^$xMNvkCU=UKuT5EhVZcNgzo5;@T6;D<Q=5(iA@Wo`}(fs_%kO4N#cqHn5Jd<=RYCjH~!Tnk^mtHE+iBvdp9$Ccr1#o0Ei'
    '{c^A#Ii)gyI5`stjw>MY8+?5PfWP7AYZ<MI6b!>-fIA>^z~7TdC2X&vZcihmE7e*RL}tX{?e3PD6`kUbGB!e)dQLx)0u<w%)}k$60IvnE+Xfs9q+hqeI`^lp'
    '>MI%?_IHH<x_Bt&Ou!-e;)@p=v1(Fi<5dB2%_T2@7yL*CR26duf1bzD3Ma@H<A{YBlX)yPjbr8}u?t9woG`~p!&Izb3Z+{C(xi9?!6*OU|MST&zQGjmPyXZI'
    'KmN;afesD}#qs*%`27?7=eUc%x^9&lFV4%y^V;R@vwldTpP`E>Ic7E^G8+2~SG*|lDVhM#Q0OZ<O$YhB<jR&D#8??UU(Gg$HYCm~yrH<lO}=WV`m$YByn5je'
    'b?7s7D-Iem!<-xL#(W0dNsAwFM8S{{F^mnWvMG}vIJfO=Z{y^jxN5!@Nq2xJ-&Qs5s5D$15@M0<)xujWrWQM}_Xn+3_{2tAva1SVTXoTkMSXjf<YxJ06uE6r'
    'e8)Z%hlk`*B|d;oReLV^L0?fkoydy3$rTm5s2v7El6TkEe5WFuBER|^D}W;F!Uw6x`k|<$YJfR{5IuVsw`_sMpV6GRaUv&y#vHMSpKYPy4(N}4z4l%MaQb6m'
    'ZI6SgSq2^*W#jIuIQwN^N-i%Xe;W!053+PbR*w2B%3EDCF47d7pj}j^A^4qpH<D{_-njOqTX(<czA$?}%ZKu+MLxK1@f#tpXK2UG#{HAK#q6rM9h{G0ay$LX'
    '2}{7i-{d8`4<4k$v`kW?JblG$?9>nZrk;egk3&7DJB%^BpKIz%`W4+o;fG5b!REsyBx0ARR9O<qaUL#lBs}FG2ozbB$Jl-)4S9UGrz@elYn`TfnKjS~^?z=f'
    's4vLjPSO%|2W4|+QD)QCzhpNpXV4!yiehy0Ci~>?rZy%oMi1r9>3sCI^fNZRi#LnuHOlvV`;K;ChJR-Ff;v8!H?4~zTZ|boKw6zQ98a1$9S1Xt$)j?kQo|lC'
    'twP0)T;S*{^8)<1$cQN>vsR;4jhw@H;Oi%GrMt7ll#LzsbLOYqSDil%=_&p*c{Fx>CI)nUFnH9=Wv<O2-h6o?GoHdv#1TYuxtUOQsU*QUq{!@0ypJi%O*$;Y'
    '2xeZAfTj;`Q8?ODGY%Nkq?}~^UP6D8#JL7KCd%_&4%xX$BBt}|T6q_(y(a3oYb&IAmA$LF<PJ>7rdOvoJU<xSz{Ri8ZCICw1rdXdJU&ncMgF#fX|mW$IjE7E'
    'zDyBJp^3kkR8#es>4W{e#DgIer><S>%K**_P{RsXrR@TETYIp%T`~9eRwuYc$4uzfYN(ozOW1|ZF!y`F^)_{s(tf{~(zQeU0zPCi9N|HbPWR@cj7}8{GT15V'
    'Hxs8IE<lkZMuj$IE9|;u$GydL-7?mf*IpmNE;l&Ivmvw<l=%qc!qn&>A(hUcqYCOPpL=6vC!0af@o`mqw-xTsW|MMbb#<Sy^e(V8u-K|yG?MpaU(<=Y^B$_+'
    'K)L;Fn$MUJ+T1qBa(jzr4<0nw$t1ba0yQm$3ZYm$O4$kC!Q7qjo?AnUK2ew7vhHnzf>Lnw^D#0j74I;9a~xnlEc)qA%PZ+$UH`&m>*Dh-BY-&!#0k5d9yn~q'
    'B?f1iPH-K$A!7_Xo{x55dO)#3j+bvjj{;hoN;YtuJW~urD$#Buu2MHHg6D}1^Fg^y%K?--+6O7sPlq_U4JLP9LfzQoSJu8@_4m5CYEtx+0jv<_fetm7TQ|1b'
    '?XImiH+<*zy_#079MX-gUbc=y+?8X+JINvCV?sd5IViFncWUqCJ`)Pn9rg?srDui}Mfo7JG`GVJ?_Xi}s+<-(RT5W>E#Y!ZiLcB3Vh(aknfb7I3^Sb3!562+'
    'd;%}u#_MAOc@G0f$z6l%9ca5Q&xTG^@`R)&G#cE2j^x3cC{<(|L3roQs~BU}rls|5KUf2Ifcy5<C@7%(^avHfFTQ<MtTLV8U6`q6x_sKTv&CS95LYO5F=!0^'
    'gGrH-lz>kNrg|4M4%kEbA=yd5R-{*m4)B?plno3M=SeWi(($AyNNj-gpgsy$5g$Tai-@maB&oGda8t<H0W1xu3=mo%yn{_BTbj8SC)u=GYSQU(e%L9dd#jVd'
    '?rQh)+N+)Q%iULA?Z9J%!ye?cCevAFvA)KP<eVds9NU+>+a1@{^uWr{+imowY~wXWI3hKSy^R+j59be;wrvIn#xBMy?1I6dx?3d>V1rQp!o#AoDNr#Ib8wKE'
    '^A2-C+IQ=<nR}mm?ei||0AN0vkAgCLG>2)P;c1dRQXF8Ya3DClj!2`;fXU<>#6i{Mw8)S;O~^+MNYVXDHE6AM*19gNP`MeT(*a%CDOE2d2{7TfpQ6DHbKjyb'
    'vTBw&>cq2PN0zP?!#<~=RjV?7Ix1x*?H0(cXDe&3xv+)m`4CVfs<71-VFwIDzG>vbUuhnS^K1{{_72RSIu64Fz7}_Z7(n@ejDwtB$re6u#Z}Vq9uolL3}0ob'
    'O)H@f9ml!?btD+UC}AGGH6EUrfg!`}FvWA#yTP_d-Cib~PwB=|bw<tdEI9&oSLf8GzS7xYZ)asMmrItNnS>U<@CqAKZ;Px17@#sxhaPt-01A}a76ZQq|8>?}'
    'Lys<&0Ay2M3<QV1Q{20u;9xe2aZkva;r8{sueWNeEF>6EH$*izt+oULh_1XN`@1j*?!NWl#s-erNipXsi^tg!;e#gC%#3v%P{HgNM5oGDKnfu6L4eF$jB%H?'
    '?U6^&ts_)I`^5-WXa){yvkc};tPmrbA-a5TG(73(Uv3a8z@kWp%$<#5n~f=SXPe7!Ul0ySnEB4iVLHu`nPK3FWQbu2o2zR~hc3pXj&Ck=URaKUWhTh+*Wd*9'
    'qIn;Go}|+;eFUN)63ZUf>hf@yj?&A^TQS?5Nx;yW7+JY+5=K(T5!(=})3Duwz3R4;i+%n7Eg`*Wx=Su7g&j%a)nTOK%*L7`J%U(>Z(uEMyX84bH!45|31HM<'
    '3>t~-0g(EbEz#MS5+-Pjw+~zi@5C}yVsXAy9S4^@C(;Bh#UcmF{tDmk@yafe$}krnb30bsW>TvRBN>I}nqUh?Z53%5_NK}-64itGRLe82hj*||EgDZ8-K%un'
    'v+)jNd)@=lAs@%UbV8;<Ge^NzB#DM6451i-Ku?7pAI!&!A_v-?DxaCgDy2=e2<y`)2Ga3q=5AB6;x5@p$e6Rsmz;9Gx0liJUTV!nShR7FQFt+m;woDyRGh4x'
    ';9fdSM;T^NKzmfi<nT^0N4@BT?cVSLfk`oB!apnkj?^5Pobu!hNk<c@e>M{La@X6}a#$Qar@oRA;F<Sz-PKjlA*M5BuHY3_o7Cm4Q}ua~xq{@t=GnS;3|NM*'
    'dw91|YJoy`XATk=!P5zK6rpU_T~@}z67Oo@f;BA_3yr&3vd0Ya!(5qlW3SCYDd-4n+D6KJdW6Q}zb<q>N_MB&qf;#a7L~=~bzi5iM%kkP+ryRngd{ttR+tD+'
    '(U=`*?!&(Ax#EcpE`TWR-^yW*qn;e);{o*Y%$pn1Km!gn;85l8D&om%cyUwgeNQVvc>mq2k$8=qM}o7-n1W4g8>R{dqpbzC>$Ff*Y|g=uX#&XvXacUh7w<B3'
    'r3Ecwvz{j(;8jJfqy3yKn0PK3-|o(#J1#<3>e2v&3EHPxnKfKJ5ddhrCU<KU-6zvxC*8@1AX3Oq!60mJZ%<BU`$AE=a^*^3`O`EvG;^uy4*ROEa+M9U7BW{u'
    'IXf9<E;ka^(YKjB&E*!=0%!U=!&kU9RzCMeeRJZ1o~k#YR?=wlYM)=bqwE-3c7!=t)cf&(+LiB=?OLD+Z7b;tBObEBz8z#I=*Um<y>T(kk`(XrA)ib?W3eg|'
    'ymv{OflD5!B7*Z}5^e8=Je7N?$J|G;(-YIfxfj>COnT%JiEG>&z02LW*7;x;FwNUqfy$WL6V}nV(Tv$6Gi6+3Q|zI|Fm{{pdh6MW+DB<$BfC^wYnHO^RIdy~'
    'J#PMq-G-c=ak-|wDp;_M&&J?ca<kV}Nn**Po{k%)nPb<)8@;BxfD?U71Y@&{cULQe-PH!_(xro=bh;;k#kFg<=$56*|JhTx)ykjjwp#amj9}DPosPZ0e>1o3'
    'D(u;F!Kh<9`g->4aMUq<bG@MUx=E_%&=`6ShB5COH)|L#Fx!<4n9d(ocT;O)p)bolo8|Po&v2!}^mKT67->boJG4xl=z&cRpP<OUsV|1Ek{<dwDDSRmymfOB'
    '`gVGXY_j;rbYb*t5oFZSoCrNlAUQ@~WOtUoO=y)}x~Iq>S1GZd>1pD~C~0diJzIj9R1gvbdoO+)Wn97pBd{E=bzb^13kwp#HB&saF^1(E^}8uu!Rj=6O@xJ@'
    'RDr#5jjY<JI%mjgX|Rk7^)fTyU^v%dhI@Qa$(E&cc)4SriPQ+n9f!!bWZinWL^kpznUYj3z2Ui0R(uB~K(@>xJzYikS7U{-BB|KzfZC)m?>;mF@f|Cj<6Ugt'
    'O{oce9+qbZ!Rc{=pGN%Ky^{wRMyZ9%RO%a7ibpA)k$1$8qk$ooQIm1-U=Dm62fsDuTr2Lw!zT167Qfc1Jm~^HVZ%#~$L{#M$_p&@_N&((bZ_2u2J2y`^(eb*'
    '_$m&rGkfwiCPnO;d)YYSiY~ACMfv7Vew>eMgZp?Cl@a{;Mp|a1kiSKr--Vs^Jy3Buqli#Y>tnQF9N{}}-Mf+8x_j$Ea`U(DUVCust-JSYA(%0YTKsd%QvvB<'
    'z_KPKgQCy;@L~eEKB3yrwg&uG&FUP0)Tlt2a-9=UzVFfT+-Xuwlk5@Zis>fnLYd?7kNjzxO@=U866Vv@aTMfj9lp)uz`eN2VB{K_n|h;0Q|l`y(!wF0EKL30'
    'Tet24FTC|ma{aAqv<=gu3vsq5*WS8&^VS!wLh=9wgGb^C%{`u;l^wDZRo!HdxAEV1?%c<VxtTqQ-mF}qCbURYUaHka?}{eNF8RhCjH+Xsx1*>n;5r3N1EOTe'
    'f&haDugcrQYc5!6x;Fku98iNENW*-{#2ze(WvF0n-BH3^s=%irCZZRRB4l4et5bk(RMv7B)5W+0zVFtGbq2c6+PoNCFFY$a?qsyNtRC1D$XazK#ZlOeIvChQ'
    '`2h~HY%<74w4;hzHXK~yOU1N&G@XUAt(V<|irlx@9*O)dR;LsKX}1cqYo#%;k!cI|^})Ljm)@V@QWq*sbm*9E^qBGC`%WAmpUW&FCGUpNJ(^~dUb<mcdF?1T'
    'c*Z{6XM*Uiy`pZnF~Eaa090_<At)#&M6S+pYjtnkHW}bx!Iqnw3bl>4xawMHBb$owLakA0tM=~eNY#%YymkGpAbhV#i)aHzpVVR0ySMY*?O-Px7DsVVNIN-h'
    '^WM1T+G;R$(7wH%A??}ZF@fHJs$)_pfRP-hD<3Xxy9Roj0-49KkKJLm*^_Du0CI<18yL3iviiY`X))hpeil{qDQ&Wn+cM_96I>k*S-52thtF)>6itn!8{h`u'
    '@RiOhubDt2-zK0UoC^zOW3W*%NbD-gcnZtuT;?nc+>RyTkMlES^nve`ugF!vXOvmm@XZ<n%wr!txHzvmZkOY?T_>=pu^sZDjhRSR*W7a)P8$G@j4^C~sSQ(b'
    'l$J6=19(#G+^iH6JT}g^uk_vYpc2Ue0u`lDLny8(#!5u(?ys?$SHx7?QR1kad9a$`7$S<#dQs0Mz(_dmB5cRjVpH$CmiZ?iACe0)Hq9xO^~Gu>RIrNqS`ovh'
    '+7t_>?ljHzU_%PKft@*??0^vzt>ZXXn1IADvEX>dEXyXGDoqcsB9N5eR38KnOG>2QGhtDBTtU;WvFDws(@;4$78S+~j4~XR1@++i!!g+wF&w+a5LP23C^R=`'
    'n`5OFymnx3$_ngcC&hTc>_?3A%-H2HMPNcwD&)b8I#h2yrGqnO#ay1vkWpAHCtjEy7Wn{{G+J;b(36~wl92LJjxdZ)My5q|mm3NeXR0Bl##0F#Vd^l$I>l;S'
    '@l89r*$7Ev<^env=X&IV$;%vt9{=lN?iLH=Y1v(BHmKO(l5`U6z&`qbjRx{qK1}V%GQZs7FRWx~jP$S9$SO2v`QTph*P}thgGaN4>EfQ@wL+n(^ByDxIewGz'
    '>bOYurs+T^nD$ogSQi$?g*337Fv(!rQBmZ&jHrIW<Pu4};|`Ziy1Drpo-iR<v@<OF2ex2Ovg1igF@uJX+$4=V*obMt-eTKF(6m{34GFe)+FaYh%yF1SPC03<'
    '6jPg0-*56ZpfTBu+J~D!e~^%D`x*eXvp^h#*#nNB@4);VsmCu@)g9UD1hnVX?iaeBe>F09Fk-D2#OvrfnO;bJBI6M|=?YrC@`W;(L^Lnd^1p%l#ApI4B=bu>'
    'T<TBes$^twQN(EJd{*?+a;AdqX-%;BV1Bi-bw>(}9ctX6HNt|n!Ed^UMSPDsL_a8#VU|EEJ=JJO3h+LZ52?kNH5|pX*IY4^Rrd?T397<amDXK)`Ivu%U!lRl'
    'R|khk<N*&+r#S5}>+E#`44#a5EOv@rnypNxg@}dF!`))TR3yGSqHr0i`0pjp(xEJ9ggO4}FKgK!05lpf$aI18mP{BiLa|?;$=-neq*<2ir^DTcV@iEpy$Fv5'
    '<FG~goq}0y@SA!r6-HBrf04<hqb}-{()@U{)9JwEg=xeD!OHp;B~ar29LiA63z<b=$_x80JF8Q%)e0srJ6=~g>I}3XHc6O=4Yg`is2NdQUqV<>4sdT#(zxi7'
    'o!LDwkQU@}_LVF_{ML1HUZtqIGGN>K;GS^lRdyeKdf*YAdqjDG=MRB-%3@l@J{hO29|hqyqtF8?xE*bfH;DmZl@;&afv0%YQb%Wdckw=IeHq#xW`YyAr&4Z8'
    'I{K7w!&wB1N^cWf?U5lo&`js!h^F%~9T-)jQ`A9gKC<qLVq4Z=EK75V3=U+`tOI%8I;F_kqy4<UPXUgiIudqM?I<1aw?v~+q%6eQL>NVW+I@nW#B_*yV37`R'
    '@n8X+8Xh!_A2C*A$4|54bVMFM*p?fn2zHj<g`WeZ{gm6)SgOL!9lT+JTlJ-ij8O?luq}#99OIE;hK=6vK`k^2ZEyjl1IwsZ&FBoUF*iX+KFCUIMQYVKNm6zu'
    'r>IcV33{L9lNg><J>#7*Dcek^(%CbkSK?g4J0v+(F|p08+v=o=wrOK9&1r4q^Uo=J_#!c=4vYAOSVt9k?6+J!I}+Pa!<(=Da^-9&_lQ|OP@#DQi!1=fO=767'
    'WRLKo0)WJ898@(9JT(sV8hq{HO~nF2+x2Dy`;KWktX3TH;tu<Zi55t|<IgTTKa^?ZG3#q-U0V6G^x!dWTUsoJOq_IW2qoLY_aKYjW6rbNbS@<{k%&F*#KBIU'
    'mTr^IeDZAI4vM3(2eJc>-J%~dXr0X9#zGr5b-KgXt#(d9hyO~)JXD$s^<fA#K#|Z<IWdi|XVWyZNtVu%gF~)oibt)DawAYh5-dCDrrW>>#VA<0qJ7-3C5xIY'
    'Ls8~YmN7pgYm~-qn1S34kD2NSdl>eu>`{w6GM+2A+*`R<ur7xrl({|M5m!f&T3dS}ToiaUR}7z{hTS=(y4|2w*>cv>Vp8S(!ycO1=sDwP_1M>17cOqhtGzjG'
    '&-w1Z4qd{)#9EiMdIzQ%b)g0*aIO!kkq1j2f3T&gO71%{LkKF0@&E#Hz;cy@<W@;;vEM1?!QCtyEVDcuod5u{lvNGqRW^d#p!%^1#6xK)o=yZJK@}_z+?S_%'
    'X`Q%Fx2dTaXv&96##-C3LTso3FEKMjgFbVtI9Oq#2*>I|>q))4LAHe9E#kL+HM1;n7QgB4!zDcj7<1Vt2{A*GbVMM{W1=ZWehPo-w!~mt_@vacvp7WL_;kBY'
    'VaT)T6cr_Q2AP}hrz|))9S%wVL-Qi{d9jo=_dq0-87ZPb;wfp8AcgUq8Xf{yP;xiRl2c*06l)UzQ%^e%U+mcw01E!*fyKOVb7Z$7ovb!tpRWUM71OLVT_4)4'
    'm!RxA@=md;FDerC@chC0ZT)+8B(v(IbOki4A;(o(5ohGFE25cKaRfGNGcVMK>XJdD2+fq{=sB};ei?~(180GIW~!q<;%7neOlFzm9%|<#sKzDSgXy=eRM_6&'
    'Q(1ZU*fl5c{sx?@+Y*?pD&IVgIyJEhRQ6G8&9XxJlFsy84>Y2qsK~)W(CuZfZ99k0shz^pP7Q4uv}Rig_C_()DU+@PLU^Hpvf!!ZjgTVK<Yd*tW!hg;JLRU-'
    'rPyRg1%72{m4s*`z;m-u&^#(7&q;VoumEX+{DO$vQcM9r;ozIP%C6B+LPR?~j}?M-6j!<NAcdkQRm2dB+(I?q1=c-AaUCvOTU&~3+9|&h2i$^^*eWvKXrL{O'
    'WcsMoUPTh<TNZ`?ae(y-2lW-$4AZL|x~q5*Qy)8rd3F@4J=LdPtk1j}H;&eo>Y=6q$1bQ{Qw9;#$MbAXTjyM>_t|3f9%$#3)ijneL0pPD9il}|ljjL{6D;JT'
    'vZQIbg4p3a5lZFMHhC!Dwkn;rgRpy@+v}=~3b6Ocjx*`3VM$T((yfA<^0H&u7!O1l_`(C-S$}8}5`1PVjMcPUX!zq6R7^q0=e>!zh$wWoE8+}kS$E<f>PrdJ'
    'X^2G>E}SadgZw<`<579h@ZHzO#VH_b)j)Ob#`>Id`<y)GD?gOQ=-R?l6JW;{2_+hH37}ClB|5k85(bKHq;}v*TT<wQ_U19l`}@q>(MSFwwr2Jr!<w&j!jJ?y'
    '?j0W6;HwZDyUSSXlHoFD)}G*C^^oTSA-l-IA=gk|53D_mHaL~_O2x*z*YFFw3Xr2K9#EPdGfr4=dZ|QA%|`Q~msJ$O*o}kBQ53Aw@vjcW;OjzE<0M^a(l`kz'
    'IxY~*jOT)4(PEV14ljQ}^G6@g8JawYyB7wpXEeYqOZYz(d+r<VZW;F+8Klr2UsfB^`CvXlzl|cefmL@Wm%CmXgHkJUC8fhoS?ta*LQO3soYC-?{Pg<K3iRJ='
    'E4Farw`z}X+Kl!lJ@>XR4n&eORzN(Nt?|r|1_SABsO)LV7zGH|qi~zZmTgo2P?vGZXQ$dhZY#}hD`_lY@u6Mi`wJlO7<L?U%!ljSMZRzM@o}wLMJKFU+fYn8'
    'bV1<WyqIf2FaG+{7HVom>Y*b)D#{)$RKA#FigVlEgVfGBc9Pz{ZM|wF4h+WC1iWRby@@B}q^Y$V>gHRQ(Uu)@rIU5aoMu;6#7YD$D`%GIFb4vA5V`hCrlf7_'
    'jc;>XE%P-98gRXe1Ni^hd)MtYj$~c%RjQ>mTLrmTBp`{BxHsG_>tge?ZbxFbyNBck3P6D@6F|WzfD~<OzsCHWHEYe!S@R0B)|nTYN0_+ey|M}ylCs)|-n(t0'
    'sxmY378wy4@da#_%OY4z=J^602?cf`#*(1BUN%Fl&Ql7Zzffy;V%Z9H5|Rs!-<}7PG#wO->Q8~;D5*(A>FJ^eY&LlIejh1uM}ZyAk`ar8(?t51pbRMP1`yl6'
    'NVOx66y8v5yA-_}$H6lw$_xpwvN0y6%DPPl0J~oyrVw>r+>bNrkh~FARx=xS!2$%f?b=rfuyPibGGtd;%<ogIw~G3%HHN$2+6NAL<?aO^6}OV|Eb0{j(Pbc<'
    '<IsM{pHm!TZ!w<obwVpPLHO<lIzaV5;8h@kn$gE3pKguPAsla`>;wTlg#i(d!AJl+U7!~Sp5?Hv;0p;8z_YW(LF!x_9atjv`oIv6aWYVOGl)=4#<y)Gkv9|O'
    '{?2;n(QYIF5lE$^8p#|D{S?zth(Gsy?wc=?eCyf%>!i0$O>L=FwvCv}uyBj*d2+ISj*ahFBlKN*LAn>wj;r4uGKqTmbBC|t@bo&9e}@9`Rbo-m4pCp?Em{Mu'
    '^n(T7%_PCa3~y^lq6y{rH)0nW!iXt`F~LabX$QZQqL4^{v@8rdyfu#f&t)9DHD}$Lvu@2<x8|%{bJndn>yKj2(n*BLXwgC06{^~accdO_880|G#&7Vj2L*QI'
    '`<Jf{P`X<X54a%dZ7#BXU?L}kAtSaXLx)JplP#e`FJZVrzmg)KSio9@s5O*1@K{}P{KZmG35yI}&`3)omuuZt9Q=TyE~!W;eVdg_GRFierwTQ9*_y_{%;Y12'
    '>1HnH<Vb>vCkj+aQ;`ZHuEGi_8=$OBW_>KJLW1hOQXDtG25D7BC3QsbQRu@RhW(FFHj)x9j89bQlj(wAyD>m_gCC^cAjELiiZ!?aumg~dVM+Yr=3oL9a0qHg'
    'f$E4Ie0+2V1eCNR$}ud^K*WVFr{Ab5uBvC1*m7UBES*X45mDEMj!*{`YqUtyaUs`J<x-8ru}|6{DXpV~;|HqjWn{%bC_zkXLhkx(X-$pXGnkY|Z$m@IP#fgO'
    'l#f5!4zv^WUPEqPVXHF-^K-rT(=q`Rp>d4b8dc2RmMZH@)d`5@k|taD0Bnfj0)!1kZgdeXfXvG=A*?25uPrujB>dKku%RcwRcx#}_)qCE2}9=F+fK1N?5f%@'
    '1&4B7j+f%M_BQt*fvTnDM(LJdNsW4{dgS2{rFvXbt8L<~f))#_*L_qe#Z<4|sALT`>;}$DR(G6?b!Nbv6|ro8CqaBo0U@dzptk+f`BtO<+J1G;XSnDrvrKyh'
    '(~X;@`I@ZxE8P)%?S}pT{^QHHt<JN9gO_g+Q>63!^|Oy3Uw+icKSy`c1Wq%QzHMWBB;R|0nD@L_1yJI1={n&Br(h!hxHTeas{N!fD&cy=Y&_6WF19fx_!R<&'
    '4`G(#$bAE~q2y4NJb<wESkR-pGcs99QUp|1_tzRqKX?DVLMdKM)WHVZgCZ>n!6KzBs|&Ospm4jyYZL!P?IKLRf9)OCkaryR-5D?YO^-)Y1;YROy(rNrg-hXP'
    'qgqJNY1=4><#oUSbJ8K$KudOO#(;KXmanOaKe9#H=Onn2eOc9jX5`~~OY_oi|CMacK8SnQNAnd5>?KFd$}dWfY)4HO63B^gFUKUw;BTWw3@B$kL5Rw%XrONZ'
    'iOO9WN~D5nX*kK&+O3moyp7r6=H|UJmJkjJp7cEi5ft3hVYit^S7^5Kk~U?&RFKIP_;AMI>%Gfh$4SRkTDiV_J3hVEO_Z9OsZovUWC$>(R5H`S$gA4mW11QX'
    '^onk<m<&=+>M^b!l}J(J(9U)om&LKlOt8KV?GHIkn#>Fv1zjiANS9)o*^nq!5{|DTj80mPo|hMw#WcmR=P9Qh_NEsk@R?Fmjo8iBY}qlx?tu}HZ7i*ZyS4-<'
    '(NrX>HPTU|;FofmWHWrv&vVo=*1V=`?vB!mTsqviDq#c%=@e743hTdW&V&cim||?Pmw;mOaa^G$SQ&EHU@l8(XDh#xKUP3m2m`<FN-{F9GhBvJ1!({SSZ<dt'
    'GP?4Uu94zY+ekD<1T^I&qsNlYU<reF*c~j(oXecGmw{FeJYSS!LlEcc8AKZM^wq}~?>UJuF^Mtoid?3h&nCF+iU5V{auXmU=h{M7X7Z+1QDlP!#Wy)osMT=7'
    'KKZKVs_`yxr_s=79kI%=2G(&@8U!Pon47lk$eCfy8P&EMs5@cWwTkqrO0&&i+V;<A$)V9MJ&7u~aa;Wr)v=ql{WDs+!`t-;sx=7HY8ib~y^QV#uLv&}bh-)Y'
    '8LO<q+BH}uiC12gdusu8V&wv>^iyUiXa{z+k!uuKW)P4SebeR$L3?kwwP{hfx|Y&*vTG5-hbUnh5I!C}`*48x^&byD?7#gUN1_A)AZ`OJ2fo9ull7o`90$)a'
    'JB3JI3(7ki3kG?Aq4jC$_d{+~S>ND&g4>O<9<flRKC&o-(h)Y!w1mZ?PgyJxlmX5rY5y*gb4*}Bv-ykU{(MRr3m=%zf(|C*5-*+iV>M)EuR)3#6?u>ZGV_Nx'
    'C6&eFIF!y+(@TL_KRemRbP3xGVY3aNp3cXkyH3FdrUVxu#S@X~7=02GbbOknCyP-sQ>}6lIy7eKkP|mz=j8kHDC39;7jk{UAl%tTu&5H5SP1LvrKd1-1W&~e'
    '6%N0-c10k1c-S>;INf9SEPPW=h{vJRIt4gMGpY&0(=#2#c1AxgBAXtflieUs$=>XmV3L95J_-%y{fs)Ud>qD{sgOM<Pn~>#asgwAZz*N-VfVkB6(8e9=*ZmH'
    '>7aWoRx@WH71Z!D5;Vm!jSO$WzxeWegwi}m$s|#PX7^eiRo=oR5zdke!K=0e8<TPO=w4dL>vde@<O!QkV+8>3-Ta)`!!YqV9S)t}a^6MOATHRp6MslVdy%>X'
    'Mr)XTBCuzcU<985et~H`3PSwMkdx_5Hw+z%S_gmIndF^fI?Co9_6#mK^I^#-snQ#DinDCmq11^2)sZrr5QG>~Qp%Yqi%Fu)oFz6X>JvVZ<q-J3P*<rcC716j'
    'oXM>3d}5eX$rdVBld!9fEyF8~=6vAXj@0B7vynBWbGM2%b~m^1?UvW5$vlbiWKKq%5VKcMZa^fU4gYJDZns5gj7MIAPHue3tpMq{Wdx>6Xg}!+FY89A7j^`R'
    'sIj~gMstVdI*xVM^trtM_Wj=uK2{&hjunT6b`!d<LKAh=DqAacM@wp<`LJHqq%@3pTwP*jb(v*@9j%1+gVq)4-YkMk@Ge^bayTPBkKIdUNr^BZ<h?nuA>o*I'
    '*nPH`=Lbj=u5c7Z&~9SNDC%X5M(;t|^+5mmzwdM~BB0ZO5_qh#Ii)C3c=#h~Z){P*q1t~@{P{SH<L#4CUKB7icj@_-?HyKO4O@5O>NL|gSTUSP3@8p_8T$2G'
    'NUV4|SOE-d6ibVk3o~0pe!{m1Fzekz*eA{7?w&ggTS1d$JYFDkIVkt<<2J!gri1(UBe&2_MIl+BX1HzhpgZX5?H0am{Ohf&{h`OwCY=94keSNJ5iQe<(3sU7'
    '@JZNFoS`dl9@!>AYZQ%Zc57ZsTZtgtxfcoJ`ihtvxqhS|8PP`yhaW_c!>&&CHl34u4@iJ%M-pmph3;|LkzkA<P!#b3b<a3<hi7DvB_KGAVeP@%JY?Y#R=O6='
    'li8>2oCNcvS8O+*Hh1FYgNM7#hO(2oSoXL}61h7F)>Cq?Io_C`T*khoFRjTO$V8$S%+=DxUscRl{<S!*n=t!2o?$S4*<PV6X0=CwCzU2nk>(r(LrY5rd5G&6'
    '%Lqgr-RKr_nVQ|N+AH&0`O0Cr&I*Syq_~JK>W>n~v%-2@KWMPYx0GMBEtrRvZP_r<j)pq!h5_T*30i#6pFv-?JdoKWR@3JVfgD?5S?6g&*-shSMJN$wrtAx6'
    '?g=9KE3!&Vz=<c77X99(q2b_spj8hx|Dr&X`3ZS?%SJIk3J#X8og$kD-SA5?UX>>O;DS=v!>;6;1<JXR4))ybl%TF=WU-8SevbnwDdJ`T;ug*|%Hi2L+u4GX'
    '1;`|tc^E+VAWySuZvoz?lkKx}roHj+E;-8Sy<4@cXwQm-AYHWw_y!1894W7~^QZ!3<H%`S#^sUYzT?^cR*;9q3FKWJe%Aq}NuPl%7%TioGJu>-@{5U#&>W>F'
    'N&nJgq7)NUo;$4xqg*JwY-kT8<J2lE%-}^Z#7m<>V4&l%Gs!(iAu$2ay6`kCDeX+BUA<1R#mjiL6BnCV$VTw58ErY^{AIT2oWjQG8-i|Lh^V<iKDp<ZFC++a'
    'Hq*CJ)@9mBcAiey>?5sHjVfl}Y;<Y0lj_e`iv}r<8F9t)!gYTx*p}?SFJ7@$D<Ne)cYQb2F|tP26vUu0(kOZ0<n&K-8u1+D0}<Sdp96yxrxSU^2-B&s0O6W<'
    '9o%rbhl>$vfJ*YAt5v+?n&q&Z&tW;7<`*1DJb_>K)C+N%Z*L5<)Om`T5=h{klou~e@jV^mV_aj6^85^#J1pl}_9^ywO{{XQ`(>_lR~0g^bme}-*X^pu4zT#r'
    't9H>D*T3Rw{jpunMOUm2SnaueFv)fooUacu|8LfxUx{=&mia(75;}`xSu(0%I!aIq!GkWY^6<IV+xF|cX~!Edya3KPF5eDai(FLtOs+?1L`sH37LzH}B;&mY'
    '`*SnYo1>g`Dl#P~GJJui8@DEW4|<I2EyADU?aDlL^O60F*{9hm?K4&T&@2pe-#{pxQuGxN{r3$BRI!J6U)@*x>m|xz?EQq2Q6R2DOuKXUxFVT|#{qV8h~ZjS'
    'S-<C`X}44kv+%G(%NZWK5SM7ybNp-dvy%5Fi;-e$v@J7iSSlR_J}*j_%!Uu^SX9n8Vy|KL98ZD6vU_sb_mr5n0gOChr8KNs0gYqtbvZk+`y7Ycb#BEjpzihh'
    'vFZ?2RjHHIm@8?kW#Xn4sJet9&r3TfP~I!(Yd7FLj8QVAe1<whl+WE37p#rDfYCZEJ82U_RbuJVx-%;(swxtZq1F!aEMVnqPdVv)tPs8x_gd1m*KfRUtl<N!'
    '#761=s}H%d_v-KM3Tzg}op|W+MJ?B1sSl;*&BEofb$-DcxE=e8IDXj6N*NN;@{c!lRwgguzfIs<_5_ZC?~)=lB&i?aaD5N6WBUG<h3U5Ena^Xzyx1~gga!90'
    'eZEBGZM*Sk-mf$r&hr&Wsqijp_;jPT@we$;Q{RyKtn66B!g1UBl@0Bm%c?d$c0>8}<(>ecXaxT$o+>5WsDC;B+a1iWKrrDfg<WD|D$<XUJbBwkEuwU4Hgt5S'
    'QOa6cePXEaKN$pGj(4u|ZdyI6MnF!umPJafi-InkxCnwyW;97>V9|+c6U(edEp*m(<2d$3)ilbF%=IPCN%N#Ulc-1EJrBYO2nf>b?77{c0dJI68C$d5MtoyQ'
    '1J9^O>U!L1CJLzi(qi#8r9BlY!6T{=QhX-&cJP8ztr7|0YWc$JD`96GlWe!cjjOG%A4xT4{fumQXbO_XM_M{%n;|L)bA@?=e-zrPI<OPx`Bs@oM4=3p3~nL<'
    'N*=2e7t-WJw_-eHu+A_|N-{xFY?e=FXmC<Xj#a4Y7FV}?OSAV~9{DZ$vBj3wM#aV{gCm$+%(H~-Z`x&HO>;=MddG0!ub(EaNa6TUvY#`T1gRjr&_PWO8x`x{'
    'X7M|<Ap;VvAz{U%V!<0^MP)+2vByThxi*hgl4AQcKdg-NGm($7%Gt0!X*<0#a?9;_a7`-^Z)pP%LdnDxX;(+Mt%uyP3c3-vuCQXCcqWht!N&P6IVI@BX@br!'
    '30>TYYh$)FxWVoIgeNY<kmE@2U<{e0RkedY1|?yvQ7LQD!nNc0u?FR(Tq<sejGnTS^wV73@XOQ09e#Fy;BSO!Wn$|4Wud_<r<Zq7x$oc9`)*%@NJHle*%h?A'
    'I9mz)6?PDhxZ>2Y21PlHME;NKLLxldpyG>~7>J64xZ<m2F%KoNsgmTHP=}h$6fG6nD&!1z@~lY+Hy?W^EM>i^Xs<ml?G2}?z4pYl*PFujrZd`Jd6w&<Tif;o'
    'AMPA`7(ks;;$>-eovj-F*z~oxEMUaLhLMxmt-`V2SQ_26G^8sT44fM#Mg*2u+SWPai+ZVJ%4;)Z8f-bpo{_hd9?$uswQy54aHuLbLO<epZYvZ%><`oS<I|*A'
    '><`cBkUJ;$H0Mm$4MyehiJWOa#&*vwZNFp`t*{F@tP3a8YO6M;&S55PgXp?2y>fplLiPAl-A@<!#Rf9ZXU~ayzJBAq{D}gY?lGUd$v$NY+q^T+KxYt!Y6=M8'
    '(18*A#t1)ciW|5}K4Aat3%kFx&6_L#hs&;A>>Cj#-O3N<yE^b(z<82^eLY5f4h6b4(|)-+*T65S6kKdYGR5$_wD~K+y?$XlEKzApruCs)z2JeOQt`{*XDz_E'
    '9W^VdorUYX0{P1<rPv!~?$p=HV$3MXhuq#a2aBvoqju-{ySK0Qzwf+y_W|3vy=1wzOO{d6gR^u!n*XQ0n2~9#NBK!Mr~Po2iX_{JyBe9eFX)=wDTBXQ*E;^S'
    'Z{F(6R)BQjDZDWUas1~sH-UnU4N}cs>p;TfWPxtL((gaO?#IgGMg#|>8TkuQPy)x5#BuDA>q{rtF?kPfHFtIdN{Hccr}zrGu`<6W`@i(Wla6m#nw>jEoRSBD'
    'guByV&rj?EieKdCsl2^qRP)?uO3Z-SDPJE-?x*Fo1hWW>Y3Y1CPe%O(2KokETv5fYsO5UQf?jg2?e26D6WLP*NM;_e9n-UwZjllK7ENKm-{II$qZ~~n>cUYL'
    'c`>ls6D6ZX!jX{M7OQB?L(F|h<{pe!svF%83v`_wr{jEfNqIQ>rvinkK!^1C{4ywDcG1}gJvYhm1G7pLDhD{Zg3M0LKsLe{vgYOkF_XfIczXFMLB9^`S-0Dr'
    'Ud~TVCy#Hx{kEJ}1tZ+^**vg9jG~f+d*sw(iIK*4ZZvZE{@K9~or8CHO8))jhl7_dIzPVL|Ne)Aj}5!RxojIMwP;zP#)Ef%eff6(e|`C(;gSgg;&{s8wLhdV'
    'Ejan)mkaE^k0AnohV?q3NAqMv0czv)&n8pDTKEtnQ@J22<ccH7+l4b@`{>Se?%Yjc1$U(?klN2>Ru^H!X7Nxy6j!UUTH;ZFD5cC7+YffC1~6uJ%7RsBHUZof'
    'Ly*Nj5fUifQ^KAbrOBkTn07=xjCEX0K}wS1IT<PR^~^d0dNkR;Hzs9KeaOkV_G#<LPobD~&k2jT3@LN#P4ERT{<TG4nj%%04jkYiyzC6q1d5`OfDJ)JF6jfJ'
    '1N=8C)D0hsi5Brg&xISA9rYIY(il6%5(B0s-`e+FYoqI|Zv^$NleX_+<h@?U$!;5eZ;19&Ywj}*oO)5cGvpRHjZulm(~gg5K8Oc7tyq!Ek&a?a<A@pqE^bUg'
    'h9^0(6jSU&(a6vt_|R`#nyqc>!<>{`9?>}p_KvNtk<isHoA#uLRT+aX3Ni^Od&}gHq3keM5xJI^$QIVuv-~LxVcAL}_(n}fbMM$|Ck(MU<4fwF5H@WF!tXus'
    'K7P$1EDuSW+A1N~0;z35asAE|eTnOLsBd!=@rvd^`MGxo!gSn;?nt#_`)3q|wyoon9>vqkJ9~F1v)v`9UTf{N9#SkCN>9N-nv7}1<C7oQ)!`K!yP%=;<6sVN'
    'K>)lPe0=}nA6u_MZI~44*8Tv=8D{BhFL*X3Z--XALncZ(FAoeV*v}$mDVpT9SRblF`unNK^^d$ze=a5&a({%3IGT{`NJ|$~S0y-(W((<%#=_<06t|*yE7jMC'
    '0}zcSXX#1$37u1B^t7Dt&l*XJ*if=6#!OKcqrel>?yJlyj}A9vGsOR6?;E^4;iu$e!4JbuXOfK5PKP$$-`_od_Fd=I{_B??J8z%8u|ckR&mt}mqF0OGF%9b-'
    'g>(&+oTP>E6+ejQpXT(HK(~JL>>r&s?_Ruo&E~7lM-Xkfu{*e+XrXN>aSv;H(0aW4gi62v@KQ>{oewX+fB6rM*b)9Y+q+l%c5nXMy;=I(y>s~A__uqb(Qo$#'
    'jo%Lc^XO;>V%pKs^w-y)@!xMhAOD$uee)at{w?@3epXoDd~?VJkMF}*#eMk;%M{;)fBw_a(Z>e;ZAEmbMT7R)<w+k;rt_A)9_UH$a!y$79kEQ0CTMnZpN76D'
    '453B^BM@a!l3C1$TTiGc8<|1Wo=zEwLT-XUm<TC@g^U}am`lR6D9j?D{s9pvJa3EWJb_Vkn#__uflX1m%xOvpix|K%L~TCS*k9;rbq*hr*P#5cpnNbGr&1z{'
    '(ZWE8FG%`E6&Jd}3gunuq9|d|kWJWIi&m>h7i!}wFkHvq-u@N4jiJ-<@*crm1VxTGQ)E!d&frPV-1$zZ!^khtwt+RJ6AHvbddP8MmM`GRqrd9FFOUw&7ZT$a'
    'lathl__Z`<gmS~k1&-)u>;(eNfnzfPRpdeIuQX1)^@0OQJnHR~kB9oiF^k+V;i46wrOz)?pjxj1zG-lWxV(ybjuwEcDP%{g91XvB{shWk6!72w1%Jc2+HByz'
    'OJDwv|HJt7fBawd-~QjmxBu_|qrMfkJh9Y}wjM|~;-E)YFX0M?2SHB<o3!rVe_~p4U<vftZF2$@I8sFq4#VjbCN=E2DazCvtPm)gI3G;#nzd<Aci2Cjg%b{f'
    'nNXtILJ%uN36yHWU$SY~!|Y*2LtMgb%rKC(Pm>HCljQva{S0AO4XW7$+3C<;c{%RoqkxK#lNjzs9l8A@{J>?af_XV|T)AFy@gfN29}Eq9p3E{_h+87|UM#nU'
    'XT5RDC`rzxyDDNGbZ|?<W28n2+GG&L=RhdE0FiIcH10!HaWWW$p{gRH0u2+T!l)hpW6h2jN#|Nr9mApLX!ecqbC9a$vqc|~8#snfn=<GYnd0(rTfzvPr=#FI'
    '{PR6DzV&h<uq5OZe8ez*vU`62{-9;pvpCLC*_cif9DNLAX96jp9lh!Tws<g0n<lfllZY@N5JntNir^P)fO8n2Nl_-gMKAUX5rdkM`kgS)KraO$V<IB`IBXLZ'
    '3dTN$GA@xwXo%2NWz=&FdbP$pdm@gEBBq(m{;FhWV=TMv|L#Z|4NqkuT7yQT90z7JfNu_u?kJsI0pJ}zeZ2EH3Yt$IKRx!+1l0tq)YED{ZRv`np-5w9W>&T|'
    'e(_I@14L>iXfW0ik@3F~8f_DNE#^nfC|Pk$xq61Ef3bzuIX9}S!YMU?oP?QJCFCCGERFE>1#Db_OBy36=d92`Mhv|vPBCoQKsr@%;3j$yNZZMbp|uGAK~`K5'
    'BdW#OAnd~KEMc;}PY}v7cW~DT7}E%tHU!=k>fx{5I#LW4DMWGe@*$E~NaXd}#k8Arse5$C(+WC0p0EXh^2IX_j6w(CLm%G$?cmkc6N+DVqB-e{n2li?lLaq$'
    'V&0Yn>-1n-0LN|@bISEqFy^XRy5XH66Fy;b##Y!>I{hEN!HJs>=jo^UZ{K>lj)EjqNU$kXX&gKgQkq0K5l6i<l^}EBi=J}qXD*o_#u0-9sr#!QXZ>~pyaj|C'
    '?JA>py}YRGosW2nVcMbZaqt%7)u+=DNQfvV^EHjR56y*C)9^;aijr-aCi93LTyKM1nZj=u5y}z$Ncj$k*EQV$WcKCK(BNb&X%r^3L5=y}hqvI+&rwRtKo}JI'
    'j^Ja|DCh=s402PP%^lNZBm|5?-65FOxecU@0~HJwGx5~3_9UAYPC}){A~9^{T1qG~HIK5iMENC;L?+Tq{7wL{_JzuI4wt;rl)?v(8+!O-!$$urx_n?D#z{=P'
    'puAH#AuCmBw`i%A%rAyP*IKI-;zoE~YJ+v@l6n$!^){4$g{76=2-*&-y=T{d90%`kTrRRAji`O!sJ8`p%o+{rG|UKU6m+5CdyXma-f#D!;2svk{|@K$>l6Kl'
    '-h6xSxGTm86m5dOR@d?ZOjl;jD&W-V>cyt-#m6}oi+vaj`WP<A5{0sV$qdDxviB1UOY&1&dcw?!POM6Wi(#e1lBOr}9`ccUY`2Y5Ir5xv?b577o|B^O__?Ts'
    'R$E-L-`cO1-+M;&+r9LWn^$Siy!arCFAbySEJFNKI8I#ezY7j1wm#W&M05|g?>kaUcaqvv8mh0NV1zp6R8StwtY4ybaV^Xi7Q~BIKAMixxDnfFv;b9l;v6)k'
    '-rLM{rGBJ0-j8%?KhoNMXaTOSAL^?c>xa=oML*OR-hOBSrmi3As~hWw(LzN()E5=El4KIdr$DX2!dPjC0~8np+5Ye_N(#Z{PAM(*o+p7V>-wlcqkTU{jQ!9o'
    '<9brd8}xSY>-k=AMvVU~VmExV!Z@3!<05Q)b^$?H18PfE2-t`~dE@rv^K9bDU_~{c`q9y3mo4PJ@0XHo*6N|XH%&W2rGumB*f$yG(NQ%U;v4UDAW0c4J51#S'
    'RHYL*fys5oD&+H0qatnkPfW`+^aN%T>H#K8-ywxZe2hjR>;gy>cgaAXPxgRTX|SQs_9{Y<FyrfF;mN3^2TK}JIbws`#A7FHG)mJ1g=Wd*jN-$TsZ`Yz3H$B{'
    'nZJ74u<E%RynT1@a?gPBLVui-8EVMF9MvO(Jq`egOD8Gb-O;@VUoI$5w%LO9G$DPaR?KRc7Sotz9yo+aA5or}0*q1_X6W=)W_dKW!Rm-lFfj%U2QZ6&hf4aV'
    '6TW^iTOmPRIN^bolbn&05sqjw!&DrIED@UpY#Y$202U=QQ0NAz3F8P&exr1LuMoO0dcSCaO-+W3-1+r$3CL)guy;<3o(izm!_qCCd$Z3H1lp@%m7&8sXRKhk'
    '1VuS!nwzG7{gp#O*@L~)iTR%MvldB&>>704w!7$~z`9sQx<9gW8n;Ovq=Xz-pPRiDrs8mO7906Ljxes}K`m%FrPUW{#TQyA)Q5`Wr=w(3eSA6>9d#mO-AgyW'
    'wNIM#J#@-B0;V;@e?gdqbT+@#V!d`J;UEii_0A^qa;ydN3G+Nsm2Q0BV(l)veKBqt<qZrys*3+0wq_EAE;00*q+>Baog1PUxTdBReuz*PD(2jnNpqLM4E&LJ'
    '`u!o5F%L$(R%izjpcxx+E^Oefb7|#q38J?ZVSp?n8BLy|Z5B7Nd(W2<lzlrGXDH%9jS<4IwnxdhH%O?BJ?9i6c7-n@cGC(`{T$-aSeEWY^}_UM*W)=ndvM~M'
    'IcGnRG)Ha(EEz|QYji^@FFImS4;_D}OkaoEW03{_K|8^`611!(v&)Lp{U>n+7iW_}`Y99yW*)&J{nSSs9u0<6QKO_kh9eVY0=mb`$SuwHxo4DWBFt+V_^GM|'
    '9JQa^PysQeT=p2{CF(jGiOtcZ+yo>@(5HB5O`r1|E+bLMMO^ncqEIEB-?W3)W<w#`5#6b4pjo-vcnix1P95SK`L?t-^mxJ8G*{aVm1sp3Ov9n&=nnjU{8OGy'
    '+z(3ZN-Xjcr{;&(%6{b;YA|8-NU~yBxQ$YpX;}q&kkD;<?8Yh~vr!S4CT~y)V8}lrjfb$ALCOrJAXjcuUv{*qc^<~{T$FUC!``uW#o)^WeiLd9$dROQXx}8$'
    'g20y114dn3InF8(JHMcuW}B7d_2PH$Ut+{e!hqlbV>Xraa6L?tL#QWAy82~sadC7BwS|^eqZ}WutG@14BkrpNo-C-j5>l2HkmqgHvCfO;ITs)lExye`BLw+>'
    'K(}R*9(8+GT51+ZsL+=2q?8s7_c=XKvS*`lQ^S*?^kiC1tl=@<is7MCt!{WG{0U+L&QnlCc#Nz;TW73<{ei}W+rH+JuvP-uLwNJ((Wv^uTrfrLicyz?`A}}a'
    'P=o~X=-_~i(keDZw_GM$5lxo9Kt7PnX31rEShC<8_L<N0iO<0I#|<&Kn$bj3^D-<><vojq%^L$*OVxdWM07S)6;BA#UVs;75=(GJ<Gd%7wrsA)4?Dt;uxg{O'
    '{T}rtm>$cF->t2>Yu)ydhK%Dc=7U(}hzxl0F33VY;cN!_?wvA&!eYlG{^B&9r5rhxsSI}Vf)CoFzGz2p!v)CQVL}Zr&0y8ERZLym^X$gd<(eUH*S=jtfxaf*'
    'g2A92?M-0-C)h6pC!wdD9wZ0-0@qtajxgkZ&ITA`!NY4A{!Po1eCz4yUP&9wF+gH|8fOL6AKDF77?5cDhoHIrUAu!|9YOf3UH)^!xROo+^}Hmw@T&9r7qu0I'
    '9=XYGE($_tOh6Afy0Zl$26WbaMHd6eIng_|bZdHO`|SI1)yZP|dy{EJ4IHm^!Etowf>^uXvOaXJ>G8!Zo2Mb&Y77?RY2jF2@D!fR+buMS&M4$YMvuZC10K~$'
    'ihh=f3mxh+Mie+uy(A)7y0>#5uoCS=sjD{gL(vPvIv?!BElG)ceW1nANb)_R5ScXH-iMBnLtJ!N){ry2AUQv59{UUi1TTtJ{;6%YP@#s}HrxuJH(|#8$SJrC'
    'r!6=Fe$|8cd5^?=dY~`|bky7gROrP#J5S>(3|1Ct6u_&<t1diQ4PQrjk3E|$rZ8lGs;aL_nHFop7@hKm&g3e6RZpP3<ou1W)?PXe%}t}n&lGiJo=8|T)>&of'
    'p^-XR%w-@1G$WEKR+pd9kGHR$<1#0uvP4$M)vD;fRl?0W!=#vh)iGzQg^6tz5Vl%58Y5ZQ<C*&kUauZSM&HHdnW*;0{;HzR_$#%WjSsaKm8$X@Le%`x0yyMh'
    '#Z)Yd8aTRx^9ek`2_Fc;UeZ6~Oh(2czgGnJwd{RA5H7n7Z?plob{lfAUdRD&pjE;S@XJ4V=)wAt0XQg|MhD0c9S#xrGSLCG`Dizc6j-%Rf7F1@&eo{%P;&<D'
    '=6?d)WYNRxdlQR(TYuUJxEV%ULEnf|g54*^8Ib7|CDDclL`Y39MtfdzK&8MZ0&>u~$d<PL;DG{5y|iyb3BKY`f-mpBOzQV2zv$>4&y22J5e2udr?cyPY;Y>~'
    'mCbbAw_lOdbFGW=kEwDtPH=uvRHlTr2HVGNZL)0K+oU4m(3_Q^TQj~$9Fr$CA=4FP3h8z5??Fxi4CiyTk{)PF==^k+FHTO0=N6Vb7YBFYa<zo9R|MNh38Z<U'
    '2Zn2`;kAt2GF*FaJ*;m%tZzN6SM#u5;@4;JQ*RK7%US(FB_1Yg|6JRb@5}fU-o%9i+j&ZC?rXStAik^;j%Wdj@eQ%X(6yLjYi*}l&;290)ll`0HLS=^M%i>*'
    'uD%-k*OjeZ9`n_~Ki|LXJpbY4^S|!D{oZf9vP?%}bI4JBk)p>D2v?UNgg!e@vJr})VzG2yC%;^BmRa#6D>_nxhlSHmJBd>y!*x3ApZZ>$XQ+r+Zv?7Fhl%U3'
    'Sp^PS)6yhQUvWQ2qm5>rL-X$YBRv)=g-v-wzB&L=q~`VCi8kkhV`ZY!XzvyQqVRiuo6kTjl!hwpt0ykQs)sJLoKGoNVg2-Az<}`MtN(CMVt1fqWcX>7#`{oN'
    '-0V?4W?5Jap*Dvc^St>w>CW=k(YtJ%oZ#jzO{3Mref-Lt&8kj1Yy9izqvS|V-%4wpQZIOSfs;m}awAp0<$YBB{#8DsL4J<XI>9SZHfXiucwNCjTYOD)nyuYF'
    'e&BF7GL9M?A#xvUZcut<a)QW$HR9$=Ye=^3@&a+l+D^OcF@trMli}RE(vs54%Hzc8xmWOYwxr=!b+(o%{La0sU=4Op9nP?%yLIg0ckZpjBs?2OZw3EmjeMVS'
    '+8N&)H#Lp#8^u$+Zw;PO&WdAnQgo%bRBoN{Cw3UYat`Iy7BWRcuu)fDQE)QKd!Vj_Xn*gJOyB9?iC$*5QNXp5uTQ%}|Dj7Wk6h?uS2rmmvK@>^PGDYGB$NO+'
    '5p&ZcSW-#`zd&hs$r3N$tD6)dOWh10ro6~=xq0ZeFL$1mF+yIKP~@T67m*iso`Wue7Zv3X@?nq!hh1mcB2dIW*{R8_3FTS6om0};MBLW#l+dNT42Vto{aoCB'
    '<QjEjh&q`BbtuTlN{JQ?7?T7TOoqvLVKrhx3E3Lq915H}i*ULooNG*uf<r1Vvvf2dU(hjdl9^M8h)UGWk43t8#=R0(R3~|UGD^kD-$Jw8))bAn&E2h5>)~-#'
    'R^9qy4ZE{)Z$G!^rS}uOFXRoEalvIRhwia-azx{woGj(0u7+2F?*<=Iz624vY~H6i;Wn$=s6bh&xzVCzSLG}hwt@XYYiDQ2Jxl&>XKb+Ng{O>_-{1u<qm-BZ'
    'WHIjXBx-xQxbvkM7b9fi3HB}>Ysjh0I8tyH3_N)xZVraW;#!njMQ)QaTK6g)f4uv|?e>do%okh9D4So(UdzSFX;W#g@$odzNAhWGI)u$rU`Yeqr7O<B&g0f<'
    'l;8z|uKnmv5N2^2N5L>n2U{XNQzKBxUa`o5W|Fcav7KQ@jG5@&GGd!0Qs1gs7YPnUO)C?iZ$S{u{Awhqx&p3??tkDUGhVDQJeZ%tmr-~bg}V=ZcT`YYHeQVN'
    't(XAF6dq`ZOGJEPOoumKpDKQWj5ZDq;4Ec?Ub=te+dLDu7US3Lc7IwFb9HGEV$gXMJZiSkGop#o0sObM^YBTdyoZwhTwKoKM_mc=UVl}0l8zRg=_QM*aC)3p'
    'T-QzV$<~O{>cMF>&&~z!6Kw>c`U{#i!VApZh})32ASK%2uOdkVwCjj{1gOU^@b7~ErfMYNlk@@~!Ox!;S^unaBEJEqtI&^PGocsehd{%0C!OM|pfJ(8m78wu'
    '?0WA@r$si(#RX>XQsfuHbt=u?iC(+r6_MUMgIPXB*O%o~?8G}h8e|1@_$f3ey>AddAn9emk233)N*RfPVK3jizLv9rWN_l%R!6Mack{lx@>N=W`1GNF+;z+9'
    'E`2K>ZrS(NwREKG(($;S4$#9^q=0L2b0w|g<5V(RxSEscT4W@!Bzdq@nx(;du?x28<;W3#brZIn*e(m=jTH$Z;@KMvF+&E>f01&tq4Vxug;JR=%4ssRK6j{n'
    '))%$_LnK^rAXwJw3+2a~Nr)W|5TcXftGHJkl+-wpr2cXphXIsd+9V-HULmUuLR3^E`?OKEiE8AHcGG?BNCWl<6$f&A_hIBaz}nsuDXQY&ZQ#$lQ4Mo#1M7^;'
    '`*~9OSlAW*Z{6pw_up$s+<S!zXTp<)KEcp{BohnItnFi(2FqNujBUI<U5rNCt%r{vKe(F}#R8MFp(3%>+<nk$J$d}-!4p51ZZ{)8BW)9-jQre<UOryU>!Ngf'
    'YBg{Eh-?DPACcXBYQ0jf=_Pa;E75OdhF?=tXq!vg`K~O^x68}iDN~lJ+vSz-DdP&lZQq7))6rIlsO_q~;wHznWtg_JCH$k7P<k%?!}+Q#l(frxNg=$Dd4xSQ'
    'ja;*i2W=m-3F+S=IW8G*@>vJtnD#`zPrC5Uix?dsUuS3Ov9VV0hWF@B$((d&&kWHl;tpm5mcbh$S5fF$ZX$0krJ52!4Tvj0BIJ;c@iwfbykarmC1@&TBH`)+'
    'nG8+ji$Z@&JW`0USof8t1=%1)U@+<9f)Gpfkzoyn(@B*$n||T-9_1?+gLv#fx7dvY3<#;JLT-4P6tFC2Jh?7SWKywOGv6O$^bwA<epm}0<;RUe5Oo|y4&zRw'
    'on{!&Ir^*F{5{>!z2na%Yn)Tm7chi4RW`++b!rd;q)NCjA|iq%2auy8yHF^56I<Fiyty8yN_ZR~0kpShSy~I&T7<SO)V#j>ver<;Ouvpjl&^;VMJ=KZ@wYb7'
    '>#>R6+6HfJgV$&qMC$sd$z;L_!Iw;9HC3t((JI5i)BUk&W#J3umDUVyWsi-4%g)jwA1z2d#n!3+RJUc(k@?G)kN56+O;Odi#>QJ?<C@0C>oRs(jyPpkfEDaZ'
    'W(_xG^0MTu#>zx}VY8Rf=vQa>623tjnZABU`<8jAv>!x1_TPqCihR7f-Q4{>j9}|nfvSx*e-v|!B^9)-`7DDd%R{zp%%a<0eeAc5MX;H{<t7Y2w?>>5jW|N9'
    '`x?zSf?3`ga{lm!oOMh%(gdTF?Q1aS_zgM2)6$v-%hzxN`;GQdqA?b`&lEnSWb0UUZor^(Yr46j>Bg`#gUW%cyf!xeMMnA@dava}cR5aZ3u+spjMcR~FeRCW'
    '_G59$FF=N=1XzTKLn>Ug<+JpX&{ot|xki1;)}&U|p>9@tT0?L8!Wz>T)RoLif8;DE6fbTpG-o!bzaF;^&L8iIXxrSu#I9x~mkuJsEL}e=9RXA3b0KlQ6i35y'
    '-COK@dztYeObp6m5JhMZ64`Av)=_0Ev{wz&CK_aX1)EEWXFeFW)pRpV1Yz)AII^)u7cAWPg7d|A#5d&fAh33gk15k)7c1<>_QaDHaETc+zmV<}=a_5&WhIwj'
    'n@{^mw!c|QwDq|~c2+^Q#k^M`oU|-^Bui`e{(a*E7t`MDl4mvFzi)k1ZeKHpSqB6Ukm#VICWnA5QS-bz0M!A5<UJ2Vu!`(?t#rQSJ9Ckx_butMB)f(vhy%5='
    '>^vQ?&?-25;rg^Yj1~D#Vcsel-bm%oV{=SDkf`qF|8QyB3!Y5^iCn<qa^EAjxuD#Bogg$hg7J4e&skGwxGci@WqxKb1b+k)L5WL8v%GUUy!i+ufkckW<n{i0'
    'M(~{G*+im`;SS89=?tVIj59#@mt1&D{_Vr-J)e6OAMaZjl7K?=c2Du??fEQCw-K7`s2t%0Lo-cgMXDb3^U)|Jxl6`Ap<Q5<^;8%W{4^?<EXLDI#D1Mj)w3D8'
    'eHQaVJs^{vsLs#`IzYUv_H>B9I)nUTf@TEqG1so*q15yD`>*9w3d8dTW7WxccF>|F`9*HHYE%yCTsLwH1GdnT1JmBa*}Uu!Q*Nvk*JYS%V4><&qY(k_3cMe^'
    'pk$POh1kc4lG}t-w><%Eu$W=f+b@LrQ*84Z-M&bBqpXuoI=$v@2MTs}A3Sb7eEg&pPo}>ZfR)~M9_?<}&ZFJV?#`o~od>%+E41_A$%g%bf}Pzbj~+jH^623T'
    '?L2C>Hfsk8VmnWEo;-bEwR2Y_?W>+RdVjSM9qAE$)O!4=>F=rC&_+W#l0*7n_rb$QPaZj49fSCgAroEzchfyyvVyj<=99^qv|tgvkanBR#}6Jnbf_uvhdVpt'
    'nvv%Z1St<U8xgL92IbL{$GeXn8td$Hd23td@?Tfj!um3$b_c?LR@p%CpH<fXBX#Ffze+u=dKcNUX3a(5snb7CT&X~~mAUQ2Y0tt!sK;K4W?mIZYf+YKZVTAc'
    '&JG)xnL0Er>pB^yiJZ@6D7G=VRr7wRUmQ!A3*(}txyucTq4Brbe5k>U%<xCk#%A`{mc8rmY`i6E?}DUcW(aeYF3t*4<me8r0JnP;5&iT%2ai=3@Q`6&+F?9d'
    '?dQI`yN5eZcWVpzF7KAK<G;IWmfNu>VTIl}^So+*>|!hSs9dPJPhtvrBVTC_n~z%S&*6iehwIPb<1a9WzTT`bhra%-Fo%z?IENOn2BUfCjEdpW9X3{47fgl`'
    'LF$0YsK#8okC5DHXU0HenBqbhrqrE}eq*GLyBiG0^F3gEL(K|pzf|3>DlbzFhR9!M$++0g#wXh<^kDleolOYBCs22eY_!zT34#@>2Qiytb;h+dn(Jjrv0h_b'
    'V&$elz0`%5v{VYjuzDu2fH3_x<>{k5*Q>2OJIQRB;-N*d;nTP+CbBom`)8eBv(MYFvBJ0L0I;)N^pg=~W*)ko5}uI4I}FMht_01}d|fkHa!%Kz%AVM3rK!2$'
    'P_=G$sG6G%RqOhP>dk2S9sStZet||F_^)OJD-bsxP2*wzMpl-Gv^7Re5_a<=*4$*oT3=+un(L3)O{_80jDLU=^+i5^Kg$P;{`{FbX>q+>t~Y{c4ac*35}zT6'
    '*H1Sxl*?&eV`xhhf9Wx%YmH4u`TX}9Wu|(gaX$Zj#`)((I@@?P|NMK6GgH0MIRE_njB~!2Z#v3Q^!FNPq<W)KRy6BzUQr76kV~S3mWv!ZKo*{}@QR<UFX2}<'
    'SRwE`?XBIxH_zVVfWzowRD!QZsLRFT<;vaA)q3S_UlUgtG=FOqBjz-=rdp(%sulfwZC1&8S8KDbmwqAcD(y8_>SOC#``BEekF6{8aUE9aqd=KGs(It}l2bPv'
    '9%5XcaMyw*T!~Z6Kh3vaqH+%uI7R`D!eW{P*f)JcO9!^*F+ccQx`Oec){M8h>464Eu<D)F4G(5_uG{Re21&+&x;jPc$}20=)7)r?S~odF%}s`=b=^a>%+OFz'
    '*yd)2^~qX!v^F&{tV>dJ{gG;YfstyiJ5txbrq(np)Dw9PW`(QMyYi4;*`#naT0I7ZRp~wdeMYxrOjwiNFFU>-L&B=`{`~ulZ^?+TCcR&Fd`kv|RjIv#`QYjl'
    'Z!o%>nGU?9mJA054MTM5>}dI6eSx{6Ijt+ywYG-xgl(awVJSsM-K2%AO+D4<rIqR{t1zQ9nNecqbUaOG38g?nZ)W;(g*LgJQGSxmJ7P*!BDQB1ER)3-%l56c'
    'QJ=oL)(^u0#x>@nK@?2V3mHuz)em61K|6p?K+&0o@KwWbgaNYT`eerlr~rXJKr!?^Jo7#L*?!0&tx~g8k4^Y>q6q#S&(h*FnWo`ZGg|7g(D)KW<FZ+s<O+w8'
    'hUIWnb#q_W7C8cgdYXn0qM#K8J5kVV7(Ojf4tUR?zyJyu6t!nWXUd8#G$*q9;y{Ql+lsdJiI=JayI;OWtOjmwA*|B+P0gq?_F^M1X+Tu~+wl-;z{Duz!_IM1'
    '<dq&mk%qa=lL3b4|J*t2oReEjSj^F9g`=76y?&QI$IfPxL3W-E&=r;;lW@7S|9xOWIsp5iozUUMV<)E64H56y%-&=w0meza$&^#M11mw4q8}#!lMr4P$znq7'
    '$rJOsNN;$|Y~lX;N6qNNb&i`U;d@4o1k@WIIa_SM{*g2KaGfJ(%I*yx!?+6!2O}ku0R@wfRj5C<&UhJ=f*yloN0>}uKFdA@-vsBBa#N&cQj{Ln8-9V^7_-Dl'
    '?MiIQfaRHDTx}mJpTPgZ!_quQE}MzUyXYAD_9-jcI}K~R*QS}raC3^Wd}Ong$lWg<TJirVl$h`j&eK^oybSWi3>b1jxCj(0$NtU~@4lF0*fB;0AeW!cQdl-='
    '2NsRWT(+2vD9{yyvi7#O1?F8p2iZKCCq+7s^VtdRxz<ke@%9e?^Wf>@-KX(S(-Rr}$1S!woGC_d?NEkM5lRZQLRo;*WK5}qVf7^!4NLsYWF!>RIvmlccP3Cr'
    'IuPJ5wC1A#J29D~P};_TFsM15!e86^Xv{#~madfAw0;I-jK>z&6Vpdl3O6_`u~u>bmrcO(hU($pXLukR&&f^w%n5uo@*kf^W;j}$^!o9e#T=8my#a9@5$>Fz'
    'M4~5#4Tz$=um9(fq>#G||7*xzm<hRUFIHW()Dl!B=M8%fs1^EFEVjS!@}A_KlNrba4dKsf#~teE6C)Y{W=UXO4PaYzhM*Fp5~YV1ZikeULZp_SE{ao%U|jS>'
    'Zs|Q4BB6s!F{p8Sn!t!`&o9#Cj6JOH6%sH2`4w&l5t^vdy2W-?lGzbhi9!`o0gdIDb`?@l8-JlIM!X?6Hpq&2s|tp*BPY!2D8hiUARB_IFJz;XfeOpe7GELe'
    '$Bb^`yw2NJI?Gsrxp@}a`pu<qjAStHwJj-CItQUs1(xa~Nyxguttbu=mZ=`k5`TwJd^-zE8}m0y$N4!c>Z|}$f}em@<FdncwSTwfJ;kh#=s27S$oSHGeP;>Z'
    's-%*`HK9MBo%1Q&PYWAf6_GMA9v%-c;2P;brg9IE3(s*HWLH)Z{hetrLy38!G<bp+X`>ORaE3_a2R>V|rjBmQXx3<s$2OSi44wR(V^x#Md=CcTEY&I8NwF0C'
    'Ondr|lj01;mc9|2oa5fl(_(UugMKpv!kK&GoR&lwi8Uqz-tL?Vfo9LB1&uKm*<gNZKmKgCruwKx(2nyB31QHX9Ce@DZjf)NH)C7>83{f?${KNtqK!F}>1Q@T'
    'kw#QqrbUzCP}_EXN1Cp-`pcM6DYn3Ivlx#r1<WqPHpAgpD7mE5RFVmXi}weUnA3Jp$T+bo=uR@)nxu;v9I@M&rVf}39CM?T|Gkw>^Fc{ap^9rnl}^(mti<Bf'
    '1KIn-_unb0A3twhozC7UnVgjgZ7m_RwK}0Rwwx0iacHnrwh<=5G~?Xt$UxL9b^dTBK$MB>6#z>|jAkUMJ!Ys00zCjZjdx)O_8=50Kth|6JyRr@g5Q`PgA=eB'
    'K{RTGsAU->CkcvG*48c5IcV<HHb#@0xp86D69YGJNvGK-*aHFr1W7WCT~1w(cGsv219j``d+=l(+A!j_zP?Az);jgU=T?2of}LpPF=MNWc)~VOs6#|b8v`Lk'
    '=m*Rhb99GAfbBUf-7OaL@p3~@+m?Ss1QtQiJ5WYwA8Pr^skeHh^e<{q($cs#($p}jr5N)PUWrn@eyhkVi*mv$>J{gZZOrCkKgOD*Ih^#xJS6PNa^vc?Y;HP3'
    'N&V=~n=~P0K((o8WCb+N9`@2Z=J6QHDSgAO3{9e?dIZJ(T9oA>pQn2u$lpT*LzoT}UI??tOCQKJh|x_4`UM{2c!-}SAhe~Q5Fpk3mRn#DL0g?M23H{IN!dJ9'
    '<JWe_I||H>w7r95^sA=byH{)3Gpx=eOL#!mv^rDJz-W;l?mdWty$8p}=Xp;rQ&{xlJR4LNZ;*4|X<9IFDkWPmvBFIf5U7)CKioNn|KE$7|I23vW@MYe4k9_-'
    '`;|@%c>npH+DD%9%_ZfVvb@?@ma1u|bT=Z-4bWB}LV>MIr=LKOM$MrZf!M7EjZZlbH0t5uRx5tG+k#^f{%Gz##$)nuYbS0!Zo)Lcf1fsYj*p`f-l*2+SEZI9'
    '3$9G<gSgpzxC_)a<EKxaJ`~hGe$d3rp?LSflgHOWEn^&9!`MFAdHRIM_Q}(xp!Q+wAyNCVxzoB9Y8f`-8mN8z<mpqQwza$aL{a<n5j@+8cOE}@v?;kh_=+pj'
    '+l(JQX+0*|9zEHSqx-b^h*oa%!R~|WA@|xAL-SGV0g>B+ah4nH>7%De?&Bv<9&Spm7ocMudTB6Q4<9y(y4~G}lDfw`yE{bPgU3MKhWlP(i(GjGVRCnNAJXK)'
    'ViY9qw4TxcK6<d@9l(-|!tErZQGbLN%gfH8$e}a>%{?C>3jRZ5Jfy7gI?vCFAV2$)&FZV{jS>;sfgnUAu&}C3BOad(vRRmZ0@10HpRs!7k^*Pb$K)IZ6*heQ'
    'sway;t$(em{<C#y#MY1kgclR2y^^$QDzwtRdgH3hIt1u3*H+s-w?oD|q)|xP<``8FvSLAkaYHmth8g4S(di&mT&JV90^!4<6+>dh#m@u;?|XV9O;UG*4{QP&'
    '@by4CSGU((MGRVNN5c1pCKBM*mL|<g>L3iu?Hbv__?pxS1l>0vEKYN@?6s@R#WC3$8)nLhbc9zSZ;@XJ#2j9mm-*t}AQ)w2Dw7ucSP{@zwX0RD0_A+!Uq(Dt'
    '{gFiBj*~Zv%n^e3mj_szpb>T)T_j#A^!1g=vgAB#SQEN`!8t&1^v)3P7OW2hbw8e_Kp1TuVSnJtl<~pBHOvsRSwU=XGLCwY<Ip)&&Vk+FV~4h!t(!n17MDQp'
    'IFqn!c?`w){(t}b|M|ZKh^}aiB>y3jD+mhdKOM@;&`@_B6_-2sRczky(1Yr09(rv}j4Y!2?gzHCEQ;`45Oed`mTd9Re3X&((SFZ27O}oUVi3hkyvUQli7&`v'
    '{yIZb{X2mR9&t7eFT`V9>ZgUn_#JQy7wPdkWFD_lawLqzb<{6xG_P(zJyKPN_FThU%uafdybvIRwKOry+$qq8NO{*`ta$jMQQfB16VcrMn`hs@eEagC^YP%>'
    '+ZWG1yy(1n`Rr{}_44Dv3-Rtblc19B$s*X&K5D_@!TbH!^z1`gAgDU#2a`6;AuP+9p3p}HyI2lbI1x|rcA+yOF<{6q3OzP-<qz9umRv^4=_MQ3-_awy+Q9pF'
    'p306+nNTr`*7zNez;*x(m5ftxG<DE7agw7W$;pIPgGjFuVK>B0MRJ~GBfM~q%9x%jqIS}icBwvP!H}q``>)G+E}YAY@@n>Gym+2Z_}aE$=yAM}&Fo!Sm7!n&'
    't2h-I^Llwcs>@u5@?0rVopA(+iYOT%wtlY&!c*Z@Liae0K&_&poDrjorMq$PoD(?Yu*+tge6e@wWT{hBcV)u6u3jhIfPtQk%mbo>$ZI%753uIrZgo$s3n_$J'
    'G;gBd_vg|ud^F(pc<7TMEIA$%TrS>ji9WB^i5YKD>_~af*3Fy#D*kT&`QPAwCGzzrVy|GqMUkSsD!ZWjR@E6v;ffvNSw!w}J&TWO(i0#u1nLFGA9Tf+0w}y&'
    ')fr74cE>@EUuCk4MnpPF&JnpJ@nE;P-Foz>yugTfQJpn8@M6{3G5hVCBfCl8gvm^c*D+<fu9`_hn>S;b&*9(6YyfSXXDmUk$SPDjE3QjEjHzLnCz56kg_J}3'
    'eO%Y=L(ut2@uz<smqvih{(1%_b<P~hyi7?zEg}JUpdyc<2b|N0F9IdwnaJd<P6{nIQ)|sCq0!L;(AA3FI%8MN2RAZwA4_ZCMh4PubW)x2cC~?o(O7#N=~%yk'
    'dSN?A0T!g!YuLo1He(CJ!w=to=XD<|c~bD*x_KL8T%xa*^-{?Kdja+ZY9!}4`|#fj?6V)fB89try6$-dV)(lJP@bic??9`f#pb~>Yamb*@nS^6n!$DeCqU%q'
    'yHz#oJ-es$s6(!vYGxfv?7?Ht0H`&!VR?IHDKed}+^n0J6wxs<)@69H1iCi=2ev-{SDP%^&vdC2=VkeA^M~>t;9qVFe&x)^4co@w+9d&hEUy!Ex>CjyEw{Jz'
    '4sgCP2FZD&M9H>V_ul$R=4c!1x?GPb<+PpkU9M9f`E6@ed$Wg@ro*3O)E?b(n<l?&JIhohh*_c`N<)wI2xtCU&6HS|-79O`evajAl562u<?ZS6rgD82Z{p+9'
    'S5cW6d3nk2$c$_+s!}C1s@g!{WoOAe5vC5KxNB6wg6^~U)&7T%2UhlFJn#G!8uXOv-iXN(M}-9uvkM&kd#{{>N8RYC#cTvY%z_@8%uwVJu?;5waV!XWxq~#J'
    '*Lz83P{t=R#j0lG;3ZvS86h&|_SqrC@?}A^AC-A_2kV406O-JRTSdrtvN%qvtLvrnixecR9nv+>moc52Iw1SkcO40B9qWnBw!OnsV@q)w_8T;owO-lqQtO3X'
    '#IQu5TDz=SWft%?y{_ef#Ceh96VFURbqZ2?;IoUw!Mn-mvfS=zLMuKU4pYu(Y#Egbntvt#dZ&Fw!-aF4t}yYo9iu$<wqaqZJ;8HQSIvZDWANHk^iT=g$kMS|'
    '$7WCx6qb#b$A}PnnA7I!yd#j*@U^j_*bnL2iAfX23S5W&WoH^F>PPcx>a{I3)%2vO)U&VLuX2;#j@kz}^cJ`r_f;R;wMlI8AH3wEtdz(kL`}4;v2Lxm6BqFm'
    '-vadvh~5+V@Zf-whJ)^88sh-Xl1rCxE@)60AH00~@!f}4uiyR19k@3i-s>8bs<Vzr?aqfW7hYcT`+pw1`|$h+QSEeITP=B)tXlKizrA_?Pf_n=T37EnBk|v!'
    '{X^9GQ}U^<Qg|KHHBnXFhA;(d!<C%45nM4fo53%(b*t8qu>I&8j6gJxJHoh~cWgq{YpZ*7N7|D*!)#W7mdbQj5GF!kP>Kl=(NGq~3vcU=h3SfywK+;^ntc7t'
    'Y0+vU3QPiOTNFF!Z-q#)UNwqR_?YN1bU2?YM6W#W8_%do3<#FN0{arx_XBN>cyBsO2jx~(sz<Bl+ohsf1l2pOZ0PN~w=b(3s;GKxd#A^0uuKG5VgmK4U0(AI'
    '9-GDw*Dz!&5Nq1Pl>A>RXZb^f9Ct@X<&FyH_$4(wmC*5__<51cgih4(+)!$>k`Uv?-(IT97N`p2lO7-bRWm@gt!jyxO5daDCsmYsty?vQEkU#KyNYU$Eo?)p'
    'q$9Cj%^YdVjFlG&u3=5|+j>&fbB{gFFe<ct-XtQL&P}0>^_$Kl@-t`umczD6+xzIw{)B3hI~wJe;LQT$zh0^Y701pJ1(wi)_GVwrrYHu-iIn${>yhBYrDL~n'
    '!$L5s_Kpdwl-0KWGpbW6+xE|>TA^#3KcXtNtF8ZxD%6>_`YWoiu(Z`*QP~vI_B@wtr7iz3-Wx-CH+VK0X*VL$X$s+5EmIxibQG~?Wggn@OrePMAxLU@O!3FZ'
    '&PsI*d0ulG)NebRtx><@mnAmB;m$EbkQvrsZvy#?WHv$Hx*S0{yt@Na9-~V+dB<Mn3yx}<U6Q#*n+}o@#>!vH-ven-M0rX{LjG+chk+v0HZj&A9m#o~4Qwxo'
    'F?*-__Ln_`C5aO_m~&UfE9l1*efVMbagbFq8d<cHLjyV&S4oXaMcwuiXFOwi%j-<sHP~%fvXhV=4)uN6aU`FJ3$x>@6<VG-ZjVXgea)4%9f2+KY|(Px+odNB'
    'k6>Ni5cWsM-eK{q|GJKuGJw-PT6GTF{DOVa7cy)1MisRC<8tl5X3<}!XBD*9c-Mx4e1i;0t5L*5YdvWeSpqZVSJ!JnC?>^c_Y%r-KPJ$WHvTIT7AcO1w#{9B'
    '2XFcNi#c}7Km&G76%i`hp+oqwjxmry{t)LzA-r?3qvYP2Dz{wq$m<RMY;tM3fgdc0DJXAn@s&81L7tqdXCzeFzt|r*HbI5BN$QoXYhpf)CllpK{`}>S+5Gf1'
    'IpM#~3(%gQ&)`2VC-WH$IX`*x@g@G@R1u@&f~=<lJfXiBp|mzkCWsmXhZg;r!^!7$amT=hb0LYyfq8z=diB=#v04_0X-t?cwjX&%BA7iKbZl&w+S+7<wc%G!'
    '*fj6xdE`F8p?c5oIT0I?pW`Ic>f$eITeB#BKT8idp6U(5<Sg;@76OV%o(S6*5LOPzPlLQ)%xAbVYhIz0#U4t~46NjV_HiFVXHI#9TZW6c@BImZ@60-|wbP#w'
    '5~|f{?uaji@3;yYn2unJcEsi{(a13(=p3Z|-D3-BORxet5N7i7NTS=MpXT<HzmsG7b0gm_%ZvU#D!28T*;6IO;<DGD`&JC@nr`)XgYVNxI!i!$lH+tV={$e;'
    '_SOFPomcNZbY8rBP9PrUARwdaJpbY4^S{1-xBvFQ`tBspPe$o>Wt5qQ)x*f5NnaeYop3G`H+3>b0dVu41}J#Gm<RtB5ab$<nq;K;aF`LC)EM8*f&~l+h>G)j'
    '<WZOeL&QLq-^|icnjquc3O@WOtA0cNd;wBBKr|qUpEFH!;FNLEkB7w~P6vza|N5!7n4Q5m&$odA6f)|n*rv`T7&gisil>92c5cCEo&I1LelnG2JY~kPuYekL'
    'HN>TZ{$h;GEjveGF`Sp)g6@lk(253wGmgWs8Sez&(6JGxGiWH-<{!n+vv~*yP@~a64>&%FB(OADI{pL3Xe($1_i@x4s4w&LZ-XAxD4!d2%j1aj8Ap#fx+AS8'
    '<D38);71E9;5j?QIXxzHV&><uwF=^rZhv$Kk6vpG@}bmN#>pkx4YEl;pJBXSY(>)vxV8S-C_PU{?Ut>n%DMkLSA}}k32g-oVC!853vzBq0yVL)t}@)CHZd@5'
    'Do4~QC{J9BbuiE|i!^xt;+3*}i`g}6h-LBbLEBnPr?7xg{Tk&L@E0r@3poV6Qq)=saoTXSabj_b1Pqk*O|SHgZ0~Q}ozW$F2#kc>#6BEiczMP^U|1YS+&f}a'
    'c<8Dt$GgE{YlIN9t;JUJIF<lUSXV3ttP|oI8UZ2r8T`4xKTS#sJzCgf#SV>oQG3)u1w|x6H8-**XB%16z^;HsI-$|9N*&0~GdT{v;y+el*o~bKSMhy!S6QFo'
    '`A3__Xar8|@e~_?PZoshc2BuFi#938mQ_(73qe)cqzyY}oP28c42*1R&&-F?jhD;r1-2!O{l%o)*-pGez0*&c_$1N4QCfvfsvi)~AO|S3Vha=V6|A7bxPv(;'
    'FQ&mPxtQ>hO5BO_Z3_~)&-LcyJ8)o#uwae4lwY}Tne5?Q?RAgfAN&hM#$Vrl4v&ryqUBc{m(Nike!-A@{?AsUYasN9#zAB&vOy{<V-D5@zROhbX#-?j5}8<1'
    '#<k#h^zy}IP;hV9Z5%KB2eh&UzmW_BR>1&zB$~a*XJ=y9_YkY`0{5SwAqlWcqx9#%?2CHX>nhK(x9<*K!h%5W9=srj%4b<JL2E18P>O&+5$L*wrwr!&0yxEg'
    'd-e~CE5=qCR!2}my^&lT`->257*ImwDW9VXUdQ7~YA~<Db`#eYNsf~N4+6mllVii8S0KpTWDYYf%v=<)a+(bVkEWP%9mz7_nBIz*Jh?mI^U0Rdae+en65qfM'
    'ppB#2kkKIJZkdqYgFOjT?YerbQa%yw!O1vYj1e>$1Q%Wr`lFusDEU<Jk$Ek)1Z%272*bn+#1>EjbB9nd!-m7}m^twh0|8tZ2X(D%&!LJ@$Za3pVbzVU!Hn@h'
    'Ff``h3PqVei~^zH0avP307a8Ja{{)<une+EGg>p9z`kpXMhu8MyA&#xRmDi@tSFb9pLolK+#bkSq6QMs<m#2fh*m_h`EGMTLwvKg7B%Nep;o(h@k$jJCU39#'
    '0M;h1N>}#O5)ZJ2aH|D}wz&q}vq#M?Y-fEoO7gK!JMojpPj_37c6XaQ4<5E2JbnDc%<56@hKKki>)2SGaxGWTQ{S=>S*)gs>CuImtn2}Z0jk1HXjGEX7F8Z`'
    '8;#Y_5Qhkp%C~4%2YxkwX<!oElcTi1m^iy!=gX<uugf-Rd9@i$%6o0lQ$KOG5(+%Iw#J<%|6*9c$a;r|QiXC|fOwTnwW3YGp1<CI&pNze@!%=NS6kn*25bkG'
    'mkNZ&-#*xIwC@;D4jj#{u*0QwNZfZ6<ODHC%nxBk)mdESaU#K$#ZUAp2{^F>Lrqs6hirGI3egdem+dPQ`NTm;tO3(91l^Dv$pm^Z6Nx6@h8gQE6j3P(Fa!ws'
    '?PT+ejNt<w%`X1Cqtmmry|Hiy%Im}Ib8k)tEIj*B@VJz+hMVj~hQZ_LaQq37Y}^OoFv;Y#4Bx%Z2(VZJi<ZM1qrl9hQI4(@Ut?|j;TF$KkjzaNH**HL6DDeo'
    'LA*&fO&ywZ!xH%o#~;U@pRSljt1_MnoHcarM_b5Dd@Ycss|%Ip4~ZhDPS3c8=&ay)_(_j*HQNy$aT+y9y28r$JVsIQjoq%l<}~S_a)HKXlWK)(;6FcDGil))'
    'm8Mm<9!y6I)|2GK$o*k-OhS|yVxhG1U3t27O{>neTVk_`USR?#pLCiawx~N7j!nmY>zSNqdc0&h29CVIk=MZVMLwK^6dkcfMlNS$j-$`GAnF|Fprr8?7D@K$'
    'Xt<^7aIb%Xd~b-Z$!(1|rP7ee;3Q%0aoLpA?Tr#v0c&F#?a(BZTZ@7?j*sm&^?a{i{@M;=G_S953_G8%c?eT(_Z5!dWb?z8@~eH-XDuhVzddSikJ{U#_V%d#'
    'S3YVDUVD4eE<I^iJ78BkSDiy|lo!iSyJoaxH;HR}tc<phA3EhZ9MUtGL%cAhc9rYV7M*NGHqJ)LESq1NN0WmAyp&j$;>?{yi%mahc+Xa0ydLC~l@wDU!lQg<'
    'xLkHk`LIjo?5?qnyCOgr-ID3jxzJG&)IZmES9Vsk<L@&K%(5;!9hCW1uA$3j|G3B2Ao#nCg+z9YmAia(;p+w^rh##I9d;dYwtF03tHI_btt`Hd80ImnD>_-Z'
    'kQbpp)t*mYs%=rGe&ozFn%CT*uZQ+=gxqM2bI4&y%Z?#uJ1_g(IveXhwKdiisx^mquf~zhD(RV;mC9<@nt9$~F%6{!x~1kL^x5R{c(3Fp!=JiB&S-dec8oIc'
    'nap}~$ceNq!Z=5jhhj{>N})F=DTo8}8IckNXN_19=lJbX3Q%l^+=UrB+wo5E8Jb_JO&U%nF$~dSq>@uwuwG06OP6~d2en2WFjT_<`%x(;`@*@+ddsGLU@kg4'
    'ogvQqv4yN$c5e1M_3>2kF#WN9R>E#_63(L+%p`{Hp0@^}8wsF7C)=W4E1uI)bF`mx(YCwne92juI~+P=_n_pKG|L9dKGMm_=(5N1ZEBRdk3FEO$h$jqhkJY5'
    '9n?l}|2|$fmTM6?<|)~;Q?)zhrF8T>>}sKg<ys?W-84$*S0@?z@pXEO;V{MIU#b<#3mD0i4&*iCX`m6OpQZ_Bm&F7Z%|;^%V1*Z0>13{=EA_6)CeYe6r>kVB'
    'qyq|1u4s>plhO68;lPW%fPm&G`hxzha+UE#tdfi){6dn^D8Ha^S%V^R@Pe4fzPo|!W+2Hda6r%v4*g0cpoCVD(VY><WZ0Q}<TxD2xFM4cM0G^NS29SZ;ucus'
    'TNAzdDMQz*q)!fV6TCMT)-*j-Hnfqb2z3)<EVh)NC7fXh5RW+uhe<!}<@p&3eF-$S4b|n_y;0uV9;8px^ub`b+aEr73g10_)Jl4f9zA*Z@NqJ14wL4C-A500'
    'TgkRy(S{ZjPcQGjZa#jv^}6}=VWge*)+6J*#e}5I)?;Y0)k~f}e7ZaAC4=GPB<=5}z2wnu`eg9rNo&wdA3TzUSOcpj>*?0(R<l`NBt|x9YjYdJO!D4g^Vs>+'
    '-NEI38ne<ZD%i1tG48_rUnG}>wGJ<Y|6gB(q6u#{G9-QCWPD=9k>?T}af8cAGS2$KDU6ybN|B{%O~qN!5e<YKPiJ+z?Q{)4;2ntXc>DR#PO3&Aw4VcSv6w0|'
    'O5z7tV);leCdJQ-H2ozFcdUx|Xe`ayR?`}DOoT0aBfC}R+4lD`%8dW(``TcwKhNeJcWo$F_|STD|Nc(=s8Kf*&JzsZ!cHN>EjQ$UW^?}KO#llYFXk|~QScxl'
    'PtqmR1;>!|mQF^ov6Msin6*m!;$TBPJCPpwhV!ZLSm<T_1^ln%iVSD?38w4?8DUoNF;sP+uQeJi9*{<So~0Lj<hi3OHiA=$c?OE;0deG+F<2hiKdL-3#pzme'
    '0Cpn$e>YCj%Ew4c$sG)#_qM&VUT7sABu8M<gWJL3eTA5D0wNdx+u)Xl+S|5TibZbjumw^)a_lK)k18#@tXh0<6JVJw89%-el5v#~Vp{3#NKZ38SCOvQ+=}eb'
    'SwOlT)=;i=zo(U%Rj&80r26P+omCZ|Wm=6Rvr=;DY9Ux=?WXM4Y?P@Kf0v8YqBhJ}T>iMQt2PgHb8Z7%%A<5*h?C=914Tc)?#xaW`GTC($LTnqUD8>rPI-3q'
    'O=se!ahdq;gmcsi^YvX@L9q8QRFl0`Bg@LfrX^F&>wI!zTWXAAW@pwb$r)4ExTi3INGo`2k^U?$YACc_X5Cfth7P~S6;J{Wu}G4oT5eplwTk?q`_?C}+yu`K'
    '_o84=6MO7K2#|DCG+}?!_U=NRzScfm%L}VoxC<*7qE*%=<{MuNN0X(@)Z}>TR;|qJSUGrusob~l5)7}+)HNIN;d`oduwM`k$3OE$@HR~c_uz<w9b@Yk8#8g$'
    'jN+h6*zxaWbyC-z`eS^gV<+jQQwX1l2Ame9pslL4R<B+7zDvQM-0Ic`(HS1y(Zq}ecFrI}^%5-ry-R(+8-ypY#k=bB?q2Y#-TY_1dQm01dp_6@Xi^kZAAWAs'
    'LFq6smcW{_hUETzOyKI`iO=#WyY-9t(Q_kSm5$sqa%<tU@pU!<Ej@IMb4%<@=U1)Z=c`tLmc{j}SmRW4E~zc+mB;0Q!q-b@QRNX<@>(#8A|4zTG1VNyIB(n0'
    '{0$zL%x3%AjIhV}55cG4ARDLfAAxIEHl3I^VRYhv0)i^GIWz>}7FwqSsPeN*>G;z&Cp@%@8gqJ<x&_Ya3y{9z6gl@7sU*z&$r_-i!}iXN><#JBZ@fLyKd!wU'
    'XCTa<*VdNNtIl~xNFR2$43Wp!8<UQ7Y>`Qe$H{!W7=`B0M`k^`ZPd*h4S5O({5<QpFdGod5YGt5U&`*P7y+Xnjg3wbEhoqx7N_}QG!R&9bZ_wMx|Dp5f*zu9'
    '7M16Sr^}z0p+?y}sv?iF_uHyLZLTtoD~@GpJc-C<gW0SfKN_fVM?U+;)wy1DR;~5(o4|!DF?Gbnw2w}aTB4li!EpMZRk<T<E1Pih3eAtm7}X}D6oK;a9-`pl'
    '2WVW$t&J4ZM~xNR6T*Sh-n4BB6;VJ}Cup5wEkXQp+YlH0&L+2OHaSznx08kY>v|j6W?dU?<*w0schhZb)3M@4wt8H@kL_VLOFP^;)<%f1u}XPqf37+w{3a2*'
    'K1(DEjdu-?@@Bk_tR-63<z69Bi((C}(9!UGj&#l^0b#uun{rGVp}e)DAE1PurZY?uO=?`%siKRxPTJSfbfd+>u%a)88nc+p^F{wO9dOKgjxh8ynU5Ur!B3zP'
    'wY=C-QwXBu-YP=Ivw1K{Q5_~<PeSK_iuwrRM>m>_-YDw_&))B=yn1dY2~aUjMk1{UUxo*y<CAHspNM--zFpF8`jbo1yRknMqv-ULMUjl8c)GeGJQ^kAL~FwQ'
    '^ELk4ndIdf@qNyatCDclX6kajwORJTyBF{Ff{$v;&(d*zPB=FGG$6VHV<qLY;6wg=jOR;lF`n}EZYyraW|bPV=s)0<!bQQ!oNkTM;hZlsa6#}$;z1I@%x5FK'
    'C}d1Woc<KA=w<}Z+so(ke7wb&YFr#~RU?kHxJVGqWsu6SQ6f~6@ogJP<n@joHW&s<wMutlSeFAzsR7Rh85$nZZ8;7;q~gy#-!1QpB;R_r|2pYyQ&Za4cw52*'
    'ZO@aF?Q?8=#~PvU(u-s!fGKHx+Cvrz9GIW+SVuV%PJ%P}cbI<ar&A4xP77Im*=`5ThH#q1z=hyqmP1V>QG{~*8?mEGnlMu^CJ4GU?ckRzFSXgPX*r^niH>4k'
    '+h1`zC~&GTtGd=&U_I)q$oq5!r8V+<a!Zxg)M-oYyZq>nsJ-4597(fzakiM~F|?m0!mE85c-I&a(E1IF_oqM^-9E|jfgfg3M3zs72upR520x+U40#kbcb2A8'
    'juip^lwquh)9$U-eO0Yn9#kMy;JGb7BoG-`0m#lLDWSBbh))B{z&*RHl`={$(O`I38&_lyDrr295m%T67&-f}C`585sQ9mc2Zr1~wbq>NU&Y|ZV=ZTx{n-|x'
    '=r)dEyxIa<dnIf%r8RRSG^Gw6@^ULnldLppOSOsbigq-P91qd{BzXVw)d3!SaF&3mcR>f+TuxTXKwc!d3EhtmKv^wN*~>=gzz^qA0qoGfb2XQ+0x!|^wg6RU'
    '#Bf2_GNNLuw&LIiTp1CPh0|`Kq6jj_z+3ufGWbXct*o>%x8P+!Zcphs%pW2>od|cQM53Fg14~d~5uJpdVw7JD@{5V?^S?0wHH9n|Im*lw@c^O;h(-fmoOp1n'
    'fqabv`4-2~Yy-OqI=ba^C(w9_cxSoMLmwtLe=HSXF>+=53cm(nB3aCHB(0N9aBq~+PRKVLMAlSO!iDjPDt$6t@N3sg`fl)p6zhbPq~NG@%qxg4z#e{a6T=X`'
    '%V|<5e{6A~gf85O-P#f581R#a<jxB88&$<s^*Dr-d^4)5<r7-^lZm=E0g@jwYNBtS6lppx<VvpGE)h%8ylzRS7zse2Dq{etfxtxxw4P3HHYGtJBiwRkumb+G'
    'ur?{OW&|Uo1y8%t$m=tyG(?z#`MDl}ekm``L$kJq>)~C&kZViTa^+IVD*Hqg;%ivlxu}ZzsG7j2WJ)w7Z098l1x{)SJ~!Itnd0WBkQz7}<kU~+GtgsfHKx6S'
    's#E1scOAB)mCRDBnURF=Ygq@WHRPJHFPj;oJ`Sddb}m@i6jic4Nt{y&&28F%IYGuCry;=5Ki$6?lNC3&iQUR;vSLdWLr40oRCD3qu7FHGzIgu(?NRI6j4UnL'
    'p<e2}jBdD=25n8$@1;L~4OhRgQ<m)#saouhVZogXE08!M**cTPCvnhabO-HWFzF!1C7GcZq7f0`K(1dPu2`21XjE55u-p8lCEu-++lAN08!5N&2J-AmHoJG%'
    '-CEsXkf(etCZjc~kO*R&8w#i6BQ_U!lGt%$r`v<$PrGyvX)QqR&4wyWR8-`QEuHHmN(fsrZI}aNQv`?rjGvxAib21C@y8`DgHz;uiX_F-%Bk^|Wu~T}>zUg2'
    '(Va9oLD#1)`!9mTF;E24y0};A%won82-~dx>@97)w)HD2e{FBTcI|2F-FWTNlc+}expjuPHOshW89Mj`FfTg$ZW&e>U=N^(l3VZs)L~K<7!RdQ7Wq~Km=&-;'
    '0XH<sCtH^PFhMENt3D{u2>mRxX+$Oo^;Y`nCAFyjn-#?(Ew-D@)}zOdW9y`__1yG4>!-2QYLD*p7lQ=U1w4#b_9v)YVSaJ2Zmm?8sGD`WlsRxe5;Q)=Eg&1m'
    'sk~Q|3}>Kry;O6`L8`!PvxE<~t`cs#WbDB<iH~LzP~VaPid=~3KY<ei%_5UQCh&AH6UoU0zjdq;w4{u%!CtO_Q4#kA)J~vM28VH27|qIX0cXU?ESa7P&>mp^'
    '&%8~8=6z0}5Kb+0sBjw?nq@2wLxRIYP}#Ng3#5R(g1!x!Z00K2<9(h2tDn&~uBxqaj7Y`I_L8W!^HnNuKE3S=-S^k>!JED8*Z4ah_rHI=|Gx9=;Na!kgZ+1J'
    'JI`M~`}pzYN6KR*ln5bo?75mjBtLMK`j-CKxkOV%3Xm4{SGMeUH2HkR(404~-!F^Dflt;5!nsDTZ=J!ZGel{f(>ER34&d<7u~Tl?Z(2UKWsmVlp%Gvl);Jwx'
    '7>Gq-@};_3zPd`~B9C8sqdMfSMqjH7`LF@%+N!&iVwSgnHLmLwh2f|yAhwV7x@?koiIHyUj7-}b#CB}}Xt7rC0RD@NP4IEmj+ZO2<Iy~h#R?Hf6D}=e1SR}z'
    'xkROjzvKy?P8D2*F4cjjZk^-N6p2-Pby>@;tF$cbAp0j1yr1dk)5{w-Q&fC*Wvhf4c2xt~;NYBjfg(%jOuoB?eR<ujFUedCr%2v$n{j>JxXwtsa`TSAYPo5@'
    'hiX?`!$_B`lvg|3+A@&Q`Z9+SUXnq-W+!;*cEahbM38R@W5w}>Se{azv(eTik*DNAbH3qa^dA$Gun=JUWK%@HG;R6dn~6C1n5JY+5CyuyVgj7Xmzz~U*2x<2'
    'W|Q(?4bpj%jcQWXV_Mh2g2d7ipR8rP2IE+bV@Iwv*=BEH&4R8qW2|GsSkHV>bM~lV`=)x>s)u&Pfo-jK6aw2_xxgy)ODN)YU{@QtHwnw^qq3rJpeD<o*(dCZ'
    'kIUz@lRq+aetQ8SL3ZvHi0LsG2r=4KPp(6SS)3@Z&jjK8rgT97LKQ->Fq%kyF_C6FPLalcgDxn*aky0>-}>zMe28_JmbN89hy+JTkqT@tsCJlrBAYO0GnBm&'
    'T5UlYK2WFb=v=5Au8j(&Y!X&dr<jg1!V&|)#>u@a?ZS49y9FL0KQRR*?*}eO*k-j*Xeru`)bSv|*M5L!s)P`+J8Ce@JeO(+>`zc}aYl#4I}+2xdJr<+*b)NG'
    '5HN{wZQcsGOqDWkW|ZrupN3plRDq=C9v~8uBCN>mEMxf@%%@I#^iyN`B3L49xg@FJW~-I$IE7_$MVawMIE+_<*N9|X!FDj2a*7*<n6NM={G)0@6S!kpl><hQ'
    '!)>q7M#Vj)+uKjcc?PUoCD#$nSZ)=KwThl?Ui5tPqS7u=U-*}FmKSa(ds$Mf^!?eI-=DAfz3PKgqJD}&TUm2*LD$`Aj6rX$FXwC>YzMm{c1pUnyK1>sb-5*j'
    'QOAha9445GUK%4{^m@ajZ|<`R&1Xk?CskKvFNAUpI2>;IB_m-97#5N-<&8>-t0;Rcybze}ERJJp+(x5>)r7<ubr3}{rb6w6Zq;>#Rd1-@Hi%4Zr(WeVE2DQm'
    'spD;OS(Oh7K}}yJ2A4eDbAZ7TT<H+wAblCPs!Uw+?45PkVK%BfgsQq8D#KZ%;W-<nskPFTRh<?(wI9C5d8oO^sa~KfGAPqP;?ir>RIwC{uZ#^OmK&~5%cT7A'
    ';Ms=*OsW0x;KTmg?@?bcMI7lvgN_7N)YoCx5z4#Aaqt|4Q}(4t!x+oHgS@}cck?n5g#vxDz5s$Obl%TV)|(}u%PZ3stGK8&W8+LqSS<Rd=&xf6>hM(otOfJe'
    'qt5e-&W?<x4->X%I$O}ag8`|L&ik>_QOsUTgi4hI{7_xRu=(&fR5`r))-cXzC)=3gd>b8&N9i_vdO9DE?m7h<n0J)MNGK4LoOq7iN}vZPi%~LDt#T1kMrL@0'
    'DNQ-@eR-5|#Ky=HXNFIu2(TIji$VdLW1W<wHOptpvG;v4OJHu&nW^CklX)m1sacxr9yj&^b~dqc+Y3F6Ec_}MPgPr$2ZfDQV*4wvnnboe#JDfzeu7hP`D{Uh'
    'c|W7ha*9ORLd3pW^HWEfwJ4S=7%!<9di%eeRUYHw?`So!dn}<$VQdvO0-RFBogYoKW+-;u!OMRfboSrA|J%XG>b>V?(Nl0r(&(mRj#f=-Q%2C=M<W{NIuX#B'
    'GO>*gf5a{kpct;m%NW)MTjQjkO-A-uGQ-K3IuRC(9=hl#l?|L>&*r7{9+DE_b)%r++3n`A`)o1K50EB26_DGY$;@O!>Saubb}7t_g}#da_`mO9hSyQ210_iN'
    'CresUqVVuX%G5NIEM@;i@#o_(j<-(`c^!u4E<N9}@tZ0vrt40eX`XZ^@eNk2!bO1dqF_a~3w<#^vEy3VdWv|wR9Iw!hBKJr3UJoDhnSl8xVz^L!&cB_HqHxV'
    'E(hiQecY1R$#ii4e&iO~sVF21)C_lGS&!X8S8uoQZNm?lcBM^IH}=K`(s-vbJy7kIGV$;vU@I(YJp&?B_en}~;p1CX@O3yx$K9F{Rc)8ouzck#<CW~@?%>?8'
    'd0bh;eVK3TOyj<cS6A+j9sMA>Vs*f3&+UUrwkyr~`XI8^!0MY{iSz`Ixy2kAo{<RwSfS~Z6HG@*CUP414z$1Tf5c15HIFnCOLeV0BNm3fgH>8O;@)E`@-k?%'
    '!wb(Vs~-O;<(@ToWRc?aewG7|z#ge{3w<8!ch$$x?H>F}_h1#2?#6eDvU6)A^M9?3;a14K6>@Kd+*={{Uqi^f6-d?-NQ{h$S3bJ?PF}@Zx#w2y`4^OXR`G{U'
    'MswRGT0VF-n&UPUjuULc6wmzr(c;T0>~3Ms)+F%2!GB@5A)38NG2S(>s>MR##nYv08PBG8sk0xWiJTjfc45=;gK`ymWj&vBt<RI@Cj4^ioxcTK0J*C2u)c<|'
    '3J*PA#KPf~_In4<Gb_75X*?UpE+v(oJx=iNW=^jbUi=q|#`PDe%3@qm%Gk{HACqV8-zM;$VXz`Mr=yjcaRvWc8)E<FGRWgUvSXvm$mA{<Hjay+tJI+`9sYR6'
    '_fK=gG6Hrr$OP0OMVYHC>ghmdvgKQia=Qm2)Ob;(!^H@%HnIAPWHM)lPiJvH3i5N<->3NnLn}_;*W4mX$k)iTbMx-yJjFZP!h=NuGLpU`xhL{|O{`4mqdY%b'
    'O!4k&mVH{y{rEa%-7l|eY^-l=sB>J4-m$S<_O-iBUqwdyQgWIHoTj!vYl=qKE*@36r2Bvg%g=m%$K{+drvFIk^CjYU$p_Q>m8Qdaz5*!~Zs85T_}aF9LA%z^'
    'y5<nc#`xIW`a<+G`rN4O64k47Cv#G{kv&mXapR3qw%RM*I%V}|xQr#8Lb{xoGFcfS%qw1a{9E15I}g6`c)mL=)ro89Lt(|VdPrbzf9-0|5JUb7I~e9$0sn4|'
    'kJ8a2|HsQvA(jF!Rc_U2DK>ls9+^81R)4}T-HrB&9jgn%YTJEsa-6OhI^o@8&I4Gbc1z9Ust}F34YA2)C;>{D61o?_gwv6q)1E%3Oy^FJklkwJbe3|6otxaQ'
    '9PjCf`p)7RA>^OgrW0J`L>`sM(QCw}tTuf*MpqaL_ZLYaSg+Eq6@zMJ8}sHw^oMv#yF%MBO!&VZBJgcThd;Sdp=$+%T6wgrfN8Uus?10T>{i0NYw4ltc+s2M'
    '{&;#x(qBL`z;!g6chC_K6d4$~&O8_KK2mM3^9B=i{snZoQPS6kx~eV|8=W#;I2+-|Y<?;{xFahPsa;xP*OIxg(q6EsouM|{;UCF5Va!ymq2dFzSXjE_tu*~R'
    'O4DwFOOGRXIiJd%e@e(wgsfpKGhNNmokkf~SEd&>l57gltn!_?`4!=oU@9D=8qCbuSQ_rDj{u$4vaSA#ti9Q;Olp1?hc%@$4S3u=)a5_Tg=eht-(4Uox1HHx'
    'F&SlNsg011_V1FT@ZEq@w3d<#t(41&lfEN@UG*VQ!v1q1z|o#`j63|$NwzdUnRiq$eKG6HAIG)UtagB&#nWV#b{-Bo_RG+bZXF=CGNP-LW*mJ(JPvDYp1)HF'
    'b`_&&`yNJ>_KmjjcNJPtNiZ}-U{i*#``#<ZgrzT?J@Kq47Gy{~x>L-v{@LZxo#5rWk8HgYfyoiIC#{MYe{g}&qwra59*{X=KPi-<Qp1?}7PtT2adjajHHFrj'
    'jimi@*CV^V2B;P8AUF4pJ#pdHN90lgzf6RiFBmA2?YAxZqB>Lphwyl%EmAI4g*|}(=am^l^Fj;}XALeYVPKIt;!Pn&Enubq5`#hK!U(vuFOAjnd>V5#qG^8s'
    '5WH-Z6&eo1_Xg)n9mN~(uI0>?FY{K^e*Ag`jt<M%773<4Rw{TmFe1HG90{H))9L6kpe>GLi9QjQONGqiJm*Mb3vmfiw)x03PE#yE0-@3g_eyKu>4tF_yE&)L'
    '=8CH3L>@V7C|7Plf&<=aHMd&Lm(yxqeo9c;wjXx8-RUK;IOEZ`-+o*6u)wfEgn_jDC?l7X9XZvMIfggb3*`5cJU<zw+l<n&g%BQF1c}+)-D<TSk^pa3JeMRO'
    'f}BaxW>wpj-xZxrCNN#&<V@rXLCqqa%gf0&!c2b>7oUho%2$LSkJGiT4pze#CbWrcMIfuTbzj26qdQ43J<Xvdy6fpDb2wE3=jk9nIl6NkRS*)%)h2GV;PrIw'
    '>7*84iO%r;eJ02mXrF{858cu`yQnJ^$zqnMfX4)X7=%h*YXq(W3d^*Nx^%;esQ=*b3HXj8x4B?nsR4fN3V<!yFh0xGjI~_Hmo(QdH`uzkGuL9WB|(QYohuk@'
    'tr~Gn3zMyc$;AyIGwJ9gIs!XDeh*xNH;4@_pk7bhJ3INtQCv)$@d_{)dR#V<hAIN2Zd4w!*5(=#k?95fRm&lDuBfX3GTs_y|A2<sWj0M~byqH*cGZh7VPyN`'
    '8_w+0WQp0#mc&f6Rk>V=>HAz+u58nqedRW-*;iuIvY@<c57!C@c2&qgkQK$^HNC{wk<?t-2O+Ik<WaaN0{01o$H4@l<(pm<Zdc~f9*hR0e|co;V8hX;TubL7'
    '3kTmn46|7=r^Gv61ZJP$=4<Hc=+CP_kDGH~WZJb6RAaukGI@%e3_UY3NKa`p51mVxX603v0wY)BP4(FHIn+8%Jr#(r_3def=kU6ULI!lJ7xvW$s_NbWFxjh`'
    'msytnvIb_}n46lI&223}S)rt`hE8Cw60E9u=-&j-a8|zBAwoB?4NPUd(^LE_G)tSEG0nzgrYakzZxy7kTtTYJLI2fQcHi&6mIhslOm!@i{06hUrNz8h!!oQQ'
    'T%$4zs<#arA70%$Je6b!>#z>Iy$0*>Vm2bqhv0O3+uIqAALe(0$xr4`d`>WRu!dVZ&Bxn2{Lh1@k9VKOKTS`}spPSS;!qz=L!3|3Nhrr1Gc1SD8p4jw(qzo?'
    '2M9(NC7P5iS@k?*y4v)ge_fmK5X6tg>}#?Fo4nf5Fih^%u3%+fn-*Hf$SPI!RqxpSf{G)hY$~6udHubatRWOy14~q_7P=6yA;A3??Al-L01i^&uI5QCTCNwD'
    'tIS6)vV9}8Bc{T08>DojSs~w7C)vgf?d6slOA>BZj)Da~8!eIT)*K0|?xka4HoD1ZSgkD^4|9^#U4rq@NCathq8<2JvxU=FsA@(!r2>2Gk#`CTsiqzwAV7iV'
    '!$Pi1`s$EfbtT)Oqmd~J&9zbLO!dKS)nSjQldm$xM1T>B&ED=6i=NE15Mq=ACb6U>%$ODsdezHD+5EEBNyTu)(0|q|5tbNk{wQf52Do&s^`c~^(0J2J%iW;H'
    'a((_LQum=DmWTDNOUQM*gp@+%SJMz|{1jt`){4Ft9UNriZeuS{zH-L>sKUnST{_lHT%Zyz+OTSH1foD%7NVYz{eTFGcA=P)0d;Xxjx6}|jm;tqRfq<wN`X;D'
    'Z{1pc54RSxafYU5^KVWOprI*&#BuD|3qsq8%Y4kE|9I4oyM!+;%y|-iL859Sc6vTefk~$$M7_XU6bets(m}(F?1rUVkWeFzz<=Xih6f%ngUP%ew(0iT^Tw1v'
    'S`yAZ<gGi<YS)^sx^Ir<Vbgb=XO*}KgQv3AqG4nOk;T;GOLBynZ1P`0nAps0c@5HsJ=#|kKiFn=^YVwG+FVrv`2q&luS0^lHN9?TdR<X+arTiZx;PANP1!{n'
    ';%-HkKal7mY#5I4a%;gZTd=DH7JCG5Ok}y*<t%SszDD$3$6jabjz&QC8C^CPmTtt(Q%kIYvAeQh_Q$${HW$tx;?nt62KrrOAUy6lC$eeMX`!K~C|P=1Xz78+'
    '$2Hcei<Gf9yV4k>O+nqD$OsMHXNr=!$+BXpkcKq3iSQ_}1-7fjRjnw<)m*hZm+g3CP;{p0tRp8&-L*TfN~^F}C~rtuVB<U+gl?Gz3_J`a!bQv^EOg#W{{Qyg'
    '{k@UnI28R?=1?~u0WF9nK&~E);B&aUl$HtiaY(LZU6KbJ00VNgzzjSySQ3`^ezFr=en^&c?d!y`9b54uxw(Fwe6}N7lKsD&EO(`!{4d;kboX@k^nf6D<=8p$'
    'Cv!0~-PPUI)zwwiRn>IB5(cRa5*SXUzdQE|0pD5Nga20QjX%4`=`p#3izu(mze2c$@Au}71+vtrKl^9J;sv^y|H$U{k>L7~;5w_|`jM6L{8mcVoJC%0r_66)'
    'SrwdO&BP=<c=K)eDW?;Y)uQou(RidwgDay}v*z`}VG9$F+zEma{<hrF9eq;}wWreo>Z9nwbi&$A!(lk^lQ5g$fe`N@S};n!rx5=vuqfVwhJH324=P$;YUWV0'
    'O0}NFdVzW`xEpqZYE@oitY7vV9FOelc`w+f>`wOO*ix+_)H5?!f$rgG6@y)L6~fMGwA%AWnY^q(-ee1{9F9A{0f!e^@>{@s;y$3~Vh5BqCcTTDLEO2x*a&)^'
    'XBz(U(~YP4&o24P&vb(2r4Ib?`;FzG`E0ib4^Q{?iYx{t=|!}4vTR7>5%OhyG<jm9`OMQ+(#lDo#02<Vv2j6Zu@<X%-2_fX%l<Cr$hzL50ZYz5XaM8|ad~eP'
    'q-{(=WjBDaK24ZlXuEJbIF#1|Ixu-FYeYE5Ql?RDJ!#LQeY^Q!koPAY4=db)TnSaXd75;nR38R@R5@Gr9%Vu(h+H+hmo=|+P1Fn~Q?qze)g4?_!JgL;tumMj'
    'dP}1z?8m0jLIa|!hLs{-TGWK&GIwp9t-`=d%abr}W5waWAVU0fbh*wML@@)A4rpwO5#e11p3TC#+#!pb6LFrTLn4-gMOYz#2KxOZek3I1u5r7(h!UZ=5U=8U'
    '14_v$jx@w;aRsz=q9q3*J3~6R&v!?I!YT6HSG}NqP$wChbix}!R@Zxd#t@X}<AlImF@`YR+w(DonFzxd8a}Kvf-?u2U{u(^@1VC`OOF)ecjj(+1@vTbxsDO<'
    '<8;&0E21-3h8`Jom$u=Srd;7>^n^Q`lb+N2@iBPG0WQRh<2@Hf@&Hx=@~;JHAx|^w>RoB*9#07Iy24xUW1wad!vo;!J)D4Sq6-gyO#XI{W(+o#TaY-d!tE<J'
    'Z#=*LLi_ofo7l{GBXnjr(()|)DZAl4inTRkYpZi+rkY&^@y>qv;PeYvHW(bT5#%1@OrDIpwwa4baF8H@<8e9O8mI0m6qn}Vje6qlm3qDIzF<&WfqbW+w{G5v'
    'B{^%$YwUZzXSIp)3ilM#16>CpY+>t@V>DFZSCB~8`)0HIu9Z)atv$2|{YnC_7S`~3p3$Ugoc4dH$+XOy0c%tD&1qKcM>V(Gj)<(WMPq-^PJ^IVS*jvq1f~P)'
    '3f9Vn^%rj3++1sKu5I1k*#6@;QOm5kM{MXE#6BhB^SyoT@OxiO72YQ^?!dm4=*yqMr0g_8LYTZ+61}$L;b5Y9?f3AqN@=GaB>{36G6tG;II99)W0Z=dxtZs8'
    ';=B6YE(!^4`(VMkRq~M!E=-}pO@oi)3`3dYtzpgRIdPtlHPS5#wsU)O83AItKuC@PDN1KYG4qh3O`Hk`J>FZ}VeeSk6Y73)F!YZrpjy_$EEra*E_?d!+${-%'
    'n^?iq29~b`KMQ+JVrM&!*7S?-Y(U<|>gBbKc4Mk7{UgivAJDRWK9lzUE>>)HpPU^aY9)yPjp=TX;J$z=Zb6Bi22kv5Xwb1fvKgphWUyt4Ki5`lBchzgyo3#P'
    'COe{#RWzb1m=lLFUf;{*=+eLhFDVo@IeZgDerF)hL`Qz&4+g;?Oow(B9HGkc545)mUgb37$)rDmu<_V$5DZDvD9O#Q*RmwM75B|Sr467AkxPWGFg_Syu_=)F'
    'NL#CjA@&%UocckGSF<Lb`zugv^*sGVMQ{*s7uu$<>F&7m2B%Y-8Dd78(pF}lyOxbx*L6}rtnEHA!s}aW8}MYC+s{7NVgX3dlJ-(qG67z`P?OW5-c-+oJU4s^'
    'wQ!BrsVJH{t<!Nv3Wd-o%8*xr4t2$Zh1ncEqD1xVBbr7*(+70<${HrA*q}b1ZA97xJc0>bGij$g&6T|m&-(3D)z4%G7uM{ldBV<^AMCWeU~{!OdGEE5#V_VY'
    's<t`-np>Hpi?k;R8n(RVVsn|i>aUFt4xr%kK5QrJQMc}8pi$NF*6poZS3j|^0h>bvl093`5GyO*>d5aN1`EylA{o7fTuI2gh{1cBClnin(e%I;Z{uoj*19Gm'
    ')X0QliuXd-ltre-mXy4#e)qRHUTs5ziBDQqn{LsRpf00i@H`iC)n_Hds6s<nkBB&m6*R0GwUj1yJv<JbUDa=+;JR&8mlOc@JlcXfS#U6^p*$}`0DIJ(numCm'
    'Ka+99OIOx#wP|#b%|{r2U(6$?x?eE|BKJeXJaq-gFH6R#CHbPhHrq#sAqZnfi9galvm&1tB8#g0m`6VHmB$`)-n_W7$jImn)ye%mP}2HnR>3=8Q5MXP3CW2`'
    'kLfT~W~hq2Pywg16Z?C7X)FMfjs9O&tLlD81*1@+7vO0Qd{=kKh7rz+ygb@@dAqIYSIT;Biz#_zOkNo;IUyTQn9x<pKsK6TA;)>efdLnMZI((Qo7n>~<8as+'
    'vwpX;J7*1#y#p0k<rQcZQ>j08=Um8;M5*8u%5Uc_OgTKVTCL>(r{)crQq63P38vh*j%Sc7(nU94=MFA_>@hF0Ctef^?zybkWMf&;c$~wjni-Mo3!1md##Xag'
    'l%4&TMxoe}qkfTRIl7qpU9KQIC^g#)%aCTd=k~ZI#p(zP`D6XoOWQX$uUyj&_4*#MQl4-4I^MlJ46?(RvXF>NrTCg~<&R2MqXgy_Rn&n}LDNcFB~C#T01tJ('
    'Ena{VCod@h+DA&**PP>_+CHB99i;mn{2gO<57GlI@}5s~MwADSYpe=nAqFm`p?yT{J}Z#OuC#d+k^a!ne5mTUpn}q9RyqKtvb?j<Kn~fphV`CXc1^4M60|XZ'
    'x_4_3;8vOh{XDb}&DCjQoY4t;9vNB5s>j_=r!#2>qm~QStLy{4?bp10T7&R=Pd%tNd;2b}5?QJw504Ia(T0IXJCbHZt{a}3Ufo**fh(`l^&|Z1dwdXxAkZHm'
    'r6@*mI9P18g2r3PCPc9+oy=rfe^RoO15>*jLd)d#Wdb(g05MGBu{Rv2WW+AEx*w+PbOal68#nm&Q4FL)_WFIV0)qs_g;T-!!%lcGMo%+#f^Z<ET6Z{o;4#^)'
    '^^H0o6VP9rhG<ledVZ4l6MWK-9n~aRwes1^>*nu_gUixM<$YM6QPD@9PO+Hee2U{YIhlNLLl*EG<KeA|M_oW@w<g=>VNXtn*q^$0IR=z<5B(?t{bqr())Mbf'
    'ujm3aCU2f#*vwEcguYnTN9ObSrS?SR*a>w>y|oqW@9Afc=q=XwLD!@)@6#UPMAud@Ksy%EFu@_l>i38kn9n1XDyppb-7e4$9bLvb*9G<=yPN{S(x}A?hs`4D'
    '1;@aQhq3cM@da<62@^)UKacjJAkF6YS8(%_jf%JVT7-=O-}8i5pk@@S9-++|9Xq5{>#|+sfck3H*@P}SH=pm1TCaTEHpAg~=<%oA=Uqlc1UWa}*8?hI#V-W}'
    'SN{O*!6+jWBT0L*6?5+k`|R4U8WK!ej>tVArVd7TFK$DHHlS`EID;Uf>Iz5zIZcN3pkAW|BTNL6OVB`bT7h-f3sE(Ip18)uLn!c*95+Dzz^$eps<T*NAYN=S'
    'ig<}}B0)+ZKnij!vYj*#dy?ImT$;+);`H9Eg6?yd&P24jXfnpEXf$;>hKP)D0s)r>P2i%{p6kwI0U>{h9nfG7xwvKW1y_gH>7KP3AC1a8&IM>n4Fk+F3%lci'
    'pLl)Dmj#>0ojQzDQ6!&|AVXWRt)OzvWPMyKe3UPQ!RH6@5$p+MEW^BZAQZ(NOc9Wlth-UB!4S;Y+Eh+~5REhl%-!`Z>Xh0q`^Q@KzgYc*xBs%c_G$STcJ_6N'
    'cYkliTP?iH*+xGe0IHB&Ii#k5<ONThyQ16wlTUgJ3#5?HO5oczdvfpV2a;RS0jKC6GO0r-=aNn9#kS0N1hc_Vx~cFjz}RaxNATH@=DtEudLp6VMB`!=?2A1G'
    '`-14L&<M3t0PA~lch0&qckW3X3;M}$o^kT`%6%JAKt_$g63Por6CSD#MCgZvjD}nIMlOh%Q0|tei%O7Txh2V0oB-uo%_Cme900)~H$vEmQFa^=QVmEj&^s7T'
    '1n}$jmCkvON>0$^(W45EQR%5%;uJb%TkZ$uV;&|b{;g@;ww3>^X}gIQ#W+3LW{%5}!4%R!(2(5T6sHX2G&EY1dkS`UY**bq<>euJTQ>^0O!yL*b{)*%^a-9)'
    'IV~xx^ScWACbjSs0)0q~9;2d5$YKx668xcwhb?}>n)D>9>+sK$thiUF*cF_H0JLU?B&MIH@8^!xxy~<*j|IipRPdD-DN8LzH(cs)*$9lw@~lo&Y)t4VWADX3'
    'BE;z(%F(Bf)HE=Bi0CtLQ-;L<#*XEgNPR?SJE4p_GEM?GX>7eq=Ro-x20i0V@VUjMuuripfQ1%5#+a;m+n40;d^QUe`4WT`htb&pZ!_g$4@xcME;v?^Y0w1b'
    'QAV%e>L>rUJHaHaxO;kGTKiS^F`RMi^M^W!+twN_9KQ)sAm7@>`B0pypLkd%elm%xN1(rmO-n>$=^={z@`R}i1L}u^z<H{-{nD+q_LXaESN;;CMZDE$QuD5&'
    'KT+Ylk1|wf9ZO-bsImZQ16q6(?}qN<KoI8d!g6S*AhNc#0}F=3|KRLpacA70?T{`%;483;bDZ+6_Id&UpTWV_X5d9EmNUau#{s9F0uExLu$nx;K{#E2-AlPg'
    '_?;p%L_x|g#V;#uqV#r~_|M00URk}|etvypZL57__4?Wt)Olrh?rz-mJ8hzZxfS%HpjqQpxI1@`8`-+GzIJ7;ef!4Mwatz78*6l0^S{3F8Ky_;s&pl+Ii|q8'
    '*vdMdbeu5O>FNy%q1xFJWOWNWxK}9+)4XWVU+x}G`;UHh3Gyp$Jv0G3`;OkCUPbB%&?#oEgB>Flni3gc)BytszPD36)f5Vgxqs7MLnl%B(zf>-bf@=)sf;om'
    'NWdjWkNLC-jowz)&x<aPVIBk#o`?)4coiv3%#(Fk#7}xaw!qM!zg4mw6#@wuOB7Uv9Yiz0#ESytzW57vVopn+g&2J0YZI->fN{#aKw0ZcO_jTI&s|JEwmY})'
    'Isgr#o^FcfwzPJ}(Va821EtFvf=^L=6p02vJ7=hzH-lwhVR6)ulIuu<2@1t8^471i9G{YhIje<cniRD+%LEy9<2dlKV2@#paq%I&D$NNW0uHEPos<M0vgJB$'
    'dD3vFbrC^vkJ_Y7!oNb_uNdbt<(8;b1;G;#q{V<bEDQuL;z~y%kc%ZVXb}a8sg1oTq5v?<O`wd*`^Ngx55VLNdi!Dx*9o6Hw<y|#94#=m5b$U=bH4R@F-wBr'
    ';?M^&NG@_fOI03EATDZbIrn1L)^VZ=@wJpx+F>W!XE)V$in9$B_gB0%c3e%z9Xu35riIij_PK)1V?XKyX-45rP<Km!A%|B=q|dBy9EAt)0)v)7(F~d1F5g;P'
    'xC+8@Ouh=X+HZiyeQdGTxVKM+XW3a?KY(Z2Q;Vi-69C2?t#HjC&Erb660q;$vN;9LbKWAr0Wuj6nUMnn&_0uOsgLBrntGu@RE7>GBb=)g3RebxmKFVciFc)B'
    ';bSdo0a!L*6W%NU*aaDOx}Z?u4-R73>JEo>$0YlM(V>siGmK-Rk9>ag=~0NSQQvz!H(Caw2`+n(U+^#u5btx;L4nLYn%bV4T4l(mRXNq1Z?dnamR)SiI#;Ox'
    '<Bi6{4u)IzRoFCdz;p&e0VrBGdNBx3N9dJ;(wf7HZ}CtvDHb@!VL;1`PL_y;6HEe0hw^0EC+;bD6kX-%aLLKH`BZ651wl@PVi)v4cH)tyUXk3(vU%<~U2ICd'
    'Uh@9b92<5^?kF|KF56O@Xw9)(xvG}#6D`Zd#A522j|nO)>|z4?i_62tCcD;C%e4|}AoH>IQEK<PZ;EB>D#+}@YO*`0lp51En(y9)2%B6Lsi;z*6`;O1;;wlE'
    '{UJ|k(_*h5xmDW9Xrv)CzcEOI4-gA}%W07J(;1nQOIFiHp-O#7qtM`&j6#)gjv|*sr;S3D`jAGU!7mwwDp4A!Gw9cCxd-;ldlU~~y_d%X94BRdEC4l`Wy@SH'
    'c4mTHVQFb>q_m^bJcQ~js2-ZBym}7Ewi}>@M#aE#Y?~!=E|ybO!3jf@$XAc$9$w1MO21eTtodlUX4a}w;BWQ|tGvnaj3=w4#r_PklZpGyi7PcP;#`l0b+<%a'
    'dGbluI5jFDdnW}OTl9d8Ht7J1MI%&y#f?m8ykAf7`t7O<b)!y`jztVpl(VReYsD&cic!9W#p}<5c(+AXxr%Jl&UGZ&LqhX!0J@>RZNaC?^qO@{UeOy5hZBtY'
    '?T5$G<<rsKx?T4=%5K=%iuI*hiZ}+e1+pg<-;TvNs}5b26tRui?-4#^l^)^9g&65vkX!H0X+zI0nbtUX-oBKdAuze1fS%KC{;oi@y7gc7nwS)+V_ZkAgGcTh'
    '0;bNG>xy0#x<+r2v&*O=vaN#Xhv`e0p;W@^iWiaq9HLPIdUO!z%Z4s&zh}X$k4HT}6Xkr|`vkbQNP<+HHqq(Fhz650viNE-n?eCwkY#Ejlq{~WA2ui%d;0r^'
    'COxVGN^k^K3R%UZedwoFgJxzCO(s-Orkss!2C#5~yPAtBdSMW{gprIrlSrjRi(C}jmJga2(@z<bW?*Qk@E)*T><Xq6nPVhp%=^Oq5%b<Ktt+d>Y3{Z;rlna2'
    'ap3eTl<pwwS8WYjn->GhX>Lk#3IU&Zd!h1&g%&vHtVHw34->o5@;RPCAuRWvw6#KIWTVbukOsFad_FVj3}{W0;lwwC`;dk;^c)%}*PARjp!_Cmnj}u(VTPE-'
    '-ip_tW_+{lnztxlm)TTpx&{ez%8GFw^h;Uo^d;>ewU+kVSk>Bx#<W#T0{Ik=51A~~Ly2gGa~GQvWO`_5^jqR?BSIl}7WXhGA_y_s5ULm&X=F8>sW|*~7HH(~'
    '9us43qribebJ%BBP)w029XQFP;4V<H*s;cpPvMzYV_H1`7R7eYKfqHM!1Or9gl%F45ij(@tnQ#k10_|;HbyDk<g1+pKwwK~A4(s9FV@7#lhr!@s5q~!C0?x7'
    'itHS+?^jId_t5z1^~dPK;=!(+lI@JnjXJ@he>X&fl|O;Lv;S~!JQ`pmaL{uFtnbbdY6QRN1Y`oF-NA8Fu7eg8DrWJ^QH7DR3OIXwl-8wueF8B0_7Fu=f*W=4'
    'kdxgkr60F&aK<fPBp^gr@bpe@1I93y{3Inma}f4?WY;<WUBK<j&X<(|Jdgt}fi4&D=@VWcsaa<@kGeRdqU;cqycAO?_3>CTk4pi>$7B<b{?Y%*cKtK$of|q*'
    'rtEpR_FB`{!ddqQi_6181F)E8nKG@+QW*Z~t(gEG(*o^`!$FVj&uJS47i(wL+{jKT`5(M6FJt$N^8qP{>+xSc5s`7gY>=2^QKon>351D^JMt-`AM8A)y&KO_'
    '4RHd`lNHcIKee5_Nmeut?-pb{wyd_|?SH~U>-xT4$nHMu!TYOJ%)2a0e;8{WnVE3%KD1>gm2pjMSARlWVTTWO0O($&UcY#N*(UsC@&vtJAW<U^?HQpBBOxCM'
    'YqrbDQb)ItPRfKUT_ijo%h4cA6#{gCGk2;{J*`cXUp+o)wGB!IrS53A^;56>ZNa8O@72&K^GUm1N5?tC4vnRD(0NLv*PYudq>hs64HkWjBTAhkO3e|0j%o}_'
    '9}L><O%10!cX1&h3@{|j8laA~>=T1IdqI{SMoc{jcm(6HX{bZ{qirQNtky|@$-CMk%t|rl1lVE^qIiEig28<R{w>-B`8EQa!~t2$vZRiZ7rm)aLT-=$vX5q6'
    'Yow}~4AXd=Ak#j_Jp@agXM#(9=PAFp+<*3|V5!+zYCh9^s{d@z=seT!HJ^U^*#>;+>J<a@?)S6UaT5VN4@X%$hz`Oi2)NIaCpMm4)&y-oiX+(M(3VS;G;{%D'
    '<_jxybhd+IAIOeTKnJ-+pK-x<4d6W)c2hxMW9c-9v(>AfA@<uD*UE0;EM*&wT3gW6!8jbQN+RfFge3{x5$kybNd||*^POOn)xGC6Z!4~apXXn>H_uxob_j%s'
    'i5GXf<0Mh;S6IRkP2NGb1FV;39KT1I>gHdWClz=epmId$XoP9%Q#AbnB@ZUm=_a13XmL$NYkHOX%W>#ZscGRlRw%|CR)w2d!p?Sd83u;1SEja9E!Brzn;mz('
    'tQXcL7NT?#)YEaNg2_yF(+kW4Jp9L9oI8$3$L1Xt8jI1{m9*3vOlo&7cQ1y9D51G&<PwL_Dn3Dp#^`xBO=smZg)~oNK(T5AJ$~?18HPOng^^G7zu;kffl(n6'
    'sTaNrG*%r!7Pg&-j699p$b$(g(ZEw^QY+EeLn`9)j=^BM-BJxo)?L|ipJJsS))AIElQtsOkUs1$Wye_qd3dUJN<P`aO^iP%NY$#DDaS#Av&92RE}we=%M2F='
    'JAW~Nb{2dQD9yPB_U&`deG#vX9Ce(E<QRGs0%bP^mMl~Ze{keaQfiL)r<U);q6(ug8arMQVWKv|@WwqG#=UF71Xp4#693}@06}7(`*IKh58mIIzcyd<<^%jQ'
    'ncv&j@4kbN|LYr{xu(7uecHwWW2$T(5_iMf$9||eyGMLujw4uk(1b=+V88BN3lefs^rPIr7ZsL$-J2(OgyuXqHhn>nWj!8s-wGA7u_fSnZUpTtuk7tlV<#Fz'
    'E(#({i8II_F8<`2S_UZl*kX4aU2tyHy){O^;EoqOaSR_EuXx5-XKmA5dx9$#I+1swi6O(Zt6McW%3=Pw`Pw|Zgufp{Y9g`%k}6?XJ0=i0$t!yW2HW!7MahrI'
    '<bIE?^vI)aVcv~FP1G2Y<Ewt$P-IEkt|mO^Nz6w}D@Ma>2cvu%J}n?n%^$l1aO>Re;~#(6Qz!e5@oQgY&NRxUGG+V`7eTjKIvr#r7+5(C8In_&*jVpv$E3Mc'
    'xDRf6)Ato)7RiGl6+SfiRHjqsGPl-A$%KZ0QlGG*3p1J6bF6`|JPn*;+-a*dy0d4b2bYq6%QNf3s=wO~)!Aav3x^JIMFfZElq#Met33!xl*kK&a{IVH2}lFy'
    'h*L?V=B76^uUFQMK(mWCnznBZt0b?lhL;O17SoCa%|a%zh#=bH1XQJ6s0go!@!~=5OtU&AX>qY`9W|vWU*M<r%A$kwp`oyPaxk{;@W{+2^+C<a<CgLTYJCvv'
    'lHT@0%y%9qlNC-Rh*lPgMC9F2v(=-%AuMI2git7{t%V5ci_`kZ&kpNh3h*mtL$;#L0g&~85?FvJ^zl6K3SoGaSIqkNs(P*wPrWzpu*eZ^PN3m{DtqZrVa+CM'
    'Ud6%-Q1|ZK0xTF9u>FblQDic>1_}`XJ7yf`UfJcTebqg`lj6SxTbTD>T7@#^7&$RwJ{kC~0A|IG4CFxLqJyRd+e#y0PWvn^<-VXMWYiarlq+cHTHP_SPV_mU'
    'o!f9iR%<d<ev881NFJfC<l6nPew2h+P;tR^srGh<IXqoN5KwA_dUl-6t;}I|uK@<w1g%!1-dMyyay&ZfX02AUzD$o{Ofe(yV!7TBZw`T3O@c|#0hW(n>QB`h'
    '{2|Lmt=3Y#u~c887io9^ixZ(UG>lY4-D?3wPN=$Af12M<!noC1mVHPkQFkE=hXIVF)w)z)B=}jDU~88cRwv9-pr6hpV?fK)i4GufiQ0oYPw{g%P6F-KWkB>G'
    'q!A;WE^P8)AD)|w0Qwp1c=ymx(jZ&Ny;pHi%k*gwcJH)W&(xdP8qP>OPDT@iwp4Gzn@M+o#i1!GA?`j(ad0$L<i$x33p_=Sptn))@gWGKctnGF2I~gLnCc4H'
    '+d&eKN8ALI>icQtk3wprL4)r1huDQho(G(-L5#hmiNH4lKRsLk{6#ofFi3a>J<sBJKpngEEWu6UPX!TLV6drY=v9jQ4WVYaPI(3n!mQW9Iyk`H=!G!jXhdqP'
    'zI+KvBz_Mk^C`r@qI|}D6F_-PR}Fj{$9JjE4Mf{vqd~t3Cx`t731pE=!<*wEV=}_hKxT4M7wgaRi(@R^B*YzZE`>hcn|C<w;D@EBuwvMYeCp|<NLapp6MMAu'
    'G*0bbgwe}>lLrEiFAoD^THFXgeL3z7!VW%O!tZy&qcDvJcX76{Ux3ts|Eb9WUNXfL0LU2n$(>$&6fK0=0!|T3Adc`lydICX{C<GVH@IXr8X#eIuoRKXus2Qy'
    '3Pr;qThHkQtQ&YU8jAiAT~LmHUc65MfQEP@yyj`GK~3mp6l9o7?HC&(B8_n5v><RzEW?ZOkZBsG9UswJ>@Rf39Uvt5rvX@M;6xHBsjof!0|cv=Xf&TTOGI~*'
    'tUI+tGLCTRPbm{c;^VWpgkf0P$9_MPH5)iOoGSGZuiuhifhM5;jb$A&&u3mAXc);W|2{&-koOEprJ*%H9MCMomrGAys$XiT4^g)}5Y?LXCQZ2fa`)vBW<9!7'
    'Hs0uNcI^6=#UrLgpbgxnx_1{wVkr`<sXvV$l9yAlI2$<ABfopc$F)IxbP-<wiy|IQoMIHBI{;ya7Vl`_XDH=?tU;V&2`3ygz-AN5!AFp3(V`);i}^^@qLm>I'
    'NRCU~dWs1tTCJyf6^tf6NLA2UbBR~jXwt)g{w^(w29mgZ)8^$4y(BIRuX$O+AgEq|EIXrdEMgaLO%7<IBS_D3mFVtpKsXeQ<6IIZa1|`!8^qoM`loX@n|0ue'
    'bhGE8`pQcR+QQbMeg5((v3oSJ<(F2kZ{V00@i77A_gH)dF0+hp_!>JJ4K<vEU0lUW&>VgT5*GUh3?yA*Nn7kGG)XD&4aM;YP54G8`XclLeWl&-4sad5l$lZ0'
    'RBs1?pF~JLPm7nl+#AH%(qKqPUl9I-ghWD`p%ilK&pykqlgUEbyF-Jb*-sCH0S5UV`a>pD7+M;lF;G%uljb}PkEvzic<FHHk9fn!Zbf0gkGnvFi4*Tq9RTSu'
    'iE+=5d|<>h8BJ;do(F!1_d-P9L@+YOToxq5Zajz+=JL>Dc8IQ&c%M`^8{k_aW)KMaxQ&X5yF?p?vON-xo5U!yc!;7zlldq<iQcdTL`fof&uxe&e@q)2zFNG1'
    '%X(RS;6qdbO^C@p*&E`Ldx}`Ou5aPnW!kbqL(Amei;o~wCHX$eS$G!02siOe0H?{}FKv~BgFrU;)2ApUnxf}%^r>*fdmw;>4GDIrCDDI)z8k{IB*07b;s`}D'
    'fggZ&)IAD&xJTjqGyo$<K_>;-ge0Pd5bQ9KK_-X*iX0EpV-k21UYbD5$H#{t!Kmq(_#;{(%S-%}kc8Q!Cy;bG7b&G$nwsDUcbWWwb1aJ|+5l0&k47S|PVr(*'
    'dW{kSU?jiG1J6w83u-FzbuS1;_$+fL<;%5B?dB<Z{9aqjql8mY$3JF9BH+^PaCp#64mzTs%so~2$5|}z6W4+IfnD;GM?ut-W&S%p*y)X^bwzmQH7VQjwGFld'
    '9|sJ)9r>_L$!a(J@NRGgcZexhzi?V8>jD@Gxt<v#u478OlVQ})@$>u`m%tU+Kvr2ji7CjNe!_=M%ods`q~to2N-l-K+KBiz-=@|ECRSs#!)UvAMxelFZCDtf'
    'vZr)oU5)MxZ3n|qHff8yrk#kCZ3zAUGQQ*M&1?Kn<EfMH=;8S|x#Evgf3R`AW?Tg>OBJpMryJZqzHz<1eQo{5U$(bi+S*>b-oCYY^ZKoA&|K%|=P_}ouxoIF'
    'aZK=V2a^|%Gm-b$2fj{8A6Vd&;UL2hZ+wY*?4ckW7U&B(+UKrpz`QX{({8lw21h4SgE$q@ECBvF>cq#q=PrsLnUBfxFj2JnRs@9J$$1rSmR`azQJa2Y?ZzYE'
    '0VJmAbTn{Mq)fY6K@PlF|9s;vz%hr-+2^%U%hX%zxJiO|a5q?wDDMvF$1A0Grx3N)yt}?IM`*X8@LR<`tjG2NMgakP&`U(YQ9BQ)u-(jzP*FQ_+t2MR>bF%!'
    '4-(d5K?`tHhXkxFXLV*-z4KUBf1U;x$8N*McUBz(8&KYOw#xR2hqV*pRcIV8^0bQE#_qJcgPIU#lUC!Y#oFw(F#4d)(p9Cgh~ZzC;7{|?rRrIM&>iQk`IK(`'
    'o}aO{?Wu>hrW_$feqNEIY&Op_N;I5MY-?+|`G-ahp4dttP#gdI!{Amm-iZf2JA27#|HC?XrodSP8fR@1Y68)>S&m#hXKf2<vjfqJ=1h=SAS~E6S!9K$Vs*vK'
    '%W5AfIik_t{GGdasGZKQVAzB;qIq=aDfu+=)2w|bn20pWP$(CwDmjmo*FX*i!?yT|w@Hc%2M7KHbO@bp?iDP^TKAQw5huKdHHEf0X{RR7APVW0<%_PH8nEh$'
    'nWDxR4j@V^x>Fdfb`~KJXxET<)twL!n|3)y@t!eVBF_{DlhkAjH3dLJuZ(17r`FVBta!@SzH=9Aw<Hut3544s;5E<6|G{8eJJ{L1T1f(gmbs_dK#a|zRoqW%'
    '&O|lE8PSF<a)h<X%F`CJPP1)QL#Hhpqg8ViS!+a_I`rt|-z6o*<e@_({wgU|ARZ`hy|Y*LW1@d?PgZ`loUsB(3=+9~Kw#jkJ;tiQI6aW~p%?}SL7D%U1#`-Z'
    'F^&SEueQ&@69O&x&HQw}wT8kH=cBK<kJG0wxfzySey3DxhyLAjz{s`XR#C2VK|I4cbcE5KeLX^lDyBqS^JZC|WiuOQ<uePq{bQNhF#DsNT2nXqBbeH7HcsoB'
    '%Kyx~7A3Kd`0f9O{FZG9AMxCec<x6$_e12lSeS)cJDT3Uor7!(H8%?M*WLa>o3-<<KM2s7q-K5$hX>+YqkgHTb8A%<C|N+}9Ijpw!f8d&(g~<)RTk^ZHLp=O'
    'j44<m3D5(mJq#l}>~()6-}&9SgF)Q!2S(D_{1<B9c-WL?!6<E^;Yagw7ViOhVLxlp85WW5%AbOwT)uf=;A)gTZrxbW8ECwctuWE;J&OM8x#sK%jtPO8b`f7;'
    'LL78C%fz&yURmw3Nt#yY)Cp_LYNyQ{7Ce3Su+;P!q!Q<uMJpl(7IbkkXq7wX!d(AnoC>@03^QR@nQbC)P=8YMFeVS8peH(Kc3vK86?O{;4{O=Gs@3_DmDVcz'
    ';Hd8)j#kNgtG3Cnji<7H@@{25mh>qvs-9dW<?f_EILc{zl=)a*mbNVG*)}tpyOQ}>NZ(tKK}#o-XlE3sAr}VLT*{z!=JA)K%KY9b@IN^Yl1ZDwgMwKbFzS@I'
    'kdq88yVHPM#ZT?IDu2e_7wS1%V#1$s3ZU@WW<Y~J!vxsiPscyPp9%d6^eouddOaiZ?VeA^y%{w@hwg%Ba!`)ol@?#cvmafiF2?DE?1N-qLTOl^8I}o1lPa&('
    'ycg9yo%A*Tc3AU{;M+Ct;zgaqi!BK~n)%8R1aV3c9RX_B;2-rQpmr;b8nsw+03v@G+M~|?wR`ACk&y~ob%3><AdJ%Su(CTB9UhH7*<JWp)IA&>eG&tRxny;w'
    'zL&NP!}6XyrQ1iRlrSg=<$|7wYmVeWTW)u#=NIBss?|D`v?`N_(rmS<XgLD1xWYwRIpCT{ZMIZDt0mN*H!6bD6uyfU_)1$q(_;OScLD!a>)inm7oO)fs<2q<'
    'qaYcMGn$Mh9`Hip=A}zTP~^Ktmp#(u*%bzj*&`}U5o6eDp8@craiv-(FThIG6=P$pzv;TOb|7l#!+GZX6-+7sy}N{t0lk>7NeWaNXtrKv!H_P)87J_iKZt#-'
    'flBL2*|D?qI;`j?uzyTi<yrMgbO4rdycN^TRde{vRL?56(o8*y*y_CXxp*%E&u>fpvW2gFrd!(Yntl0lPYu<f;O@-p86yjO-&CQkYD6V!(^0tN*WR~QYKwcW'
    '+WbT0(V2hl)8ViU{r~jG=4tinuW63ygCkingVEj9Zb#v@_08N>t>W@uV}rkTUoGBG(2lza&A7ZuCr-kH5O2=l<=Kssgx#jw(siLMd&GnP=NBhm{lfjXfAHX^'
    'zr1;6^T9V>d+_Za|Nh<AAAIlE58n9c{onoc!B4)=FCTpC7Z1Mi^OLW9_vD|x%x_$_2S?s1qjM{s4&nFje&OMde|-P#SE2RazxzcyUeXxFil_HR)cnoscFoiB'
    '7V>~kzWSAifB)Aop2A2h{J@^#lh5uDcQuj9i1LnAlKJ<4_sx@czjyNH4<7!{H%`9w+mpX~^W?SPy#Lz2{QIk4s_guae}DDMdl#zk7tnd~g?Apl^XAE~e{uhv'
    '@85s>H}v(h^!2mz#rZO9>ShnD(a4W@f5-0Jl>WS8f~iwnUZqjJ=j@-mbQ4Q|*2wZ&j1kv*WMa}7)ck`l{LO=Zee1!uzj*TI-#q-v&p-I~tMC8lGdRIt|20g^'
    '$+ux*em##4s5TWivnHxWFPPM(<By|<jnA&mb>d@`U@%b*&i4FsyStypceJ?S_s1%`yS)q5n){!f|M48ms;mQi0&8eV$W-(2IgmO3+#lO<!v23d&a*VSzKv)S'
    'Z?FKt07Q5UJg8^`uo#0KCVUJc#{~X=406W={*Qr3=%kuYd1?psc4O!jZXzVxL|NsyQALmNtTK^5j+^pLvsx|T1T?@M1q@FBu<>c7867g$fU4>|zTnguSfUq-'
    '8>Hy5QRCOmA|~2|i#B0{em5zZCs!*JZc^bU6(;1ho3e1zLKub}C5<*-LQ}zJfJCu{s&~;VDN4AVaM8)MqD=+0$xxe9ib|+WhUzLRW3VfErx)_3KrjW#6tn8*'
    '9nC*a_%r>b@KN`yq<Dc4&1Z#Y6@xzQ*$^@Bc>zbDy6CH&w8DJCrNAAumm2${Wk6iyoC#kS&5yXeXg+ay<vpp>MeDANhUQ|D)ZjRKr2fd6qj_^og;r8b98FtE'
    'No}0GsBl45WJ@&#sp`5mB1ggn>{1gZ_1I2MYh4X;_W<Wmp-zW!kuoA_bBnW?KQa?uT6A!gv`rcGT70v80p$foG#C##Z;2eTF=ZOaJy*>(M$e^(@t_AF%mG;h'
    'TpT~iU_EDcm7_yUfWZYEM51as?xT`NKibIlTz8&Q{WQp2`4$`*81RxKWP5`Kq9p*mkQ9cBXr@|{zXtE$!wT5mO1aN5Dn?;64t67V8N}={WkntjD+ZbFVFE&$'
    'GM=sh@i`<O`t1hC_(K#)I+;=!fxk>cZ#sv*6JBVn>|r8WjERMZ>Xd;3K3<*yA0|%9N|b*gd^KmnS5w1RvpTisekZMnnOG1_RK4d4E8&!J<;bs7^kwg~mL<L7'
    'ZCRC<tFu5aO<7cqHf_yF=^*cENu#P);Y^DtUy@p5Hpw&l!B1j>u+T~wusZI96<}?xfj{i@e6PFWS@Yf4bMw4n)p)StEu2m<8o^YjGI(r&@P|Cq!H8xZssTqM'
    '8Yn)seVI#6uWoo!D4oFG*Bz?G!|(m#!5iN_`N`it{PB-I_{YC~@XIfM@bz!L|FfUX%RA{U4TG})-+$+a_uu|9s3PUnPgPE|uPOg)>5r#2Qh)Nl?(Rkv_@`R='
    'uPg8q{=4w$|D#&e8EXjm>do!djg8_>s>P11xOa-v6^bG#rAn5uPg~8PHFUbF@px1Y2g#UvhwP0mv1;xw@YSWNmD1V$_uhE;-cMPbMRUnN{=<W}K7S@sD}it-'
    'wP9ChA~R<LkBQQ>PMn*>G#F>5tv@(jDFSh6v%2mx2*~+Ya8yfvE47H6eBpZ^eDeors+yUKprl@R2Fekga}ke2@hWX;2aQ9!>E4q$A+;nS!JnPQg=c^N?%yo@'
    '{@rix{r=rAGHtoGM~m8Ax}TuGsUDLB+BAphASh8l9YN^D6(#1m1!oh!Pp5aS*;&cF{RgFSgLp}7nr?nk=k3$?X{I9}A|NH^Nl5&kdGp~vz2hQUba9`5_}%~S'
    '!B4)=*4u~geC@$MeEZ}_p9Ls0P`T>WtaJ{1a7q_HUDlQA&hOv-`xW><E`;>SGOqd|Qa&Kp9;}0|R1rb%b`=N3pO?aorpjm>IVsXP{yP!>9t&~djI5odbhHEr'
    'vS-+W(76HaK`jAps#-16W;;Qese=IXVjDx12_t2AM8iBDcvj}as6Ssli(-Sg#rR_yNsWIZ4gk}pX|xU)AG)nldeTc-zs1`VXRws;G3BK~+bN2mHa5%3s&!WZ'
    'J4Nq9#w_7IXP|E7^iXFD)AHx(6^)rxy*|tApjj&ig19s~^PJaMT%0lj7mwV@0V>JV>{dYq0`<rxo|?x5`6L%ehpp<i-oh)*veqjod#CKPMsu!lu9O+Rc*QzV'
    'Q`Wiz7dr}=`&^+cv9SduZ?OQyM6XWpVtr~oFIk*nU$<2nEzi(5)i0O!Qc_9YU#=w9>J%z5w)~Ypx+$)Y;!#Bbta;WA2D8sH0n)gZ1xSk;u}mV`L52lVM{9l%'
    '*;40_4&tFa<az^B59-`!3u4i_DaA71<lVYJ=f=<FC!C-D*el14d$m_4_~*EZzwi%2zxSLx)co|v+y}eDZqwHeSWojZ3YE`8JCsfjcCd~nHj*nN8h+(v$&_Vl'
    'hZ1bFgB1PU(|txLZIHO_M|Z0M#)r%rFOzDscZRD_JB|huX|b=F{z_9mafYShH}tKc4hjF3s0I6k9-ou@*w|G?ii+I9X0g6hD~(KAIyExM>9V~e1Bwg`76N^0'
    'QzC>OXCRX1;hAFgnI?7){_5dQNzfk%xw#vUqfF&#$!~NjqY@*UmT5qqLhbC8`KFOOQ6Y~?)iDz@`?JaEIkiHew{)mxUmeJa$!N)8Znv(VX2Rc5q8hW+qN`{S'
    'VJcLS7e_cia>Wjbt&OJ~8yC^9`+@}5sw$C!(g{$^P-G&3A(Z8CE7mSy?Jl4t3DJs`9aHQ~4x__2^TRmyn!>Ek3fHPR)v`e+e8d+qMJR41I+t|LQkdesN+hx8'
    'HR(piG9BXVBz+F#>U}vK=ozr911_T0MPTJf()dRivXtP=F;ZHBM-5WRsI-}EIr6We>o(o(BwtiYpic$Q{mM(9ch=Zrzhv4teF2&*T}+`=$JRN%(aAavd}n24'
    'p|SU<G<qhMlK5d7cppcf*)@5S#_cE_k47Z1aS!PV`4#UK{70XLdxg6*>@YSJMdylDb|J-O7b*!>o)&p*W2>y56^(2ypW2khoN;AST5$!Jl@)hJm^lhh4+e8Z'
    '>CYTLch)Gjvq!So-gCKpA`iL&QSL%07VOOT62G5Od_5Gj$^Owe4K!RWWOI0Qc0n?4VTH<21bwLxV>jx%ywfd*bW?z?*-r0<U|zMaqH^G0cBE<r1pLuSi=zbX'
    'oAdYI{>cY_`-=xZ`Qh*1{rh?Ef<mltW5?@RBj8v&3yq>Hyr3zOnlKnOh_EDtX{E^>qolOx7_>WICf)*+;RpvzAgs&B3WX5OrxcVUXi~ZPd6z2tVT#Zn?mF~U'
    'x=@uR#7MDPr;8xqq2Jbu>ZI&|7WDLhZxdcO33!zc0dkneavRgdpqk#TX)H@kwOnFG2da<E#HNEIS(=VA{0K&1YnEpAC8cw4Xb)@?=nDz|)bWq&Y~r!7NJ<0c'
    '>c{!L*6UL|)^o!});%PVOdRw0gF!`|@HiQvsh~>^K0klTF_LKM94G!I0-V-7+Pmh@Ztj|Q;#(@295eF%&;IS?XTLsq?`tPteB*<+eyQ9o9(@1R2fzF>pAf$P'
    'yKlY!*6R=d^|$wb_ftGAk52yWUmw2n&6A)0{r%s)_24r<1Gc;yHR_&T^W<B<h1VzF{$~uF`POgmzx}0?H@|oCrSD;w1U2^HpWiz9hcC0+5LbNvYu|bB*5|2#'
    'rUZs<y!Xb*cfR`Id#|1R->-|}0E3*4poIrNd-ML=|8nx`H$Qmwi+tkg8HfVJ1Kt7cfOS<{cU;o&MNRKU_us=_-hbzvlP|qRLtF4pKL5spAHQ|~ov%N9=kvRB'
    '55JAW+nxLUyRQR+FfNBiWDWn{U;WH%V|X1Vpos31&;9cU-}y&{beN|PUVG!=XMcV2)7Kw-;j4Vad-88Ty#L;}8U4S1_lxiU(`Qe9_FbzRQ1ax>FBN-2;WM9q'
    '@DE>o@OM8u`Rb1ze*3!*e)4@T2A|&l@((}w&OhjQqVnGq{x@Ix62oVIFLtV6cM6`q^K=GCg@S)zw?FyG-@)U9zx&FAzx%E9{`ov3`Q+EX68<Ki`;`+8`v$?J'
    'eDLaLPk!)ECqI9ayN6Bu`d9D2^~00TeU7K#{h$5t<OhFkkpuOAsjUAneJ5Y|e;<DRpKo5-eE+ZBJNeSLPJa8hCtvtEkT;&Vf8({2&wuXZr@uzR`0HPteDy0Q'
    'Z~hF2{JAfP7Jv6mAl8RpeC@$cf5|aU_kZ)1ldt{D2d}-Zs&KfNSdf}iovWYA=fE?T`R)%5=Kvyrlh;1~@U5RBmVWeEAVR120DNTvU@ro_j95Fn(TwX>My!=G'
    's0%Hf1x``e&TQM6(XR8@t!1;B$^KY02lp8=dcZt<=gs$j_Zrg?64VSaaTqX&o1`2_`Sr7<Liki-pu=$x=<c`)3WfFl{KFsr_?7TN;~sKyT4X|eFDw6w?Crgh'
    'g5~l#ZP?EC&IqNbddyU9o}G5)-EvJj9^?VpR%EV*W<x$Ig~C5TWE~CsZcv%`{{7W2<MrhD&rCSerx-z%*3ng1E?Ui_jL)&a<>*dZ;<#2(z(2?@$DU^On#J-p'
    'gZ1lCEMVuhf}P6}b`ERUnJr@HvWlJEGIk#8*x4;)4lCJoyMSgmlayFJ6Q=`-<-y<n3bx%>G)-?IR{`%8EE$fisBC$jQ*~EMxCGwgR^Der_^c}YhoHuv!wUE4'
    '!q7Raaz)7iODer`9#zUWKl}cxe+7bQ6Qp^N0zkEU|BL^M*P(z|3X(g@_@H(E=IfNs;5Cp6mEer;zxCS&)?-QX_1~O)_q!}Q-hc0FtQ-MVpZwqrkSh2G&^Cx2'
    'Aim!J-FLXy2jBn7gYUh2|Lv~`SlH$}-vHg|<cIHZ2`KvD_0QaY`_=bf`vo@p)z<;a{kPvhf&RV!dGfV4idC=^)J>3RAAIlMPTqYJ<(gCNSlYcfVksc|v6n;X'
    '##l2>=ISCtk?lm|cQIC0TQGX7k80~W2UVVrBF{;Urz}M5XK^taSUcaT03|{{)$FC^L3d_@Jj0>k`LKEhdFXVd=L`yu4)C-CY;B*xn39v9f1Y!>knKpaf^Yp6'
    'bZ3Wdi?$|Y9D1=_UnT}3j4B3mz@?80Eb$Q3WPpdf^M&_+`!&$btulfQ5ajuDW7j2Zp6g`wBN_7}8FMBXv*a+TFSx!hG%xF*)kywCRrs5JD<PnL&~BfWp?pPM'
    'ExMz-u!%qfBP};2OH!rNEOC{3^2s}Xa*%4Ca(IC$`^c?Elf9KvSF?E@?TP?NTda68ywK2xhH}@1DShNCBq=Kozpw*z_$ct!An?;b-L~mx9=^ytgg}ga#S6mh'
    'Fu(u?(LkIK+`6qMyRh$aCbdor^^T+eileW2-CUzUk01kx9P7+5tWBb9287t)aGcFS*5m|NPb0_O2+5GHe5IUn75a<8JBMuM8lqzwZ0-l6acg((nZ*~v%e!;='
    'QJ7GbcIUPeAHBhSObYl)4uoA`xa<?&#>RDzFLv`dHC`Br#Mt?jZ!kpdW>`vj*v6Z;>3ZQxoYoO+{pC1}jK1#9X}88<d1$z5!>)hwPCMdDFY`nL+UG0`+D91E'
    '5)3;*4;_|LGhK&Pmad~iq>Tpw9RpFXugBY0DZ^Z{f!}k}>=WL{!|js6<Lwb@@Wp#?&wzs*l7$291Ew7`N2Pr%+_xw02QB@);5D=%uC5U+<s-{47Qvrtkm73R'
    '${+^*N4;-N8I>Q@wVS8&K?$CLt@0hJ2sY@=YZOO!4VR!@^+vm$SYyO@!*mRtgT3FrI8uBWM};dTbNXoIKXt~N<Fb|iteL4!<#+Y7X4c1Li4(!8R&jBrz$;r%'
    '8~)XC6b0QZPTC-wx8`Ad<MwDmo(b*Dj}G+<Gz*zqTTRPnU6i12jpj4b!`hey4T*HmWatl|xHcK&R8hP>+6ayr_|d@_<uQ3Du6-|bB}Z$u;776{Y71(xdSReY'
    'fG=DqUCC#q2p?DQRL~ME;i6&=rj|Ec^U7Laay`9`x`GA2$fCm!Ip4y&H17rp4d!kWGj77p5Cwjsjj%>xDY{A}x<RNwkKl0^4?M)eF;|wlbltmUE-U@IV)J@-'
    'W@G>?!_6Y-Y0JE7)wS+a=Ah@|{pb}{LF*aGMxMIp-R_@#RkC?4ya+S6)M(bc##1#<Lyqe%WUF#>arv2&%GRaJR>fzZ!CQ4Hz+t&+ZLagz{bQJ^VVq1nSnb{m'
    'w{GJeYi`c>^0~<`b0=-samjndA}4FPb6c%f!WCNtuzh6M)d`)UMAl_?m8n##_w*o<xJ@o*Vz+El22{MdVRw!Q9v!L5_Rl>Rv+2)Y<#f<67C=QlFbxZ-KMd56'
    'sa;BDllN-gT%UDq1qtfXndCg`B0??*661^HpK;3HjgLSyP2gE(g~6~1?~QkgVVku>9WH-R`G--+EW<y>`8HPcV6-I7@+8_h#y~&LDh^yxCU^F8TUj*xn2`kT'
    'P|#YjRxw=U4hIy<=A4Na<j0*msH^B*ohho%29Wof%UadWgNpyK;d6PpuaMZ(Jq)^cMj$0+ZP2MO=$LlyDQh~fF-^*1`ZonFJv2@Avz7nUzdL*%YX;h&%3&Pe'
    'Np+#AY7EiJ@_t%!cO_klOX<0M7FTp&T8?z(lIfXf6)XR)Kd89&L6#~>flY%9)|`h$$7JHAx^jEwzcMQ=@ATS^W(mXEeQ}Q40j(S5jNMSGZ6WXQ!fj9wN(s~U'
    'D8!-ZaV&htq#nYe_z0I@u|w}D*F4U~sLpBU3wkc~fHJo=FI@s<K&w|!JnbCzvU1hAOhqLPtQ>ZVYA+vq8TS4fe_?u7HhI3RfUnB{IYYD{s>$^F<yIYG@UB?!'
    'cq|*MH?I7jH@Bqlf~GfrfM^$v19%+BOKoQchw9o4F3YtUT&=}6QFV^gpc(yJP7G+VkbELI#z3-)LaZ*VLz&^7UYKZKi%r(?BYJ_TLiaAdB4<N+o?jGx=l3ji'
    '0aVp!=E}->Vt20o^5}r2ih6dO!S{RUOK~(`_59T9>zhISC<(KmBIm9;cdrIQNg8&yHs@AencG;uvUX!@4WZ1f%$=54!do5TC5+w%@SP}~o{8e)z|fF^H0vPw'
    '{e>TmeGu=Oi;K(UwGOjvv~uy{(a}-er-tfra&U3Ljina_=C;>1uWxy)H?DeDZr-@MzP*0)#+LW|%}ww2)>_TmT)VY-^Xly@__{`=uC8xwZ?0dyjc){qM%}v_'
    '^hp81BnG(=X?IQzZO%&%k=BQSA9)#!84p}iVv-=r^tdK%mg6+QtYM=B_DFK4Ry2%-F=s{+cE-d^VRCxdE-;kN#M_eDgBs945|0lKy=T3cB2b}R+#L_`HjiTG'
    ';-uI+JlaW6-}K_62xMsJ7gP?ji3g(bAuR4sQJWH!t4en0lR@tQxN-*C1QinKFnbXk_ycc^z!!Tm#%N?7M&N;Rj*w(m03T3RLW;@GN#8;cUa5UN4@hFzK|mdo'
    '4+H8tvOIcA0eA3iIE+=NL`f0YfPlEky7xSqPoXEHaRP!SG@}r#2%we^O~8Rs;doN75>~l(e1tn3Faww~k782!H*SDl7nJt^%Sk}|jqrf3-x2T$oJ(vw9d{2!'
    '=W5>3VL&(lVxX3N0&d{vC`9rCFqIJcLNk>fh9d;k5BnKuOqdkYt6W<AvnsV01Ckj33eq@}5sEln37!NoNPyW1B0vQyohEp#r^fWZ6psNvuwn3TAjYar18BE~'
    'mhOhVF+xl{gIolZ;23(1d-nnMe;B4IEd(MxUK}(>F6!HYCcC&kNzt^350A@(1T=a6%AWUkuo)0cLKr2VTdNrlqwZi#Xn_?<8^a(RhS&v|)->*CM@VrgHR+++'
    'R#OX;AW1k}q9)lxKRg)o6&@a1G))e3v-2_#hhqSKG~su=_hYY6n1GLXYZ3HatsEsd`I5-!%|JZ%J>O$Ac`RKS!<0a*(c-Cph)XS|{)y4bD_DBwxD6J^mKdHv'
    '0L06B81zD)0!Pi!zZfTXicI4O-crAbSs?}Hivtu4F_?U5Fb>7&hCZI8-1Wl&#wrQ^pfRr+@?E6Kt}lqo&-sSrfq)H^%q0_Kj0Ga8kmqL^?ni_u*<T5x0%Hh{'
    '$*&e`ilZH>iA7iG^dmfmbvB5Pss=Ky21$4q5PCQ85HG2T7^DYm$c+kt0||^#A-mu4F_SjX0xr`Ywt`Cnh>6)5Hbzs23*_i9>>lde0%i{M3FHS!z}C0a@cO8M'
    'DC>9PL_T499iDT2jY(*@GX^Q$cLvkoLv!(fR*Di~9kXBL+lrO-4c1WeY#a&<ayChtM1n8&mn0baVZLn9a7ILfxS^4Nj2R3jurQ)KguD(A4w6g+I$BjuKOIl?'
    'K@`L7q^57G3bTc7A%=puZ_NJ{WcorlaL>KHDAY>U8&!xD%TVsVsxt`9m>i<%o{(Y`sF-m?)u2Gx5Ng^gLCX-G#10g1Psbf#DuSy>iA5xGfX-6~#oC~jh}jp!'
    '5jEemJ5JqHbP)&HHnoq$-w6PreZa;Pv1c|30p>~#E6)zZGJ;~;P%prUx+Df>UBk)fpa%giu>@;IB(#hpF$o?n3Iq2+jxNMjmgb9xrXj6O-LP_|rndtB<-G!4'
    '84j>E<<F?;JDZZBI9o?*uz1*@#{q6KUD`}V5vBv&Az8*zVw#Ton$B7ba?p^DxC2SKJ5EU^peBdJNrfExBJn|O_X>_BBFy1R(uC2Zqp&-U$0;ml(8%u~Lrij^'
    'O37-V{~km>Cp(~-BJ4Ve5;@rJ+zr6G@9Ar|u7{{-%i5OI)$rw_JY7=KQHi{BXf^E}LZ3Q8fC(zQ0r7e0yWWttF4N$XV<4shHr$N?^SrgA)U2&Y$)cKd?*$ad'
    'u;nWX5mF}ew#K{(3(|G!NBYv!nRWoXsi&dIL(T&ICIOJ72#_pcq#zuQf^4i^unSCa&^roIHi_bBfyf|*VdLWh`s$$#D2XTjAe$`olK^caAQ0Y-yU6E@`@YaF'
    'u}P`JK^1WzRAhk~yC01^P#JKE7RrGSl&{`Ew|PHK>5ULVbZyX37Zjf+k-N~QW7j8!Bj;&Jo2OfRjoyEXNvc4-V3gsaOEbAZsf!|sYZd)5aJbqeg18UR4*k0U'
    '$%U#*q@~CGK8l7hI++Y=;x90oILUZ^6n79Jwh$zV`>4So8hEN?yZ&f2Koyo!rV*-<r-{yW2Yv`h=d$|v0F?wqN3r6l5v=Ak^^=emYafsIq+%3=YIoE*K_#t%'
    'c8!iivCt@oAiOA*fa=*5)R<Ur7CSPGfQ3S{d!mgaoDaDf*1h#Ul0mLerNF(AzSY!Z;Q@ETKk)H0F+O1tsO0-uE|Vv5nl2F5Jjx+BMrn(mfdG7O;2)*qFvHOf'
    'f&<<m0Bf@Qn&e=y%4tlGHVp21D)e^^P&c2Vi5!`nl_ANt0EeYElO&aJB~@FoYNV2quQ#zJO8JX-4P37{hl*zWR0?<KIx4A9p%}oS!r7CoVY%*Y2D%BVPOT69'
    'NzUzT#ss`Elv0q%qNd1%G&3k=LTh7Sd~DrJ=-)WswM{LY_xTd;R?9Uh!iq*@!yw@4?#BbrsaOb*>`cpIQ1Pog+%Zty0rnZY$hrzla0uf>My5;4v5`)-Fj}8>'
    '4qFNNbKYiEvmLEjwhiS1IO<_&5@5Sgg5(6MM;IZ^unwB)JwV2%C>(*JDd2!`5-=dU@vhc*5@fLCYEs<RYyzZLK&J{CsMfY>GbfH3uDN{otO;tbA-n7aC}q`j'
    'k%K5U%NL*+Ae%KDJu2{8Q_^F$DuGb<dNB#Xum#~r5vjZelT6#zr08!C)<#{girh@mf>0lE)<NjIxxId64a7O}dqm9RS`-bTtgbhvuQ-i0I2KO<!!&U^EUCo$'
    'Fq?jll+~OD11EZsTag2b4oC1i;ut&_8bxgu{OWMrsCMF)2v2u$)!~Rb`4XnHjdVqJ)R(=?5uRhoq_A{qFaBI-^#;KiYu&VOQJQ(le{fel$oI%%194JwpxT#`'
    'p=L!BdW8d1eS6`MoB>KCPZ>Z=dJ8ztiJBpF^+e4Z<&VGzRd)N3)i`8p4qR)~PBI4T<jRf+x<{@9qa<T>NUSyDV-w?aKy501uZO=ARF`zp)WOK!3oOmDC~7<f'
    '=muyEixe`nmY~OGZ#<M@u0c7H*RTpHC$+%n2+PtY0;q5<EV6=uy2a8$GA<AzBd7fI#EGa}8zDKDjtW>#uxx7jj35-{MxJ*Y?R{aNT&6X-*dg|7hC{~@7Q@kw'
    'uHwF<KQ(O)^-0^Dl;|+J=})aI0yx&v0GnMX9G@8*wUX4)a81%T5|nA_Q7Vq5KUni}iS#6K3c=cLE;Xig?{)<HY)W$%979{(5LIY`sU6bf_L7M$c4}5K%{*3W'
    '64Mj}Y}2+4v1r)QE!}6*>x4{39n~m@VX&~<QzxSg;6|KbEp>=QJ5MKOjSiRb0V#*LzfkAWF>DiQ&<prL3fF`_GomRLLD&cjs8rh20ceD@2qt2+kUAF}2VI@@'
    '6F*i+NrD4E;R7yPX;B9dPu0<DlI4^Xxuqt3_u_zK8AwpqPNfk2;&_jx9Cd!;578p0Bp<r!J`rwv^a$M&^vs1N0n1L;G-H-fbd%td<4~Mm;f9~alz@$=79<1o'
    'Etyb%$nPrc!XU+j<mw7qg9|obxwho=iG9McH%N3nUH7hrDJfofqTKghM8B2@PgTW=P`&L;SO+Hk9ThZ<G0=39;*;CgYWc(oZjk1^tzdW2_)ySt_2OtAHD<Yr'
    'wldfZcIQ^Ny!9;*)h@4Yt#2tLzqr1A?dI)m@5R;4&D9&*>uX!y%}w2h;O6t*>W!DYzg)j@wI-u*V7(_e(tJ1}ahslIip^IP*<yXky(XY46T(QHN*ufwaJs#|'
    'y|GpU^xs%mzw!L$`i&RXuCLwLu6ftjHm_WRey(0#-&o&%iKyxM_3ayLTkP$yDuLWu-Gr&Ty|KFK-MYPb>*m%P@5g*{Il$8l7{Ume4e6wVPJ~#ov?vFrFo{P='
    'h%!A5x(}3wWr!AYR;<}%*^-&2AUffQ6(bK*V)|*^4VAXd{9YXWlF3&;`z>n3k`AA#!yAcMthN#Qop3<M`|G$ldmtu7d{Y=hfmZ{vr9wABW7kbL@*E9lDbvjl'
    'QE)H_4=~QCT2qJdHN#?@+mcSD{t8PIc!n?tJ0$y2_YTmuksp=FMl*DjNa^(6y;_*78aoRbdDP4eLTXwV7ifxo^iDEP46wTNTFCtugu}UhBo6Bf<X}F!KtYGC'
    '#&{qopi0)QVxv8b7pFGT+>j%3zgWsLp>(YH&Y0hXk(g4A-5R!s%G9HJ*)JUVAZF4%h~wT-I54b|cVO3!M<a3+M7d#%eee6>0HXlNSzyo~N4W?@Tcpct1rHsN'
    'j&+=H(?JRZganBqy=~u<K$Owm@7)dQsJ1WsOkgz#L`WA50bj0$XY1Z-7q<&Uh2+TC_-ekDYYXH>y0h6_y@dnIsfT@1WbGctG22+ljB2_7kWCjvyna9&2G}3<'
    'k!}m3fsEM5DmdbVC@>gC=mw+N*%-M4*?lkW41{Tmq??P#h$yOt=n{s6Yey(%VQQW-fm(7cK0*zO^-zU0!m0*2A0qkRL<9ZsMG5KR6o#zC;x%&DoVQYsYwTSp'
    'i))NNH$7?eAgs=)Y=wR1+PMCC`4fKnIevOUAG92<LU(%?4fK98B-SK_afL~3b&r!IKdBSOX^dAOKUZO+tracPor#b~v@uN(3prwy_<y94r6ysj-bh+ryKxnF'
    'U6)6nD6x9$78F|l#0t(O*&Kk;PK0BH?u&@us5?j62^M+_&RC@;d^}8}gp$N#SVhSQBO!z`Sj+7J{V*8xQV;eHSpU2obnv7!$ndmqXD>f677m<ZJDf<;B7Q5>'
    'Mok~DdzGtkG_U*^^kpj>_%BtDw0=@`QxHOcEI>0*eGyuPwzKP}#JGCW3Gn)3bs$T|0qz1Y52!gv@nnok3sbD(@Kl(o2#5}4TvpmhE*;73TAuNCg50f!PHI(O'
    'Fw_80MPBLf5&1KSZ+>c>!3rN8>?%-Vm^%pw1j;jDW$(#NCU_4bo}f$0$j>vIi;ZU%nv2b*C){GaRra?KB?YncG;PZyqnhsO>Q%6S=0;lmbAlo@8M-lmwl`s_'
    'm$E<Il!c-tu}no<xhSL=58@74;C#anD2dwF!@y7Rs1@G73(#><X%@l0Ohyszas)>+)fFAG@R&j`Fhy>EM4??u>x{pxATYWiX`VzNpdJ;T39<uh8<FjvD6KEK'
    'n4cC02Tje@=8~k>iGr++bXOlu=2qtTn2{rzb|bb*;cs#N=tbln97X!KdKAgeQY=BsRO&syE6a6|Xw#$)8~Y%wgBFZ<oyc0NphL7b*Iqyu=$Ax|I`*lJ<z*?@'
    'SbUJw+7!Gh!d7c^lFCugHE$Fi2N)2X9*+C{LBRR+c-3zPDLF7)#lc@5#N9iUD5_tlfFbEk&sjWWNso$pR9127+k#tdZeG3Yb?_5Dt!-_-uzEukT1^i07$gi5'
    '?gu`}@mY1nyK?A9Q82*w9oAUMQHxv=fpKF|kn?h@215Xu9Iu(PfTV)Wzy~al9oF@s<e`X|<AU(ku5e}x*{Rl3%QYQ-YpJOR7m$UdQbZn|O82XE1+{9X&Y-y0'
    '2F9#J^(z$mT9GXp&DFf6n%98;T=kpI`rZW-z^u@g%Ahs9C(j3QxeVgcc_CgZgShO1$k@!mSpWw~Fc>qyjljP%*$g&rS3s8|@wuH2;t^h7)<m(B_WNR1^C$A?'
    'XWz7syVhc5C!hJ6HjCQtCGGdJ_WROa&0DNDsye{}VFRD;33@%&sq(lXGjPxtD|x$>V?7oBakHdcQ-;j?S*3Ce(D5qTM%AdhR8n`TP!|)S7wa#V)L(YihyO1X'
    'VO)|h2uY<C;s48AS)aSL;V4YdAA5j4{H(*me3N4drWvB`Ad%dzPntTWl9zr#<}I!w5DdEXBtg009UJ@=7~-iM6&dHO+;!nbko}4C))^Qp4D?i13VSrtyK4t?'
    'HRPz?e6;FIXRAKb5=PrfD<2=M+p=#>jE^h*x@3L*kjNezwdnr88B&XoX<$yP`jP-5E?X7wi9NoBjesneeeblzQWH*3Ae_R;#4$s0o$%v|w-z1ZfO_5*65Ikn'
    'j0b$)^8%mtqA6vyhc4MZIn&E-DM&A(@SmYp#V`qFj7i|H<PXRX4Y59M#96Sy#};8qDWhUmvhs?!KhTe+T<*|pUA4y-cLpF7$E*>PV7H(II|^o?5K1)5Q9^L3'
    '#9={Mikv0McZkJjpNxYTIm#`O0Y4g$O^8%D!91`85Jev&l=`4Tu);eUXXHxG$6V;2FA+joTNY4~j%kc4jEYRg;8fUz#hQTdrdHx0PrKHmu|)PMq78n<durJP'
    '%|fq$O%y313ne~F#u2<@GO9~J>5A7}Iv>}~`?AzL3kxQ6*EB2&qH8TX#9ta#tGP5A&jxBem2ESDx42CsCz<%t@|sp6f}BWN+V3YlZ3^gGL<OKRBY?(J4q;Q>'
    'k84Vt8rp0D(*V(ldlS0!Me1K-H&fWuu<YIur|de^KhZ*&3Q^X(C%o~9FA#769giC4fT`DSK2rU|<dLcme8-gK03@M*NJ_yen>v!FM*2kpC%)9O^>h?wd1W@M'
    'Hl-Wj8fOHzY=A3dEfLj^wK#NsV=a*izSck?UA7BF@ohz|46aixfEN0k0x_Lz^+qf;0tvbDJf|QW#c1-aFd5L$+kGFi*tB3GG1o<F9&=6wX*OSV^r-}V^HITT'
    'lA5C3inFo9MBuCHlDu{!Ob4(T_=5%BakZ2Hb1QR~x2}3i3s(l@!m48acca{Rir%Kp#U`wXPr!iu*n81OqxtG!AkS$qd?~prJd#acWInKER|XM?gm2JkJ>ih5'
    ')Zs|1J0NkGIGO0kqsH_YF7puPh@~JSPvM5`Ax&q8dR&y|Nt^q<6s*#<`~-@UZUPX!#i(a>r>8tC2s6!MH=sEU6|x)BX?sdfs1syhK4VX$pivBQ7H4lF%oAu9'
    'C0Z@kRoSZzUvnJTQXK?eHp}97IwKGk>%8N5nnlO<d7)HytJU1^Q?h_}Nc$iYG|10!!iUJ{NgZc_bh=56$k!YzDz00)Z7BQiC<(P_z1;LmBiX*TzU6J*e17}I'
    ')y*{ze&5=>`SJCuYgfI?FL~S7*1Rh>Z@sj+{=&6w@7m3ct81I;NQ=`?+=Lg(#tmx(zM<uM?Gv{)(Wm3)rni3m*2X%(%Z=SNZ~exVjoVk(Z@f_R00z40cpK~2'
    '*SDeA_RSi$yBihjp;_E=Afe6}4kQlG5f6s88(m%By0Wpletqp~9omOBy|s_8-PrcFuB~otIEN~X^7%k7uR%9gFK+-eY6}K>b$xT~$~KND|8)iM2R+=Vd0V&E'
    'uB_vawNI?Ua8@^8stKT5YybT=6oQY?v(@XXFTn6Br=SI<=F06&@}xmjY~8-RwY|Q5dwb1$;pWY&i0s{HYi;x6!0NXC+}pUhMHr(5B3D<pSE*@$0qBCy@cZ)Z'
    't#v}&`i<?i&CT1l@bIAugMSe)1F){FLfxx`;+r>Uw1B0Xn=c_$#Dj3*dGXpByx&Bm61G+m3z(hc%C=T)H-Z)ci`&{Ty&G#UY^=YqcH_z#_U9&od2xMftqL=?'
    'zJ&$Wx!o66q0!qkYMdzO7J$Rw8u8X>KE3tlz16E9U&oG%(m(=R>w;VeZC9>&jB-5HxOeaW2N0;=NB'
)
_vendor_raw = _vendor_zlib.decompress(_vendor_base64.b85decode(_VENDOR_SOURCES_B85))
if _vendor_hashlib.sha256(_vendor_raw).hexdigest() != _VENDOR_SOURCES_SHA256:
    raise ValueError('Embedded upstream source checksum mismatch')
VENDOR_SOURCES = _vendor_json.loads(_vendor_raw)
del _VENDOR_SOURCES_B85, _vendor_raw
WORKER_SOURCE = '"""Generated verbatim by the notebook. No project imports and no network."""\nfrom __future__ import annotations\nimport os, sys, json, time, traceback, pathlib, socket, hashlib\n\nfor _key in (\'HF_HUB_OFFLINE\',\'TRANSFORMERS_OFFLINE\',\'HF_DATASETS_OFFLINE\',\'DIFFUSERS_OFFLINE\'):\n    os.environ[_key] = \'1\'\nos.environ[\'HF_HUB_DISABLE_TELEMETRY\'] = \'1\'\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\n\ndef deny_network(event, args):\n    if event==\'socket.connect\' and isinstance(args[1],str):\n        return # Local Unix-domain IPC used by compiler workers; no IP network.\n    if event in (\'socket.connect\', \'socket.getaddrinfo\'):\n        raise RuntimeError(\'OFFLINE: network access prohibited in image worker\')\nsys.addaudithook(deny_network)\n\ndef atomic_json(path, obj):\n    path = pathlib.Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(\'.tmp\'); tmp.write_text(json.dumps(obj, ensure_ascii=False), encoding=\'utf8\'); tmp.replace(path)\n\ndef rrdb_model():\n    # Inference architecture from XPixelGroup/BasicSR rrdbnet_arch.py (Apache-2.0).\n    # Identical released x4plus layer names/shapes/forward; training init omitted.\n    import torch\n    from torch import nn\n    from torch.nn import functional as F\n    class ResidualDenseBlock(nn.Module):\n        def __init__(self):\n            super().__init__()\n            for i in range(1,6):\n                setattr(self, \'conv\'+str(i), nn.Conv2d(64+(i-1)*32, 32 if i<5 else 64, 3,1,1))\n            self.lrelu = nn.LeakyReLU(.2, inplace=True)\n        def forward(self,x):\n            values=[x]\n            for i in range(1,5): values.append(self.lrelu(getattr(self,\'conv\'+str(i))(torch.cat(values,1))))\n            return self.conv5(torch.cat(values,1))*.2+x\n    class RRDB(nn.Module):\n        def __init__(self):\n            super().__init__(); self.rdb1=ResidualDenseBlock(); self.rdb2=ResidualDenseBlock(); self.rdb3=ResidualDenseBlock()\n        def forward(self,x): return self.rdb3(self.rdb2(self.rdb1(x)))*.2+x\n    class RRDBNet(nn.Module):\n        def __init__(self):\n            super().__init__(); self.conv_first=nn.Conv2d(3,64,3,1,1)\n            self.body=nn.Sequential(*[RRDB() for _ in range(23)])\n            self.conv_body=nn.Conv2d(64,64,3,1,1); self.conv_up1=nn.Conv2d(64,64,3,1,1)\n            self.conv_up2=nn.Conv2d(64,64,3,1,1); self.conv_hr=nn.Conv2d(64,64,3,1,1); self.conv_last=nn.Conv2d(64,3,3,1,1)\n            self.lrelu=nn.LeakyReLU(.2,inplace=True)\n        def forward(self,x):\n            x=self.conv_first(x); x=x+self.conv_body(self.body(x))\n            x=self.lrelu(self.conv_up1(F.interpolate(x,scale_factor=2,mode=\'nearest\')))\n            x=self.lrelu(self.conv_up2(F.interpolate(x,scale_factor=2,mode=\'nearest\')))\n            return self.conv_last(self.lrelu(self.conv_hr(x)))\n    return RRDBNet()\n\ndef starts(length,tile,overlap):\n    if not 0 <= overlap < tile: raise ValueError(\'Require 0 <= overlap < tile\')\n    if length <= tile: return [0]\n    values=list(range(0,length-tile+1,tile-overlap))\n    if values[-1]!=length-tile: values.append(length-tile)\n    return values\n\ndef tiled_tadisr(image, predict, tile, overlap):\n    """CPU accumulation, independent axes; no uncovered pixels on thin crops."""\n    import torch\n    from torch.nn import functional as F\n    if tile % 32 or overlap % 32: raise ValueError(\'TADiSR tile/overlap must be multiples of 32\')\n    h,w=image.shape[-2:]\n    image=F.pad(image,(0,max(0,tile-w),0,max(0,tile-h)),mode=\'replicate\')\n    ph,pw=image.shape[-2:]\n    sums=torch.zeros(1,3,ph,pw); masks=torch.zeros(1,1,ph,pw); weights=torch.zeros(1,1,ph,pw)\n    blend=torch.hann_window(tile,periodic=False).clamp_min(1e-3)\n    blend=(blend[:,None]*blend[None,:])[None,None]\n    ys,xs=starts(ph,tile,overlap),starts(pw,tile,overlap)\n    seam_diffs=[]\n    # Compare independently predicted patches in their actual overlap before blending.\n    for top in ys:\n        for left in xs:\n            sr,mask=predict(image[:,:,top:top+tile,left:left+tile])\n            sr,mask=sr.detach().float().cpu(),mask.detach().float().cpu()\n            if tuple(sr.shape)!=(1,3,tile,tile) or tuple(mask.shape)!=(1,1,tile,tile):\n                raise ValueError(\'TADiSR output violates tile contract\')\n            old_weights=weights[:,:,top:top+tile,left:left+tile]\n            old=sums[:,:,top:top+tile,left:left+tile]\n            occupied=(old_weights>0).expand_as(old)\n            if occupied.any():\n                previous=old/old_weights.clamp_min(1e-12)\n                seam_diffs.append(float((previous-sr).abs()[occupied].mean()))\n            old += sr*blend\n            masks[:,:,top:top+tile,left:left+tile] += mask*blend\n            old_weights += blend\n    if not (weights>0).all(): raise ValueError(\'Tiled output has uncovered pixels\')\n    return (sums/weights)[:,:,:h,:w],(masks/weights)[:,:,:h,:w],{\n        \'tiles\':len(ys)*len(xs),\'overlap_patch_mae\':max(seam_diffs,default=0.),\n        \'covered_all_pixels\':True,\'visual_seam_review\':\'required\'}\n\ndef load_backend(model, assets, options):\n    if model in DOCUMENT_VARIANTS: return document_load(model, assets, options)\n    import torch\n    device=options.get(\'device\',\'cuda\')\n    if model==\'realesrgan_x4\':\n        net=rrdb_model()\n        sd=torch.load(assets[\'checkpoint\'],map_location=\'cpu\',weights_only=True)\n        net.load_state_dict(sd.get(\'params_ema\',sd.get(\'params\',sd)),strict=True)\n        net=net.eval().to(device)\n        if options.get(\'half\',False): net.half()\n        return net\n    if model==\'tadisr_x4\':\n        from tadisr.inference import CogView4Config,load_cogview4\n        if options.get(\'placement\')==\'sharded\': return load_tadisr_multi_gpu(assets,options)\n        # Upstream inspect_checkpoint runs before GPU construction and validates decoder.\n        return load_cogview4(CogView4Config(pathlib.Path(assets[\'base_model\']),\n            pathlib.Path(assets[\'checkpoint\']),pathlib.Path(assets[\'prompt_embeddings\']),device))\n    if model==\'difftsr\':\n        from omegaconf import OmegaConf\n        from model.IDM.utils.util import instantiate_from_config\n        cfg=OmegaConf.load(assets[\'config\'])\n        cfg.model.params.Text_Prediction_config.params={\'ckpt_path\':assets[\'transocr\'],\n            \'alphabet_path\':str(pathlib.Path(assets[\'vendor\'])/\'model/TDM/utils/benchmark.txt\')}\n        net=instantiate_from_config(cfg.model); net.load_model(assets[\'checkpoint\'])\n        net.IDM_ddim_schedule_init(ddim_num_steps=options[\'steps\'],ddim_eta=.2,verbose=False)\n        return net\n    if model==\'unihir_restoration\':\n        from utils import load_model,UniHIRInferencer\n        m,t,v,vt,vg,it,nt=(load_unihir_multi_gpu(assets,options) if options.get(\'placement\')==\'sharded\' else\n            load_model(assets[\'base_model\'],max_mem_per_gpu=options.get(\'max_memory\',\'80GiB\')))\n        sizes={}\n        def capture(name,transform):\n            class ObservedTransform:\n                def __getattr__(self,key): return getattr(transform,key)\n                def __call__(self,image,*a,**kw):\n                    result=transform(image,*a,**kw)\n                    sizes[name]={\'input_size\':list(image.size),\'output_shape\':list(result.shape) if hasattr(result,\'shape\') else None}\n                    return result\n            return ObservedTransform()\n        net=UniHIRInferencer(model=m,tokenizer=t,vae_model=v,vae_transform=capture(\'vae\',vt),\n            vae_transform_gen=capture(\'vae_gen\',vg),vit_transform=capture(\'vit\',it),new_token_ids=nt)\n        net._observed_processor_sizes=sizes\n        net._vtm_device_map=getattr(m,\'_vtm_device_map\',getattr(m,\'hf_device_map\',{}))\n        return net\n    raise ValueError(model)\n\ndef run_backend(model,net,image,options):\n    if model in DOCUMENT_VARIANTS: return document_run(model,net,image,options)\n    import torch, numpy as np\n    from PIL import Image\n    import torch.nn.functional as F\n    device=options.get(\'device\',\'cuda\');diagnostics={};extras={}\n    if model==\'realesrgan_x4\':\n        # Same local tile padding/cropping as RealESRGANer; RGB tensors throughout.\n        tile=options.get(\'tile\',256);pad=options.get(\'overlap\',24)\n        h,w=image.height,image.width;output=np.empty((h*4,w*4,3),dtype=np.uint8);processor_sizes=set()\n        dtype=next(net.parameters()).dtype\n        for y in range(0,h,tile):\n            for x in range(0,w,tile):\n                x0,y0=max(0,x-pad),max(0,y-pad);x1,y1=min(w,x+tile+pad),min(h,y+tile+pad)\n                a=np.asarray(image.crop((x0,y0,x1,y1)),dtype=np.float32)/255\n                tensor=torch.from_numpy(a.copy()).permute(2,0,1)[None].to(device=device,dtype=dtype)\n                processor_sizes.add((x1-x0,y1-y0))\n                result=net(tensor).clamp(0,1)[0].float().cpu().permute(1,2,0).numpy()\n                actual_w,actual_h=min(tile,w-x),min(tile,h-y)\n                output[y*4:(y+actual_h)*4,x*4:(x+actual_w)*4]=(result[\n                    (y-y0)*4:(y-y0+actual_h)*4,(x-x0)*4:(x-x0+actual_w)*4]*255).round().astype(np.uint8)\n        return Image.fromarray(output),extras,{\'visual_seam_review\':\'required\',\'tile_padding\':pad,\n            \'processor_input_sizes\':sorted(processor_sizes),\'processor_output_scale\':4}\n    if model==\'tadisr_x4\':\n        a=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()/255).permute(2,0,1)[None]\n        a=F.interpolate(a,size=(image.height*4,image.width*4),mode=\'bicubic\',align_corners=False)\n        def predict(patch): return net(patch.to(device))\n        sr,mask,diagnostics=tiled_tadisr(a,predict,options[\'tile\'],options[\'overlap\'])\n        diagnostics.update(processor_tile_size=[options[\'tile\'],options[\'tile\']],pre_upsample_size=[image.width*4,image.height*4])\n        out=(sr[0].clamp(0,1).permute(1,2,0).numpy()*255).round().astype(np.uint8)\n        extras[\'tadisr_text_mask\']=Image.fromarray((mask.sigmoid()[0,0].numpy()*255).round().astype(np.uint8))\n        return Image.fromarray(out),extras,diagnostics\n    if model==\'difftsr\':\n        # Only reviewed, near-4:1 horizontal lines; isotropic letterbox, never rotate.\n        canvas=Image.new(\'RGB\',(512,128),\'white\');copy=image.copy();copy.thumbnail((512,128),Image.Resampling.BICUBIC)\n        offset=((512-copy.width)//2,(128-copy.height)//2);canvas.paste(copy,offset)\n        result=Image.fromarray(net.DiffTSR_sample(canvas),\'RGB\')\n        # Released output is 512x128, not an arbitrary x4 contract.\n        result=result.crop((offset[0],offset[1],offset[0]+copy.width,offset[1]+copy.height))\n        return result,extras,{\'processor_size\':[512,128],\'content_size\':list(copy.size),\n            \'processor_offset\':list(offset),\'steps\':options[\'steps\'],\n            \'warning\':\'upstream_text_prior_uses_zhconv_zh_cn; inspect_traditional_forms\'}\n    if model==\'unihir_restoration\':\n        from utils import invert_image\n        result=net(image=invert_image(image),text=\'请对图中破损位置进行定位，并输出破损定位图。\',\n            und_only=False,gen_only=False,reflection_count=options[\'ref_count\'],think=False,\n            max_think_token_n=10000,do_sample=False,cfg_text_scale=1.,cfg_img_scale=1.,\n            cfg_interval=[0.,1.],timestep_shift=3.,num_timesteps=options[\'steps\'],\n            cfg_renorm_min=0.,cfg_renorm_type=\'global\')\n        final=invert_image(result[\'final_image\'].resize(image.size,Image.Resampling.LANCZOS))\n        extras[\'predictions\']={k:result.get(k) for k in (\'text_step1\',\'text_step2\',\'text_merge\')}\n        extras[\'predictions\'].update(refinement_count=options[\'ref_count\'],confidence=None,\n            confidence_status=\'upstream_does_not_supply_calibrated_confidence\')\n        import re\n        extras[\'predictions\'][\'damage_predictions\']=[\n            {\'box_normalized_1000\':[int(x) for x in match[:4]],\'predicted_character\':match[4],\n             \'evidence\':\'model_generated; not_source_observation\'}\n            for match in re.findall(r\'<\\|box_start\\|>(\\d+),(\\d+),(\\d+),(\\d+),(.?)<\\|box_end\\|>\',result.get(\'text_merge\',\'\'))]\n        # Keep damage/draft and all available intermediate diagnostic images separately.\n        for key in (\'draft\',):\n            if isinstance(result.get(key),Image.Image): extras[key]=result[key]\n        for key in (\'tmp_img\',\'tmp_img_gen\'):\n            for i,v in enumerate(result.get(key,[])):\n                if isinstance(v,Image.Image):extras[f\'{key}_{i:02d}\']=v\n        return final,extras,{\'generative_reconstruction\':True,\'alignment\':\'nominal_resize_only; generated_glyphs_unverified\',\n            \'observed_processor_sizes\':net._observed_processor_sizes}\n    raise ValueError(model)\n\ndef verify_active_records(module_name, distribution_name, *, kernel_torch_source_policy=\'strict\',\n                          kernel_torch_source_path=None):\n    """Audit active imports; only inherited Torch Python source may report drift."""\n    import base64, csv, importlib, importlib.metadata as md, io\n    from packaging.version import Version\n    from packaging.tags import sys_tags, parse_tag\n    if kernel_torch_source_policy not in (\'strict\', \'warn\'):\n        raise ValueError(\'kernel_torch_source_policy must be strict or warn\')\n    module=importlib.import_module(module_name);dist=md.distribution(distribution_name)\n    actual=getattr(module,\'__version__\',None)\n    if actual and Version(actual).public!=Version(dist.version).public:\n        raise RuntimeError(f\'Active module/metadata mismatch: {module_name} {actual} vs {dist.version}\')\n    wheel_tags=set()\n    for line in (dist.read_text(\'WHEEL\') or \'\').splitlines():\n        if line.startswith(\'Tag: \'):wheel_tags.update(parse_tag(line[5:]))\n    if wheel_tags and not wheel_tags.intersection(set(sys_tags())):\n        raise RuntimeError(f\'Active distribution has incompatible wheel ABI: {distribution_name}\')\n    record=dist.read_text(\'RECORD\')\n    if not record:raise RuntimeError(f\'Cannot verify active distribution RECORD: {distribution_name}\')\n    rows=list(csv.reader(io.StringIO(record)))\n    if any(len(row)!=3 for row in rows):\n        raise RuntimeError(f\'Malformed active distribution RECORD: {distribution_name}\')\n    module_file=pathlib.Path(module.__file__).resolve()\n    verify_paths=[module_file]\n    if module_name==\'torch\':verify_paths.append(pathlib.Path(module._C.__file__).resolve())\n    if module_name==\'numpy\':\n        try:native=importlib.import_module(\'numpy._core._multiarray_umath\')\n        except ImportError:native=importlib.import_module(\'numpy.core._multiarray_umath\')\n        verify_paths.append(pathlib.Path(native.__file__).resolve())\n    verified=[];notices=[]\n    for path in verify_paths:\n        matching=[r for r in rows if pathlib.Path(dist.locate_file(r[0])).resolve()==path]\n        if len(matching)!=1 or not matching[0][1]:\n            raise RuntimeError(f\'Active file not owned by metadata RECORD: {path}\')\n        algorithm,expected=matching[0][1].split(\'=\',1)\n        hasher=hashlib.new(algorithm)\n        with path.open(\'rb\') as stream:\n            for block in iter(lambda:stream.read(1024*1024),b\'\'):hasher.update(block)\n        observed=base64.urlsafe_b64encode(hasher.digest()).decode().rstrip(\'=\')\n        if observed!=expected:\n            # Use the actual import origin captured by the notebook kernel, not\n            # guessed site directories: Debian dist-packages can differ from sysconfig.\n            inherited_torch_source=(module_name==\'torch\' and path==module_file\n                and path==pathlib.Path(dist.locate_file(\'torch/__init__.py\')).resolve()\n                and kernel_torch_source_path is not None\n                and path==pathlib.Path(kernel_torch_source_path).resolve())\n            if kernel_torch_source_policy!=\'warn\' or not inherited_torch_source:\n                raise RuntimeError(f\'Active package RECORD checksum differs: {path}; \'\n                    f\'expected {algorithm}={expected}, observed {algorithm}={observed}; \'\n                    f\'policy={kernel_torch_source_policy}, kernel Torch source={kernel_torch_source_path}\')\n            notices.append({\'distribution\':distribution_name,\'version\':dist.version,\'path\':str(path),\n                \'status\':\'kernel_source_record_mismatch\',\'expected_hash\':f\'{algorithm}={expected}\',\n                \'observed_hash\':f\'{algorithm}={observed}\',\n                \'note\':\'Source drift recorded; compatibility is checked separately; RECORD integrity unverified\'})\n        else:verified.append(str(path))\n    return {\'verified\':verified,\'warnings\':notices}\n\n\ndef probe(model, options):\n    import importlib,importlib.metadata as md,platform\n    imports = {\'realesrgan_x4\': [\'torch\', \'numpy\', \'PIL\'], \'tadisr_x4\': [\'torch\', \'numpy\', \'diffusers\', \'peft\', \'transformers\', \'sentencepiece\', \'tadisr.inference\', \'tadisr.pipelines\'], \'difftsr\': [\'torch\', \'torchvision\', \'omegaconf\', \'pytorch_lightning\', \'zhconv\', \'einops\', \'model.DiffTSR\', \'model.IDM.models.autoencoder\', \'model.TDM.models.text_prediction\', \'model.TDM.models.modules\'], \'unihir_restoration\': [\'torch\', \'flash_attn\', \'transformers\', \'accelerate\', \'fontTools\', \'utils\'], \'docres_appearance\': [\'torch\', \'numpy\', \'PIL\', \'cv2\', \'einops\', \'skimage\'], \'docres_deblurring\': [\'torch\', \'numpy\', \'PIL\', \'cv2\', \'einops\', \'skimage\'], \'docres_deshadowing\': [\'torch\', \'numpy\', \'PIL\', \'cv2\', \'einops\', \'skimage\'], \'docres_binarization\': [\'torch\', \'numpy\', \'PIL\', \'cv2\', \'einops\', \'skimage\'], \'docdiff_deblurring\': [\'torch\', \'torchvision\', \'numpy\', \'PIL\', \'yaml\', \'model.DocDiff\', \'schedule.diffusionSample\'], \'restormer_motion_deblurring\': [\'torch\', \'numpy\', \'PIL\', \'einops\'], \'restormer_defocus_deblurring\': [\'torch\', \'numpy\', \'PIL\', \'einops\'], \'restormer_real_denoising\': [\'torch\', \'numpy\', \'PIL\', \'einops\'], \'restormer_gaussian_denoising\': [\'torch\', \'numpy\', \'PIL\', \'einops\'], \'docentr_binarization\': [\'torch\', \'numpy\', \'PIL\', \'cv2\', \'einops\', \'vit_pytorch\'], \'degan_binarization\': [\'torch\', \'numpy\', \'PIL\', \'h5py\', \'keras\'], \'degan_deblurring\': [\'torch\', \'numpy\', \'PIL\', \'h5py\', \'keras\'], \'degan_watermark_removal\': [\'torch\', \'numpy\', \'PIL\', \'h5py\', \'keras\']}[model]\n    for name in imports: importlib.import_module(name)\n    import torch,numpy as np\n    if torch.from_numpy(np.ones((2,2),dtype=np.float32)).numpy().shape!=(2,2):raise RuntimeError(\'NumPy bridge failed\')\n    records_verified=[];record_warnings=[]\n    source_policy=options.get(\'kernel_torch_source_policy\',\'strict\')\n    for module_name,distribution_name in [(\'torch\',\'torch\'),(\'numpy\',\'numpy\'),(\'PIL\',\'pillow\'),\n            *([(n,n) for n in (\'diffusers\',\'peft\',\'transformers\')] if model==\'tadisr_x4\' else [])]:\n        audit=verify_active_records(module_name,distribution_name,kernel_torch_source_policy=source_policy,\n            kernel_torch_source_path=options.get(\'kernel_torch_source_path\'))\n        records_verified.extend(audit[\'verified\']);record_warnings.extend(audit[\'warnings\'])\n    report={\'python\':sys.version,\'abi\':sys.implementation.cache_tag,\'platform\':platform.platform(),\n        \'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'numpy\':np.__version__,\n        \'cuda_available\':torch.cuda.is_available(),\'imports\':imports,\'pid\':os.getpid(),\n        \'active_import_records_verified\':records_verified,\'active_import_record_warnings\':record_warnings,\n        \'kernel_torch_source_policy\':source_policy,\'kernel_torch_source_path\':options.get(\'kernel_torch_source_path\'),\n        \'python_executable\':sys.executable,\'prefix\':sys.prefix,\'base_prefix\':sys.base_prefix}\n    if options.get(\'device\',\'cuda\').startswith(\'cuda\'):\n        report.update(restoration_gpu_probe(torch,model,options))\n        report.update(gpu=torch.cuda.get_device_name(0),capability=list(torch.cuda.get_device_capability(0)),\n            compiled_arches=torch.cuda.get_arch_list(),total_vram=torch.cuda.get_device_properties(0).total_memory)\n    if model==\'tadisr_x4\':\n        from diffusers import CogView4Transformer2DModel\n        from transformers import GlmModel\n    if model==\'unihir_restoration\':\n        from flash_attn import flash_attn_func\n        sample=torch.randn(1,16,2,64,device=\'cuda\',dtype=torch.bfloat16)\n        flash_attn_func(sample,sample,sample);torch.cuda.synchronize()\n    # Check only the dependency closure actually imported, not unrelated host notebook packages.\n    from packaging.requirements import Requirement\n    pending=[{\'PIL\':\'pillow\',\'fontTools\':\'fonttools\',\'skimage\':\'scikit-image\',\'cv2\':\'opencv-python-headless\',\'yaml\':\'pyyaml\'}.get(n,n) for n in imports if \'.\' not in n]\n    checked=set()\n    while pending:\n        name=pending.pop()\n        if name in checked:continue\n        checked.add(name)\n        try:dist=md.distribution(name)\n        except md.PackageNotFoundError:continue\n        for spec in dist.requires or []:\n            req=Requirement(spec)\n            if req.marker and not req.marker.evaluate({\'extra\':\'\'}):continue\n            version=md.version(req.name)\n            if req.specifier and not req.specifier.contains(version,prereleases=True):\n                raise RuntimeError(f\'Dependency conflict: {dist.metadata["Name"]} requires {req}, got {version}\')\n            pending.append(req.name)\n    report[\'dependency_closure_checked\']=sorted(checked)\n    return report\n\ndef main():\n    cfg=json.loads(pathlib.Path(sys.argv[1]).read_text(encoding=\'utf8\'))\n    if cfg.get(\'vendor\'):sys.path.insert(0,cfg[\'vendor\'])\n    result_path=cfg[\'ready\']\n    try:\n        if cfg.get(\'action\')==\'isolation_probe\':\n            try:socket.create_connection((\'127.0.0.1\',9),timeout=.1)\n            except RuntimeError as error:\n                blocked=\'OFFLINE\' in str(error)\n            else:blocked=False\n            if not blocked:raise RuntimeError(\'Worker offline guard did not reject a connection\')\n            atomic_json(result_path,{\'state\':\'ready\',\'pid\':os.getpid(),\'offline\':True,\'network_block_verified\':blocked});return\n        report=probe(cfg[\'model\'],cfg[\'options\'])\n        if cfg.get(\'action\')==\'probe\':atomic_json(result_path,{\'state\':\'ready\',\'report\':report});return\n        if cfg.get(\'action\')==\'prepare_prompt\':\n            import runpy,torch\n            target=pathlib.Path(cfg[\'assets\'][\'prompt_output\']);target.parent.mkdir(parents=True,exist_ok=True)\n            temporary=target.with_suffix(\'.part\')\n            sys.argv=[\'prepare_prompt_embeddings.py\',\'--base-model\',cfg[\'assets\'][\'base_model\'],\n                \'--output\',str(temporary),\'--device\',cfg[\'options\'].get(\'prompt_device\',cfg[\'options\'].get(\'device\',\'cuda\'))]\n            torch.manual_seed(cfg[\'options\'].get(\'seed\',42));torch.cuda.reset_peak_memory_stats()\n            started=time.monotonic()\n            runpy.run_path(str(pathlib.Path(cfg[\'vendor\'])/\'scripts/prepare_prompt_embeddings.py\'),run_name=\'__main__\')\n            temporary.replace(target);torch.cuda.synchronize()\n            atomic_json(result_path,{\'state\':\'ready\',\'report\':report,\'prompt_seconds\':time.monotonic()-started,\n                \'peak_vram_bytes\':torch.cuda.max_memory_allocated(),\'derived_offline\':True});return\n        import torch, numpy as np\n        torch.manual_seed(cfg[\'options\'].get(\'seed\',42));np.random.seed(cfg[\'options\'].get(\'seed\',42))\n        started=time.monotonic();net=load_backend(cfg[\'model\'],cfg[\'assets\'],cfg[\'options\'])\n        restoration_cuda_metrics(torch)\n        atomic_json(result_path,{\'state\':\'ready\',\'report\':report,\'load_seconds\':time.monotonic()-started,\n            \'device_map\':getattr(net,\'_vtm_device_map\',{}), \'devices\':restoration_cuda_metrics(torch)})\n    except Exception:\n        atomic_json(result_path,{\'state\':\'failed\',\'error\':traceback.format_exc(),\n            \'runtime_context\':{\'python_executable\':sys.executable,\'prefix\':sys.prefix,\'base_prefix\':sys.base_prefix,\n                \'kernel_torch_source_policy\':cfg.get(\'options\',{}).get(\'kernel_torch_source_policy\'),\n                \'kernel_torch_source_path\':cfg.get(\'options\',{}).get(\'kernel_torch_source_path\')}});return\n    for line in sys.stdin:\n        job=json.loads(line);output=pathlib.Path(job[\'output\']);output.mkdir(parents=True,exist_ok=True)\n        started=time.monotonic()\n        try:\n            from PIL import Image\n            restoration_cuda_metrics(torch,reset=True)\n            with Image.open(job[\'input\']) as src:image=src.convert(\'RGB\')\n            # Per-region seed makes interrupted/resumed diffusion independent of job ordering.\n            torch.manual_seed(job[\'seed\']);np.random.seed(job[\'seed\']%(2**32-1))\n            with torch.inference_mode():restored,extras,diagnostics=run_backend(cfg[\'model\'],net,image,cfg[\'options\'])\n            restored.save(output/\'result.png\')\n            extra_refs={}\n            for key,value in extras.items():\n                if isinstance(value,Image.Image):value.save(output/(key+\'.png\'));extra_refs[key]=key+\'.png\'\n                else:atomic_json(output/(key+\'.json\'),value);extra_refs[key]=key+\'.json\'\n            restoration_cuda_metrics(torch)\n            peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0\n            reserved=torch.cuda.max_memory_reserved() if torch.cuda.is_available() else 0\n            atomic_json(job[\'response\'],{\'state\':\'ok\',\'seconds\':time.monotonic()-started,\n                \'peak_vram_bytes\':peak,\'peak_reserved_vram_bytes\':reserved,\n                \'per_gpu_memory\':restoration_cuda_metrics(torch),\'device_map\':getattr(net,\'_vtm_device_map\',{}),\n                \'per_gpu_memory\':restoration_cuda_metrics(torch),\'device_map\':getattr(net,\'_vtm_device_map\',{}),\'diagnostics\':diagnostics,\'extras\':extra_refs})\n        except Exception:\n            atomic_json(job[\'response\'],{\'state\':\'failed\',\'seconds\':time.monotonic()-started,\'error\':traceback.format_exc()})\n\n# BEGIN DOCUMENT RESTORATION ADAPTERS\nimport ast\nimport hashlib\nimport importlib.util\nimport os\nfrom pathlib import Path\nimport tempfile\nfrom types import SimpleNamespace\nimport zipfile\nLEARNED_DOCUMENT_TASKS = {\'docentr_binarization\': (\'docentr\', None, 0.5), \'degan_binarization\': (\'degan\', \'weights/binarization_generator_weights.h5\', 0.95), \'degan_deblurring\': (\'degan\', \'weights/deblur_weights.h5\', None), \'degan_watermark_removal\': (\'degan\', \'weights/watermark_rem_weights.h5\', None)}\n\ndef learned_document_options(name, options):\n    if name not in LEARNED_DOCUMENT_TASKS:\n        raise ValueError(name)\n    if options.get(\'patch_size\', 256) != 256:\n        raise ValueError(\'Released learned document checkpoints require 256x256 native patches; no resize\')\n    if options.get(\'device\', \'cuda\') not in (\'cpu\', \'cuda\', \'cuda:0\'):\n        raise ValueError(\'Select one CPU/CUDA device; learned document models run sequentially\')\n\ndef learned_document_load(name, assets, options):\n    import torch\n    learned_document_options(name, options)\n    family, member, _ = LEARNED_DOCUMENT_TASKS[name]\n    vendor = Path(assets[\'vendor\'])\n    device = options.get(\'device\', \'cuda\')\n    if family == \'docentr\':\n        from vit_pytorch import ViT\n        spec = importlib.util.spec_from_file_location(\'docentr_pinned_binae\', vendor / \'models/binae.py\')\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        encoder = ViT(image_size=256, patch_size=8, num_classes=1000, dim=768, depth=6, heads=8, mlp_dim=2048)\n        network = module.BinModel(encoder=encoder, decoder_dim=768, decoder_depth=6, decoder_heads=8)\n        network.load_state_dict(torch.load(assets[\'checkpoint\'], map_location=\'cpu\', weights_only=True), strict=True)\n        return SimpleNamespace(network=network.eval().to(device), family=family)\n    os.environ[\'KERAS_BACKEND\'] = \'torch\'\n    import keras\n    if keras.backend.backend() != \'torch\':\n        raise RuntimeError(\'DE-GAN requires a fresh worker with KERAS_BACKEND=torch; kernel backend is not changed\')\n    keras.backend.clear_session()\n    tree = ast.parse((vendor / \'models/models.py\').read_text(encoding=\'utf8\'))\n    definitions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'generator_model\']\n    if len(definitions) != 1:\n        raise ValueError(\'Pinned DE-GAN generator definition missing\')\n    scope = {n: getattr(keras.layers, n) for n in (\'Input\', \'Conv2D\', \'MaxPooling2D\', \'Dropout\', \'UpSampling2D\', \'concatenate\')}\n    scope[\'Model\'] = keras.Model\n    exec(compile(ast.Module(body=definitions, type_ignores=[]), \'degan-upstream-generator\', \'exec\'), scope)\n    network = scope[\'generator_model\'](biggest_layer=512 if name == \'degan_watermark_removal\' else 1024)\n    record = assets.get(\'archive_members\', {}).get(member)\n    if not record or not record.get(\'sha256\'):\n        raise ValueError(\'DE-GAN archive member integrity manifest is missing; run preparation script\')\n    with tempfile.TemporaryDirectory(prefix=\'vtm_degan_\') as temporary:\n        target = Path(temporary) / Path(member).name\n        digest = hashlib.sha256()\n        written = 0\n        with zipfile.ZipFile(assets[\'checkpoint\']) as archive:\n            if archive.getinfo(member).file_size != record[\'bytes\']:\n                raise ValueError(\'DE-GAN archive member size differs from manifest\')\n            with archive.open(member) as source, target.open(\'wb\') as output:\n                for block in iter(lambda: source.read(4 * 1024 ** 2), b\'\'):\n                    digest.update(block)\n                    written += len(block)\n                    output.write(block)\n        if written != record[\'bytes\'] or digest.hexdigest() != record[\'sha256\']:\n            raise ValueError(\'DE-GAN archive member checksum mismatch\')\n        network.load_weights(target)\n    return SimpleNamespace(network=network.to(device).eval(), family=family, keras_version=keras.__version__, backend=\'torch\')\n\ndef learned_document_run(name, backend, image, options):\n    import numpy as np\n    import torch\n    from PIL import Image\n    learned_document_options(name, options)\n    family, _, threshold = LEARNED_DOCUMENT_TASKS[name]\n    device = options.get(\'device\', \'cuda\')\n    if family == \'docentr\':\n        array = np.asarray(image.convert(\'RGB\'))[:, :, ::-1].copy().astype(np.float32) / 255\n        channels = 3\n    else:\n        array = np.asarray(image.convert(\'L\'), dtype=np.float32)[:, :, None] / 255\n        channels = 1\n    width, height = image.size\n    pw, ph = (-width % 256, -height % 256)\n    padded = np.ones((height + ph, width + pw, channels), dtype=np.float32)\n    padded[:height, :width] = array\n    output = np.empty_like(padded)\n    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]\n    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]\n    count = 0\n    with torch.inference_mode():\n        for y in range(0, height + ph, 256):\n            for x in range(0, width + pw, 256):\n                patch = padded[y:y + 256, x:x + 256]\n                if family == \'docentr\':\n                    tensor = torch.from_numpy((patch.transpose(2, 0, 1) - mean) / std)[None].to(device)\n                    _, _, tokens = backend.network(tensor, torch.zeros_like(tensor))\n                    if tuple(tokens.shape) != (1, 1024, 192):\n                        raise ValueError(\'DocEnTR released base/8 checkpoint output contract changed\')\n                    value = tokens.reshape(1, 32, 32, 8, 8, 3).permute(0, 5, 1, 3, 2, 4).reshape(3, 256, 256)\n                    result = value.detach().cpu().numpy() * std + mean\n                    result = result.transpose(1, 2, 0)\n                else:\n                    tensor = torch.from_numpy(patch[None]).to(device)\n                    prediction = backend.network(tensor, training=False)\n                    if tuple(prediction.shape) != (1, 256, 256, 1):\n                        raise ValueError(\'DE-GAN released generator output contract changed\')\n                    result = prediction.detach().cpu().numpy()[0]\n                if not np.isfinite(result).all():\n                    raise ValueError(\'Learned document model produced non-finite pixels\')\n                output[y:y + 256, x:x + 256] = np.clip(result, 0, 1)\n                count += 1\n    output = output[:height, :width]\n    if threshold is not None:\n        output = (output > threshold).astype(np.float32)\n    pixels = (output * 255).round().astype(\'uint8\')\n    if channels == 1:\n        pixels = np.repeat(pixels, 3, axis=2)\n    else:\n        pixels = pixels[:, :, ::-1].copy()\n    return (Image.fromarray(pixels), {}, dict(task=name, patches=count, patch_size=256, padding=[0, pw, 0, ph], padding_value=1.0, geometry=\'native_patches; padding_removed; identity\', orientation=\'unchanged; no_rotation\', threshold=threshold, channel_convention=\'upstream_BGR_ImageNet_normalization\' if family == \'docentr\' else \'upstream_Pillow_luminance\', output_conversion=\'clip_01_to_uint8; no_matplotlib_display_autocontrast\', keras_backend=getattr(backend, \'backend\', None), keras_version=getattr(backend, \'keras_version\', None), warnings=[\'learned_prior; may_erase_or_add_strokes; red_ink_color_not_preserved\', \'nonoverlapping_native_256_patches_match_upstream; inspect_seams\']))\n\nimport ast\nimport importlib.util\nfrom pathlib import Path\nfrom types import SimpleNamespace\nDOCUMENT_VARIANTS = {\'docentr_binarization\': (\'docentr\', \'binarization_base_2018_patch8\', \'model_8_2018_base.pt\'), \'degan_binarization\': (\'degan\', \'binarization\', \'degan_weights.bundle\'), \'degan_deblurring\': (\'degan\', \'deblurring\', \'degan_weights.bundle\'), \'degan_watermark_removal\': (\'degan\', \'watermark_removal\', \'degan_weights.bundle\'), \'docres_appearance\': (\'docres\', \'appearance\', \'docres.pkl\'), \'docres_deblurring\': (\'docres\', \'deblurring\', \'docres.pkl\'), \'docres_deshadowing\': (\'docres\', \'deshadowing\', \'docres.pkl\'), \'docres_binarization\': (\'docres\', \'binarization\', \'docres.pkl\'), \'docdiff_deblurring\': (\'docdiff\', \'deblurring\', \'init.pth\'), \'restormer_motion_deblurring\': (\'restormer\', \'motion\', \'motion_deblurring.pth\'), \'restormer_defocus_deblurring\': (\'restormer\', \'defocus\', \'single_image_defocus_deblurring.pth\'), \'restormer_real_denoising\': (\'restormer\', \'real\', \'real_denoising.pth\'), \'restormer_gaussian_denoising\': (\'restormer\', \'gaussian_color_blind\', \'gaussian_color_denoising_blind.pth\')}\n\ndef document_module(path, name):\n    spec = importlib.util.spec_from_file_location(name, path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\ndef docres_prompt_module(vendor):\n    """Use verbatim upstream prompt definitions without importing unused MBD.\n\n    inference.py imports dewarp/MBD eagerly, although these four tasks do not\n    use it. Selecting only the named definitions avoids its unrelated dependency\n    graph. No function bodies are rewritten and source integrity is checked by\n    preflight before this loader runs.\n    """\n    import cv2\n    import numpy as np\n    utils = document_module(Path(vendor) / \'utils.py\', \'docres_upstream_utils\')\n    scope = dict(cv2=cv2, np=np, utils=utils)\n    names = {\'appearance_prompt\', \'deshadow_prompt\', \'deblur_prompt\', \'binarization_promptv2\'}\n    tree = ast.parse((Path(vendor) / \'inference.py\').read_text(encoding=\'utf8\'))\n    definitions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name in names]\n    if {n.name for n in definitions} != names:\n        raise ValueError(\'Pinned DocRes DTSPrompt definitions missing\')\n    exec(compile(ast.Module(body=definitions, type_ignores=[]), \'docres-upstream-prompts\', \'exec\'), scope)\n    return SimpleNamespace(**{n: scope[n] for n in names})\n\ndef document_load(model, assets, options):\n    if model.startswith((\'docentr_\', \'degan_\')):\n        return learned_document_load(model, assets, options)\n    import torch\n    family, task, _ = DOCUMENT_VARIANTS[model]\n    vendor = Path(assets[\'vendor\'])\n    device = options.get(\'device\', \'cuda\')\n    if family == \'docres\':\n        arch = document_module(vendor / \'models/restormer_arch.py\', \'docres_upstream_arch\')\n        net = arch.Restormer(inp_channels=6, out_channels=3, dim=48, num_blocks=[2, 3, 3, 4], num_refinement_blocks=4, heads=[1, 2, 4, 8], ffn_expansion_factor=2.66, bias=False, LayerNorm_type=\'WithBias\', dual_pixel_task=True)\n        state = torch.load(assets[\'checkpoint\'], map_location=\'cpu\', weights_only=True)[\'model_state\']\n        if state and all((k.startswith(\'module.\') for k in state)):\n            state = {k[7:]: v for k, v in state.items()}\n        net.load_state_dict(state, strict=True)\n        return SimpleNamespace(network=net.eval().to(device), prompts=docres_prompt_module(vendor), task=task)\n    if family == \'docdiff\':\n        import yaml\n        from model.DocDiff import DocDiff\n        from schedule.schedule import Schedule\n        from schedule.diffusionSample import GaussianDiffusion\n        cfg = yaml.safe_load((vendor / \'conf.yml\').read_text(encoding=\'utf8\'))\n        if (cfg[\'CHANNEL_X\'], cfg[\'CHANNEL_Y\'], cfg[\'TIMESTEPS\'], cfg[\'SCHEDULE\'], str(cfg[\'PRE_ORI\'])) != (3, 3, 100, \'linear\', \'True\'):\n            raise ValueError(\'DocDiff released deblurring checkpoint/config contract changed\')\n        if options.get(\'steps\', 100) != 100:\n            raise ValueError(\'Released DocDiff sampler uses 100 steps; changing T changes its schedule\')\n        net = DocDiff(input_channels=cfg[\'CHANNEL_X\'] + cfg[\'CHANNEL_Y\'], output_channels=cfg[\'CHANNEL_Y\'], n_channels=cfg[\'MODEL_CHANNELS\'], ch_mults=cfg[\'CHANNEL_MULT\'], n_blocks=cfg[\'NUM_RESBLOCKS\'])\n        net.init_predictor.load_state_dict(torch.load(assets[\'checkpoint\'], map_location=\'cpu\', weights_only=True), strict=True)\n        net.denoiser.load_state_dict(torch.load(assets[\'denoiser\'], map_location=\'cpu\', weights_only=True), strict=True)\n        net.eval().to(device)\n        sampler = GaussianDiffusion(net.denoiser, 100, Schedule(\'linear\', 100)).eval().to(device)\n        return SimpleNamespace(network=net, sampler=sampler, task=task)\n    arch = document_module(vendor / \'basicsr/models/archs/restormer_arch.py\', \'restormer_upstream_arch\')\n    params = dict(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8], ffn_expansion_factor=2.66, bias=False, LayerNorm_type=\'BiasFree\' if task in (\'real\', \'gaussian_color_blind\') else \'WithBias\', dual_pixel_task=False)\n    net = arch.Restormer(**params)\n    state = torch.load(assets[\'checkpoint\'], map_location=\'cpu\', weights_only=True)\n    net.load_state_dict(state[\'params\'], strict=True)\n    return SimpleNamespace(network=net.eval().to(device), task=task)\n\ndef document_pad(tensor, multiple=8, *, top_left=False, mode=\'reflect\'):\n    import torch.nn.functional as F\n    h, w = tensor.shape[-2:]\n    ph, pw = (-h % multiple, -w % multiple)\n    padding = (pw, 0, ph, 0) if top_left else (0, pw, 0, ph)\n    actual = \'replicate\' if mode == \'reflect\' and (ph >= h or pw >= w) else mode\n    return (F.pad(tensor, padding, mode=actual), dict(padding=list(padding), mode=actual, original_size=[w, h], padded_size=[w + pw, h + ph], image_to_input=[1, 0, 0, 1, padding[0], padding[2]]))\n\ndef document_tiled(tensor, predict, tile, overlap):\n    """Independent axes, CPU accumulation; no resize/rotation, no uncovered border."""\n    import torch\n    h, w = tensor.shape[-2:]\n    if not isinstance(tile, int) or tile < 8 or tile % 8 or (not isinstance(overlap, int)) or (not 0 <= overlap < tile) or overlap % 8:\n        raise ValueError(\'Document tile/overlap must be multiples of 8; 0 <= overlap < tile\')\n\n    def positions(length):\n        if length <= tile:\n            return [0]\n        return sorted(set([*range(0, length - tile + 1, tile - overlap), length - tile]))\n    sums = torch.zeros(1, 3, h, w, dtype=torch.float32)\n    weights = torch.zeros(1, 1, h, w, dtype=torch.float32)\n    count = 0\n    for y in positions(h):\n        for x in positions(w):\n            patch = tensor[:, :, y:min(y + tile, h), x:min(x + tile, w)]\n            result = predict(patch).detach().float().cpu()\n            if tuple(result.shape) != (1, 3, patch.shape[-2], patch.shape[-1]) or not torch.isfinite(result).all():\n                raise ValueError(\'Document model violated output shape/finite-pixel contract\')\n            sums[:, :, y:y + result.shape[-2], x:x + result.shape[-1]] += result\n            weights[:, :, y:y + result.shape[-2], x:x + result.shape[-1]] += 1\n            count += 1\n    if not (weights > 0).all():\n        raise ValueError(\'Uncovered document pixels\')\n    return (sums / weights, count)\n\ndef document_run(model, backend, image, options):\n    if model.startswith((\'docentr_\', \'degan_\')):\n        return learned_document_run(model, backend, image, options)\n    import cv2\n    import numpy as np\n    import torch\n    from PIL import Image\n    family, task, _ = DOCUMENT_VARIANTS[model]\n    rgb = np.asarray(image.convert(\'RGB\'))\n    if family == \'docres\':\n        bgr = rgb[:, :, ::-1].copy()\n        if task in (\'deblurring\', \'binarization\'):\n            top, left = (-image.height % 8, -image.width % 8)\n            padded = cv2.copyMakeBorder(bgr, top, 0, left, 0, cv2.BORDER_REPLICATE)\n            prompt = backend.prompts.deblur_prompt(padded) if task == \'deblurring\' else backend.prompts.binarization_promptv2(padded)\n            array = np.concatenate((padded, prompt), axis=2)\n            padding = dict(padding=[left, 0, top, 0], mode=\'replicate\', original_size=list(image.size), padded_size=[array.shape[1], array.shape[0]])\n        else:\n            prompt = backend.prompts.appearance_prompt(bgr) if task == \'appearance\' else backend.prompts.deshadow_prompt(bgr)\n            array = np.concatenate((bgr, prompt), axis=2)\n        tensor = torch.from_numpy(array.astype(np.float32) / 255).permute(2, 0, 1)[None]\n        if task in (\'appearance\', \'deshadowing\'):\n            tensor, padding = document_pad(tensor, top_left=True, mode=\'replicate\')\n\n        def predict(patch):\n            value = backend.network(patch.to(options.get(\'device\', \'cuda\')))\n            if task == \'binarization\':\n                value = value[:, :2].softmax(1).argmax(1, keepdim=True).float().repeat(1, 3, 1, 1)\n            return value\n    else:\n        tensor = torch.from_numpy(rgb.astype(np.float32) / 255).permute(2, 0, 1)[None]\n        tensor, padding = document_pad(tensor)\n\n        def predict(patch):\n            patch = patch.to(options.get(\'device\', \'cuda\'))\n            if family == \'docdiff\':\n                coarse = backend.network.init_predictor(patch, 0)\n                return coarse + backend.sampler(torch.randn_like(patch), coarse, \'True\')\n            return backend.network(patch)\n    with torch.inference_mode():\n        output, tiles = document_tiled(tensor, predict, options.get(\'tile\', 512), options.get(\'overlap\', 32))\n    left, _, top, _ = padding[\'padding\']\n    output = output[:, :, top:top + image.height, left:left + image.width]\n    pixels = (output[0].clamp(0, 1).permute(1, 2, 0).numpy() * 255).round().astype(\'uint8\')\n    if family == \'docres\':\n        pixels = pixels[:, :, ::-1].copy()\n    return (Image.fromarray(pixels), {}, dict(task=task, padding=padding, tiles=tiles, orientation=\'unchanged\', geometry=\'padding_removed; identity\', inference_mode=\'native_resolution_tiled; no_upstream_square_resize\', fidelity=\'unverified; inspect_strokes_and_red_ink\', seam_review=\'required\'))\n# END DOCUMENT RESTORATION ADAPTERS\n\n# BEGIN MULTI GPU ADAPTERS\ndef restoration_memory_limits(torch, options):\n    cap = options.get(\'max_memory\', \'18GiB\')\n    if not isinstance(cap, str) or not cap.endswith(\'GiB\'):\n        raise ValueError(\'max_memory must be a per-GPU limit such as 18GiB\')\n    requested = int(float(cap[:-3]) * 1024 ** 3)\n    reserve = int(float(options.get(\'activation_reserve_gib\', 4)) * 1024 ** 3)\n    if requested <= 0 or reserve < 1024 ** 3:\n        raise ValueError(\'Positive weight budget and >=1 GiB activation reserve required\')\n    limits = {}\n    for index in range(torch.cuda.device_count()):\n        free, total = torch.cuda.mem_get_info(index)\n        limit = min(requested, free - reserve)\n        if index == 0:\n            limit -= int(options.get(\'entry_reserve_gib\', 2) * 1024 ** 3)\n        if limit < 4 * 1024 ** 3:\n            raise MemoryError(f\'GPU {index}: insufficient free VRAM after activation reserve\')\n        limits[index] = limit\n    return limits\n\ndef verify_restoration_device_map(mapping, count, *, require_multiple=True):\n    devices = set()\n    for value in mapping.values():\n        if isinstance(value, int):\n            index = value\n        else:\n            label = str(value)\n            if not label.startswith(\'cuda:\'):\n                raise MemoryError(f\'Model would spill to {label}; reduce scope or change explicit placement budgets\')\n            index = int(label.split(\':\')[1])\n        if not 0 <= index < count:\n            raise ValueError(f\'Device map refers to unavailable GPU {index}\')\n        devices.add(index)\n    if require_multiple and len(devices) < 2:\n        raise ValueError(\'Sharded placement did not distribute the model across multiple GPUs\')\n    return mapping\n\ndef restoration_gpu_probe(torch, model, options):\n    """Validate every visible GPU; aggregate capacity is only a placement precheck."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required but unavailable\')\n    devices = []\n    for index in range(torch.cuda.device_count()):\n        (torch.ones(8, device=f\'cuda:{index}\') * 2).sum().item()\n        torch.cuda.synchronize(index)\n        devices.append(dict(index=index, name=torch.cuda.get_device_name(index), total_bytes=torch.cuda.get_device_properties(index).total_memory, free_bytes=torch.cuda.mem_get_info(index)[0], capability=list(torch.cuda.get_device_capability(index))))\n    sharded = options.get(\'placement\') == \'sharded\'\n    if sharded and len(devices) < 2:\n        raise RuntimeError(f\'{model}: sharded placement requires >=2 visible GPUs\')\n    if model in (\'unihir_restoration\', \'tadisr_x4\') and sharded:\n        limits = restoration_memory_limits(torch, options)\n        minimum = 32 if model == \'unihir_restoration\' else 28\n        if sum(limits.values()) < minimum * 1024 ** 3:\n            raise MemoryError(f\'{model}: insufficient aggregate weight budgets; inference workspace is reserved separately\')\n    elif model == \'unihir_restoration\' and devices[0][\'total_bytes\'] < 75 * 1024 ** 3:\n        raise RuntimeError(\'Single-GPU UniHIR needs >=75 GiB; use placement=sharded for multiple L4 GPUs\')\n    elif model == \'tadisr_x4\' and devices[0][\'total_bytes\'] < 24 * 1024 ** 3:\n        raise RuntimeError(\'Single-GPU float32 TADiSR needs >=24 GiB; use placement=sharded on L4\')\n    if model == \'unihir_restoration\' and any((d[\'capability\'][0] < 8 for d in devices)):\n        raise RuntimeError(\'UniHIR BF16/FlashAttention requires Ampere-or-newer GPUs\')\n    return dict(devices=devices, placement=options.get(\'placement\', \'single\'), cuda_visible_devices=__import__(\'os\').environ.get(\'CUDA_VISIBLE_DEVICES\'), weight_capacity_check_only=True)\n\ndef load_unihir_multi_gpu(assets, options):\n    import torch\n    import types\n    import utils as upstream\n    from accelerate import infer_auto_device_map, load_checkpoint_and_dispatch\n    limits = restoration_memory_limits(torch, options)\n\n    def infer(model, **kwargs):\n        kwargs.update(max_memory=limits, dtype=torch.bfloat16, no_split_module_classes=[\'Qwen2MoTDecoderLayer\'])\n        return verify_restoration_device_map(infer_auto_device_map(model, **kwargs), len(limits))\n\n    def dispatch(model, **kwargs):\n        verify_restoration_device_map(kwargs[\'device_map\'], len(limits))\n        return load_checkpoint_and_dispatch(model, **kwargs)\n    loader = types.FunctionType(upstream.load_model.__code__, {**upstream.load_model.__globals__, \'infer_auto_device_map\': infer, \'load_checkpoint_and_dispatch\': dispatch}, argdefs=upstream.load_model.__defaults__)\n    result = loader(assets[\'base_model\'], max_mem_per_gpu=options.get(\'max_memory\', \'18GiB\'))\n    result[0]._vtm_device_map = {k: str(v) for k, v in result[0].hf_device_map.items()}\n    return result\n\ndef load_tadisr_multi_gpu(assets, options):\n    import pathlib\n    import torch\n    from accelerate import dispatch_model, infer_auto_device_map\n    from tadisr.inference import CogView4Config, load_cogview4\n    net = load_cogview4(CogView4Config(pathlib.Path(assets[\'base_model\']), pathlib.Path(assets[\'checkpoint\']), pathlib.Path(assets[\'prompt_embeddings\']), \'cpu\'))\n    limits = restoration_memory_limits(torch, options)\n    no_split = getattr(net.transformer, \'_no_split_modules\', None) or [\'CogView4Block\']\n    mapping = infer_auto_device_map(net.transformer, max_memory=limits, no_split_module_classes=no_split, dtype=torch.float32)\n    verify_restoration_device_map(mapping, len(limits))\n    net.transformer = dispatch_model(net.transformer, device_map=mapping, main_device=0, force_hooks=True)\n    net.execution_device = torch.device(\'cuda:0\')\n    net.vae.to(net.execution_device)\n    net.js_decoder.to(net.execution_device)\n    net.prompt_embeds = net.prompt_embeds.to(net.execution_device)\n    net._vtm_device_map = {k: str(v) for k, v in mapping.items()}\n    return net\n\ndef restoration_cuda_metrics(torch, *, reset=False):\n    records = []\n    if torch.cuda.is_available():\n        for index in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(index)\n            if reset:\n                torch.cuda.reset_peak_memory_stats(index)\n            records.append(dict(index=index, peak_allocated_bytes=torch.cuda.max_memory_allocated(index), peak_reserved_bytes=torch.cuda.max_memory_reserved(index)))\n    return records\n# END MULTI GPU ADAPTERS\n\nif __name__==\'__main__\':main()\n'
RESTORATION_CODE_SIGNATURE = '884b974c5da614b1534612838310d69bfa615175a6fd8830375e1342504c05ce'


## 4. Bootstrap offline, preflight, cache, QC và output

Chế độ process chỉ dùng file local. Worker bật các cờ offline và audit hook chặn
kết nối mạng. PDFium thiếu được cài từ wheel tương thích vào thư mục riêng;
không sửa kernel. TADiSR có venv overlay riêng, tái dùng Torch/NumPy đã probe,
kiểm tra dependency closure thực tế. DiffTSR/UniHIR dùng overlay từ manifest đã chuẩn bị;
UniHIR biên dịch FlashAttention offline từ source có CUTLASS, rồi probe GPU. Runtime mặc định ở /tmp; kernel được giữ nguyên. Thiếu asset báo đúng đường dẫn.

Preflight kiểm checksum toàn bộ model trước GPU. Kiểm import/NumPy bridge/CUDA và
phép tính GPU trong worker; đây chưa thay smoke inference. Không tuyên bố model
sẵn sàng chỉ vì có weights. `ALLOW_PARTIAL=False` dừng trước PDF khi thiếu model
phù hợp đầu vào; lỗi region giữ cache nhưng không thay `dataset.json` hoàn chỉnh.

Trên Kaggle, `KERNEL_TORCH_SOURCE_POLICY='warn'` cho phép riêng `torch/__init__.py` của kernel khác hash `RECORD`; hash kỳ vọng/thực tế được ghi trong `preflight.json` và tham gia cache key, không được tính là đã xác minh checksum. Native Torch/NumPy, source trong overlay, wheel và checkpoint vẫn kiểm tra nghiêm ngặt. Import, dependency, NumPy bridge và phép tính CUDA thực vẫn phải qua. Chọn `'strict'` để dừng cả khi source Torch kernel khác hash.

Đường dẫn Torch được lấy trực tiếp từ kernel rồi truyền cho worker, không suy đoán từ thư mục site-packages. Cấu hình cũ chưa có policy tự dùng mặc định theo môi trường. Đường dẫn Python của venv được giữ nguyên để không đi qua symlink về Python hệ thống; môi trường thiếu interpreter hoặc receipt hỏng được dựng lại từ wheel local. Thông báo lỗi worker kèm interpreter, policy và log.


In [ ]:
from __future__ import annotations
import os, sys, json, time, hashlib, io, shutil, subprocess, tempfile, uuid
from pathlib import Path
from dataclasses import asdict

CLASSICAL = ('original','clahe','bicubic_x2','bicubic_x4','denoised', *EXTRA_CLASSICAL)
NEURAL = ('realesrgan_x4', 'tadisr_x4', 'difftsr', 'unihir_restoration', 'docentr_binarization', 'degan_binarization', 'degan_deblurring', 'degan_watermark_removal', 'docres_appearance', 'docres_deblurring', 'docres_deshadowing', 'docres_binarization', 'docdiff_deblurring', 'restormer_motion_deblurring', 'restormer_defocus_deblurring', 'restormer_real_denoising', 'restormer_gaussian_denoising')
# BEGIN DOCUMENT ASSETS
DOCUMENT_VARIANTS = {'docentr_binarization': ('docentr', 'binarization_base_2018_patch8', 'model_8_2018_base.pt'), 'degan_binarization': ('degan', 'binarization', 'degan_weights.bundle'), 'degan_deblurring': ('degan', 'deblurring', 'degan_weights.bundle'), 'degan_watermark_removal': ('degan', 'watermark_removal', 'degan_weights.bundle'), 'docres_appearance': ('docres', 'appearance', 'docres.pkl'), 'docres_deblurring': ('docres', 'deblurring', 'docres.pkl'), 'docres_deshadowing': ('docres', 'deshadowing', 'docres.pkl'), 'docres_binarization': ('docres', 'binarization', 'docres.pkl'), 'docdiff_deblurring': ('docdiff', 'deblurring', 'init.pth'), 'restormer_motion_deblurring': ('restormer', 'motion', 'motion_deblurring.pth'), 'restormer_defocus_deblurring': ('restormer', 'defocus', 'single_image_defocus_deblurring.pth'), 'restormer_real_denoising': ('restormer', 'real', 'real_denoising.pth'), 'restormer_gaussian_denoising': ('restormer', 'gaussian_color_blind', 'gaussian_color_denoising_blind.pth')}
AUTO_EXCLUDED_VARIANTS = ('difftsr',)
DOCUMENT_SOURCE_RECORDS = {'degan': [{'path': 'image_restoration/sources/degan/README.md', 'bytes': 3542, 'sha256': 'bdeeb6ac40deefcf22a1eb8c357f16199dc8c8410248fef303885684d26500c0'}, {'path': 'image_restoration/sources/degan/LICENSE', 'bytes': 35149, 'sha256': '3972dc9744f6499f0f9b2dbf76696f2ae7ad8af9b23dde66d6af86c9dfb36986'}, {'path': 'image_restoration/sources/degan/requirements.txt', 'bytes': 210, 'sha256': '883a1ac162065401c16d28e990872244b546a67d071e4fdbd5d7c174ff37dd01'}, {'path': 'image_restoration/sources/degan/enhance.py', 'bytes': 1845, 'sha256': '04a248452b8bef138e4551e544d67454de0bd2f17529beb1cc6c18de8958e3aa'}, {'path': 'image_restoration/sources/degan/models/models.py', 'bytes': 4975, 'sha256': '8b36b5b525fd1b5c7ba7689b27ee910e51d08e016b69c69235a463b0ad02174b'}, {'path': 'image_restoration/sources/degan/utils.py', 'bytes': 1837, 'sha256': '26d1cb343f371abae621c1d0e8169c5d2adbb1d554e2f0f191862c4aabf01dac'}], 'docdiff': [{'path': 'image_restoration/sources/docdiff/LICENSE', 'bytes': 1070, 'sha256': '321d494f33abbfe86bd181c1feaf73575bc145b69c6a662378d76ff0fe0ec525'}, {'path': 'image_restoration/sources/docdiff/README.EN.md', 'bytes': 10342, 'sha256': '2f6b498599a25fc395e025472664987250d19636c2482e1f780cec6fd06ec158'}, {'path': 'image_restoration/sources/docdiff/README.md', 'bytes': 10050, 'sha256': '62958e3374ee5bc0f925bbd1ae19e2f903ff70c5d4a0b230ec24abcdfc36f102'}, {'path': 'image_restoration/sources/docdiff/conf.yml', 'bytes': 2351, 'sha256': '5b2ad9b38db48c204f44325fae177bcf0a718810354c208f903a00ed0b5b1dac'}, {'path': 'image_restoration/sources/docdiff/demo/inference.ipynb', 'bytes': 2961354, 'sha256': 'd9e013f9f26ae3427dc7f864104125f285d37c5a0ecaff3346f544580a036f7a'}, {'path': 'image_restoration/sources/docdiff/model/DocDiff.py', 'bytes': 14497, 'sha256': '7ad026fb16c5f8dea19d9ec7731a94cd2b042405544cb1dbb1b25fc76077012b'}, {'path': 'image_restoration/sources/docdiff/schedule/diffusionSample.py', 'bytes': 4116, 'sha256': 'e03e1ce4204dcf6be098a0809ada8cb19852dd268a08b1d53678013854df2d95'}, {'path': 'image_restoration/sources/docdiff/schedule/dpm_solver_pytorch.py', 'bytes': 68846, 'sha256': '5137898f74145446458a2bf89cd69bc347bc7b776a0dee1abee2f4f3262cd05c'}, {'path': 'image_restoration/sources/docdiff/schedule/schedule.py', 'bytes': 1825, 'sha256': 'd916ae0df5067bbc8ec70ee17ad3bea0617df7851212ddd029e8b390f3663e6b'}, {'path': 'image_restoration/sources/docdiff/src/sobel.py', 'bytes': 1473, 'sha256': '660f4d547bf7d1d7ecfde9065e718e7d9887350ebeea8bb8505905aa8a78c39e'}], 'docentr': [{'path': 'image_restoration/sources/docentr/README.md', 'bytes': 8428, 'sha256': '2885fa36ae07b175ac5d8953ce4cfddc9ca97f58fa1f72e4507c6e45533bfd36'}, {'path': 'image_restoration/sources/docentr/LICENSE', 'bytes': 19346, 'sha256': '3711f963c05d0be80d53e5923308a6dee31b203da23435c9cfb7c7b6e4dd5e19'}, {'path': 'image_restoration/sources/docentr/requirements.txt', 'bytes': 165, 'sha256': '62d08b7f17c58c05eb47b842374ea18e3dd226d3613a1e4b4ac369a0233973cf'}, {'path': 'image_restoration/sources/docentr/predict.py', 'bytes': 6468, 'sha256': '713f2f0669d312fbae33cf9dbb8b7fd3c09f6315c0f9ae9325ccc649c7c0f9b4'}, {'path': 'image_restoration/sources/docentr/models/binae.py', 'bytes': 2595, 'sha256': '659f84c618d8d96c64801f35471b72b927e7dcf2cfb3f8250dd1e6389864e581'}], 'docres': [{'path': 'image_restoration/sources/docres/LICENSE', 'bytes': 1069, 'sha256': '2fa58d878abc9ea7b2e4780e736eb45c904ace4cb167504b8549210f3d32df1e'}, {'path': 'image_restoration/sources/docres/README.md', 'bytes': 3668, 'sha256': '394a208036a692b64c2472671de98c3a5d6481ba720eba38dc167847404340ed'}, {'path': 'image_restoration/sources/docres/data/MBD/MBD.py', 'bytes': 4543, 'sha256': '9d9e150bcd75d8668bde48744e13d32572cd2de2e0c9bcbd9b5f6bd8cdbd4119'}, {'path': 'image_restoration/sources/docres/data/MBD/MBD_utils.py', 'bytes': 11492, 'sha256': '7575c694fe7b44d71feab77b0c27dc035835fa314fe5e4f9c42e635a437df2cc'}, {'path': 'image_restoration/sources/docres/data/MBD/infer.py', 'bytes': 6054, 'sha256': 'c012ba3e8032d11c32547cb2ef45bc8b0c8f8fab23553d45a94f8fcead8be777'}, {'path': 'image_restoration/sources/docres/data/MBD/model/__init__.py', 'bytes': 2084, 'sha256': 'dca616a744ddb1855bc00a73ad3db329a8acfdf336d424c28cd9a89e301751a4'}, {'path': 'image_restoration/sources/docres/data/MBD/model/cbam.py', 'bytes': 3859, 'sha256': '7a1be8a910d83a9eda3df4606ae95da02c1fb55284956d570cc2ecb3e859545c'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/__init__.py', 'bytes': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/aspp.py', 'bytes': 3614, 'sha256': 'caf846f6bdca7db3427565cd99f39a6a0b0fecbc3f9a6c295c970a9caf2eb8fd'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/backbone/__init__.py', 'bytes': 526, 'sha256': '8b377895b4180684785128c8d539bd3c8b1fb5fd70d76215b928f15080007097'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/backbone/drn.py', 'bytes': 14632, 'sha256': 'b290cb9b3c30ae6945faf4052492925e9f827aa63297146e3f2ae20d019a2a71'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/backbone/mobilenet.py', 'bytes': 5402, 'sha256': '42c9dcad4786e24b62e4dcc3e0f1709a39efeca3ea1e0af51355cb3aa67c5c11'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/backbone/resnet.py', 'bytes': 6535, 'sha256': '159d46ccdfb3bd2d4c8cfeff330ac0ce100ed7079078cd3f4d95b453c615271f'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/backbone/xception.py', 'bytes': 11566, 'sha256': '0e331acdcfb0a00acbae08c1a61e07748096a99885e741aca093e8c2fab90a04'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/decoder.py', 'bytes': 2377, 'sha256': '8d9fb253c36638b9d849a8606df59aa913f018751e96c1e4bcfe4a8e7dff907e'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/deeplab.py', 'bytes': 3030, 'sha256': 'f7b34691562146c802a8bc55512c87113cace12c1e13c20584badd88b9127dd4'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/sync_batchnorm/__init__.py', 'bytes': 447, 'sha256': '302e1c1df4ad71871e109cb1e053c601bf78299054ab38d53f757afade6744ce'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/sync_batchnorm/batchnorm.py', 'bytes': 12932, 'sha256': '938743d77c735bee3b4820a5e28883dad3252dadcf71e8b4e89610827e5e7243'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/sync_batchnorm/comm.py', 'bytes': 4440, 'sha256': 'fad714b9c85de1431453b3d26f52f4f517bf84018cfe80743d24bac17504387d'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/sync_batchnorm/replicate.py', 'bytes': 3218, 'sha256': 'b0ef534c558612943a5891b2ef0987267036a1b19dc72ab0018691dc409a90d2'}, {'path': 'image_restoration/sources/docres/data/MBD/model/deep_lab_model/sync_batchnorm/unittest.py', 'bytes': 834, 'sha256': 'c152f08abfbe56b0809870568ab4b213d3a533d5ca88b990fe422c634e79eea7'}, {'path': 'image_restoration/sources/docres/data/MBD/model/densenetccnl.py', 'bytes': 14377, 'sha256': '734e16ba31a87b8e6a6faac0cc9b13c3228ebc2e2a32d48263d502ca79bee56d'}, {'path': 'image_restoration/sources/docres/data/MBD/model/gienet.py', 'bytes': 31924, 'sha256': '4c3233225089e1e517113ddf96761b6402fe2fe2b85e25c5f3f217cfe0b92f62'}, {'path': 'image_restoration/sources/docres/data/MBD/model/unetnc.py', 'bytes': 3785, 'sha256': '604d625d22796c420ecfe2e18f049216f5670abe4074f9065101067621d39e2e'}, {'path': 'image_restoration/sources/docres/data/MBD/modify_stn_model/stn_head.py', 'bytes': 4565, 'sha256': '404e511e59da196dfe19694c21a613185303becf750024f8d0dd2936e887eeda'}, {'path': 'image_restoration/sources/docres/data/MBD/modify_stn_model/tps_spatial_transformer.py', 'bytes': 9055, 'sha256': 'e9d79281d7ee12948cb351e5ac4491414dfd8d67b4b938664c688e4afb5bf526'}, {'path': 'image_restoration/sources/docres/data/MBD/stn_model/stn_head.py', 'bytes': 4565, 'sha256': '404e511e59da196dfe19694c21a613185303becf750024f8d0dd2936e887eeda'}, {'path': 'image_restoration/sources/docres/data/MBD/stn_model/tps_spatial_transformer.py', 'bytes': 7408, 'sha256': '8a2769562aabc90b0abb534f4dd1bbf96814683684c189b9e76e30963edc6d1f'}, {'path': 'image_restoration/sources/docres/data/MBD/tps_grid_gen.py', 'bytes': 3431, 'sha256': 'd661fb9d40142f53fd3948caf1047b2947c2cfa3c41b7e6a233eaf221b9095f3'}, {'path': 'image_restoration/sources/docres/data/MBD/utils.py', 'bytes': 8140, 'sha256': '28492bf442cf144afb4e02cd037d51ac81d01d8cd8d6de194a532b98d16714ad'}, {'path': 'image_restoration/sources/docres/data/preprocess/crop_merge_image.py', 'bytes': 4605, 'sha256': 'c0baa9260845744116e2e48d134abcb6eab393030afc0e70a0c5a7a4c5cea6b6'}, {'path': 'image_restoration/sources/docres/inference.py', 'bytes': 13207, 'sha256': '32dcebbebb98f55a1b1e1022f61519662489807ddf146e76d4fa7b09c55b6ded'}, {'path': 'image_restoration/sources/docres/models/restormer_arch.py', 'bytes': 12203, 'sha256': '4a6f50552cf21c016f3c38f0302ff9c9cea07a38111c219ef9e67279e60d6915'}, {'path': 'image_restoration/sources/docres/requirements.txt', 'bytes': 180, 'sha256': '21470c4d1ef5c2ea70abfc207960389c4ca1ff1e2330a4ab7278800fb86d2315'}, {'path': 'image_restoration/sources/docres/utils.py', 'bytes': 14103, 'sha256': '607e665244da535de3563e1f254ae651acdc3d333c8ab7f7decaa49c016efc85'}], 'restormer': [{'path': 'image_restoration/sources/restormer/Defocus_Deblurring/Options/DefocusDeblur_DualPixel_16bit_Restormer.yml', 'bytes': 3175, 'sha256': '2325dabb98fbdd5431369db95eb9adba43414db86bc8b754c985e472d02220b4'}, {'path': 'image_restoration/sources/restormer/Defocus_Deblurring/Options/DefocusDeblur_Single_8bit_Restormer.yml', 'bytes': 2992, 'sha256': '73b29581ca925158b6f5dad75ac43d4190ecb8a30a6811826c3a00ad47f64542'}, {'path': 'image_restoration/sources/restormer/Defocus_Deblurring/README.md', 'bytes': 1367, 'sha256': '9feb837cf4761d876f324003df4b2f59aa3d44497f9038c126db065f21a5e05a'}, {'path': 'image_restoration/sources/restormer/Defocus_Deblurring/pretrained_models/README.md', 'bytes': 145, 'sha256': '46b453f40eb4f08bcf058f1c8fe4736774e68f683b344732b89ccecb1ef898a5'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianColorDenoising_Restormer.yml', 'bytes': 2984, 'sha256': 'e77c7ac4dbb602ff64d646e5a9aaf406934707ba478e92a73b70809813e8341a'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianColorDenoising_RestormerSigma15.yml', 'bytes': 2990, 'sha256': 'cdfd64b57955596da123907853514a606377365f9296a27a68f47697acecc6c8'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianColorDenoising_RestormerSigma25.yml', 'bytes': 2990, 'sha256': '5c99e637ad1974c56d2f2251aa9919817d88e97f4d5070e84af14cef1ed0cc6f'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianColorDenoising_RestormerSigma50.yml', 'bytes': 2990, 'sha256': '9896f59e20430f9a015ef6262c4bd03b65accdc6dcf752238840453a39441f12'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianGrayDenoising_Restormer.yml', 'bytes': 2995, 'sha256': '1c2fba477d4891c2b7dc21939088a42f153ad102382b20aa7c9c0e85e9e41753'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianGrayDenoising_RestormerSigma15.yml', 'bytes': 3002, 'sha256': '79b4739bb69a0f4ddd7c2d5505998d25248217824a0e902c5fd787be043c5081'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianGrayDenoising_RestormerSigma25.yml', 'bytes': 3002, 'sha256': '3eba8830063fb4dec8c49c0c4c3b7a9bb170932fa4b96932396eb5794a3cc205'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/GaussianGrayDenoising_RestormerSigma50.yml', 'bytes': 3002, 'sha256': 'af533339ac4a3bbfeefd9de8c1bdea99e7ac1cf0eb396b6230ef02cd3dbd9530'}, {'path': 'image_restoration/sources/restormer/Denoising/Options/RealDenoising_Restormer.yml', 'bytes': 2941, 'sha256': '697bf6402c58d22e1f7314f1f87fbed2a5de6f5f912ae9dcc96bba203b3b5b32'}, {'path': 'image_restoration/sources/restormer/Denoising/README.md', 'bytes': 5292, 'sha256': '5c7f00a76bc2590cfd4a2ae64b7044098363e0fe78ae2d53e3fef1730605001a'}, {'path': 'image_restoration/sources/restormer/Denoising/pretrained_models/README.md', 'bytes': 133, 'sha256': 'e2e64edfff5b9558378337bca044db6af5145dc2a67eaf57c92f2cef852c5368'}, {'path': 'image_restoration/sources/restormer/INSTALL.md', 'bytes': 1289, 'sha256': 'ea6aeef9c58fa6cde333dac8ef073d0f5f714c8813cde287bc5f02abd6bc4807'}, {'path': 'image_restoration/sources/restormer/LICENSE.md', 'bytes': 1090, 'sha256': '2b93776512924bc095ec5d97a79d76cf1ab401ae641d1ae1f46e94f1c53a2e59'}, {'path': 'image_restoration/sources/restormer/Motion_Deblurring/Options/Deblurring_Restormer.yml', 'bytes': 2974, 'sha256': '8f4403ec761973f1c0677fae07841df3935075cea2f3871932d3cd26d1b16e73'}, {'path': 'image_restoration/sources/restormer/Motion_Deblurring/README.md', 'bytes': 1730, 'sha256': '2db2809814de46f3137921be93d79533bb528740f436b1974ac00725e79a4d6b'}, {'path': 'image_restoration/sources/restormer/Motion_Deblurring/pretrained_models/README.md', 'bytes': 134, 'sha256': 'd401316a609f0bd16658f9ee46d3204ab2794c9eceb44757deda949cea6fddc3'}, {'path': 'image_restoration/sources/restormer/README.md', 'bytes': 9620, 'sha256': '6a1ce0d30f78378ac64347840bf22985b0da75751442f55dd1ba7b14e7426d3e'}, {'path': 'image_restoration/sources/restormer/basicsr/models/archs/restormer_arch.py', 'bytes': 11428, 'sha256': '3be243fa3c8e2cb2c9459eeac062b04d6084dbcc328292dae1c3e52ba6f7434a'}, {'path': 'image_restoration/sources/restormer/demo.py', 'bytes': 7719, 'sha256': '14164c47978fe1f692e0bb1b097f5fa8861e70f3d43a48bd13adeec23ed6442a'}]}
DOCUMENT_WEIGHT_RECORDS = {'degan': [{'path': 'models/image_restoration/degan/degan_weights.bundle', 'url': 'https://drive.usercontent.google.com/download?id=1J_t-TzR2rxp94SzfPoeuJniSFLfY3HM-&export=download&confirm=t', 'bytes': 289138005, 'sha256': '91f33d11c87d042a00de2912e3828fe291be5247ab76a5ffb09551d47a3cc70b', 'kind': 'checkpoint', 'revision': 'google-drive-file-1J_t-TzR2rxp94SzfPoeuJniSFLfY3HM-', 'source_verification': 'Checkpoint link in official dali92002/DE-GAN README at 7281836455608337c940fa0de67cceffbc5c026d', 'checksum_origin': 'first_TLS_download_pinned_in_catalog', 'container_format': 'zip'}], 'docdiff': [{'path': 'models/image_restoration/docdiff/init.pth', 'url': 'https://raw.githubusercontent.com/Royalvice/DocDiff/f422a3af10657eeabeca90f2e862d88c72725845/checksave/init.pth', 'bytes': 16161781, 'git_blob_sha1': 'ded5e03c6fd6420ebeb63496771634fbafd99694', 'kind': 'checkpoint', 'revision': 'f422a3af10657eeabeca90f2e862d88c72725845', 'sha256': '2a1a68010683e9b692034bafb572ec1c476e47c6bd1753d0a086f95e01790fbc'}, {'path': 'models/image_restoration/docdiff/denoiser.pth', 'url': 'https://raw.githubusercontent.com/Royalvice/DocDiff/f422a3af10657eeabeca90f2e862d88c72725845/checksave/denoiser.pth', 'bytes': 16753405, 'git_blob_sha1': 'dd7cb52b37c0a59559b789a14eaa4d5d44c2b7d0', 'kind': 'checkpoint', 'revision': 'f422a3af10657eeabeca90f2e862d88c72725845', 'sha256': 'a97ef81372b20e733a5b52fdb6ec3a338ec045e372846c01798e77736581d983'}], 'docentr': [{'path': 'models/image_restoration/docentr/model_8_2018_base.pt', 'url': 'https://drive.usercontent.google.com/download?id=1CpvS9ahZolRz2sJ4PHobofOXVJBMWwD0&export=download&confirm=t', 'bytes': 275213537, 'sha256': 'd0294ad99fcde046c4298821183f7d38e712e3219728305fe0c52fec8ef75498', 'kind': 'checkpoint', 'revision': 'google-drive-file-1CpvS9ahZolRz2sJ4PHobofOXVJBMWwD0', 'source_verification': 'Checkpoint link in official dali92002/DocEnTR README at 004159e10f69f2b33f5800eeba4175d6f93cb17a', 'checksum_origin': 'first_TLS_download_pinned_in_catalog'}], 'docres': [{'path': 'models/image_restoration/docres/docres.pkl', 'url': 'https://huggingface.co/spaces/qubvel-hf/documents-restoration/resolve/e9d734906ea26ec569a8daeac0bfd14f495990ab/checkpoints/docres.pkl', 'bytes': 183260711, 'sha256': '1d6a89d754fe1e58ffd1865eab0ef3f03344798d39197b2d9a77ce4fbc8c02fd', 'kind': 'checkpoint', 'revision': 'e9d734906ea26ec569a8daeac0bfd14f495990ab', 'source_verification': 'Space linked directly by official DocRes README'}], 'restormer': [{'path': 'models/image_restoration/restormer/gaussian_color_denoising_blind.pth', 'url': 'https://github.com/swz30/Restormer/releases/download/v1.0/gaussian_color_denoising_blind.pth', 'bytes': 104611957, 'sha256': None, 'kind': 'checkpoint', 'revision': 'release-v1.0', 'release_asset_id': 61003978, 'source_verification': 'Official GitHub release; publisher does not supply SHA256, first-download digest locked in receipt'}, {'path': 'models/image_restoration/restormer/motion_deblurring.pth', 'url': 'https://github.com/swz30/Restormer/releases/download/v1.0/motion_deblurring.pth', 'bytes': 104700429, 'sha256': None, 'kind': 'checkpoint', 'revision': 'release-v1.0', 'release_asset_id': 61003755, 'source_verification': 'Official GitHub release; publisher does not supply SHA256, first-download digest locked in receipt'}, {'path': 'models/image_restoration/restormer/real_denoising.pth', 'url': 'https://github.com/swz30/Restormer/releases/download/v1.0/real_denoising.pth', 'bytes': 104611957, 'sha256': None, 'kind': 'checkpoint', 'revision': 'release-v1.0', 'release_asset_id': 61004273, 'source_verification': 'Official GitHub release; publisher does not supply SHA256, first-download digest locked in receipt'}, {'path': 'models/image_restoration/restormer/single_image_defocus_deblurring.pth', 'url': 'https://github.com/swz30/Restormer/releases/download/v1.0/single_image_defocus_deblurring.pth', 'bytes': 104700429, 'sha256': None, 'kind': 'checkpoint', 'revision': 'release-v1.0', 'release_asset_id': 61003811, 'source_verification': 'Official GitHub release; publisher does not supply SHA256, first-download digest locked in receipt'}]}
DOCUMENT_REVISIONS = {'degan': '7281836455608337c940fa0de67cceffbc5c026d', 'docdiff': 'f422a3af10657eeabeca90f2e862d88c72725845', 'docentr': '004159e10f69f2b33f5800eeba4175d6f93cb17a', 'docres': 'd3e3a18c7c7ad10e4615ca2eb6a10f83128d8560', 'restormer': '68dc6ac472db26f16361150cb7a96a1bc87da93f'}

def prep_document_assets(model, models_dir, packages_dir):
    family, task, filename = DOCUMENT_VARIANTS[model]
    folder = Path(models_dir)/'image_restoration'/family
    required = [filename] + (['denoiser.pth'] if family == 'docdiff' else [])
    weights = prep_check_manifest(folder, required)
    # Input datasets may mount models and packages separately. All references in
    # the pinned source table are relative to the packages root.
    vendor = Path(packages_dir)/'image_restoration/sources'/family
    records = DOCUMENT_SOURCE_RECORDS[family]
    for r in records:
        p = dataset_path(Path(packages_dir), r['path'])
        if not p.is_file(): raise FileNotFoundError(p)
        if p.stat().st_size != r['bytes'] or dataset_file_hash(p) != r['sha256']:
            raise ValueError(f'Pinned upstream source checksum mismatch: {p}')
    inventory = json.loads((folder/'assets.json').read_text(encoding='utf8'))
    for r in DOCUMENT_WEIGHT_RECORDS[family]:
        if Path(r['path']).name not in required: continue
        p = folder/Path(r['path']).name
        if p.stat().st_size != r['bytes'] or (r.get('sha256') and dataset_file_hash(p) != r['sha256']):
            raise ValueError(f'Released checkpoint identity mismatch: {p}')
    if inventory.get('revision') != DOCUMENT_REVISIONS[family]:
        raise ValueError('Model manifest revision differs from pinned upstream revision')
    assets = {'checkpoint': str((folder/filename).resolve()), 'vendor': str(vendor.resolve())}
    if family == 'docdiff': assets['denoiser'] = str((folder/'denoiser.pth').resolve())
    if family == 'degan':
        if not inventory.get('archive_members'): raise ValueError('DE-GAN member manifest missing; run prepare_learned_restoration_assets.py')
        assets['archive_members'] = inventory['archive_members']
    return assets, dataset_hash({'weights': weights, 'sources': records, 'revision': DOCUMENT_REVISIONS[family], 'task': task})
# END DOCUMENT ASSETS
SUPPORTED = (*CLASSICAL,*NEURAL,'tadisr_text_mask')
PROVENANCE = {**dict.fromkeys(CLASSICAL,'deterministic_transformation'),
    'original':'source_observation',**dict.fromkeys(NEURAL,'neural_restoration'),
    'tadisr_text_mask':'neural_restoration','unihir_restoration':'generative_reconstruction'}

_canonical_dataset_write=dataset_write
# MULTI GPU RUNTIME V1
from threading import RLock
_prep_io_lock=RLock()
_storage_guard=None
def dataset_write(root,relative,data):
    with _prep_io_lock:
        global _storage_guard
        path=dataset_path(root,relative)
        if _storage_guard and Path(root).resolve()==_storage_guard['root']:
            growth=max(0,len(data)-(path.stat().st_size if path.exists() else 0))
            if _storage_guard['used']+growth>_storage_guard['limit']:
                raise RuntimeError('max_storage_bytes exceeded before atomic write')
            ref=_canonical_dataset_write(root,relative,data)
            _storage_guard['used']+=growth
            return ref
        return _canonical_dataset_write(root,relative,data)

PREPARATION_FAILURE_STATES = {'failed', 'missing_assets', 'runtime_incompatible', 'inference_failed', 'timeout', 'resource_exceeded'}

def restoration_failure_state(error, *, phase='inference'):
    message = str(error).lower()
    if isinstance(error, (TimeoutError,)) or 'timeout' in message or 'timed out' in message:
        return 'timeout'
    if isinstance(error, MemoryError) or any((s in message for s in ('out of memory', 'exceeds max_', 'storage budget', 'no space left', 'disk full'))):
        return 'resource_exceeded'
    if isinstance(error, FileNotFoundError):
        return 'missing_assets'
    return 'runtime_incompatible' if phase == 'preflight' else 'inference_failed'







def select_preparation_variants(mode, selected):
    if mode == 'selected':
        unavailable={n:UNAVAILABLE_RESTORATION_METHODS[n] for n in selected if n in UNAVAILABLE_RESTORATION_METHODS}
        if unavailable: raise ValueError('No runnable upstream/checkpoint for requested methods: '+str(unavailable))
        return list(selected)
    if mode == 'classical': return list(CLASSICAL)
    if mode == 'all_eligible': return list(CLASSICAL) + [n for n in NEURAL if n not in AUTO_EXCLUDED_VARIANTS]
    raise ValueError('VARIANT_MODE must be classical, selected or all_eligible')

def prep_normalize(config):
    cfg=json.loads(json.dumps(config));variants=cfg['variants']
    validate_applicability(cfg)
    for name, opts in cfg['variant_options'].items():
        if name in EXTRA_CLASSICAL: classical_parameters(name, opts)
    if 'unihir' in variants:variants[variants.index('unihir')]='unihir_restoration'
    if set(variants)-set(SUPPORTED) or len(set(variants))!=len(variants):raise ValueError('Unknown/duplicate variants')
    if 'original' not in variants:variants.insert(0,'original')
    if 'tadisr_x4' in variants and 'tadisr_text_mask' not in variants:variants.append('tadisr_text_mask')
    if 'tadisr_text_mask' in variants and 'tadisr_x4' not in variants:raise ValueError('Mask requires tadisr_x4')
    if not cfg['crop_types'] or set(cfg['crop_types'])-{'page','group','column'}:raise ValueError('Invalid crop_types')
    for key in ('max_source_pixels','max_output_pixels','max_storage_bytes','timeout_seconds','load_timeout_seconds'):
        if cfg[key]<=0:raise ValueError(f'{key} must be positive')
    # Keep older user settings compatible when only the implementation cells were updated.
    source_policy=cfg.get('kernel_torch_source_policy',
        'warn')
    if source_policy not in ('strict','warn'):raise ValueError('KERNEL_TORCH_SOURCE_POLICY must be strict or warn')
    cfg['kernel_torch_source_policy']=source_policy
    for opts in cfg['model_options'].values(): opts['kernel_torch_source_policy']=source_policy
    for model in NEURAL:
        opts=cfg['model_options'].setdefault(model, {'tile':512,'overlap':32,'device':'cuda','seed':42})
        opts['kernel_torch_source_policy']=source_policy
        if opts.get('tile',1)<=0 or not 0<=opts.get('overlap',0)<opts.get('tile',1):raise ValueError(f'{model}: invalid tiling')
    tad=cfg['model_options']['tadisr_x4']
    if tad['tile']%32 or tad['overlap']%32:raise ValueError('TADiSR tile/overlap require multiples of 32')
    if cfg['batch_size']!=1:raise ValueError('Only batch_size=1 has a validated memory contract')
    if not 1<=cfg['model_options']['difftsr']['steps']<=200:raise ValueError('DiffTSR steps must be 1..200')
    if cfg['deskew'] or cfg['dewarp']:raise ValueError('Deskew/dewarp need a separately validated geometry profile; disabled')
    cfg.setdefault('gpu_ids',[0])
    cfg.setdefault('execution_mode','sequential')
    if cfg['execution_mode'] not in ('sequential','parallel_models'):
        raise ValueError('EXECUTION_MODE must be sequential or parallel_models')
    gpu_worker_environment(cfg['gpu_ids'])
    small=0
    for model in NEURAL:
        opts=cfg['model_options'].setdefault(model, {'tile':512,'overlap':32,'device':'cuda','seed':42})
        opts.setdefault('gpu_ids', [cfg['gpu_ids'][small % len(cfg['gpu_ids'])]])
        if not set(opts['gpu_ids']) <= set(cfg['gpu_ids']): raise ValueError(f'{model}: GPU IDs outside GPU_IDS')
        gpu_worker_environment(opts['gpu_ids'])
        if model in cfg['variants'] and len(opts['gpu_ids'])==1: small+=1
    return cfg

def prep_worker_environment():
    env=dict(os.environ)
    for key in ('PYTHONPATH','PYTHONHOME','PYTHONUSERBASE'):env.pop(key,None)
    env.update(PYTHONNOUSERSITE='1',PYTHONUTF8='1',HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',
        DIFFUSERS_OFFLINE='1',HF_DATASETS_OFFLINE='1',HF_HUB_DISABLE_TELEMETRY='1',
        PIP_NO_INDEX='1',PIP_DISABLE_PIP_VERSION_CHECK='1',TOKENIZERS_PARALLELISM='false',KERAS_BACKEND='torch')
    return env

def prep_bootstrap(packages_dir, runtime_dir):
    """Install missing PDFium into a private import directory; never into kernel."""
    if Path('/kaggle/input').exists() and Path(runtime_dir).resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('RUNTIME_DIR must be writable and outside /kaggle/input')
    from importlib.util import find_spec
    import PIL,numpy,cv2
    # Existing NumPy/OpenCV must operate together; no ABI replacement in a running kernel.
    cv2.cvtColor(numpy.ones((4,4,3),dtype=numpy.uint8),cv2.COLOR_RGB2GRAY)
    if find_spec('pypdfium2') is None:
        candidates=sorted(Path(packages_dir).rglob('pypdfium2-*.whl'))
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
        tags=set(sys_tags());candidates=[p for p in candidates if tags & parse_wheel_filename(p.name)[3]]
        if not candidates:raise FileNotFoundError(f'Missing compatible PDFium wheel below {packages_dir}; attach the offline packages Dataset')
        target=Path(runtime_dir)/'base_lib';target.mkdir(parents=True,exist_ok=True)
        subprocess.run([sys.executable,'-m','pip','--isolated','install','--no-index','--no-deps',
            '--target',str(target),str(candidates[-1])],check=True,env=prep_worker_environment())
        sys.path.insert(0,str(target))
    import pypdfium2
    return {'pillow':PIL.__version__,'numpy':numpy.__version__,'opencv':cv2.__version__,
        'pdfium':str(pypdfium2.PDFIUM_INFO),'python':sys.version}

def prep_vendor(runtime_dir, model):
    name={'tadisr_x4':'TADiSR','difftsr':'DiffTSR','unihir_restoration':'UniHIR'}.get(model)
    if not name:return None
    folder=Path(runtime_dir)/'vendor'/name
    for relative,source in VENDOR_SOURCES[name].items():
        p=dataset_path(folder,relative);p.parent.mkdir(parents=True,exist_ok=True)
        if not p.exists() or p.read_text(encoding='utf8')!=source:p.write_text(source,encoding='utf8')
    return str(folder.resolve())

def prep_kernel_torch_source():
    """Capture the notebook kernel's import origin without importing Torch or loading CUDA."""
    from importlib.util import find_spec
    spec=find_spec('torch')
    return str(Path(spec.origin).resolve()) if spec and spec.origin else None

def prep_log_tail(log, maximum=8000):
    try:
        with Path(log).open('rb') as stream:
            stream.seek(0,2);stream.seek(max(0,stream.tell()-maximum))
            return stream.read().decode('utf8',errors='replace')
    except OSError:return '(worker log unavailable)'

def prep_wait(process, response, timeout, log, cancel_event=None):
    started=time.monotonic();response=Path(response)
    while not response.exists():
        if cancel_event is not None and cancel_event.is_set(): raise InterruptedError('Restoration cancelled')
        if process.poll() is not None:
            if response.exists():break # A short-lived probe can publish and exit between checks.
            raise RuntimeError(f'Worker exit {process.returncode}; see {log}\n'+prep_log_tail(log))
        if time.monotonic()-started>timeout:
            process.kill();process.wait();raise TimeoutError(f'Worker exceeded {timeout}s; see {log}')
        time.sleep(.1)
    data=json.loads(response.read_text(encoding='utf8'))
    if data['state']=='failed':
        context=json.dumps(data.get('runtime_context',{}),ensure_ascii=False)
        raise RuntimeError(f"{data['error']}\nRuntime context: {context}\nWorker log: {log}")
    return data

class PrepWorker:
    def __init__(self, runtime_dir, python, model, assets, options, timeout, action='run',cancel_event=None):
        self.cancel_event=cancel_event
        options={**options,'kernel_torch_source_path':prep_kernel_torch_source()}
        self.folder=Path(runtime_dir)/'jobs'/uuid.uuid4().hex;self.folder.mkdir(parents=True)
        self.log=self.folder/'worker.log';self.stream=self.log.open('w',encoding='utf8')
        self.counter=0;vendor=assets.get('vendor') or prep_vendor(runtime_dir,model)
        if vendor:assets={**assets,'vendor':vendor}
        # UniHIR upstream fonts are relative to its unmodified source cwd.
        if model=='unihir_restoration' and 'font_dir' in assets:
            font=Path(vendor)/'font'
            font.mkdir(parents=True,exist_ok=True)
            for source in Path(assets['font_dir']).iterdir():
                if source.is_file():
                    target=font/source.name
                    if not target.is_file() or dataset_file_hash(target)!=dataset_file_hash(source):
                        temporary=target.with_suffix(target.suffix+'.tmp');shutil.copyfile(source,temporary);temporary.replace(target)
        self.config={'model':model,'assets':assets,'options':options,'action':action,
            'vendor':vendor,'ready':str((self.folder/'ready.json').resolve())}
        spec=self.folder/'worker_config.json';spec.write_text(json.dumps(self.config),encoding='utf8')
        worker=Path(runtime_dir)/('image_worker_'+self.folder.name+'.py');worker.write_text(WORKER_SOURCE,encoding='utf8')
        try:
            self.process=subprocess.Popen([str(python),'-u',str(worker.resolve()),str(spec.resolve())],
                stdin=subprocess.PIPE,stdout=self.stream,stderr=subprocess.STDOUT,text=True,
                encoding='utf8',env=gpu_worker_environment(options.get('gpu_ids',[0]),prep_worker_environment()),cwd=vendor or self.folder,
                creationflags=getattr(subprocess,'CREATE_NO_WINDOW',0))
            self.ready=prep_wait(self.process,self.config['ready'],timeout,self.log,self.cancel_event)
        except BaseException:self.close();raise
    def infer(self, input_path, seed, timeout):
        self.counter+=1;folder=self.folder/f'region_{self.counter:07d}';folder.mkdir()
        response=(folder/'response.json').resolve()
        job={'input':str(Path(input_path).resolve()),'output':str(folder.resolve()),'seed':seed,'response':str(response)}
        self.process.stdin.write(json.dumps(job)+'\n');self.process.stdin.flush()
        return folder,prep_wait(self.process,response,timeout,self.log,self.cancel_event)
    def close(self):
        if hasattr(self,'process'):
            if self.process.stdin:
                try:self.process.stdin.close()
                except OSError:pass
            try:self.process.wait(timeout=10)
            except subprocess.TimeoutExpired:self.process.kill();self.process.wait()
        self.stream.close()

def prep_overlay(packages_dir, runtime_dir, model, config):
    """A separate venv per model, local wheels only, kernel files untouched."""
    override = config['runtime_pythons'].get(model)
    if override:
        if not Path(override).is_file():
            raise FileNotFoundError(override)
        return str(Path(override).absolute())
    if model == 'realesrgan_x4':
        return sys.executable
    if model in ('difftsr', 'unihir_restoration') or model in DOCUMENT_VARIANTS:
        import importlib.util
        filename = 'learned_documents_runtime.json' if model.startswith(('docentr_', 'degan_')) else 'document_runtime.json' if model in DOCUMENT_VARIANTS else 'unihir_runtime.json' if model == 'unihir_restoration' else 'difftsr_runtime.json'
        root = Path(packages_dir) / 'image_restoration'
        plan = json.loads((root / 'manifests' / filename).read_text(encoding='utf8'))
        if 'bootstrap' not in plan:
            raise RuntimeError('Missing prepared runtime bundle; run tools/prepare_kaggle_offline.py --download-small locally')
        record = plan['bootstrap']
        helper = dataset_path(root, record['path'])
        if helper.stat().st_size != record['bytes'] or dataset_file_hash(helper) != record['sha256']:
            raise ValueError(f'Runtime bootstrap checksum mismatch: {helper}')
        spec = importlib.util.spec_from_file_location('vtm_restoration_runtime', helper)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module.prepare_runtime(packages_dir, runtime_dir, model)
    manifest = Path(packages_dir) / 'image_restoration/manifests/tadisr_runtime.json'
    if not manifest.is_file():
        raise FileNotFoundError(f'Missing {manifest}; attach the offline models Dataset')
    plan = json.loads(manifest.read_text(encoding='utf8'))
    wheel_root = manifest.parent.parent
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    tags = set(sys_tags())
    wheels = []
    for record in plan['wheels']:
        wheel = dataset_path(wheel_root, record['path'])
        if not wheel.is_file():
            raise FileNotFoundError(wheel)
        if dataset_file_hash(wheel) != record['sha256']:
            raise ValueError(f'Wheel checksum mismatch: {wheel}')
        if not tags & parse_wheel_filename(wheel.name)[3]:
            raise RuntimeError(f'Incompatible wheel ABI: {wheel.name}')
        wheels.append(str(wheel.resolve()))
    key = dataset_hash({'plan': plan, 'python': sys.version, 'prefix': sys.prefix, 'executable': sys.executable})
    folder = Path(runtime_dir) / 'environments' / ('tadisr_' + key[:16])
    python = folder / 'bin/python'
    done = folder / 'install.json'
    try:
        installed = json.loads(done.read_text(encoding='utf8'))
    except (OSError, ValueError):
        installed = {}
    if not isinstance(installed, dict):
        installed = {}
    if installed.get('key') != key or not python.is_file() or (not (folder / 'pyvenv.cfg').is_file()):
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(folder)], check=True, env=prep_worker_environment())
        if wheels:
            subprocess.run([str(python), '-m', 'pip', '--isolated', 'install', '--ignore-installed', '--no-index', '--no-deps', *wheels], check=True, env=prep_worker_environment(), stdout=subprocess.DEVNULL)
        dataset_write(folder, done.name, dataset_json_bytes({'key': key}))
    return str(python.absolute())

def prep_check_manifest(model_dir, required):
    """Verify complete file list and hashes before GPU loading (not only checkpoint)."""
    root=Path(model_dir);manifest=root/'assets.json'
    if not manifest.is_file():raise FileNotFoundError(f'Missing {manifest}; prepare/inventory assets first')
    data=json.loads(manifest.read_text(encoding='utf8'));indexed={e['path']:e for e in data['files']}
    missing=[]
    for relative in required:
        p=dataset_path(root,relative)
        if not p.is_file():missing.append(str(p));continue
        if relative not in indexed:raise ValueError(f'Asset not inventoried: {p}')
    if missing:raise FileNotFoundError('Missing assets:\n'+'\n'.join(missing))
    for relative,record in indexed.items():
        p=dataset_path(root,relative)
        if not p.is_file():raise FileNotFoundError(p)
        if p.stat().st_size!=record['bytes'] or dataset_file_hash(p)!=record['sha256']:
            raise ValueError(f'Asset checksum mismatch: {p}')
    return dataset_hash({'files':data['files']})

def prep_assets(model, models_dir, prompt_override=None):
    root=Path(models_dir)/'image_restoration'
    if model=='realesrgan_x4':
        folder=root/'realesrgan';required=['RealESRGAN_x4plus.pth']
        paths={'checkpoint':str((folder/required[0]).resolve())}
    elif model=='tadisr_x4':
        folder=root/'tadisr';base=folder/'CogView4'
        required=['tadisr_cogview4_realce_17500.pkl',
            'CogView4/model_index.json','CogView4/vae/config.json','CogView4/vae/diffusion_pytorch_model.safetensors',
            'CogView4/transformer/config.json','CogView4/transformer/diffusion_pytorch_model.safetensors.index.json',
            'CogView4/text_encoder/config.json','CogView4/text_encoder/model.safetensors.index.json',
            'CogView4/tokenizer/tokenizer.json','CogView4/tokenizer/tokenizer_config.json',
            'CogView4/tokenizer/special_tokens_map.json','CogView4/scheduler/scheduler_config.json']
        if prompt_override is None:required.append('CogView4/saved_prompt_tokens_nocfg.pt')
        for name in ('transformer/diffusion_pytorch_model.safetensors.index.json','text_encoder/model.safetensors.index.json'):
            p=base/name
            if p.is_file():required.extend('CogView4/'+str(Path(name).parent/part).replace('\\','/')
                for part in sorted(set(json.loads(p.read_text())['weight_map'].values())))
        paths={'checkpoint':str((folder/required[0]).resolve()),'base_model':str(base.resolve()),
            'prompt_embeddings':str(Path(prompt_override).resolve()) if prompt_override else str((base/'saved_prompt_tokens_nocfg.pt').resolve())}
    elif model=='difftsr':
        folder=root/'difftsr';required=['DiffTSR.ckpt','transocr.pth']
        paths={'checkpoint':str((folder/required[0]).resolve()),'transocr':str((folder/required[1]).resolve())}
    else:
        folder=root/'unihir';required=['model.safetensors','ae.safetensors','llm_config.json','vit_config.json',
            'tokenizer_config.json','tokenizer.json','vocab.json','merges.txt']
        paths={'base_model':str(folder.resolve()),'font_dir':str((folder/'font').resolve())}
        required.extend('font/'+name for name in ('KaiXinSongA.ttf','KaiXinSongB.ttf',
            'TH-Tshyn-P0.ttf','TH-Tshyn-P1.ttf','TH-Tshyn-P16.ttf','TH-Tshyn-P2.ttf'))
    identity=prep_check_manifest(folder,required)
    if model=='tadisr_x4' and dataset_file_hash(paths['checkpoint'])!='7cec8d0917305037cbafcb1b1bf81062add1613b561f5d7e2d1e599fe0baf5e0':
        raise ValueError('TADiSR adapter differs from official release checksum')
    if prompt_override:
        identity=dataset_hash({'base_assets':identity,'derived_prompt_sha256':dataset_file_hash(prompt_override)})
    return paths,identity

def prep_offline_prompt(config,runtime_dir):
    """Derive upstream's fixed prompt offline, before loading the restoration model."""
    folder=Path(config['models_dir'])/'image_restoration/tadisr'
    if (folder/'CogView4/saved_prompt_tokens_nocfg.pt').is_file():return None
    # Verify every non-derived component before loading the text encoder.
    base_required=['CogView4/'+name for name in ('model_index.json','scheduler/scheduler_config.json',
        'tokenizer/tokenizer.json','tokenizer/tokenizer_config.json','tokenizer/special_tokens_map.json',
        'text_encoder/config.json','text_encoder/model.safetensors.index.json',
        'transformer/config.json','transformer/diffusion_pytorch_model.safetensors.index.json',
        'vae/config.json','vae/diffusion_pytorch_model.safetensors')]
    for prefix,index in (('text_encoder','model.safetensors.index.json'),('transformer','diffusion_pytorch_model.safetensors.index.json')):
        p=folder/'CogView4'/prefix/index
        if not p.is_file():raise FileNotFoundError(p)
        base_required.extend(f'CogView4/{prefix}/{part}' for part in set(json.loads(p.read_text())['weight_map'].values()))
    identity=prep_check_manifest(folder,['tadisr_cogview4_realce_17500.pkl',*base_required])
    key=dataset_hash({'assets':identity,'upstream':UPSTREAM_REVISIONS['TADiSR'],'prompt_contract':'upstream_fixed_quality_nocfg_v1'})
    output=Path(runtime_dir)/'derived_prompts'/key/'saved_prompt_tokens_nocfg.pt'
    receipt=output.with_suffix('.json')
    if output.is_file() and receipt.is_file():
        cached=json.loads(receipt.read_text())
        if cached.get('sha256')==dataset_file_hash(output):return str(output)
        raise ValueError('Derived prompt cache corrupted; preserve it for review and choose a fresh runtime directory')
    python=prep_overlay(config['packages_dir'],runtime_dir,'tadisr_x4',config)
    worker=PrepWorker(runtime_dir,python,'tadisr_x4',{'base_model':str((folder/'CogView4').resolve()),
        'prompt_output':str(output.resolve())},config['model_options']['tadisr_x4'],
        config['load_timeout_seconds'],action='prepare_prompt')
    try:report=worker.ready
    finally:worker.close()
    dataset_write(output.parent,receipt.name,dataset_json_bytes({'key':key,'sha256':dataset_file_hash(output),'worker':report}))
    return str(output)

def prep_applicability(model, book, page, region, config, size=None):
    state, reason = preparation_applicability(model, book, page, region, config)
    if state != 'ready': return state, reason
    key=f'{book}:{page+1}:{region["id"]}'
    review=preparation_region_review(book,page,region['id'],config)
    if model in review.get('excluded_variants',[]): return 'not_applicable','variant explicitly excluded for this reviewed region'
    if model=='degan_watermark_removal' and not review.get('watermark_removal_approved'):
        return 'not_applicable','Requires explicit review: genuine red ink must not be treated as a watermark'
    if model=='difftsr':
        if not review.get('upright_horizontal_single_line') or not 1<=review.get('character_count',0)<=24:
            return 'not_applicable','Requires reviewed upright single horizontal line with 1..24 characters; no vertical rotation/reflow'
        if not review.get('allow_simplified_text_prior',False):
            return 'not_applicable','Upstream Text_Prediction uses zhconv zh-cn; traditional-form fidelity not established'
        box=region['source_box'];ratio=(box[2]-box[0])/(box[3]-box[1])
        if not 1<=ratio<=8:return 'not_applicable','DiffTSR requires 1 <= width/height <= 8'
        # Stricter than upstream: near-native processor ratio avoids heavy aspect warp.
        if abs(ratio/4-1)>.05:return 'not_applicable','No validated aspect-preserving 512x128 contract for this crop (require near 4:1)'
    # UniHIR is a selectable restoration experiment on every chosen page/crop.
    # It does not require a manually supplied damage label.
    return 'ready',None

def prep_preflight(config, runtime_dir):
    cfg=prep_normalize(config);report={};failures=[]
    print('Methods without runnable released resources:',json.dumps(UNAVAILABLE_RESTORATION_METHODS,ensure_ascii=False))
    if Path('/kaggle/input').exists() and Path(runtime_dir).resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Runtime artifacts must be outside /kaggle/input')
    # DiffTSR alone requires reviewed horizontal-line input; UniHIR accepts all chosen crops.
    for model in NEURAL:
        if model not in cfg['variants']:continue
        if model=='degan_watermark_removal' and not any(r.get('watermark_removal_approved') for r in cfg['reviewed_regions'].values()):
            report[model]={'state':'not_applicable','reason':'No reviewed region approves watermark removal; genuine ink is retained in original'};continue
        if model=='difftsr':
            if not any(r.get('upright_horizontal_single_line') for r in cfg['reviewed_regions'].values()):
                report[model]={'state':'not_applicable','reason':'No reviewed input matches released model contract'};continue
        try:
            prompt_override=None
            if model=='tadisr_x4' and cfg.get('generate_tadisr_prompt_offline',False):
                prompt_override=prep_offline_prompt(cfg,runtime_dir)
            assets,identity=(prep_document_assets(model,cfg['models_dir'],cfg['packages_dir']) if model in DOCUMENT_VARIANTS else prep_assets(model,cfg['models_dir'],prompt_override=prompt_override))
            python=prep_overlay(cfg['packages_dir'],runtime_dir,model,cfg)
            if model=='difftsr':assets['config']=str(Path(prep_vendor(runtime_dir,model))/'model/DiffTSR_config.yaml')
            checkpoint=Path(assets.get('checkpoint',str(Path(assets.get('base_model',''))/'model.safetensors')))
            inventory=json.loads((checkpoint.parent/'assets.json').read_text(encoding='utf8'))
            checkpoint_record=next(item for item in inventory['files'] if item['path']==checkpoint.name)
            worker=PrepWorker(runtime_dir,python,model,assets,cfg['model_options'][model],cfg['load_timeout_seconds'],action='probe')
            try:probe=worker.ready['report']
            finally:worker.close()
            report[model]={'state':'ready','assets':assets,'checkpoint_identity':identity,'python':python,'probe':probe,
                'checkpoint':{'filename':checkpoint.name,'sha256':checkpoint_record['sha256'],'bytes':checkpoint_record['bytes']},
                'vendor_identity':dataset_hash(VENDOR_SOURCES.get({'tadisr_x4':'TADiSR','difftsr':'DiffTSR','unihir_restoration':'UniHIR'}.get(model,''),{}))}
            for notice in probe.get('active_import_record_warnings',[]):
                print(f"{model}: cảnh báo source Torch kernel khác RECORD: {notice['path']}; hash thực tế lưu trong {Path(runtime_dir)/'preflight.json'}")
        except Exception as exc:
            report[model]={'state':restoration_failure_state(exc,phase='preflight'),'reason':str(exc)}
            failures.append(f'{model}: {exc}')
    Path(runtime_dir).mkdir(parents=True,exist_ok=True)
    (Path(runtime_dir)/'preflight.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
    if failures and not cfg['allow_partial']:raise RuntimeError('Preflight stopped before PDF processing:\n'+'\n'.join(failures))
    return report

def prep_classical(image, name, options):
    if name in EXTRA_CLASSICAL: return extra_classical_variant(image,name,options)
    from PIL import Image
    if name=='clahe' and (not 0<options.get('clip_limit',1.5)<=4 or not 2<=options.get('tile_grid',8)<=32):
        raise ValueError('Conservative CLAHE requires clip_limit (0,4] and tile_grid [2,32]')
    if name in ('original','clahe','bicubic_x2'):return dataset_variant(image,name,options)
    if name=='bicubic_x4':return image.resize((image.width*4,image.height*4),Image.Resampling.BICUBIC),{
        'method':'bicubic','parameters':{'scale':4},'warnings':['interpolation_does_not_recover_missing_strokes']}
    if name=='denoised':
        import cv2,numpy as np
        params={'h':2.,'template_window':7,'search_window':21,**options}
        if not 0<params['h']<=5 or params['template_window']%2!=1 or params['search_window']%2!=1 or not 3<=params['template_window']<=params['search_window']<=63:
            raise ValueError('Conservative NLM requires 0<h<=5 and odd windows')
        # Denoise luminance only: retain chromatic notes/stamps, independent from CLAHE.
        lab=cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2LAB)
        lab[:,:,0]=cv2.fastNlMeansDenoising(lab[:,:,0],None,float(params['h']),
            params['template_window'],params['search_window'])
        return Image.fromarray(cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)),{'method':'nlm_lab_l',
            'parameters':params,'warnings':['may_remove_weak_strokes; opt_in_visual_review_required']}
    raise ValueError(name)

def prep_image_qc(source, output, name):
    import cv2,numpy as np
    from PIL import Image
    expected_scale=2 if name=='bicubic_x2' else 4 if name in ('bicubic_x4','realesrgan_x4','tadisr_x4','tadisr_text_mask') else 1
    if name!='difftsr' and output.size!=(source.width*expected_scale,source.height*expected_scale):
        raise ValueError(f'{name}: invalid output dimensions {output.size}')
    if name=='difftsr' and abs(output.width/output.height-source.width/source.height)>.06:
        raise ValueError('DiffTSR geometry distortion exceeds rounding tolerance')
    # Luminance comparison after area reduction is diagnostic, not a glyph-fidelity score.
    reduced=output.convert('RGB').resize(source.size,Image.Resampling.BOX)
    a=cv2.cvtColor(np.asarray(source),cv2.COLOR_RGB2GRAY)
    b=cv2.cvtColor(np.asarray(reduced),cv2.COLOR_RGB2GRAY)
    def components(gray):
        _,binary=cv2.threshold(gray,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
        return int(cv2.connectedComponents(binary)[0]-1)
    before,after=components(a),components(b)
    return {'state':'needs_visual_review' if name!='original' else 'verified_pixel_identity',
        'expected_scale':expected_scale if name!='difftsr' else None,
        'luminance_mae_at_source':float(np.abs(a.astype(float)-b.astype(float)).mean()),
        'components_before':before,'components_after':after,
        'component_loss_warning':after<before*.9,'glyph_fidelity':'unverified',
        'ocr_accuracy':'not_measured'}

def prep_usage(root):return sum(p.stat().st_size for p in Path(root).rglob('*') if p.is_file())

def prep_store(root,image,budget):
    with _prep_io_lock:
        # Pre-encode to enforce storage bound before object write; content addressing deduplicates.
        image=image.convert('RGB');stream=io.BytesIO();image.save(stream,format='PNG',compress_level=6)
        data=stream.getvalue();sha=hashlib.sha256(data).hexdigest();relative=f'objects/{sha[:2]}/{sha}.png'
        p=dataset_path(root,relative)
        if not p.exists() or dataset_file_hash(p)!=sha:
            if budget['used']+len(data)>budget['limit']:raise RuntimeError('max_storage_bytes exceeded')
            dataset_write(root,relative,data);budget['used']+=len(data)
        return {'path':relative,'sha256':sha,'size':list(image.size),'pixel_sha256':hashlib.sha256(image.tobytes()).hexdigest()}

def prep_receipt_valid(root, receipt, key):
    data=json.loads(dataset_path(root,receipt).read_bytes())
    if data['key']!=key or data['state']!='ok':raise ValueError('Incomplete/mismatched cache')
    dataset_load_image(root,data['image'])
    for extra in data.get('extras',{}).values():
        if 'pixel_sha256' in extra:dataset_load_image(root,extra)
        else:dataset_read(root,extra)
    return data

def prep_run(config, runtime_dir):
    global _storage_guard
    cfg=prep_normalize(config);root=Path(cfg['output_dir']).resolve();root.mkdir(parents=True,exist_ok=True)
    if Path('/kaggle/input').exists() and root.is_relative_to(Path('/kaggle/input').resolve()):raise ValueError('Output must not be in /kaggle/input')
    for key in ('data_dir','models_dir','packages_dir'):
        if not cfg[key] or not Path(cfg[key]).is_dir():raise FileNotFoundError(f'Configure valid {key}: {cfg[key]}')
        if root==Path(cfg[key]).resolve() or root.is_relative_to(Path(cfg[key]).resolve()):raise ValueError('Output must be outside input assets')
    started=time.monotonic();preflight=prep_preflight(cfg,runtime_dir)
    _storage_guard={'root':root,'used':prep_usage(root),'limit':cfg['max_storage_bytes']}
    # Conservative pre-allocation estimate after bounded PDF selection; no downscale fallback.
    # Canonical receipts are staged, and dataset.json is published only after restoration QC.
    canonical_config=ImageDatasetConfig(variants=['original'],max_source_pixels=cfg['max_source_pixels'],**cfg['layout'])
    profile_dir=Path(runtime_dir)/'profiles';profile_dir.mkdir(parents=True,exist_ok=True)
    for entry in list_pdfs(cfg['data_dir']):
        profile=cfg.get('book_layouts',{}).get(entry['stem'],{})
        (profile_dir/(entry['stem']+'.json')).write_text(json.dumps(profile),encoding='utf8')
    stats=prepare_image_dataset(cfg['data_dir'],root,config=canonical_config,books=cfg['books'],
        pages=cfg['pages'],limit=cfg['limit'],max_books=cfg['max_books'],profile_dir=profile_dir,publish=False)
    _,base=dataset_open(stats['dataset']);budget={'used':prep_usage(root),'limit':cfg['max_storage_bytes']}
    if budget['used']>budget['limit']:raise RuntimeError('Canonical dataset exceeds max_storage_bytes')
    plan=plan_restoration_variants(root,base,cfg,applicability=prep_applicability)
    live_plan={**plan,'storage_already_used_bytes':budget['used'],'storage_limit_bytes':budget['limit'],
        'storage_budget_risk_before_cache_deduplication':budget['used']+plan['additional_storage_estimate_bytes']>budget['limit']}
    dataset_write(root,'last_plan.json',dataset_json_bytes(live_plan))
    print('Restoration plan:',json.dumps(live_plan,ensure_ascii=False))
    if plan['resource_warnings'] and not cfg['allow_partial']:
        raise MemoryError('Restoration preflight exceeds max_pixel limits; explicitly select smaller regions or review limits; no downscale')
    if live_plan['storage_budget_risk_before_cache_deduplication']:
        print('Storage estimate exceeds budget before cache/deduplication; hard byte limit remains enforced at every write. Review last_plan.json before a large run.')
    stage=Path(runtime_dir)/('manifest_'+uuid.uuid4().hex+'.jsonl');stage.parent.mkdir(parents=True,exist_ok=True)
    counts={};measurements=[];reused=0;failures=[];rows=0
    def write_row(stream,row):
        nonlocal rows
        with _prep_io_lock:
            row['runtime_status']={'state':row['state'],'inference_verified':row['state']=='ok',
                'device':'cpu' if row['variant'] in CLASSICAL else 'worker_report',
                'verification_scope':'this output only; not OCR accuracy or other devices'}
            row.setdefault('parameters',row.get('operation',{}).get('parameters',cfg['variant_options'].get(row['variant'],cfg['model_options'].get(row['variant'],{}))))
            stream.write(dataset_json_bytes(row));stream.flush();rows+=1
            counts[row['state']]=counts.get(row['state'],0)+1
    # Lock covers variant generation/publication; canonical builder has its own lock.
    with dataset_lock(root),stage.open('wb') as manifest:
        def run_variant(name,cancel_event):
            nonlocal reused
            with stage.with_name(stage.stem+'_'+name+'.jsonl').open('wb') as manifest:
                worker=None
                try:
                    for entry in base['pages']:
                        record=json.loads(dataset_read(root,entry['record']))
                        for region in record['regions']:
                            if cancel_event.is_set(): raise InterruptedError('Restoration cancelled')
                            original=region['images']['original'];x1,y1,x2,y2=region['source_box']
                            common={'book':entry['book'],'page':entry['page']+1,'page_index':entry['page'],
                                'source_sha256':entry['source_sha256'],'source_pdf':entry['source_pdf'],
                                'source_image':record['source'],'source_image_hash':record['source']['sha256'],
                                'source_box':region['source_box'],'crop_type':region['crop_type'],
                                'region_id':region['id'],'reading_order':region['reading_order'],
                                'canonical_image':original,'variant':name,'provenance':PROVENANCE[name],
                                'method_identity':variant_spec(name),'input_pixel_sha256':original['pixel_sha256']}
                            state,reason='ready',None
                            if region['crop_type'] not in cfg['crop_types']:state,reason='skipped_by_configuration','crop_type deselected'
                            elif cfg.get('region_ids') and region['id'] not in cfg['region_ids']:state,reason='skipped_by_configuration','region_id deselected'
                            else:state,reason=prep_applicability(name,entry['book'],entry['page'],region,cfg)
                            common['applicability']={'state':state,'reason':reason,'evidence':'explicit_configuration; no_automatic_degradation_classification'}
                            model_info=preflight.get(name,{})
                            if state=='ready' and name in NEURAL and model_info.get('state')!='ready':
                                state,reason=model_info.get('state','failed'),model_info.get('reason','Model not ready')
                            if state!='ready':
                                row={**common,'state':state,'reason':reason};write_row(manifest,row)
                                if name=='tadisr_x4':write_row(manifest,{**row,'variant':'tadisr_text_mask'})
                                if state in PREPARATION_FAILURE_STATES:failures.append(row)
                                continue
                            options=classical_parameters(name,cfg['variant_options'].get(name,{})) if name in EXTRA_CLASSICAL else cfg['model_options'].get(name,cfg['variant_options'].get(name,{}))
                            identity={'model':model_info.get('checkpoint_identity'),'checkpoint':model_info.get('checkpoint'),
                                'architecture':{'realesrgan_x4':'RRDBNet(3,3,64,23,32), scale=4','tadisr_x4':'CogView4-RealCE with upstream joint segmentation decoder',
                                    'difftsr':'DiffTSR released IDM/TDM/MoM + TransOCR prior','unihir_restoration':'UniHIR BAGEL/MLLM'} .get(name),
                                'vendor':model_info.get('vendor_identity'),
                                'runtime':model_info.get('probe'), 'document_task':globals().get('DOCUMENT_VARIANTS',{}).get(name)}
                            # PID is diagnostic, never an input identity; changing process must not invalidate cache.
                            if identity['runtime']:
                                identity['runtime']=json.loads(json.dumps({k:v for k,v in identity['runtime'].items() if k!='pid'}))
                                for device in identity['runtime'].get('devices',[]): device.pop('free_bytes',None)
                            key=dataset_hash({'source':original['sha256'],'box':region['source_box'],'name':name,'options':options,
                                'model':identity,'code':RESTORATION_CODE_SIGNATURE,'opencv':__import__('cv2').__version__})
                            receipt=f'restoration_cache/{key}.json';result=None
                            if cfg['resume']:
                                try:
                                    result=prep_receipt_valid(root,receipt,key)
                                    with _prep_io_lock: reused+=1
                                except (OSError,ValueError,KeyError):pass
                            if result is None:
                                tick=time.monotonic()
                                try:
                                    source=dataset_load_image(root,original)
                                    scale=4 if name in ('bicubic_x4','realesrgan_x4','tadisr_x4') else 2 if name=='bicubic_x2' else 1
                                    if source.width*source.height*scale**2>cfg['max_output_pixels']:
                                        raise ValueError(f'{name} output exceeds max_output_pixels; split crop or explicitly raise limit')
                                    extras={};info={};operation={'method':name,'parameters':options}
                                    if name in CLASSICAL:output,operation=prep_classical(source,name,options)
                                    else:
                                        if worker is None:
                                            worker=PrepWorker(runtime_dir,model_info['python'],name,model_info['assets'],options,cfg['load_timeout_seconds'],cancel_event=cancel_event)
                                        folder,info=worker.infer(dataset_path(root,original['path']),int(key[:8],16),cfg['timeout_seconds'])
                                        info['model_load_seconds']=worker.ready.get('load_seconds')
                                        from PIL import Image
                                        with Image.open(folder/'result.png') as loaded:output=loaded.convert('RGB')
                                        for extra,relative in info['extras'].items():
                                            p=dataset_path(folder,relative)
                                            if p.suffix=='.png':
                                                with Image.open(p) as im:
                                                    if im.width*im.height>cfg['max_output_pixels']:raise ValueError('Extra output exceeds pixel budget')
                                                    extras[extra]=prep_store(root,im,budget)
                                            else:
                                                data=p.read_bytes();h=hashlib.sha256(data).hexdigest()
                                                extras[extra]=dataset_write(root,f'diagnostics/{h}.json',data)
                                    qc=prep_image_qc(source,output,name)
                                    if name=='tadisr_x4' and 'tadisr_text_mask' not in extras:
                                        raise ValueError('TADiSR must emit both SR image and text mask')
                                    if 'tadisr_text_mask' in extras:
                                        mask=dataset_load_image(root,extras['tadisr_text_mask'])
                                        if mask.size!=output.size:raise ValueError('TADiSR mask and SR dimensions differ')
                                    image=prep_store(root,output,budget)
                                    mapping=[(x2-x1)/output.width,0,0,(y2-y1)/output.height,x1,y1]
                                    result={'key':key,'state':'ok','image':{**image,'image_to_source':mapping},'extras':extras,
                                        'operation':operation,'model_identity':identity,'qc':qc,'seconds':round(time.monotonic()-tick,6),
                                        'coordinate_mapping_evidence':'nominal_canvas_only; generated_content_unverified' if name=='unihir_restoration' else 'crop_and_scale_geometry',
                                        'worker':info,'warnings':['possible_added_removed_merged_strokes; inspect_against_original'] if name in NEURAL else operation.get('warnings',[])}
                                    dataset_write(root,receipt,dataset_json_bytes(result))
                                    # Worker scratch is bounded per region; completed artifacts are already hashed/atomic.
                                    if name in NEURAL:
                                        for p in folder.iterdir():
                                            if p.is_file():p.unlink()
                                        folder.rmdir()
                                except Exception as exc:
                                    result={'key':key,'state':restoration_failure_state(exc),'reason':str(exc),'seconds':round(time.monotonic()-tick,6)}
                                    dataset_write(root,receipt,dataset_json_bytes(result));failures.append({**common,**result})
                                    # Restart a failed/timed-out worker for the next region.
                                    if worker is not None:worker.close();worker=None
                            row={**common,**result};write_row(manifest,row)
                            if result['state']=='ok':measurements.append({k:result[k] for k in ('seconds','worker')}|{'variant':name})
                            if name=='tadisr_x4':
                                mask=result.get('extras',{}).get('tadisr_text_mask')
                                maskrow={**row,'variant':'tadisr_text_mask'}
                                if mask:
                                    maskrow['image']={**mask,'image_to_source':row['image']['image_to_source']}
                                    maskrow['provenance']='neural_restoration';maskrow['warnings']=['model_predicted_mask_is_not_source_character_evidence']
                                elif result['state']=='ok':maskrow.update(state='inference_failed',reason='Missing paired TADiSR text mask')
                                write_row(manifest,maskrow)
                finally:
                    if worker is not None:worker.close()
        names=[n for n in cfg['variants'] if n!='tadisr_text_mask']
        if cfg['execution_mode']=='parallel_models':
            # Small models run concurrently; heavy sharded models get their own wave.
            names=([n for n in names if n in CLASSICAL]+
                [n for n in names if n not in CLASSICAL and len(cfg['model_options'][n]['gpu_ids'])==1]+
                [n for n in names if n not in CLASSICAL and len(cfg['model_options'][n]['gpu_ids'])>1])
            tasks=[(n,[] if n in CLASSICAL else cfg['model_options'][n]['gpu_ids']) for n in names]
            run_gpu_task_waves(tasks,run_variant,cfg['gpu_ids'])
        else:
            from threading import Event
            cancel_event=Event()
            for name in names: run_variant(name,cancel_event)
        for name in names:
            shard=stage.with_name(stage.stem+'_'+name+'.jsonl')
            with shard.open('rb') as stream: shutil.copyfileobj(stream,manifest)
            shard.unlink()
        manifest.flush();os.fsync(manifest.fileno())
    data=stage.read_bytes();sha=hashlib.sha256(data).hexdigest()
    manifest_ref=dataset_write(root,f'restoration_manifests/{sha}.jsonl',data)
    snapshot_preflight=json.loads(json.dumps(preflight))
    for info in snapshot_preflight.values():
        if 'probe' in info:info['probe'].pop('pid',None)
    report={'format':'vtm-image-restoration','version':1,'canonical_snapshot':base['snapshot_id'],
        'manifest':manifest_ref,'selected_variants':cfg['variants'],'crop_types':cfg['crop_types'],
        'states':counts,'partial':bool(failures),'preflight':snapshot_preflight,'provenance_categories':sorted(set(PROVENANCE.values())),
        'configuration':cfg,'code_sha256':RESTORATION_CODE_SIGNATURE,'cost_plan':plan,'not_integrated_methods':UNAVAILABLE_RESTORATION_METHODS}
    report_ref=dataset_write(root,f'restoration_manifests/{dataset_hash(report)}.json',dataset_json_bytes(report))
    # Validate every row, hash and expected pair before publishing immutable snapshot/marker.
    prep_validate_extension(root,base,report)
    summary={'canonical':stats,'states':counts,'manifest_rows':rows,'variants_reused':reused,
        'seconds':round(time.monotonic()-started,3),'storage_bytes':prep_usage(root),'measurements':measurements,
        'report':report_ref,'failures':len(failures),'cost_plan':plan,'not_integrated_methods':UNAVAILABLE_RESTORATION_METHODS}
    if failures and not cfg['allow_partial']:
        dataset_write(root,'last_attempt.json',dataset_json_bytes(summary))
        raise RuntimeError(f'{len(failures)} variants failed; completed cache retained; dataset.json not replaced. See last_attempt.json')
    final={k:v for k,v in base.items() if k!='snapshot_id'}
    final['restoration']={'format':'vtm-image-restoration','version':1,'report':report_ref,'partial':bool(failures)}
    final['snapshot_id']=dataset_hash(final)
    with dataset_lock(root):
        dataset_write(root,f'snapshots/{final["snapshot_id"]}.json',dataset_json_bytes(final))
        dataset_write(root,'dataset.json',dataset_json_bytes(final))
    summary['snapshot_id']=final['snapshot_id'];summary['dataset']=str(root/'dataset.json')
    dataset_write(root,'last_run.json',dataset_json_bytes(summary))
    return summary

def prep_validate_extension(root, base, report):
    expected={}
    for entry in base['pages']:
        record=json.loads(dataset_read(root,entry['record']));dataset_validate_page(root,record)
        for r in record['regions']:
            for name in report['selected_variants']:
                expected[(entry['book'],entry['page']+1,r['id'],name)]=r
    seen=set();states={}
    for line in dataset_read(root,report['manifest']).splitlines():
        row=json.loads(line);key=(row['book'],row['page'],row['region_id'],row['variant'])
        if key not in expected or key in seen:raise ValueError('Unexpected/duplicate restoration manifest entry')
        seen.add(key);region=expected[key]
        states[row['state']]=states.get(row['state'],0)+1
        if row['provenance']!=PROVENANCE[row['variant']]:raise ValueError('Invalid evidence provenance')
        if row['source_box']!=region['source_box'] or row['reading_order']!=region['reading_order']:
            raise ValueError('Restoration crop geometry changed')
        if row['state']=='ok':
            image=dataset_load_image(root,row['image']);x1,y1,x2,y2=region['source_box']
            scale=2 if row['variant']=='bicubic_x2' else 4 if row['variant'] in ('bicubic_x4','realesrgan_x4','tadisr_x4','tadisr_text_mask') else 1
            if row['variant']!='difftsr' and image.size!=((x2-x1)*scale,(y2-y1)*scale):
                raise ValueError('Restoration variant dimensions violate scale contract')
            if row['variant']=='original' and row['image']['pixel_sha256']!=row['canonical_image']['pixel_sha256']:
                raise ValueError('Original variant differs from canonical crop pixels')
            if row['image']['image_to_source']!=[(x2-x1)/image.width,0,0,(y2-y1)/image.height,x1,y1]:raise ValueError('Invalid source mapping')
            for ref in row.get('extras',{}).values():dataset_read(root,ref)
        elif row['state'] not in {'not_applicable','skipped_by_configuration'} | PREPARATION_FAILURE_STATES:
            raise ValueError('Invalid variant state')
    if seen!=set(expected):raise ValueError('Selected variants silently omitted')
    if states!=report['states']:raise ValueError('Manifest state counts differ from report')
    if any(states.get(s,0) for s in PREPARATION_FAILURE_STATES) and not report['partial']:
        raise ValueError('Incomplete selected methods falsely labelled complete')
    return {'rows':len(seen),'validated':True,'glyph_fidelity':'unverified'}

def prep_open(directory):
    root,base=dataset_open(directory)
    extension=base.get('restoration')
    if not extension or extension['version']!=1:raise ValueError('Missing/unsupported restoration extension')
    report=json.loads(dataset_read(root,extension['report']))
    return root,base,report

def prep_validate(directory):
    root,base,report=prep_open(directory)
    core=validate_image_dataset(directory)
    return {'canonical':core,'restoration':prep_validate_extension(root,base,report),'partial':report['partial']}

def prep_image_export_plan(directory):
    """Name successful snapshot images by method/book/page/crop, with portable paths."""
    import re, unicodedata
    root, descriptor, report = prep_open(directory)

    def book_slug(book):
        text = unicodedata.normalize('NFC', str(book))
        text = 'book'
        if re.fullmatch('(?i)(CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])', text.split('.')[0]):
            text = '_' + text
        return text
    books = {(p['book'], p['source_sha256']) for p in descriptor['pages']}
    groups = {}
    for identity in sorted(books):
        groups.setdefault(book_slug(identity[0]).casefold(), []).append(identity)
    names = {}
    for identities in groups.values():
        for identity in identities:
            suffix = '--' + dataset_hash(list(identity))[:8] if len(identities) > 1 else ''
            names[identity] = book_slug(identity[0]) + suffix
    entries = []
    seen = set()
    methods = {}
    skipped = {}
    for line in dataset_read(root, report['manifest']).splitlines():
        row = json.loads(line)
        if row['state'] != 'ok':
            skipped[row['state']] = skipped.get(row['state'], 0) + 1
            continue
        method = row['variant']
        if method not in SUPPORTED:
            raise ValueError(f'Unknown image export method: {method}')
        page = row['page']
        order = row['reading_order']
        kind = row['crop_type']
        if not isinstance(page, int) or page < 1 or (not isinstance(order, int)) or (order < 0):
            raise ValueError('Image export requires 1-based pages and nonnegative crop order')
        if kind not in ('page', 'group', 'column'):
            raise ValueError(f'Unknown export crop type: {kind}')
        suffix = '' if kind == 'page' else f'-{kind}{order + 1:03d}'
        book = names[row['book'], row['source_sha256']]
        relative = f'{method}/{book}-p{page:05d}{suffix}.png'
        if relative.casefold() in seen:
            raise ValueError(f'Duplicate readable image name: {relative}')
        seen.add(relative.casefold())
        source = dataset_path(root, row['image']['path'])
        if source.suffix.lower() != '.png' or not source.is_file():
            raise ValueError(f'Missing PNG: {source}')
        entries.append({'name': relative, 'source': source, 'sha256': row['image']['sha256']})
        methods[method] = methods.get(method, 0) + 1
    if not entries:
        raise ValueError('No successful selected images to export')
    return (root, sorted(entries, key=lambda entry: entry['name']), {'files': len(entries), 'methods': dict(sorted(methods.items())), 'skipped': skipped, 'partial': report['partial'], 'snapshot_id': descriptor['snapshot_id']})

def prep_export_images(directory, destination):
    """Refresh only managed images; keep snapshot/cache files under the dataset root."""
    root,entries,summary=prep_image_export_plan(directory)
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    receipt_root=destination/'_dataset'
    receipt=f'exports/{dataset_hash(str(destination))}.json'
    try:previous=json.loads(dataset_path(receipt_root,receipt).read_text(encoding='utf8'))
    except FileNotFoundError:previous={}
    if not isinstance(previous,dict):raise ValueError('Invalid image export receipt')
    owned=previous.get('files',{})
    if not isinstance(owned,dict):raise ValueError('Invalid image export file list')
    for relative in owned:
        parts=Path(relative).parts
        if len(parts)!=2 or parts[0] not in SUPPORTED or Path(relative).suffix.lower()!='.png':
            raise ValueError(f'Invalid managed image path: {relative}')
        dataset_path(destination,relative)
    current={entry['name']:entry['sha256'] for entry in entries}
    # Validate every source and every replacement before modifying readable output.
    for entry in entries:
        if dataset_file_hash(entry['source'])!=entry['sha256']:
            raise ValueError(f"Image checksum mismatch: {entry['source']}")
        target=dataset_path(destination,entry['name'])
        if target.exists():
            digest=dataset_file_hash(target)
            if digest not in (entry['sha256'],owned.get(entry['name'])):
                raise FileExistsError(f'Unmanaged or edited image already exists: {target}')
    for entry in entries:
        target=dataset_path(destination,entry['name']);target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists() and dataset_file_hash(target)==entry['sha256']:continue
        temporary=target.with_name('.'+target.name+'.'+uuid.uuid4().hex+'.tmp')
        try:
            try:os.link(entry['source'],temporary) # Immutable objects; avoid duplicate storage where supported.
            except OSError:shutil.copyfile(entry['source'],temporary)
            temporary.replace(target)
        finally:
            if temporary.exists():temporary.unlink()
    # A new selection removes only unchanged files owned by the previous export.
    for relative,digest in owned.items():
        if relative in current:continue
        target=dataset_path(destination,relative)
        if target.is_file() and dataset_file_hash(target)==digest:
            target.unlink()
            try:target.parent.rmdir()
            except OSError:pass
    dataset_write(receipt_root,receipt,dataset_json_bytes({'destination':str(destination),'files':current}))
    return {**summary,'directory':str(destination)}

def prep_package_images(directory, archive):
    """Stream only final PNGs into method folders; replace ZIP only after success."""
    import zipfile
    root,entries,summary=prep_image_export_plan(directory)
    archive=Path(archive).absolute()
    if archive.suffix.lower()!='.zip':raise ValueError('Image archive must have a .zip extension')
    archive.parent.mkdir(parents=True,exist_ok=True)
    temporary=archive.with_name('.'+archive.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with zipfile.ZipFile(temporary,'x',compression=zipfile.ZIP_STORED,allowZip64=True) as output:
            for entry in entries:
                info=zipfile.ZipInfo(entry['name'],date_time=(1980,1,1,0,0,0))
                info.external_attr=0o100644<<16;hasher=hashlib.sha256()
                with entry['source'].open('rb') as source,output.open(info,'w',force_zip64=True) as target:
                    for block in iter(lambda:source.read(1024*1024),b''):
                        hasher.update(block);target.write(block)
                if hasher.hexdigest()!=entry['sha256']:
                    raise ValueError(f"Image checksum mismatch: {entry['source']}")
        temporary.replace(archive)
    finally:
        if temporary.exists():temporary.unlink()
    return {**summary,'archive':str(archive),'bytes':archive.stat().st_size}

def prep_previews(directory, maximum=3, detail_box=(.1,.15,.4,.45)):
    from PIL import Image,ImageDraw
    root,base,report=prep_open(directory);chosen={}
    # Bounded number of identical regions, include every selected method/status for comparison.
    for line in dataset_read(root,report['manifest']).splitlines():
        row=json.loads(line);key=(row['book'],row['page'],row['region_id'])
        if key not in chosen and len(chosen)<maximum and row['crop_type'] in ('group','page'):chosen[key]=[]
        if key in chosen:chosen[key].append(row)
    outputs=[]
    if detail_box and (len(detail_box)!=4 or not 0<=detail_box[0]<detail_box[2]<=1 or not 0<=detail_box[1]<detail_box[3]<=1):
        raise ValueError('Preview detail_box requires fractional x1,y1,x2,y2 in [0,1]')
    for key,rows in chosen.items():
        cell_w,cell_h=300,350;columns=min(4,len(rows));height=((len(rows)+columns-1)//columns)*cell_h
        sheet=Image.new('RGB',(columns*cell_w,height),'#eeeeee');draw=ImageDraw.Draw(sheet)
        for i,row in enumerate(rows):
            left,top=i%columns*cell_w,i//columns*cell_h
            draw.text((left+8,top+5),row['variant']+' / '+row['state'],fill='black')
            if row['state']=='ok':
                image=dataset_load_image(root,row['image']);image.thumbnail((cell_w-16,cell_h-50))
                sheet.paste(image,(left+8,top+30))
            else:
                reason=row.get('reason','');draw.multiline_text((left+8,top+35),'\n'.join(reason[j:j+38] for j in range(0,min(240,len(reason)),38)),fill='black')
        path=root/'previews'/f'{key[0]}_p{key[1]}_{key[2]}.png';path.parent.mkdir(exist_ok=True)
        tmp=path.with_suffix('.tmp');sheet.save(tmp,format='PNG');tmp.replace(path);outputs.append(str(path))
        if detail_box:
            # All panels show exactly the same subregion in canonical source coordinates.
            detail=Image.new('RGB',(columns*400,((len(rows)+columns-1)//columns)*420),'#eeeeee')
            draw_detail=ImageDraw.Draw(detail)
            for i,row in enumerate(rows):
                left,top=i%columns*400,i//columns*420
                draw_detail.text((left+8,top+5),row['variant']+' / source-aligned detail',fill='black')
                if row['state']=='ok':
                    image=dataset_load_image(root,row['image'])
                    box=[round(detail_box[j]*image.size[j%2]) for j in range(4)]
                    box[0]=min(box[0],image.width-1);box[1]=min(box[1],image.height-1)
                    box[2]=min(image.width,max(box[0]+1,box[2]));box[3]=min(image.height,max(box[1]+1,box[3]))
                    patch=image.crop(box)
                    factor=min(384/patch.width,380/patch.height)
                    patch=patch.resize((max(1,round(patch.width*factor)),max(1,round(patch.height*factor))),
                        Image.Resampling.NEAREST if row['variant']=='original' else Image.Resampling.LANCZOS)
                    detail.paste(patch,(left+8,top+30))
                else:draw_detail.text((left+8,top+30),row['state'],fill='black')
            path=path.with_name(path.stem+'_detail.png');tmp=path.with_suffix('.tmp')
            detail.save(tmp,format='PNG');tmp.replace(path);outputs.append(str(path))
    return outputs


In [ ]:
# Chỉ dùng để kiểm thử checksum bằng fixture tổng hợp.
def assets_inventory(folder,license_info,source_info):
    folder=Path(folder);files=[]
    for path in sorted(folder.rglob('*')):
        if path.is_file() and path.name!='assets.json' and not path.name.endswith('.part'):
            files.append({'path':path.relative_to(folder).as_posix(),'sha256':dataset_file_hash(path),'bytes':path.stat().st_size})
    obj={'format':'vtm-local-model-assets','version':1,'license':license_info,'sources':source_info,'files':files,
        'inference_verified':False}
    folder.mkdir(parents=True,exist_ok=True);target=folder/'assets.json'
    data=dataset_json_bytes(obj)
    # Inventories are regenerated from actual files; weights themselves are never overwritten.
    temporary=target.with_suffix('.tmp');temporary.write_bytes(data);temporary.replace(target)
    return obj


## 5. Tự kiểm thử nhúng

Fixture PDF/image là tổng hợp; kiểm tra native/fallback/rotation/overlay/mask,
tọa độ và RTL, cả năm phương pháp cổ điển, missing assets, process isolation,
resume sau ngắt, đổi tham số/hash trọng số, ảnh hỏng, manifest, pixel/storage budget.
Test tiling dùng tensor thật và hàm identity: **không phải inference model thật**.
Kiểm thử không cần file trong tests/scripts. Nó không đo CER hay fidelity từng glyph.


In [ ]:
import zlib
from PIL import Image, ImageDraw
def make_scan_pdf(path, *, size=(120,80), pages=2, rotation=0, clip=False,
                  overlay=False, multiple=False, annotation=False, reflected=False,
                  cropped=False, alpha=False, inverted=False):
    """Minimal explicit PDF fixture exercises transformations PDFium must honor."""
    w,h = size
    image = Image.new("RGB",size,"#e8ddc4")
    draw = ImageDraw.Draw(image)
    for x in range(12,w-10,12):
        for y in range(10,h-10,12):
            draw.rectangle((x,y,x+3,y+6),fill="black")
    image.putpixel((2,3),(255,0,0))
    objects = [b"<< /Type /Catalog /Pages 2 0 R >>",b""]
    def add(data):
        objects.append(data); return len(objects)
    def stream(options,data):
        return b"<< "+options+b" /Length "+str(len(data)).encode()+b" >>\nstream\n"+data+b"\nendstream"
    mask_id = add(stream(f"/Type /XObject /Subtype /Image /Width {w} /Height {h} /ColorSpace /DeviceGray /BitsPerComponent 8 /Filter /FlateDecode".encode(),zlib.compress(bytes([128])*(w*h)))) if alpha else None
    image_id = add(stream((f"/Type /XObject /Subtype /Image /Width {w} /Height {h} /ColorSpace /DeviceRGB /BitsPerComponent 8 /Filter /FlateDecode"+
        (f" /SMask {mask_id} 0 R" if alpha else "")+ (" /Decode [1 0 1 0 1 0]" if inverted else "")).encode(),zlib.compress(image.tobytes())))
    page_ids = []
    for index in range(pages):
        command = (f"q {'10 0 '+str(w-10)+' '+str(h)+' re W n ' if clip else ''}"
                   f"{-w if reflected else w} 0 0 {h} {w if reflected else 0} 0 cm /Scan Do Q\n")
        if overlay:
            command += "0 0 1 rg 5 5 10 10 re f\n"
        if multiple:
            command += "q 20 0 0 20 1 1 cm /Scan Do Q\n"
        content = add(stream(b"",command.encode()))
        annot = add(b"<< /Type /Annot /Subtype /Square /Rect [3 3 30 30] /C [1 0 0] /F 4 >>") if annotation else None
        page_ids.append(add((f"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 {w} {h}] /Rotate {rotation} "+
            (f"/CropBox [5 5 {w-5} {h-5}] " if cropped else "")+
            f"/Resources << /XObject << /Scan {image_id} 0 R >> >> /Contents {content} 0 R "+
            (f"/Annots [{annot} 0 R] " if annot else "")+">>").encode()))
    objects[1] = f"<< /Type /Pages /Count {pages} /Kids [{' '.join(str(i)+' 0 R' for i in page_ids)}] >>".encode()
    output = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for i,obj in enumerate(objects,1):
        offsets.append(len(output)); output.extend(f"{i} 0 obj\n".encode()+obj+b"\nendobj\n")
    start = len(output)
    output.extend(f"xref\n0 {len(objects)+1}\n0000000000 65535 f \n".encode())
    for offset in offsets[1:]:
        output.extend(f"{offset:010d} 00000 n \n".encode())
    output.extend(f"trailer\n<< /Size {len(objects)+1} /Root 1 0 R >>\nstartxref\n{start}\n%%EOF\n".encode())
    Path(path).write_bytes(output)
    return image


import unittest,tempfile,copy,types,importlib.util
from unittest.mock import patch

def prep_test_config(folder):
    folder=Path(folder);inputs=folder/'input';inputs.mkdir(exist_ok=True)
    models=folder/'models';models.mkdir(exist_ok=True);packages=folder/'packages';packages.mkdir(exist_ok=True)
    return prep_normalize({'data_dir':str(inputs),'models_dir':str(models),'packages_dir':str(packages),
        'output_dir':str(folder/'dataset'),'books':['scan'],'pages':'1-2','limit':2,'max_books':1,
        'variants':['original','clahe','bicubic_x2','bicubic_x4','denoised'],
        'crop_types':['page','group','column'],'resume':True,'allow_partial':False,
        'max_source_pixels':1000000,'max_output_pixels':16000000,'max_storage_bytes':100000000,
        'timeout_seconds':20,'load_timeout_seconds':90,'batch_size':1,'deskew':False,'dewarp':False,
        'accept_unihir_noncommercial':False,'runtime_pythons':{},'reviewed_regions':{},'book_layouts':{},
        'layout':{'dpi':72,'column_cuts':[.25,.5,.75]},'variant_options':{},
        'model_options':{'realesrgan_x4':{'tile':64,'overlap':8,'device':'cuda','half':False},
            'tadisr_x4':{'tile':64,'overlap':32,'device':'cuda','seed':42},
            'difftsr':{'steps':20,'device':'cuda'},'unihir_restoration':{'steps':50,'ref_count':6,'device':'cuda'}}})

class OfflineImageTests(unittest.TestCase):

    def test_pdf_native_and_composition_fallback(self):
        import pypdfium2 as pdfium
        from PIL import ImageChops
        with tempfile.TemporaryDirectory() as directory:
            for options in ({}, {'rotation': 90}, {'cropped': True}, {'overlay': True}, {'alpha': True}, {'annotation': True}):
                path = Path(directory) / 'scan.pdf'
                source = make_scan_pdf(path, pages=1, **options)
                with pdfium.PdfDocument(path) as doc:
                    p = doc[0]
                    image, metadata = acquire_pdf_image(p, dpi=72)
                    if not options:
                        self.assertEqual(metadata['method'], 'native_validated')
                        self.assertEqual(source.tobytes(), image.tobytes())
                    else:
                        self.assertEqual(metadata['method'], 'pdf_render')
                        bitmap = p.render(scale=metadata['render_scale'], draw_annots=True)
                        expected = bitmap.to_pil().convert('RGB')
                        bitmap.close()
                        self.assertIsNone(ImageChops.difference(expected, image).getbbox())
                    p.close()

    def test_analysis_mapping_partition_and_rtl(self):
        from PIL import Image
        image = Image.new('RGB', (601, 503), 'white')
        cfg = ImageDatasetConfig(analysis_max_side=123, column_cuts=[0.17, 0.34, 0.51, 0.68, 0.85])
        layout, _, _ = dataset_plan_layout(image, cfg)
        self.assertEqual(layout['analysis_size'], [123, 103])
        self.assertEqual(layout['analysis_to_source'], [601 / 123, 0, 0, 503 / 103, 0, 0])
        for kind in ('page', 'group', 'column'):
            regions = [r for r in layout['regions'] if r['crop_type'] == kind]
            dataset_partition_qc(regions, image.size, 'vertical_rtl')
            self.assertEqual(sorted((r['reading_order'] for r in regions)), list(range(len(regions))))
            self.assertEqual(sum(((r['source_box'][2] - r['source_box'][0]) * 503 for r in regions)), 601 * 503)

    def test_all_classical_dimensions_alignment(self):
        from PIL import Image, ImageDraw
        im = Image.new('RGB', (64, 48), 'white')
        ImageDraw.Draw(im).rectangle((13, 10, 25, 37), fill='black')
        for method in CLASSICAL:
            output, _ = prep_classical(im, method, {})
            qc = prep_image_qc(im, output, method)
            self.assertEqual(qc['glyph_fidelity'], 'unverified')
            if method == 'original':
                self.assertEqual(output.tobytes(), im.tobytes())

    def test_missing_model_fails_before_pdf(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg['variants'] = ['original', 'realesrgan_x4']
            with self.assertRaisesRegex(RuntimeError, 'Preflight stopped'):
                prep_run(cfg, Path(d) / 'runtime')
            self.assertFalse((Path(cfg['output_dir']) / 'canonical.json').exists())

    def test_manifest_resume_invalidation_corruption(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            runtime = Path(d) / 'runtime'
            first = prep_run(cfg, runtime)
            original_acquire = globals()['acquire_pdf_image']

            def no_acquire(*a, **kw):
                raise AssertionError('Unnecessary source acquisition')
            globals()['acquire_pdf_image'] = no_acquire
            try:
                second = prep_run(cfg, runtime)
            finally:
                globals()['acquire_pdf_image'] = original_acquire
            self.assertEqual(first['snapshot_id'], second['snapshot_id'])
            self.assertEqual(second['states']['ok'], second['variants_reused'])
            cfg['variant_options'] = {'clahe': {'clip_limit': 2.0}}
            third = prep_run(cfg, runtime)
            self.assertNotEqual(second['snapshot_id'], third['snapshot_id'])
            self.assertEqual(third['canonical']['source_reused'], 2)
            root, base, report = prep_open(cfg['output_dir'])
            rows = [json.loads(l) for l in dataset_read(root, report['manifest']).splitlines()]
            victim = next((r for r in rows if r['variant'] == 'clahe'))
            dataset_path(root, victim['image']['path']).write_bytes(b'corrupt')
            with self.assertRaises(ValueError):
                prep_validate(cfg['output_dir'])
            repaired = prep_run(cfg, runtime)
            dataset_load_image(root, victim['image'])
            self.assertTrue(prep_validate(cfg['output_dir'])['restoration']['validated'])

    def test_interruption_preserves_last_published_snapshot(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg['variants'] = ['original']
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            runtime = Path(d) / 'runtime'
            first = prep_run(cfg, runtime)
            cfg['variants'] += ['clahe']
            original = globals()['prep_classical']
            calls = [0]

            def interrupt(im, name, opts):
                if name == 'clahe':
                    calls[0] += 1
                    if calls[0] == 2:
                        raise KeyboardInterrupt('simulated interruption')
                return original(im, name, opts)
            globals()['prep_classical'] = interrupt
            try:
                with self.assertRaises(KeyboardInterrupt):
                    prep_run(cfg, runtime)
            finally:
                globals()['prep_classical'] = original
            self.assertEqual(dataset_open(cfg['output_dir'])[1]['snapshot_id'], first['snapshot_id'])
            completed = prep_run(cfg, runtime)
            self.assertGreater(completed['variants_reused'], 0)
            self.assertNotEqual(completed['snapshot_id'], first['snapshot_id'])

    def test_failure_does_not_publish_complete_marker(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            cfg.update(variants=['original', 'clahe'], crop_types=['page'], pages='1', limit=1)
            original = prep_classical

            def fail_variant(image, name, options):
                if name == 'clahe':
                    raise RuntimeError('simulated variant failure')
                return original(image, name, options)
            with patch.dict(globals(), {'prep_classical': fail_variant}):
                with self.assertRaisesRegex(RuntimeError, 'variants failed'):
                    prep_run(cfg, Path(d) / 'runtime')
            root = Path(cfg['output_dir'])
            self.assertFalse((root / 'dataset.json').exists())
            self.assertFalse(json.loads((root / 'last_plan.json').read_text(encoding='utf8'))['resource_warnings'])
            attempt = json.loads((root / 'last_attempt.json').read_text(encoding='utf8'))
            self.assertEqual(attempt['failures'], 1)
            self.assertEqual(attempt['states']['ok'], 1)

    def test_preflight_pixel_limit_does_not_publish_complete_marker(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            cfg.update(variants=['original', 'bicubic_x2'], crop_types=['page'], pages='1', limit=1, max_output_pixels=10000)

            def unexpected_variant(*args):
                self.fail('Pixel preflight must stop before variant execution')
            with patch.dict(globals(), {'prep_classical': unexpected_variant}):
                with self.assertRaisesRegex(MemoryError, 'Restoration preflight exceeds'):
                    prep_run(cfg, Path(d) / 'runtime')
            self.assertFalse((Path(cfg['output_dir']) / 'dataset.json').exists())
            plan = json.loads((Path(cfg['output_dir']) / 'last_plan.json').read_text(encoding='utf8'))
            self.assertEqual({row['variant'] for row in plan['resource_warnings']}, {'bicubic_x2'})

    def test_not_applicable_and_partial_states_are_explicit(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg['variants'] = ['original', 'difftsr', 'unihir_restoration', 'realesrgan_x4']
            cfg['allow_partial'] = True
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            result = prep_run(cfg, Path(d) / 'runtime')
            self.assertGreater(result['states']['not_applicable'], 0)
            self.assertGreater(result['states']['missing_assets'], 0)
            self.assertTrue(prep_validate(cfg['output_dir'])['partial'])

    def test_contract_does_not_rotate_or_reflow(self):
        cfg = prep_test_config(tempfile.mkdtemp())
        r = {'id': 'column_000', 'source_box': [0, 0, 20, 200]}
        self.assertEqual(prep_applicability('difftsr', 'scan', 0, r, cfg)[0], 'not_applicable')
        cfg['reviewed_regions']['scan:1:column_000'] = {'upright_horizontal_single_line': True, 'character_count': 12, 'allow_simplified_text_prior': True}
        self.assertEqual(prep_applicability('difftsr', 'scan', 0, r, cfg)[0], 'not_applicable')

    def test_worker_process_isolation(self):
        with tempfile.TemporaryDirectory() as d:
            worker = PrepWorker(d, sys.executable, 'realesrgan_x4', {}, {}, 90, action='isolation_probe')
            try:
                self.assertNotEqual(worker.ready['pid'], os.getpid())
                self.assertTrue(worker.ready['offline'])
                self.assertTrue(worker.ready['network_block_verified'])
            finally:
                worker.close()

    def test_thin_tiled_coverage_real_tensor_identity(self):
        import torch
        spec = importlib.util.spec_from_loader('embedded_image_worker', loader=None)
        mod = importlib.util.module_from_spec(spec)
        import ast
        tree = ast.parse(WORKER_SOURCE)
        for node in tree.body:
            if isinstance(node, ast.FunctionDef) and node.name in ('starts', 'tiled_tadisr'):
                exec(compile(ast.Module(body=[node], type_ignores=[]), '<tiling_test>', 'exec'), mod.__dict__)
        for h, w in ((211, 19), (17, 199), (80, 90), (17, 19)):
            a = torch.rand(1, 3, h, w)
            out, mask, info = mod.tiled_tadisr(a, lambda p: (p, p[:, :1] * 0), 64, 32)
            self.assertTrue(torch.allclose(a, out, atol=2e-06))
            self.assertTrue(info['covered_all_pixels'])

    def test_storage_and_output_limits(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            cfg['max_storage_bytes'] = 1024
            with self.assertRaisesRegex(RuntimeError, 'max_storage_bytes'):
                prep_run(cfg, Path(d) / 'runtime')
            self.assertFalse((Path(cfg['output_dir']) / 'dataset.json').exists())

    def test_path_traversal_rejected(self):
        with self.assertRaises(ValueError):
            dataset_path('.', '../escape.png')

    def test_checkpoint_identity_changes_cache_key(self):
        with tempfile.TemporaryDirectory() as d:
            root = Path(d)
            p = root / 'test.pth'
            p.write_bytes(b'first')
            assets_inventory(root, 'synthetic fixture', [])
            a = prep_check_manifest(root, ['test.pth'])
            p.write_bytes(b'second')
            assets_inventory(root, 'synthetic fixture', [])
            b = prep_check_manifest(root, ['test.pth'])
            self.assertNotEqual(a, b)

    def test_kernel_torch_record_source_drift_keeps_native_strict(self):
        import ast, base64, importlib, importlib.metadata as md, site
        function = next((node for node in ast.parse(WORKER_SOURCE).body if isinstance(node, ast.FunctionDef) and node.name == 'verify_active_records'))
        scope = {'pathlib': __import__('pathlib'), 'hashlib': hashlib, 'sys': sys}
        exec(compile(ast.Module(body=[function], type_ignores=[]), 'worker-record-audit', 'exec'), scope)
        with tempfile.TemporaryDirectory() as d:
            root = Path(d)
            source = root / 'torch/__init__.py'
            source.parent.mkdir()
            native = root / 'torch/_C.fixture.so'
            source.write_bytes(b'original source')
            native.write_bytes(b'original native')

            def record(path):
                digest = base64.urlsafe_b64encode(hashlib.sha256(path.read_bytes()).digest()).decode().rstrip('=')
                return f'{path.relative_to(root).as_posix()},sha256={digest},{path.stat().st_size}\n'
            metadata = {'RECORD': record(source) + record(native), 'WHEEL': 'Tag: py3-none-any\n'}
            dist = types.SimpleNamespace(version='2.11.0', read_text=metadata.get, locate_file=lambda r: root / r)
            module = types.SimpleNamespace(__file__=str(source), __version__='2.11.0', _C=types.SimpleNamespace(__file__=str(native)))
            with patch.object(md, 'distribution', return_value=dist), patch.object(site, 'getsitepackages', return_value=[str(root / 'different-site-directory')]), patch.object(importlib, 'import_module', return_value=module):
                audit = scope['verify_active_records']
                self.assertEqual(len(audit('torch', 'torch')['verified']), 2)
                source.write_bytes(b'simulated kernel source patch')
                with self.assertRaisesRegex(RuntimeError, 'RECORD checksum differs'):
                    audit('torch', 'torch')
                result = audit('torch', 'torch', kernel_torch_source_policy='warn', kernel_torch_source_path=str(source))
                self.assertEqual(result['verified'], [str(native.resolve())])
                self.assertEqual(result['warnings'][0]['status'], 'kernel_source_record_mismatch')
                self.assertNotEqual(result['warnings'][0]['expected_hash'], result['warnings'][0]['observed_hash'])
                native.write_bytes(b'changed native')
                with self.assertRaisesRegex(RuntimeError, 'RECORD checksum differs'):
                    audit('torch', 'torch', kernel_torch_source_policy='warn', kernel_torch_source_path=str(source))

    def test_legacy_settings_use_kaggle_source_policy(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg.pop('kernel_torch_source_policy', None)
            with patch.object(Path, 'is_dir', return_value=True):
                normalized = prep_normalize(cfg)
            self.assertEqual(normalized['kernel_torch_source_policy'], 'warn')
            self.assertTrue(all((o['kernel_torch_source_policy'] == 'warn' for o in normalized['model_options'].values())))

    def test_overlay_preserves_real_venv_interpreter(self):
        with tempfile.TemporaryDirectory() as d:
            folder = Path(d) / 'venv'
            subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(folder)], check=True, env=prep_worker_environment(), capture_output=True)
            python = folder / 'bin/python'
            selected = prep_overlay('unused', 'unused', 'tadisr_x4', {'runtime_pythons': {'tadisr_x4': str(python)}})
            prefix = subprocess.check_output([selected, '-c', 'import sys; print(sys.prefix)'], env=prep_worker_environment(), text=True).strip()
            self.assertEqual(Path(prefix).resolve(), folder.resolve())

    def test_worker_passes_captured_kernel_origin_to_real_subprocess(self):
        import base64
        with tempfile.TemporaryDirectory() as d:
            runtime = Path(d) / 'runtime'
            source = runtime / 'torch/__init__.py'
            source.parent.mkdir(parents=True)
            source.write_text("from types import SimpleNamespace\nfrom . import _C\n__version__='0.0.0'\nversion=SimpleNamespace(cuda=None)\ncuda=SimpleNamespace(is_available=lambda:False)\ndef from_numpy(a):return SimpleNamespace(numpy=lambda:a)\n", encoding='utf8')
            native = source.parent / '_C.py'
            native.write_bytes(b'# native-file fixture\n')
            info = runtime / 'torch-0.0.0.dist-info'
            info.mkdir()
            (info / 'METADATA').write_text('Metadata-Version: 2.1\nName: torch\nVersion: 0.0.0\n')
            (info / 'WHEEL').write_text('Wheel-Version: 1.0\nTag: py3-none-any\n')
            rows = []
            for file in (source, native):
                digest = base64.urlsafe_b64encode(hashlib.sha256(file.read_bytes()).digest()).decode().rstrip('=')
                rows.append(f'{file.relative_to(runtime).as_posix()},sha256={digest},{file.stat().st_size}\n')
            (info / 'RECORD').write_text(''.join(rows), encoding='utf8')
            with source.open('a', encoding='utf8') as stream:
                stream.write('# simulated kernel source patch\n')
            with patch.object(importlib.util, 'find_spec', return_value=types.SimpleNamespace(origin=str(source))):
                worker = PrepWorker(runtime, sys.executable, 'realesrgan_x4', {}, {'device': 'cpu', 'kernel_torch_source_policy': 'warn'}, 30, action='probe')
            try:
                report = worker.ready['report']
                warning, = report['active_import_record_warnings']
                self.assertEqual(warning['path'], str(source.resolve()))
                self.assertNotEqual(warning['expected_hash'], warning['observed_hash'])
                self.assertNotIn(str(source.resolve()), report['active_import_records_verified'])
            finally:
                worker.close()

    def test_missing_paired_tadisr_mask_blocks_publication(self):
        from PIL import Image
        original_preflight = globals()['prep_preflight']
        original_worker = globals()['PrepWorker']

        class IncompleteBackend:

            def __init__(self, *a, **kw):
                self.ready = {}
                self.index = 0

            def infer(self, input_path, seed, timeout):
                self.index += 1
                folder = Path(runtime) / str(self.index)
                folder.mkdir(parents=True)
                with Image.open(input_path) as image:
                    image.resize((image.width * 4, image.height * 4)).save(folder / 'result.png')
                return (folder, {'extras': {}, 'seconds': 0.0, 'state': 'ok'})

            def close(self):
                pass
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg['variants'] = ['original', 'tadisr_x4']
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf')
            runtime = Path(d) / 'runtime'
            globals()['prep_preflight'] = lambda *a: {'tadisr_x4': {'state': 'ready', 'python': sys.executable, 'assets': {}, 'checkpoint_identity': 'synthetic_backend_for_missing_mask_failure_test'}}
            globals()['PrepWorker'] = IncompleteBackend
            try:
                with self.assertRaisesRegex(RuntimeError, 'variants failed'):
                    prep_run(cfg, runtime)
            finally:
                globals()['prep_preflight'] = original_preflight
                globals()['PrepWorker'] = original_worker
            self.assertFalse((Path(cfg['output_dir']) / 'dataset.json').exists())

    def test_method_folders_and_image_only_zip(self):
        import zipfile
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            output = Path(d) / 'readable'
            cfg['output_dir'] = str(output / '_dataset')
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf', pages=2)
            prep_run(cfg, Path(d) / 'runtime')
            exported = prep_export_images(cfg['output_dir'], output)
            archive = Path(d) / 'images.zip'
            packed = prep_package_images(cfg['output_dir'], archive)
            self.assertEqual(exported['files'], packed['files'])
            self.assertEqual({p.name for p in output.iterdir()}, {'_dataset', *cfg['variants']})
            with zipfile.ZipFile(archive) as zipped:
                self.assertIsNone(zipped.testzip())
                self.assertIn('original/scan-p00001.png', zipped.namelist())
                self.assertIn('clahe/scan-p00002-column001.png', zipped.namelist())
                self.assertTrue(all((len(name.split('/')) == 2 and name.endswith('.png') for name in zipped.namelist())))
                for name in zipped.namelist():
                    self.assertEqual(zipped.read(name), (output / name).read_bytes())

    def test_unihir_missing_assets_are_not_silently_skipped(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg.update(variants=['original', 'unihir_restoration'], allow_partial=True, reviewed_regions={}, accept_unihir_noncommercial=False)
            report = prep_preflight(cfg, Path(d) / 'runtime')
            self.assertEqual(report['unihir_restoration']['state'], 'missing_assets')
            for kind in ('page', 'group', 'column'):
                region = {'id': kind + '_000', 'source_box': [0, 0, 30, 100]}
                self.assertEqual(prep_applicability('unihir_restoration', 'scan', 0, region, cfg), ('ready', None))

    def test_unihir_all_pages_crops_validate_and_export_without_damage_labels(self):
        with tempfile.TemporaryDirectory() as d:
            cfg = prep_test_config(d)
            cfg.update(variants=['original', 'unihir_restoration'], reviewed_regions={}, accept_unihir_noncommercial=False)
            make_scan_pdf(Path(cfg['data_dir']) / 'scan.pdf', pages=2)
            runtime = Path(d) / 'runtime'

            class IdentityFixture:

                def __init__(self, *args, **kwargs):
                    self.ready = {'load_seconds': 0}

                def infer(self, input_path, seed, timeout):
                    folder = runtime / 'synthetic_unihir' / uuid.uuid4().hex
                    folder.mkdir(parents=True)
                    with Image.open(input_path) as image:
                        image.save(folder / 'result.png')
                    return (folder, {'state': 'ok', 'extras': {}, 'seconds': 0, 'diagnostics': {'fixture': 'identity; not real neural inference'}})

                def close(self):
                    pass
            preflight = {'unihir_restoration': {'state': 'ready', 'python': sys.executable, 'assets': {}, 'checkpoint_identity': 'synthetic_identity_fixture'}}
            with patch.dict(globals(), {'PrepWorker': IdentityFixture, 'prep_preflight': lambda *args: preflight}):
                first = prep_run(cfg, runtime)
                self.assertTrue(prep_validate(cfg['output_dir'])['restoration']['validated'])
                root, descriptor, report = prep_open(cfg['output_dir'])
                rows = [json.loads(line) for line in dataset_read(root, report['manifest']).splitlines()]
                rows = [row for row in rows if row['variant'] == 'unihir_restoration']
                self.assertEqual({row['page'] for row in rows}, {1, 2})
                self.assertEqual({row['crop_type'] for row in rows}, {'page', 'group', 'column'})
                self.assertTrue(all((row['state'] == 'ok' and row['provenance'] == 'generative_reconstruction' for row in rows)))
                packed = prep_package_images(cfg['output_dir'], Path(d) / 'images.zip')
                self.assertEqual(packed['methods']['unihir_restoration'], len(rows))
                second = prep_run(cfg, runtime)
                self.assertEqual(second['snapshot_id'], first['snapshot_id'])
                self.assertEqual(second['variants_reused'], first['states']['ok'])

def run_image_selftests():
    suite=unittest.defaultTestLoader.loadTestsFromTestCase(OfflineImageTests)
    result=unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():raise RuntimeError('Embedded selftests failed; stop before processing real PDFs')
    return {'tests':result.testsRun,'failures':len(result.failures),'errors':len(result.errors),
        'real_neural_inference':False,'note':'Synthetic PDF/image/tensor tests plus a real isolated subprocess'}


## 6. Chạy offline trên Kaggle

Gắn Dataset packages, models và PDF; điền đường dẫn ở ô cấu hình rồi Run All.
Notebook chỉ dùng asset đã gắn, không tải từ Internet. Giữ phạm vi một trang trước khi mở rộng.


In [ ]:
CONFIG['variants'] = select_preparation_variants(VARIANT_MODE, VARIANTS)
if RUN_PREPARATION:
    for key in ('packages_dir','models_dir','data_dir'):
        if not CONFIG[key] or not Path(CONFIG[key]).is_dir():
            raise FileNotFoundError(f'Điền đường dẫn {key} hợp lệ trong ô cấu hình: {CONFIG[key]!r}')
    BASE_ENVIRONMENT = prep_bootstrap(PACKAGES_DIR,RUNTIME_DIR)
    print('Môi trường cơ sở:',BASE_ENVIRONMENT)
    SELFTEST_REPORT = run_image_selftests() if RUN_SELF_TESTS else {'skipped_by_configuration':True}
    SUMMARY = prep_run(CONFIG,RUNTIME_DIR)
    print(json.dumps({k:v for k,v in SUMMARY.items() if k!='measurements'},ensure_ascii=False,indent=2))


## 7. Xác minh và xem cùng vùng giữa các phương pháp

`source_observation`, `deterministic_transformation`, `neural_restoration` và
`generative_reconstruction` là các loại bằng chứng khác nhau. UniHIR có thể thêm
chữ không được pixel gốc xác nhận; intermediate predictions lưu JSON riêng,
không đưa vào canonical/OCR. Text mask TADiSR là dự đoán, không phải chứng cứ có chữ.

Contact sheet dùng cùng region, giữ original để so nét thêm/mất, bộ thủ gộp,
chữ gãy, nền sắc giả và seam. `overlap_patch_mae` chỉ cảnh báo bất đồng ở tile,
không thay kiểm tra bằng mắt. Hãy thử thêm vùng mờ/hiếm bằng `BOOKS=['4']; PAGES='50'; LIMIT=1`.
NLM giữ mặc định không chọn; component count giảm là cảnh báo, không phải kết luận mất chữ.


In [ ]:
if RUN_PREPARATION:
    if VALIDATE_OUTPUT:
        VALIDATION = prep_validate(DATASET_DIR)
        print(json.dumps(VALIDATION,ensure_ascii=False,indent=2))
    PREVIEWS = prep_previews(DATASET_DIR,maximum=3) if EXPORT_PREVIEWS else []
    try:
        from IPython.display import display,Image as DisplayImage
        for path in PREVIEWS: display(DisplayImage(filename=path))
    except ImportError:
        print('Preview PNG:',PREVIEWS)


## 8. Ảnh theo phương pháp và ZIP chỉ ảnh

`OUTPUT_DIR` mặc định là `/kaggle/working/image_dataset`. Mỗi phương pháp có một
thư mục ảnh riêng, dùng cùng tên sách–trang–vùng để dễ đối chiếu:

```text
image_dataset/
  original/
    1-p00001.png
    1-p00001-group001.png
    1-p00001-column001.png
  clahe/
    1-p00001.png
    1-p00001-group001.png
    1-p00001-column001.png
  bicubic_x2/
  bicubic_x4/
  realesrgan_x4/
  tadisr_x4/
  tadisr_text_mask/
  _dataset/
```

Chỉ các phương pháp có ảnh thành công mới xuất hiện. Trang đếm từ 1, thêm số 0
để sắp xếp đúng thứ tự. `group001`/`column001` là thứ tự vùng trong nhóm cắt,
theo hướng đọc đã cấu hình. Tên sách giữ Unicode; tên không hợp lệ hoặc trùng
trên Windows được xử lý để không ghi đè ảnh của sách khác.

`_dataset/` gom metadata, snapshot, cache và preview phục vụ resume/xác minh.
`DATASET_DIR = OUTPUT_DIR + '/_dataset'` là đường dẫn dùng cho notebook 3:
`PREPARED_IMAGES=DATASET_DIR`, `PREPARED_VARIANT='original'`. Giữ nguyên thư mục
này và `RESUME=True` để tiếp tục xử lý mà không tạo lại phần đã hoàn thành.

Bật `EXPORT_DATASET_ZIP=True` để tạo `DATASET_ZIP`. ZIP **chỉ chứa ảnh PNG**,
mỗi phương pháp một folder ngay ở gốc ZIP, giống cây ảnh bên trên. Chỉ lấy ảnh
`ok` của snapshot hiện tại. Chạy lại sẽ cập nhật ZIP sau khi kiểm tra checksum.
ZIP ảnh dùng để đọc/đối chiếu; để notebook 3 resume bằng manifest, giữ `_dataset/`.

Có thể xuất lại dataset cũ mà không chạy model: đặt `RUN_PREPARATION=False`,
trỏ `DATASET_DIR` đến thư mục cũ có `dataset.json`, đặt `OUTPUT_DIR` là thư mục
ảnh muốn xuất, rồi chạy ô cuối. Nếu chỉ cần ZIP, đặt `EXPORT_IMAGE_FOLDERS=False`
và `EXPORT_DATASET_ZIP=True`.

Nguồn/canonical, các trạng thái thiếu hoặc lỗi và provenance vẫn được lưu trong
`_dataset/`. Các phép đo ảnh không kết luận OCR tốt hơn hoặc glyph đúng. Không tự upload.


In [ ]:
if (Path(DATASET_DIR) / 'dataset.json').is_file():
    root,descriptor,restoration = prep_open(DATASET_DIR)
    print('Dataset:',root/'dataset.json')
    print('Snapshot:',descriptor['snapshot_id'])
    print('Trạng thái:',restoration['states'])
    print('Không tự upload. Chưa đo OCR accuracy/CER.')
    # Ví dụ đọc tuần tự: chỉ nạp image của region cần xem.
    # for line in dataset_read(root,restoration['manifest']).splitlines():
    #     row=json.loads(line)
    #     if row['state']=='ok' and row['variant']=='tadisr_x4':
    #         image=dataset_load_image(root,row['image'])
    #         print(row['source_box'],row['image']['image_to_source'])
    #         break

    if EXPORT_IMAGE_FOLDERS:
        IMAGE_EXPORT = prep_export_images(DATASET_DIR, OUTPUT_DIR)
        print('Ảnh theo phương pháp:', IMAGE_EXPORT['directory'], IMAGE_EXPORT['methods'])
    if EXPORT_DATASET_ZIP:
        IMAGE_ARCHIVE = prep_package_images(DATASET_DIR, DATASET_ZIP)
        print('ZIP chỉ ảnh:', IMAGE_ARCHIVE['archive'], IMAGE_ARCHIVE['files'], 'PNG')
